In [1]:
import os, subprocess, sys, time, json, shutil, platform
from pathlib import Path
started = time.time()
print({'python': sys.version, 'libc': platform.libc_ver(), 'machine': platform.machine()})
os.environ.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1', TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='1', VLLM_USE_DEEP_GEMM='0')


{'python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]', 'libc': ('glibc', '2.35'), 'machine': 'x86_64'}


In [2]:
import csv
gpu_csv = subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader,nounits'], text=True)
gpu_inventory = list(csv.reader(gpu_csv.splitlines()))
assert len(gpu_inventory) == 4 and all('L4' in row[1] for row in gpu_inventory), gpu_inventory
print(open('/proc/meminfo').read().split('\nSwap')[0], flush=True)
print(subprocess.run(['nvidia-smi', 'topo', '-p2p', 'r'], capture_output=True, text=True).stdout, flush=True)
print('cpus', os.cpu_count(), len(os.sched_getaffinity(0)), flush=True)
os.makedirs('/kaggle/tmp', exist_ok=True)
print(subprocess.run(['df', '-h', '/kaggle/working', '/kaggle/tmp', '/tmp', '/dev/shm'], capture_output=True, text=True).stdout, flush=True)
subprocess.check_call([sys.executable, '-m', 'venv', '--without-pip', '--clear', '/kaggle/tmp/venv-probe'])
subprocess.check_call([sys.executable, '-m', 'pip', '--python', '/kaggle/tmp/venv-probe/bin/python', '--version'])


MemTotal:       197858988 kB
MemFree:        191151976 kB
MemAvailable:   194481372 kB
Buffers:          812736 kB
Cached:          3885728 kB
 	GPU0	GPU1	GPU2	GPU3	
 GPU0	X	OK	OK	OK	
 GPU1	OK	X	OK	OK	
 GPU2	OK	OK	X	OK	
 GPU3	OK	OK	OK	X	

Legend:

  X    = Self
  OK   = Status Ok
  CNS  = Chipset not supported
  GNS  = GPU not supported
  TNS  = Topology not supported
  NS   = Not supported
  U    = Unknown

cpus 48 46
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  2.1M   20G   1% /kaggle/working
overlay         8.0T  7.0T  1.1T  88% /
overlay         8.0T  7.0T  1.1T  88% /
shm              86G     0   86G   0% /dev/shm



Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)


0

In [3]:
deployment = {'package': 'vllm', 'vllm_version': '0.30.0+merlin252.sm89', 'wheel_sources': ['nisshokuitsuki/qwen-merlin-h95-wheels', 'nisshokuitsuki/qwen-scipy-wheel'], 'wheel_kernel_sources': [], 'model_architecture': 'Qwen4ExpForConditionalGeneration', 'model_files': {'.gitattributes': {'size_bytes': 1635}, 'README.md': {'size_bytes': 2418}, 'chat_template.jinja': {'size_bytes': 8952}, 'config.json': {'size_bytes': 266426}, 'generation_config.json': {'size_bytes': 214}, 'model-00001-of-00017.safetensors': {'size_bytes': 2899311784}, 'model-00003-of-00017.safetensors': {'size_bytes': 5370548192}, 'model-00004-of-00017.safetensors': {'size_bytes': 5370910416}, 'model-00005-of-00017.safetensors': {'size_bytes': 5370140704}, 'model-00006-of-00017.safetensors': {'size_bytes': 5370205184}, 'model-00007-of-00017.safetensors': {'size_bytes': 5370927288}, 'model-00008-of-00017.safetensors': {'size_bytes': 5344445352}, 'model-00009-of-00017.safetensors': {'size_bytes': 5370355648}, 'model-00010-of-00017.safetensors': {'size_bytes': 5370927288}, 'model-00011-of-00017.safetensors': {'size_bytes': 5324173744}, 'model-00012-of-00017.safetensors': {'size_bytes': 5370438520}, 'model-00013-of-00017.safetensors': {'size_bytes': 5370927288}, 'model-00014-of-00017.safetensors': {'size_bytes': 5342689856}, 'model-00015-of-00017.safetensors': {'size_bytes': 5050474736}, 'model-00017-of-00017.safetensors': {'size_bytes': 1284526608}, 'model-ple-00001-of-00022.safetensors': {'size_bytes': 4800023992}, 'model-ple-00002-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00003-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00004-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00005-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00006-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00007-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00008-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00009-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00010-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00011-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00012-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00013-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00014-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00015-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00016-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00017-of-00022.safetensors': {'size_bytes': 4800024000}, 'model-ple-00018-of-00022.safetensors': {'size_bytes': 4800024008}, 'model-ple-00019-of-00022.safetensors': {'size_bytes': 4800024008}, 'model-ple-00020-of-00022.safetensors': {'size_bytes': 4800024008}, 'model-ple-00021-of-00022.safetensors': {'size_bytes': 4800024008}, 'model-ple-00022-of-00022.safetensors': {'size_bytes': 1600008000}, 'model.safetensors.index.json': {'size_bytes': 23794973}, 'model_extra_tensors.safetensors': {'size_bytes': 5214492152}, 'ple_reshard_manifest.json': {'size_bytes': 16582}, 'preprocessor_config.json': {'size_bytes': 443}, 'processor_config.json': {'size_bytes': 1220}, 'quantization_config.json': {'size_bytes': 94026}, 'tokenizer.json': {'size_bytes': 19989325}, 'tokenizer_config.json': {'size_bytes': 1165}}, 'wheel_files': {'agent_detector-2.0.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '22c6a1c9c23894a12f55aea625f060b8d3330fba774de72d02d5442bb580417f'}, 'aiohappyeyeballs-2.7.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '9243213661e29250eb41368e5daa826fc017156c3b8a11440826b2e3ed376472'}, 'aiohttp-3.14.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '543906c127fb1d929b95076db19b83fa2d46751006ff1e23b093aa5ac4d8db42'}, 'aiosignal-1.4.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '053243f8b92b990551949e63930a839ff0cf0b0ebbe0597b0f3fb19e1a0fe82e'}, 'annotated_doc-0.0.5-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101'}, 'annotated_types-0.8.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'f072f4d804ea359e4eaf198b1af7a8b0943881a87f31bb764f8bf219bb9419e0'}, 'anthropic-1.7.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '6b681b6ee00f232bb54f50f9a0d6d30e7be368c1b6f754bfd470c8385b8b6058'}, 'anyio-4.15.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101'}, 'apache_tvm_ffi-0.1.11-cp312-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2843f084cdc94dedacd8b257a395a2b71b8a3dc7fc99711b148bf1d161983128'}, 'astor-0.8.1-py2.py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '070a54e890cefb5b3739d19f30f5a5ec840ffc9c50ffa7d23cc9fc1a38ebbfc5'}, 'attrs-26.1.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c647aa4a12dfbad9333ca4e71fe62ddc36f4e63b2d260a37a8b83d2f043ac309'}, 'blake3-1.0.9-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'bcd56a7a972c4185070f7042ccc20166927eec3c0f98b8405f375d007b604a0b'}, 'cachetools-7.2.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '3045213f186b89fdd95d94441354c4bd87c570b7a38d8c3fd1d9dd37f6dc90d8'}, 'cbor2-6.1.4-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '310f3dfb296ba48fe9b63c5cf26e691e3548a1eae6901d2f0c18e941d151f220'}, 'certifi-2026.7.22-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775'}, 'cffi-2.1.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c1453022f490d2459a11819d83ad1d586e9ff65a12ac3e705ffebd46d3685dcf'}, 'charset_normalizer-3.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08'}, 'click-8.5.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360'}, 'cloudpickle-3.1.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '9acb47f6afd73f60dc1df93bb801b472f05ff42fa6c84167d25cb206be1fbf4a'}, 'compressed_tensors-0.17.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '4a1b89b508f7efb8ffb4eee8a6e69e0452d9b080cae130146025c64fbe9fa9aa'}, 'cryptography-50.0.1-cp311-abi3-manylinux_2_34_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '51afcfceb15597cf2635068e4ac9a56b2abde622edde17f37d85fd7b5306497a'}, 'cuda_bindings-13.4.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '40c811caf26976eb319462db614422112438f1d02b4f3df3a4a96f9bbcc77b6d'}, 'cuda_core-1.2.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'cd36063535f75a88c4eb44ba9a678441491a5f67a72f62c2e3b797e8a052d438'}, 'cuda_pathfinder-1.8.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e'}, 'cuda_python-13.4.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b6b114882beb8325139a318a02bcf8feb821dfd724f6f6104c262e2d1e2035ef'}, 'cuda_tile-1.6.0-cp312-cp312-manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7e38f97bfedff359c926710387a40fbfc8f36efb7fe9ad2cb759d5a098eedde3'}, 'cuda_toolkit-13.0.3.0-py2.py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f'}, 'depyf-0.20.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd31effad4261cebecb58955d832e448ace88f432328f95f82fd99c30fd9308d4'}, 'detect_installer-0.2.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '87e2ce7d05cb108b178f8e45bb80d3d88c0cb4b397983345c18ad6562e02e620'}, 'dill-0.4.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '1e1ce33e978ae97fcfcff5638477032b801c46c7c65cf717f95fbc2248f79a9d'}, 'dnspython-2.8.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '01d9bbc4a2d76bf0db7c1f729812ded6d912bd318d3b1cf81d30c0f845dbf3af'}, 'docstring_parser-0.18.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b3fcbed555c47d8479be0796ef7e19c2670d428d72e96da63f3a40122860374b'}, 'einops-0.8.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '54058201ac7087911181bfec4af6091bb59380360f069276601256a76af08193'}, 'email_validator-2.3.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '80f13f623413e6b197ae73bb10bf4eb0908faf509ad8362c5edeb0be7fd450b4'}, 'fastapi-0.136.3-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '3d2a69bdf04b7e9f3afa292c3bc7a98816bbfafa10bc9b45f3f3700d2f761620'}, 'fastapi_cli-0.0.32-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '8dcc286fa32f01bbd3f65dd09cfd5a2540ed5f2230b77db7fd30978d6165f3c4'}, 'fastapi_cloud_cli-0.26.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '94029f92f3dc4289b376e1213f0d6cc7c30d4f32de7e47b5effc6dc754ab9ce9'}, 'fastar-0.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'deb0b3882626832361dd0fcee1f4ef1c6e2a7aaa4165e51c86ae0bab95caccd6'}, 'fastsafetensors-0.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd1ecbb199d98d222c827ae1cc9c7b09dd9ef2d34aa0eb01baa8f81f70345dc0a'}, 'filelock-4.0.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '481a321a27bef441e23c53371c6abc8d7d16e26b97090074ba44f7538a3fd55a'}, 'flashinfer_python-0.6.18.post1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'adb6d2952b47131138e108bd2999b1fe18b71f1a4c0b2684f168753db9563991'}, 'frozenlist-1.8.0-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '494a5952b1c597ba44e0e78113a7266e656b9794eec897b19ead706bd7074383'}, 'fsspec-2026.9.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f'}, 'googleapis_common_protos-1.75.3-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'a018d2bf098ca9fb6faa08d5bb780e2a2c2f73c566f069761331386c9596d3f2'}, 'grpcio-1.84.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b44f0a0fc7bc6677d38cc80bca1a32814ce6c8f200fb8b3c1a61c9d77eaefbf3'}, 'h11-0.16.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86'}, 'hf_xet-1.6.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f'}, 'httpcore-1.0.9-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55'}, 'httpcore2-2.13.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '35ae5be347aa40467b4a5dc032ac67ebb6d27189fc97e8cebcf99616f6a1bb9e'}, 'httptools-0.8.0-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b15fc622b0f869d19207c4089a501d9bcc63ca5e071ffdd2f03f922df882dcb2'}, 'httpx-0.28.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad'}, 'httpx2-2.13.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'fc12720cedf72faa26cca6b4ca394e05c894e7d7933fc45cafe767960804e49a'}, 'huggingface_hub-1.32.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b0c7c80561969d9cdacdd55fce67ba9584cca0b9d4ea80957a3a5c1445fac5c8'}, 'humming_kernels-0.1.12-py3-none-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'cd3ef712a93f3a9075ea99de2c72bcd3ec89dab3759b3a248d869f5507b60331'}, 'idna-3.20-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c'}, 'ijson-3.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e4fcebfe1685bb7ba06a8255a5d428ea6b4b895d7acf979cb637d8bbc9db2f47'}, 'instanttensor-0.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '879e8b524b391ed916b8412ef82c3ab40d933ad8d038f40b340d0bd58b7894d3'}, 'interegular-0.3.3-py37-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b0c07007d48c89d6d19f7204972d369b2a77222722e126b6aa63aa721dc3b19c'}, 'jinja2-3.1.6-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67'}, 'jiter-0.17.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '8c21265b251d99bbb40080d178a8953e35601d3a1564e05c4de4c0d2ca616797'}, 'jmespath-1.1.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'a5663118de4908c91729bea0acadca56526eb2698e83de10cd116ae0f4e97c64'}, 'jsonschema-4.26.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd489f15263b8d200f8387e64b4c3a75f06629559fb73deb8fdfb525f2dab50ce'}, 'jsonschema_specifications-2025.9.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '98802fee3a11ee76ecaca44429fda8a41bff98b00a0f2838151b113f210cc6fe'}, 'lark-1.2.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c2276486b02f0f1b90be155f2c8ba4a8e194d42775786db622faccd652d8e80c'}, 'llguidance-1.7.6-cp39-abi3-manylinux_2_31_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ceec951d29a74309984e3be0fe7f5f56c1362434cd937abd517b259a60908b1e'}, 'llvmlite-0.47.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '5853bf26160857c0c2573415ff4efe01c4c651e59e2c55c2a088740acfee51cd'}, 'lm_format_enforcer-0.11.3-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'cf586350875def1ae7a8fba84fcbbfc8371424b6c9d05c1fcba70aa233fbf06f'}, 'loguru-0.7.3-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '31a33c10c8e1e10422bfd431aeb5d351c7cf7fa671e3c4df004162264b28220c'}, 'markdown_it_py-4.2.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a'}, 'markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d'}, 'mcp-2.2.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'bde982589473a060ae145e3406e9a5333fe538c97229ba841f5a7f92be004f81'}, 'mcp_types-2.2.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ea476b73ee86709ab5abc9452385ed36cc05907e582355622e294595c9a04f13'}, 'mdurl-0.1.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8'}, 'mistral_common-1.11.7-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'a9511b88eacacbe7dacddd9d3498c1739f56847b7fdddbd5a22e7844fd9def95'}, 'ml_dtypes-0.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '3b4a480aa8fd54a1805b8ac10f3f91763926a74f73c0c364c10f9231854f4170'}, 'model_hosting_container_standards-0.1.16-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '47f4f65713120bc3a69feb022981a38db9e557aedf88dbd72077f20588caa12b'}, 'mpmath-1.3.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c'}, 'msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '21995e74b5c598c2e004110ad66ec7f1b8c20bf2bcf3b2de8fd9a3094422d3ff'}, 'multidict-6.9.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '976fd7689d69ec78d67d31d38d396d8adb562f7e8368279f76aed4aa451fa06d'}, 'nccl4py-0.5.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '51156cbc35eb9122b870aa7ba1244d0c4781266442036c2bb3e1abb7e88d9376'}, 'networkx-3.7-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0'}, 'ninja-1.13.2-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '65a24341b5ac09fcadcc37082660be40a94174e51a937fabf6e2cae26225fa2c'}, 'numba-0.65.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'a52d92ffd297c10364bce60cd1fcb88f99284ab5df085f2c6bcd1cb33b529a6f'}, 'numpy-2.4.6-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '90f9849678c75fe7afa2d348ac842c168b0a4d3d61919687216dfc547976d853'}, 'nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436'}, 'nvidia_cuda_cccl-13.0.85-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e0da7ad981f3a8aff08241b5bfc1af868742a63e2762f53a5171c492ef242649'}, 'nvidia_cuda_crt-13.0.88-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2c8043c7c9e02492716426e9919fc78d2c5b3b2a7a768a88e952676b08aa55a4'}, 'nvidia_cuda_cupti-13.0.85-py3-none-manylinux_2_25_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8'}, 'nvidia_cuda_nvcc-13.0.88-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '56fe502eb77625a12f25172caa3cdddb4e4c8ba2c8c17dba44b164761b380f03'}, 'nvidia_cuda_nvdisasm-13.4.92-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'f35a5b6ddb64b6758c22c0b078c5b6ff6d54f7c240cd74bf131ab5bf13b6c3fe'}, 'nvidia_cuda_nvrtc-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575'}, 'nvidia_cuda_runtime-13.0.96-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548'}, 'nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0c45dd8eeb50b603f07995b1b300c62ffe6a1980482b82b3bcf94a4ca9d49304'}, 'nvidia_cudnn_frontend-1.29.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '596c9b7591457b4b49227fa2e22bd5960957a198e4336ce9ada8b5d2a1e4a5a1'}, 'nvidia_cufft-12.0.0.61-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3'}, 'nvidia_cufile-1.15.1.6-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44'}, 'nvidia_curand-10.4.0.35-py3-none-manylinux_2_27_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc'}, 'nvidia_cusolver-12.0.4.66-py3-none-manylinux_2_27_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112'}, 'nvidia_cusparse-12.6.3.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b'}, 'nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0'}, 'nvidia_cutlass_dsl-4.7.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '765850c74868a0b5a02fefbfc1b380842b64e5f7a1c8b1655e322a3b14c01726'}, 'nvidia_cutlass_dsl_libs_base-4.7.1-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c2b67dacabfc905e4d132784fe4848647f40b0b97b1cdde3ce4a482642536b84'}, 'nvidia_cutlass_dsl_libs_core-4.7.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c65662abc9ab18fd997c57547895736928e559f4624bbe2aa1aa91418ea014ca'}, 'nvidia_cutlass_dsl_libs_cu12-4.7.1-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c8a0f93a1cde41404696b3194dd1f589c6f3a7b426bcabe97845f8fe96f949f5'}, 'nvidia_cutlass_dsl_libs_cu13-4.7.1-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '20c14e16125cb8e36bc9947a9f0a57a660f44651365337b911c84abde4967f5f'}, 'nvidia_ml_py-13.610.43-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'f13c72698edef492f985cc225f14faafe68ae065a2e407f45bdf6f4b9b43fde8'}, 'nvidia_nccl_cu13-2.29.7-py3-none-manylinux_2_18_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'edd81538446786ec3b73972543e53bb43bcaf0bfc8ef76cb679fcc390ffe136d'}, 'nvidia_nvjitlink-13.4.92-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323'}, 'nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80'}, 'nvidia_nvtx-13.0.85-py3-none-manylinux1_x86_64.manylinux_2_5_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4'}, 'nvidia_nvvm-13.0.88-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c5f41ffeb6466944a026dfa5317d7d85355c119bbec279205d22f1869d1054e0'}, 'nvtx-0.2.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '3ca8030a6d197952318013dd1c12c22da1d4b9feb76ba72e0fcd449961183c2c'}, 'openai-3.17.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '712bc2982a6da1af10f6948a1d2d9ff5ce3ee3604ded88002a53765a77cf5ae5'}, 'openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c007d277218a50db8839e599ed78e0fffe5130f614c3f6d93ae257f282071a29'}, 'opencv_python_headless-5.0.0.93-cp37-abi3-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ed709fdf9aa0bd1f2ed8549e71d19449b03a675bb581eb292285f6861953be37'}, 'opentelemetry_api-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '94b98c893a91b88657eaac1e3ba89618cdb85be6918196705354f34728b2cdef'}, 'opentelemetry_exporter_otlp-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '4a498fa8d8fd8be9e8e2d175fe5524a3fe581ccffadd8509db86526a5fb97051'}, 'opentelemetry_exporter_otlp_proto_common-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '9a9fe61bba73d802904bc989f1d6b4a7b1ee40f06c40e98d6f85af65aaebb694'}, 'opentelemetry_exporter_otlp_proto_grpc-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '6a1a645ea182a2f59440c51fa8301d309f3324a8f9d65f8395584b064b67ee4e'}, 'opentelemetry_exporter_otlp_proto_http-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '838592fce774c1c8bb7b9a0a7facbfa82e17be5a8a4e94cef10cb84ae026bae3'}, 'opentelemetry_proto-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '898b155a0e1557afd867478fb6158e8122a46329ca0bb8dc53cc55e98f017f56'}, 'opentelemetry_sdk-1.44.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'df081c4c6bcfdb1211e3e86140376792643128a25f8d72d1d27675936e7e96ad'}, 'opentelemetry_semantic_conventions-0.65b0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '1cacde7b0ad306f84c5ef08c3dbe1bbaf20165bba6f8bff43b670e555a086bcb'}, 'opentelemetry_semantic_conventions_ai-0.5.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '25aeb22bd261543b4898a73824026d96770e5351209c7d07a0b1314762b1f6e4'}, 'outlines_core-0.2.14-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '3bb2060c240c4507f334965a8948dbeeb22007560d797f6debd92346c0b620cb'}, 'packaging-26.3-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c'}, 'partial_json_parser-0.2.1.1.post7-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '145119e5eabcf80cbb13844a6b50a85c68bf99d376f8ed771e2a3c3b03e653ae'}, 'pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91'}, 'prometheus_client-0.26.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'fa93d06737aa02bacd05794768508bb97d2fbee28cb3bca04eaae92f0ca953d6'}, 'prometheus_fastapi_instrumentator-8.1.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b9f40b2cff3f7891ca0610b3ae4fc6ec723fd326b04bb659819aaeb821a0fc7d'}, 'propcache-0.5.4-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2814ecd8e818f487bee4b0f921bc4d1c176cc5fc71ac0f072d0fa67eda4ac14b'}, 'protobuf-7.36.2-cp310-abi3-manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '89f23aa53c24553a2416fd4fd1ec06f74fa42b14b546d8883128813f775bbfd2'}, 'psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9'}, 'py_cpuinfo-9.0.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '859625bc251f64e21f077d099d4162689c762b5d6a4c3c97553d56241c9674d5'}, 'pybase64-1.5.0-cp312-cp312-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '1a2b9cf39b4d30f600df8c56cccbc03adfc6e1ae8c04cd6b181105a432d4a515'}, 'pycountry-26.2.16-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '115c4baf7cceaa30f59a4694d79483c9167dbce7a9de4d3d571c5f3ea77c305a'}, 'pycparser-3.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b727414169a36b7d524c1c3e31839a521725078d7b2ff038656844266160a992'}, 'pydantic-2.13.5-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '346a034f080da3755d8e9cb5e00e8b07de1d39e4f6e2c87d8ab7cafa0b269a73'}, 'pydantic_core-2.46.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0fc5be0abd4a407e200d844b404e33639a554e7bd0d448e7b9ae181be4789ac2'}, 'pydantic_extra_types-2.11.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '1722ea2bddae5628ace25f2aa685b69978ef533123e5638cfbddb999e0100ec1'}, 'pydantic_settings-2.15.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0ba092c291c94baceb5eff768aa0d56400a457585bc0175925a5a5510303da42'}, 'pygments-2.21.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9'}, 'pyjwt-2.14.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ad0cef71c756a56e74863c2919cf0985f72decbcfcb550ee2f422e7c62b5eedc'}, 'pynvvideocodec-2.0.4-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b59cec7a1a3f78fad13fead78cad8b6d9686827f9ff4477080245457675a01d0'}, 'python_dotenv-1.2.3-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '904552145e8bfed22162c09dab1c2b9b54fefa7b23ba780f4f26ca0316b0f0d9'}, 'python_json_logger-4.2.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '158a52126fcd6869e09574d2b66272666f3dc8f468c62637ef9a1fa883719cb9'}, 'python_multipart-0.0.32-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ff6d3f776f16878c894e52e107296ffc890e913c611b1a4ec6c44e2821fe2e23'}, 'pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc'}, 'pyzmq-27.2.0-cp312-abi3-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'dea74fd65f1fc5f7fe167916a473ebe6ed6174e5e5d9de11ea6583661be6cf43'}, 'quack_kernels-0.6.5-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'df1ddd31366c82eb9692fe087c671209189dcbde53c909c81a865d4ee66b158d'}, 'referencing-0.37.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '381329a9f99628c9069361716891d34ad94af76e461dcb0335825aecc7692231'}, 'regex-2026.9.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0'}, 'requests-2.34.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0'}, 'rich-15.0.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb'}, 'rich_toolkit-0.20.5-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e21fd616db9c0539d3c50f3433fb1173db95c19d3cbf215615e224182fad6c18'}, 'rignore-0.8.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2e68cfc4ee0a2909952af2aebd608d1cd22f7d1cdce332cb0b5ea3762939865d'}, 'rpds_py-2026.6.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'ecabd69db66de867690f9797f2f8fa27ba501bbc24540cbdbdc649cd15888ba6'}, 'safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774'}, 'sentencepiece-0.2.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c8a168b040bc61681293f79a949b5d911c8e25086f4260285b8d97ab5f1195da'}, 'sentry_sdk-2.69.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '25bc0a6c55bba0b83398212a7247f4f9f3190f189a1909b68e4807ef4b785895'}, 'setproctitle-1.3.7-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2906b6c7959cdb75f46159bf0acd8cc9906cf1361c9e1ded0d065fe8f9039629'}, 'setuptools-80.10.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '95b30ddfb717250edb492926c92b5221f7ef3fbcc2b07579bcd4a27da21d0173'}, 'shellingham-1.5.4-py2.py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686'}, 'six-1.17.0-py2.py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274'}, 'sniffio-1.3.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '2f6da418d1f1e0fddd844478f41680e794e6051915791a034ff65e5f100525a2'}, 'sse_starlette-3.4.11-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c7b2244bdff016fe7f64e10075e89a3e6bbf899649cc89b0fe884b5545042453'}, 'starlette-1.6.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'a86dd39d14bb45f85a3d18525215a9ef0cfd1f192ac793220e72598c90335f0c'}, 'supervisor-4.3.0-py2.py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0bcb763fddafba410f35cbde226aa7f8514b9fb82eb05a0c85f6588d1c13f8db'}, 'sympy-1.14.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5'}, 'tabulate-0.10.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'f0b0622e567335c8fabaaa659f1b33bcb6ddfe2e496071b743aa113f8774f2d3'}, 'tiktoken-0.14.0-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7896eea257fe497a2b7134474d909156c6744ce8da35bce88011a960e008aa0d'}, 'tilelang-0.1.12-cp38-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'bbeb5573cbe2544a51a5c9a6cc737c5e3b5c2d95411caa3687fd9b08eb9d5f97'}, 'tokenizers-0.23.2-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '41c2f84d172449b4dadb9cdc508e3e364076613c35b16e76ecfe47a60d1e3305'}, 'tokenspeed_mla-0.1.8-py3-none-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '6a7526d7327746893f8c20d24aa63ba5b8a123d0dfd6e66388e13b768b6452c6'}, 'tokenspeed_triton-3.8.10.post20260920-cp312-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'eb5eae6c4d3db11e13b7058c89b57e075fa7df905d43169cfdb066e161886421'}, 'torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '796633c4cdf0fe2cdced72d8f88f22e73dbcfce83132763162f6d4bff13b820b'}, 'torch_c_dlpack_ext-0.1.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e6f9da4bb9af70e27facc777458be62e10dbbbddda7672d16138db0553c5a524'}, 'torchaudio-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf'}, 'torchcodec-0.16.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '5309c29e31b719f81dba75fcfa49a9ce6a38d069d7f570767466fe634783e7ba'}, 'torchvision-0.28.0-cp312-cp312-manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '028a3d481b37d785605620d7cdad897064c5a55bae2aa1f2658766333e291940'}, 'tqdm-4.70.1-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73'}, 'transformers-5.18.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'd79e5a515a572ee3deb33eb0d578be47d91e70c75d54cf21bf6eede6edef14a9'}, 'triton-3.7.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7e40869937a68206ec70d7f25bb7ec6433cb083f9135e1f36dbd318dc449a728'}, 'truststore-0.10.4-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'adaeaecf1cbb5f4de3b1959b42d41f6fab57b2b1666adb59e89cb0b53361d981'}, 'typer-0.27.2-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b3a5fc4342d5fc8fda8fc3010b1cf117e9249aab7fae800c2eff62fd3842d97d'}, 'typing_extensions-4.16.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8'}, 'typing_inspection-0.4.4-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '65b8397ba37ccbce054456aaccddfc91e6e3083c92824df348d96ca832f3f147'}, 'urllib3-2.8.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3'}, 'uvicorn-0.53.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e8dca71ec86dce5f04e333f0d56cdedf942446e6643b9cea1af0d6d3a02cb03e'}, 'uvloop-0.22.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7b5b1ac819a3f946d3b2ee07f09149578ae76066d70b44df3fa990add49a82e4'}, 'vllm-0.30.0+merlin252.sm89-cp38-abi3-manylinux_2_34_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'db0212be0c281ffc3c16bbfb1dda1eb5d6eb10b5f60fdc8fc31b18cb8ebea9e5'}, 'watchfiles-1.3.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'b5768b49e426fd5b550b012c866db347cdf15c398ef98dc557b6e6b72fa74cd1'}, 'websockets-17.1-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '0c863507ada5805517ca6dff1c524dcd42942efe6304dacf06700878398d21a6'}, 'wheel-0.48.0-py3-none-any.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '3217dcc807155e45db462d7ef2431f5ddda0d7273b700d05a67b271ceb1287ab'}, 'xgrammar-0.2.7-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'e4397798f73512ab5abccbadf0a7b6aaf0753ccc4f3171c43b14c4d22df1ed28'}, 'yarl-1.25.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': 'c6f117789d22dce188e5754e8bc65b7e6ebf8cb73963b9fa761f672a5883769d'}, 'z3_solver-4.15.4.0-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl': {'source': 'nisshokuitsuki/qwen-merlin-h95-wheels', 'sha256': '7e103a6f203f505b8b8b8e5c931cc407c95b61556512d4921c1ddc0b3f41b08e'}, 'scipy-1.18.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl': {'source': 'nisshokuitsuki/qwen-scipy-wheel', 'sha256': 'f55fa87b6c612ecd6b058f167c53231b1d14e412efe361d3d6e38b3631c73218'}}, 'isolated_wheel_install': False, 'runtime': {}, 'model_dir_contains': 'intel-qwen3.8-flash-next-w4a16-autoround', 'venv': '/kaggle/tmp/h95-venv', 'uv_sources': ['nisshokuitsuki/uv-wheel']}


In [4]:
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    exec(compile('from importlib.metadata import version, PackageNotFoundError\npython = sys.executable\nif deployment[\'venv\']:  # a fresh venv without pip: the image\'s packages stay out of it\n    python = deployment[\'venv\'] + \'/bin/python\'\n    subprocess.check_call([sys.executable, \'-m\', \'venv\', \'--without-pip\', \'--clear\', deployment[\'venv\']])\n    installed = None\nelse:\n    try:\n        installed = version(deployment[\'package\'])\n    except PackageNotFoundError:\n        installed = None\nif installed != deployment[\'vllm_version\'] or deployment[\'wheel_files\']:\n    command = [sys.executable, \'-m\', \'pip\'] + ([\'--python\', python] if deployment[\'venv\'] else []) + [\'install\', \'--no-index\']\n    wheel_directories = {}\n    for reference in deployment[\'wheel_sources\'] + deployment[\'wheel_kernel_sources\']:\n        matches = [p for p in Path(\'/kaggle/input\').rglob(reference.split(\'/\')[1]) if p.is_dir()]\n        assert len(matches) == 1, (reference, matches)\n        wheel_directories[reference] = matches[0]\n        command += [\'--find-links\', str(matches[0])]\n    if deployment[\'wheel_files\']:\n        install_target = Path(\'/kaggle/working/offline-site\') if deployment[\'isolated_wheel_install\'] else None\n        requirements = []\n        for name, record in sorted(deployment[\'wheel_files\'].items()):\n            matches = list(wheel_directories[record[\'source\']].rglob(name))\n            if not matches and \'+\' in name:  # a Kaggle dataset drops the \'+\' of a local version from file names,\n                renamed = list(wheel_directories[record[\'source\']].rglob(name.replace(\'+\', \'\')))  # pip needs it back\n                assert len(renamed) == 1, (name, renamed)\n                matches = [Path(\'/kaggle/tmp/wheel-links\') / name]\n                matches[0].parent.mkdir(parents=True, exist_ok=True)\n                matches[0].unlink(missing_ok=True)\n                matches[0].symlink_to(renamed[0])\n            assert len(matches) == 1, (name, matches)\n            wheel = matches[0]\n            assert wheel.is_file(), wheel\n            requirements.append(wheel.as_uri() + \' --hash=sha256:\' + record[\'sha256\'])\n        requirements_path = Path(\'/kaggle/working/wheel-requirements.txt\')\n        requirements_path.write_text(\'\\n\'.join(requirements) + \'\\n\')\n        command += [\'--only-binary=:all:\', \'--require-hashes\']\n        if install_target is not None:\n            install_target.mkdir(parents=True)\n            command += [\'--target\', str(install_target), \'--no-compile\', \'--no-deps\']\n        elif deployment[\'venv\']:\n            command += [\'--no-compile\', \'--no-deps\']\n        else:\n            command += [\'--force-reinstall\']\n        command += [\'-r\', str(requirements_path)]\n    else:\n        command += [\'--upgrade\', deployment[\'package\'] + \'==\' + deployment[\'vllm_version\']]\n    install_started, install_path = time.time(), \'pip\'\n    try:\n        import zipfile\n        uv = Path(\'/kaggle/tmp/uv/uv\')\n        wheels = [w for ref in deployment[\'uv_sources\'] for d in Path(\'/kaggle/input\').rglob(ref.split(\'/\')[1]) if d.is_dir()\n                  for w in d.rglob(\'uv-*.whl\')]\n        assert len(wheels) == 1, wheels\n        with zipfile.ZipFile(wheels[0]) as archive:\n            member = next(n for n in archive.namelist() if n.endswith(\'.data/scripts/uv\'))\n            uv.parent.mkdir(parents=True, exist_ok=True)\n            uv.write_bytes(archive.read(member))\n        uv.chmod(0o755)\n        links = [x for i, a in enumerate(command) if a == \'--find-links\' for x in (a, command[i + 1])]\n        uv_command = [str(uv), \'pip\', \'install\', \'--python\', python, \'--no-index\', *links, \'--only-binary\', \':all:\',\n                      \'--require-hashes\', \'--no-deps\', \'-r\', str(requirements_path)]\n        subprocess.run(uv_command, check=True, env={**os.environ, \'UV_CACHE_DIR\': \'/kaggle/tmp/uv-cache\', \'UV_LINK_MODE\': \'copy\', \'UV_NO_PROGRESS\': \'1\'})\n        install_path = \'uv\'\n    except Exception as error:\n        print(\'uv install did not finish, pip instead:\', repr(error)[:300], flush=True)\n        subprocess.check_call([sys.executable, \'-m\', \'venv\', \'--without-pip\', \'--clear\', deployment[\'venv\']])\n        subprocess.check_call(command)\n        install_path = \'pip after uv failed\'\n    print(json.dumps({\'wheel_install\': install_path, \'seconds\': round(time.time() - install_started, 1)}), flush=True)\n    if deployment[\'wheel_files\'] and install_target is not None:\n        sys.path.insert(0, str(install_target))\n        os.environ[\'PYTHONPATH\'] = str(install_target)\nif deployment[\'venv\']:\n    probe = f"from importlib.metadata import version; print(version({deployment[\'package\']!r}))"\n    assert subprocess.check_output([python, \'-c\', probe], text=True).strip() == deployment[\'vllm_version\']\nelse:\n    assert version(deployment[\'package\']) == deployment[\'vllm_version\']\n', 'notebook_cell_3', 'exec'))


In [5]:
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    exec(compile('sources = {\'src/utils/kaggle_runtime.py\': \'"""File, process and submission helpers shared by the Kaggle notebooks and the native-tool serving runtimes."""\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport signal\\nimport subprocess\\nimport threading\\nfrom typing import Any\\n\\nfrom src.utils.grid import normalize_grid\\n\\n\\ndef atomic_json(path: Path, value) -> None:\\n    temporary = path.with_suffix(path.suffix + ".tmp")\\n    temporary.write_text(json.dumps(value, indent=2) + "\\\\n")\\n    temporary.replace(path)\\n\\n\\ndef stop_group(process: subprocess.Popen | None) -> None:\\n    if process is None:\\n        return\\n    try:\\n        os.killpg(process.pid, signal.SIGTERM)\\n    except ProcessLookupError:\\n        pass\\n    try:\\n        process.wait(timeout=20)\\n    except subprocess.TimeoutExpired:\\n        pass\\n    # A reaped process leader does not establish that its worker group has exited.\\n    try:\\n        os.killpg(process.pid, signal.SIGKILL)\\n    except ProcessLookupError:\\n        pass\\n    process.wait(timeout=20)\\n\\n\\ndef capture_server_log(stream, path: Path, ready: threading.Event, startup_complete: threading.Event) -> None:\\n    try:\\n        with path.open(\\\'a\\\') as log:\\n            for line in stream:\\n                log.write(line)\\n                log.flush()\\n                if \\\'Application startup complete\\\' in line:\\n                    startup_complete.set()\\n                    ready.set()\\n    finally:\\n        ready.set()\\n\\n\\ndef capture_and_close(stream, path: Path, ready: threading.Event, complete: threading.Event) -> None:\\n    with stream:\\n        capture_server_log(stream, path, ready, complete)\\n\\n\\ndef validate_model_files(model: Path, files: dict[str, Any]) -> None:\\n    """Check an attached model against a complete reference file inventory. Entries without "sha256" are checked by size\\n    only (for checkpoints too large to hash within the notebook\\\'s time budget)."""\\n    index = json.loads((model/"model.safetensors.index.json").read_text())\\n    required = set(index["weight_map"].values()) | {\\n        "config.json", "model.safetensors.index.json",\\n        "tokenizer.json", "tokenizer_config.json", "chat_template.jinja",\\n    }\\n    if (model / "generation_config.json").exists():\\n        required.add("generation_config.json")\\n    if not required <= files.keys():\\n        raise ValueError("reference model inventory omits weights or configuration")\\n    for name, expected in files.items():\\n        if not name or Path(name).name != name or name in (".", ".."):\\n            raise ValueError("model inventory entries must be plain file names")\\n        path = model/name\\n        if path.stat().st_size != expected["size_bytes"]:\\n            raise ValueError(f"model file size differs: {name}")\\n        if "sha256" not in expected:\\n            continue\\n        with path.open("rb") as handle:\\n            if hashlib.file_digest(handle, "sha256").hexdigest() != expected["sha256"]:\\n                raise ValueError(f"model file hash differs: {name}")\\n\\n\\ndef validate_submission(tasks: dict[str, Any], result: dict[str, Any]) -> None:\\n    if not tasks or set(result) != set(tasks):\\n        raise ValueError("submission task coverage differs from challenges")\\n    for task_id, task in tasks.items():\\n        if len(result[task_id]) != len(task["test"]):\\n            raise ValueError("submission test output count differs from challenges")\\n        for entry in result[task_id]:\\n            if set(entry) != {"attempt_1", "attempt_2"}:\\n                raise ValueError("submission requires exactly two attempts")\\n            for grid in entry.values():\\n                if normalize_grid(grid) is None or any(type(cell) is not int for row in grid for cell in row):\\n                    raise ValueError("submission contains an invalid integer grid")\\n\', \'src/utils/grid.py\': \'"""ARC grid validity and rule-preserving augmentation, shared by the induction and transduction lines.\\n\\nValidity: `normalize_grid` returns a JSON-compatible grid or None; `grid_key` is a hashable canonical form.\\n\\nRule-preserving test-time views shared by grid and code generation.\\n\\nA *view* is an ARC-rule-preserving disguise of a whole task: a D8 dihedral\\nsymmetry (optional transpose then k 90-degree rotations) x a color permutation\\n(the 10 colors are arbitrary labels) x a demonstration-order shuffle. The frozen\\nmodel solves each disguised view; `invert_grid` maps its predicted grid back to\\nthe canonical frame so candidates from different views pool together.\\n\\nThese are the same symmetries NVARC\\\'s per-task TTT augments over (cf.\\n`arc_loader.permute_mod` and `ArcDataset.augment`); here they drive prompt-level\\nTTA instead of building a training set. Color and geometry commute, so forward\\napplies color then geometry and `invert_grid` undoes geometry then color.\\n"""\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom typing import Any\\n\\nimport numpy as np\\n\\n\\nGrid = list[list[int]]\\n\\n\\ndef normalize_grid(value: Any) -> Grid | None:\\n    """Return a JSON-compatible ARC grid, or ``None`` for invalid values."""\\n    try:\\n        array = np.asarray(value)\\n        if array.ndim != 2 or array.size == 0 or any(not 0 < size <= 30 for size in array.shape):\\n            return None\\n        if not np.issubdtype(array.dtype, np.integer):\\n            if not np.all(np.equal(array, np.floor(array))):\\n                return None\\n        array = array.astype(int)\\n        if array.min() < 0 or array.max() > 9:\\n            return None\\n        return array.tolist()\\n    except (TypeError, ValueError, OverflowError):\\n        return None\\n\\n\\ndef serialize_spaced(grid) -> str:\\n    """Serialize a grid as rows of space-separated digits."""\\n    return "\\\\n".join(" ".join(str(int(cell)) for cell in row) for row in grid)\\n\\n\\ndef is_valid_grid(value: Any) -> bool:\\n    return normalize_grid(value) is not None\\n\\n\\ndef grid_key(value: Any) -> tuple[tuple[int, ...], ...]:\\n    grid = normalize_grid(value)\\n    if grid is None:\\n        raise ValueError("value is not a valid ARC grid")\\n    return tuple(tuple(row) for row in grid)\\n\\n# The 8 elements of the dihedral group of the square: optionally transpose, then\\n# rotate by k * 90 degrees. (transpose, k) enumerates all of D8 exactly once.\\nD8 = [(t, k) for t in (False, True) for k in range(4)]\\n\\n\\n@dataclass(frozen=True)\\nclass View:\\n    """One TTA disguise. Frozen so it is hashable (used for de-duplication).\\n\\n    transpose / k: the D8 geometry. color: a length-10 permutation, color[c] is\\n    the new label for cell value c. demo_order: a permutation of the demo indices.\\n    """\\n    transpose: bool\\n    k: int\\n    color: tuple  # length-10 permutation of 0..9\\n    demo_order: tuple  # permutation of range(n_demos)\\n\\n\\ndef d8_transform(grid, transform_index: int) -> list:\\n    """Apply one of the eight square-grid symmetries."""\\n    if not 0 <= transform_index < len(D8):\\n        raise ValueError("transform_index must be in [0, 7]")\\n    g = np.asarray(grid, dtype=int)\\n    if transform_index >= 4:\\n        g = np.fliplr(g)\\n    return np.rot90(g, transform_index % 4).tolist()\\n\\n\\ndef apply_grid(grid, view: View) -> list:\\n    """Forward: relabel colors, then apply the D8 geometry. Returns nested ints."""\\n    g = np.asarray(grid, dtype=int)\\n    g = np.asarray(view.color, dtype=int)[g]  # color[c] relabel, per cell\\n    if view.transpose:\\n        g = g.T\\n    if view.k:\\n        g = np.rot90(g, view.k)\\n    return g.tolist()\\n\\n\\ndef invert_grid(grid, view: View) -> list:\\n    """Inverse of `apply_grid`: undo the geometry, then undo the color relabel."""\\n    g = np.asarray(grid, dtype=int)\\n    if view.k:\\n        g = np.rot90(g, -view.k)\\n    if view.transpose:\\n        g = g.T\\n    inv_color = np.argsort(np.asarray(view.color, dtype=int))  # inverse permutation\\n    g = inv_color[g]\\n    return g.tolist()\\n\\n\\ndef apply_task(task: dict, view: View) -> dict:\\n    """Disguise a whole task: reorder demos and transform every grid. Test outputs\\n    are unknown, so only test *inputs* are transformed."""\\n    train = [task["train"][i] for i in view.demo_order]\\n    new_train = [\\n        {"input": apply_grid(p["input"], view), "output": apply_grid(p["output"], view)}\\n        for p in train\\n    ]\\n    new_test = [{"input": apply_grid(t["input"], view)} for t in task["test"]]\\n    return {"train": new_train, "test": new_test}\\n\\n\\ndef make_views(n_demos: int, n_views: int, seed: int = 0) -> list:\\n    """Sample `n_views` distinct views. The identity view is always first so the\\n    untransformed task is one of the candidates."""\\n    rng = np.random.default_rng(seed)\\n    identity = View(False, 0, tuple(range(10)), tuple(range(n_demos)))\\n    views = [identity]\\n    seen = {identity}\\n    max_attempts = max(50, n_views * 50)\\n    attempts = 0\\n    while len(views) < n_views and attempts < max_attempts:\\n        attempts += 1\\n        transpose, k = D8[int(rng.integers(len(D8)))]\\n        color = tuple(int(x) for x in rng.permutation(10))\\n        demo_order = tuple(int(x) for x in rng.permutation(n_demos))\\n        v = View(bool(transpose), int(k), color, demo_order)\\n        if v in seen:\\n            continue\\n        seen.add(v)\\n        views.append(v)\\n    return views\\n\\n\\n__all__ = [\\n    "D8",\\n    "View",\\n    "apply_grid",\\n    "apply_task",\\n    "d8_transform",\\n    "invert_grid",\\n    "is_valid_grid",\\n    "make_views",\\n]\\n\', \'src/utils/prompt.py\': \'"""The ARC task prompt and answer parsing shared by direct answers and the agent harness, after arcprize/arc-agi-benchmarking\\n(prompts/system_prompt.txt with convert_task_pairs_to_prompt, sent as the user message; utils/parsing.py: \\\\\\\\boxed{} first,\\nelse the last bracketed JSON list of lists), and the fixed second D8 view per test output.\\n"""\\nimport hashlib\\nimport json\\nimport re\\n\\nfrom src.utils.grid import D8, is_valid_grid\\n\\nDIRECT_PROMPT = (\\n    "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\\\\n\\\\n"\\n    "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation "\\n    "in the training examples that maps the input to the output, then apply that pattern to the test input to give a "\\n    "final output.\\\\n\\\\nRespond in the format of the training output examples\\\\n\\\\n--Training Examples--\\\\n"\\n    "{training_examples}\\\\n--End of Training Examples--\\\\n\\\\n--Test Input--\\\\n{test_input}\\\\n--End of Test Input--\\\\n\\\\n"\\n    "Your response:")\\n# One request for all test inputs of a task (job_unit "task"): the same instructions, every test input in order, and one\\n# labelled output per test input. A task with one test input gets DIRECT_PROMPT unchanged.\\nTASK_PROMPT = (\\n    "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\\\\n\\\\n"\\n    "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or transformation "\\n    "in the training examples that maps the input to the output, then apply that pattern to each test input to give its "\\n    "final output.\\\\n\\\\nRespond in the format of the training output examples, with one output per test input, in order, "\\n    "each after its own line \\\\"--Test Output k--\\\\" (k = 1, 2, ...)\\\\n\\\\n--Training Examples--\\\\n"\\n    "{training_examples}\\\\n--End of Training Examples--\\\\n\\\\n{test_inputs}\\\\n\\\\nYour response:")\\n\\n\\ndef format_grid(grid: list, grid_format: str = "json") -> str:\\n    """"json": the benchmark\\\'s json.dumps (about 3 tokens per cell); "compact": one line of digits per row (1 token per cell)."""\\n    if grid_format == "compact":\\n        return "\\\\n".join("".join(str(cell) for cell in row) for row in grid)\\n    return json.dumps(grid)\\n\\n\\ndef direct_prompt(task: dict, index: int, grid_format: str = "json") -> str:\\n    examples = "".join(f"--Example {i}-- \\\\n\\\\n INPUT: \\\\n\\\\n{format_grid(pair[\\\'input\\\'], grid_format)}\\\\n\\\\nOUTPUT: \\\\n\\\\n"\\n                       f"{format_grid(pair[\\\'output\\\'], grid_format)}\\\\n\\\\n" for i, pair in enumerate(task["train"]))\\n    return DIRECT_PROMPT.format(training_examples=examples, test_input=format_grid(task["test"][index]["input"], grid_format))\\n\\n\\ndef task_prompt(task: dict, grid_format: str = "json") -> str:\\n    """The prompt for all test inputs of a task at once (DIRECT_PROMPT when there is one)."""\\n    if len(task["test"]) == 1:\\n        return direct_prompt(task, 0, grid_format)\\n    examples = "".join(f"--Example {i}-- \\\\n\\\\n INPUT: \\\\n\\\\n{format_grid(pair[\\\'input\\\'], grid_format)}\\\\n\\\\nOUTPUT: \\\\n\\\\n"\\n                       f"{format_grid(pair[\\\'output\\\'], grid_format)}\\\\n\\\\n" for i, pair in enumerate(task["train"]))\\n    tests = "\\\\n\\\\n".join(f"--Test Input {k}--\\\\n{format_grid(test[\\\'input\\\'], grid_format)}\\\\n--End of Test Input {k}--"\\n                         for k, test in enumerate(task["test"], 1))\\n    return TASK_PROMPT.format(training_examples=examples, test_inputs=tests)\\n\\n\\ndef all_grids(text: str) -> list:\\n    """Every top-level bracketed JSON list of lists that is a valid ARC grid, in order."""\\n    grids, depth, start = [], 0, None\\n    for position, char in enumerate(text):\\n        if char == "[":\\n            if depth == 0:\\n                start = position\\n            depth += 1\\n        elif char == "]" and depth:\\n            depth -= 1\\n            if depth == 0:\\n                try:\\n                    grid = json.loads(text[start:position + 1])\\n                except json.JSONDecodeError:\\n                    continue\\n                if is_valid_grid(grid):\\n                    grids.append(grid)\\n    return grids\\n\\n\\ndef answer_grids(text: str, count: int, grid_format: str = "json") -> list:\\n    """`count` grids for the test outputs of a task answer: the grid after each "--Test Output k--" line, else (no\\n    labels) the last `count` grids in order; a missing one is None."""\\n    parts = re.split(r"--\\\\s*Test Output\\\\s*(\\\\d+)\\\\s*--", text)\\n    found = {}\\n    for label, body in zip(parts[1::2], parts[2::2]):\\n        found[int(label)] = answer_grid(body, grid_format)\\n    if found:\\n        return [found.get(k) for k in range(1, count + 1)]\\n    if grid_format == "compact":\\n        return [answer_grid(text, grid_format)] + [None] * (count - 1) if count else []\\n    grids = all_grids(text)[-count:] if count else []\\n    return grids + [None] * (count - len(grids))\\n\\n\\ndef last_digit_grid(text: str) -> list | None:\\n    """The last block of consecutive lines that are each only digits, all of one length (fences and blank lines end a\\n    block); kept only when it is a valid ARC grid."""\\n    rows, blocks = [], []\\n    for line in text.splitlines() + [""]:\\n        line = line.strip().strip("`")\\n        if line.isdigit() and (not rows or len(line) == len(rows[0])):\\n            rows.append(line)\\n        else:\\n            if rows:\\n                blocks.append(rows)\\n            rows = [line] if line.isdigit() else []\\n    for rows in reversed(blocks):\\n        grid = [[int(cell) for cell in row] for row in rows]\\n        if is_valid_grid(grid):\\n            return grid\\n    return None\\n\\n\\ndef last_grid(text: str) -> list | None:\\n    """\\\\\\\\boxed{...} first, else the last bracketed JSON list of lists; kept only when it is a valid ARC grid."""\\n    candidates = [match.group(1) for match in re.finditer(r"\\\\\\\\boxed\\\\{(.*?)\\\\}", text, re.DOTALL)][:1]\\n    end = text.rfind("]")\\n    if end >= 0:\\n        depth = 0\\n        for start in range(end, -1, -1):\\n            depth += {"]": 1, "[": -1}.get(text[start], 0)\\n            if depth == 0:\\n                candidates.append(text[start:end + 1])\\n                break\\n    for candidate in candidates:\\n        try:\\n            grid = json.loads(candidate)\\n        except json.JSONDecodeError:\\n            continue\\n        if is_valid_grid(grid):\\n            return grid\\n    return None\\n\\n\\ndef answer_grid(text: str, grid_format: str = "json") -> list | None:\\n    """last_grid; with the compact format the last digit block first, then last_grid."""\\n    if grid_format == "compact":\\n        grid = last_digit_grid(text)\\n        if grid is not None:\\n            return grid\\n    return last_grid(text)\\n\\n\\ndef parse_answer(tokenizer, tokens: list[int], grid_format: str = "json") -> tuple[str, list | None, int | None]:\\n    """(answer text after </think>, its grid or None, the index of </think> or None) for a completion\\\'s token ids;\\n    the deployment and the RL reward (archive/rl/verl/direct_loop.py) both parse with this."""\\n    think_end = tokenizer.convert_tokens_to_ids("</think>")\\n    end = tokens.index(think_end) if think_end in tokens else None\\n    answer = tokenizer.decode(tokens[end + 1:] if end is not None else tokens, skip_special_tokens=False,\\n                              clean_up_tokenization_spaces=False)\\n    return answer, answer_grid(answer, grid_format), end\\n\\n\\ndef second_view(task_id: str, index: int) -> list:\\n    """[transpose, k] of a fixed non-identity D8 symmetry per test output (from a hash of task and output index)."""\\n    return list(D8[1 + int(hashlib.sha256(f"{task_id}_{index}".encode()).hexdigest(), 16) % 7])\\n\', \'src/utils/completion.py\': \'"""Token-id completions with vLLM\\\'s thinking_token_budget behaviour applied by the client (budgeted), from SGLang\\\'s\\n/generate or an OpenAI-compatible /v1/completions."""\\n\\n\\ndef sglang_completion(http, url: str, prompt: list[int], settings: dict, seed: int, max_tokens: int, think_start: int,\\n                      think_end: int) -> tuple[list[int], str]:\\n    """(token ids, finish reason) from SGLang\\\'s /generate with vLLM\\\'s thinking_token_budget behaviour (budgeted)."""\\n    def generate(ids: list[int], limit: int) -> tuple[list[int], str]:\\n        body = {"input_ids": ids, "sampling_params": {\\n            "temperature": settings["temperature"], "top_p": settings["top_p"], "top_k": settings["top_k"],\\n            "max_new_tokens": limit, "skip_special_tokens": False, "sampling_seed": seed}}\\n        response = http.post(url, json=body, timeout=settings["request_timeout"])\\n        response.raise_for_status()\\n        out = response.json()\\n        return list(out["output_ids"]), out["meta_info"]["finish_reason"]["type"]\\n\\n    return budgeted(generate, prompt, settings, max_tokens, think_start, think_end)\\n\\n\\ndef openai_completion(http, url: str, prompt: list[int], settings: dict, seed: int, max_tokens: int, think_start: int,\\n                      think_end: int) -> tuple[list[int], str]:\\n    """The same from an OpenAI-compatible /v1/completions with token ids in and out (vLLM; the budget is applied here, so\\n    an engine build without thinking_token_budget behaves the same); settings["served_model"] names the model."""\\n    def generate(ids: list[int], limit: int) -> tuple[list[int], str]:\\n        body = {"model": settings["served_model"], "prompt": ids, "max_tokens": limit, "temperature": settings["temperature"],\\n                "top_p": settings["top_p"], "top_k": settings["top_k"], "seed": seed, "return_token_ids": True,\\n                "add_special_tokens": False, "skip_special_tokens": False}\\n        response = http.post(url, json=body, timeout=settings["request_timeout"])\\n        response.raise_for_status()\\n        choice = response.json()["choices"][0]\\n        return list(choice.get("token_ids") or []), choice.get("finish_reason")\\n\\n    return budgeted(generate, prompt, settings, max_tokens, think_start, think_end)\\n\\n\\ndef budgeted(generate, prompt: list[int], settings: dict, max_tokens: int, think_start: int,\\n             think_end: int) -> tuple[list[int], str]:\\n    """vLLM\\\'s thinking_token_budget behaviour for engines without it, over generate(ids, limit) -> (new ids, finish):\\n    tokens after the prompt\\\'s last <think> count toward the budget, and when they reach it </think> is forced and the\\n    answer continues up to max_tokens. The first request stops at the budget; if it ends there, a second request\\n    continues from prompt + output (+ </think> when thinking is still open)."""\\n    budget = settings.get("thinking_token_budget")\\n    opened = prompt[::-1].index(think_start) if think_start in prompt else None  # tokens after the last <think>\\n    first = max_tokens if budget is None or opened is None else min(max_tokens, max(0, budget - opened))\\n    tokens, finish = generate(prompt, first)\\n    if finish == "length" and len(tokens) < max_tokens:\\n        forced = [] if think_end in tokens else [think_end]\\n        rest, finish = generate(prompt + tokens + forced, max_tokens - len(tokens) - len(forced))\\n        tokens = tokens + forced + rest\\n    return tokens, finish\\n\', \'src/utils/tensors.py\': \'"""A safetensors file\\\'s header without reading tensor data: 8-byte little-endian length, then the JSON header."""\\nfrom contextlib import nullcontext\\nimport json\\nfrom pathlib import Path\\nimport struct\\nfrom typing import BinaryIO\\n\\n\\ndef safetensors_header(source: Path | BinaryIO) -> tuple[dict, int]:\\n    """(name -> {"dtype", "shape", "data_offsets"} without __metadata__, byte offset where the data section starts) of a\\n    file path or of a binary stream positioned at the file\\\'s start (e.g. a remote file opened with HfFileSystem)."""\\n    with (open(source, "rb") if isinstance(source, (str, Path)) else nullcontext(source)) as stream:\\n        size = struct.unpack("<Q", stream.read(8))[0]\\n        if not 0 < size < 100_000_000:\\n            raise ValueError(f"{source}: invalid safetensors header length {size}")\\n        entries = json.loads(stream.read(size))\\n    entries.pop("__metadata__", None)\\n    return entries, 8 + size\\n\', \'src/qwen/direct.py\': \'"""Direct (transductive) answers: the model thinks and writes the test output grid, without tools, with the shared prompt\\nand parsing of src.utils.prompt.\\n"""\\nfrom pathlib import Path\\nimport time\\n\\nfrom src.utils.completion import openai_completion, sglang_completion\\nfrom src.utils.grid import View, apply_task, invert_grid\\nfrom src.utils.kaggle_runtime import atomic_json\\nfrom src.utils.prompt import answer_grids, direct_prompt, parse_answer, second_view, task_prompt\\n\\nSEED_OFFSET = 10_000_000  # keeps direct-answer seeds apart from the tool trajectories\\\' seeds\\n\\n\\ndef direct_jobs_for(tasks: dict, settings: dict) -> list[dict]:\\n    """settings["direct_attempts"] (default 0) direct jobs per test output, attempt-major. With settings["second_view"],\\n    attempts after the first carry "view": the task is shown under that symmetry and the answer mapped back;\\n    settings["view_every_attempt"] gives the first attempts that view too (measuring the second view on its own)."""\\n    jobs = []\\n    for attempt in range(settings.get("direct_attempts", 0)):\\n        for task_id, task in tasks.items():\\n            # job_unit "task": one job for all test inputs of a task (test_index None, task_prompt)\\n            for index in [None] if settings.get("job_unit") == "task" else range(len(task["test"])):\\n                jobs.append({"key": f"d{len(jobs):04d}", "task_id": task_id, "test_index": index, "attempt": attempt,\\n                             "seed": settings["seed"] + SEED_OFFSET + len(jobs)})\\n                if (attempt or settings.get("view_every_attempt")) and settings.get("second_view"):\\n                    jobs[-1]["view"] = second_view(task_id, 0 if index is None else index)\\n    return jobs\\n\\n\\ndef direct_answer(job: dict, task: dict, root: Path, model: str, settings: dict, tokenizer, client) -> dict:\\n    """One completion with the agent\\\'s sampling, thinking budget and token cap; writes root/<key>.json. With\\n    settings["max_model_len"] the token cap shrinks so prompt plus answer fit the server\\\'s context. With\\n    settings["engine"] == "sglang", client is an httpx client for settings["generate_url"] (sglang_completion); with\\n    "openai", for settings["completions_url"] (openai_completion: the thinking budget applied by the client)."""\\n    if any("output" in test for test in task["test"]):\\n        raise ValueError("Test labels must not enter the prompt")\\n    view = View(job["view"][0], job["view"][1], tuple(range(10)), tuple(range(len(task["train"])))) if job.get("view") else None\\n    task = apply_task(task, view) if view else task\\n    grid_format = settings.get("grid_format", "json")\\n    content = task_prompt(task, grid_format) if job["test_index"] is None else direct_prompt(task, job["test_index"], grid_format)\\n    text = tokenizer.apply_chat_template([{"role": "user", "content": content}],\\n                                         tokenize=False, add_generation_prompt=True, enable_thinking=True)\\n    prompt = tokenizer.encode(text, add_special_tokens=False)\\n    extra = {"top_k": settings["top_k"], "return_token_ids": True, "add_special_tokens": False, "skip_special_tokens": False}\\n    if settings.get("thinking_token_budget") is not None:\\n        extra["thinking_token_budget"] = settings["thinking_token_budget"]\\n    result, started = {**job, "status": "RUNNING", "prediction": None}, time.monotonic()\\n    max_tokens = min(settings["max_tokens"], settings.get("max_model_len", 1 << 30) - len(prompt))\\n    try:\\n        if settings.get("engine") in ("sglang", "openai"):\\n            complete, url = ((sglang_completion, settings["generate_url"]) if settings["engine"] == "sglang"\\n                             else (openai_completion, settings["completions_url"]))\\n            tokens, finish = complete(client, url, prompt, settings, job["seed"], max_tokens,\\n                                      tokenizer.convert_tokens_to_ids("<think>"), tokenizer.convert_tokens_to_ids("</think>"))\\n        else:\\n            response = client.completions.create(\\n                model=model, prompt=prompt, n=1, temperature=settings["temperature"], top_p=settings["top_p"],\\n                max_tokens=max_tokens, seed=job["seed"], extra_body=extra, timeout=settings["request_timeout"]).model_dump()\\n            choice = response["choices"][0]\\n            tokens, finish = choice.get("token_ids") or [], choice.get("finish_reason")\\n        answer, grid, thinking = parse_answer(tokenizer, tokens, grid_format)\\n        grids = [grid] if job["test_index"] is not None else answer_grids(answer, len(task["test"]), grid_format)\\n        grids = [invert_grid(g, view) if view and g is not None else g for g in grids]\\n        if job["test_index"] is not None:\\n            result["prediction"] = grids[0]\\n        else:\\n            result["predictions"] = grids\\n        result.update(status="ANSWERED" if any(g is not None for g in grids) else "NO_GRID", prompt_tokens=len(prompt),\\n                      completion_tokens=len(tokens), thinking_tokens=thinking, finish_reason=finish,\\n                      answer_tail=answer[-2000:])\\n    except Exception as error:\\n        result.update(status="ERROR", error=f"{type(error).__name__}: {error}")\\n    result["elapsed_seconds"] = time.monotonic() - started\\n    atomic_json(root / f"{job[\\\'key\\\']}.json", result)\\n    return result\\n\', \'src/qwen/runtime.py\': \'"""Kaggle runtime for Qwen3.8 direct (transductive) answers: one vLLM server over all GPUs, direct answers in\\nattempt-major order (every output\\\'s first attempt before any second attempt) until the deadline, then submission.json.\\n\\nA valid submission ([[0]] everywhere) is written before the server starts and rewritten at the end. Each output\\\'s\\nattempt_1 and attempt_2 are its first two distinct parsed grids in attempt order; an output without one keeps [[0]].\\nA job not started by the generation deadline is skipped; requests still running then are abandoned. A server that dies\\nmid-run is restarted with the same command and the jobs it failed run again (answer, recover; status.json "restarts"),\\nor one level down config["fallback_env"] when its log reports a GPU allocation failure (status.json "fallback").\\n\\nWith evaluation agent_mode "free" (job_unit "task", engine "openai") each job is a src.agent free-mode trajectory\\n(free_answer): the direct prompt with the optional arc_python_executor tool, up to max_rounds rounds sharing the\\nthinking budget; config["trajectories"] (default the results directory) holds its per-round files.\\n    python -m src.qwen.runtime --config CONFIG.json\\n"""\\nimport argparse\\nfrom collections import Counter\\nfrom concurrent.futures import ThreadPoolExecutor, wait\\nfrom datetime import datetime, timezone\\nimport errno\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport re\\nimport shutil\\nimport statistics\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\n\\nfrom src.qwen.direct import direct_answer, direct_jobs_for\\nfrom src.utils.cuda_linker import prepare_driver_linker\\nfrom src.utils.kaggle_runtime import atomic_json, stop_group, validate_submission\\n\\nSAMPLING = ("temperature", "top_p", "top_k", "thinking_token_budget", "max_tokens", "max_model_len")\\nFREE = ("agent_mode", "max_rounds", "execution_timeout", "earlier_reasoning", "thinking_scope", "tool", "context_brake",\\n        "budget_notice", "helpers", "permissive", "reasoning_effort", "exact_tool_text", "token_continuity", "avoid_previous",\\n        "previous_program", "previous_program_prompt", "second_hypothesis", "second_hypothesis_tokens", "second_hypothesis_guard", "second_hypothesis_prompt",\\n        "deadline_brake", "repeat_refuse", "previous_program_slice", "round_clamp", "doubt_hint", "scipy_ndimage")\\n\\n\\ndef filled_slots(row: dict, count: int) -> bool:\\n    """skip_filled: whether a first attempt\\\'s task-level row already gives each of its count test outputs two distinct\\n    grids (its answer, then its program prediction: the first two assemble keeps), so no later attempt can reach the\\n    submission."""\\n    predictions, programs = row.get("predictions") or [], row.get("program_predictions") or []\\n    return count > 0 and all(\\n        len({json.dumps(grid) for grid in (predictions[i] if i < len(predictions) else None,\\n                                            programs[i] if i < len(programs) else None) if grid is not None}) == 2\\n        for i in range(count))\\n\\n\\ndef rule_doubt(task: dict, predictions: list) -> bool:\\n    """doubt_rules: whether an answer breaks an invariant that every training pair of its task keeps, among the three\\n    that on the AZ family\\\'s four eval-120 runs flagged wrong verified answers only (11 of 155 wrong outputs, none of 574\\n    right): the output uses exactly the input\\\'s colors; the output, of the input\\\'s shape, keeps every background cell of\\n    the input; the output has as many non-background cells as the input (src.agent.tools.broken_invariants)."""\\n    from src.agent.tools import broken_invariants\\n\\n    return bool(broken_invariants(task, predictions))\\n\\n\\nSECOND_HYPOTHESIS_GUARD = 1200  # settings second_hypothesis_guard\\\'s default: seconds before the generation deadline\\n\\n\\ndef server_command(config: dict) -> list[str]:\\n    """vLLM\\\'s OpenAI server, or with config["engine"] == "sglang" SGLang\\\'s server (sglang_command)."""\\n    if config.get("engine") == "sglang":\\n        return sglang_command(config)\\n    settings = config["evaluation"]\\n    return [sys.executable, "-m", "vllm.entrypoints.openai.api_server",\\n            "--model", config["model"], "--host", "127.0.0.1", "--port", str(config["port"]),\\n            "--served-model-name", config["served_model"], "--tensor-parallel-size", str(len(config["gpus"])),\\n            "--dtype", "bfloat16", "--max-model-len", str(settings["max_model_len"]),\\n            "--max-num-seqs", str(settings["concurrency"]), "--max-num-batched-tokens", str(config.get("max_num_batched_tokens", 4096)),\\n            "--gpu-memory-utilization", str(config["gpu_memory_utilization"]), "--kv-cache-dtype", config["kv_cache_dtype"],\\n            "--language-model-only", "--generation-config", "vllm", "--reasoning-parser", "qwen3",\\n            "--compilation-config", json.dumps(config["compilation_config"])] + config.get("server_args", [])\\n\\n\\ndef sglang_cuda_home(environment: dict, driver_link: str | None, nvidia_paths=None) -> str:\\n    """Point SGLang\\\'s JIT kernel builds at the CUDA 13 toolkit its wheels install (nvidia/cu13): Kaggle\\\'s system CUDA is\\n    too old for flashinfer 0.6.18\\\'s headers. The JIT links with -L$CUDA_HOME/lib64 -lcudart (and -lcuda from\\n    lib64/stubs), names the wheel layout lacks, so they are added as links."""\\n    if nvidia_paths is None:\\n        # Not `import nvidia`: an image\\\'s own regular `nvidia` package shadows the wheels\\\' namespace directory.\\n        entries = environment.get("PYTHONPATH", "").split(os.pathsep) + sys.path\\n        nvidia_paths = [Path(entry) / "nvidia" for entry in entries if entry]\\n    home = next(Path(path) / "cu13" for path in nvidia_paths if (Path(path) / "cu13" / "bin" / "nvcc").exists())\\n    if not (home / "lib64").exists():\\n        (home / "lib64").symlink_to("lib")\\n    if not (home / "lib" / "libcudart.so").exists():\\n        (home / "lib" / "libcudart.so").symlink_to("libcudart.so.13")\\n    if driver_link and not (home / "lib" / "stubs" / "libcuda.so").exists():\\n        (home / "lib" / "stubs").mkdir(exist_ok=True)\\n        (home / "lib" / "stubs" / "libcuda.so").symlink_to(driver_link)\\n    # The wheels pair nvcc 13.4 with 13.0 runtime headers, which CCCL\\\'s same-major.minor check rejects; the headers are\\n    # only used to compile, so the check is skipped (CCCL_DISABLE_CTK_COMPATIBILITY_CHECK).\\n    flags = environment.get("NVCC_APPEND_FLAGS", "")\\n    environment.update(CUDA_HOME=str(home), PATH=f"{home / \\\'bin\\\'}:{environment.get(\\\'PATH\\\', \\\'\\\')}",\\n                       NVCC_APPEND_FLAGS=f"{flags} -DCCCL_DISABLE_CTK_COMPATIBILITY_CHECK".strip())\\n    return str(home)\\n\\n\\ndef sglang_command(config: dict) -> list[str]:\\n    """SGLang\\\'s server with the same model, context, concurrency, KV dtype and memory share; speculative decoding and\\n    backends come from config["server_args"]."""\\n    settings = config["evaluation"]\\n    return [sys.executable, "-m", "sglang.launch_server", "--model-path", config["model"], "--host", "127.0.0.1",\\n            "--port", str(config["port"]), "--served-model-name", config["served_model"], "--tp-size", str(len(config["gpus"])),\\n            "--context-length", str(settings["max_model_len"]), "--max-running-requests", str(settings["concurrency"]),\\n            "--kv-cache-dtype", config["kv_cache_dtype"], "--random-seed", str(settings["seed"]),\\n            # SGLang\\\'s own warmup request waits behind real requests and stops the server when it times out\\n            "--skip-server-warmup"] + (\\n            ["--mem-fraction-static", str(config["gpu_memory_utilization"])] if config.get("gpu_memory_utilization") else []\\n            ) + config.get("server_args", [])\\n\\n\\nCAPPED, UNVERIFIED, RUNNING, VERIFIED = range(4)  # JobQueue\\\'s order of later attempts, by how the first attempt ended\\nLATER_ORDER = ("capped", "unverified", "running", "verified")  # their names, for settings["later_order"]\\n\\n\\ndef previous_answer(job: dict, first_keys: dict, results: Path, given: dict | None) -> list | None:\\n    """The earlier answer a job with avoid_previous is shown: given[task_id] (config["previous_answers"], e.g. another\\n    run\\\'s first attempts), else the grids of the task\\\'s first attempt in results; None when there is none with a grid\\n    (the first attempt not finished, or without a grid), and the job then runs as without avoid_previous."""\\n    if given is not None:\\n        grids = given.get(job["task_id"])\\n    else:\\n        key = first_keys.get((job["task_id"], job["test_index"]))\\n        path = results / f"{key}.json"\\n        grids = json.loads(path.read_text()).get("predictions") if key not in (None, job["key"]) and path.exists() else None\\n    return grids if grids and any(grid is not None for grid in grids) else None\\n\\n\\ndef earlier_program(job: dict, first_keys: dict, trajectories: Path, task: dict, given: dict | None,\\n                    viewed: set, slice_definitions: bool = False) -> dict | None:\\n    """The first attempt\\\'s previous_program record a job with previous_program is shown beside previous_answer:\\n    given[task_id] (config["previous_programs"]), else read from the first attempt\\\'s trajectory; None when there is\\n    none, or when the first attempt ran under a view (its programs see the task transformed)."""\\n    from src.agent.loop import previous_program\\n\\n    if given is not None:\\n        return given.get(job["task_id"])\\n    key = first_keys.get((job["task_id"], job["test_index"]))\\n    directory = trajectories / key if key not in (None, job["key"]) else None\\n    return (previous_program(directory, task, slice_definitions) if directory is not None and directory.is_dir() and key not in viewed\\n            else None)\\n\\n\\ndef attempt_settings(job: dict, rank: dict, verified_later: bool, tokens: dict | None = None,\\n                     max_first_tokens: int | None = None) -> int:\\n    """The job\\\'s per-attempt settings: 0 for a first attempt, 1 (later_attempts) for a later one, 2 (also\\n    verified_later_attempts) for a later one whose first attempt ended VERIFIED in JobQueue.rank and, with\\n    max_first_tokens, generated at most that many completion tokens (JobQueue.tokens). A later attempt taken while its\\n    first attempt still runs (no rank yet, near the end of the first attempts) gets 1. An extra attempt (JobQueue\\\'s extra)\\n    gets 0: another sample of the first attempt\\\'s prompt."""\\n    if job["attempt"] == 0 or job.get("extra"):\\n        return 0\\n    key = (job["task_id"], job["test_index"])\\n    short = max_first_tokens is None or (tokens or {}).get(key, 0) <= max_first_tokens\\n    return 2 if verified_later and rank.get(key) == VERIFIED and short else 1\\n\\n\\nclass JobQueue:\\n    """Hands jobs to the request threads: every first attempt in order, then the later attempts. With priority, a later\\n    attempt whose first attempt hit the budget or gave no grid goes first, then one whose first attempt ended by itself\\n    with an answer that is not a verified program\\\'s (src.agent.loop.verified), then one whose first attempt is still\\n    running, and last one whose first attempt ended by itself with a verified answer (cap64/F3/448 @Ada first attempts\\n    with lift/F3 cut at 64K as the second: 3.38 points per million second-attempt tokens on budget-hit first attempts,\\n    about 1.5 on the others; lift/F3 @Ada first attempts with lift/F2 as the second: 92 of 120 verified and 85% right,\\n    the rest 32% right, so lift/F2 only where lift/F3 is not verified gives 79.4 for 7.34M tokens, against 84.4 for\\n    11.19M on every task). order (a permutation of LATER_ORDER\\\'s names) serves those four groups in another order.\\n    shortest_first takes, within a group, the later attempt whose first attempt generated the fewest completion tokens\\n    first (a running first attempt counts 0); otherwise a group goes in task order (on the 12-hour simulator, 240 tasks\\n    with plain second attempts on verified tasks: 164 against 130 second attempts done in 11.2 hours, plain second\\n    attempts rescuing 4.72 points per million tokens after 10-20K-token first attempts, 1.17 after 40-80K). It also keeps\\n    the finished requests\\\' generation rates for deadline_budget.\\n    Jobs marked "extra" (settings extra_attempts) go only after every first and later attempt has been taken, and only\\n    for a task with no verified answer from any attempt so far (a doubted one does not count) whose first attempt did\\n    not fill both slots: those with no attempt in flight first, then in attempt and task order; the others are dropped\\n    unstarted (AZ-Focus eval-120 runs of 10-24, one run\\\'s attempts 1 and 2 plus the other run\\\'s attempt 1 as a\\n    third: 20 and 25 tasks without a verified answer after two attempts, 5 and 10 of them verified by the third, +0.84\\n    and +4.44 points with verified later answers ahead of unverified ones in the slots, assemble)."""\\n\\n    def __init__(self, jobs: list[dict], priority: bool = False, order=LATER_ORDER, shortest_first: bool = False,\\n                 skip_filled: dict | None = None, doubt: dict | None = None):\\n        """skip_filled ({task_id: test outputs}, settings skip_filled): a task whose first attempt filled both slots of\\n        every output (filled_slots) has its later attempts dropped unstarted, counted in skipped. doubt ({task_id: task},\\n        settings doubt_rules): a verified first answer that rule_doubt flags ranks UNVERIFIED, counted in\\n        doubted."""\\n        if sorted(order) != sorted(LATER_ORDER):\\n            raise ValueError(f"later_order is a permutation of {LATER_ORDER}, got {order!r}")\\n        if shortest_first and not priority:\\n            raise ValueError("later_shortest_first orders within second_priority\\\'s groups; set second_priority")\\n        self.first = [job for job in jobs if job["attempt"] == 0]\\n        self.later = [job for job in jobs if job["attempt"] > 0 and not job.get("extra")]\\n        self.extra = [job for job in jobs if job.get("extra")]\\n        self.verified, self.inflight, self.extra_taken = set(), {}, 0  # keys with a verified answer; taken, unfinished jobs\\n        self.priority, self.shortest_first, self.rank, self.tokens, self.rates, self.taken = priority, shortest_first, {}, {}, [], 0\\n        self.position = {LATER_ORDER.index(name): index for index, name in enumerate(order)}  # rank -> place in the order\\n        self.lock = threading.Lock()\\n        self.outputs, self.filled, self.skipped = skip_filled, set(), 0\\n        self.doubt, self.doubted = doubt, set()\\n        self.generated, self.first_done = 0, set()  # timing_code\\\'s run rate; first attempts with a result (a retry once)\\n        self.later_done = set()  # timing_code\\\'s L: later attempts (not extra) with a result\\n\\n    def record(self, job: dict, result: dict, budget: int | None) -> None:\\n        with self.lock:\\n            self.generated += result.get("completion_tokens") or 0\\n            if job["attempt"] == 0:\\n                self.first_done.add((job["task_id"], job["test_index"]))\\n            elif not job.get("extra"):\\n                self.later_done.add((job.get("task_id"), job.get("test_index"), job["attempt"]))\\n            if result.get("completion_tokens") and result.get("elapsed_seconds"):\\n                self.rates.append(result["completion_tokens"] / result["elapsed_seconds"])\\n            if result.get("verified") and not (self.doubt is not None and job["test_index"] is None\\n                                               and rule_doubt(self.doubt[job["task_id"]], result.get("predictions"))):\\n                self.verified.add((job["task_id"], job["test_index"]))\\n            if job["attempt"] == 0:\\n                thinking = result.get("thinking_tokens")\\n                answered = result.get("prediction") is not None or any(g is not None for g in result.get("predictions") or [])\\n                natural = result["natural"] if "natural" in result else (\\n                    thinking is not None and (budget is None or thinking < budget - 1) and answered)\\n                rank = (VERIFIED if result.get("verified") else UNVERIFIED) if natural else CAPPED\\n                if (rank == VERIFIED and self.doubt is not None and job["test_index"] is None\\n                        and rule_doubt(self.doubt[job["task_id"]], result.get("predictions"))):\\n                    rank = UNVERIFIED\\n                    self.doubted.add(job["task_id"])\\n                self.rank[(job["task_id"], job["test_index"])] = rank\\n                self.tokens[(job["task_id"], job["test_index"])] = result.get("completion_tokens") or 0\\n                if self.outputs is not None and job["test_index"] is None and filled_slots(result, self.outputs[job["task_id"]]):\\n                    self.filled.add(job["task_id"])\\n\\n    def rate(self) -> float | None:\\n        """Median generated tokens per second of the last 32 finished requests (None before 8)."""\\n        with self.lock:\\n            return statistics.median(self.rates[-32:]) if len(self.rates) >= 8 else None\\n\\n    def defer(self, job: dict) -> None:\\n        """Puts a taken job back behind every waiting one (a job the server died on twice: whether it kills the server\\n        or was only in flight, the jobs still waiting run first)."""\\n        with self.lock:\\n            (self.extra if job.get("extra") else self.later).append(job)\\n            self.taken -= 1\\n            self.extra_taken -= bool(job.get("extra"))\\n            self._finish(job)\\n\\n    def finish(self, job: dict) -> None:\\n        """A taken job\\\'s request thread is done with it (answered, failed or deferred)."""\\n        with self.lock:\\n            self._finish(job)\\n\\n    def _finish(self, job: dict) -> None:\\n        key = (job["task_id"], job["test_index"])\\n        self.inflight[key] = self.inflight.get(key, 1) - 1\\n\\n    def order(self, job: dict) -> tuple[int, int]:\\n        """A later attempt\\\'s place: its group\\\'s position, then (shortest_first) its first attempt\\\'s completion tokens."""\\n        key = (job["task_id"], job["test_index"])\\n        return self.position[self.rank.get(key, RUNNING)], self.tokens.get(key, 0) if self.shortest_first else 0\\n\\n    def take(self) -> dict | None:\\n        with self.lock:\\n            if not self.first and self.filled:\\n                dropped = [job for job in self.later if job["task_id"] in self.filled and job["test_index"] is None]\\n                self.later = [job for job in self.later if job not in dropped]\\n                self.skipped += len(dropped)\\n            if self.first:\\n                job = self.first.pop(0)\\n            elif self.later:\\n                job = min(self.later, key=self.order) if self.priority else self.later[0]\\n                self.later.remove(job)\\n            else:\\n                self.extra = [job for job in self.extra if (job["task_id"], job["test_index"]) not in self.verified\\n                              and not (job["task_id"] in self.filled and job["test_index"] is None)]\\n                if not self.extra:\\n                    return None\\n                job = min(self.extra, key=lambda job: self.inflight.get((job["task_id"], job["test_index"]), 0) > 0)\\n                self.extra.remove(job)\\n                self.extra_taken += 1\\n            key = (job["task_id"], job["test_index"])\\n            self.inflight[key] = self.inflight.get(key, 0) + 1\\n            self.taken += 1\\n            return job\\n\\n\\ndef deadline_budget(rate: float | None, remaining: float, budget: int, answer_room: int, minimum: int) -> int | None:\\n    """Thinking budget for a request started with `remaining` seconds left: the full budget while `rate` (generated\\n    tokens per second of one request) leaves time for it, else what fits in 90% of the time minus the answer room, so\\n    the answer is forced before the deadline instead of being abandoned; None (do not start) below `minimum`."""\\n    if rate is None:\\n        return budget\\n    fits = int(0.9 * rate * remaining) - answer_room\\n    return budget if fits >= budget else fits if fits >= minimum else None\\n\\n\\ndef assemble(tasks: dict, results: list[dict]) -> dict:\\n    """attempt_1/attempt_2 per output: its first two distinct grids in attempt order, else [[0]]; a row\\\'s\\n    program_prediction (a free-mode tool "f3" answer\\\'s passing program output, see finalize) right after its own grid,\\n    and a row\\\'s fill_prediction (finalize\\\'s slot_fill) only after every row\\\'s answers and program outputs. After the\\n    first attempt, verified rows go before unverified ones (the same order with one later attempt; with extra attempts,\\n    a verified third answer takes the slot of an unverified second)."""\\n    grids = {}\\n    for row in sorted(results, key=lambda row: (row["attempt"] > 0, row["attempt"] > 0 and not row.get("verified"), row["attempt"])):\\n        found = grids.setdefault((row["task_id"], row["test_index"]), [])\\n        for grid in (row.get("prediction"), row.get("program_prediction")):\\n            if grid is not None and grid not in found:\\n                found.append(grid)\\n    for row in sorted(results, key=lambda row: row["attempt"]):\\n        found = grids[(row["task_id"], row["test_index"])]\\n        if row.get("fill_prediction") is not None and row["fill_prediction"] not in found:\\n            found.append(row["fill_prediction"])\\n    submission = {}\\n    for task_id, task in tasks.items():\\n        submission[task_id] = []\\n        for index in range(len(task["test"])):\\n            found = grids.get((task_id, index), []) + [[[0]], [[0]]]\\n            submission[task_id].append({"attempt_1": found[0], "attempt_2": found[1]})\\n    return submission\\n\\n\\ndef fill_predictions(directory: Path, count: int, prediction: list) -> list:\\n    """slot_fill: per test output, the recorded program output of a free-mode trajectory (every executor call and every\\n    function variant it checked) that differs from the prediction\\\'s grid, most training pairs passed first, then the\\n    latest; None where there is none. Recorded outputs only (in the trajectory\\\'s view: callers skip viewed jobs)."""\\n    found = [[] for _ in range(count)]\\n    order = 0\\n    for path in sorted(directory.glob("[0-9][0-9]_execution.json")):\\n        try:\\n            turn = json.loads(path.read_text())\\n        except ValueError:  # a round still being written\\n            break\\n        for call in turn.get("calls") or []:\\n            for run in [call, *(call.get("functions") or {}).values()]:\\n                for index, grid in enumerate((run.get("test_outputs") or [])[:count]):\\n                    if grid is not None:\\n                        found[index].append((run.get("train_passed") or 0, order, grid))\\n                order += 1\\n    mine = list(prediction or []) + [None] * count\\n    return [next((grid for _, _, grid in sorted(options, key=lambda item: (-item[0], -item[1])) if grid != mine[index]), None)\\n            for index, options in enumerate(found)]\\n\\n\\ndef salvage(directory: Path, job: dict, task: dict) -> list | None:\\n    """The program fallback (src.agent.loop.program_fallback) of a free-mode trajectory\\\'s recorded rounds, up to the\\n    first unreadable one (a round still being written)."""\\n    from src.agent.loop import job_view, program_fallback\\n\\n    turns = []\\n    for path in sorted(directory.glob("[0-9][0-9]_execution.json")):\\n        try:\\n            turns.append(json.loads(path.read_text()))\\n        except ValueError:\\n            break\\n    return program_fallback(turns, job_view(job, task)[1])\\n\\n\\ndef free_answer(job: dict, task: dict, root: Path, model: str, settings: dict, tokenizer, client,\\n                trajectories: Path | None = None) -> dict:\\n    """agent_mode free: one src.agent.loop.trajectory under trajectories/<key>/ (default root), its row written to\\n    root/<key>.json as direct_answer writes it. A retry moves the failed attempt\\\'s rounds to <key>.error/; a trajectory\\n    without a grid (an ERROR included) keeps the test outputs of its last passing program (salvage). natural: the model\\n    answered by itself, before the last round and before the thinking budget forced the answer (with tool "f3", before\\n    any final round: forced)."""\\n    from src.agent.loop import FINAL_ANSWER_RESERVE, trajectory\\n\\n    trajectories = trajectories or root\\n    directory = trajectories / job["key"]\\n    if directory.exists():\\n        directory.rename(trajectories / f"{job[\\\'key\\\']}.error")\\n    started = time.monotonic()\\n    try:\\n        result = trajectory(job, task, trajectories, model, settings, tokenizer, client)\\n    except Exception as error:\\n        result = {**job, "status": "ERROR", "prediction": None, "turns": 0, "thinking_tokens": 0,\\n                  "error": f"{type(error).__name__}: {error}", "elapsed_seconds": time.monotonic() - started}\\n    predictions = result["prediction"]\\n    if not any(grid is not None for grid in predictions or []):\\n        predictions = salvage(directory, job, task)\\n    responses = directory.glob("[0-9][0-9]_response.json")\\n    row = {key: value for key, value in result.items() if key != "prediction"}\\n    second = result.get("second_hypothesis") or {}  # its continuation does not count: the first answer ended by itself\\n    row.update(predictions=predictions,\\n               completion_tokens=sum(json.loads(path.read_text())["usage"]["completion_tokens"] for path in responses),\\n               natural=(result["status"] == "ANSWERED" and "fallback" not in result\\n                        and result["turns"] - second.get("rounds", 0) < settings["max_rounds"]\\n                        and result["thinking_tokens"] - second.get("thinking_tokens", 0) < settings["thinking_token_budget"] - FINAL_ANSWER_RESERVE\\n                        and not result.get("forced") and "deadline_brake" not in result))\\n    atomic_json(root / f"{job[\\\'key\\\']}.json", row)\\n    return row\\n\\n\\nSIZE_MODULUS = 10_000_000  # timing_code\\\'s number is the final submission file\\\'s size modulo this\\n\\n\\ndef mount_class(fstype: str | None) -> int | None:\\n    """0 nfs, 1 fuse (fuse, fuseblk, gcsfuse), 2 any other filesystem; None when unknown."""\\n    if not fstype:\\n        return None\\n    return 0 if fstype.startswith("nfs") else 1 if fstype.startswith("fuse") or "gcsfuse" in fstype else 2\\n\\n\\ndef scipy_ndimage_check(requests: list[dict]) -> dict | None:\\n    """settings scipy_ndimage at the start of a run: None when no attempt asks for it; else whether the executor\\\'s\\n    workers import scipy.ndimage (src.agent.executor.ndimage_error). When they cannot, the setting leaves every\\n    attempt\\\'s requests, which then run without it: no import in the workers and no library line in the prompt."""\\n    if not any(own.get("scipy_ndimage") for own in requests):\\n        return None\\n    from src.agent.executor import ndimage_error\\n\\n    started = time.monotonic()\\n    error = ndimage_error()\\n    if error:\\n        for own in requests:\\n            own.pop("scipy_ndimage", None)\\n    return {"importable": error is None, "error": error, "seconds": round(time.monotonic() - started, 1)}\\n\\n\\ndef timing_code(state: dict) -> dict:\\n    """The run\\\'s infrastructure figures as one number below SIZE_MODULUS, which the final submission file carries in\\n    its size (finalize\\\'s size_code), the only figure a scored competition run reports back:\\n    F * 1,000,000 + R * 100,000 + C * 10,000 + V * 1,000 + L * 100 + D * 10 + Z. Nothing in it comes from the answers.\\n    F: 1 + 3 m + p for the mounts the served model\\\'s shards (m, state["model_fstype"]) and its PLE table\\\'s shards (p,\\n       state["ple_fstype"]) are read from (mount_class: 0 nfs, 1 fuse, 2 other), 0 when either is unknown.\\n    R: the server restarts attempted (at most 9).\\n    C: the run\\\'s generation rate (state["generation_tok_s"]: finished requests\\\' completion tokens over the time from the\\n       first request to the end of generation) in 50 tok/s units, at most 9; 0 when unknown. (Until 10-27: the\\n       copy-first stage, 5 on every run since copy-first was dropped; it stays in the returned copy_stage.)\\n    V: the first start: 0 ready, 1 the server exited during startup, 2 the startup deadline passed, 3 another error\\n       before the server was ready; plus 4 when settings scipy_ndimage was dropped because the workers\\\' interpreter\\n       does not import scipy.ndimage (state["scipy_ndimage"], scipy_ndimage_check).\\n    L: the later attempts done (state["later_fraction"]: those with a result or dropped by skip_filled, over all later\\n       attempts but the extra ones) in tenths, rounded down, at most 9 (9: 90% or more); 0 when unknown. Whether the second\\n       attempts ran is what more throughput buys.\\n    D: when ready, the minutes from the runtime\\\'s start to the first ready server in 5-minute units rounded up; when\\n       never ready, the GiB written to the copy at the decision in 2 GiB units; at most 9 (until 10-30 two digits, at\\n       most 99, where L now is).\\n    Z: the first attempts without a result at the end of generation (state["first_unfinished"]: abandoned or never\\n       started) in units of 3, at most 9; 0 when unknown. (Until 10-27: the copy disk\\\'s free space, 0 on every run\\n       since; it stays in the returned free_100gib.)"""\\n    model, ple = mount_class(state.get("model_fstype")), mount_class(state.get("ple_fstype"))\\n    mount = 0 if model is None or ple is None else 1 + 3 * model + ple\\n    attempts = [record for record in state.get("restarts") or [] if "refused" not in record]\\n    restarts = min(len(attempts), 9)\\n    copy = state.get("copy")\\n    if not copy:\\n        stage = 5\\n    elif copy["served"]:\\n        stage = 0\\n    else:\\n        stage = 1 if "EIO" in copy["errors"] else 2 if "ENOSPC" in copy["errors"] else 3 if copy["ended"] else 4\\n    error = state.get("error") or ""\\n    ready = "ready_seconds" in state\\n    server = 0 if ready else 1 if "exited during startup" in error or "failed at startup" in error else 2 if error.startswith("TimeoutError") else 3\\n    detail = math.ceil(state["ready_seconds"] / 300) if ready else int((copy or {}).get("copied_gib", 0) // 2)\\n    detail = min(detail, 9)\\n    free = min(int(state.get("copy_disk_free_gib") or 0) // 100, 9)\\n    try:  # the end-of-run path: a figure that cannot be read is 0, never an exception before finalize\\n        speed = min(int(float(state.get("generation_tok_s") or 0) // 50), 9)\\n        late = min(max(int(state.get("first_unfinished") or 0), 0) // 3, 9)\\n    except (TypeError, ValueError):\\n        speed, late = 0, 0\\n    try:\\n        later = min(max(int(10 * float(state.get("later_fraction") or 0)), 0), 9)\\n    except (TypeError, ValueError):\\n        later = 0\\n    dropped = int((state.get("scipy_ndimage") or {}).get("importable") is False)\\n    return {"code": mount * 1_000_000 + restarts * 100_000 + speed * 10_000 + (server + 4 * dropped) * 1_000 + later * 100 + detail * 10 + late,\\n            "mount": mount, "model_fstype": state.get("model_fstype"), "ple_fstype": state.get("ple_fstype"),\\n            "restarts": restarts, "speed_50tok_s": speed, "server_stage": server, "ndimage_dropped": dropped, "detail": detail,\\n            "first_unfinished_3": late, "later_done_tenths": later,\\n            "copy_stage": stage, "free_100gib": free,\\n            "restart_wait_seconds": round(sum(record.get("wait_seconds", 0) for record in attempts), 1)}\\n\\n\\ndef decode_timing(total_bytes: int) -> dict:\\n    """timing_code\\\'s figures back from a scored submission file\\\'s size (get_competition_submission\\\'s total_bytes)."""\\n    code = int(total_bytes) % SIZE_MODULUS\\n    return {"code": code, "mount": code // 1_000_000, "restarts": code // 100_000 % 10, "speed_50tok_s": code // 10_000 % 10,\\n            "server_stage": code // 1_000 % 10 % 4, "ndimage_dropped": code // 1_000 % 10 // 4,\\n            "later_done_tenths": code // 100 % 10, "detail": code // 10 % 10, "first_unfinished_3": code % 10}\\n\\n\\ndef model_fstype(model: str, ple: bool = False) -> str | None:\\n    """`stat -f -c %T` of the served model directory\\\'s first safetensors file, through its symlink: the type of the\\n    mount it is read from (on Kaggle under /kaggle/input); with ple, of its first PLE-table shard (model-plefp8-*).\\n    None when it cannot be told."""\\n    try:\\n        first = sorted(path for path in Path(model).glob("*.safetensors") if ("plefp8" in path.name) == ple)[0].resolve()\\n        return subprocess.run(["stat", "-f", "-c", "%T", str(first)], capture_output=True, text=True, timeout=30,\\n                              check=True).stdout.strip() or None\\n    except (IndexError, OSError, subprocess.SubprocessError):\\n        return None\\n\\n\\nERRNO_TEXT = re.compile(r"\\\\[Errno (\\\\d+)\\\\]")\\n\\n\\ndef copy_outcome(spec: dict, served: bool) -> dict:\\n    """The local_model_copy_first copy at the first start\\\'s decision: whether the start serves it, whether the\\n    prefetch had ended (its summary line in spec["log"]), the errno names of its read errors with their counts (the\\n    summary\\\'s "errors" when it has them, else each "[Errno N]" in the log\\\'s text), and the GiB written into\\n    spec["copy"] (allocated blocks, since a copier may size each .part file up front)."""\\n    try:\\n        text = Path(spec["log"]).read_text() if spec.get("log") else ""\\n    except OSError:\\n        text = ""\\n    errors = Counter(errno.errorcode.get(int(number), number) for number in ERRNO_TEXT.findall(text))\\n    summary = next((line for line in reversed(text.splitlines()) if \\\'"files"\\\' in line), None)\\n    try:  # the dual copier\\\'s summary counts every read error by name (its file rows keep only each file\\\'s last)\\n        errors = Counter(json.loads(summary)["errors"]) if summary and "errors" in json.loads(summary) else errors\\n    except ValueError:\\n        pass\\n    written = 0\\n    for path in Path(spec["copy"]).glob("*") if spec.get("copy") else ():\\n        try:\\n            written += path.stat().st_blocks * 512\\n        except OSError:\\n            pass\\n    return {"served": served, "ended": summary is not None, "errors": dict(errors), "copied_gib": round(written / 2**30, 1)}\\n\\n\\ndef disk_free_gib(path: str) -> float | None:\\n    """Free GiB on the file system that holds path (its nearest existing ancestor); None when it cannot be told."""\\n    probe = Path(path)\\n    while not probe.exists() and probe != probe.parent:\\n        probe = probe.parent\\n    try:\\n        return round(shutil.disk_usage(probe).free / 2**30, 1)\\n    except OSError:\\n        return None\\n\\n\\ndef copy_rate(log: Path) -> float | None:\\n    """The notebook prefetch\\\'s read rate in MB/s from its summary line in prefetch.log (None without one)."""\\n    try:\\n        lines = log.read_text().splitlines()\\n    except OSError:\\n        return None\\n    summary = next((json.loads(line) for line in reversed(lines) if \\\'"files"\\\' in line), None)\\n    return summary.get("mb_per_s") if summary else None\\n\\n\\ndef finalize(tasks: dict, results_dir: Path, destination: Path, jobs: list[dict] | None = None,\\n             trajectories: Path | None = None, size_code: int | None = None, slot_fill: bool = False) -> dict:\\n    """With jobs (agent_mode free), a job whose trajectory directory exists under trajectories (default results_dir)\\n    but has no result row (abandoned at the deadline) counts with its salvaged program outputs; nothing is written for\\n    it, so threads still running cannot race this. A row\\\'s program_predictions (tool "f3": the last passing program\\\'s\\n    test outputs where they differ from the answer) follow its answer in each output\\\'s candidates (assemble), so a slot\\n    keeps the program\\\'s output when the written answer differs from it. With size_code, spaces after the JSON make\\n    the file\\\'s size modulo SIZE_MODULUS equal it (timing_code); the JSON and its answers are unchanged. slot_fill\\n    (settings "slot_fill", agent_mode free): each first attempt\\\'s other recorded program outputs (fill_predictions) fill\\n    a slot nothing else fills, so an output whose later attempt never finished does not leave attempt_2 empty."""\\n    # ponytail: no mid-trajectory forced answer at the deadline; only programs are salvaged\\n    results = []\\n    trajectories = trajectories or results_dir\\n    rows = {path.stem: json.loads(path.read_text()) for path in sorted(results_dir.glob("d[0-9]*[0-9].json"))}\\n    unfinished = [{**job, "predictions": salvage(trajectories / job["key"], job, tasks[job["task_id"]]) or []}\\n                  for job in jobs or [] if job["key"] not in rows and (trajectories / job["key"]).is_dir()]\\n    viewed = {job["key"] for job in jobs or [] if job.get("view")}\\n    for row in [*rows.values(), *unfinished]:\\n        # a task-level answer (test_index None) counts as one row per test output\\n        programs = row.get("program_predictions") or []\\n        count = len(tasks[row["task_id"]]["test"])\\n        fills = []\\n        if slot_fill and row.get("test_index") is None and row["attempt"] == 0 and row["key"] not in viewed:\\n            try:\\n                fills = fill_predictions(trajectories / row["key"], count, row.get("predictions"))\\n            except Exception:  # an unreadable record leaves the slot as without slot_fill, never the submission\\n                fills = []\\n        results += [row] if row.get("test_index") is not None else [\\n            {**row, "test_index": index, "prediction": grid,\\n             **({"program_prediction": programs[index]} if index < len(programs) else {}),\\n             **({"fill_prediction": fills[index]} if index < len(fills) and fills[index] is not None else {})}\\n            for index, grid in enumerate((row.get("predictions") or []) + [None] * (len(fills) - len(row.get("predictions") or [])))]\\n    submission = assemble(tasks, results)\\n    validate_submission(tasks, submission)\\n    if size_code is None:\\n        atomic_json(destination, submission)\\n    else:  # atomic_json\\\'s text, then the padding\\n        body = json.dumps(submission, indent=2) + "\\\\n"\\n        temporary = destination.with_suffix(destination.suffix + ".tmp")\\n        temporary.write_text(body + " " * ((size_code - len(body.encode())) % SIZE_MODULUS))\\n        temporary.replace(destination)\\n    return {"answers": len(results), "grids": sum(row.get("prediction") is not None for row in results),\\n            "outputs_with_grid": sum(entry["attempt_1"] != [[0]] for entries in submission.values() for entry in entries),\\n            "unfinished": len(unfinished)}\\n\\n\\ndef local_model(config: dict) -> Path | None:\\n    """config["local_model_copy"] = {"source": the served model directory, "copy": where the notebook\\\'s prefetch writes\\n    a copy of each of its safetensors files, "link": a directory to make}: when every safetensors file of the served\\n    model has a complete copy (the same size), `link` is made with the served model\\\'s entries, those files pointing at\\n    the copies and the rest at their own targets, and returned, so a restart reads local disk instead of the source\\n    (Kaggle\\\'s /kaggle/input); otherwise None and the restart reads the source."""\\n    spec = config.get("local_model_copy")\\n    if not spec:\\n        return None\\n    source, copy, link = (Path(spec[key]) for key in ("source", "copy", "link"))\\n    entries = sorted(source.iterdir())\\n    shards = [entry for entry in entries if entry.name.endswith(".safetensors")]\\n    if not shards or any(not (copy / entry.name).is_file() or (copy / entry.name).stat().st_size != entry.stat().st_size\\n                         for entry in shards):\\n        return None\\n    shutil.rmtree(link, ignore_errors=True)\\n    link.mkdir(parents=True)\\n    for entry in entries:\\n        (link / entry.name).symlink_to(copy / entry.name if entry.name.endswith(".safetensors") else entry.resolve())\\n    return link\\n\\n\\nLINK_PROBE = """\\nimport json, time, torch\\nn, size = torch.cuda.device_count(), 256 << 20\\nsrc = [torch.empty(size, dtype=torch.uint8, device=f"cuda:{i}") for i in range(n)]\\ndst = [torch.empty(size, dtype=torch.uint8, device=f"cuda:{i}") for i in range(n)]\\nhost = torch.empty(size, dtype=torch.uint8).pin_memory() if n else None\\ndef timed(copy):\\n    copy()\\n    for d in range(n):\\n        torch.cuda.synchronize(d)\\n    start = time.perf_counter()\\n    copy()\\n    for d in range(n):\\n        torch.cuda.synchronize(d)\\n    return round(size / (time.perf_counter() - start) / 1e9, 2)\\nprint(json.dumps({"gpus": n, "d2d_gb_s": {f"{i}>{j}": timed(lambda: dst[j].copy_(src[i])) for i in range(n) for j in range(n) if i != j},\\n                  "h2d_gb_s": {str(i): timed(lambda: dst[i].copy_(host, non_blocking=True)) for i in range(n)}}))\\n"""\\n\\n\\ndef link_probe(environment: dict, timeout: float = 180.0) -> dict:\\n    """config["link_probe"]: GB/s of one 256 MiB copy per ordered GPU pair and from pinned host memory to each GPU, in a\\n    subprocess of its own (its CUDA contexts are gone before the server starts), so a run on a machine with a slow GPU\\n    link can be told from a slow change; {"error": ...} when it fails, never raised."""\\n    try:\\n        done = subprocess.run([sys.executable, "-c", LINK_PROBE], env=environment, capture_output=True, text=True, timeout=timeout)\\n        if done.returncode:\\n            return {"error": done.stderr.strip()[-300:]}\\n        return json.loads(done.stdout.strip().splitlines()[-1])\\n    except Exception as error:\\n        return {"error": f"{type(error).__name__}: {error}"[:300]}\\n\\n\\ndef cgroup_memory_gib(root: Path = Path("/sys/fs/cgroup"), own: Path = Path("/proc/self/cgroup")) -> dict:\\n    """The container\\\'s memory cgroup, which /proc/meminfo does not show (a rented box\\\'s 124 GB limit under a 251 GB\\n    MemTotal): limit (None when unlimited), usage and peak in GiB and the OOM kills so far, from cgroup v2 (memory.max,\\n    memory.current, memory.peak, memory.events, in this process\\\'s cgroup from `own`: the namespace root inside a\\n    container) or v1 (memory/memory.limit_in_bytes, usage_in_bytes, max_usage_in_bytes, oom_control); {} where neither\\n    can be read."""\\n    def read(path: Path) -> str | None:\\n        try:\\n            return path.read_text().strip()\\n        except OSError:\\n            return None\\n\\n    def gib(text: str | None) -> float | None:\\n        return round(int(text) / 2**30, 1) if text and text.isdigit() and int(text) < 1 << 60 else None\\n\\n    def kills(text: str | None) -> int | None:\\n        return next((int(line.split()[1]) for line in (text or "").splitlines() if line.startswith("oom_kill ")), None)\\n\\n    path = next((line[3:] for line in (read(own) or "").splitlines() if line.startswith("0::")), "/")\\n    v2 = root / path.lstrip("/")\\n    if (limit := read(v2 / "memory.max")) is not None:\\n        return {"version": 2, "limit": gib(limit), "usage": gib(read(v2 / "memory.current")),\\n                "peak": gib(read(v2 / "memory.peak")), "oom_kills": kills(read(v2 / "memory.events"))}\\n    v1 = root / "memory"\\n    if (limit := read(v1 / "memory.limit_in_bytes")) is not None:\\n        return {"version": 1, "limit": gib(limit), "usage": gib(read(v1 / "memory.usage_in_bytes")),\\n                "peak": gib(read(v1 / "memory.max_usage_in_bytes")), "oom_kills": kills(read(v1 / "memory.oom_control"))}\\n    return {}\\n\\n\\ndef meminfo_gib() -> dict:\\n    """MemAvailable, Cached and Dirty from /proc/meminfo, in GiB ({} where it cannot be read)."""\\n    try:\\n        fields = {line.split(":")[0]: int(line.split()[1]) for line in Path("/proc/meminfo").read_text().splitlines()}\\n    except (OSError, ValueError, IndexError):\\n        return {}\\n    return {key: round(fields.get(key, 0) / 2**20, 1) for key in ("MemAvailable", "Cached", "Dirty")}\\n\\n\\ndef first_start_model(config: dict, deadline: float, poll: float = 10.0) -> Path | None:\\n    """config["local_model_copy"] with "first" (local_model_copy_first): the first start waits, until `deadline`, for the\\n    notebook\\\'s prefetch to copy every served file, and serves the link directory to the copies (local_model); None\\n    (serve the source) when the prefetch ended without a complete copy (the summary line in config["local_model_copy"]\\n    ["log"]) or the wait ran out. Reading a slow network mount (Kaggle\\\'s /kaggle/input, NFS or GCSFUSE) once, in large\\n    parallel reads, then serving local disk, replaces the servers\\\' own scattered reads of the mount."""\\n    spec = config.get("local_model_copy") or {}\\n    log = Path(spec["log"]) if spec.get("log") else None\\n    while True:\\n        link = local_model(config)\\n        if link or time.time() >= deadline or (log and log.exists() and \\\'"files"\\\' in log.read_text()):\\n            return link or local_model(config)\\n        time.sleep(poll)\\n\\n\\ndef fallback_levels(config: dict) -> list[dict]:\\n    """config["fallback_env"]: one set of variables, or a list of them (a ladder: each level replaces the primary\\\'s\\n    variables, the next one taken after the current one fails)."""\\n    value = config.get("fallback_env")\\n    return [] if not value else list(value) if isinstance(value, list) else [value]\\n\\n\\ndef wait_ready(process: subprocess.Popen, config: dict, deadline: float, log_path: Path | None = None) -> None:\\n    import httpx\\n\\n    url = f"http://127.0.0.1:{config[\\\'port\\\']}/v1/models"\\n    while time.time() < deadline:\\n        if process.poll() is not None:\\n            raise RuntimeError(f"the server exited during startup with code {process.returncode}")\\n        # a failed SGLang scheduler can leave the launcher alive (its cleanup may hit /proc permissions)\\n        if log_path and "Received sigquit from a child process" in log_path.read_text(errors="replace"):\\n            raise RuntimeError("the SGLang scheduler failed at startup")\\n        try:\\n            if [row["id"] for row in httpx.get(url, timeout=5, trust_env=False).json()["data"]] == [config["served_model"]]:\\n                return\\n        except (httpx.HTTPError, ValueError, KeyError):\\n            pass\\n        time.sleep(5)\\n    raise TimeoutError("vLLM was not ready before the startup deadline")\\n\\n\\nOUT_OF_MEMORY = ("CUDA out of memory", "OutOfMemoryError", "cudaErrorMemoryAllocation", "CUBLAS_STATUS_ALLOC_FAILED")\\n\\n\\ndef out_of_memory(log_path: Path, tail: int = 1 << 20) -> bool:\\n    """Whether the last `tail` bytes of a server log report a GPU allocation failure."""\\n    try:\\n        with log_path.open("rb") as stream:\\n            stream.seek(max(0, log_path.stat().st_size - tail))\\n            text = stream.read().decode(errors="replace")\\n    except OSError:\\n        return False\\n    return any(marker in text for marker in OUT_OF_MEMORY)\\n\\n\\ndef server_down(process: subprocess.Popen, config: dict, silence: float = 60.0, pause: float = 5.0) -> bool:\\n    """After a failed request: True when the server process has exited, its health request answers other than 200\\n    (vLLM\\\'s /health answers 503 once its engine died; SGLang\\\'s /health generates behind the running requests, so its\\n    /v1/models is asked instead), or it answers nothing for `silence` seconds (a briefly busy API loop must not cost\\n    every running trajectory). One 200 means the server is up and the failure was the request\\\'s own."""\\n    import httpx\\n\\n    path = "/v1/models" if config.get("engine") == "sglang" else "/health"\\n    end = time.time() + silence\\n    while True:\\n        if process.poll() is not None:\\n            return True\\n        try:\\n            return httpx.get(f"http://127.0.0.1:{config[\\\'port\\\']}{path}", timeout=10, trust_env=False).status_code != 200\\n        except httpx.HTTPError:\\n            if time.time() >= end:\\n                return True\\n        time.sleep(pause)\\n\\n\\ndef generation_metrics(config: dict) -> tuple[float, float] | None:\\n    """(tokens generated so far, requests running) from a vLLM server\\\'s Prometheus /metrics, summed over engines; None\\n    when it does not answer. A server without those series reads (0, 0), which never looks stalled."""\\n    import httpx\\n\\n    try:\\n        reply = httpx.get(f"http://127.0.0.1:{config[\\\'port\\\']}/metrics", timeout=10, trust_env=False)\\n        reply.raise_for_status()\\n    except httpx.HTTPError:\\n        return None\\n    total = {"vllm:generation_tokens_total": 0.0, "vllm:num_requests_running": 0.0}\\n    for line in reply.text.splitlines():\\n        name = line.split("{", 1)[0].split(" ", 1)[0]\\n        if name in total:\\n            total[name] += float(line.rsplit(" ", 1)[1])\\n    return total["vllm:generation_tokens_total"], total["vllm:num_requests_running"]\\n\\n\\ndef probe_generation(config: dict) -> None:\\n    """One short greedy generation (SGLang\\\'s /generate; /v1/completions on a vLLM server, such as h95\\\'s, which has no\\n    /generate), so a server that starts but cannot decode with its parallel layout or patches fails before the run\\n    depends on it."""\\n    import httpx\\n\\n    url = f"http://127.0.0.1:{config[\\\'port\\\']}"\\n    if config.get("engine") == "sglang":\\n        reply = httpx.post(url + "/generate", timeout=900, trust_env=False,\\n                           json={"text": "1 + 1 =", "sampling_params": {"max_new_tokens": 16, "temperature": 0}})\\n        reply.raise_for_status()\\n        text = reply.json().get("text")\\n    else:\\n        reply = httpx.post(url + "/v1/completions", timeout=900, trust_env=False,\\n                           json={"model": config["served_model"], "prompt": "1 + 1 =", "max_tokens": 16, "temperature": 0})\\n        reply.raise_for_status()\\n        text = (reply.json().get("choices") or [{}])[0].get("text")\\n    if not text:\\n        raise RuntimeError("the probe generation returned no text")\\n\\n\\nSMOKE_MARKERS = ("Traceback", "UNCOMMITTED", "PLE fatal")\\n\\n\\ndef smoke_check(config: dict, log_path: Path) -> dict:\\n    """config["smoke"] = {"requests": N, "prompt_tokens": T, "max_tokens": M, "timeout": S}: N concurrent completions of\\n    about T tokens (distinct prefixes) right after startup. A failed or late request, or a fatal marker in the server log,\\n    raises, so a check run of an unproven server patch ends within minutes instead of serving its whole window."""\\n    import httpx\\n\\n    smoke, started = config["smoke"], time.time()\\n    body = " ".join(str(i % 10) for i in range(smoke["prompt_tokens"]))\\n\\n    def one(index: int) -> int:\\n        reply = httpx.post(f"http://127.0.0.1:{config[\\\'port\\\']}/v1/completions", timeout=smoke["timeout"], trust_env=False,\\n                           json={"model": config["served_model"], "prompt": f"{index} {body}", "max_tokens": smoke["max_tokens"],\\n                                 "temperature": 0, "ignore_eos": True})\\n        reply.raise_for_status()\\n        return reply.json()["usage"]["completion_tokens"]\\n\\n    with ThreadPoolExecutor(smoke["requests"]) as pool:\\n        generated = list(pool.map(one, range(smoke["requests"])))\\n    found = [marker for marker in SMOKE_MARKERS if marker in log_path.read_text(errors="replace")]\\n    if found or min(generated) <= 0:\\n        raise RuntimeError(f"smoke check failed: generated {generated}, server log markers {found}")\\n    return {"requests": smoke["requests"], "generated": sum(generated), "seconds": round(time.time() - started, 1)}\\n\\n\\nIDENTITY_TURNS = ((0, 1536), (256, 512), (256, 512))  # (new prompt tokens, generated tokens) per turn\\n\\n\\ndef identity_conversations(config: dict, conversations: list[dict], log_path: Path, timeout: float) -> list[list[dict]]:\\n    """config["identity_check"]\\\'s greedy conversations, one request at a time (the same batches on every server): turn 1 is\\n    a conversation\\\'s "first" tokens, every later turn the previous prompt + its output + the next slice of "extra". Per\\n    turn: the output token ids, the prompt\\\'s cached tokens and the GDN park restores the server logged meanwhile."""\\n    import httpx\\n\\n    url = f"http://127.0.0.1:{config[\\\'port\\\']}/v1/completions"\\n    out = []\\n    for conversation in conversations:\\n        prompt, turns, used = list(conversation["first"]), [], 0\\n        for new, generate in IDENTITY_TURNS:\\n            prompt += conversation["extra"][used:used + new]\\n            used += new\\n            mark = log_path.stat().st_size\\n            started = time.time()\\n            reply = httpx.post(url, timeout=timeout, trust_env=False, json={\\n                "model": config["served_model"], "prompt": prompt, "max_tokens": generate, "temperature": 0.0,\\n                "return_token_ids": True, "ignore_eos": True})\\n            reply.raise_for_status()\\n            body = reply.json()\\n            tokens = body["choices"][0]["token_ids"]\\n            with log_path.open("rb") as stream:\\n                stream.seek(mark)\\n                restores = stream.read().decode(errors="replace").count("GDN park: restored")\\n            turns.append({"prompt": len(prompt), "token_ids": tokens, "restores": restores,\\n                          "cached": ((body.get("usage") or {}).get("prompt_tokens_details") or {}).get("cached_tokens"),\\n                          "seconds": round(time.time() - started, 2)})\\n            prompt += tokens\\n        out.append(turns)\\n    return out\\n\\n\\ndef identity_summary(stock: list[list[dict]], candidate: list[list[dict]]) -> dict:\\n    """PASS: every turn\\\'s output token ids equal on both servers and at least one restore on the candidate."""\\n    same = [[a["token_ids"] == b["token_ids"] for a, b in zip(sa, sb)] for sa, sb in zip(stock, candidate)]\\n    restores = sum(turn["restores"] for turns in candidate for turn in turns)\\n    return {"pass": all(all(row) for row in same) and restores > 0, "identical": same, "restores": restores,\\n            "cached": [[(a["cached"], b["cached"]) for a, b in zip(sa, sb)] for sa, sb in zip(stock, candidate)],\\n            "candidate_restores": [[turn["restores"] for turn in turns] for turns in candidate],\\n            "seconds": [[(a["seconds"], b["seconds"]) for a, b in zip(sa, sb)] for sa, sb in zip(stock, candidate)]}\\n\\n\\ndef identity_stock_phase(config: dict, command: list[str], environment: dict, output: Path, deadline: float) -> dict:\\n    """The stock half of config["identity_check"]: the serving with identity["stock_env"] over its env (ARC_VLLM_GDN_PARK 0)\\n    on a server of its own, the conversations, then the server stopped."""\\n    identity = config["identity_check"]\\n    conversations = json.loads(Path(identity["prompts"]).read_text())\\n    log_path = output / "server_identity_stock.log"\\n    started = time.time()\\n    with log_path.open("x") as log:\\n        server = subprocess.Popen(command, env={**environment, **identity["stock_env"]}, stdout=log, stderr=subprocess.STDOUT,\\n                                  start_new_session=True)\\n        try:\\n            wait_ready(server, config, min(deadline, time.time() + identity.get("startup_timeout", 2400)), log_path)\\n            turns = identity_conversations(config, conversations, log_path, identity.get("timeout", 900))\\n            return {"turns": turns, "ready_seconds": round(time.time() - started, 1), "conversations": conversations}\\n        except Exception as error:  # the check never costs the run: the candidate serves on\\n            return {"error": f"{type(error).__name__}: {error}"}\\n        finally:\\n            stop_group(server)\\n\\n\\ndef run(config: dict) -> dict:\\n    import httpx\\n    from openai import OpenAI\\n    from transformers import AutoTokenizer\\n\\n    from src.agent.executor import configure\\n\\n    configure(**config.get("executor", {}))  # {"sandbox": "bwrap", "memory_gb": G} for the free agent\\\'s program workers\\n    started = time.time()\\n    output = Path(config["output"])\\n    output.mkdir(parents=True, exist_ok=False)\\n    (output / "results").mkdir()\\n    tasks = json.loads(Path(config["challenges"]).read_text())\\n    if any("output" in test for task in tasks.values() for test in task["test"]):\\n        raise ValueError("Test labels must not reach the runtime")\\n    # config["trajectories"]: a fresh directory for free-mode trajectories (e.g. on a larger disk than the results)\\n    trajectories = output / "results"\\n    if config.get("trajectories"):\\n        trajectories = Path(config["trajectories"])\\n        trajectories.mkdir(parents=True)\\n    settings = config["evaluation"]\\n    free = settings.get("agent_mode") == "free"\\n    # settings["extra_attempts"] = K: K more jobs per task after the attempts, each another first-attempt sample (no\\n    # view), served by JobQueue only to tasks without a verified answer once every other job has been taken\\n    extra = settings.get("extra_attempts", 0)\\n    if type(extra) is not int or extra < 0:\\n        raise ValueError(f"extra_attempts is a non-negative int, got {extra!r}")\\n    jobs = direct_jobs_for(tasks, {"seed": settings["seed"], "direct_attempts": settings["attempts"] + extra,\\n                                   "second_view": settings.get("second_view", False), "job_unit": settings.get("job_unit", "output"),\\n                                   "view_every_attempt": settings.get("view_every_attempt", False)})\\n    for job in jobs:\\n        if job["attempt"] >= settings["attempts"]:\\n            job.pop("view", None)\\n            job["extra"] = True\\n    # settings["later_order"] (with second_priority): the later attempts\\\' groups in another order than LATER_ORDER\\\'s;\\n    # settings["later_shortest_first"]: within a group, the shortest first attempts\\\' later attempts first\\n    # settings["skip_filled"] (job_unit task): a task whose first attempt filled both slots of every output gets no later\\n    # attempt (JobQueue, filled_slots); the run\\\'s skipped_filled counts them\\n    if settings.get("skip_filled") and settings.get("job_unit") != "task":\\n        raise ValueError("skip_filled runs with job_unit task")\\n    # settings["doubt_rules"] (job_unit task): a verified first answer rule_doubt flags has its later attempt served as an\\n    # unverified one\\\'s (JobQueue); the run\\\'s doubted_rules lists them\\n    if settings.get("doubt_rules") and settings.get("job_unit") != "task":\\n        raise ValueError("doubt_rules runs with job_unit task")\\n    queue = JobQueue(jobs, priority=settings.get("second_priority", False), order=settings.get("later_order", LATER_ORDER),\\n                     shortest_first=settings.get("later_shortest_first", False),\\n                     skip_filled={task_id: len(task["test"]) for task_id, task in tasks.items()} if settings.get("skip_filled") else None,\\n                     doubt=tasks if settings.get("doubt_rules") else None)\\n    destination = Path(config["final_submission"])\\n    # the competition rerun: no placeholder file before serving, and a run whose server was never ready or that answered\\n    # no output ends in an error with no file, so that Kaggle reports a failed submission instead of scoring a\\n    # placeholder 0.00 (a run with any answered output keeps its file, unanswered outputs as placeholders)\\n    fail_loudly = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))\\n    if not fail_loudly:\\n        finalize(tasks, output / "results", destination, jobs if free else None, trajectories, slot_fill=bool(settings.get("slot_fill")))\\n    generation_deadline = config["deadline"] - config["cleanup_reserve"]\\n    state = {"status": "STARTING", "started_utc": datetime.now(timezone.utc).isoformat(), "jobs": len(jobs)}\\n    atomic_json(output / "status.json", state)\\n    # The GPUs are the ones the job sees (all four on Kaggle); a scheduler\\\'s CUDA_VISIBLE_DEVICES is left as it is.\\n    # config["env"]: extra variables for the server (e.g. the knobs of a serve script given as config["command"]).\\n    environment = {**os.environ, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "TOKENIZERS_PARALLELISM": "false",\\n                   "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "1", **config.get("env", {})}\\n    server, log, pool, stopping = None, None, None, threading.Event()\\n    try:\\n        if config.get("driver_linker", True):\\n            state["driver_linker"] = prepare_driver_linker(output / "cuda-driver-link", environment)\\n        if config.get("engine") == "sglang":\\n            state["cuda_home"] = sglang_cuda_home(environment, state.get("driver_linker", {}).get("link"))\\n        # config["fallback_server_args"]: server arguments to restart with when the server with config["server_args"]\\n        # fails to start or to answer probe_generation (an untested parallel layout must not cost the whole run);\\n        # config["fallback_env"] (fallback_levels): variables replaced for that start and every later restart (e.g. a\\n        # patch switched off), each level of a ladder tried in turn until one starts; a restart after a server that ran\\n        # out of GPU memory takes the next level (recover), with config["fallback_command"]\\n        log_path = output / "server.log"\\n        log = log_path.open("x")\\n        command = config.get("command") or server_command(config)\\n        state.update(model_fstype=model_fstype(config["model"]), ple_fstype=model_fstype(config["model"], ple=True))\\n        if config.get("link_probe"):\\n            state["link_probe"] = link_probe(environment)\\n            atomic_json(output / "status.json", state)\\n        if (config.get("local_model_copy") or {}).get("first"):  # local_model_copy_first: serve the local copy from the start\\n            state["copy_disk_free_gib"] = disk_free_gib(config["local_model_copy"]["copy"])\\n            first = first_start_model(config, min(generation_deadline, time.time() + config["local_model_copy"].get("wait", 5400)))\\n            state["first_start_model"] = {"model": str(first) if first else "source", "waited_seconds": round(time.time() - started, 1)}\\n            state["copy_mb_per_s"] = copy_rate(Path(config["local_model_copy"]["log"])) if config["local_model_copy"].get("log") else None\\n            state["copy"] = copy_outcome(config["local_model_copy"], first is not None)\\n            atomic_json(output / "status.json", state)\\n            if first:\\n                environment = {**environment, "MODEL": str(first)}\\n        # config["identity_check"] = {"prompts": path, "stock_env": {...}, "drop_file": path, "assumed_blocks_per_request": N,\\n        # "startup_timeout": s, "timeout": s}: before the run, the same greedy multi-turn conversations on a stock server\\n        # (identity_stock_phase) and then on this serving with drop_file present (vllm_gdn_park\\\'s ARC_VLLM_GDN_PARK_DROP: a\\n        # parked state loses its GPU copy at once, so every hit on a held state is a restore from the host); identity.json\\n        # records per turn the equality of the output token ids, the cached tokens and the restores; the file is removed\\n        # and the run goes on with the same server. A failed check is recorded, never raised.\\n        identity = config.get("identity_check")\\n        stock = None\\n        if identity:\\n            stock = identity_stock_phase(config, command, environment, output, generation_deadline)\\n            Path(identity["drop_file"]).touch()\\n            state["identity"] = {"stock_error": stock.get("error"), "stock_ready_seconds": stock.get("ready_seconds")}\\n            atomic_json(output / "status.json", state)\\n        primary_environment = environment  # the ladder\\\'s levels replace variables of this one\\n        server = subprocess.Popen(command, env=environment, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\\n        try:\\n            wait_ready(server, config, min(generation_deadline, time.time() + config["startup_timeout"]), log_path)\\n            if config.get("fallback_server_args") is not None:\\n                probe_generation(config)\\n        except (RuntimeError, TimeoutError, httpx.HTTPError) as error:\\n            if config.get("fallback_server_args") is None:\\n                raise\\n            state["fallback"] = {"error": f"{type(error).__name__}: {error}", "seconds": round(time.time() - started, 1)}\\n            atomic_json(output / "status.json", state)\\n            stop_group(server)\\n            log.close()\\n            config = {**config, "server_args": config["fallback_server_args"]}\\n            command = config.get("fallback_command") or server_command(config)\\n            levels = fallback_levels(config) or [{}]\\n            for level, extra in enumerate(levels, 1):  # a level that exits while it starts: the next one\\n                environment = {**primary_environment, **extra}\\n                log_path = output / ("server_fallback.log" if level == 1 else f"server_fallback{level}.log")\\n                log = log_path.open("x")\\n                server = subprocess.Popen(command, env=environment, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\\n                try:\\n                    wait_ready(server, config, generation_deadline, log_path)  # no server answers nothing: no time limit\\n                    state["fallback"]["level"] = level\\n                    break\\n                except RuntimeError as failure:\\n                    stop_group(server)\\n                    log.close()\\n                    state["fallback"].setdefault("failed_levels", []).append(f"{type(failure).__name__}: {failure}"[:300])\\n                    atomic_json(output / "status.json", state)\\n                    if level == len(levels):\\n                        raise\\n        if identity:\\n            result = {"assumed_blocks_per_request": identity.get("assumed_blocks_per_request")}\\n            try:\\n                if state.get("fallback"):\\n                    result["error"] = "the candidate serving did not start (fallback serving)"\\n                elif stock.get("error"):\\n                    result["error"] = f"stock phase: {stock[\\\'error\\\']}"\\n                else:\\n                    candidate = identity_conversations(config, stock["conversations"], log_path, identity.get("timeout", 900))\\n                    result.update(identity_summary(stock["turns"], candidate))\\n            except Exception as error:\\n                result["error"] = f"{type(error).__name__}: {error}"\\n            finally:\\n                Path(identity["drop_file"]).unlink(missing_ok=True)\\n            result.setdefault("pass", False)\\n            atomic_json(output / "identity.json", result)\\n            state["identity"].update({"pass": result["pass"], "error": result.get("error")})\\n            print(f"IDENTITY {\\\'PASS\\\' if result[\\\'pass\\\'] else \\\'FAIL\\\'}: {json.dumps({k: v for k, v in result.items() if k != \\\'seconds\\\'})}",\\n                  flush=True)\\n        if config.get("smoke"):\\n            state["smoke"] = smoke_check(config, log_path)\\n        watchdog = config.get("server_watchdog", True)\\n        state.update(status="GENERATING", ready_seconds=round(time.time() - started, 1), **({"restarts": []} if watchdog else {}),\\n                     memory_at_ready=meminfo_gib(),  # after the PLE table and the pinned host K/V: what the page cache kept\\n                     cgroup_at_ready=cgroup_memory_gib())\\n        atomic_json(output / "status.json", state)\\n        if free:\\n            from src.agent.protocol import load_native_tokenizer\\n            from src.agent.run import VLLMCompletions\\n\\n            tokenizer = load_native_tokenizer(config["model"])  # with the terminator ids the loop stops on\\n        else:\\n            tokenizer = AutoTokenizer.from_pretrained(config["model"], local_files_only=True)\\n        request = {key: settings[key] for key in SAMPLING}\\n        request.update({key: settings[key] for key in FREE if key in settings})\\n        transport = httpx.Client(trust_env=False, follow_redirects=False)\\n        client = OpenAI(base_url=f"http://127.0.0.1:{config[\\\'port\\\']}/v1", api_key="EMPTY", max_retries=0, http_client=transport)\\n        if config.get("engine") == "sglang":\\n            request.update(engine="sglang", generate_url=f"http://127.0.0.1:{config[\\\'port\\\']}/generate")\\n            client = transport\\n        elif config.get("engine") == "openai":  # a vLLM build served by config["command"]; the client applies the budget\\n            request.update(engine="openai", completions_url=f"http://127.0.0.1:{config[\\\'port\\\']}/v1/completions",\\n                           served_model=config["served_model"])\\n            client = transport\\n        if free:\\n            if config.get("engine") != "openai":\\n                raise ValueError("agent_mode free runs on the openai engine (the client applies the thinking budget)")\\n            # config["rollout_logprobs"]: each sampled token\\\'s log-prob in the trajectories (archive.rl.hfrl.dapo\\\'s rollouts)\\n            client = VLLMCompletions(transport, request["completions_url"], tokenizer, logprobs=bool(config.get("rollout_logprobs")))\\n\\n        # settings["later_attempts"]: sampling / free-mode settings for every attempt after the first (e.g. another tool or\\n        # other caps, so the two attempts of a task differ by more than the D8 view)\\n        # settings["verified_later_attempts"]: over those, for a later attempt of a task whose first attempt ended by itself\\n        # with a verified answer (JobQueue\\\'s VERIFIED, the attempts served last), e.g. a lower max_model_len: the context cap\\n        # at which the loop asks for the answer; settings["verified_later_max_first_tokens"] = N: only where that first\\n        # attempt generated at most N completion tokens (a longer one keeps later_attempts)\\n        later, verified_later = settings.get("later_attempts") or {}, settings.get("verified_later_attempts") or {}\\n        max_first = settings.get("verified_later_max_first_tokens")\\n        if max_first is not None and not (type(max_first) is int and max_first > 0 and verified_later):\\n            raise ValueError(f"verified_later_max_first_tokens is a positive int with verified_later_attempts, got {max_first!r}")\\n        if (set(later) | set(verified_later)) - set(SAMPLING) - set(FREE):\\n            raise ValueError(f"later_attempts and verified_later_attempts take {SAMPLING + FREE} keys, got "\\n                             f"{sorted(set(later) | set(verified_later))}")\\n        f3_only = ("context_brake", "budget_notice", "helpers", "permissive")  # src.agent.loop refuses them on another tool\\n        per_attempt = [settings, {**settings, **later}, {**settings, **later, **verified_later}]\\n        requests = [request, {**request, **later}, {**request, **later, **verified_later}]\\n        for index in (1, 2):\\n            if per_attempt[index].get("tool") != "f3":\\n                per_attempt[index], requests[index] = ({k: v for k, v in d.items() if k not in f3_only}\\n                                                       for d in (per_attempt[index], requests[index]))\\n        # settings["avoid_previous"] (in later_attempts; in settings for a run given config["previous_answers"]): the\\n        # attempt is shown an earlier answer it must not repeat (previous_answer, src.agent.prompt.AVOID_PREVIOUS); the\\n        # row\\\'s "previous" says whether one was shown\\n        if any(own.get("avoid_previous") for own in per_attempt) and (not free or settings.get("job_unit") != "task"):\\n            raise ValueError("avoid_previous runs in agent_mode free with job_unit task")\\n        # settings["previous_program"] (F4, with avoid_previous): the earlier answer comes with the first attempt\\\'s last\\n        # program and its check on the training pairs (src.agent.loop.previous_program, from its trajectory, or\\n        # config["previous_programs"][task_id] beside config["previous_answers"]); the row\\\'s "previous_program" says\\n        # whether one was shown\\n        if any(own.get("previous_program") and not own.get("avoid_previous") for own in per_attempt):\\n            raise ValueError("previous_program needs avoid_previous")\\n        # settings["previous_program_prompt"] "edges" (with previous_program): a previous program that passed every training\\n        # pair comes with src.agent.prompt.PREVIOUS_PROGRAM_EDGES (the test input\\\'s uncovered cases first)\\n        if any(own.get("previous_program_prompt", "edges") != "edges" or ("previous_program_prompt" in own and not own.get("previous_program"))\\n               for own in per_attempt):\\n            raise ValueError("previous_program_prompt is \\\\"edges\\\\" with previous_program")\\n        # settings["repeat_refuse"] (with avoid_previous, tool f3): a later attempt\\\'s submission that repeats the earlier\\n        # answer it was shown, every grid, is refused once outside the final round (src.agent.tools.submit_answer)\\n        if any(own.get("repeat_refuse") and not (own.get("avoid_previous") and own.get("tool") == "f3") for own in per_attempt):\\n            raise ValueError("repeat_refuse needs avoid_previous and tool f3")\\n        # settings["previous_program_slice"] (with previous_program): the program shown comes with the earlier calls\\\'\\n        # definitions it uses (src.agent.loop.defining_statements)\\n        if any(own.get("previous_program_slice") and not own.get("previous_program") for own in per_attempt):\\n            raise ValueError("previous_program_slice needs previous_program")\\n        # settings["second_hypothesis"] (first attempts, tool f3): a first answer from a program passing every training pair\\n        # continues in the same conversation for a second such program, at most second_hypothesis_tokens tokens\\n        # (src.agent.loop.second_hypothesis_turn), its differing output kept as the row\\\'s program_predictions; none starts\\n        # within second_hypothesis_guard seconds of the generation deadline\\n        if settings.get("second_hypothesis_prompt", "edges") != "edges" or ("second_hypothesis_prompt" in settings and not settings.get("second_hypothesis")):\\n            raise ValueError(f"second_hypothesis_prompt is \\\\"edges\\\\" with second_hypothesis, got {settings.get(\\\'second_hypothesis_prompt\\\')!r}")\\n        if settings.get("second_hypothesis") and not (free and settings.get("tool") == "f3"):\\n            raise ValueError("second_hypothesis runs in agent_mode free with tool f3")\\n        # settings["deadline_brake"] (seconds, agent_mode free): a round that starts within that long of the generation\\n        # deadline is the trajectory\\\'s final round (src.agent.loop.start_round\\\'s force), so it answers instead of being\\n        # abandoned at the deadline; the row\\\'s deadline_brake names the round\\n        if settings.get("deadline_brake") and not free:\\n            raise ValueError("deadline_brake runs in agent_mode free")\\n        # settings["round_clamp"] (seconds, agent_mode free): every round generates at most what one request can before\\n        # the generation deadline less that margin, at the rate measured when its job started (src.agent.loop.round_clamp);\\n        # a round left with only the answer room is the final round, recorded as the brake is\\n        # settings["doubt_hint"] (tool f3): the reply to a program passing every training pair names each invariant its\\n        # test outputs break that every training pair keeps (src.agent.tools.invariant_hints), asking for a check\\n        for own in (settings, {**settings, **later}, {**settings, **later, **verified_later}):\\n            if own.get("doubt_hint") and not (free and own.get("tool") == "f3"):\\n                raise ValueError("doubt_hint runs in agent_mode free with tool f3")\\n            # settings["scipy_ndimage"] (tool f3): the executor also predefines and imports scipy.ndimage, and the\\n            # system prompt names it among the libraries\\n            if own.get("scipy_ndimage") and not (free and own.get("tool") == "f3"):\\n                raise ValueError("scipy_ndimage runs in agent_mode free with tool f3")\\n        if settings.get("round_clamp") and not free:\\n            raise ValueError("round_clamp runs in agent_mode free")\\n        for key, low in (("second_hypothesis_tokens", 1), ("second_hypothesis_guard", 0), ("deadline_brake", 1), ("round_clamp", 1)):\\n            if key in settings and not (type(settings[key]) is int and settings[key] >= low):\\n                raise ValueError(f"{key} is an int of at least {low}, got {settings[key]!r}")\\n        if (check := scipy_ndimage_check(requests)) is not None:\\n            state["scipy_ndimage"] = check\\n            print(f"scipy_ndimage: {check}", flush=True)\\n        first_keys = {(job["task_id"], job["test_index"]): job["key"] for job in jobs if job["attempt"] == 0}\\n        viewed = {job["key"] for job in jobs if job.get("view")}\\n        given = json.loads(Path(config["previous_answers"]).read_text()) if config.get("previous_answers") else None\\n        given_programs = json.loads(Path(config["previous_programs"]).read_text()) if config.get("previous_programs") else None\\n        # config["resume_turns"] ({task_id: {"turns", "thinking_tokens", "forced"}}, an earlier run\\\'s first attempts that\\n        # ended with a verified answer): a first attempt of such a task starts at that answer and runs only the second\\n        # hypothesis (src.agent.loop.resume_rounds), the earlier attempt\\\'s answer kept as its own\\n        resume = json.loads(Path(config["resume_turns"]).read_text()) if config.get("resume_turns") else {}\\n        if resume and not settings.get("second_hypothesis"):\\n            raise ValueError("resume_turns runs the second hypothesis: set second_hypothesis")\\n        restart_lock, done, refused = threading.Lock(), [], False\\n\\n        def recover(seen: int, error: str, stalled: bool = False) -> dict | None:\\n            """config["server_watchdog"] (default on), after an ERROR row of an attempt begun after `seen` restarts: the\\n            restart that followed it (its state["restarts"] record), made here when none has and the server is down\\n            (server_down) or `stalled` (stall_watch: up but not generating): the old process group stopped, the run\\\'s\\n            command (the fallback\\\'s once it switched) started again and waited for. One thread restarts while the others\\n            wait on the lock and find it done. None when the server is up (the failure was the request\\\'s own), the\\n            restart fails or the run is ending, or it is refused: config["server_restarts"] attempts made (default no\\n            limit). A start waits until the generation deadline, since without a server nothing is answered and a slow\\n            start costs nothing; the run\\\'s finally stops it there and finalizes. config["restart_prefetch"] (a command;\\n            the restart\\\'s log path appended) runs beside it, so that its weight reads find the page cache as the first\\n            start\\\'s do after the notebook\\\'s prefetch. A start that fails (the server exits) counts as an attempt, and\\n            the next ERROR tries again. A server that ended for another reason than GPU memory restarts as it was; a\\n            restart after a log that reports a GPU allocation failure takes the next level of config["fallback_env"]\\n            (fallback_levels, with config["fallback_command"]; status.json "fallback" level), and stays at the last.\\n            Refused once, the run goes on as without the watchdog."""\\n            nonlocal server, refused, command, environment, log_path\\n            with restart_lock:\\n                if len(done) > seen:\\n                    return done[seen]\\n                if refused or stopping.is_set() or not (stalled or server_down(server, config)):\\n                    return None\\n                at, restarts = time.time(), state["restarts"]\\n                reason = "generation stalled" if stalled else "no health answer" if server.poll() is None else f"exit code {server.returncode}"\\n                record = {"at_seconds": round(at - started, 1), "reason": reason, "error": error[:300],\\n                          "in_flight": queue.taken - len(statuses)}\\n                spent = config.get("server_restarts") is not None and len(restarts) >= config["server_restarts"]\\n                restarts.append(record)  # before the wait: a start still waiting when the run finalizes is in status.json\\n                if spent:\\n                    refused = True\\n                    record["refused"] = "restart budget spent"\\n                else:\\n                    try:\\n                        stop_group(server)\\n                        for _ in range(120):  # its engine and workers must let go of GPU and pinned memory first\\n                            try:\\n                                os.killpg(server.pid, 0)\\n                            except ProcessLookupError:\\n                                break\\n                            time.sleep(1)\\n                        restart_log = output / f"server_restart{len(restarts)}.log"\\n                        levels, level = fallback_levels(config), (state.get("fallback") or {}).get("level", 0)\\n                        if level < len(levels) and out_of_memory(log_path):\\n                            # a serving that ran out of GPU memory restarts one level down (e.g. a smaller KV pool)\\n                            environment = {**primary_environment, **levels[level]}\\n                            command = config.get("fallback_command") or command\\n                            state["fallback"] = {**(state.get("fallback") or {}), "error": "the server ran out of GPU memory",\\n                                                 "at_seconds": record["at_seconds"], "level": level + 1}\\n                            record.update(fallback=True, fallback_level=level + 1)\\n                        log_path = restart_log\\n                        if config.get("restart_prefetch"):\\n                            subprocess.Popen([*config["restart_prefetch"], str(output / f"prefetch_restart{len(restarts)}.log")],\\n                                             stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, start_new_session=True)\\n                            record["prefetch"] = True\\n                        local = local_model(config)  # config["local_model_copy"]: the prefetch\\\'s copy when complete\\n                        record["model"] = str(local) if local else "source"\\n                        with restart_log.open("x") as handle:\\n                            server = subprocess.Popen(command, env={**environment, **({"MODEL": str(local)} if local else {})},\\n                                                      stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)\\n                        if stopping.is_set():  # the run\\\'s finally may have stopped the old process only\\n                            stop_group(server)\\n                            return None\\n                        wait_ready(server, config, generation_deadline, restart_log)\\n                        record.update(ready_seconds=round(time.time() - at, 1), requeued=[], deferred=[])\\n                        done.append(record)\\n                    except Exception as failure:  # counts as an attempt; the next ERROR tries again\\n                        stop_group(server)\\n                        record["failed"] = f"{type(failure).__name__}: {failure}"\\n                    record["wait_seconds"] = round(time.time() - at, 1)  # stop to ready or failure, for timing_code\\n                print(json.dumps({"server_restart": {key: value for key, value in record.items() if key not in ("requeued", "deferred")}}), flush=True)\\n                return done[seen] if len(done) > seen and not stopping.is_set() else None\\n\\n        def stall_watch(limit: float, poll: float) -> None:\\n            """config["stall_restart"] seconds (off by default; vLLM servers): a server whose generated-token count has\\n            not moved for that long while it runs requests, or whose /metrics has not answered for that long, is\\n            restarted through recover as a dead one would be (a hung engine keeps answering /health, and its requests\\n            wait until the generation deadline, so no ERROR row ever calls recover). The requests it held then fail and\\n            run again from the start like those of a dead server. state["stall_metrics"] keeps the last reading (tokens\\n            generated, requests running), so a run shows that the series were found."""\\n            progress, last = time.time(), None\\n            while not stopping.wait(poll) and time.time() < generation_deadline:\\n                seen, metrics = len(done), generation_metrics(config)\\n                state["stall_metrics"] = metrics\\n                if metrics is not None and (metrics[1] == 0 or metrics[0] != last):\\n                    progress, last = time.time(), metrics[0]\\n                elif time.time() - progress >= limit:\\n                    running = "" if metrics is None else f" with {metrics[1]:.0f} running"\\n                    recover(seen, f"no generation for {time.time() - progress:.0f} s{running}", stalled=True)\\n                    progress, last = time.time(), None\\n\\n        def answer(job):\\n            """A request that fails (ERROR: an HTTP or server error, not a missing grid) is retried once; the retry\\n            overwrites the first result, which is kept as <key>.error.json. With settings["deadline_budget"] (the\\n            smallest thinking budget worth starting), deadline_budget shortens requests that would not finish. A failure\\n            the server was restarted after (recover) is not recorded and does not use the retry: the job runs again from\\n            the start with the same seeds (a free-mode trajectory from round 0: resuming would have to rebuild the\\n            messages, thinking count and prefill of the lost round from its files), its died row and trajectory kept as\\n            <key>.died.json and <key>.died (then .died-b, .died-c, ...); a job failing with a second restart goes back\\n            behind every waiting job (queue.defer; None returned, nothing recorded), so a request that kills the server\\n            costs the jobs still waiting nothing, and the restart budget bounds how often it can."""\\n            status, retry, deaths, aside = "SKIPPED", False, 0, job.pop("aside", None)\\n            while True:\\n                if time.time() >= generation_deadline:\\n                    break\\n                index = attempt_settings(job, queue.rank, bool(verified_later), queue.tokens, max_first)\\n                own = per_attempt[index]\\n                budget, answer_room = own["thinking_token_budget"], own["max_tokens"] - own["thinking_token_budget"]\\n                if settings.get("deadline_budget"):\\n                    budget = deadline_budget(queue.rate(), generation_deadline - time.time(), budget, answer_room, settings["deadline_budget"])\\n                    if budget is None:\\n                        break\\n                if aside:\\n                    name, n = aside, 0\\n                    while aside == "died" and ((output / "results" / f"{job[\\\'key\\\']}.{name}.json").exists()\\n                                               or (trajectories / f"{job[\\\'key\\\']}.{name}").exists()):\\n                        n += 1\\n                        name = f"died-{chr(97 + n)}"  # no trailing digit: finalize\\\'s d[0-9]*[0-9].json must not match\\n                    (output / "results" / f"{job[\\\'key\\\']}.json").rename(output / "results" / f"{job[\\\'key\\\']}.{name}.json")\\n                    if aside == "died" and (trajectories / job["key"]).exists():\\n                        (trajectories / job["key"]).rename(trajectories / f"{job[\\\'key\\\']}.{name}")\\n                    aside = None\\n                seen = len(done)\\n                sampling = {**requests[index], "thinking_token_budget": budget, "max_tokens": budget + answer_room,\\n                            "request_timeout": max(1.0, generation_deadline - time.time())}\\n                show_program = sampling.pop("previous_program", False)\\n                slice_program = sampling.pop("previous_program_slice", False)\\n                guard = sampling.pop("second_hypothesis_guard", SECOND_HYPOTHESIS_GUARD)\\n                if sampling.get("deadline_brake"):\\n                    sampling["deadline_brake_at"] = generation_deadline - sampling.pop("deadline_brake")\\n                if sampling.get("round_clamp"):\\n                    margin, rate = sampling.pop("round_clamp"), queue.rate()\\n                    if rate:  # none before 8 finished requests: those early jobs run unclamped\\n                        sampling.update(round_clamp_at=generation_deadline - margin, round_clamp_rate=rate)\\n                if sampling.pop("second_hypothesis", False) and job["attempt"] == 0:\\n                    sampling.update(second_hypothesis=True, second_hypothesis_until=generation_deadline - guard)\\n                    if job["task_id"] in resume and job["test_index"] is None:\\n                        sampling["resume"] = resume[job["task_id"]]\\n                if sampling.pop("avoid_previous", False):\\n                    previous = previous_answer(job, first_keys, output / "results", given)\\n                    job["previous"] = previous is not None\\n                    if previous is not None:\\n                        sampling["previous_answer"] = previous\\n                    if show_program:\\n                        try:\\n                            program = earlier_program(job, first_keys, trajectories, tasks[job["task_id"]], given_programs, viewed,\\n                                                      slice_program)\\n                        except Exception:  # an unreadable first attempt: the job runs as with avoid_previous alone\\n                            program = None\\n                        job["previous_program"] = previous is not None and program is not None\\n                        if job["previous_program"]:\\n                            sampling["previous_program"] = program\\n                if free:\\n                    result = free_answer(job, tasks[job["task_id"]], output / "results", config["served_model"], sampling,\\n                                         tokenizer, client, trajectories)\\n                else:\\n                    result = direct_answer(job, tasks[job["task_id"]], output / "results", config["served_model"], sampling,\\n                                           tokenizer, client)\\n                status = result["status"]\\n                restart = recover(seen, result.get("error", "")) if status == "ERROR" and watchdog else None\\n                if restart:\\n                    deaths += 1\\n                    restart["requeued" if deaths == 1 else "deferred"].append(job["key"])\\n                    if deaths == 1:\\n                        aside = "died"\\n                        continue\\n                    queue.defer({**job, "aside": "died"})\\n                    return None\\n                queue.record(job, result, budget)\\n                if status != "ERROR" or retry:\\n                    break\\n                retry, aside = True, "error"\\n            return status\\n\\n        statuses = []\\n\\n        def worker():\\n            while time.time() < generation_deadline and (job := queue.take()) is not None:\\n                if (status := answer(job)) is not None:\\n                    statuses.append(status)\\n                    queue.finish(job)\\n\\n        # ponytail: a thread per in-flight request; the server\\\'s max_num_seqs bounds what runs at once\\n        pool = ThreadPoolExecutor(max_workers=settings["concurrency"])\\n        generation_started = time.time()\\n        if watchdog and config.get("stall_restart") and config.get("engine") != "sglang":\\n            threading.Thread(target=stall_watch, args=(config["stall_restart"], config.get("stall_poll", 30.0)), daemon=True).start()\\n        futures = [pool.submit(worker) for _ in range(settings["concurrency"])]\\n        wait(futures, timeout=max(0.0, generation_deadline - time.time()))\\n        finished = list(statuses)\\n        # timing_code\\\'s C and Z: the run\\\'s generation rate, and the first attempts without a result (abandoned or never\\n        # started) at the end of generation\\n        state.update(generation_tok_s=round(queue.generated / max(1.0, time.time() - generation_started), 1),\\n                     first_unfinished=sum(1 for job in jobs if job["attempt"] == 0) - len(queue.first_done),\\n                     later_fraction=round((len(queue.later_done) + queue.skipped)\\n                                          / max(1, sum(1 for job in jobs if job["attempt"] > 0 and not job.get("extra"))), 3))\\n        state.update(status="FINALIZING", finished=len(finished), abandoned=queue.taken - len(finished),\\n                     not_started=len(jobs) - queue.taken, status_counts={name: finished.count(name) for name in sorted(set(finished))},\\n                     **({"skipped_filled": queue.skipped} if settings.get("skip_filled") else {}),\\n                     **({"doubted_rules": sorted(queue.doubted)} if settings.get("doubt_rules") else {}),\\n                     **({"extra_started": queue.extra_taken} if extra else {}), cgroup_at_end=cgroup_memory_gib())\\n    except BaseException as error:\\n        state.update(status="ERROR", error=f"{type(error).__name__}: {error}")\\n    finally:\\n        stopping.set()  # before `server` is read: a restart in progress then stops what it started (recover)\\n        if pool is not None:\\n            pool.shutdown(wait=False, cancel_futures=True)\\n        if server is not None:\\n            stop_group(server)\\n        if log is not None:\\n            log.close()\\n        try:\\n            state["timing"] = timing_code(state)\\n            if fail_loudly and "ready_seconds" not in state:\\n                failure = "the server was never ready"\\n            else:\\n                state["finalization"] = finalize(tasks, output / "results", destination, jobs if free else None, trajectories,\\n                                                 size_code=state["timing"]["code"], slot_fill=bool(settings.get("slot_fill")))\\n                failure = "no output has a grid" if fail_loudly and not state["finalization"]["outputs_with_grid"] else None\\n            if failure:\\n                destination.unlink(missing_ok=True)\\n                state.update(status="ERROR", failed_loudly=failure)\\n            elif state["status"] != "ERROR":\\n                state["status"] = "COMPLETE"\\n        except Exception as error:\\n            state.update(status="ERROR", finalization_error=f"{type(error).__name__}: {error}")\\n        state["released_utc"] = datetime.now(timezone.utc).isoformat()\\n        atomic_json(output / "status.json", state)\\n    return state\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    parser.add_argument("--config", type=Path, required=True)\\n    report = run(json.loads(parser.parse_args().config.read_text()))\\n    print(json.dumps(report), flush=True)\\n    os._exit(1 if report["status"] == "ERROR" else 0)  # abandoned request threads must not hold the interpreter open\\n\', \'src/utils/cuda_linker.py\': \'"""Expose the loaded CUDA driver to JIT linkers without replacing runtime libraries."""\\nimport ctypes\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport signal\\nimport subprocess\\n\\n\\ndef mapped_driver(maps: str) -> Path:\\n    paths = set()\\n    for line in maps.splitlines():\\n        fields = line.split(maxsplit=5)\\n        if len(fields) == 6 and fields[5].startswith(\\\'/\\\'):\\n            path = Path(fields[5])\\n            if path.name.startswith(\\\'libcuda.so.\\\') and path.is_file():\\n                paths.add(path.resolve())\\n    if len(paths) != 1:\\n        raise RuntimeError(f\\\'Expected one loaded CUDA driver, found {len(paths)}\\\')\\n    driver = paths.pop()\\n    if \\\'stubs\\\' in driver.parts:\\n        raise RuntimeError(\\\'The loaded CUDA library is a stub\\\')\\n    return driver\\n\\n\\ndef compile_probe(command, source, environment, directory: Path, timeout: float = 60):\\n    process = subprocess.Popen(command, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,\\n        env={**environment, \\\'TMPDIR\\\': str(directory)}, text=True, start_new_session=True)\\n    stdout, stderr = \\\'\\\', \\\'\\\'\\n    try:\\n        stdout, stderr = process.communicate(source, timeout=timeout)\\n        if process.returncode:\\n            raise RuntimeError(f\\\'CUDA driver link probe failed: {stderr}\\\')\\n        return stdout, stderr\\n    except subprocess.TimeoutExpired as error:\\n        stdout = error.stdout.decode(errors=\\\'replace\\\') if isinstance(error.stdout, bytes) else error.stdout or \\\'\\\'\\n        stderr = error.stderr.decode(errors=\\\'replace\\\') if isinstance(error.stderr, bytes) else error.stderr or \\\'\\\'\\n        raise RuntimeError(\\\'CUDA driver link probe exceeded its time limit\\\') from error\\n    finally:\\n        try:\\n            os.killpg(process.pid, signal.SIGKILL)\\n        except ProcessLookupError:\\n            pass\\n        process.wait(timeout=5)\\n        for stream in (process.stdin, process.stdout, process.stderr):\\n            if stream is not None:\\n                stream.close()\\n        (directory / \\\'compiler.log\\\').write_text(stdout + stderr)\\n\\n\\ndef prepare_driver_linker(directory: Path, environment: dict[str, str]) -> dict:\\n    driver = ctypes.CDLL(\\\'libcuda.so.1\\\')\\n    version = ctypes.c_int()\\n    if driver.cuDriverGetVersion(ctypes.byref(version)) != 0 or version.value <= 0:\\n        raise RuntimeError(\\\'CUDA driver version query failed\\\')\\n    loaded = mapped_driver(Path(\\\'/proc/self/maps\\\').read_text())\\n    directory.mkdir(parents=True, exist_ok=False)\\n    directory = directory.resolve()\\n    link = directory / \\\'libcuda.so\\\'\\n    link.symlink_to(loaded)\\n    previous = environment.get(\\\'LIBRARY_PATH\\\', \\\'\\\')\\n    environment[\\\'LIBRARY_PATH\\\'] = str(directory) + (os.pathsep + previous if previous else \\\'\\\')\\n    # Force a real driver symbol dependency so a successful link is meaningful.\\n    source = \\\'extern "C" int cuDriverGetVersion(int*); extern "C" int query(int* v) { return cuDriverGetVersion(v); }\\\\n\\\'\\n    probe_path = directory / \\\'driver_link_check.so\\\'\\n    command = [\\\'c++\\\', \\\'-shared\\\', \\\'-fPIC\\\', \\\'-x\\\', \\\'c++\\\', \\\'-\\\', \\\'-Wl,--no-as-needed\\\', \\\'-Wl,-t\\\', \\\'-lcuda\\\', \\\'-o\\\', str(probe_path)]\\n    stdout, stderr = compile_probe(command, source, environment, directory)\\n    linked_paths = {Path(line.strip()).resolve() for line in (stdout + stderr).splitlines()\\n                    if line.strip().startswith(\\\'/\\\') and line.strip().endswith(\\\'/libcuda.so\\\')}\\n    if linked_paths != {loaded}:\\n        raise RuntimeError(\\\'CUDA link probe did not use the loaded driver\\\')\\n    probe = ctypes.CDLL(str(probe_path))\\n    linked_version = ctypes.c_int()\\n    if probe.query(ctypes.byref(linked_version)) != 0 or linked_version.value != version.value:\\n        raise RuntimeError(\\\'Linked CUDA driver differs from the loaded driver\\\')\\n    report = {\\\'status\\\': \\\'LOADED_CUDA_DRIVER_LINK_VERIFIED\\\', \\\'driver\\\': str(loaded), \\\'driver_version\\\': version.value,\\n              \\\'link\\\': str(link), \\\'library_path\\\': environment[\\\'LIBRARY_PATH\\\'], \\\'command\\\': command,\\n              \\\'runtime_library_path_changed\\\': False}\\n    (directory / \\\'report.json\\\').write_text(json.dumps(report, indent=2) + \\\'\\\\n\\\')\\n    return report\\n\', \'src/qwen/vllm_patch/sitecustomize.py\': \'"""Put this directory first on PYTHONPATH of a vLLM server to change how vLLM 0.29 groups a hybrid model\\\'s KV cache\\nlayers when a speculative drafter brings sliding-window layers (DFlash2, DSpark): Python imports sitecustomize in every\\nprocess it starts (API server, engine core, workers), and the hook below patches vllm.v1.core.kv_cache_utils right\\nafter it loads.\\n\\nvLLM gives every KV cache group the same number of layers and takes that number from the smallest layer bucket. Qwen3.8\\nhas 16 full-attention and 48 Gated DeltaNet layers; DFlash2 adds 5 sliding-window layers, so the group size drops from\\n16 to 5 and the 16 full-attention layers are padded to 20 ("Add 4 padding layers, may waste at most 25.00% KV cache\\nmemory"): every cached token pays for 4 empty layers. Here, when sliding-window buckets exist, the group size is the\\ngreatest common divisor of the other buckets (16 for Qwen3.8), so only the sliding-window group is padded, and it holds\\nonly its window (about two blocks per request). Without sliding-window buckets vLLM\\\'s own grouping is kept.\\n\\nWith ARC_SGLANG_GPTQ_MARLIN_MOE=1 (an SGLang server) three faults of SGLang 0.5.21\\\'s GPTQ Marlin MoE, the path a\\nbfloat16 W4A16 model such as Qwen3.8-Flash-Next takes under expert parallelism (whole 640-wide experts), are fixed:\\n- the scales are created in the model dtype: SGLang creates them as float16 whatever the model dtype, and its Marlin MoE\\n  asserts that the scales match the activations, so the model cannot start. The checkpoint\\\'s float16 scales are cast to\\n  bfloat16 while loading; against the Triton path (float16 scale, float32 product, rounded to bfloat16) 12% of the\\n  dequantized expert weights move by one or two bfloat16 steps (at most 0.78% relative, layer 0, 6 tensors);\\n- the dispatcher\\\'s expert map reaches the kernel, as in the compressed-tensors and MXFP4 Marlin MoE: under expert\\n  parallelism the dispatcher turns other ranks\\\' experts into id -1, which Marlin skips only when it is told it runs EP,\\n  so without the map it reads expert -1\\\'s weights (an illegal address, reported by the next launch check:\\n  "moe_sum_reduce CUDA kernel (small-token) launch failed" while capturing the 128-token verify graph, H100 EP2);\\n- the weights replaced by the Marlin repack are freed layer by layer: the repacked tensors are plain tensors, so they are\\n  registered as new parameters and the old ones lived until a later garbage collection, 28.5 GB per rank at EP2 (one\\n  more copy of the rank\\\'s experts) when loading ended.\\n\\nWith ARC_SGLANG_SAVE_MEMORY=1 (an SGLang server; outputs unchanged) two memory leaks of Flash-Next under SGLang 0.5.21\\nare closed: --language-model-only is allowed for Qwen4ExpForConditionalGeneration (its loader already skips the vision\\nweights, but the allowlist rejects the flag, so the unused vision tower is built; SGLang PR #37793), and the MTP draft\\\'s\\nown embed_tokens / lm_head copies are replaced by the target\\\'s before the KV pools are sized rather than after (SGLang\\nissue #42510: the scheduler sizes the target pool, allocates the draft pool, and only then shares the heads, so the pool\\nis sized around ~0.6 GiB per rank of soon-freed weights). Pass --disable-prefill-cuda-graph with --language-model-only: SGLang keeps its\\nbreakable prefill CUDA graph off only for multimodal models, and capturing it fails with --ple-offload-embedding\\n("Cannot copy between CPU and CUDA tensors during CUDA graph capture"), so the flag keeps the multimodal behaviour.\\n\\nWith ARC_SGLANG_QSA_TRITON=1 (an SGLang server on L4) Flash-Next\\\'s QSA decode attention runs SGLang\\\'s own Triton packed\\nsparse-GQA kernel (sparse_gqa_packed_decode_triton, its ROCm path) instead of FlashAttention 2: on sm89 the backend looks\\nfor FA2 and otherwise falls back to FA4 CuTe, which fails at head_dim 256 there (SGLang PR #36968), and the Kaggle\\nwheelhouse has no FA2. Same inputs (each row\\\'s selected KV packed back to back); on the local RTX 4070 at the L4 TP4 shape\\n(6 query heads over 1 KV head, head_dim 256, 2,048 selected entries) it matches a float32 reference to 1e-3 and reads KV\\nat ~240 GB/s (docs/design/qsa_bench.py).\\n\\nWith ARC_SGLANG_QSA_HISPARSE=1 (an SGLang Flash-Next server; needs --max-running-requests) the target\\\'s QSA layers keep\\ntheir K/V in registered host memory, the GPU keeps only the QSA index, the MTP layer and a per-request hot buffer of\\nARC_SGLANG_QSA_HOT_TOKENS (default 4096) entries per layer filled by HiSparse\\\'s LRU swap-in kernel, so the KV pool grows\\n~4x on an L4 with bit-identical attention inputs (qsa_hisparse.py; design in\\ndocs/design/hisparse_qsa_design.md).\\n\\nWith ARC_VLLM_QSA_HOST_KV=1 (a vLLM Flash-Next server on halt95\\\'s Merlin engine, h95) the QSA K/V of the target and\\nthe MTP drafter moves to registered host memory as its own host-resident KV cache group with a private block pool;\\nthe GPU pool keeps the QSA index, the key ring, the GDN and PLE states and is sized after reserving per-request hot\\nbuffers that vLLM\\\'s HiSparse kernels fill, so the pool holds several times the tokens with bit-identical attention\\ninputs (vllm_qsa_host_kv.py; design in docs/design/hisparse_h95_design.md).\\n\\nWith ARC_VLLM_RETAIN_LAST_STATE=1 (h95, engine core only) a finished or preempted request leaves the newest decode-time\\nGated DeltaNet state the next turn can resume at (the newest 4096 multiple at or below its last 64-token key minus one\\nkey), plus that decode-tail attention key, instead of only its prompt-tail state; superseded decode states lose their\\nhash before they are freed, so they never age in the LRU as with --prefix-cache-retention-interval (vllm_last_state.py;\\ndesign in docs/design/last_state_retention_design.md). Composes with\\nARC_VLLM_QSA_HOST_KV.\\n\\nWith ARC_VLLM_FP8_DENSE=1 (h95; =int8g128 for INT8 weights with a scale per group of 128) the target\\\'s bfloat16 non-expert linears (GDN and QSA projections, shared experts, PLE\\nkey/value, hyper-connection mixers) are quantized after loading to FP8 E4M3 with per-output-channel scales and run vLLM\\\'s\\nFP8 Marlin GEMM (W8A16), before profiling, so the freed memory goes to the KV cache; the router, the shared expert gate,\\nthe QSA indexer and the MTP drafter stay bfloat16 (vllm_fp8_dense.py). ARC_VLLM_FP8_DENSE_DRAFTER=1 converts the MTP\\ndrafter\\\'s dense linears too, in the same mode. With ARC_VLLM_FP8_DENSE unset, ARC_VLLM_FP8_DENSE_DRAFTER=<mode> converts the\\ndrafter alone (lossless: the target verifies every draft), and ARC_VLLM_FP8_DENSE=0 still turns both off, as a bfloat16\\nfallback sets it.\\n\\nWith ARC_VLLM_DFLASH2=1 (h95) Flash-Next becomes the target of a DFlash2 drafter (speculative method dflash) under\\npipeline parallelism: its model captures the drafter\\\'s target-layer features (each target layer\\\'s completed output,\\naveraged over the hyper-connection streams) and relays the first stage\\\'s captures to the last one through Merlin\\\'s\\nauxiliary-state slots, and the model classes take the SupportsEagle3 interface (vllm_dflash2.py). Off, the forward is\\nMerlin\\\'s own.\\n\\nWith ARC_SGLANG_REAP_KEEP=<keep.json> (an SGLang server; needs the repo root on its path for src.qwen.reap) Flash-Next\\nloads a full checkpoint as its REAP-pruned version: load_weights drops the pruned experts, renumbers the kept ones and\\nslices the routers exactly as src.qwen.reap.prune writes the pruned files, so a public full copy plus the keep list (and\\na config.json with the kept expert count) replaces uploading the pruned weights. The MTP drafter is pruned with the same\\nlist (its mtp.layers.0 entry): under expert parallelism SGLang\\\'s expert-location table is shared with the target.\\nARC_VLLM_REAP_KEEP=<keep.json> does the same on a vLLM server (the halt95 Merlin engine; vllm_reap.py names its classes).\\nARC_VLLM_READ_FILTER=1 makes a vLLM GPU worker skip, before reading, the checkpoint tensors it would drop (the other\\npipeline stage, the PLE table, pruned and other-EP-rank experts) and the PLE offload worker read only the PLE tensors;\\nARC_VLLM_WEIGHT_DIGEST=1 logs a digest of the loaded weights and the bytes read, to compare starts (vllm_read_filter.py).\\n\\nWith ARC_SGLANG_ROUTER_STATS=<file> (an SGLang server run for REAP expert pruning, CUDA graphs off so the hooks run)\\nevery MoE TopK accumulates, per expert, how often it is routed, the sum of its routing weights, and the sum of the\\nnorms of its routing-weighted outputs g_j(x) f_j(x) (REAP\\\'s saliency numerator), read from the Triton MoE\\\'s\\n[tokens, top_k, hidden] buffer just before the experts are summed; under TP each rank sees its partial f_j(x). TopK\\nmodules are numbered in construction order (the target\\\'s layers first, then the MTP layer), and every TP rank writes\\n{index: (layer_id, tensor[3, experts])} to <file>.rank<R> every ROUTER_STATS_EVERY calls of the first TopK.\\n\\nWith ARC_VLLM_PLE_LOADED_FLAG=<path> (h95/serve_l4.sh, a fresh path per start) Merlin\\\'s PLE offload worker (the process\\nholding Flash-Next\\\'s per-layer embedding table in host memory) writes <path> once its table is loaded, and every GPU\\nworker waits for it (at most PLE_WAIT_SECONDS, then goes on as before, logged) before profiling its memory, so no rank\\nallocates its KV cache or pins host memory while the table is still being read. The worker streams the whole checkpoint\\nthrough its prefix filter (116 GiB for a 48 GiB table, from the network filesystem on Kaggle); when the other ranks pinned\\ntheir host KV pools before it had finished, it stopped making progress and the rank waiting for it never became ready\\n(Kaggle, 10-20 and 10-21: 14 and 24 of 25 shards read; ready in ~600 s when it finished first). The full read stays: the\\nGPU workers read their expert slices from the page cache it fills (reading only the PLE tensors made their shards take\\n120 s instead of 14 s on Kaggle).\\n"""\\nimport importlib.abc\\nimport importlib.util\\nimport json\\nimport math\\nimport os\\nimport sys\\nimport time\\nfrom collections import defaultdict\\n\\n\\ndef group_layers(buckets: list[list[str]], sliding: list[bool]) -> list[list[str]] | None:\\n    """Layer groups for the buckets (layer names per spec bucket), or None to keep vLLM\\\'s grouping."""\\n    others = [len(names) for names, is_sliding in zip(buckets, sliding) if not is_sliding]\\n    if not any(sliding) or not others or math.gcd(*others) < 4:\\n        return None\\n    size = math.gcd(*others)\\n    groups = []\\n    for names in buckets:\\n        count = -(-len(names) // size)\\n        groups.extend(names[i::count] for i in range(count))\\n    return groups\\n\\n\\ndef patch(module) -> None:\\n    original = module._get_kv_cache_groups_uniform_page_size\\n\\n    def grouped(kv_cache_spec):\\n        same_type = defaultdict(list)\\n        for name, spec in kv_cache_spec.items():\\n            same_type[spec].append(name)\\n        buckets, spec_buckets = [], []\\n        for spec, names in same_type.items():  # the same merge of compatible specs as vLLM\\\'s\\n            for bucket, specs in zip(buckets, spec_buckets):\\n                try:\\n                    type(specs[0]).merge([*specs, spec])\\n                except (AssertionError, ValueError):\\n                    continue\\n                bucket.extend(names)\\n                specs.append(spec)\\n                break\\n            else:\\n                buckets.append(list(names))\\n                spec_buckets.append([spec])\\n        groups = group_layers(buckets, [isinstance(specs[0], module.SlidingWindowSpec) for specs in spec_buckets])\\n        if groups is None:\\n            return original(kv_cache_spec)\\n        module.logger.info("KV cache groups of %s layers (sitecustomize patch)", [len(g) for g in groups])\\n        return module.create_kv_cache_group_specs(kv_cache_spec, groups)\\n\\n    module._get_kv_cache_groups_uniform_page_size = grouped\\n\\n\\ndef patch_marlin_moe_scale_dtype(module) -> None:\\n    import inspect\\n\\n    import torch\\n\\n    scheme = module.GPTQMarlinMoEScheme\\n    create = scheme.create_weights\\n\\n    def create_weights(self, layer, *args, **kwargs):\\n        create(self, layer, *args, **kwargs)\\n        dtype = inspect.signature(create).bind(self, layer, *args, **kwargs).arguments["params_dtype"]\\n        for name in ("w13_scales", "w2_scales"):\\n            old = getattr(layer, name)\\n            if old.dtype != dtype:\\n                new = torch.nn.Parameter(torch.empty_like(old, dtype=dtype), requires_grad=False)\\n                new.__dict__.update(old.__dict__)  # the weight loader and SGLang\\\'s other attributes\\n                layer.register_parameter(name, new)\\n\\n    scheme.create_weights = create_weights\\n\\n\\nGPTQ_MARLIN_MOE_TENSORS = ("w13_qweight", "w2_qweight", "w13_scales", "w2_scales", "w13_g_idx", "w2_g_idx",\\n                           "w13_g_idx_sort_indices", "w2_g_idx_sort_indices")\\n\\n\\ndef patch_gptq_marlin_moe_kernel(module) -> None:\\n    import torch\\n\\n    kernel = module.GPTQMarlinMoEKernel\\n    process = kernel.process_weights_after_loading\\n\\n    def process_weights_after_loading(self, layer):\\n        before = {name: getattr(layer, name) for name in GPTQ_MARLIN_MOE_TENSORS if hasattr(layer, name)}\\n        process(self, layer)\\n        for name, old in before.items():\\n            if getattr(layer, name) is not old:\\n                old.data = old.data.new_empty(0)  # whoever still holds the replaced parameter, its memory goes now\\n        if torch.cuda.is_available():\\n            torch.cuda.empty_cache()\\n\\n    def apply(self, layer, dispatch_output):  # SGLang 0.5.21\\\'s, plus the expert map\\n        dispatcher = getattr(layer, "dispatcher", None)\\n        expert_map = getattr(dispatcher, "local_expert_mapping", None)  # built by the dispatcher\\\'s first EP dispatch\\n        quant_info = module.MarlinMoeQuantInfo(\\n            w13_qweight=layer.w13_qweight, w2_qweight=layer.w2_qweight, w13_scales=layer.w13_scales,\\n            w2_scales=layer.w2_scales, w13_g_idx=layer.w13_g_idx, w2_g_idx=layer.w2_g_idx,\\n            w13_g_idx_sort_indices=layer.w13_g_idx_sort_indices, w2_g_idx_sort_indices=layer.w2_g_idx_sort_indices,\\n            weight_bits=self.quant_config.weight_bits, is_k_full=self.is_k_full, expert_map=expert_map,\\n            global_num_experts=dispatcher.num_experts if expert_map is not None else -1)\\n        return self.runner.run(dispatch_output, quant_info)\\n\\n    kernel.process_weights_after_loading = process_weights_after_loading\\n    kernel.apply = apply\\n\\n\\nROUTER_STATS_EVERY = 256\\n\\n\\n_ROUTER = {"modules": [], "stats": {}, "calls": 0, "last": None}\\n\\n\\ndef patch_router_stats(module, path: str) -> None:\\n    import torch\\n\\n    topk = module.TopK\\n    init = topk.__init__\\n    modules, stats = _ROUTER["modules"], _ROUTER["stats"]\\n\\n    def __init__(self, *args, **kwargs):\\n        init(self, *args, **kwargs)\\n        self._arc_index = len(modules)\\n        modules.append(self)\\n\\n    def record(self, out) -> None:\\n        ids, weights, logits = out.topk_ids, out.topk_weights, out.router_logits\\n        experts = logits.shape[-1]\\n        table = stats.get(self._arc_index)\\n        if table is None:\\n            table = stats[self._arc_index] = torch.zeros(3, experts, dtype=torch.float32, device=ids.device)\\n        valid = (ids >= 0) & (ids < experts)\\n        flat = ids.clamp(0, experts - 1).reshape(-1).long()\\n        table[0].index_add_(0, flat, valid.reshape(-1).float())\\n        table[1].index_add_(0, flat, (weights.float() * valid).reshape(-1))\\n        _ROUTER["last"] = (table, flat, valid.reshape(-1).float(), ids.shape)\\n        if self._arc_index == 0:\\n            _ROUTER["calls"] += 1\\n            if _ROUTER["calls"] % ROUTER_STATS_EVERY == 0:\\n                rank = torch.distributed.get_rank() if torch.distributed.is_initialized() else 0\\n                target = f"{path}.rank{rank}"\\n                torch.save({index: (modules[index].layer_id, table.cpu()) for index, table in stats.items()}, target + ".part")\\n                os.replace(target + ".part", target)\\n\\n    for name in ("forward_cuda", "forward_native"):\\n        original = getattr(topk, name)\\n\\n        def wrapped(self, *args, _original=original, **kwargs):\\n            out = _original(self, *args, **kwargs)\\n            record(self, out)\\n            return out\\n\\n        setattr(topk, name, wrapped)\\n    topk.__init__ = __init__\\n\\n\\ndef patch_expert_output_norms(module) -> None:\\n    """Wrap the Triton MoE\\\'s sum over the top-k experts: add each slot\\\'s output norm to the expert routed there."""\\n    for name in ("moe_sum_reduce", "moe_sum_reduce_triton", "moe_sum_reduce_torch_compile"):\\n        original = getattr(module, name, None)\\n        if original is None:\\n            continue\\n\\n        def wrapped(x, *args, _original=original, **kwargs):\\n            last = _ROUTER["last"]\\n            if last is not None and tuple(x.shape[:2]) == tuple(last[3]):\\n                table, flat, valid, _ = last\\n                table[2].index_add_(0, flat, x.float().norm(dim=-1).reshape(-1) * valid)\\n            return _original(x, *args, **kwargs)\\n\\n        setattr(module, name, wrapped)\\n\\n\\ndef patch_reap_keep(module, path: str, class_name: str) -> None:\\n    """Flash-Next\\\'s load_weights drops the experts REAP pruned and renumbers the kept ones, slicing each router to the\\n    kept rows, as src.qwen.reap.prune writes them: the full checkpoint (e.g. a public copy) serves as the pruned model.\\n    path is prune\\\'s <output>.reap_keep.json; config.json must already declare the kept expert count. Patched on the\\n    target (Qwen4ExpForConditionalGeneration) and on the MTP drafter (Qwen4ExpForCausalLMMTP): under expert parallelism\\n    SGLang shares one expert-location table between them, so the drafter must keep as many experts as the target."""\\n    import json\\n\\n    import torch\\n    from src.qwen.reap import ROUTER, renamed\\n\\n    with open(path) as stream:\\n        kept = {tuple(key.rsplit(".", 1)): experts for key, experts in json.load(stream).items()}\\n    kept = {(prefix, int(layer)): experts for (prefix, layer), experts in kept.items()}\\n    position = {key: {expert: new for new, expert in enumerate(experts)} for key, experts in kept.items()}\\n    cls = getattr(module, class_name)\\n    original = cls.load_weights\\n\\n    def load_weights(self, weights):\\n        def pruned():\\n            for name, tensor in weights:\\n                router = ROUTER.match(name)\\n                if router:\\n                    yield name, tensor[torch.tensor(kept[(router.group(1), int(router.group(2)))])].contiguous()\\n                    continue\\n                new = renamed(name, kept, position)\\n                if new is not None:\\n                    yield new, tensor\\n\\n        return original(self, pruned())\\n\\n    cls.load_weights = load_weights\\n\\n\\ndef patch_ple_offload_load(module) -> None:\\n    original = module.PleOffloadRunner._load_weights\\n\\n    def _load_weights(self) -> None:\\n        started = time.time()\\n        original(self)\\n        with open(os.environ["ARC_VLLM_PLE_LOADED_FLAG"] + ".part", "w") as handle:\\n            handle.write(f"{time.time()}\\\\n")\\n        os.replace(os.environ["ARC_VLLM_PLE_LOADED_FLAG"] + ".part", os.environ["ARC_VLLM_PLE_LOADED_FLAG"])\\n        memory = {line.split(":")[0]: int(line.split()[1]) for line in open("/proc/meminfo")}\\n        print(f"PLE table loaded in {time.time() - started:.1f} s; MemAvailable {memory.get(\\\'MemAvailable\\\', 0) / 2**20:.1f} GiB, "\\n              f"Cached {memory.get(\\\'Cached\\\', 0) / 2**20:.1f} GiB", file=sys.stderr, flush=True)\\n\\n    module.PleOffloadRunner._load_weights = _load_weights\\n\\n\\nPLE_WAIT_SECONDS = 10800  # Merlin\\\'s own PLE ready timeout in h95/serve_l4.sh (VLLM_PLE_OFFLOAD_READY_TIMEOUT)\\n\\n\\ndef wait_for_file(path: str, timeout: float) -> float | None:\\n    """Seconds waited until path exists, or None when it did not within timeout."""\\n    started = time.time()\\n    while not os.path.exists(path):\\n        if time.time() - started >= timeout:\\n            return None\\n        time.sleep(1)\\n    return time.time() - started\\n\\n\\ndef patch_wait_ple_loaded(module) -> None:\\n    worker = module.Worker\\n    original = worker.determine_available_memory\\n\\n    def determine_available_memory(self):\\n        flag = os.environ["ARC_VLLM_PLE_LOADED_FLAG"]\\n        waited = wait_for_file(flag, PLE_WAIT_SECONDS)\\n        print(f"PLE table flag {flag}: " + (f"present after {waited:.1f} s" if waited is not None\\n                                            else f"absent after {PLE_WAIT_SECONDS} s, profiling anyway"), file=sys.stderr, flush=True)\\n        return original(self)\\n\\n    worker.determine_available_memory = determine_available_memory\\n\\n\\ndef patch_marlin_meta_workspace(module) -> None:\\n    """A Marlin layer built on meta gets a meta workspace: the halt95 engine\\\'s PLE offload process builds the whole model\\n    on meta with CUDA blocked, and the workspace size (the GPU\\\'s SM count) is the one query in an AutoRound/GPTQ MoE\\n    layer\\\'s create_weights that reaches CUDA. That process never runs the layer."""\\n    import torch\\n\\n    make = module.marlin_make_workspace_new\\n\\n    def make_workspace(device, *args, **kwargs):\\n        if torch.device(device).type == "meta":\\n            return torch.empty(0, dtype=torch.int, device="meta")\\n        return make(device, *args, **kwargs)\\n\\n    module.marlin_make_workspace_new = make_workspace\\n\\n\\ndef patch_qwen4_exp_local_logits(module) -> None:\\n    """--enable-batch-sharded-sampling (each TP rank samples a slice of the batch from full-vocab rows it gets by an\\n    all-to-all, instead of every rank all-gathering every row\\\'s full logits) asks the model for compute_logits_local, this\\n    rank\\\'s vocab shard of the logits. The Merlin engine\\\'s Qwen4ExpForConditionalGeneration forwards it to its\\n    Qwen4ExpForCausalLM, which lacks it; this is Qwen3.5\\\'s."""\\n    cls = module.Qwen4ExpForCausalLM\\n    if not hasattr(cls, "compute_logits_local"):\\n        cls.compute_logits_local = lambda self, hidden_states: self.logits_processor(self.lm_head, hidden_states, skip_gather=True)\\n\\n\\ndef patch_qwen4_exp_topk_logits(module, k: int) -> None:\\n    """ARC_VLLM_TOPK_LOGITS=k: Qwen4ExpForCausalLM.compute_logits all-gathers each TP shard\\\'s k largest logits (values\\n    and vocabulary ids) instead of its whole vocabulary slice, and returns full rows that hold those values and -inf\\n    elsewhere. Every request the runtime sends samples with top_k <= k (the deployment\\\'s top-k 20), so top-k masks every\\n    other entry before the softmax: the sampled distribution, the probabilistic draft acceptance and the sampler\\\'s random\\n    draws are unchanged, at 2k gathered values per row instead of vocab / TP."""\\n    import torch\\n    from vllm.distributed import get_tensor_model_parallel_world_size, tensor_model_parallel_all_gather\\n\\n    def compute_logits(self, hidden_states):\\n        processor, head = self.logits_processor, self.lm_head\\n        assert processor.soft_cap is None and processor.scale == 1.0 and not processor.logits_as_input\\n        local = processor._apply_head(head, hidden_states, None)\\n        padding = head.shard_indices.num_org_vocab_padding\\n        if padding:\\n            local[..., -padding:] = float("-inf")\\n        values, ids = local.topk(k, dim=-1)\\n        ids = ids + head.shard_indices.org_vocab_start_index\\n        if get_tensor_model_parallel_world_size() > 1:\\n            values = tensor_model_parallel_all_gather(values.contiguous(), dim=-1)\\n            ids = tensor_model_parallel_all_gather(ids.contiguous(), dim=-1)\\n        rows = torch.full((local.shape[0], processor.org_vocab_size), float("-inf"), dtype=local.dtype, device=local.device)\\n        return rows.scatter_(-1, ids, values)\\n\\n    module.Qwen4ExpForCausalLM.compute_logits = compute_logits\\n\\n\\ndef patch_pp_balance(module, balance: bool, every: int = 4096) -> None:\\n    """ARC_VLLM_PP_BALANCE=1: under pipeline parallelism with the V2 model runner and async scheduling, a decoding\\n    request is due again pp_size steps after it was scheduled, so requests that become due on the same step stay together\\n    for good; when all of them share one step (they started together), every other step is empty and the first stage\\n    idles while the last stage samples and drafts. Before each schedule() at most ceil(decoding / pp_size) due decoding\\n    requests stay due and the rest wait one more step, so the pipeline alternates two halves of the batch. A request\\n    waiting longer than pp_size steps is what an exhausted token budget already does: the worker applies step T\\\'s\\n    sampled tokens at step T + pp_size whatever that step holds. Sampling is unchanged; only batch shapes change.\\n    ARC_VLLM_PP_BALANCE=stats only logs, every `every` schedule() calls (4096; stats:<calls> sets it) and whenever the\\n    state changes: empty steps and requests per non-empty step, then the split: schedule() calls by step % pp_size (the\\n    micro-batch a decoding request stays in) and by scheduled requests that are not mid-prefill, as JSON (12 requests split 4/8 under PP2:\\n    {"0": {"4": n}, "1": {"8": n}}). Every mode logs both lines. ARC_VLLM_PP_BALANCE=file:<path> balances while <path>\\n    exists (an on/off comparison on one server)."""\\n    cls = module.AsyncScheduler\\n    schedule = cls.schedule\\n    counts = [0, 0, 0]  # schedule() calls, empty steps, scheduled requests\\n    split = defaultdict(int)  # (step % pp_size, scheduled requests not mid-prefill) -> schedule() calls\\n    flag = balance[len("file:"):] if isinstance(balance, str) and balance.startswith("file:") else None\\n    last = [None]\\n\\n    def report(state):\\n        busy = counts[0] - counts[1]\\n        print(f"[pp_balance] balance={state} calls={counts[0]} empty={counts[1]} requests_per_step={counts[2] / max(busy, 1):.2f} "\\n              f"time={time.time():.1f}", file=sys.stderr, flush=True)\\n        histogram = {}\\n        for (slot, decoding), calls in sorted(split.items()):\\n            histogram.setdefault(str(slot), {})[str(decoding)] = calls\\n        print(f"[pp_balance] split balance={state} {json.dumps(histogram)}", file=sys.stderr, flush=True)\\n        counts[:] = [0, 0, 0]\\n        split.clear()\\n\\n    def balanced(self, *args, **kwargs):\\n        on = os.path.exists(flag) if flag else balance is True\\n        if on != last[0]:\\n            if last[0] is not None:\\n                report(last[0])\\n            print(f"[pp_balance] now={on}", file=sys.stderr, flush=True)  # marks the server log for the comparison\\n            last[0] = on\\n        if on and self.pp_size > 1 and self.use_v2_model_runner:\\n            step = self.current_step + 1  # schedule() advances the counter first\\n            decoding = [r for r in self.running if not r.is_prefill_chunk]\\n            due = [r for r in decoding if r.next_decode_eligible_step <= step]\\n            for request in due[math.ceil(len(decoding) / self.pp_size):]:\\n                request.next_decode_eligible_step = step + 1\\n        output = schedule(self, *args, **kwargs)\\n        scheduled = len(output.num_scheduled_tokens)\\n        counts[0], counts[1], counts[2] = counts[0] + 1, counts[1] + (scheduled == 0), counts[2] + scheduled\\n        decoding = sum(not getattr(self.requests.get(r), "is_prefill_chunk", True) for r in output.num_scheduled_tokens)\\n        split[self.current_step % self.pp_size, decoding] += 1\\n        if counts[0] == every:\\n            report(on)\\n        return output\\n\\n    cls.schedule = balanced\\n\\n\\ndef chain(name: str, hook) -> None:\\n    """Register hook for module name after any hook already registered for it."""\\n    previous = _PATCHES.get(name)\\n    _PATCHES[name] = hook if previous is None else (lambda module: (previous(module), hook(module)))\\n\\n\\nFLASHNEXT_ARCHITECTURE = "Qwen4ExpForConditionalGeneration"\\n\\n\\ndef patch_language_model_only(module) -> None:\\n    args = module.ServerArgs\\n    if FLASHNEXT_ARCHITECTURE not in args.LANGUAGE_MODEL_ONLY_ARCHITECTURES:\\n        args.LANGUAGE_MODEL_ONLY_ARCHITECTURES = (*args.LANGUAGE_MODEL_ONLY_ARCHITECTURES, FLASHNEXT_ARCHITECTURE)\\n\\n\\ndef patch_draft_head_sharing(module) -> None:\\n    scheduler = module.Scheduler\\n    init_pools = scheduler.init_memory_pools\\n\\n    def init_memory_pools(self):\\n        draft = getattr(self, "draft_worker", None)\\n        if draft is not None and hasattr(draft, "init_token_map") and hasattr(draft, "init_lm_head"):\\n            import torch\\n\\n            draft.init_token_map()\\n            draft.init_lm_head()  # repeated by alloc_memory_pool with the same shared tensors\\n            if torch.cuda.is_available():\\n                torch.cuda.empty_cache()\\n        init_pools(self)\\n\\n    scheduler.init_memory_pools = init_memory_pools\\n\\n\\ndef qsa_triton_varlen(q, k, v, cu_seqlens_q, cu_seqlens_k, max_seqlen_q, max_seqlen_k, softmax_scale, causal=True):\\n    """flash_attn_varlen_func for QSA\\\'s packed decode (one query per row) through SGLang\\\'s Triton kernel."""\\n    import torch\\n    from sglang.srt.layers.attention.qsa.sparse_attn import sparse_gqa_packed_decode_triton\\n\\n    if max_seqlen_q != 1:\\n        raise ValueError(f"the Triton packed decode takes one query per row, got max_seqlen_q={max_seqlen_q}")\\n    valid = (cu_seqlens_k[1:] - cu_seqlens_k[:-1]).to(torch.int32)\\n    rows = valid.shape[0]\\n    relative = torch.arange(max_seqlen_k, dtype=torch.int32, device=q.device).expand(rows, -1)\\n    relative = relative.masked_fill(relative >= valid[:, None], -1).contiguous()\\n    return sparse_gqa_packed_decode_triton(q.contiguous(), k, v, relative, cu_seqlens_q, cu_seqlens_k, valid, softmax_scale)\\n\\n\\ndef patch_qsa_triton_decode(module) -> None:\\n    module._resolve_flash_attn_varlen_func = lambda: qsa_triton_varlen\\n\\n\\n_PATCHES = {"vllm.v1.core.kv_cache_utils": patch}\\nif os.environ.get("ARC_VLLM_PLE_LOADED_FLAG"):\\n    chain("vllm.v1.ple_offload.worker", patch_ple_offload_load)\\n    chain("vllm.v1.worker.gpu_worker", patch_wait_ple_loaded)\\nif os.environ.get("ARC_SGLANG_GPTQ_MARLIN_MOE") == "1":\\n    _PATCHES["sglang.srt.layers.quantization.gptq.schemes.gptq_moe"] = patch_marlin_moe_scale_dtype\\n    _PATCHES["sglang.srt.hardware_backend.gpu.quantization.gptq_kernels"] = patch_gptq_marlin_moe_kernel\\nif os.environ.get("ARC_SGLANG_SAVE_MEMORY") == "1":\\n    _PATCHES["sglang.srt.server_args"] = patch_language_model_only\\n    _PATCHES["sglang.srt.managers.scheduler"] = patch_draft_head_sharing\\nif os.environ.get("ARC_SGLANG_QSA_TRITON") == "1":\\n    _PATCHES["sglang.srt.layers.attention.qwen_sparse_attn_backend"] = patch_qsa_triton_decode\\nif os.environ.get("ARC_SGLANG_REAP_KEEP"):\\n    chain("sglang.srt.models.qwen4_exp", lambda module, keep=os.environ["ARC_SGLANG_REAP_KEEP"]: patch_reap_keep(\\n        module, keep, "Qwen4ExpForConditionalGeneration"))\\n    _PATCHES["sglang.srt.models.qwen4_exp_mtp"] = lambda module: patch_reap_keep(\\n        module, os.environ["ARC_SGLANG_REAP_KEEP"], "Qwen4ExpForCausalLMMTP")\\nif os.environ.get("ARC_VLLM_WEIGHT_OVERRIDE"):  # trained bf16 tensors (src.qwen.fnweights export); the REAP hook\\n    # below wraps this loader, so it sees the pruned, renumbered stream\\n    chain("vllm.models.qwen4_exp.nvidia.model", lambda module, path=os.environ["ARC_VLLM_WEIGHT_OVERRIDE"], keep=os.environ.get("ARC_VLLM_REAP_KEEP"):\\n          __import__("src.qwen.fnweights", fromlist=["_"]).patch_weight_override(module, path, "Qwen4ExpForConditionalGeneration", keep))\\nif os.environ.get("ARC_VLLM_REAP_KEEP"):\\n    for _name, _hook in __import__("vllm_reap").patches(os.environ["ARC_VLLM_REAP_KEEP"], patch_reap_keep).items():\\n        chain(_name, _hook)\\n    _PATCHES["vllm.model_executor.layers.quantization.utils.marlin_utils"] = patch_marlin_meta_workspace\\nif os.environ.get("ARC_VLLM_READ_FILTER") == "1" or os.environ.get("ARC_VLLM_WEIGHT_DIGEST") == "1":  # vllm_read_filter.py\\n    for _name, _hook in __import__("vllm_read_filter").patches(os.environ.get("ARC_VLLM_READ_FILTER") == "1", os.environ.get("ARC_VLLM_REAP_KEEP"),\\n                                                               os.environ.get("ARC_VLLM_WEIGHT_DIGEST") == "1").items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_DRAFT_VOCAB"):\\n    _PATCHES.update(__import__("vllm_draft_vocab").patches(os.environ["ARC_VLLM_DRAFT_VOCAB"], os.environ.get("ARC_VLLM_DRAFT_VOCAB_BALANCE") == "1",\\n                                                           os.environ.get("ARC_VLLM_DRAFT_VOCAB_SAMPLE") == "restricted"))\\nif os.environ.get("ARC_VLLM_TOPK_LOGITS"):  # after the REAP hook on the same module\\n    chain("vllm.models.qwen4_exp.nvidia.model", lambda module, k=int(os.environ["ARC_VLLM_TOPK_LOGITS"]): patch_qwen4_exp_topk_logits(module, k))\\nif os.environ.get("ARC_VLLM_PP_BALANCE", "").startswith(("1", "stats", "file:")):\\n    _PATCHES["vllm.v1.core.sched.async_scheduler"] = lambda module, value=os.environ["ARC_VLLM_PP_BALANCE"]: patch_pp_balance(\\n        module, value if value.startswith("file:") else value == "1", int(value[len("stats:"):]) if value.startswith("stats:") else 4096)\\nif os.environ.get("ARC_VLLM_LOCAL_LOGITS") == "1":  # after the REAP hook on the same module\\n    chain("vllm.models.qwen4_exp.nvidia.model", patch_qwen4_exp_local_logits)\\nif os.environ.get("ARC_SGLANG_ROUTER_STATS"):\\n    _PATCHES["sglang.srt.layers.moe.topk"] = lambda module: patch_router_stats(module, os.environ["ARC_SGLANG_ROUTER_STATS"])\\n    _PATCHES["sglang.srt.layers.moe.moe_runner.triton_utils.fused_moe"] = patch_expert_output_norms\\nif os.environ.get("ARC_VLLM_MEM_REPORT"):  # a measurement: each worker\\\'s memory after capture and every N steps\\n    for _name, _hook in __import__("vllm_mem_report").patches().items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_QSA_HOST_KV") == "1":  # after the grouping patch on vllm.v1.core.kv_cache_utils\\n    for _name, _hook in __import__("vllm_qsa_host_kv").patches().items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_LATE_COMMIT") == "1":  # vllm_late_commit.py, after the host-KV runner hook\\n    for _name, _hook in __import__("vllm_late_commit").patches().items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_GDN_REUSE") in ("1", "check"):  # vllm_gdn_reuse.py\\n    for _name, _hook in __import__("vllm_gdn_reuse").patches().items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_RETAIN_LAST_STATE") == "1":  # after ARC_VLLM_QSA_HOST_KV\\\'s hooks on the same modules\\n    for _name, _hook in __import__("vllm_last_state").patches().items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_GDN_PARK", "0") not in ("", "0"):  # after ARC_VLLM_RETAIN_LAST_STATE\\\'s (its hold maps)\\n    for _name, _hook in __import__("vllm_gdn_park").patches().items():\\n        chain(_name, _hook)\\n_dense, _drafter = os.environ.get("ARC_VLLM_FP8_DENSE"), os.environ.get("ARC_VLLM_FP8_DENSE_DRAFTER", "")\\nif _dense not in (None, "", "0"):  # after the other hooks on the target and drafter modules\\n    for _name, _hook in __import__("vllm_fp8_dense").patches(_dense, _drafter == "1").items():\\n        chain(_name, _hook)\\nelif _dense is None and _drafter not in ("", "0"):  # the drafter alone, in its own mode\\n    for _name, _hook in __import__("vllm_fp8_dense").patches(_drafter, True, target=False).items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_VLLM_DFLASH2") == "1":  # after the other hooks on the target module (its forward replaced last)\\n    for _name, _hook in __import__("vllm_dflash2").patches().items():\\n        chain(_name, _hook)\\nif os.environ.get("ARC_SGLANG_QSA_HISPARSE") == "1":  # after ARC_SGLANG_QSA_TRITON: register() chains onto it\\n    __import__("qsa_hisparse").register(_PATCHES)\\n\\n\\nclass _Hook(importlib.abc.MetaPathFinder):\\n    def find_spec(self, name, path=None, target=None):\\n        if name not in _PATCHES:\\n            return None\\n        sys.meta_path.remove(self)\\n        try:\\n            spec = importlib.util.find_spec(name)\\n        finally:\\n            sys.meta_path.insert(0, self)\\n        load = spec.loader.exec_module\\n\\n        def exec_module(module):\\n            load(module)\\n            _PATCHES[name](module)\\n\\n        spec.loader.exec_module = exec_module\\n        return spec\\n\\n\\nsys.meta_path.insert(0, _Hook())\\n\', \'src/qwen/flashnext.py\': \'"""Assemble Qwen3.8-Flash-Next serving directories from read-only copies (Kaggle mounts) with links instead of copies:\\n\\n- model: Intel\\\'s W4A16 AutoRound checkpoint (public Kaggle copy) with its bf16 n-gram (PLE) shards replaced by the FP8\\n  table (albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE\\\'s model-plefp8-* shards) and without the MTP tensors, which is\\n  exactly albucino\\\'s FP8PLE index; config.json declares the FP8 table and, with a REAP keep list, the kept expert\\n  count (the server prunes while loading: ARC_SGLANG_REAP_KEEP in src/qwen/vllm_patch).\\n- drafter: albucino\\\'s int4 MTP head with a quantization config SGLang 0.5.21 accepts (its own targets "RoutedExperts",\\n  a module class of the vLLM image it was made for, and ignores nothing, so SGLang finds no scheme for its other\\n  layers): experts are matched by name, everything else stays bf16. With a keep list its config declares the kept\\n  expert count too, and the server prunes its experts with the keep list\\\'s mtp.layers.0 entry.\\n- dflash2 (with --dflash2): the DFlash2 drafter\\\'s files (incoai/Qwen3.8-Flash-Next-DFlash2) plus the target\\\'s input\\n  embedding as embed_tokens.weight in its own file: the drafter has none, and under pipeline parallelism the last stage,\\n  where it runs, holds no copy of the target\\\'s (h95\\\'s DFLASH2, src/qwen/vllm_patch/vllm_dflash2.py).\\n    python -m src.qwen.flashnext --intel DIR --fp8-table DIR --drafter DIR [--keep KEEP.json] [--dflash2 DIR] --output DIR\\n"""\\nimport argparse\\nimport json\\nfrom pathlib import Path\\nimport struct\\n\\nfrom src.utils.tensors import safetensors_header\\n\\nMTP_SHARD = "model_extra_tensors.safetensors"\\nEMBEDDING = "embed_tokens.weight"  # the DFlash loader adds "model."; the target\\\'s tensor ends with it outside the MTP head\\nFP8_TABLE_GLOB = "model-plefp8-*.safetensors"\\nDRAFT_QUANTIZATION = {\\n    "config_groups": {"mtp_routed_experts": {\\n        "format": "pack-quantized", "input_activations": None, "output_activations": None, "targets": [r"re:.*\\\\.mlp\\\\.experts\\\\..*"],\\n        "weights": {"actorder": None, "block_structure": None, "dynamic": False, "group_size": 32, "num_bits": 4, "observer": "minmax",\\n                    "observer_kwargs": {}, "strategy": "group", "symmetric": True, "type": "int"}}},\\n    "format": "pack-quantized", "global_compression_ratio": None, "ignore": [r"re:^(?!.*\\\\.mlp\\\\.experts\\\\.).*$"],\\n    "kv_cache_scheme": None, "quant_method": "compressed-tensors", "quantization_status": "compressed", "sparsity_config": {},\\n    "transform_config": {}}\\n\\n\\ndef link_others(source: Path, output: Path, skip: set[str]) -> None:\\n    for entry in source.iterdir():\\n        if entry.name not in skip and not entry.name.startswith(".") and not entry.name.endswith(".safetensors") \\\\\\n                and not (output / entry.name).exists():\\n            (output / entry.name).symlink_to(entry)\\n\\n\\ndef assemble_model(intel: Path, fp8_table: Path, output: Path, num_experts: int | None = None) -> Path:\\n    weight_map = {name: file for name, file in json.loads((intel / "model.safetensors.index.json").read_text())["weight_map"].items()\\n                  if file != MTP_SHARD}\\n    tables = sorted(fp8_table.glob(FP8_TABLE_GLOB))\\n    if not tables:\\n        raise FileNotFoundError(f"no {FP8_TABLE_GLOB} in {fp8_table}")\\n    for table in tables:\\n        weight_map.update({name: table.name for name in safetensors_header(table)[0]})\\n    output.mkdir(parents=True, exist_ok=False)\\n    for file in sorted(set(weight_map.values())):\\n        (output / file).symlink_to((fp8_table if file.startswith("model-plefp8-") else intel) / file)\\n    config = json.loads((intel / "config.json").read_text())\\n    text = config.get("text_config", config)\\n    text["ple_embedding_dtype"] = "float8_e4m3fn"\\n    if num_experts is not None:\\n        text["num_experts"] = num_experts\\n    (output / "config.json").write_text(json.dumps(config, indent=2))\\n    (output / "model.safetensors.index.json").write_text(json.dumps({"metadata": {}, "weight_map": weight_map}))\\n    link_others(intel, output, {"config.json", "model.safetensors.index.json"})\\n    return output\\n\\n\\ndef assemble_drafter(drafter: Path, output: Path, num_experts: int | None = None) -> Path:\\n    """The drafter with SGLang\\\'s quantization config; num_experts as the target\\\'s when REAP prunes both (SGLang shares\\n    one expert-location table between target and drafter under expert parallelism)."""\\n    output.mkdir(parents=True, exist_ok=False)\\n    for entry in drafter.iterdir():\\n        if entry.name != "config.json":\\n            (output / entry.name).symlink_to(entry)\\n    config = json.loads((drafter / "config.json").read_text())\\n    config["quantization_config"] = DRAFT_QUANTIZATION\\n    if num_experts is not None:\\n        config.get("text_config", config)["num_experts"] = num_experts\\n    (output / "config.json").write_text(json.dumps(config, indent=2))\\n    return output\\n\\n\\ndef assemble_dflash2(dflash2: Path, intel: Path, output: Path) -> Path:\\n    """The DFlash2 drafter\\\'s files linked, and the target\\\'s input embedding written as EMBEDDING in embed.safetensors\\n    (its bytes copied as stored)."""\\n    weight_map = json.loads((intel / "model.safetensors.index.json").read_text())["weight_map"]\\n    names = [name for name in weight_map if name.endswith(EMBEDDING) and "mtp" not in name]\\n    if len(names) != 1:\\n        raise ValueError(f"expected one target {EMBEDDING} in {intel}, found {names}")\\n    shard = intel / weight_map[names[0]]\\n    header, start = safetensors_header(shard)\\n    entry = header[names[0]]\\n    begin, end = entry["data_offsets"]\\n    output.mkdir(parents=True, exist_ok=False)\\n    for item in dflash2.iterdir():\\n        if not item.name.startswith(".") and item.name != "embed.safetensors":\\n            (output / item.name).symlink_to(item)\\n    new_header = json.dumps({EMBEDDING: {"dtype": entry["dtype"], "shape": entry["shape"], "data_offsets": [0, end - begin]}}).encode()\\n    new_header += b" " * (-len(new_header) % 8)\\n    with open(shard, "rb") as source, open(output / "embed.safetensors", "wb") as target:\\n        target.write(struct.pack("<Q", len(new_header)) + new_header)\\n        source.seek(start + begin)\\n        remaining = end - begin\\n        while remaining:\\n            chunk = source.read(min(remaining, 1 << 26))\\n            target.write(chunk)\\n            remaining -= len(chunk)\\n    return output\\n\\n\\ndef kept_experts(keep: Path) -> int:\\n    counts = {len(experts) for experts in json.loads(keep.read_text()).values()}\\n    if len(counts) != 1:\\n        raise ValueError(f"layers keep different expert counts: {counts}")\\n    return counts.pop()\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    parser.add_argument("--intel", type=Path, required=True, help="Intel\\\'s Qwen3.8-Flash-Next-W4A16-AutoRound files")\\n    parser.add_argument("--fp8-table", type=Path, required=True, help="directory with the model-plefp8-* shards")\\n    parser.add_argument("--drafter", type=Path, required=True, help="albucino\\\'s int4 MTP drafter files")\\n    parser.add_argument("--keep", type=Path, help="REAP keep list (src/qwen/reap.py); the config then declares its expert count")\\n    parser.add_argument("--dflash2", type=Path, help="DFlash2 drafter files (incoai/Qwen3.8-Flash-Next-DFlash2): OUTPUT/dflash2 with the target\\\'s embedding")\\n    parser.add_argument("--output", type=Path, required=True)\\n    args = parser.parse_args()\\n    experts = kept_experts(args.keep) if args.keep else None\\n    model = assemble_model(args.intel, args.fp8_table, args.output / "model", experts)\\n    drafter = assemble_drafter(args.drafter, args.output / "drafter", experts)\\n    dflash2 = assemble_dflash2(args.dflash2, args.intel, args.output / "dflash2") if args.dflash2 else None\\n    print(json.dumps({"model": str(model), "drafter": str(drafter), **({"dflash2": str(dflash2)} if dflash2 else {})}))\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'src/qwen/reap.py\': \'"""REAP expert pruning (Router-weighted Expert Activation Pruning, Lasby et al. 2025) for Qwen3.8-Flash-Next checkpoints.\\n\\nAn expert\\\'s REAP saliency is the mean, over the tokens routed to it, of g_j(x) * ||f_j(x)||: its routing weight times\\nthe norm of its output. The statistics come from an SGLang run with ARC_SGLANG_ROUTER_STATS (src/qwen/vllm_patch): every\\nTP rank writes {topk index: (layer_id, tensor[3, experts])} with the routed count, the routing-weight sum and the sum of\\n||g_j(x) f_j(x)|| per expert; under TP a rank sees its share of f_j(x), so the ranks\\\' saliencies are averaged. TopK\\nindices follow construction order: the target\\\'s layers, then the MTP layer.\\n\\nEvery MoE layer (and the MTP layer) keeps its `keep` highest-saliency experts in their original order, renumbered from\\n0; its router rows are sliced to match. Shards holding expert or router tensors are rewritten, every other file is\\nsymlinked to the source, and config.json gets num_experts = keep.\\n    python -m src.qwen.reap --stats STATS.pt.rank0 [STATS.pt.rank1 ...] --source MODEL --output OUT --keep 384\\n"""\\nimport argparse\\nimport hashlib\\nimport json\\nimport os\\nimport re\\nfrom pathlib import Path\\n\\nEXPERT = re.compile(r"^(model\\\\.language_model\\\\.layers|mtp\\\\.layers)\\\\.(\\\\d+)\\\\.mlp\\\\.experts\\\\.(\\\\d+)\\\\.(.+)$")\\nROUTER = re.compile(r"^(model\\\\.language_model\\\\.layers|mtp\\\\.layers)\\\\.(\\\\d+)\\\\.mlp\\\\.gate\\\\.weight$")\\nTARGET, DRAFT = "model.language_model.layers", "mtp.layers"\\n\\n\\ndef keep_sha256(keep: Path) -> str:\\n    """sha256 of the keep list\\\'s canonical JSON (sorted keys, no spaces): reformatting the file does not change it."""\\n    canonical = json.dumps(json.loads(Path(keep).read_text()), sort_keys=True, separators=(",", ":"))\\n    return hashlib.sha256(canonical.encode()).hexdigest()\\n\\n\\ndef saliency(stats_files: list[Path], target_layers: int) -> dict[tuple[str, int], "torch.Tensor"]:\\n    """(prefix, layer) -> REAP saliency per expert, averaged over the ranks\\\' files."""\\n    import torch\\n\\n    per_rank = [torch.load(path) for path in stats_files]\\n    scores = {}\\n    for index in per_rank[0]:\\n        layer_id, _ = per_rank[0][index]\\n        key = (TARGET, layer_id) if index < target_layers else (DRAFT, index - target_layers)\\n        rank_scores = []\\n        for stats in per_rank:\\n            table = stats[index][1].double()\\n            rank_scores.append(torch.where(table[0] > 0, table[2] / table[0].clamp(min=1), torch.zeros_like(table[0])))\\n        scores[key] = torch.stack(rank_scores).mean(0)\\n    return scores\\n\\n\\ndef keep_sets(scores: dict, keep: int) -> dict[tuple[str, int], list[int]]:\\n    return {key: sorted(score.argsort(descending=True)[:keep].tolist()) for key, score in scores.items()}\\n\\n\\ndef renamed(name: str, kept: dict[tuple[str, int], list[int]], position: dict) -> str | None:\\n    """The tensor\\\'s name in the pruned checkpoint, or None when its expert is pruned."""\\n    match = EXPERT.match(name)\\n    if not match:\\n        return name\\n    prefix, layer, expert, rest = match.group(1), int(match.group(2)), int(match.group(3)), match.group(4)\\n    if (prefix, layer) not in kept:\\n        raise KeyError(f"no REAP statistics for {prefix}.{layer}")\\n    new = position[(prefix, layer)].get(expert)\\n    return None if new is None else f"{prefix}.{layer}.mlp.experts.{new}.{rest}"\\n\\n\\ndef prune(source: Path, output: Path, kept: dict[tuple[str, int], list[int]]) -> dict:\\n    import torch\\n    from safetensors import safe_open\\n    from safetensors.torch import save_file\\n\\n    position = {key: {expert: new for new, expert in enumerate(experts)} for key, experts in kept.items()}\\n    index = json.loads((source / "model.safetensors.index.json").read_text())\\n    weight_map = index["weight_map"]\\n    output.mkdir(parents=True, exist_ok=False)\\n    touched = {file for name, file in weight_map.items() if EXPERT.match(name) or ROUTER.match(name)}\\n    new_map, dropped = {}, 0\\n    for file in sorted(set(weight_map.values())):\\n        names = [name for name, shard in weight_map.items() if shard == file]\\n        if file not in touched:\\n            os.symlink(os.path.realpath(source / file), output / file)\\n            new_map.update({name: file for name in names})\\n            continue\\n        tensors = {}\\n        with safe_open(str(source / file), framework="pt") as stream:\\n            for name in names:\\n                router = ROUTER.match(name)\\n                new = renamed(name, kept, position)\\n                if new is None:\\n                    dropped += 1\\n                    continue\\n                tensor = stream.get_tensor(name)\\n                if router:\\n                    tensor = tensor[torch.tensor(kept[(router.group(1), int(router.group(2)))])].contiguous()\\n                tensors[new] = tensor\\n        save_file(tensors, str(output / file), metadata={"format": "pt"})\\n        new_map.update({name: file for name in tensors})\\n    (output / "model.safetensors.index.json").write_text(json.dumps({**index, "weight_map": new_map}, indent=2))\\n    config = json.loads((source / "config.json").read_text())\\n    text = config.get("text_config", config)\\n    experts = {len(v) for v in kept.values()}\\n    if len(experts) != 1:\\n        raise ValueError(f"layers keep different expert counts: {experts}")\\n    text["num_experts"] = experts.pop()\\n    (output / "config.json").write_text(json.dumps(config, indent=2))\\n    for entry in source.iterdir():  # tokenizer, templates, n-gram table shards outside the index, runtime files\\n        if not (output / entry.name).exists():\\n            os.symlink(os.path.realpath(entry), output / entry.name)\\n    return {"rewritten_files": len(touched), "dropped_tensors": dropped, "num_experts": text["num_experts"]}\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    parser.add_argument("--stats", type=Path, nargs="+", required=True, help="ARC_SGLANG_ROUTER_STATS files, one per TP rank")\\n    parser.add_argument("--source", type=Path, required=True)\\n    parser.add_argument("--output", type=Path, required=True)\\n    parser.add_argument("--keep", type=int, required=True, help="experts kept per MoE layer (Flash-Next has 512)")\\n    parser.add_argument("--target-layers", type=int, default=48, help="TopK modules of the target before the MTP layer")\\n    args = parser.parse_args()\\n    kept = keep_sets(saliency(args.stats, args.target_layers), args.keep)\\n    (args.output.parent / f"{args.output.name}.reap_keep.json").write_text(\\n        json.dumps({f"{prefix}.{layer}": experts for (prefix, layer), experts in kept.items()}))\\n    print(json.dumps(prune(args.source, args.output, kept)))\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'kaggle/qwen-transductive/flashnext/reap448_keep.json\': \'{"model.language_model.layers.0": [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 136, 138, 139, 141, 142, 143, 144, 145, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 238, 239, 240, 241, 242, 243, 246, 247, 248, 249, 250, 251, 253, 254, 256, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272, 273, 274, 275, 277, 278, 279, 280, 282, 283, 285, 286, 287, 289, 290, 291, 292, 293, 294, 296, 297, 298, 299, 301, 302, 303, 304, 305, 306, 308, 309, 312, 313, 315, 316, 318, 319, 320, 321, 322, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 369, 370, 371, 372, 375, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 387, 389, 390, 392, 393, 394, 395, 396, 398, 399, 400, 401, 403, 404, 406, 407, 408, 409, 410, 411, 412, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 470, 471, 472, 473, 474, 475, 476, 478, 479, 480, 481, 482, 483, 484, 485, 486, 489, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.1": [0, 2, 3, 4, 5, 6, 10, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 142, 143, 144, 145, 146, 147, 148, 152, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 184, 185, 186, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 200, 201, 202, 203, 204, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 222, 223, 225, 226, 227, 228, 229, 230, 231, 232, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 254, 257, 258, 259, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 291, 292, 293, 294, 295, 296, 297, 300, 301, 302, 303, 304, 305, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 324, 325, 326, 328, 329, 332, 334, 336, 338, 339, 340, 342, 343, 345, 346, 347, 348, 350, 351, 352, 353, 354, 355, 356, 357, 359, 361, 362, 363, 364, 365, 366, 368, 369, 370, 372, 373, 374, 375, 376, 377, 378, 379, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 413, 414, 415, 416, 417, 418, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 458, 460, 461, 462, 464, 465, 466, 467, 468, 469, 470, 471, 472, 474, 475, 476, 477, 478, 479, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510], "model.language_model.layers.2": [0, 1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 24, 25, 26, 27, 28, 30, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 68, 69, 70, 71, 73, 74, 75, 76, 77, 79, 80, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 99, 100, 101, 102, 103, 104, 105, 106, 107, 109, 110, 112, 113, 114, 116, 117, 118, 119, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 142, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 188, 190, 191, 192, 193, 195, 196, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 220, 221, 222, 224, 225, 226, 227, 228, 229, 230, 231, 232, 234, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 263, 264, 265, 267, 268, 269, 270, 271, 272, 273, 274, 276, 277, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 319, 320, 321, 322, 323, 324, 325, 326, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 357, 358, 359, 360, 362, 363, 364, 365, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 388, 390, 391, 392, 393, 394, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 412, 413, 414, 415, 416, 417, 418, 419, 421, 422, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 442, 443, 444, 445, 447, 448, 449, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 485, 486, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 500, 501, 502, 503, 504, 505, 506, 508, 509, 510, 511], "model.language_model.layers.3": [0, 1, 2, 3, 4, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 28, 29, 30, 31, 32, 35, 36, 37, 38, 39, 41, 43, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 61, 63, 64, 65, 66, 70, 71, 72, 73, 75, 76, 78, 79, 80, 82, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 138, 139, 140, 141, 143, 144, 145, 146, 148, 149, 150, 151, 152, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 185, 186, 187, 188, 189, 190, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 219, 220, 221, 222, 223, 225, 226, 227, 228, 229, 231, 233, 235, 236, 237, 238, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 282, 283, 284, 287, 288, 289, 290, 291, 292, 293, 295, 296, 297, 298, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 321, 323, 324, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 359, 360, 361, 362, 363, 364, 366, 367, 368, 369, 370, 372, 373, 374, 375, 377, 378, 379, 381, 382, 383, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 412, 413, 415, 416, 417, 418, 419, 421, 422, 423, 424, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 441, 442, 443, 444, 445, 447, 448, 449, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.4": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 99, 100, 102, 103, 104, 105, 106, 108, 109, 110, 113, 114, 115, 116, 117, 118, 119, 120, 121, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 134, 135, 137, 138, 139, 140, 141, 142, 143, 145, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 198, 199, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 215, 216, 218, 219, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 233, 235, 236, 237, 238, 239, 241, 242, 243, 244, 245, 246, 247, 248, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 263, 264, 265, 266, 267, 268, 270, 271, 272, 273, 275, 276, 277, 278, 279, 280, 281, 283, 284, 285, 286, 287, 288, 289, 290, 292, 293, 294, 295, 296, 297, 298, 299, 300, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 318, 319, 320, 321, 322, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 335, 336, 338, 339, 340, 341, 342, 343, 344, 345, 346, 348, 349, 350, 351, 352, 354, 355, 357, 358, 359, 360, 361, 362, 363, 364, 366, 368, 369, 370, 371, 372, 373, 374, 375, 377, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 408, 410, 411, 412, 413, 414, 416, 417, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 448, 449, 451, 452, 453, 454, 455, 457, 458, 460, 461, 462, 463, 464, 465, 466, 468, 470, 471, 472, 473, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 498, 499, 501, 502, 503, 504, 505, 506, 507, 508, 509, 511], "model.language_model.layers.5": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 20, 21, 23, 24, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 96, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 125, 126, 127, 129, 130, 131, 132, 133, 135, 136, 137, 138, 139, 140, 142, 143, 144, 146, 147, 148, 149, 150, 151, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 193, 194, 195, 196, 197, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 213, 215, 216, 217, 218, 219, 220, 221, 222, 224, 225, 226, 227, 228, 229, 231, 233, 234, 235, 236, 237, 238, 239, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 255, 257, 258, 259, 261, 262, 263, 264, 266, 267, 268, 269, 270, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 329, 330, 331, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 364, 367, 368, 369, 370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 387, 388, 389, 391, 392, 393, 394, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 420, 421, 422, 423, 424, 426, 427, 428, 430, 431, 434, 435, 436, 437, 438, 439, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 465, 467, 468, 470, 471, 473, 474, 475, 476, 477, 478, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 507, 508, 509, 510, 511], "model.language_model.layers.6": [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 20, 21, 22, 23, 24, 25, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 90, 91, 93, 95, 96, 97, 98, 99, 100, 102, 104, 106, 108, 109, 110, 111, 112, 113, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 156, 157, 158, 159, 161, 162, 164, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 177, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 193, 194, 195, 196, 197, 198, 199, 200, 201, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 216, 217, 219, 220, 222, 223, 224, 226, 227, 228, 229, 230, 231, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 258, 259, 260, 261, 263, 264, 265, 266, 267, 268, 269, 270, 272, 273, 274, 275, 276, 277, 279, 280, 281, 283, 284, 285, 286, 287, 288, 289, 291, 292, 293, 294, 295, 296, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 347, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 368, 369, 371, 372, 373, 374, 375, 376, 377, 378, 379, 381, 382, 383, 384, 385, 386, 387, 388, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 412, 414, 415, 416, 417, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 451, 452, 453, 454, 455, 456, 457, 458, 459, 461, 462, 463, 465, 466, 468, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 495, 496, 497, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.7": [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 43, 44, 46, 47, 48, 49, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 101, 102, 103, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 145, 147, 148, 150, 152, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200, 202, 204, 206, 207, 208, 210, 212, 213, 214, 215, 216, 217, 218, 219, 221, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 234, 235, 237, 238, 239, 240, 241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 267, 268, 269, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 286, 287, 288, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 314, 315, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 357, 358, 360, 361, 362, 363, 364, 365, 366, 367, 368, 370, 371, 372, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 389, 390, 391, 392, 394, 395, 396, 397, 398, 399, 400, 401, 402, 404, 405, 406, 407, 408, 409, 410, 412, 413, 414, 415, 416, 417, 418, 419, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 436, 437, 438, 439, 441, 442, 444, 445, 446, 447, 449, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.8": [1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 26, 27, 28, 30, 31, 32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 71, 72, 73, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 102, 103, 104, 106, 107, 108, 109, 110, 111, 113, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 154, 155, 156, 157, 158, 159, 161, 162, 164, 165, 166, 167, 169, 170, 171, 173, 174, 175, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 227, 228, 229, 230, 231, 232, 233, 234, 236, 237, 238, 239, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 291, 292, 293, 295, 296, 297, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 363, 364, 365, 366, 368, 369, 370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 383, 386, 387, 390, 392, 394, 395, 396, 397, 398, 399, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 416, 417, 418, 419, 420, 423, 424, 425, 426, 427, 428, 429, 430, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 474, 475, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 494, 495, 496, 497, 499, 500, 501, 503, 504, 505, 506, 508, 509, 510, 511], "model.language_model.layers.9": [0, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 53, 54, 56, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 70, 71, 72, 74, 75, 76, 77, 78, 79, 80, 82, 83, 85, 86, 88, 89, 91, 93, 94, 95, 96, 97, 98, 101, 102, 103, 104, 105, 106, 108, 110, 111, 112, 113, 115, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 202, 203, 204, 205, 206, 207, 210, 211, 212, 213, 214, 215, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 252, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 268, 269, 270, 271, 272, 273, 275, 277, 278, 279, 281, 282, 283, 284, 285, 286, 287, 290, 291, 292, 293, 294, 295, 296, 297, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 316, 318, 319, 320, 321, 322, 323, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 346, 347, 348, 349, 350, 352, 353, 354, 355, 356, 357, 358, 359, 361, 362, 363, 364, 366, 367, 368, 369, 370, 371, 372, 373, 374, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 410, 411, 412, 413, 414, 415, 417, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 459, 460, 461, 463, 464, 465, 466, 468, 470, 471, 472, 474, 475, 476, 478, 480, 481, 482, 483, 484, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 508, 509, 510, 511], "model.language_model.layers.10": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 26, 27, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 95, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 110, 111, 112, 113, 114, 115, 117, 119, 120, 121, 122, 123, 124, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 155, 156, 157, 158, 161, 162, 163, 164, 165, 166, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 185, 188, 189, 191, 193, 194, 195, 196, 197, 198, 199, 200, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 220, 221, 222, 223, 224, 225, 228, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 290, 291, 292, 293, 294, 295, 296, 298, 299, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 320, 322, 323, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 361, 362, 363, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 415, 416, 417, 418, 419, 420, 421, 422, 424, 425, 426, 427, 428, 430, 431, 432, 433, 434, 435, 436, 437, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 467, 468, 469, 471, 472, 473, 474, 475, 477, 479, 480, 482, 483, 484, 485, 486, 487, 488, 490, 491, 492, 494, 495, 496, 497, 498, 499, 501, 502, 503, 504, 506, 507, 508, 509, 510, 511], "model.language_model.layers.11": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 21, 22, 24, 25, 26, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 55, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 135, 136, 137, 138, 139, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 187, 188, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 216, 218, 219, 220, 221, 222, 223, 224, 225, 226, 229, 230, 231, 232, 233, 234, 236, 237, 239, 241, 242, 243, 244, 246, 247, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 308, 309, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 342, 343, 344, 345, 346, 349, 350, 351, 352, 354, 355, 356, 357, 358, 359, 360, 361, 362, 364, 365, 367, 368, 369, 370, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 405, 406, 407, 408, 410, 412, 413, 414, 416, 417, 418, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 442, 443, 444, 445, 446, 447, 449, 450, 451, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 468, 469, 470, 471, 472, 473, 474, 475, 476, 478, 480, 481, 482, 483, 484, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 497, 498, 499, 500, 501, 502, 503, 504, 506, 507, 509, 510, 511], "model.language_model.layers.12": [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 46, 47, 48, 50, 51, 52, 53, 54, 55, 57, 59, 60, 61, 62, 63, 64, 65, 66, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 106, 107, 108, 109, 110, 111, 112, 113, 114, 116, 117, 119, 120, 121, 122, 123, 127, 128, 129, 131, 132, 134, 135, 136, 137, 138, 139, 140, 141, 143, 144, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 184, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 216, 217, 218, 219, 220, 221, 223, 224, 226, 227, 228, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 269, 272, 273, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 313, 314, 315, 317, 318, 319, 320, 321, 323, 324, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 382, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 399, 400, 401, 403, 404, 405, 406, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 424, 425, 426, 427, 428, 429, 430, 432, 433, 434, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 462, 463, 464, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 478, 479, 480, 482, 484, 485, 487, 488, 489, 490, 491, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.13": [0, 1, 2, 3, 4, 6, 8, 9, 10, 11, 12, 13, 15, 16, 17, 19, 20, 21, 22, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 62, 63, 64, 65, 67, 69, 70, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 145, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 163, 164, 165, 166, 167, 168, 169, 170, 171, 173, 175, 176, 177, 178, 179, 180, 181, 182, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 203, 205, 206, 207, 208, 209, 210, 211, 212, 213, 215, 217, 218, 219, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 257, 259, 261, 262, 263, 265, 266, 267, 268, 269, 270, 271, 273, 274, 275, 276, 277, 278, 279, 280, 281, 283, 285, 286, 287, 288, 290, 291, 292, 293, 294, 297, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 323, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 346, 347, 348, 349, 350, 351, 352, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 380, 381, 382, 384, 385, 387, 388, 389, 390, 391, 392, 394, 395, 397, 398, 399, 401, 402, 403, 404, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 417, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 432, 433, 435, 436, 437, 438, 439, 440, 441, 445, 446, 447, 448, 449, 450, 451, 452, 454, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 469, 471, 472, 473, 474, 475, 476, 477, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 501, 502, 503, 504, 505, 506, 508, 509, 510, 511], "model.language_model.layers.14": [0, 1, 3, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 96, 97, 98, 99, 101, 102, 103, 104, 105, 107, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 130, 131, 132, 133, 134, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 149, 150, 151, 152, 153, 154, 156, 158, 159, 160, 161, 162, 164, 166, 167, 169, 170, 171, 172, 173, 174, 175, 177, 178, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 218, 219, 220, 221, 223, 224, 225, 226, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 256, 257, 258, 259, 260, 261, 262, 263, 265, 266, 267, 268, 270, 272, 273, 274, 275, 276, 277, 278, 279, 280, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 325, 326, 328, 329, 330, 331, 332, 333, 334, 336, 337, 340, 341, 342, 343, 344, 345, 346, 348, 349, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 393, 394, 395, 396, 397, 398, 401, 402, 403, 404, 405, 406, 408, 409, 411, 412, 413, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 461, 462, 463, 464, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 481, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 496, 497, 498, 499, 500, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.15": [0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 83, 84, 86, 88, 89, 90, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 139, 140, 143, 144, 146, 147, 148, 149, 150, 151, 152, 153, 154, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 174, 175, 176, 177, 179, 180, 182, 183, 184, 185, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 204, 206, 207, 208, 209, 210, 211, 212, 213, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 244, 245, 247, 248, 249, 250, 251, 252, 254, 255, 256, 258, 259, 262, 263, 264, 265, 266, 267, 268, 270, 271, 272, 273, 274, 275, 277, 278, 279, 280, 281, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 299, 300, 301, 302, 304, 305, 307, 308, 309, 310, 311, 312, 313, 314, 315, 318, 319, 320, 321, 322, 323, 325, 326, 327, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 346, 347, 348, 350, 351, 352, 353, 354, 355, 356, 357, 358, 360, 361, 362, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 404, 406, 407, 408, 409, 410, 412, 414, 415, 416, 417, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 484, 486, 487, 488, 489, 490, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.16": [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 83, 84, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 106, 108, 109, 110, 111, 112, 113, 115, 116, 118, 119, 120, 121, 123, 124, 126, 127, 128, 129, 130, 131, 132, 134, 135, 136, 138, 139, 140, 142, 143, 144, 145, 146, 147, 148, 151, 152, 153, 154, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 249, 250, 251, 252, 253, 254, 255, 256, 258, 259, 260, 262, 263, 264, 265, 266, 267, 270, 271, 272, 273, 274, 275, 276, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 300, 301, 303, 304, 306, 308, 309, 310, 311, 313, 314, 315, 316, 317, 318, 319, 320, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 337, 338, 339, 340, 341, 342, 343, 345, 346, 347, 348, 349, 350, 351, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 407, 408, 409, 410, 411, 412, 413, 414, 418, 420, 421, 422, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 436, 437, 438, 439, 440, 441, 442, 443, 444, 447, 448, 449, 450, 451, 452, 453, 454, 456, 457, 458, 459, 460, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 491, 492, 494, 495, 496, 497, 499, 501, 502, 503, 504, 505, 506, 507, 508, 509, 511], "model.language_model.layers.17": [0, 1, 3, 6, 7, 8, 9, 10, 11, 13, 14, 16, 17, 18, 19, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 108, 109, 110, 111, 112, 113, 114, 116, 117, 118, 119, 120, 121, 122, 123, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 143, 144, 145, 146, 147, 148, 149, 150, 151, 153, 154, 156, 158, 159, 160, 162, 163, 164, 165, 166, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 227, 228, 229, 230, 231, 233, 234, 235, 236, 237, 238, 240, 242, 243, 244, 245, 246, 247, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 309, 310, 311, 312, 313, 314, 315, 317, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 336, 337, 338, 340, 343, 344, 345, 346, 347, 348, 349, 350, 351, 354, 355, 356, 357, 358, 359, 360, 361, 362, 364, 365, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 389, 390, 391, 392, 394, 395, 396, 397, 398, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 412, 413, 415, 416, 418, 419, 420, 422, 423, 425, 426, 427, 428, 429, 430, 431, 432, 433, 435, 436, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 472, 475, 477, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.18": [0, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 42, 43, 44, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 60, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 100, 101, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120, 121, 122, 125, 126, 127, 128, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 176, 178, 179, 180, 181, 182, 183, 184, 185, 188, 189, 190, 192, 193, 194, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 246, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 264, 265, 266, 267, 268, 269, 270, 271, 272, 274, 275, 276, 277, 278, 280, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 295, 296, 297, 298, 299, 300, 302, 303, 304, 305, 306, 307, 308, 309, 310, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 324, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 340, 341, 343, 344, 345, 346, 347, 349, 351, 352, 353, 354, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 389, 390, 391, 392, 394, 395, 396, 397, 398, 399, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 426, 427, 428, 429, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 446, 447, 448, 449, 451, 452, 453, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 477, 478, 479, 480, 481, 482, 484, 485, 486, 487, 488, 491, 492, 493, 494, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 509, 510, 511], "model.language_model.layers.19": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 85, 86, 88, 89, 90, 91, 92, 93, 94, 95, 98, 99, 100, 101, 102, 104, 105, 106, 107, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 165, 166, 167, 168, 169, 170, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 184, 185, 187, 188, 189, 190, 191, 192, 193, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 207, 208, 209, 210, 211, 212, 213, 215, 216, 217, 218, 220, 221, 222, 223, 225, 226, 228, 229, 230, 232, 233, 234, 236, 237, 238, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 265, 267, 268, 269, 270, 271, 273, 275, 276, 277, 278, 279, 280, 281, 282, 284, 285, 286, 288, 289, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 324, 325, 326, 327, 328, 330, 331, 332, 333, 334, 335, 337, 338, 339, 341, 342, 343, 344, 345, 347, 348, 349, 350, 352, 353, 354, 355, 356, 357, 358, 359, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 405, 406, 407, 408, 409, 411, 412, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 426, 427, 428, 429, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 477, 479, 480, 482, 483, 484, 486, 487, 488, 489, 491, 492, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 510, 511], "model.language_model.layers.20": [2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 81, 82, 83, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 112, 113, 114, 115, 116, 117, 118, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 135, 136, 137, 138, 140, 141, 142, 143, 144, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 166, 167, 168, 170, 171, 172, 173, 174, 175, 176, 177, 178, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 202, 203, 204, 205, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 225, 226, 227, 228, 229, 230, 231, 232, 235, 236, 237, 238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 249, 251, 252, 253, 254, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 270, 271, 272, 273, 274, 275, 277, 278, 280, 281, 282, 283, 284, 285, 287, 288, 289, 290, 291, 293, 295, 296, 297, 298, 300, 302, 303, 304, 305, 306, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 327, 328, 329, 330, 331, 332, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 365, 366, 368, 369, 370, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 430, 432, 433, 434, 436, 437, 438, 440, 441, 443, 444, 445, 448, 449, 450, 451, 452, 453, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.21": [0, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 128, 129, 130, 131, 132, 133, 134, 135, 136, 138, 139, 141, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 177, 178, 179, 182, 183, 185, 186, 188, 189, 190, 191, 192, 194, 195, 196, 197, 198, 199, 200, 201, 202, 204, 205, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 233, 234, 236, 237, 238, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 263, 264, 266, 267, 268, 269, 270, 271, 273, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 301, 302, 304, 305, 306, 307, 308, 309, 310, 311, 312, 314, 316, 317, 318, 320, 321, 322, 323, 324, 325, 326, 327, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 343, 344, 345, 347, 348, 349, 350, 351, 352, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 366, 367, 368, 369, 370, 372, 373, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 388, 389, 391, 392, 393, 394, 395, 396, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 421, 422, 424, 425, 426, 427, 429, 431, 432, 433, 434, 435, 438, 439, 440, 441, 442, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 508, 509, 510, 511], "model.language_model.layers.22": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 118, 120, 121, 123, 124, 125, 126, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 143, 145, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 195, 196, 197, 198, 199, 200, 201, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 214, 215, 216, 217, 218, 219, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 260, 261, 262, 264, 265, 269, 270, 273, 274, 275, 276, 277, 279, 280, 281, 282, 283, 284, 285, 287, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 315, 316, 317, 318, 319, 320, 321, 322, 324, 325, 326, 327, 328, 329, 330, 331, 332, 334, 336, 338, 339, 340, 342, 343, 344, 345, 346, 347, 348, 349, 350, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 366, 368, 369, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 404, 405, 406, 407, 408, 409, 410, 411, 413, 414, 415, 416, 417, 418, 419, 420, 422, 423, 424, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 440, 441, 442, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 456, 457, 458, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 479, 481, 482, 483, 485, 487, 489, 490, 491, 492, 493, 494, 495, 496, 498, 499, 500, 501, 503, 504, 505, 506, 508, 509, 511], "model.language_model.layers.23": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 35, 36, 39, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 137, 138, 139, 141, 142, 144, 145, 146, 149, 150, 151, 152, 153, 154, 155, 156, 157, 159, 160, 161, 162, 163, 164, 165, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 198, 199, 200, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 217, 219, 220, 221, 222, 224, 226, 227, 228, 229, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 262, 263, 264, 265, 266, 267, 270, 271, 272, 275, 276, 278, 279, 280, 281, 282, 284, 285, 286, 289, 290, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 315, 316, 317, 318, 319, 321, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 356, 357, 358, 359, 360, 361, 363, 364, 365, 366, 367, 368, 369, 372, 373, 374, 375, 377, 379, 380, 381, 382, 383, 384, 385, 386, 388, 389, 390, 391, 393, 394, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 456, 457, 458, 459, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 488, 490, 491, 492, 493, 494, 495, 496, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.24": [0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 17, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 57, 59, 60, 61, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117, 119, 120, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 154, 155, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 184, 186, 187, 188, 189, 191, 192, 193, 194, 195, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 224, 225, 227, 228, 229, 230, 231, 232, 233, 234, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 281, 282, 283, 284, 286, 287, 290, 291, 292, 293, 294, 295, 296, 297, 298, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 360, 361, 363, 364, 365, 366, 367, 368, 369, 370, 371, 373, 374, 375, 376, 377, 378, 381, 383, 384, 386, 387, 388, 389, 390, 392, 393, 394, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 416, 417, 418, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 436, 437, 439, 440, 441, 442, 443, 445, 446, 447, 448, 449, 451, 452, 453, 454, 455, 456, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 477, 478, 479, 480, 481, 482, 484, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 503, 504, 505, 507, 508, 509, 510, 511], "model.language_model.layers.25": [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 14, 15, 16, 17, 19, 20, 21, 23, 24, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 105, 106, 107, 108, 109, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 125, 126, 127, 128, 129, 130, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 144, 145, 146, 148, 149, 152, 153, 154, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 177, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 210, 211, 212, 213, 214, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 227, 228, 229, 231, 232, 233, 234, 235, 236, 237, 239, 240, 241, 242, 243, 244, 245, 246, 247, 249, 250, 252, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272, 273, 274, 275, 276, 277, 278, 280, 282, 284, 285, 286, 287, 288, 289, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 325, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 344, 345, 346, 347, 348, 349, 350, 351, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 459, 460, 461, 462, 463, 464, 466, 470, 471, 472, 473, 474, 475, 476, 477, 478, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 497, 498, 499, 500, 501, 502, 503, 505, 506, 508, 510, 511], "model.language_model.layers.26": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 38, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 57, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 71, 72, 73, 74, 75, 76, 77, 78, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 95, 97, 98, 99, 100, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 119, 120, 121, 122, 123, 124, 126, 128, 130, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 145, 146, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 188, 189, 190, 191, 193, 194, 195, 196, 197, 198, 199, 200, 202, 203, 204, 205, 206, 207, 208, 209, 211, 212, 213, 214, 215, 216, 217, 218, 220, 221, 222, 223, 224, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 284, 285, 286, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 343, 345, 346, 347, 348, 349, 351, 352, 353, 354, 355, 356, 357, 358, 360, 361, 362, 363, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 426, 427, 428, 430, 432, 433, 434, 435, 436, 437, 440, 441, 442, 443, 444, 445, 446, 447, 448, 450, 451, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 477, 478, 479, 480, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 497, 500, 501, 502, 503, 504, 506, 507, 508, 509], "model.language_model.layers.27": [0, 1, 2, 3, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 28, 30, 33, 34, 35, 36, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 71, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 92, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 131, 132, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 146, 147, 148, 149, 150, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 191, 192, 193, 194, 195, 196, 197, 198, 203, 204, 205, 206, 207, 208, 209, 210, 211, 213, 214, 215, 216, 217, 218, 220, 221, 222, 223, 224, 225, 226, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 241, 242, 243, 244, 246, 247, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 260, 261, 262, 263, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 281, 282, 283, 284, 285, 287, 288, 289, 290, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 308, 309, 310, 311, 312, 313, 314, 315, 316, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 339, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 356, 357, 358, 359, 360, 361, 364, 365, 366, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 395, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 412, 413, 414, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 434, 436, 437, 438, 439, 440, 442, 443, 444, 445, 446, 448, 449, 451, 452, 453, 454, 455, 457, 458, 459, 460, 461, 462, 463, 464, 465, 467, 468, 469, 470, 471, 473, 474, 475, 476, 478, 479, 480, 481, 483, 484, 485, 486, 487, 488, 490, 491, 492, 493, 494, 495, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.28": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 44, 45, 46, 48, 50, 51, 52, 53, 54, 55, 57, 59, 60, 61, 62, 63, 64, 65, 66, 67, 70, 71, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 106, 107, 108, 109, 110, 111, 112, 113, 114, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 130, 133, 134, 135, 136, 137, 138, 139, 140, 141, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 198, 200, 202, 203, 204, 205, 206, 207, 208, 209, 211, 213, 215, 216, 217, 219, 220, 221, 223, 224, 225, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 246, 248, 249, 250, 251, 252, 253, 254, 256, 257, 258, 259, 261, 262, 263, 264, 265, 266, 267, 268, 269, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 320, 321, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 338, 339, 340, 341, 342, 343, 345, 346, 347, 348, 349, 350, 351, 352, 353, 355, 356, 358, 359, 360, 361, 362, 363, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 383, 384, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 403, 404, 405, 406, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 421, 423, 424, 425, 426, 427, 428, 429, 430, 432, 433, 434, 435, 438, 440, 442, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 478, 479, 480, 482, 484, 485, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.29": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 69, 70, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117, 118, 119, 120, 121, 122, 123, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 190, 191, 193, 194, 195, 196, 197, 198, 199, 202, 203, 204, 205, 207, 208, 209, 210, 211, 212, 213, 214, 215, 217, 218, 219, 221, 222, 223, 224, 225, 226, 228, 229, 230, 231, 232, 233, 234, 235, 237, 238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 249, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 263, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 286, 287, 288, 290, 291, 292, 293, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 312, 313, 314, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 339, 341, 342, 343, 344, 346, 347, 348, 349, 350, 351, 352, 353, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 375, 376, 377, 378, 379, 380, 381, 382, 384, 385, 387, 388, 389, 390, 391, 392, 394, 395, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 410, 411, 412, 413, 414, 415, 416, 417, 419, 420, 421, 422, 423, 425, 426, 427, 428, 429, 432, 433, 435, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 469, 470, 471, 472, 473, 474, 475, 477, 479, 481, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 508, 510], "model.language_model.layers.30": [1, 3, 5, 6, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 34, 36, 37, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 50, 51, 52, 53, 54, 55, 56, 57, 58, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 95, 96, 97, 98, 99, 102, 103, 104, 105, 106, 107, 109, 110, 111, 112, 114, 115, 116, 117, 118, 119, 120, 121, 123, 124, 125, 126, 127, 128, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 145, 146, 147, 149, 150, 151, 152, 153, 154, 155, 156, 158, 159, 160, 161, 162, 164, 166, 167, 169, 170, 171, 173, 174, 175, 177, 178, 179, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 194, 195, 196, 197, 198, 199, 200, 201, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 218, 219, 220, 221, 222, 223, 225, 226, 227, 228, 229, 230, 231, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 246, 247, 248, 249, 250, 251, 252, 253, 254, 256, 258, 259, 260, 261, 262, 263, 264, 267, 268, 269, 270, 271, 273, 274, 275, 276, 277, 278, 279, 280, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 340, 341, 342, 343, 344, 345, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 401, 402, 403, 404, 405, 406, 408, 409, 411, 412, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 441, 443, 444, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 481, 483, 484, 485, 486, 487, 489, 491, 493, 494, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 511], "model.language_model.layers.31": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 105, 106, 108, 110, 111, 112, 114, 115, 116, 117, 118, 120, 121, 124, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 182, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 216, 217, 218, 219, 220, 221, 222, 223, 224, 226, 227, 228, 229, 231, 233, 234, 235, 237, 238, 239, 240, 241, 245, 247, 248, 249, 250, 252, 253, 254, 255, 256, 258, 260, 261, 262, 263, 264, 265, 266, 267, 268, 270, 271, 272, 273, 274, 275, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 313, 314, 315, 318, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 346, 347, 348, 350, 351, 352, 353, 356, 357, 358, 360, 361, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 410, 411, 412, 413, 415, 416, 417, 418, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 443, 444, 445, 446, 447, 449, 450, 451, 452, 453, 454, 455, 456, 457, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 478, 479, 480, 481, 482, 483, 484, 486, 487, 489, 490, 491, 492, 493, 494, 495, 496, 497, 499, 500, 501, 502, 503, 505, 506, 507, 509, 510, 511], "model.language_model.layers.32": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 68, 70, 71, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 83, 84, 85, 87, 88, 90, 91, 92, 93, 94, 95, 96, 98, 99, 100, 101, 102, 106, 108, 109, 110, 111, 112, 113, 115, 116, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 178, 179, 180, 181, 182, 183, 184, 187, 188, 189, 190, 191, 192, 193, 194, 195, 197, 198, 199, 200, 201, 203, 204, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 230, 231, 232, 233, 234, 235, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 249, 250, 251, 252, 253, 254, 255, 256, 258, 259, 260, 261, 262, 263, 265, 266, 267, 269, 270, 271, 272, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 299, 300, 303, 304, 306, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 337, 338, 340, 341, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 389, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 413, 414, 415, 416, 419, 421, 422, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 436, 437, 438, 439, 440, 441, 442, 443, 444, 446, 447, 448, 449, 451, 452, 453, 454, 455, 459, 460, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 486, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 510, 511], "model.language_model.layers.33": [0, 1, 3, 4, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 22, 23, 24, 25, 27, 30, 31, 33, 34, 35, 38, 39, 40, 41, 42, 43, 44, 47, 49, 50, 51, 52, 53, 54, 55, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 93, 94, 95, 96, 97, 98, 99, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 153, 156, 157, 158, 159, 160, 162, 163, 164, 165, 166, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 204, 205, 206, 207, 208, 209, 210, 211, 214, 215, 216, 218, 219, 220, 222, 223, 224, 225, 227, 228, 230, 231, 232, 233, 234, 235, 236, 237, 238, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272, 273, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 312, 313, 314, 315, 316, 317, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 336, 337, 338, 340, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 354, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 394, 395, 396, 398, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 418, 419, 420, 422, 423, 425, 426, 427, 428, 429, 430, 431, 432, 433, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 452, 453, 454, 455, 456, 457, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 477, 478, 479, 480, 482, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.34": [0, 1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 32, 33, 34, 35, 36, 37, 38, 39, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 62, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 79, 80, 81, 82, 83, 84, 85, 87, 88, 89, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 124, 125, 126, 127, 128, 130, 131, 132, 133, 135, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 189, 190, 191, 192, 193, 194, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 210, 211, 212, 213, 214, 216, 217, 218, 219, 220, 221, 222, 223, 224, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 246, 248, 249, 250, 251, 252, 253, 255, 256, 257, 258, 260, 261, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 275, 276, 277, 278, 279, 280, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 302, 303, 304, 305, 306, 307, 308, 309, 310, 312, 313, 314, 315, 316, 318, 319, 320, 321, 322, 324, 325, 326, 327, 329, 330, 331, 332, 333, 335, 336, 337, 338, 340, 341, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 357, 358, 359, 360, 361, 362, 363, 365, 366, 368, 369, 370, 373, 374, 375, 376, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 430, 431, 432, 433, 434, 435, 437, 438, 439, 440, 441, 442, 443, 444, 447, 448, 449, 450, 451, 453, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 477, 478, 479, 480, 481, 482, 484, 485, 486, 487, 488, 492, 493, 494, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 509, 510, 511], "model.language_model.layers.35": [0, 1, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 86, 88, 89, 90, 91, 92, 93, 94, 96, 99, 100, 101, 102, 103, 104, 105, 106, 107, 109, 110, 112, 113, 114, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 165, 166, 167, 168, 170, 171, 172, 173, 174, 175, 176, 177, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 196, 199, 200, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 220, 221, 222, 223, 225, 226, 228, 230, 231, 233, 234, 237, 238, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 279, 280, 281, 282, 285, 286, 287, 288, 289, 290, 291, 292, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 306, 307, 308, 309, 310, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 347, 348, 349, 350, 352, 353, 354, 355, 356, 357, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 371, 372, 373, 374, 375, 376, 377, 378, 381, 382, 383, 384, 385, 386, 387, 388, 389, 391, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 405, 406, 407, 408, 409, 411, 412, 413, 415, 417, 418, 419, 421, 422, 423, 424, 426, 427, 428, 429, 430, 431, 433, 434, 435, 436, 437, 438, 439, 440, 441, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 481, 482, 483, 484, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 498, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510], "model.language_model.layers.36": [0, 2, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 82, 83, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 112, 113, 114, 115, 116, 117, 118, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 184, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 203, 204, 205, 207, 208, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 260, 261, 262, 263, 264, 265, 266, 267, 269, 271, 272, 273, 274, 275, 276, 277, 278, 281, 282, 283, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 300, 301, 302, 303, 304, 306, 308, 309, 310, 311, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 327, 328, 331, 332, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 351, 352, 353, 354, 355, 356, 358, 359, 360, 361, 362, 363, 366, 367, 368, 369, 370, 371, 372, 373, 374, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 394, 395, 396, 397, 399, 400, 401, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 416, 417, 418, 420, 421, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 451, 452, 454, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 473, 474, 475, 477, 478, 480, 481, 483, 484, 485, 486, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 501, 503, 504, 506, 507, 508, 509, 510, 511], "model.language_model.layers.37": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 95, 96, 97, 98, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 111, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 124, 125, 128, 129, 130, 131, 133, 134, 135, 136, 138, 139, 140, 141, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 158, 159, 160, 162, 163, 164, 165, 166, 167, 168, 169, 170, 172, 173, 175, 178, 179, 180, 181, 182, 183, 185, 186, 187, 188, 189, 191, 192, 194, 195, 196, 197, 198, 199, 200, 202, 203, 204, 205, 206, 208, 210, 211, 212, 213, 214, 215, 216, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 257, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 270, 271, 272, 273, 275, 277, 278, 279, 280, 281, 282, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 316, 318, 320, 321, 322, 323, 324, 325, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 365, 366, 367, 368, 369, 370, 371, 372, 373, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 388, 389, 391, 392, 393, 394, 395, 396, 398, 399, 400, 401, 402, 403, 404, 407, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 424, 425, 426, 427, 429, 432, 433, 434, 435, 436, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 490, 492, 493, 494, 495, 496, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.38": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58, 59, 60, 62, 63, 64, 65, 66, 67, 69, 71, 72, 73, 74, 75, 76, 77, 78, 79, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 99, 100, 101, 102, 104, 105, 106, 107, 109, 110, 111, 112, 113, 114, 115, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 129, 130, 131, 132, 133, 134, 135, 136, 138, 139, 140, 141, 142, 143, 146, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200, 201, 203, 204, 205, 206, 209, 210, 211, 212, 214, 215, 216, 218, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 264, 265, 266, 267, 269, 270, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 287, 289, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 334, 335, 336, 338, 339, 340, 343, 344, 345, 346, 347, 348, 349, 350, 353, 354, 355, 356, 357, 358, 359, 360, 361, 363, 364, 365, 366, 367, 369, 371, 372, 373, 374, 375, 377, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 408, 409, 410, 411, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 440, 441, 442, 443, 444, 445, 446, 447, 449, 450, 451, 452, 453, 454, 456, 457, 458, 459, 460, 461, 462, 463, 465, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 478, 479, 481, 482, 483, 484, 485, 487, 489, 490, 491, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 511], "model.language_model.layers.39": [0, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 15, 16, 17, 19, 21, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 35, 36, 38, 39, 41, 42, 43, 44, 45, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 78, 79, 81, 82, 83, 84, 85, 87, 89, 90, 91, 92, 95, 96, 97, 98, 99, 100, 101, 102, 103, 105, 106, 107, 108, 109, 110, 111, 112, 114, 115, 116, 117, 118, 119, 120, 121, 122, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 217, 218, 219, 220, 221, 222, 223, 224, 226, 227, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 262, 263, 264, 266, 267, 269, 270, 271, 272, 273, 274, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 299, 300, 301, 303, 304, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 320, 321, 322, 323, 324, 326, 327, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 365, 367, 368, 369, 370, 372, 373, 374, 375, 376, 377, 379, 380, 381, 382, 384, 385, 386, 387, 388, 389, 390, 391, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 419, 420, 421, 422, 423, 424, 425, 426, 427, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 445, 446, 447, 449, 450, 451, 452, 453, 454, 456, 457, 458, 459, 461, 463, 464, 465, 466, 467, 468, 470, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 493, 494, 495, 496, 497, 498, 499, 500, 501, 503, 504, 505, 506, 507, 508, 509, 511], "model.language_model.layers.40": [0, 1, 2, 4, 6, 8, 9, 10, 12, 13, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 28, 29, 30, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 63, 64, 65, 66, 68, 69, 70, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 91, 92, 93, 94, 95, 96, 97, 98, 100, 101, 103, 104, 105, 106, 107, 108, 109, 110, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 150, 151, 153, 155, 156, 157, 158, 159, 160, 161, 162, 165, 166, 168, 169, 170, 171, 172, 173, 174, 176, 177, 178, 179, 180, 182, 183, 185, 186, 187, 188, 189, 190, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 230, 231, 232, 233, 234, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 274, 275, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 325, 326, 327, 328, 329, 330, 331, 333, 334, 335, 336, 337, 338, 340, 341, 342, 343, 344, 345, 346, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 370, 373, 374, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 431, 432, 434, 435, 436, 437, 438, 439, 440, 441, 444, 445, 447, 448, 449, 450, 451, 452, 453, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 469, 470, 471, 472, 473, 474, 475, 476, 478, 479, 480, 481, 482, 483, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.41": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 21, 22, 24, 25, 26, 27, 29, 30, 31, 32, 33, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 53, 54, 55, 56, 58, 59, 60, 61, 62, 63, 65, 66, 67, 68, 69, 71, 72, 73, 74, 75, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 123, 124, 125, 126, 127, 128, 129, 130, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 162, 163, 164, 166, 167, 168, 169, 170, 171, 172, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184, 186, 187, 188, 189, 190, 192, 193, 194, 196, 198, 200, 201, 202, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 222, 223, 225, 226, 228, 229, 230, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 243, 244, 245, 246, 247, 248, 250, 251, 252, 253, 254, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 272, 273, 274, 276, 277, 279, 280, 281, 283, 284, 286, 287, 289, 290, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 306, 309, 310, 311, 312, 313, 314, 315, 316, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 388, 389, 390, 391, 392, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 406, 407, 408, 409, 410, 411, 412, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 471, 472, 473, 475, 476, 477, 478, 479, 480, 481, 482, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 507, 508, 509, 510, 511], "model.language_model.layers.42": [0, 1, 2, 3, 5, 6, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 110, 111, 112, 113, 114, 116, 117, 118, 119, 120, 121, 123, 124, 125, 128, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 147, 151, 152, 153, 154, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 179, 180, 181, 182, 184, 185, 187, 188, 189, 190, 192, 193, 194, 197, 198, 199, 200, 201, 202, 203, 204, 205, 207, 208, 210, 211, 212, 213, 214, 215, 216, 217, 218, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 283, 284, 285, 286, 287, 288, 290, 291, 293, 294, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 307, 308, 309, 310, 311, 312, 313, 314, 315, 317, 319, 320, 321, 322, 323, 325, 326, 327, 330, 331, 332, 333, 334, 335, 336, 337, 339, 340, 341, 342, 343, 344, 345, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 393, 394, 396, 397, 398, 399, 400, 401, 402, 403, 404, 407, 408, 409, 410, 411, 413, 415, 416, 417, 418, 419, 420, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 438, 439, 440, 443, 444, 445, 447, 448, 449, 450, 451, 453, 454, 455, 456, 457, 458, 459, 462, 463, 464, 465, 466, 467, 468, 470, 471, 472, 473, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 499, 500, 502, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.43": [0, 1, 2, 3, 4, 5, 6, 8, 9, 10, 12, 13, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 73, 74, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 155, 156, 157, 158, 160, 161, 162, 163, 164, 166, 167, 169, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 185, 187, 188, 189, 190, 193, 194, 195, 198, 199, 201, 203, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 219, 220, 221, 223, 224, 225, 226, 227, 228, 230, 231, 232, 233, 234, 235, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 250, 251, 252, 253, 254, 255, 256, 258, 260, 261, 263, 264, 265, 266, 267, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 284, 285, 286, 287, 289, 290, 291, 292, 293, 294, 296, 297, 298, 299, 300, 301, 302, 303, 304, 306, 307, 308, 309, 310, 311, 313, 314, 315, 316, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 335, 336, 337, 339, 340, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 387, 388, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 430, 431, 434, 435, 436, 437, 438, 439, 441, 442, 444, 445, 446, 447, 448, 450, 451, 452, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 494, 495, 496, 497, 498, 499, 501, 502, 504, 505, 506, 507, 508, 509, 510, 511], "model.language_model.layers.44": [0, 1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 14, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 68, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 127, 128, 129, 130, 131, 132, 134, 135, 136, 137, 139, 140, 141, 142, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 181, 182, 183, 184, 185, 186, 187, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 202, 203, 204, 206, 207, 208, 209, 210, 211, 212, 213, 214, 216, 217, 218, 220, 221, 222, 223, 224, 225, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 269, 270, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 283, 284, 285, 287, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 305, 306, 309, 310, 311, 312, 313, 315, 317, 319, 320, 321, 322, 323, 324, 326, 327, 328, 329, 331, 332, 333, 334, 336, 337, 338, 339, 340, 341, 342, 345, 346, 347, 348, 350, 351, 352, 353, 355, 356, 358, 359, 360, 361, 363, 365, 366, 367, 368, 369, 370, 371, 372, 374, 375, 376, 377, 378, 379, 380, 381, 383, 384, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 454, 455, 456, 457, 458, 459, 461, 462, 463, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 482, 483, 484, 485, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 499, 500, 501, 502, 503, 504, 505, 507, 508, 509, 510, 511], "model.language_model.layers.45": [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 20, 22, 23, 24, 25, 26, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 86, 87, 89, 90, 93, 94, 97, 98, 99, 100, 101, 102, 103, 104, 105, 107, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 188, 189, 191, 193, 194, 195, 196, 198, 199, 200, 201, 202, 203, 204, 205, 206, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 242, 244, 245, 246, 247, 249, 250, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 269, 270, 271, 272, 274, 275, 276, 278, 279, 280, 281, 282, 283, 284, 285, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 315, 316, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 332, 333, 334, 335, 336, 338, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 358, 359, 363, 364, 365, 366, 368, 369, 371, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 389, 390, 391, 392, 393, 394, 395, 396, 397, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 417, 418, 419, 421, 422, 423, 424, 425, 426, 427, 428, 429, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 462, 464, 465, 466, 467, 469, 470, 471, 472, 473, 474, 475, 476, 477, 479, 480, 481, 482, 483, 484, 485, 486, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 508, 509, 511], "model.language_model.layers.46": [0, 1, 2, 3, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 20, 21, 22, 23, 24, 25, 26, 30, 31, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 59, 60, 61, 64, 65, 66, 67, 68, 69, 70, 71, 73, 74, 76, 77, 78, 80, 81, 82, 83, 84, 85, 87, 88, 89, 90, 92, 93, 94, 95, 97, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 161, 162, 163, 164, 165, 166, 167, 169, 170, 171, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 204, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 236, 237, 239, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 274, 275, 276, 277, 278, 280, 281, 282, 283, 284, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 302, 303, 304, 305, 306, 310, 311, 312, 314, 315, 316, 317, 318, 319, 321, 322, 323, 324, 325, 326, 327, 328, 329, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 343, 344, 345, 346, 347, 348, 350, 351, 352, 353, 354, 356, 357, 358, 359, 360, 361, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 397, 398, 399, 401, 402, 403, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 440, 441, 442, 443, 444, 445, 446, 447, 448, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 466, 467, 468, 469, 470, 471, 472, 473, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 489, 490, 491, 492, 493, 494, 495, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 509, 510], "model.language_model.layers.47": [0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 31, 33, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 59, 60, 61, 62, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 82, 83, 84, 85, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 100, 101, 102, 103, 104, 105, 107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 131, 132, 133, 134, 135, 136, 137, 138, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 156, 157, 158, 159, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 197, 199, 200, 201, 202, 203, 204, 205, 206, 207, 209, 210, 211, 212, 213, 215, 216, 217, 218, 220, 221, 222, 223, 224, 225, 226, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 240, 241, 242, 244, 245, 246, 247, 248, 249, 250, 252, 253, 254, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 292, 293, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 346, 347, 349, 350, 351, 352, 354, 355, 356, 357, 358, 360, 361, 362, 363, 364, 365, 366, 368, 369, 370, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 385, 386, 387, 389, 390, 391, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 423, 424, 425, 426, 427, 428, 429, 430, 431, 433, 434, 435, 438, 439, 440, 441, 442, 443, 444, 445, 446, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 461, 462, 463, 464, 465, 466, 467, 469, 470, 471, 472, 474, 475, 476, 477, 478, 479, 480, 481, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511], "mtp.layers.0": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 35, 36, 37, 38, 39, 41, 42, 43, 44, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 73, 74, 76, 78, 79, 80, 81, 82, 83, 84, 86, 87, 88, 89, 90, 91, 92, 93, 94, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 139, 140, 141, 142, 143, 145, 146, 147, 148, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 201, 202, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 224, 225, 226, 227, 228, 229, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 270, 272, 273, 274, 275, 276, 278, 279, 280, 281, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 296, 298, 299, 300, 301, 303, 304, 305, 306, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 352, 353, 354, 355, 357, 359, 360, 361, 362, 363, 364, 366, 367, 368, 369, 370, 372, 373, 374, 375, 376, 377, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 423, 424, 425, 427, 429, 430, 431, 432, 433, 435, 436, 437, 438, 439, 440, 441, 442, 443, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 466, 468, 469, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 485, 486, 487, 488, 489, 490, 491, 492, 493, 495, 496, 497, 498, 499, 500, 501, 503, 504, 508, 509, 510, 511]}\', \'src/qwen/vllm_patch/vllm_reap.py\': \'"""ARC_VLLM_REAP_KEEP=<keep.json>: load-time REAP expert pruning for Qwen3.8-Flash-Next served by vLLM (the halt95\\nMerlin engine, vLLM 0.30.0 + its patch series), the counterpart of ARC_SGLANG_REAP_KEEP.\\n\\nvLLM hands both loaders the checkpoint\\\'s own tensor names, so the SGLang wrapper (sitecustomize.patch_reap_keep: drop the\\npruned experts, renumber the kept ones, slice each router to the kept rows, as src.qwen.reap.prune writes them) applies\\nunchanged; this module only names vLLM\\\'s classes. The target\\\'s config.json and the drafter\\\'s must both declare the kept\\nexpert count, as src.qwen.flashnext --keep writes them (served with --speculative-config \\\'{"method": "mtp", "model":\\n<assembly>/drafter, ...}\\\'). The repo root must be on the server\\\'s PYTHONPATH for src.qwen.reap.\\n"""\\n\\nTARGET = ("vllm.models.qwen4_exp.nvidia.model", "Qwen4ExpForConditionalGeneration")\\nDRAFTER = ("vllm.models.qwen4_exp.nvidia.mtp", "Qwen4ExpMTP")\\n\\n\\ndef patches(keep: str, patch_reap_keep) -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook; patch_reap_keep is sitecustomize\\\'s wrapper."""\\n    return {module: (lambda loaded, name=name: patch_reap_keep(loaded, keep, name)) for module, name in (TARGET, DRAFTER)}\\n\', \'kaggle/qwen-transductive/h95/serve_l4.sh\': \'#!/usr/bin/env bash\\n# Serve Qwen3.8-Flash-Next on 4 GPUs of ~22.5 GiB (Kaggle 4x L4, or a larger GPU behind a memory ballast) with halt95\\\'s\\n# Merlin engine (vllm-0.30.0+merlin252.sm89: vLLM 0.30.0 + the v2.5.2 patch series, built for sm89; Kaggle installs it\\n# from the qwen-merlin-h95-wheels wheelhouse, build_wheelhouse.sh): halt95\\\'s serve-v2.5.sh (v2.5.2) shape and\\n# environment, adapted to our files and to sm89:\\n#   - our files: Intel\\\'s W4A16 AutoRound target + the FP8 n-gram table, assembled by src.qwen.flashnext --keep (config\\n#     declares 448 experts), pruned while loading by ARC_VLLM_REAP_KEEP; the assembled albucino int4 g32 MTP drafter,\\n#     given to vLLM as a separate draft model (its config declares 448 too, so it is pruned with the keep list\\\'s\\n#     mtp.layers.0 entry, as on SGLang)\\n#   - FP8 KV at scale 1.0 (no sidecar), as our SGLang path; SCALES=<sidecar json> applies static per-layer scales\\n#   - --language-model-only (no vision tower), CUDA graphs up to 64 tokens (16 sequences x 4 verify tokens),\\n#     --max-model-len 131072 (the Kaggle runtime\\\'s), --kv-cache-memory-bytes per rank instead of halt95\\\'s 4.1e9 pin\\n#   - no tool/chat/sampling server defaults: the client sends its own\\n# Env knobs: VENV (the env holding the wheel), REPO (this repository: src/qwen/vllm_patch and src.qwen.reap), MODEL,\\n# DRAFTER, KEEP, KV_BYTES (per rank), UTIL (vLLM\\\'s start check, a share of the whole card: ~0.95 on an L4, ~0.45 on\\n# 47.5 GiB cards behind a 25.1 GiB ballast), MAX_SEQS, SIZES, PORT, CACHE_ROOT, PP_SPLIT, SCALES, RETENTION, BSS,\\n# KV_DTYPE (bfloat16 on other GPUs: the QSA FP8 KV path is built for sm86/sm89), DP (2: data-parallel attention, two\\n# engine replicas of TP1 x PP2 sharing the experts through EP; MAX_SEQS and KV_BYTES are then per replica / rank),\\n# TP4 (1: TP4 x EP4 without pipeline parallelism, every GPU holding all 48 layers; PP_SPLIT then defaults to 48),\\n# SSM_DTYPE (e.g. bfloat16: the GDN recurrent state\\\'s dtype; unset keeps the engine\\\'s float32).\\nset -euo pipefail\\nVENV=${VENV:?the env with vllm-0.30.0+merlin252.sm89}\\nREPO=${REPO:?repository root}\\nMODEL=${MODEL:?assembled target dir (src.qwen.flashnext --keep ... -> <out>/model)}\\nDRAFTER=${DRAFTER:?assembled drafter dir (<out>/drafter)}\\nKEEP=${KEEP:?REAP keep list json}\\nKV_BYTES=${KV_BYTES:-4300000000}\\nUTIL=${UTIL:-0.95}\\nMAX_SEQS=${MAX_SEQS:-16}\\nPORT=${PORT:-8580}\\nCACHE_ROOT=${CACHE_ROOT:-$PWD/.vllm-cache-merlin}\\n# torch.compile\\\'s cache key does not see ARC_VLLM_FP8_DENSE\\\'s post-load conversion, and a graph compiled for the\\n# bfloat16 weights fails its shape checks on the FP8 ones: separate caches, and another one when\\n# ARC_VLLM_FP8_DENSE_DRAFTER=1 converts the drafter too (its graphs share the cache), and one for the drafter alone\\n# (ARC_VLLM_FP8_DENSE unset and ARC_VLLM_FP8_DENSE_DRAFTER=<mode>, as sitecustomize reads them; =0 is the bfloat16 fallback)\\nDENSE_TAG=; [ "${ARC_VLLM_FP8_DENSE_DRAFTER:-0}" != 1 ] || DENSE_TAG=-drafter\\ncase "${ARC_VLLM_FP8_DENSE:-0}" in 0|"") ;; 1) CACHE_ROOT=$CACHE_ROOT/fp8-dense$DENSE_TAG ;; *) CACHE_ROOT=$CACHE_ROOT/dense-$ARC_VLLM_FP8_DENSE$DENSE_TAG ;; esac\\nif [ -z "${ARC_VLLM_FP8_DENSE+set}" ] && [ "${ARC_VLLM_FP8_DENSE_DRAFTER:-0}" != 0 ]; then CACHE_ROOT=$CACHE_ROOT/drafter-$ARC_VLLM_FP8_DENSE_DRAFTER; fi\\n# nor the DFlash2 drafter\\\'s draft length: a K 7 server loaded K 3\\\'s drafter graph and failed its shape checks (16 x 7 rows\\n# against 16 x 3), so each DFLASH2 draft length keeps its own cache\\n[ -z "${DFLASH2:-}" ] || CACHE_ROOT=$CACHE_ROOT/dflash2-k${MTP_K:-3}\\n[ ! -d "$PWD/vllm" ] || { echo "start from a directory without a vllm/ beside it (it would shadow the installed engine)"; exit 1; }\\n\\nCU=$("$VENV/bin/python" -c \\\'import sysconfig; print(sysconfig.get_paths()["purelib"])\\\')/nvidia/cu13\\nexport CUDA_HOME=$CU PATH="$CU/bin:$VENV/bin:$PATH"  # FlashInfer and Triton JIT on the first requests (needs cc + Python.h)\\n# FlashInfer JIT links with -lcudart, and the CUDA 13 wheel ships only libcudart.so.13\\nmkdir -p "$CACHE_ROOT/cudalink" && ln -sf "$CU/lib/libcudart.so.13" "$CACHE_ROOT/cudalink/libcudart.so"\\nexport LIBRARY_PATH="$CACHE_ROOT/cudalink:$CU/lib${LIBRARY_PATH:+:$LIBRARY_PATH}"\\nexport PYTHONPATH="$REPO/src/qwen/vllm_patch:$REPO" ARC_VLLM_REAP_KEEP=$KEEP\\nexport VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0 CUDA_DEVICE_ORDER=PCI_BUS_ID\\nexport VLLM_SKIP_P2P_CHECK=1 NCCL_P2P_LEVEL=SYS NCCL_PROTO=LL  # P2P is on between all four L4s\\n[ "${TP4:-0}" != 1 ] || PP_SPLIT=${PP_SPLIT:-48}\\nexport VLLM_PP_LAYER_PARTITION=${PP_SPLIT:-25,23}\\nfor v in $(env | sed -n \\\'s/^\\\\(MERLIN_[A-Z0-9_]*\\\\)=.*/\\\\1/p\\\'); do unset "$v"; done\\nunset VLLM_PLE_CPU_OFFLOAD VLLM_QSA_KV_SCALES VLLM_QSA_KV_COLLECT VLLM_QSA_KV_CLIP_COUNT VLLM_E2_ALLOW_BREAKABLE \\\\\\n      PYTORCH_CUDA_ALLOC_CONF || true\\n# the PLE offload worker reads the whole checkpoint, which on Kaggle\\\'s network filesystem took from ~10 minutes to over an\\n# hour (10-21: 14 to 190 s per shard): a slow start is waited for, as the runtime\\\'s startup_timeout does\\nexport VLLM_E2_PLE_PULL_TRANSPORT=1 VLLM_PLE_OFFLOAD_READY_TIMEOUT=10800 VLLM_PLE_OFFLOAD_HOME_DEVICE=0\\n# every GPU worker waits until the PLE offload worker has loaded its table before profiling its memory (sitecustomize);\\n# a fresh flag per start, so a restart never finds the previous start\\\'s\\nexport ARC_VLLM_PLE_LOADED_FLAG=${TMPDIR:-/tmp}/arc-ple-loaded.$$; rm -f "$ARC_VLLM_PLE_LOADED_FLAG"\\nexport VLLM_HOST_EMBED_TABLE=1 VLLM_USE_BREAKABLE_CUDAGRAPH=0 VLLM_CACHE_ROOT=$CACHE_ROOT HF_HUB_OFFLINE=1\\nexport MERLIN_FULL_K=1 MERLIN_QSA_SORT=1 MERLIN_TIE_DET=1 MERLIN_TIE_RECENT=1 MERLIN_TOPK_SORTED_EMIT=1\\nexport VLLM_E2_GUARD_MODE=warn VLLM_E2_GUARD_IDLE=1\\nexport VLLM_E2_SIDE_CACHE_LAYOUT=1 VLLM_USE_V2_MODEL_RUNNER=1 VLLM_E2_MAMBA_RETAIN_CHECKPOINT=1\\nif [ -n "${SCALES:-}" ]; then export VLLM_QSA_KV_SCALES=$SCALES; fi\\n# CUDA graphs up to 64 tokens, every verify step (1 + 3 drafts) of 16 requests: an 80-token graph for 20 requests does\\n# not fit an L4 next to KV_BYTES 4.3e9 (out of memory at capture on an L4-sized GPU), so MAX_SEQS stays at 16 by default; SIZES\\n# overrides the capture sizes (e.g. adding 80 for 20 requests next to a smaller KV_BYTES)\\nSIZES=${SIZES:-1,2,3,4,6,8,12,16,24,32,48,64}\\n# RETENTION (tokens, a multiple of the 4096 block): the linear-attention layers also keep a reusable state every RETENTION\\n# tokens while decoding. Unset (the engine\\\'s 0) keeps only prompt-boundary states, so the next free-agent round reuses\\n# its previous prompt but re-prefills everything the previous round generated (measured reuse 0.41 at 64K).\\n# ARC_VLLM_RETAIN_LAST_STATE=1 (src/qwen/vllm_patch/vllm_last_state.py) instead keeps only each request\\\'s newest usable\\n# decode state, so the next round resumes inside the previous round\\\'s output; leave RETENTION unset with it.\\n# BSS=1: batch-sharded sampling, each TP rank samples half the requests from full-vocab rows it gets by an all-to-all\\n# instead of every rank all-gathering every row\\\'s full logits (same sampling; src/qwen/vllm_patch adds the\\n# compute_logits_local that Merlin\\\'s Qwen4ExpForCausalLM lacks)\\nSAMPLING=()\\nif [ "${BSS:-0}" = 1 ]; then export ARC_VLLM_LOCAL_LOGITS=1; SAMPLING=(--enable-batch-sharded-sampling); fi\\n\\n# BLOCK_SIZE (default 4096): tokens per KV block (--block-size). INDEXER_KV_DTYPE (default unset: the engine\\\'s bf16): the\\n# QSA indexer\\\'s compressed-key cache dtype (attention_config indexer_kv_dtype, e.g. fp8: half the index bytes; with\\n# BLOCK_SIZE 8192 a pool block then holds twice the tokens\\\' index, so a request holds about a third fewer blocks)\\n# MTP_K: draft tokens per step (default 3, as deployed). SIZES stays as it is: the V2 model runner rounds each size up to a\\n# multiple of MTP_K + 1 for its uniform decode graphs and drops those above the largest size or MAX_SEQS requests\\n# DFLASH2=<dir> (default off): a DFlash2 drafter (src.qwen.flashnext dflash2: its weights plus the target\\\'s embedding)\\n# drafts MTP_K tokens per step instead of the MTP head, the target capturing its features (ARC_VLLM_DFLASH2,\\n# vllm_dflash2.py); DFLASH2_ATTN names the drafter\\\'s attention backend when the default refuses its non-causal\\n# sliding-window layers. The MTP-only FR-Spec vocabulary (ARC_VLLM_DRAFT_VOCAB) does not apply to it.\\n# REJECTION (default unset: vLLM\\\'s "standard"): the verify step\\\'s rejection_sample_method, e.g. "block" (block verification,\\n# Sun et al. 2024: the drafted tokens are accepted jointly as a block; same output distribution, longer accepted runs)\\nSPEC="{\\\\"method\\\\":\\\\"mtp\\\\",\\\\"model\\\\":\\\\"$DRAFTER\\\\",\\\\"num_speculative_tokens\\\\":${MTP_K:-3},\\\\"draft_sample_method\\\\":\\\\"probabilistic\\\\"${REJECTION:+,\\\\"rejection_sample_method\\\\":\\\\"$REJECTION\\\\"}}"\\nif [ -n "${DFLASH2:-}" ]; then\\n  export ARC_VLLM_DFLASH2=1; unset ARC_VLLM_DRAFT_VOCAB\\n  SPEC="{\\\\"method\\\\":\\\\"dflash\\\\",\\\\"model\\\\":\\\\"$DFLASH2\\\\",\\\\"num_speculative_tokens\\\\":${MTP_K:-3}${DFLASH2_ATTN:+,\\\\"attention_backend\\\\":\\\\"$DFLASH2_ATTN\\\\"}}"\\nfi\\nPARALLEL=(--tensor-parallel-size 2 --pipeline-parallel-size 2 --enable-expert-parallel)\\nif [ "${DP:-1}" = 2 ]; then PARALLEL=(--data-parallel-size 2 --tensor-parallel-size 1 --pipeline-parallel-size 2 --enable-expert-parallel); fi\\nif [ "${TP4:-0}" = 1 ]; then PARALLEL=(--tensor-parallel-size 4 --pipeline-parallel-size 1 --enable-expert-parallel); fi\\n\\nexec "$VENV/bin/vllm" serve "$MODEL" --served-model-name qwen --host 127.0.0.1 --port "$PORT" \\\\\\n  "${PARALLEL[@]}" \\\\\\n  --speculative-config "$SPEC" \\\\\\n  --max-model-len 131072 --max-num-seqs "$MAX_SEQS" --max-num-batched-tokens 1024 \\\\\\n  --gpu-memory-utilization "$UTIL" --kv-cache-memory-bytes "$KV_BYTES" --kv-cache-dtype "${KV_DTYPE:-fp8_e4m3}" \\\\\\n  --language-model-only \\\\\\n  --compilation-config "{\\\\"cudagraph_mode\\\\":\\\\"FULL_AND_PIECEWISE\\\\",\\\\"cudagraph_capture_sizes\\\\":[$SIZES]}" \\\\\\n  --enable-prefix-caching --enable-prompt-tokens-details --no-enable-flashinfer-autotune \\\\\\n  --reasoning-parser qwen3 --shutdown-timeout 60 \\\\\\n  --block-size "${BLOCK_SIZE:-4096}" --use-replayssm --prefix-match-unit 64 \\\\\\n  ${INDEXER_KV_DTYPE:+--attention-config "{\\\\"indexer_kv_dtype\\\\":\\\\"$INDEXER_KV_DTYPE\\\\"}"} \\\\\\n  ${RETENTION:+--prefix-cache-retention-interval "$RETENTION"} ${SSM_DTYPE:+--mamba-ssm-cache-dtype "$SSM_DTYPE"} "${SAMPLING[@]}" \\\\\\n  "$@"\\n\', \'src/qwen/vllm_patch/vllm_gdn_reuse.py\': \'"""ARC_VLLM_GDN_REUSE=1: one GDN RecoverSSM metadata build per step serves every Gated DeltaNet (GDN) layer group of the\\nworker. Each GDN layer is its own KV cache group (18 per worker at PP2), and Merlin ran the full builder for each of them\\nevery step: about half of a tp0 worker\\\'s host time at 20 x 32K decode (py-spy, 10-25). The batch-level fields\\n(row classification, query offsets, token indices, accepted counts) are the same for every group, so the groups after\\nthe first take vLLM\\\'s update_block_table path: re-gather the group\\\'s state indices (MRV2 precomputes the aligned ones\\nfor all groups in one launch) and give the group its own commit (its state indices, its block table, its commit\\ncontext). Merlin keeps RecoverSSM builders off that path because their commit metadata is per group; the commit is\\nrebuilt here per group, the rest of it (spec rows, query offsets, computed tokens) is batch-level.\\n\\nARC_VLLM_GDN_REUSE=check also runs the stock full build (without the aligned precompute) for every reused group, checks\\nthe source group\\\'s aligned indices against the stock gather, and raises on the first field that differs."""\\nimport os\\nfrom dataclasses import fields, is_dataclass, replace\\n\\nimport numpy as np\\nimport torch\\n\\nCHECK = os.environ.get("ARC_VLLM_GDN_REUSE") == "check"\\n_STATS = {"reused": 0, "checked": 0}\\n\\n\\ndef same(a, b, path: str = "metadata") -> None:\\n    """Raise unless a and b hold the same values (tensors by shape, dtype, device and content; dataclasses per field)."""\\n    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):\\n        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)) or a.shape != b.shape or a.dtype != b.dtype \\\\\\n                or a.device != b.device or not torch.equal(a, b):\\n            raise RuntimeError(f"ARC_VLLM_GDN_REUSE=check: {path} differs from the full build "\\n                               f"({a if not isinstance(a, torch.Tensor) else (tuple(a.shape), a.dtype)} vs "\\n                               f"{b if not isinstance(b, torch.Tensor) else (tuple(b.shape), b.dtype)})")\\n    elif is_dataclass(a) and is_dataclass(b):\\n        for f in fields(a):\\n            if f.name != "recoverssm_context":\\n                same(getattr(a, f.name), getattr(b, f.name), f"{path}.{f.name}")\\n    elif isinstance(a, np.ndarray) or isinstance(b, np.ndarray):\\n        if not np.array_equal(a, b):\\n            raise RuntimeError(f"ARC_VLLM_GDN_REUSE=check: {path} differs from the full build")\\n    elif isinstance(a, dict) and isinstance(b, dict) and a.keys() == b.keys():\\n        for key in a:\\n            same(a[key], b[key], f"{path}[{key!r}]")\\n    elif a != b:\\n        raise RuntimeError(f"ARC_VLLM_GDN_REUSE=check: {path} differs from the full build ({a!r} vs {b!r})")\\n\\n\\ndef cloned(value):\\n    """A copy whose tensors no longer alias the builder\\\'s persistent buffers."""\\n    if isinstance(value, torch.Tensor):\\n        return value.clone()\\n    if is_dataclass(value):\\n        return replace(value, **{f.name: cloned(getattr(value, f.name)) for f in fields(value) if f.name != "recoverssm_context"})\\n    if isinstance(value, dict):\\n        return {key: cloned(item) for key, item in value.items()}\\n    return value\\n\\n\\ndef patch_gdn_recoverssm(module) -> None:\\n    cls, base = module.GDNRecoverSSMMetadataBuilder, module.GDNAttentionMetadataBuilder\\n    init, build = cls.__init__, cls.build\\n\\n    def __init__(self, kv_cache_spec, layer_names, vllm_config, device):\\n        init(self, kv_cache_spec, layer_names, vllm_config, device)\\n        # the stock GDN builder\\\'s condition, without its exact-type check\\n        self.supports_update_block_table = vllm_config.use_v2_model_runner and device.type == "cuda"\\n        if self.supports_update_block_table:\\n            self.mamba_aligned_state_indices = None  # opts into MRV2\\\'s aligned-index precompute\\n\\n    def update_block_table(self, metadata, blk_table, slot_mapping):\\n        full = checked_build(self, blk_table, slot_mapping) if CHECK and _LAST else None\\n        meta = base.update_block_table(self, metadata, blk_table, slot_mapping)\\n        source = metadata.recoverssm_commit\\n        if source is None:\\n            meta = replace(meta, recoverssm_commit=None, recoverssm_context=None)\\n        else:\\n            indices = meta.spec_state_indices_tensor[:, :1]  # the checkpoint column, as build() keeps\\n            align = self.vllm_config.cache_config.mamba_cache_mode == "align"\\n            commit = replace(source, state_indices=indices, block_table=blk_table if align else None)\\n            meta = replace(meta, spec_state_indices_tensor=indices, recoverssm_commit=commit,\\n                           recoverssm_context=self._get_context())\\n        _STATS["reused"] += 1\\n        if full is not None:\\n            same(meta, full)\\n            if (meta.recoverssm_context is None) != (full.recoverssm_context is None):\\n                raise RuntimeError("ARC_VLLM_GDN_REUSE=check: recoverssm_context differs from the full build")\\n            _STATS["checked"] += 1\\n            if _STATS["checked"] in (1, 100) or _STATS["checked"] % 10000 == 0:\\n                module.logger.info("ARC_VLLM_GDN_REUSE=check: %d reused group builds equal the full build", _STATS["checked"])\\n        elif _STATS["reused"] == 1:\\n            module.logger.info("ARC_VLLM_GDN_REUSE: GDN RecoverSSM groups share one metadata build per step")\\n        return meta\\n\\n    def checked_build(self, blk_table, slot_mapping):\\n        """The stock full build of this group (its own block table, the stock state-index gather), detached."""\\n        common, args, kwargs = _LAST[0]\\n        aligned, self.mamba_aligned_state_indices = self.mamba_aligned_state_indices, None\\n        try:\\n            full = build(self, 0, common.replace(block_table_tensor=blk_table, slot_mapping=slot_mapping), *args, **kwargs)\\n        finally:\\n            self.mamba_aligned_state_indices = aligned\\n        context = full.recoverssm_context\\n        return replace(cloned(full), recoverssm_context=context)\\n\\n    def checked(self, common_prefix_len, common_attn_metadata, *args, **kwargs):\\n        # a capture build (positional arguments, dummy batch) clears it: the reused groups of a capture are not checked\\n        _LAST[:] = [] if self._capturing else [(common_attn_metadata, args, kwargs)]\\n        if not self._capturing:\\n            aligned = getattr(self, "mamba_aligned_state_indices", None)\\n            if aligned is not None:\\n                from vllm.v1.attention.backends.utils import mamba_get_block_table_tensor\\n                m = common_attn_metadata\\n                same(aligned[: m.num_reqs], mamba_get_block_table_tensor(\\n                    m.block_table_tensor, m.seq_lens, self.kv_cache_spec, self.vllm_config.cache_config.mamba_cache_mode),\\n                    "mamba_aligned_state_indices")\\n        return build(self, common_prefix_len, common_attn_metadata, *args, **kwargs)\\n\\n    cls.__init__ = __init__\\n    cls.update_block_table = update_block_table\\n    if CHECK:\\n        cls.build = checked\\n\\n\\n_LAST: list = []  # check mode: the last full build\\\'s (common metadata, args, kwargs); groups differ only in block table\\n\\n\\ndef patches() -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook."""\\n    return {"vllm.v1.attention.backends.gdn_recoverssm": patch_gdn_recoverssm}\\n\', \'src/qwen/vllm_patch/vllm_late_commit.py\': \'"""ARC_VLLM_LATE_COMMIT=1: under halt95\\\'s Merlin engine (V2 model runner, pipeline parallelism, MTP), the last pipeline\\nstage commits its RecoverSSM states (model_state.postprocess_state: the accepted-count scatter, the GDN / PLE commit\\nlaunches and the align postprocess) after the draft tokens are broadcast to the other stages instead of before the draft\\npass. The first stage waits for that broadcast to start its next step, and the drafter reads none of the commit\\\'s outputs\\n(it has no recurrent layers, its attention metadata is built from the request states, not the model state), so the same\\nkernels run on the same inputs in the same stream, only later: the target\\\'s states and the drafts are unchanged.\\nGPUModelRunner.sample_tokens is wrapped: on the last stage of a pipeline with a speculator, every postprocess_state call\\nthe original makes is recorded and run, in order, when it returns (also when it raises). Other stages and runs without\\npipeline parallelism or a speculator are untouched.\\n"""\\nimport logging\\n\\nlogger = logging.getLogger("vllm_late_commit")\\n\\n\\ndef patch_model_runner(module) -> None:\\n    cls = module.GPUModelRunner\\n    sample = cls.sample_tokens\\n\\n    def sample_tokens(self, grammar_output):\\n        if not (getattr(self, "is_last_pp_rank", False) and getattr(self, "pp_handler", None) is not None\\n                and getattr(self, "speculator", None) is not None):\\n            return sample(self, grammar_output)\\n        state = self.model_state\\n        own = state.__dict__.get("postprocess_state")\\n        commit, deferred = state.postprocess_state, []\\n        state.postprocess_state = lambda *args, **kwargs: deferred.append((args, kwargs))\\n        try:\\n            return sample(self, grammar_output)\\n        finally:\\n            if own is None:\\n                del state.postprocess_state\\n            else:\\n                state.postprocess_state = own\\n            for args, kwargs in deferred:\\n                commit(*args, **kwargs)\\n\\n    cls.sample_tokens = sample_tokens\\n    logger.info("ARC_VLLM_LATE_COMMIT: the last pipeline stage commits RecoverSSM states after the draft broadcast")\\n\\n\\ndef patches() -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook (chained after hooks already registered there)."""\\n    return {"vllm.v1.worker.gpu.model_runner": patch_model_runner}\\n\', \'src/qwen/vllm_patch/vllm_mem_report.py\': \'"""ARC_VLLM_MEM_REPORT=N (a measurement, off by default): each GPU worker logs where its memory went, once after CUDA graph\\ncapture and then every N model steps: the device\\\'s used memory (total - free), PyTorch\\\'s reserved and allocated bytes\\n(now and peak), the reserved bytes no tensor holds (inactive splits: fragmentation the caching allocator keeps), and\\nthe rest of the device\\\'s used memory that PyTorch does not hold (CUDA context, NCCL buffers, cuBLAS workspaces, other\\ndirect cudaMalloc). The KV pool and the weights are inside PyTorch\\\'s allocated bytes.\\n"""\\nimport logging\\nimport os\\n\\nlogger = logging.getLogger("vllm.arc_mem_report")\\n_STATE = {"steps": 0}\\n\\n\\ndef report(tag: str) -> None:\\n    try:\\n        log(tag)\\n    except Exception as error:  # a measurement never takes the server down\\n        logger.warning("memory report %s failed: %r", tag, error)\\n\\n\\ndef log(tag: str) -> None:\\n    import torch\\n\\n    free, total = torch.cuda.mem_get_info()\\n    stats = torch.cuda.memory_stats()\\n    gib = 1 << 30\\n    used, reserved, allocated = total - free, torch.cuda.memory_reserved(), torch.cuda.memory_allocated()\\n    logger.info("memory report %s (device %d): used %.3f GiB of %.3f; torch reserved %.3f (peak %.3f), allocated %.3f "\\n                "(peak %.3f), inactive splits %.3f; not torch %.3f; %d segments", tag, torch.cuda.current_device(),\\n                used / gib, total / gib, reserved / gib, stats.get("reserved_bytes.all.peak", 0) / gib, allocated / gib,\\n                stats.get("allocated_bytes.all.peak", 0) / gib, stats.get("inactive_split_bytes.all.current", 0) / gib,\\n                (used - reserved) / gib, stats.get("segment.all.current", 0))\\n\\n\\ndef patch_model_runner(module) -> None:\\n    every = int(os.environ.get("ARC_VLLM_MEM_REPORT", "0") or 0)\\n    runner = module.GPUModelRunner\\n    capture, execute = runner.capture_model, runner.execute_model\\n\\n    def capture_model(self, *args, **kwargs):\\n        result = capture(self, *args, **kwargs)\\n        if not kwargs.get("profile_only"):\\n            report("after graph capture")\\n        return result\\n\\n    def execute_model(self, *args, **kwargs):\\n        result = execute(self, *args, **kwargs)\\n        _STATE["steps"] += 1\\n        if every > 0 and _STATE["steps"] % every == 0:\\n            report(f"after {_STATE[\\\'steps\\\']} steps")\\n        return result\\n\\n    runner.capture_model, runner.execute_model = capture_model, execute_model\\n\\n\\ndef patches() -> dict:\\n    return {"vllm.v1.worker.gpu.model_runner": patch_model_runner}\\n\', \'src/qwen/vllm_patch/vllm_gdn_park.py\': \'"""ARC_VLLM_GDN_PARK=N (default 0: off): park the align-mode GDN (Mamba) states that requests hold for later turns in\\npinned host memory instead of the GPU pool: N host slots of one pool block each per worker (N x the pool block bytes of\\npinned host memory per GPU; 1,893,376 B with ARC_VLLM_GDN_PACK=tight).\\n\\nUnder VLLM_E2_MAMBA_RETAIN_CHECKPOINT a decoding request keeps, per Mamba group, its live state plus held states for its\\nnext turn: stock\\\'s newest hashed retired state (or its moved tail\\\'s CoW copy), and with ARC_VLLM_RETAIN_LAST_STATE\\n(vllm_last_state.py) also its prompt-tail state and the previous decode state until the newest is usable. On merlin\\n(10 groups) the holds are 10-30 of the ~24-40 blocks such a request takes. With this switch every hold map\\n(HOLD_MAPS) keeps positions but no block: a block put into one is copied to a host slot keyed by its prefix hash and its\\nhold reference is released, so it ages as an ordinary cached free block, and the null block stands in for it. A later\\nprefix lookup that finds no GPU block at a position but finds that position\\\'s states in host slots for every group it\\nasks about takes fresh pool blocks, gives them the hashes and returns them as the hit, with host-to-GPU copies queued\\nfor this step. The copies ride in SchedulerOutput.kv_cache_block_copies with host slot ids at PARK_BASE - slot (apart\\nfrom the QSA host K/V patch\\\'s negative ids) and run before the forward on the step\\\'s stream: restores first (a\\nsame-step copy-on-write may read a restored block), then the GPU copies, then the parks (a parked CoW copy is filled by\\nthis step\\\'s GPU copy). In order per worker, so a parked block is copied out before anything reuses it and a restore\\nlands before any forward reads it. Each copy moves its group\\\'s pages of the block (mamba_group_ranges: the whole block\\nwith ARC_VLLM_GDN_PACK=tight), so a restored state is the parked one bit for bit (RecoverSSM\\\'s replay record lives\\ninside the page; num_speculative_blocks is 0 under RecoverSSM). Host ids carry the slot and the group.\\n\\nA host entry is the states of one prefix position for all its groups (one slot each), evicted together (LRU), so a\\nlookup never restores some of a position\\\'s groups and misses others. The hybrid lookup asks each Mamba spec group once\\nper candidate length, bounded by the full-attention groups\\\' hit, so restores land only where the other groups also hit.\\nNot covered: blocks held by pins (grace pins, waiting turns\\\' pins) and the block-granular lookup path (merlin\\\'s is\\nfine-grained, --prefix-match-unit 64). Register after ARC_VLLM_RETAIN_LAST_STATE (its hold maps must exist).\\nPatched modules: see patches().\\n"""\\nimport logging\\nimport os\\nfrom collections import OrderedDict\\n\\nlogger = logging.getLogger("vllm.arc_gdn_park")\\n\\nPARK_BASE = -(1 << 40)  # host slot s rides as PARK_BASE - s; the QSA host K/V patch\\\'s host ids are -1 - id (> PARK_BASE)\\nSTATS_EVERY = int(os.environ.get("ARC_VLLM_GDN_PARK_STATS", "2000"))  # log the counters every this many parks + restores\\n# Test knobs (off by default): ARC_VLLM_GDN_PARK_DROP=1 (or a file path: while the file exists) takes a parked block\\\'s\\n# hashes off the GPU pool at once, so every later hit on a held state is a restore from the host, each logged ("GDN park:\\n# restored"); ARC_VLLM_GDN_PARK_TIMING=1 times each step\\\'s park / restore copies on the worker (synchronizes).\\nTIMING = os.environ.get("ARC_VLLM_GDN_PARK_TIMING") == "1"\\n\\n\\ndef drop_active() -> bool:\\n    value = os.environ.get("ARC_VLLM_GDN_PARK_DROP", "")\\n    return value == "1" or (value not in ("", "0") and os.path.exists(value))\\n\\n\\ndef park_slots() -> int:\\n    return int(os.environ.get("ARC_VLLM_GDN_PARK", "0") or 0)\\n\\n\\nGROUPS = 256  # a host id carries its slot and the cache group whose pages it holds: PARK_BASE - (slot x GROUPS + group)\\n\\n\\ndef encode_park(gpu_block: int, slot: int, group: int) -> tuple[int, int]:\\n    return gpu_block, PARK_BASE - (slot * GROUPS + group)\\n\\n\\ndef encode_restore(slot: int, group: int, gpu_block: int) -> tuple[int, int]:\\n    return PARK_BASE - (slot * GROUPS + group), gpu_block\\n\\n\\ndef split_park_copies(pairs) -> tuple[list, list[tuple[int, int, int]], list[tuple[int, int, int]]]:\\n    """(other copies as given, parks (GPU block, host slot, group), restores (host slot, group, GPU block))."""\\n    other, parks, restores = [], [], []\\n    for pair in pairs:\\n        src, dst = pair[0], pair[1]\\n        if dst <= PARK_BASE:\\n            parks.append((src, *divmod(PARK_BASE - dst, GROUPS)))\\n        elif src <= PARK_BASE:\\n            restores.append((*divmod(PARK_BASE - src, GROUPS), dst))\\n        else:\\n            other.append(pair)\\n    return other, parks, restores\\n\\n\\nclass ParkStore:\\n    """Engine core: which prefix positions have their Mamba states in which host slots, and this step\\\'s copies."""\\n\\n    def __init__(self, slots: int):\\n        self.free = list(range(slots - 1, -1, -1))\\n        self.entries: OrderedDict = OrderedDict()  # token hash -> {group id: slot}, least recently used first\\n        self.pending: list[tuple[int, int]] = []\\n        self.counts = {"parks": 0, "restores": 0, "evictions": 0, "restore_blocks": 0}\\n\\n    def _take_slot(self) -> int | None:\\n        while not self.free and self.entries:\\n            _, old = self.entries.popitem(last=False)\\n            self.free.extend(old.values())\\n            self.counts["evictions"] += 1\\n        return self.free.pop() if self.free else None\\n\\n    def park(self, token_hash, group_id: int, gpu_block: int) -> bool:\\n        """Queue a copy of gpu_block\\\'s state (the state at token_hash\\\'s position for group_id) into a host slot."""\\n        entry = self.entries.get(token_hash)\\n        if entry is not None and group_id in entry:  # the same position\\\'s state: same bytes, already parked\\n            self.entries.move_to_end(token_hash)\\n            return False\\n        slot = self._take_slot()\\n        if slot is None:\\n            return False\\n        self.entries.setdefault(token_hash, {})[group_id] = slot\\n        self.entries.move_to_end(token_hash)\\n        self.pending.append(encode_park(gpu_block, slot, group_id))\\n        self.counts["parks"] += 1\\n        self._maybe_log()\\n        return True\\n\\n    def slots_for(self, token_hash, group_ids) -> list[int] | None:\\n        entry = self.entries.get(token_hash)\\n        if entry is None or any(g not in entry for g in group_ids):\\n            return None\\n        return [entry[g] for g in group_ids]\\n\\n    def restore(self, block_pool, token_hash, group_ids, num_tokens: int, make_key) -> list | None:\\n        """Fresh pool blocks holding token_hash\\\'s states for group_ids (hashed, cached free until the caller touches\\n        them), or None when the position is not parked for every group or the pool has too few free blocks."""\\n        slots = self.slots_for(token_hash, group_ids)\\n        if slots is None or block_pool.get_num_free_blocks() < len(group_ids):\\n            return None\\n        blocks = block_pool.get_new_blocks(len(group_ids))\\n        for block, group_id, slot in zip(blocks, group_ids, slots):\\n            block_pool._insert_block_hash(make_key(token_hash, group_id), block, num_tokens)\\n            self.pending.append(encode_restore(slot, group_id, block.block_id))\\n        block_pool.free_blocks(blocks)  # hashed: to the free queue\\\'s end, evicted last\\n        self.entries.move_to_end(token_hash)\\n        self.counts["restores"] += 1\\n        self.counts["restore_blocks"] += len(blocks)\\n        if drop_active():\\n            logger.info("GDN park: restored %d groups at %d tokens", len(group_ids), num_tokens)\\n        self._maybe_log()\\n        return blocks\\n\\n    def take(self) -> list[tuple[int, int]]:\\n        pending, self.pending = self.pending, []\\n        return pending\\n\\n    def _maybe_log(self) -> None:\\n        if (self.counts["parks"] + self.counts["restores"]) % STATS_EVERY == 0:\\n            logger.info("GDN park: %s, %d host entries, %d free slots", self.counts, len(self.entries), len(self.free))\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Engine core\\n\\n\\nclass ParkingHolds(dict):\\n    """A Mamba manager\\\'s hold map ({request: (position, block)}) whose entries keep their positions but no GPU block: a\\n    block put in is parked and its hold\\\'s reference released (the table\\\'s reference that became the hold\\\'s, or the\\n    touch the hold took), and the null block stands in for it, so every later release of the hold frees nothing and\\n    comparisons of positions run as before."""\\n\\n    def __init__(self, manager, items=()):\\n        super().__init__()\\n        self.manager = manager\\n        for key, value in dict(items).items():\\n            self[key] = value\\n\\n    def __setitem__(self, key, value) -> None:\\n        position, block = value\\n        manager = self.manager\\n        if not block.is_null:\\n            if block.block_hash is not None:\\n                manager._park.park(manager._park_hash(block.block_hash), manager._park_group(block.block_hash),\\n                                   block.block_id)\\n                if drop_active():  # the identity check: the GPU copy loses its hash even while a request still holds it\\n                    manager.block_pool._maybe_evict_cached_block(block)  # (hash metadata only), so a later hit restores\\n            manager.block_pool.free_blocks([block])\\n            block = manager.block_pool.null_block\\n        super().__setitem__(key, (position, block))\\n\\n\\nHOLD_MAPS = ("_retained_checkpoints", "_arc_previous", "_arc_prompt_state")  # stock retention, vllm_last_state\\\'s two\\n\\n\\ndef patch_single_type_manager(module) -> None:\\n    from vllm.v1.core.kv_cache_utils import get_block_hash, get_group_id, make_block_hash_with_group_id\\n\\n    manager = module.MambaManager\\n    init = manager.__init__\\n    find = manager.find_longest_cache_hit.__func__\\n    slots = park_slots()\\n\\n    def __init__(self, kv_cache_spec, block_pool, **kwargs):\\n        init(self, kv_cache_spec, block_pool, **kwargs)  # after vllm_last_state\\\'s __init__ (its hold maps exist)\\n        self._park = None\\n        if self.mamba_cache_mode == "align" and self.enable_caching and getattr(self, "_retain_checkpoint", False):\\n            store = getattr(block_pool, "gdn_park", None)\\n            if store is None:\\n                store = block_pool.gdn_park = ParkStore(slots)\\n                logger.info("GDN park: %d host slots per worker; Mamba holds are parked on the host (%s)", slots,\\n                            [name for name in HOLD_MAPS if hasattr(self, name)])\\n            self._park, self._park_hash, self._park_group = store, get_block_hash, get_group_id\\n            for name in HOLD_MAPS:\\n                if hasattr(self, name):\\n                    setattr(self, name, ParkingHolds(self, getattr(self, name)))\\n\\n    def find_longest_cache_hit(cls, block_hashes, max_length, kv_cache_group_ids, block_pool, kv_cache_spec,\\n                               drop_eagle_block, alignment_tokens, dcp_world_size=1, pcp_world_size=1):\\n        computed, hit = find(cls, block_hashes, max_length, kv_cache_group_ids, block_pool, kv_cache_spec,\\n                             drop_eagle_block, alignment_tokens, dcp_world_size, pcp_world_size)\\n        store = getattr(block_pool, "gdn_park", None)\\n        block_size = kv_cache_spec.block_size\\n        if store is None or not store.entries or not (alignment_tokens < block_size and block_size % alignment_tokens == 0):\\n            return computed, hit  # merlin\\\'s lookup is fine-grained (--prefix-match-unit 64); the block path is not parked\\n        hashes = module.resolve_block_hashes(block_hashes, block_pool.hash_block_size, block_size,\\n                                             supports_fine_grained_hash_lookup=cls.supports_fine_grained_hash_lookup,\\n                                             alignment_tokens=alignment_tokens)\\n        unit, scale = alignment_tokens, block_size // alignment_tokens\\n        for idx in range(min(max_length // unit, len(hashes)) - 1, -1, -1):\\n            num_tokens = (idx + 1) * unit\\n            if num_tokens <= hit:\\n                break\\n            blocks = store.restore(block_pool, hashes[idx], kv_cache_group_ids, num_tokens, make_block_hash_with_group_id)\\n            if blocks is not None:\\n                return tuple([block_pool.null_block] * (idx // scale) + [block] for block in blocks), num_tokens\\n        return computed, hit\\n\\n    manager.__init__ = __init__\\n    manager.find_longest_cache_hit = classmethod(find_longest_cache_hit)\\n\\n\\ndef patch_kv_cache_manager(module) -> None:\\n    take = module.KVCacheManager.take_kv_cache_block_copies\\n\\n    def take_kv_cache_block_copies(self):\\n        copies, retained = take(self)\\n        store = getattr(self.block_pool, "gdn_park", None)\\n        pairs = store.take() if store is not None else []\\n        if not pairs:\\n            return copies, retained\\n        return [*copies, *(module.KVCacheBlockCopy(src, dst) for src, dst in pairs)], retained\\n\\n    module.KVCacheManager.take_kv_cache_block_copies = take_kv_cache_block_copies\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Worker\\n\\n_WORKER: dict = {}  # "slab": HostSlab\\n\\n\\ndef register_host_memory(tensor) -> None:\\n    """Pin an existing CPU tensor (exact size, unlike pin_memory=True\\\'s power-of-two blocks) and map it for the GPU."""\\n    import torch\\n\\n    error = torch.cuda.cudart().cudaHostRegister(tensor.data_ptr(), tensor.numel() * tensor.element_size(), 1 | 2)\\n    if error != torch.cuda.cudart().cudaError.success:  # cudaHostRegisterPortable | cudaHostRegisterMapped\\n        raise RuntimeError(f"GDN park: cudaHostRegister of {tensor.numel() / 2**30:.2f} GiB failed: {error}")\\n\\n\\ndef mamba_group_ranges(kv_cache_config) -> tuple[dict[int, list[tuple[int, int]]], int, int]:\\n    """({group id: [(start, length), ...]}, block stride, allocation bytes): where each GPU Mamba group\\\'s pages sit in a\\n    pool block on this worker, from the block-outermost KVCacheTensor placement (layer l\\\'s page of block b at offset +\\n    l x layer_stride + b x block_stride; cache groups overlay one allocation), adjacent pages merged. A parked block\\n    copies only its group\\\'s ranges: the whole block with ARC_VLLM_GDN_PACK=tight (two GDN pages fill it), a small part\\n    where the pool also holds K/V pages."""\\n    from vllm.v1.kv_cache_interface import MambaSpec, UniformTypeKVCacheSpecs\\n\\n    page, group_of = {}, {}\\n    for gid, group in enumerate(kv_cache_config.kv_cache_groups):\\n        if getattr(group, "host_resident", False):\\n            continue\\n        spec = group.kv_cache_spec\\n        per = spec.kv_cache_specs if isinstance(spec, UniformTypeKVCacheSpecs) else {n: spec for n in group.layer_names}\\n        for name, layer_spec in per.items():\\n            if isinstance(layer_spec, MambaSpec) and name in group.layer_names:\\n                page[name], group_of[name] = layer_spec.page_size_bytes, gid\\n    ranges: dict[int, list[tuple[int, int]]] = {}\\n    strides, sizes = set(), set()\\n    for tensor in kv_cache_config.kv_cache_tensors:\\n        if getattr(tensor, "host_resident", False):\\n            continue\\n        for i, name in enumerate(tensor.layers):\\n            if name in page:\\n                strides.add(tensor.block_stride)\\n                sizes.add(tensor.size)\\n                ranges.setdefault(group_of[name], []).append((tensor.offset + i * tensor.layer_stride, page[name]))\\n    if not ranges or len(strides) != 1 or len(sizes) != 1:\\n        raise ValueError(f"GDN park: Mamba pages over {len(strides)} block strides / {len(sizes)} allocations")\\n    stride = strides.pop()\\n    merged = {}\\n    for gid, spans in ranges.items():\\n        out = []\\n        for start, length in sorted(spans):\\n            if start < 0 or start + length > stride:\\n                raise ValueError(f"GDN park needs a block-outermost layout: group {gid} bytes {start}+{length} > {stride}")\\n            if out and out[-1][0] + out[-1][1] == start:\\n                out[-1] = (out[-1][0], out[-1][1] + length)\\n            else:\\n                out.append((start, length))\\n        merged[gid] = out\\n    return merged, stride, sizes.pop()\\n\\n\\nclass HostSlab:\\n    """A worker\\\'s parking slab: slots x the largest Mamba group\\\'s pages of a pool block (one KV cache allocation)."""\\n\\n    def __init__(self, kv_caches, num_blocks: int, slots: int, layout):\\n        import torch\\n\\n        self.ranges, stride, size = layout\\n        self.view = None\\n        for cache in kv_caches:\\n            storage = cache.untyped_storage()\\n            if storage.nbytes() != size:\\n                continue\\n            blocks = torch.empty(0, dtype=torch.uint8, device=cache.device)\\n            blocks.set_(storage)\\n            self.view = blocks.view(num_blocks, -1)\\n            break\\n        if self.view is None or self.view.shape[1] != stride:\\n            raise ValueError(f"GDN park: no KV cache storage of {size} B with {stride} B blocks")\\n        width = max(sum(length for _, length in spans) for spans in self.ranges.values())\\n        self.slab = torch.empty((slots, width), dtype=torch.uint8)\\n        register_host_memory(self.slab)\\n        logger.info("GDN park: %d slots x %d B pinned on the host (%.2f GB); group ranges %s", slots, width,\\n                    self.slab.numel() / 1e9, {g: len(r) for g, r in self.ranges.items()})\\n\\n    def apply(self, parks, restores) -> None:\\n        import torch\\n\\n        if TIMING:\\n            start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)\\n            start.record()\\n        moved = 0\\n        # a group with no pages on this worker (pipeline parallel: a merged Mamba group\\\'s layers all on another stage)\\n        # has nothing to copy here\\n        for slot, group, block in restores:\\n            at = 0\\n            for begin, length in self.ranges.get(group, ()):\\n                self.view[block, begin:begin + length].copy_(self.slab[slot, at:at + length], non_blocking=True)\\n                at += length\\n            moved += at\\n        for block, slot, group in parks:\\n            at = 0\\n            for begin, length in self.ranges.get(group, ()):\\n                self.slab[slot, at:at + length].copy_(self.view[block, begin:begin + length], non_blocking=True)\\n                at += length\\n            moved += at\\n        if TIMING:\\n            end.record()\\n            end.synchronize()\\n            logger.info("GDN park copies: %d parks, %d restores, %.1f MB, %.3f ms", len(parks), len(restores),\\n                        moved / 1e6, start.elapsed_time(end))\\n\\n\\ndef patch_model_runner(module) -> None:\\n    cls = module.GPUModelRunner\\n    initialize = cls.initialize_kv_cache\\n    copy_blocks = module.copy_kv_cache_blocks_inplace\\n    slots = park_slots()\\n\\n    def initialize_kv_cache(self, kv_cache_config, *args, **kwargs):\\n        initialize(self, kv_cache_config, *args, **kwargs)\\n        _WORKER["slab"] = HostSlab(self.kv_caches, kv_cache_config.num_blocks, slots,  # after any host-view filter\\n                                   mamba_group_ranges(kv_cache_config))\\n\\n    def copy_kv_cache_blocks_inplace(kv_caches, num_blocks, kv_cache_block_copies):\\n        other, parks, restores = split_park_copies(kv_cache_block_copies or [])\\n        slab = _WORKER.get("slab")\\n        if (parks or restores) and slab is None:\\n            raise RuntimeError("GDN park: host copies without a parking slab")\\n        if restores:\\n            slab.apply([], restores)\\n        copy_blocks(kv_caches, num_blocks, other)\\n        if parks:\\n            slab.apply(parks, [])\\n\\n    cls.initialize_kv_cache = initialize_kv_cache\\n    module.copy_kv_cache_blocks_inplace = copy_kv_cache_blocks_inplace\\n\\n\\ndef patches() -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook (chained after the QSA host K/V hooks on the same modules)."""\\n    return {\\n        "vllm.v1.core.single_type_kv_cache_manager": patch_single_type_manager,\\n        "vllm.v1.core.kv_cache_manager": patch_kv_cache_manager,\\n        "vllm.v1.worker.gpu.model_runner": patch_model_runner,\\n    }\\n\', \'src/qwen/vllm_patch/vllm_draft_vocab.py\': \'"""ARC_VLLM_DRAFT_VOCAB=<vocab.json from src.qwen.draft_vocab>: FR-Spec (frequency-ranked speculative sampling) for\\nvLLM\\\'s MTP drafter. Each draft token is scored on the listed token ids only (16K of the 248K vocabulary cover 99.98% of\\nthe 27B\\\'s ARC completion tokens) instead of the whole vocabulary, which saves most of the drafter\\\'s LM-head read and\\nits softmax and sampling over 248K entries per draft token. The target still verifies on the full vocabulary and the\\ndraft probabilities handed to rejection sampling are the restricted distribution scattered back to full width (zero\\noutside the set), so the output distribution is unchanged; a target token outside the set can only be rejected.\\n\\nEach tensor-parallel rank keeps the listed rows of its own LM-head shard, padded to the largest shard\\\'s count, and the\\nper-rank logits are all-gathered as the full head\\\'s are. Only the "mtp" method with an unquantized LM head is changed;\\nanything else keeps vLLM\\\'s path.\\n\\nA frequency list leans towards low token ids (the deployed 16K list has 16,173 of its 16,340 ids in the first half of the\\nvocabulary), so under TP2 both ranks read 16,173 rows every draft step. With ARC_VLLM_DRAFT_VOCAB_BALANCE=1 the V2\\nspeculator\\\'s ranks all-gather their listed rows once in load_model, before the drafter\\\'s CUDA graphs are captured, and\\neach keeps an equal contiguous part of the sorted ids instead (8,170 rows per rank). The full-width row holds the same\\ntokens\\\' logits from the same weights, and the cached draft logits that rejection sampling reads are the ones the draft\\ntoken was sampled from, so the output distribution stays exact; the draft logits themselves match only up to the GEMM\\\'s\\nrounding at the new shape (cuBLAS picks its kernel by shape: bf16, 2,560 wide, on an sm89 RTX 4070, under 0.2% of the\\nlogits move by one bf16 step at 1-16 hidden-state rows per GEMM, none at 24-64, about a quarter at 96 or more). It is opt-in for that reason (a\\nnear-tied draft token can flip, so a run is not token-identical to one without it, and the saving is to be measured on\\nL4 first) and because it adds a load-time collective that briefly holds every rank\\\'s padded rows on every rank (the gathered\\nrows are about 166 MB for the deployed list and the build peaks at about 375 MB, all freed before the KV cache is sized). The V1 proposer builds its head lazily at the first draft sample and keeps the padded\\nsplit.\\n\\nARC_VLLM_DRAFT_VOCAB_SAMPLE=restricted (V2 speculator, probabilistic drafts): the drafts are also sampled on the listed\\ncolumns and only those columns of the cached draft logits are written (the cache starts at -inf), instead of building a\\nfull-width -inf row, sampling over all 248K columns and caching all of them every draft step. Same draft distribution\\nand the same cached distribution for rejection sampling; not token-identical to a run without it (the Gumbel noise is\\ndrawn per column index).\\n"""\\nimport json\\nimport logging\\nfrom pathlib import Path\\n\\nimport torch\\n\\nlogger = logging.getLogger("vllm_draft_vocab")\\n\\n\\nclass DraftVocab:\\n    """Restricted LM head of one tensor-parallel rank: weight [m, H] (its part of the listed rows, zero-padded to m), ids\\n    [tp * m] (the token id of every gathered column, 0 for padding), pad [tp * m] (padding columns, masked to -inf)."""\\n\\n    def __init__(self, weight, ids, pad, vocab: int, gather):\\n        self.weight, self.ids, self.pad, self.vocab, self.gather = weight, ids, pad, vocab, gather\\n\\n    @classmethod\\n    def build(cls, head, token_ids: list[int], balance: bool = False):\\n        """balance: split the listed rows evenly across the ranks (a collective, so every rank of the group calls it)."""\\n        weight = getattr(head, "weight", None)\\n        if weight is None or weight.dtype not in (torch.bfloat16, torch.float16, torch.float32):\\n            logger.warning("draft vocab: the drafter\\\'s LM head is quantized or has no plain weight; keeping the full head")\\n            return None\\n        tp, rank = head.tp_size, head.tp_rank\\n        per = head.org_vocab_size_padded // tp\\n        ranges = [(r * per, min((r + 1) * per, head.org_vocab_size)) for r in range(tp)]\\n        indices = getattr(head, "shard_indices", None)\\n        if indices is not None and (indices.org_vocab_start_index, indices.org_vocab_end_index) != ranges[rank]:\\n            logger.warning("draft vocab: unexpected LM-head sharding %s; keeping the full head", ranges[rank])\\n            return None\\n        ids = torch.tensor(sorted(set(token_ids)), dtype=torch.long)\\n        ids = ids[ids < head.org_vocab_size]\\n        shards = [ids[(ids >= start) & (ids < end)] for start, end in ranges]\\n        m = max(len(shard) for shard in shards)\\n        start = ranges[rank][0]\\n        local = torch.zeros(m, weight.shape[1], dtype=weight.dtype, device=weight.device)\\n        local[:len(shards[rank])] = weight[(shards[rank] - start).to(weight.device)]\\n        if tp == 1:\\n            gather = None\\n        elif getattr(head, "parallel_group", None) is not None:\\n            gather = lambda x, dim=-1: head.parallel_group.all_gather(x, dim=dim)\\n        else:\\n            from vllm.distributed import tensor_model_parallel_all_gather\\n            gather = lambda x, dim=-1: tensor_model_parallel_all_gather(x, dim=dim)\\n        balance = balance and gather is not None\\n        if balance:  # every listed row in id order (the shards are ascending ranges), cut into equal contiguous parts\\n            gathered = gather(local, dim=0)\\n            rows = torch.cat([gathered[r * m:r * m + len(shard)] for r, shard in enumerate(shards)])\\n            m = -(-len(ids) // tp)\\n            shards = [ids[r * m:(r + 1) * m] for r in range(tp)]\\n            local = rows.new_zeros(m, weight.shape[1])\\n            local[:len(shards[rank])] = rows[rank * m:rank * m + len(shards[rank])]\\n        columns = torch.zeros(tp * m, dtype=torch.long)\\n        pad = torch.ones(tp * m, dtype=torch.bool)\\n        for r, shard in enumerate(shards):\\n            columns[r * m:r * m + len(shard)] = shard\\n            pad[r * m:r * m + len(shard)] = False\\n        logger.info("draft vocab: %d of %d tokens, %d rows per rank (balanced: %s)", len(ids), head.org_vocab_size, m, balance)\\n        return cls(local, columns.to(weight.device), pad.to(weight.device), head.org_vocab_size, gather)\\n\\n    def logits(self, hidden: torch.Tensor) -> torch.Tensor:\\n        logits = hidden.to(self.weight.dtype) @ self.weight.t()\\n        if self.gather is not None:\\n            logits = self.gather(logits)\\n        return logits.masked_fill(self.pad, float("-inf"))\\n\\n    def full_probs(self, probs: torch.Tensor) -> torch.Tensor:\\n        """The restricted probabilities at their token ids in a full-vocabulary row (padding columns carry 0)."""\\n        full = probs.new_zeros(probs.shape[0], self.vocab)\\n        return full.scatter_add_(1, self.ids.expand(probs.shape[0], -1), probs)\\n\\n\\ndef restricted_logits(state: DraftVocab):\\n    """compute_logits for a drafter: the listed tokens\\\' logits at their ids in a full-width row, -inf elsewhere, so the\\n    caller\\\'s sampling, cached draft logits and rejection sampling are unchanged."""\\n    keep = (~state.pad).nonzero().flatten()\\n    ids = state.ids.index_select(0, keep)\\n\\n    def compute_logits(hidden_states, *args, **kwargs):\\n        logits = state.logits(hidden_states)\\n        full = logits.new_full((logits.shape[0], state.vocab), float("-inf"))\\n        return full.index_copy_(1, ids, logits.index_select(1, keep))\\n\\n    return compute_logits\\n\\n\\ndef restricted_sample(state: DraftVocab, gumbel_sample):\\n    """sample_draft\\\'s probabilistic branch on the listed columns only: Gumbel-max sampling over the restricted logits\\n    (padding columns at -inf), and the cached pre-temperature draft logits that rejection sampling reads written at the\\n    listed token ids alone, the rest of the cache holding the -inf it was filled with. Gumbel-max over the listed columns\\n    has the distribution of Gumbel-max over a full-width row that is -inf elsewhere, so drafts and their cached\\n    distribution are the full-width path\\\'s in distribution (not token for token: the noise is drawn per column index)."""\\n    keep = (~state.pad).nonzero().flatten()\\n    ids = state.ids.index_select(0, keep)\\n\\n    def sample_draft(self, hidden_states, sample_src_positions, idx_mapping, temperature, seeds, draft_step, draft_logits):\\n        logits = state.logits(hidden_states)\\n        columns = gumbel_sample(logits, idx_mapping, temperature, seeds, sample_src_positions, apply_temperature=True,\\n                                is_drafting=True, use_fp64=self.use_fp64_gumbel)\\n        cache = draft_logits.view(-1, draft_logits.shape[-1])\\n        rows = idx_mapping.long() * draft_logits.shape[1] + draft_step.long()\\n        cache[rows.unsqueeze(1), ids.unsqueeze(0)] = logits.index_select(1, keep).to(cache.dtype)\\n        return state.ids[columns]\\n\\n    return sample_draft\\n\\n\\ndef patch_v2_speculator(module, path: str, balance: bool = False, sample: bool = False) -> None:\\n    """vLLM\\\'s V2 model runner (VLLM_USE_V2_MODEL_RUNNER, e.g. halt95\\\'s engine): once DraftModelSpeculator has loaded an\\n    "mtp" drafter, its compute_logits scores only the listed tokens (vllm.v1.worker.gpu.spec_decode.speculator), with\\n    the listed rows split evenly across the tensor-parallel ranks under balance. sample (ARC_VLLM_DRAFT_VOCAB_SAMPLE):\\n    probabilistic drafts are also sampled and cached on the listed columns (restricted_sample), the draft logits cache\\n    filled with -inf, unless an acceptance estimator or a draft watermarker reads the full-width logits."""\\n    token_ids = json.loads(Path(path).read_text())["token_ids"]\\n    cls = module.DraftModelSpeculator\\n    load = cls.load_model\\n\\n    def load_model(self, target_model):\\n        load(self, target_model)\\n        if getattr(self.vllm_config.speculative_config, "method", None) != "mtp":\\n            return\\n        state = DraftVocab.build(self.model.lm_head, token_ids, balance)\\n        if state is None:\\n            return\\n        self.model.compute_logits = restricted_logits(state)\\n        if (sample and self.draft_logits is not None and getattr(self, "acceptance_estimator", None) is None\\n                and getattr(self, "draft_watermarker", None) is None):\\n            self.sample_draft = restricted_sample(state, module.gumbel_sample).__get__(self)\\n            logger.info("draft vocab: probabilistic drafts sampled and cached on the %d listed columns", len(state.ids))\\n\\n    def draft_logits_spec(self, vllm_config):\\n        dtype, fill = spec(self, vllm_config)\\n        return dtype, float("-inf")  # columns restricted_sample never writes hold no draft probability\\n\\n    cls.load_model = load_model\\n    if sample:\\n        spec = cls.draft_logits_spec\\n        cls.draft_logits_spec = draft_logits_spec\\n\\n\\ndef patch_proposer(module, path: str) -> None:\\n    """Wrap LLMBaseProposer\\\'s greedy and probabilistic draft sampling (vllm.v1.spec_decode.llm_base_proposer); a vLLM\\n    without that class (0.30 renamed it, and its V2 runner drafts elsewhere) is left as it is."""\\n    token_ids = json.loads(Path(path).read_text())["token_ids"]\\n    cls = getattr(module, "LLMBaseProposer", None)\\n    if cls is None:\\n        return\\n    greedy, sample = cls._greedy_sample, cls._sample_draft_tokens\\n\\n    def vocab(self):\\n        state = self.__dict__.get("_arc_draft_vocab", False)\\n        if state is False:\\n            usable = self.method == "mtp" and not getattr(self, "use_heterogeneous_vocab", False)\\n            state = DraftVocab.build(self.model.lm_head, token_ids) if usable else None\\n            self._arc_draft_vocab = state\\n        return state\\n\\n    def _greedy_sample(self, hidden_states):\\n        state = vocab(self)\\n        if state is None:\\n            return greedy(self, hidden_states)\\n        return state.ids[state.logits(hidden_states).argmax(dim=-1)]\\n\\n    def _sample_draft_tokens(self, hidden_states, sampling_metadata):\\n        state = vocab(self)\\n        if state is None:\\n            return sample(self, hidden_states, sampling_metadata)\\n        if not self._enable_probabilistic_draft_probs or sampling_metadata.all_greedy:\\n            return self._greedy_sample(hidden_states), None\\n        columns, probs = self._sample_from_logits(state.logits(hidden_states), sampling_metadata)\\n        return state.ids[columns], (None if probs is None else state.full_probs(probs))\\n\\n    cls._greedy_sample = _greedy_sample\\n    cls._sample_draft_tokens = _sample_draft_tokens\\n\\n\\ndef patches(path: str, balance: bool = False, sample: bool = False) -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook."""\\n    return {"vllm.v1.spec_decode.llm_base_proposer": lambda module: patch_proposer(module, path),\\n            "vllm.v1.worker.gpu.spec_decode.speculator": lambda module: patch_v2_speculator(module, path, balance, sample)}\\n\', \'kaggle/qwen-transductive/draft_vocab/qwen38_arc_16384.json\': \'{"token_ids": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 98, 101, 102, 103, 130, 158, 198, 220, 223, 224, 225, 227, 228, 231, 232, 234, 237, 238, 241, 242, 243, 244, 245, 246, 247, 252, 254, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 364, 365, 366, 367, 368, 369, 370, 371, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 385, 386, 387, 388, 389, 390, 391, 394, 395, 396, 397, 399, 400, 401, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 414, 415, 416, 417, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 445, 446, 448, 449, 450, 451, 452, 453, 455, 456, 457, 458, 459, 460, 461, 462, 463, 466, 467, 468, 469, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 481, 484, 485, 486, 487, 488, 491, 492, 494, 495, 496, 498, 499, 500, 501, 504, 505, 506, 507, 508, 509, 511, 512, 513, 514, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 528, 529, 531, 532, 533, 534, 535, 536, 537, 538, 539, 540, 542, 543, 544, 547, 549, 550, 551, 552, 553, 557, 558, 559, 560, 561, 562, 563, 564, 565, 567, 568, 569, 570, 571, 572, 573, 574, 575, 576, 577, 578, 579, 580, 581, 582, 583, 585, 586, 588, 589, 591, 592, 593, 594, 595, 596, 598, 599, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 612, 614, 615, 616, 617, 620, 623, 624, 625, 626, 627, 628, 629, 631, 632, 633, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 647, 648, 650, 653, 654, 656, 658, 660, 662, 663, 664, 665, 668, 670, 671, 673, 674, 675, 677, 678, 680, 681, 682, 683, 684, 685, 687, 688, 689, 692, 694, 695, 697, 698, 701, 702, 704, 706, 708, 709, 710, 712, 713, 715, 716, 721, 722, 723, 724, 727, 728, 729, 730, 732, 733, 734, 735, 736, 737, 738, 740, 741, 742, 744, 745, 746, 747, 748, 749, 750, 751, 752, 754, 755, 757, 758, 759, 760, 763, 764, 766, 767, 768, 769, 770, 771, 773, 776, 777, 778, 779, 781, 783, 784, 785, 786, 787, 788, 790, 791, 792, 793, 794, 795, 796, 799, 801, 803, 804, 805, 806, 809, 811, 813, 814, 815, 818, 819, 821, 822, 823, 824, 826, 828, 829, 830, 831, 832, 833, 835, 836, 837, 839, 840, 841, 842, 843, 847, 848, 850, 851, 852, 853, 854, 855, 856, 859, 861, 863, 864, 866, 867, 868, 869, 871, 873, 874, 878, 879, 880, 882, 883, 884, 886, 887, 888, 889, 890, 892, 893, 894, 895, 896, 897, 898, 899, 900, 901, 902, 905, 906, 907, 908, 911, 912, 913, 914, 916, 917, 919, 924, 925, 926, 927, 928, 930, 931, 932, 933, 935, 936, 938, 940, 942, 943, 944, 945, 946, 947, 948, 950, 951, 956, 957, 958, 961, 962, 963, 965, 966, 969, 972, 974, 975, 977, 978, 981, 984, 985, 986, 988, 989, 991, 992, 995, 997, 998, 1000, 1001, 1002, 1003, 1004, 1005, 1006, 1008, 1009, 1010, 1012, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1025, 1026, 1027, 1028, 1029, 1030, 1035, 1037, 1038, 1039, 1040, 1042, 1044, 1046, 1047, 1048, 1049, 1050, 1052, 1053, 1054, 1055, 1056, 1057, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1070, 1071, 1072, 1073, 1075, 1076, 1077, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1097, 1098, 1100, 1101, 1102, 1103, 1104, 1105, 1106, 1110, 1112, 1113, 1114, 1116, 1117, 1118, 1119, 1121, 1122, 1124, 1126, 1128, 1129, 1131, 1132, 1133, 1134, 1136, 1137, 1139, 1140, 1141, 1142, 1144, 1145, 1146, 1148, 1149, 1150, 1151, 1154, 1156, 1158, 1159, 1160, 1161, 1162, 1163, 1164, 1165, 1167, 1170, 1171, 1172, 1173, 1174, 1175, 1178, 1179, 1180, 1181, 1187, 1188, 1189, 1190, 1193, 1196, 1199, 1200, 1201, 1202, 1203, 1204, 1206, 1208, 1209, 1212, 1213, 1214, 1216, 1218, 1219, 1220, 1221, 1223, 1227, 1228, 1229, 1231, 1234, 1235, 1236, 1237, 1238, 1244, 1245, 1246, 1248, 1249, 1250, 1253, 1254, 1257, 1258, 1262, 1264, 1265, 1266, 1267, 1270, 1272, 1273, 1274, 1277, 1282, 1283, 1284, 1286, 1287, 1288, 1289, 1290, 1292, 1295, 1297, 1298, 1299, 1301, 1302, 1303, 1304, 1305, 1306, 1307, 1308, 1309, 1310, 1311, 1313, 1315, 1318, 1319, 1320, 1321, 1322, 1323, 1324, 1325, 1326, 1327, 1328, 1330, 1331, 1332, 1334, 1335, 1336, 1340, 1342, 1344, 1345, 1347, 1348, 1349, 1350, 1352, 1353, 1355, 1362, 1363, 1364, 1365, 1366, 1367, 1368, 1369, 1371, 1372, 1373, 1375, 1376, 1379, 1380, 1381, 1382, 1386, 1387, 1393, 1394, 1395, 1396, 1402, 1404, 1405, 1406, 1407, 1410, 1411, 1412, 1413, 1414, 1416, 1418, 1420, 1421, 1422, 1423, 1426, 1428, 1429, 1431, 1432, 1433, 1435, 1436, 1438, 1439, 1440, 1441, 1442, 1445, 1446, 1447, 1448, 1449, 1451, 1452, 1454, 1455, 1456, 1457, 1458, 1459, 1460, 1461, 1462, 1465, 1467, 1469, 1471, 1472, 1473, 1474, 1475, 1477, 1478, 1483, 1484, 1485, 1487, 1488, 1489, 1490, 1493, 1494, 1495, 1496, 1498, 1500, 1502, 1503, 1505, 1506, 1508, 1509, 1510, 1511, 1513, 1514, 1518, 1519, 1520, 1521, 1522, 1524, 1525, 1527, 1528, 1529, 1531, 1532, 1534, 1536, 1538, 1539, 1540, 1541, 1542, 1543, 1544, 1546, 1547, 1548, 1549, 1550, 1551, 1552, 1554, 1555, 1557, 1558, 1559, 1560, 1562, 1563, 1564, 1565, 1566, 1569, 1570, 1572, 1573, 1574, 1575, 1576, 1577, 1579, 1581, 1583, 1586, 1587, 1588, 1589, 1590, 1591, 1593, 1595, 1596, 1597, 1599, 1600, 1601, 1602, 1603, 1605, 1608, 1609, 1611, 1612, 1615, 1618, 1619, 1622, 1623, 1625, 1626, 1628, 1630, 1631, 1632, 1633, 1634, 1635, 1636, 1638, 1639, 1640, 1642, 1645, 1646, 1647, 1648, 1650, 1652, 1653, 1654, 1655, 1657, 1660, 1661, 1665, 1666, 1669, 1671, 1672, 1673, 1674, 1675, 1676, 1678, 1680, 1681, 1683, 1684, 1685, 1686, 1687, 1688, 1689, 1690, 1691, 1692, 1693, 1697, 1698, 1701, 1703, 1704, 1705, 1711, 1713, 1715, 1716, 1718, 1719, 1720, 1723, 1724, 1727, 1732, 1733, 1734, 1737, 1740, 1742, 1743, 1745, 1746, 1748, 1752, 1754, 1755, 1756, 1757, 1760, 1761, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1772, 1773, 1774, 1777, 1779, 1780, 1781, 1782, 1783, 1784, 1785, 1786, 1787, 1788, 1789, 1791, 1793, 1795, 1798, 1799, 1801, 1802, 1803, 1804, 1805, 1806, 1808, 1809, 1815, 1816, 1817, 1818, 1819, 1821, 1822, 1824, 1825, 1826, 1827, 1829, 1831, 1832, 1837, 1840, 1841, 1842, 1844, 1845, 1848, 1852, 1853, 1856, 1857, 1859, 1860, 1861, 1865, 1866, 1867, 1868, 1870, 1871, 1874, 1875, 1876, 1877, 1879, 1880, 1881, 1883, 1884, 1888, 1889, 1890, 1891, 1892, 1894, 1896, 1898, 1899, 1900, 1901, 1902, 1906, 1908, 1910, 1911, 1916, 1918, 1919, 1921, 1923, 1925, 1926, 1927, 1928, 1929, 1931, 1934, 1938, 1940, 1941, 1942, 1943, 1944, 1945, 1946, 1947, 1948, 1950, 1953, 1954, 1956, 1957, 1958, 1959, 1960, 1962, 1964, 1965, 1967, 1968, 1970, 1971, 1973, 1979, 1980, 1982, 1985, 1987, 1988, 1989, 1990, 1993, 1994, 1996, 1998, 2002, 2006, 2007, 2009, 2010, 2012, 2014, 2018, 2019, 2020, 2021, 2022, 2025, 2027, 2028, 2029, 2033, 2038, 2046, 2047, 2050, 2052, 2053, 2054, 2058, 2059, 2060, 2064, 2065, 2069, 2070, 2073, 2074, 2076, 2077, 2078, 2080, 2081, 2082, 2083, 2084, 2085, 2086, 2091, 2093, 2094, 2095, 2098, 2099, 2100, 2101, 2103, 2104, 2106, 2107, 2109, 2110, 2112, 2113, 2114, 2116, 2117, 2119, 2120, 2121, 2124, 2126, 2128, 2131, 2134, 2135, 2136, 2137, 2139, 2140, 2143, 2144, 2146, 2147, 2149, 2150, 2153, 2154, 2155, 2156, 2160, 2166, 2167, 2168, 2171, 2173, 2174, 2177, 2178, 2181, 2184, 2185, 2186, 2187, 2188, 2190, 2191, 2192, 2193, 2195, 2197, 2198, 2199, 2200, 2205, 2206, 2209, 2210, 2212, 2213, 2215, 2217, 2218, 2220, 2222, 2224, 2225, 2226, 2229, 2231, 2233, 2234, 2235, 2236, 2238, 2241, 2243, 2248, 2250, 2251, 2252, 2255, 2258, 2259, 2261, 2262, 2264, 2266, 2267, 2270, 2271, 2272, 2274, 2275, 2276, 2277, 2278, 2279, 2280, 2282, 2283, 2285, 2286, 2287, 2291, 2294, 2297, 2299, 2301, 2302, 2303, 2305, 2306, 2307, 2308, 2311, 2313, 2315, 2316, 2319, 2320, 2322, 2324, 2325, 2327, 2330, 2331, 2332, 2333, 2334, 2336, 2338, 2341, 2342, 2343, 2348, 2352, 2353, 2355, 2356, 2357, 2358, 2359, 2361, 2362, 2364, 2369, 2372, 2373, 2374, 2376, 2377, 2379, 2380, 2381, 2382, 2383, 2387, 2388, 2389, 2391, 2392, 2394, 2395, 2396, 2397, 2399, 2400, 2402, 2404, 2405, 2407, 2408, 2413, 2414, 2418, 2420, 2422, 2423, 2424, 2425, 2426, 2427, 2428, 2433, 2434, 2438, 2441, 2442, 2443, 2444, 2445, 2446, 2447, 2450, 2456, 2458, 2460, 2466, 2468, 2471, 2472, 2473, 2474, 2476, 2477, 2479, 2482, 2483, 2484, 2486, 2487, 2488, 2491, 2493, 2495, 2496, 2497, 2498, 2500, 2502, 2503, 2506, 2508, 2512, 2513, 2515, 2517, 2518, 2519, 2521, 2522, 2523, 2524, 2526, 2528, 2530, 2531, 2532, 2533, 2534, 2536, 2537, 2538, 2540, 2541, 2542, 2543, 2545, 2548, 2551, 2552, 2553, 2554, 2559, 2560, 2562, 2564, 2565, 2568, 2569, 2570, 2573, 2574, 2575, 2577, 2579, 2580, 2581, 2582, 2583, 2585, 2586, 2587, 2588, 2589, 2590, 2592, 2594, 2595, 2597, 2598, 2599, 2600, 2603, 2604, 2605, 2607, 2608, 2609, 2610, 2612, 2613, 2614, 2615, 2617, 2621, 2623, 2624, 2625, 2626, 2627, 2630, 2631, 2632, 2634, 2637, 2638, 2639, 2640, 2645, 2647, 2649, 2657, 2658, 2659, 2660, 2663, 2664, 2665, 2666, 2669, 2670, 2674, 2675, 2676, 2677, 2678, 2679, 2680, 2682, 2683, 2686, 2688, 2691, 2693, 2695, 2697, 2699, 2701, 2702, 2706, 2707, 2708, 2709, 2713, 2714, 2715, 2716, 2717, 2719, 2720, 2721, 2723, 2725, 2732, 2735, 2739, 2741, 2742, 2746, 2747, 2748, 2749, 2751, 2752, 2754, 2757, 2758, 2759, 2762, 2763, 2765, 2767, 2768, 2770, 2772, 2773, 2774, 2777, 2779, 2781, 2782, 2784, 2785, 2789, 2790, 2791, 2792, 2794, 2796, 2798, 2799, 2800, 2801, 2803, 2804, 2805, 2806, 2807, 2810, 2814, 2816, 2818, 2820, 2824, 2827, 2829, 2830, 2831, 2835, 2836, 2838, 2839, 2841, 2842, 2843, 2844, 2845, 2846, 2850, 2858, 2867, 2868, 2869, 2870, 2871, 2873, 2874, 2875, 2876, 2878, 2879, 2880, 2881, 2885, 2887, 2888, 2890, 2891, 2892, 2893, 2894, 2895, 2898, 2902, 2903, 2904, 2905, 2907, 2908, 2909, 2910, 2911, 2912, 2918, 2919, 2920, 2921, 2922, 2923, 2924, 2928, 2929, 2930, 2931, 2932, 2935, 2937, 2938, 2939, 2940, 2942, 2943, 2944, 2945, 2948, 2949, 2954, 2955, 2956, 2957, 2958, 2959, 2961, 2962, 2965, 2971, 2972, 2973, 2974, 2977, 2978, 2979, 2980, 2982, 2983, 2984, 2985, 2987, 2989, 2990, 2992, 2993, 2995, 2997, 2998, 3000, 3002, 3004, 3008, 3010, 3014, 3015, 3016, 3017, 3018, 3019, 3022, 3023, 3025, 3026, 3028, 3032, 3035, 3036, 3037, 3038, 3040, 3041, 3042, 3043, 3047, 3049, 3050, 3051, 3053, 3054, 3055, 3057, 3060, 3062, 3063, 3065, 3066, 3068, 3069, 3070, 3071, 3072, 3074, 3075, 3078, 3079, 3081, 3082, 3084, 3086, 3087, 3088, 3089, 3092, 3094, 3095, 3097, 3099, 3101, 3102, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3112, 3113, 3114, 3122, 3123, 3124, 3127, 3128, 3129, 3130, 3133, 3134, 3135, 3136, 3138, 3140, 3141, 3143, 3144, 3145, 3146, 3148, 3150, 3154, 3155, 3156, 3157, 3158, 3159, 3160, 3162, 3163, 3164, 3165, 3166, 3168, 3171, 3172, 3173, 3175, 3179, 3180, 3181, 3182, 3184, 3186, 3187, 3188, 3192, 3193, 3196, 3197, 3199, 3200, 3201, 3202, 3204, 3212, 3213, 3216, 3217, 3218, 3219, 3221, 3222, 3228, 3232, 3233, 3234, 3237, 3238, 3250, 3251, 3254, 3255, 3256, 3257, 3258, 3262, 3264, 3268, 3269, 3270, 3272, 3273, 3274, 3277, 3280, 3282, 3283, 3284, 3285, 3286, 3288, 3289, 3290, 3291, 3294, 3295, 3296, 3299, 3300, 3301, 3307, 3309, 3310, 3311, 3313, 3314, 3315, 3317, 3318, 3319, 3320, 3322, 3323, 3324, 3325, 3327, 3330, 3335, 3336, 3337, 3342, 3348, 3349, 3354, 3355, 3357, 3359, 3361, 3364, 3366, 3367, 3369, 3371, 3372, 3374, 3377, 3382, 3383, 3384, 3387, 3390, 3394, 3397, 3404, 3406, 3407, 3408, 3412, 3413, 3414, 3415, 3418, 3420, 3422, 3426, 3428, 3429, 3431, 3433, 3436, 3437, 3438, 3439, 3442, 3443, 3444, 3445, 3446, 3447, 3448, 3449, 3455, 3456, 3459, 3460, 3462, 3463, 3464, 3467, 3468, 3470, 3474, 3478, 3479, 3480, 3481, 3482, 3483, 3486, 3488, 3490, 3491, 3492, 3493, 3494, 3495, 3498, 3500, 3501, 3502, 3503, 3505, 3507, 3508, 3511, 3512, 3513, 3514, 3515, 3517, 3518, 3519, 3520, 3521, 3522, 3523, 3524, 3525, 3528, 3529, 3533, 3534, 3535, 3542, 3544, 3549, 3553, 3554, 3555, 3558, 3559, 3562, 3563, 3564, 3565, 3566, 3568, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3578, 3579, 3580, 3581, 3583, 3584, 3590, 3591, 3593, 3594, 3595, 3597, 3599, 3600, 3601, 3602, 3604, 3605, 3606, 3608, 3611, 3613, 3614, 3618, 3619, 3620, 3621, 3622, 3624, 3625, 3626, 3627, 3630, 3631, 3634, 3636, 3637, 3638, 3639, 3641, 3644, 3646, 3648, 3650, 3651, 3653, 3655, 3658, 3663, 3665, 3667, 3668, 3670, 3672, 3679, 3681, 3683, 3684, 3689, 3690, 3692, 3701, 3702, 3703, 3704, 3710, 3713, 3714, 3715, 3721, 3725, 3726, 3727, 3728, 3729, 3732, 3733, 3736, 3738, 3741, 3742, 3743, 3747, 3749, 3750, 3751, 3753, 3754, 3755, 3756, 3757, 3758, 3759, 3761, 3765, 3766, 3769, 3772, 3775, 3780, 3782, 3785, 3787, 3790, 3791, 3792, 3793, 3795, 3796, 3798, 3799, 3803, 3804, 3807, 3808, 3809, 3810, 3811, 3812, 3813, 3814, 3815, 3816, 3817, 3819, 3820, 3826, 3827, 3828, 3833, 3835, 3837, 3838, 3839, 3840, 3841, 3842, 3843, 3845, 3847, 3850, 3851, 3854, 3856, 3858, 3859, 3861, 3863, 3867, 3868, 3870, 3873, 3874, 3875, 3876, 3878, 3879, 3883, 3884, 3885, 3886, 3888, 3889, 3891, 3892, 3896, 3899, 3901, 3902, 3903, 3905, 3906, 3908, 3913, 3914, 3915, 3919, 3920, 3921, 3922, 3923, 3924, 3925, 3928, 3929, 3931, 3936, 3938, 3942, 3944, 3945, 3946, 3949, 3950, 3951, 3952, 3955, 3956, 3957, 3961, 3965, 3967, 3968, 3971, 3972, 3975, 3976, 3977, 3978, 3979, 3982, 3983, 3985, 3986, 3988, 3990, 3991, 3992, 3993, 3994, 3996, 3998, 3999, 4000, 4001, 4004, 4005, 4009, 4010, 4011, 4016, 4021, 4022, 4024, 4025, 4026, 4031, 4034, 4035, 4036, 4038, 4040, 4041, 4042, 4044, 4045, 4046, 4047, 4050, 4052, 4054, 4055, 4057, 4058, 4062, 4064, 4068, 4069, 4072, 4073, 4075, 4078, 4079, 4081, 4082, 4084, 4087, 4088, 4089, 4090, 4091, 4092, 4093, 4095, 4097, 4098, 4099, 4101, 4102, 4104, 4105, 4108, 4109, 4111, 4115, 4118, 4119, 4122, 4123, 4125, 4126, 4127, 4128, 4129, 4130, 4132, 4134, 4135, 4138, 4139, 4140, 4145, 4147, 4148, 4149, 4150, 4152, 4156, 4159, 4161, 4162, 4165, 4169, 4171, 4174, 4175, 4176, 4177, 4179, 4181, 4184, 4188, 4190, 4191, 4193, 4195, 4197, 4199, 4201, 4202, 4203, 4205, 4206, 4210, 4213, 4216, 4217, 4220, 4221, 4222, 4225, 4228, 4229, 4230, 4232, 4233, 4234, 4237, 4238, 4239, 4240, 4241, 4242, 4243, 4245, 4246, 4252, 4253, 4254, 4255, 4258, 4260, 4261, 4262, 4263, 4265, 4269, 4271, 4272, 4274, 4275, 4277, 4278, 4279, 4281, 4282, 4283, 4284, 4286, 4289, 4290, 4291, 4293, 4294, 4296, 4299, 4300, 4303, 4304, 4306, 4307, 4308, 4309, 4310, 4312, 4313, 4314, 4315, 4319, 4321, 4322, 4330, 4331, 4332, 4334, 4336, 4341, 4342, 4344, 4346, 4348, 4350, 4351, 4352, 4354, 4356, 4360, 4361, 4362, 4364, 4368, 4370, 4371, 4374, 4377, 4378, 4379, 4382, 4384, 4385, 4387, 4389, 4390, 4391, 4392, 4393, 4394, 4407, 4409, 4415, 4416, 4417, 4418, 4424, 4425, 4428, 4430, 4431, 4434, 4435, 4436, 4437, 4439, 4440, 4442, 4447, 4450, 4451, 4452, 4454, 4455, 4456, 4458, 4459, 4460, 4461, 4465, 4471, 4472, 4473, 4475, 4478, 4480, 4481, 4483, 4485, 4486, 4487, 4489, 4490, 4491, 4492, 4496, 4497, 4499, 4503, 4505, 4506, 4507, 4508, 4512, 4516, 4518, 4519, 4522, 4523, 4524, 4528, 4530, 4531, 4534, 4535, 4541, 4542, 4543, 4545, 4546, 4549, 4551, 4555, 4557, 4558, 4559, 4561, 4562, 4563, 4564, 4565, 4566, 4567, 4568, 4570, 4574, 4581, 4583, 4584, 4588, 4590, 4592, 4595, 4598, 4599, 4600, 4603, 4611, 4612, 4613, 4618, 4621, 4623, 4625, 4627, 4632, 4634, 4638, 4639, 4644, 4645, 4649, 4650, 4653, 4654, 4657, 4660, 4661, 4662, 4663, 4664, 4666, 4670, 4671, 4674, 4675, 4678, 4679, 4681, 4682, 4684, 4686, 4688, 4689, 4691, 4694, 4696, 4699, 4702, 4704, 4706, 4707, 4711, 4712, 4713, 4718, 4721, 4723, 4725, 4729, 4732, 4733, 4734, 4737, 4738, 4747, 4749, 4750, 4751, 4752, 4755, 4757, 4758, 4766, 4767, 4774, 4775, 4776, 4777, 4778, 4779, 4781, 4782, 4786, 4789, 4791, 4793, 4796, 4799, 4802, 4804, 4807, 4808, 4809, 4816, 4818, 4821, 4823, 4826, 4829, 4830, 4831, 4833, 4837, 4838, 4840, 4841, 4842, 4844, 4845, 4847, 4848, 4850, 4857, 4859, 4860, 4862, 4863, 4864, 4866, 4868, 4873, 4874, 4876, 4879, 4880, 4881, 4883, 4884, 4886, 4887, 4888, 4890, 4891, 4892, 4898, 4899, 4904, 4910, 4912, 4913, 4918, 4919, 4920, 4924, 4927, 4928, 4929, 4935, 4936, 4937, 4941, 4944, 4945, 4947, 4950, 4952, 4954, 4955, 4956, 4958, 4961, 4962, 4963, 4965, 4970, 4973, 4974, 4979, 4980, 4982, 4983, 4984, 4986, 4989, 4990, 4992, 4993, 4997, 5000, 5002, 5009, 5010, 5011, 5013, 5016, 5018, 5020, 5021, 5024, 5027, 5028, 5033, 5038, 5048, 5049, 5050, 5055, 5059, 5064, 5067, 5068, 5072, 5073, 5074, 5075, 5077, 5081, 5083, 5084, 5087, 5089, 5090, 5091, 5096, 5097, 5098, 5099, 5102, 5106, 5110, 5111, 5116, 5117, 5121, 5122, 5125, 5128, 5129, 5132, 5134, 5136, 5138, 5139, 5141, 5142, 5143, 5144, 5146, 5147, 5148, 5150, 5152, 5154, 5156, 5157, 5158, 5166, 5167, 5170, 5171, 5173, 5174, 5176, 5181, 5182, 5183, 5190, 5197, 5198, 5199, 5202, 5206, 5212, 5215, 5216, 5218, 5221, 5222, 5226, 5227, 5228, 5231, 5235, 5236, 5237, 5242, 5245, 5247, 5251, 5252, 5253, 5255, 5257, 5259, 5261, 5262, 5263, 5265, 5267, 5272, 5274, 5278, 5281, 5283, 5284, 5285, 5289, 5292, 5296, 5298, 5303, 5304, 5306, 5307, 5308, 5309, 5310, 5312, 5313, 5315, 5317, 5320, 5323, 5324, 5326, 5327, 5330, 5331, 5334, 5335, 5337, 5339, 5342, 5346, 5347, 5350, 5352, 5353, 5354, 5356, 5357, 5362, 5364, 5365, 5369, 5370, 5372, 5373, 5376, 5378, 5380, 5383, 5384, 5390, 5392, 5394, 5396, 5399, 5402, 5403, 5406, 5407, 5408, 5414, 5418, 5425, 5426, 5427, 5428, 5429, 5432, 5433, 5434, 5435, 5436, 5437, 5438, 5439, 5441, 5443, 5444, 5445, 5446, 5447, 5448, 5453, 5454, 5456, 5459, 5461, 5466, 5467, 5469, 5471, 5473, 5474, 5475, 5477, 5478, 5479, 5480, 5484, 5485, 5486, 5487, 5488, 5489, 5491, 5493, 5495, 5497, 5498, 5499, 5500, 5501, 5505, 5508, 5511, 5514, 5515, 5516, 5517, 5518, 5527, 5529, 5531, 5532, 5533, 5536, 5537, 5541, 5546, 5548, 5549, 5550, 5551, 5552, 5553, 5555, 5559, 5561, 5562, 5564, 5565, 5567, 5568, 5570, 5571, 5572, 5576, 5582, 5585, 5586, 5587, 5588, 5591, 5593, 5597, 5598, 5599, 5600, 5601, 5603, 5604, 5605, 5606, 5607, 5611, 5614, 5615, 5617, 5618, 5621, 5623, 5629, 5634, 5637, 5640, 5642, 5643, 5646, 5649, 5651, 5657, 5658, 5661, 5662, 5666, 5671, 5672, 5674, 5676, 5678, 5680, 5682, 5684, 5685, 5686, 5687, 5689, 5691, 5692, 5693, 5694, 5696, 5700, 5703, 5706, 5707, 5708, 5712, 5715, 5717, 5718, 5721, 5722, 5728, 5732, 5733, 5734, 5737, 5743, 5745, 5746, 5748, 5751, 5752, 5753, 5756, 5757, 5758, 5759, 5761, 5762, 5764, 5770, 5774, 5776, 5777, 5779, 5781, 5784, 5786, 5787, 5788, 5790, 5791, 5792, 5794, 5799, 5802, 5805, 5807, 5812, 5816, 5817, 5821, 5823, 5825, 5827, 5829, 5830, 5832, 5836, 5838, 5839, 5840, 5843, 5844, 5849, 5851, 5853, 5854, 5855, 5856, 5860, 5862, 5871, 5875, 5876, 5878, 5879, 5882, 5883, 5886, 5887, 5888, 5891, 5893, 5895, 5902, 5904, 5908, 5909, 5910, 5911, 5912, 5913, 5916, 5917, 5918, 5922, 5924, 5925, 5927, 5933, 5935, 5936, 5938, 5939, 5940, 5941, 5943, 5945, 5946, 5947, 5951, 5954, 5955, 5956, 5959, 5961, 5964, 5971, 5972, 5975, 5976, 5978, 5979, 5981, 5983, 5984, 5987, 5993, 5995, 5998, 5999, 6000, 6001, 6007, 6008, 6009, 6013, 6015, 6018, 6022, 6027, 6031, 6032, 6033, 6037, 6040, 6041, 6042, 6043, 6047, 6052, 6057, 6058, 6061, 6063, 6066, 6067, 6073, 6078, 6081, 6082, 6084, 6085, 6086, 6087, 6088, 6089, 6090, 6093, 6094, 6095, 6096, 6097, 6101, 6105, 6108, 6113, 6117, 6118, 6121, 6122, 6125, 6127, 6130, 6132, 6134, 6136, 6139, 6140, 6144, 6145, 6146, 6149, 6150, 6151, 6157, 6160, 6163, 6165, 6170, 6174, 6178, 6179, 6180, 6182, 6183, 6184, 6187, 6188, 6195, 6196, 6197, 6199, 6205, 6206, 6209, 6213, 6214, 6217, 6220, 6221, 6226, 6228, 6231, 6233, 6234, 6235, 6236, 6237, 6243, 6244, 6248, 6249, 6250, 6257, 6258, 6263, 6276, 6281, 6282, 6284, 6290, 6292, 6295, 6299, 6302, 6306, 6309, 6310, 6311, 6316, 6318, 6321, 6325, 6327, 6331, 6335, 6337, 6342, 6343, 6345, 6347, 6348, 6349, 6350, 6354, 6355, 6356, 6357, 6359, 6361, 6362, 6364, 6365, 6369, 6371, 6374, 6375, 6377, 6379, 6380, 6381, 6382, 6386, 6387, 6391, 6395, 6399, 6403, 6406, 6411, 6415, 6418, 6420, 6421, 6424, 6426, 6427, 6429, 6434, 6435, 6442, 6450, 6454, 6455, 6456, 6458, 6463, 6464, 6469, 6470, 6473, 6475, 6479, 6481, 6483, 6486, 6487, 6489, 6491, 6494, 6497, 6498, 6501, 6503, 6504, 6505, 6507, 6508, 6511, 6513, 6516, 6517, 6518, 6521, 6522, 6525, 6526, 6527, 6528, 6530, 6531, 6534, 6536, 6544, 6546, 6548, 6549, 6550, 6552, 6554, 6557, 6558, 6560, 6561, 6564, 6568, 6570, 6572, 6573, 6575, 6576, 6587, 6588, 6590, 6591, 6593, 6595, 6596, 6599, 6600, 6601, 6602, 6604, 6607, 6608, 6610, 6611, 6614, 6618, 6625, 6627, 6632, 6635, 6637, 6638, 6639, 6642, 6643, 6645, 6646, 6649, 6651, 6658, 6659, 6661, 6663, 6667, 6668, 6669, 6673, 6675, 6676, 6679, 6681, 6686, 6692, 6694, 6695, 6696, 6697, 6699, 6701, 6702, 6703, 6705, 6707, 6708, 6711, 6716, 6723, 6725, 6726, 6728, 6729, 6730, 6731, 6734, 6735, 6737, 6738, 6740, 6741, 6743, 6744, 6745, 6747, 6749, 6750, 6751, 6752, 6756, 6757, 6758, 6761, 6762, 6765, 6766, 6767, 6770, 6771, 6773, 6774, 6775, 6778, 6779, 6780, 6784, 6790, 6791, 6797, 6799, 6802, 6803, 6804, 6805, 6806, 6807, 6809, 6810, 6813, 6816, 6817, 6818, 6820, 6821, 6826, 6829, 6833, 6835, 6838, 6844, 6848, 6849, 6851, 6854, 6855, 6857, 6859, 6863, 6864, 6866, 6867, 6868, 6872, 6880, 6881, 6882, 6885, 6886, 6891, 6892, 6895, 6898, 6902, 6907, 6909, 6915, 6916, 6918, 6920, 6921, 6922, 6923, 6925, 6927, 6936, 6941, 6942, 6943, 6946, 6948, 6950, 6951, 6957, 6958, 6960, 6961, 6966, 6968, 6969, 6970, 6973, 6975, 6976, 6977, 6978, 6979, 6983, 6986, 6989, 6990, 6991, 6992, 6994, 6995, 6999, 7004, 7009, 7010, 7014, 7019, 7021, 7023, 7025, 7026, 7028, 7030, 7031, 7032, 7033, 7034, 7035, 7041, 7042, 7044, 7047, 7048, 7049, 7050, 7054, 7055, 7056, 7059, 7060, 7062, 7063, 7065, 7068, 7070, 7072, 7073, 7077, 7078, 7081, 7086, 7087, 7090, 7091, 7092, 7095, 7098, 7099, 7101, 7110, 7116, 7117, 7123, 7127, 7139, 7143, 7144, 7145, 7148, 7151, 7152, 7154, 7159, 7161, 7167, 7170, 7171, 7172, 7173, 7174, 7176, 7178, 7179, 7181, 7183, 7185, 7187, 7188, 7189, 7193, 7194, 7199, 7202, 7206, 7207, 7209, 7210, 7213, 7222, 7225, 7226, 7230, 7233, 7234, 7236, 7243, 7244, 7245, 7246, 7247, 7249, 7252, 7255, 7258, 7261, 7262, 7265, 7268, 7269, 7272, 7276, 7278, 7281, 7284, 7285, 7286, 7287, 7289, 7290, 7294, 7302, 7303, 7304, 7308, 7309, 7310, 7314, 7315, 7318, 7320, 7321, 7324, 7326, 7327, 7333, 7334, 7335, 7345, 7346, 7354, 7355, 7359, 7361, 7364, 7365, 7367, 7368, 7369, 7373, 7383, 7385, 7386, 7390, 7401, 7402, 7403, 7406, 7407, 7408, 7410, 7411, 7413, 7415, 7417, 7418, 7419, 7421, 7426, 7428, 7432, 7435, 7438, 7439, 7440, 7441, 7444, 7448, 7450, 7457, 7458, 7459, 7460, 7461, 7462, 7463, 7465, 7469, 7472, 7474, 7478, 7481, 7484, 7489, 7496, 7500, 7503, 7506, 7511, 7515, 7520, 7521, 7522, 7525, 7526, 7533, 7535, 7538, 7541, 7547, 7551, 7552, 7554, 7556, 7557, 7559, 7561, 7563, 7571, 7573, 7578, 7584, 7587, 7588, 7589, 7590, 7592, 7593, 7594, 7601, 7602, 7604, 7605, 7606, 7608, 7609, 7610, 7613, 7618, 7620, 7621, 7624, 7628, 7629, 7636, 7637, 7638, 7640, 7642, 7643, 7646, 7650, 7651, 7652, 7654, 7656, 7658, 7662, 7667, 7672, 7674, 7675, 7676, 7677, 7683, 7690, 7691, 7692, 7695, 7696, 7697, 7698, 7699, 7700, 7701, 7703, 7705, 7708, 7709, 7711, 7712, 7713, 7717, 7718, 7719, 7723, 7728, 7730, 7731, 7734, 7736, 7739, 7744, 7745, 7748, 7752, 7757, 7761, 7762, 7772, 7776, 7780, 7781, 7782, 7783, 7784, 7786, 7787, 7789, 7790, 7792, 7793, 7795, 7803, 7810, 7813, 7821, 7825, 7828, 7830, 7832, 7834, 7836, 7838, 7840, 7841, 7845, 7850, 7854, 7856, 7859, 7860, 7861, 7865, 7867, 7869, 7872, 7873, 7874, 7875, 7876, 7880, 7881, 7883, 7886, 7888, 7890, 7892, 7896, 7897, 7899, 7904, 7907, 7908, 7909, 7910, 7911, 7913, 7915, 7916, 7917, 7918, 7920, 7926, 7929, 7933, 7935, 7936, 7941, 7944, 7945, 7946, 7947, 7949, 7950, 7955, 7965, 7968, 7974, 7980, 7984, 7986, 7994, 7995, 8003, 8005, 8007, 8012, 8014, 8015, 8017, 8020, 8024, 8026, 8027, 8028, 8029, 8031, 8032, 8034, 8037, 8038, 8042, 8046, 8047, 8053, 8055, 8056, 8057, 8059, 8060, 8062, 8065, 8070, 8072, 8077, 8079, 8082, 8083, 8085, 8097, 8103, 8106, 8112, 8114, 8115, 8117, 8119, 8121, 8122, 8123, 8130, 8139, 8142, 8153, 8155, 8156, 8158, 8160, 8166, 8167, 8169, 8172, 8173, 8174, 8176, 8177, 8181, 8184, 8187, 8190, 8193, 8195, 8198, 8200, 8205, 8209, 8210, 8211, 8213, 8214, 8218, 8219, 8222, 8224, 8225, 8229, 8231, 8232, 8236, 8239, 8240, 8242, 8245, 8246, 8252, 8255, 8259, 8264, 8266, 8268, 8269, 8274, 8275, 8280, 8281, 8284, 8286, 8287, 8288, 8290, 8298, 8306, 8307, 8308, 8309, 8311, 8317, 8320, 8323, 8332, 8334, 8339, 8340, 8341, 8346, 8347, 8348, 8351, 8358, 8363, 8364, 8366, 8367, 8373, 8379, 8381, 8385, 8387, 8388, 8390, 8391, 8394, 8396, 8397, 8398, 8404, 8405, 8406, 8410, 8412, 8413, 8418, 8419, 8421, 8422, 8427, 8432, 8434, 8437, 8439, 8441, 8443, 8448, 8451, 8454, 8457, 8460, 8461, 8466, 8468, 8469, 8470, 8472, 8474, 8476, 8477, 8479, 8481, 8483, 8485, 8492, 8497, 8498, 8500, 8502, 8503, 8507, 8508, 8509, 8511, 8516, 8523, 8524, 8526, 8529, 8532, 8535, 8541, 8545, 8547, 8552, 8553, 8554, 8555, 8557, 8559, 8563, 8566, 8570, 8571, 8574, 8581, 8582, 8583, 8585, 8589, 8591, 8592, 8593, 8594, 8595, 8597, 8598, 8602, 8610, 8613, 8615, 8618, 8621, 8623, 8626, 8628, 8635, 8639, 8640, 8643, 8645, 8646, 8652, 8653, 8655, 8659, 8660, 8666, 8668, 8669, 8672, 8677, 8679, 8681, 8686, 8689, 8692, 8695, 8698, 8701, 8702, 8703, 8705, 8708, 8712, 8714, 8716, 8719, 8720, 8722, 8724, 8725, 8727, 8732, 8736, 8737, 8744, 8751, 8752, 8754, 8755, 8757, 8761, 8763, 8771, 8772, 8775, 8779, 8780, 8783, 8785, 8786, 8790, 8791, 8794, 8796, 8797, 8798, 8799, 8804, 8806, 8807, 8812, 8817, 8823, 8826, 8833, 8835, 8838, 8839, 8842, 8845, 8846, 8847, 8848, 8851, 8852, 8855, 8857, 8858, 8862, 8866, 8868, 8869, 8871, 8872, 8873, 8875, 8876, 8877, 8879, 8882, 8883, 8884, 8886, 8887, 8894, 8900, 8906, 8907, 8910, 8912, 8913, 8915, 8917, 8919, 8920, 8926, 8928, 8937, 8938, 8940, 8941, 8944, 8949, 8950, 8952, 8953, 8969, 8972, 8974, 8976, 8977, 8978, 8979, 8981, 8984, 8985, 8987, 8993, 8996, 9002, 9006, 9007, 9009, 9011, 9013, 9019, 9023, 9033, 9036, 9037, 9039, 9043, 9044, 9047, 9054, 9055, 9057, 9060, 9063, 9066, 9069, 9071, 9075, 9077, 9079, 9084, 9085, 9087, 9089, 9095, 9100, 9101, 9103, 9106, 9107, 9110, 9111, 9113, 9116, 9117, 9121, 9123, 9128, 9135, 9140, 9141, 9143, 9144, 9146, 9147, 9149, 9150, 9153, 9154, 9155, 9163, 9167, 9168, 9170, 9172, 9175, 9181, 9182, 9184, 9189, 9190, 9191, 9192, 9194, 9197, 9202, 9205, 9207, 9209, 9214, 9215, 9218, 9221, 9223, 9226, 9227, 9236, 9237, 9238, 9239, 9240, 9241, 9244, 9245, 9250, 9252, 9253, 9257, 9261, 9263, 9264, 9265, 9270, 9272, 9278, 9281, 9283, 9284, 9286, 9290, 9292, 9293, 9296, 9297, 9301, 9304, 9307, 9308, 9318, 9319, 9321, 9325, 9326, 9331, 9333, 9337, 9343, 9347, 9356, 9357, 9358, 9361, 9368, 9370, 9372, 9373, 9375, 9376, 9378, 9380, 9381, 9390, 9397, 9399, 9400, 9402, 9404, 9405, 9407, 9409, 9411, 9416, 9418, 9422, 9423, 9428, 9431, 9433, 9434, 9435, 9439, 9444, 9445, 9446, 9447, 9450, 9457, 9459, 9460, 9461, 9463, 9464, 9467, 9468, 9469, 9476, 9477, 9482, 9485, 9493, 9495, 9496, 9501, 9502, 9507, 9514, 9515, 9516, 9518, 9521, 9522, 9527, 9536, 9539, 9540, 9542, 9546, 9553, 9554, 9556, 9566, 9568, 9569, 9571, 9574, 9581, 9584, 9590, 9592, 9596, 9599, 9602, 9603, 9610, 9611, 9612, 9613, 9628, 9633, 9634, 9637, 9639, 9640, 9646, 9648, 9651, 9664, 9665, 9666, 9668, 9669, 9671, 9674, 9675, 9679, 9680, 9685, 9686, 9687, 9690, 9691, 9694, 9696, 9697, 9699, 9703, 9704, 9705, 9711, 9712, 9713, 9714, 9718, 9721, 9722, 9727, 9728, 9731, 9732, 9734, 9735, 9738, 9740, 9744, 9751, 9756, 9757, 9758, 9761, 9764, 9766, 9770, 9772, 9773, 9774, 9776, 9780, 9781, 9785, 9789, 9790, 9791, 9793, 9797, 9798, 9801, 9806, 9808, 9809, 9810, 9813, 9814, 9815, 9819, 9825, 9826, 9828, 9829, 9842, 9843, 9845, 9851, 9852, 9853, 9854, 9855, 9856, 9857, 9858, 9859, 9861, 9868, 9870, 9878, 9882, 9884, 9885, 9886, 9888, 9889, 9891, 9892, 9895, 9896, 9897, 9900, 9907, 9919, 9920, 9923, 9927, 9929, 9930, 9933, 9936, 9940, 9944, 9947, 9948, 9952, 9956, 9959, 9960, 9964, 9966, 9968, 9971, 9973, 9974, 9975, 9976, 9978, 9979, 9981, 9986, 9987, 9989, 9990, 9997, 10000, 10001, 10002, 10003, 10010, 10014, 10016, 10018, 10020, 10021, 10024, 10025, 10027, 10028, 10029, 10031, 10032, 10033, 10036, 10039, 10041, 10042, 10046, 10047, 10052, 10054, 10065, 10071, 10079, 10082, 10087, 10088, 10092, 10103, 10104, 10106, 10110, 10115, 10116, 10118, 10121, 10124, 10125, 10126, 10130, 10131, 10133, 10135, 10137, 10139, 10140, 10141, 10142, 10144, 10152, 10155, 10156, 10160, 10162, 10163, 10165, 10168, 10171, 10172, 10173, 10179, 10184, 10185, 10188, 10189, 10194, 10197, 10202, 10203, 10206, 10208, 10212, 10213, 10219, 10220, 10226, 10229, 10231, 10232, 10234, 10235, 10236, 10242, 10245, 10248, 10250, 10253, 10255, 10256, 10260, 10263, 10264, 10266, 10268, 10269, 10270, 10273, 10278, 10279, 10281, 10283, 10286, 10287, 10288, 10289, 10290, 10291, 10292, 10296, 10297, 10299, 10300, 10304, 10306, 10307, 10308, 10311, 10312, 10317, 10320, 10321, 10324, 10333, 10337, 10338, 10339, 10340, 10342, 10343, 10345, 10346, 10350, 10351, 10354, 10355, 10357, 10358, 10365, 10368, 10371, 10374, 10375, 10376, 10377, 10378, 10379, 10380, 10381, 10382, 10386, 10390, 10391, 10393, 10399, 10404, 10406, 10416, 10417, 10419, 10424, 10426, 10431, 10432, 10435, 10442, 10451, 10452, 10453, 10454, 10457, 10458, 10459, 10463, 10466, 10467, 10469, 10470, 10474, 10478, 10479, 10480, 10484, 10490, 10495, 10502, 10503, 10512, 10517, 10519, 10521, 10522, 10523, 10530, 10534, 10537, 10539, 10542, 10545, 10548, 10549, 10551, 10557, 10558, 10560, 10561, 10562, 10569, 10570, 10573, 10578, 10582, 10583, 10584, 10589, 10590, 10591, 10592, 10596, 10598, 10605, 10606, 10616, 10621, 10625, 10630, 10631, 10632, 10633, 10635, 10639, 10641, 10642, 10643, 10646, 10655, 10658, 10659, 10660, 10661, 10663, 10665, 10666, 10672, 10673, 10674, 10675, 10676, 10691, 10694, 10695, 10705, 10706, 10708, 10709, 10710, 10717, 10720, 10722, 10723, 10724, 10727, 10728, 10729, 10732, 10734, 10736, 10738, 10739, 10741, 10743, 10744, 10751, 10752, 10755, 10757, 10765, 10771, 10772, 10774, 10782, 10787, 10788, 10789, 10800, 10804, 10805, 10807, 10811, 10813, 10814, 10817, 10820, 10834, 10841, 10842, 10845, 10846, 10850, 10852, 10853, 10856, 10858, 10860, 10865, 10866, 10869, 10874, 10876, 10877, 10883, 10885, 10886, 10895, 10897, 10906, 10913, 10914, 10917, 10921, 10923, 10925, 10926, 10928, 10930, 10931, 10932, 10933, 10934, 10936, 10937, 10938, 10946, 10951, 10952, 10953, 10957, 10958, 10959, 10962, 10963, 10964, 10966, 10967, 10968, 10970, 10972, 10973, 10977, 10980, 10986, 10990, 10993, 11000, 11002, 11007, 11009, 11013, 11018, 11022, 11028, 11031, 11033, 11036, 11039, 11040, 11046, 11047, 11048, 11051, 11053, 11055, 11066, 11073, 11076, 11079, 11081, 11083, 11086, 11088, 11096, 11100, 11104, 11106, 11107, 11114, 11116, 11121, 11124, 11125, 11128, 11132, 11134, 11135, 11136, 11140, 11147, 11148, 11152, 11153, 11154, 11156, 11158, 11161, 11164, 11173, 11177, 11179, 11181, 11182, 11183, 11185, 11192, 11195, 11205, 11211, 11215, 11218, 11220, 11222, 11223, 11224, 11227, 11229, 11233, 11236, 11239, 11241, 11244, 11245, 11246, 11247, 11248, 11249, 11250, 11255, 11256, 11258, 11259, 11263, 11264, 11271, 11278, 11279, 11280, 11283, 11285, 11290, 11291, 11292, 11293, 11296, 11304, 11311, 11313, 11314, 11315, 11316, 11317, 11319, 11320, 11324, 11325, 11326, 11330, 11344, 11346, 11347, 11352, 11353, 11354, 11358, 11359, 11362, 11365, 11369, 11378, 11381, 11393, 11394, 11395, 11396, 11398, 11400, 11406, 11407, 11409, 11414, 11416, 11421, 11429, 11431, 11432, 11439, 11440, 11447, 11448, 11455, 11456, 11459, 11461, 11464, 11469, 11483, 11488, 11490, 11492, 11496, 11497, 11504, 11510, 11513, 11514, 11516, 11524, 11527, 11529, 11531, 11532, 11535, 11540, 11541, 11543, 11545, 11553, 11554, 11555, 11559, 11561, 11564, 11567, 11569, 11573, 11576, 11581, 11585, 11587, 11590, 11591, 11592, 11597, 11602, 11605, 11608, 11610, 11614, 11615, 11618, 11619, 11621, 11622, 11625, 11628, 11629, 11635, 11639, 11650, 11651, 11653, 11654, 11656, 11658, 11663, 11667, 11668, 11670, 11672, 11678, 11680, 11683, 11684, 11687, 11690, 11691, 11693, 11697, 11698, 11699, 11702, 11712, 11717, 11721, 11722, 11737, 11741, 11746, 11747, 11762, 11763, 11767, 11770, 11774, 11778, 11782, 11784, 11790, 11791, 11795, 11811, 11817, 11822, 11823, 11827, 11829, 11832, 11839, 11844, 11846, 11847, 11848, 11851, 11855, 11856, 11857, 11858, 11861, 11862, 11863, 11870, 11872, 11877, 11878, 11880, 11882, 11886, 11888, 11889, 11890, 11900, 11909, 11910, 11925, 11927, 11930, 11936, 11937, 11938, 11939, 11940, 11946, 11947, 11948, 11949, 11952, 11955, 11960, 11961, 11962, 11964, 11965, 11966, 11968, 11969, 11972, 11974, 11976, 11983, 11985, 11988, 11990, 11995, 12003, 12006, 12017, 12025, 12026, 12029, 12034, 12037, 12041, 12051, 12052, 12054, 12056, 12060, 12068, 12071, 12076, 12077, 12078, 12080, 12083, 12086, 12088, 12089, 12090, 12091, 12092, 12093, 12098, 12101, 12103, 12104, 12105, 12112, 12113, 12115, 12116, 12125, 12131, 12134, 12135, 12139, 12140, 12145, 12151, 12152, 12159, 12165, 12167, 12169, 12171, 12174, 12176, 12180, 12187, 12188, 12189, 12193, 12195, 12197, 12201, 12204, 12211, 12215, 12217, 12218, 12219, 12230, 12231, 12234, 12235, 12239, 12244, 12250, 12254, 12258, 12259, 12261, 12262, 12263, 12265, 12266, 12269, 12271, 12272, 12277, 12283, 12287, 12291, 12296, 12300, 12301, 12304, 12306, 12307, 12310, 12311, 12313, 12315, 12316, 12317, 12320, 12321, 12322, 12323, 12331, 12333, 12339, 12346, 12351, 12354, 12355, 12366, 12368, 12376, 12386, 12388, 12391, 12395, 12396, 12398, 12404, 12405, 12406, 12410, 12414, 12415, 12417, 12422, 12426, 12435, 12436, 12441, 12443, 12448, 12450, 12454, 12456, 12458, 12460, 12461, 12462, 12463, 12466, 12468, 12477, 12482, 12484, 12488, 12490, 12495, 12497, 12499, 12502, 12503, 12509, 12511, 12515, 12518, 12521, 12523, 12525, 12528, 12533, 12535, 12539, 12540, 12544, 12549, 12551, 12552, 12554, 12556, 12557, 12558, 12559, 12563, 12566, 12569, 12571, 12572, 12578, 12579, 12583, 12586, 12594, 12597, 12601, 12603, 12606, 12607, 12609, 12611, 12613, 12615, 12618, 12626, 12630, 12636, 12639, 12642, 12643, 12647, 12648, 12650, 12654, 12657, 12659, 12663, 12665, 12669, 12674, 12680, 12682, 12683, 12692, 12705, 12706, 12708, 12709, 12717, 12718, 12719, 12720, 12726, 12728, 12739, 12740, 12741, 12748, 12749, 12750, 12752, 12755, 12757, 12759, 12760, 12763, 12771, 12777, 12782, 12784, 12792, 12793, 12794, 12795, 12802, 12804, 12805, 12806, 12807, 12811, 12830, 12832, 12833, 12834, 12836, 12837, 12844, 12845, 12846, 12850, 12851, 12852, 12854, 12858, 12863, 12865, 12867, 12870, 12874, 12877, 12878, 12882, 12883, 12884, 12888, 12891, 12895, 12896, 12899, 12904, 12906, 12908, 12909, 12910, 12917, 12924, 12927, 12929, 12931, 12932, 12934, 12935, 12936, 12937, 12939, 12942, 12943, 12945, 12948, 12952, 12954, 12955, 12961, 12962, 12965, 12975, 12976, 12977, 12985, 12986, 12995, 12996, 12997, 13000, 13001, 13002, 13004, 13007, 13011, 13016, 13019, 13027, 13031, 13033, 13036, 13043, 13044, 13045, 13047, 13048, 13054, 13057, 13063, 13064, 13065, 13067, 13068, 13075, 13077, 13081, 13082, 13087, 13089, 13090, 13092, 13094, 13095, 13099, 13102, 13110, 13114, 13118, 13123, 13126, 13128, 13129, 13139, 13140, 13142, 13143, 13146, 13154, 13157, 13160, 13161, 13162, 13165, 13171, 13173, 13176, 13177, 13178, 13180, 13181, 13184, 13185, 13187, 13188, 13190, 13193, 13202, 13204, 13205, 13207, 13209, 13211, 13214, 13218, 13224, 13230, 13232, 13233, 13235, 13236, 13238, 13239, 13240, 13245, 13246, 13249, 13250, 13254, 13256, 13259, 13262, 13264, 13271, 13273, 13279, 13280, 13282, 13285, 13290, 13293, 13296, 13301, 13305, 13310, 13311, 13315, 13316, 13322, 13324, 13325, 13329, 13330, 13333, 13337, 13340, 13351, 13353, 13358, 13362, 13364, 13365, 13366, 13371, 13372, 13375, 13384, 13387, 13398, 13399, 13402, 13407, 13414, 13416, 13418, 13425, 13426, 13427, 13428, 13430, 13432, 13439, 13441, 13450, 13451, 13457, 13458, 13462, 13464, 13468, 13483, 13486, 13492, 13496, 13497, 13501, 13503, 13505, 13512, 13513, 13514, 13518, 13519, 13522, 13525, 13527, 13529, 13533, 13534, 13535, 13538, 13542, 13548, 13550, 13553, 13554, 13555, 13557, 13559, 13560, 13564, 13567, 13569, 13571, 13573, 13578, 13581, 13587, 13590, 13595, 13596, 13600, 13601, 13606, 13609, 13613, 13616, 13624, 13628, 13632, 13634, 13635, 13636, 13640, 13642, 13643, 13648, 13653, 13657, 13660, 13661, 13662, 13665, 13667, 13668, 13669, 13673, 13675, 13677, 13681, 13683, 13684, 13685, 13687, 13690, 13695, 13698, 13699, 13704, 13706, 13717, 13718, 13731, 13732, 13733, 13741, 13744, 13747, 13749, 13750, 13753, 13756, 13759, 13761, 13766, 13769, 13771, 13772, 13773, 13775, 13776, 13780, 13784, 13787, 13792, 13795, 13808, 13809, 13810, 13813, 13818, 13830, 13834, 13838, 13843, 13847, 13849, 13852, 13854, 13856, 13858, 13860, 13862, 13864, 13867, 13868, 13870, 13873, 13874, 13875, 13876, 13878, 13881, 13891, 13897, 13899, 13901, 13902, 13908, 13909, 13910, 13914, 13920, 13923, 13926, 13927, 13935, 13942, 13943, 13948, 13954, 13955, 13962, 13964, 13965, 13969, 13973, 13974, 13976, 13983, 13985, 13986, 13987, 13988, 13991, 13992, 13995, 13996, 14008, 14009, 14012, 14016, 14029, 14035, 14036, 14037, 14038, 14040, 14043, 14046, 14048, 14053, 14058, 14060, 14061, 14067, 14071, 14077, 14078, 14080, 14081, 14084, 14086, 14088, 14095, 14103, 14105, 14107, 14108, 14109, 14120, 14121, 14122, 14129, 14131, 14134, 14135, 14137, 14142, 14143, 14145, 14146, 14147, 14149, 14155, 14156, 14159, 14164, 14165, 14166, 14168, 14169, 14170, 14179, 14180, 14181, 14184, 14186, 14190, 14193, 14195, 14197, 14198, 14199, 14204, 14205, 14208, 14215, 14219, 14222, 14226, 14228, 14230, 14231, 14235, 14241, 14244, 14245, 14248, 14254, 14255, 14257, 14260, 14261, 14264, 14265, 14268, 14271, 14272, 14273, 14276, 14279, 14280, 14283, 14287, 14289, 14290, 14293, 14294, 14296, 14299, 14311, 14314, 14315, 14318, 14330, 14331, 14333, 14335, 14336, 14340, 14344, 14347, 14349, 14352, 14356, 14362, 14367, 14371, 14372, 14375, 14377, 14378, 14381, 14382, 14387, 14392, 14394, 14408, 14410, 14419, 14420, 14429, 14430, 14431, 14441, 14445, 14450, 14452, 14456, 14457, 14458, 14459, 14460, 14469, 14470, 14471, 14472, 14475, 14476, 14482, 14484, 14490, 14499, 14501, 14505, 14513, 14517, 14519, 14520, 14522, 14523, 14540, 14541, 14542, 14545, 14546, 14548, 14549, 14553, 14554, 14559, 14560, 14561, 14562, 14565, 14569, 14572, 14575, 14577, 14585, 14586, 14589, 14590, 14594, 14595, 14603, 14606, 14607, 14611, 14617, 14619, 14620, 14622, 14623, 14629, 14645, 14648, 14653, 14658, 14661, 14663, 14673, 14683, 14685, 14691, 14693, 14695, 14698, 14703, 14706, 14708, 14712, 14714, 14717, 14718, 14722, 14726, 14730, 14733, 14734, 14738, 14739, 14742, 14749, 14750, 14751, 14759, 14765, 14773, 14777, 14778, 14782, 14787, 14789, 14791, 14807, 14808, 14813, 14827, 14829, 14830, 14833, 14835, 14837, 14838, 14844, 14848, 14862, 14867, 14870, 14872, 14874, 14880, 14882, 14884, 14885, 14886, 14890, 14892, 14895, 14902, 14905, 14910, 14912, 14913, 14931, 14934, 14940, 14943, 14959, 14960, 14961, 14963, 14964, 14965, 14971, 14972, 14973, 14979, 14981, 14984, 14987, 14988, 14989, 14991, 14995, 15000, 15001, 15003, 15006, 15007, 15009, 15011, 15016, 15018, 15019, 15025, 15027, 15028, 15029, 15030, 15031, 15036, 15042, 15043, 15047, 15048, 15049, 15050, 15057, 15060, 15062, 15069, 15072, 15074, 15077, 15083, 15087, 15089, 15094, 15098, 15103, 15105, 15107, 15119, 15120, 15121, 15123, 15129, 15133, 15142, 15146, 15156, 15161, 15162, 15164, 15166, 15167, 15169, 15170, 15175, 15177, 15183, 15192, 15196, 15199, 15204, 15207, 15209, 15210, 15211, 15214, 15215, 15218, 15220, 15227, 15238, 15239, 15241, 15246, 15250, 15262, 15263, 15271, 15272, 15273, 15275, 15277, 15279, 15288, 15289, 15291, 15292, 15295, 15296, 15297, 15305, 15308, 15309, 15316, 15323, 15324, 15326, 15327, 15328, 15339, 15340, 15341, 15344, 15346, 15349, 15352, 15354, 15360, 15361, 15371, 15375, 15376, 15377, 15379, 15384, 15388, 15390, 15391, 15394, 15395, 15397, 15403, 15405, 15414, 15415, 15417, 15421, 15433, 15434, 15435, 15436, 15438, 15440, 15445, 15448, 15449, 15451, 15462, 15463, 15464, 15468, 15470, 15471, 15484, 15486, 15488, 15489, 15491, 15493, 15495, 15497, 15500, 15505, 15511, 15513, 15526, 15537, 15546, 15549, 15550, 15555, 15559, 15561, 15563, 15571, 15572, 15577, 15579, 15588, 15589, 15592, 15594, 15611, 15612, 15615, 15620, 15622, 15623, 15625, 15626, 15627, 15632, 15636, 15644, 15647, 15648, 15654, 15656, 15661, 15662, 15672, 15673, 15676, 15679, 15686, 15688, 15693, 15694, 15697, 15699, 15700, 15707, 15708, 15715, 15722, 15724, 15725, 15733, 15734, 15735, 15736, 15739, 15740, 15745, 15746, 15747, 15749, 15750, 15752, 15756, 15764, 15766, 15767, 15770, 15771, 15773, 15783, 15784, 15793, 15795, 15796, 15797, 15799, 15803, 15804, 15808, 15812, 15816, 15818, 15821, 15822, 15824, 15825, 15832, 15836, 15838, 15840, 15842, 15848, 15851, 15855, 15860, 15861, 15866, 15867, 15885, 15887, 15888, 15891, 15893, 15894, 15895, 15896, 15899, 15903, 15919, 15922, 15924, 15928, 15936, 15941, 15943, 15947, 15949, 15955, 15956, 15958, 15965, 15969, 15977, 15983, 15999, 16000, 16001, 16005, 16009, 16015, 16018, 16021, 16022, 16023, 16025, 16026, 16028, 16030, 16036, 16037, 16042, 16045, 16050, 16058, 16059, 16066, 16067, 16069, 16073, 16075, 16086, 16088, 16093, 16097, 16099, 16100, 16101, 16112, 16113, 16117, 16121, 16126, 16134, 16140, 16141, 16148, 16149, 16154, 16158, 16162, 16164, 16165, 16167, 16168, 16169, 16175, 16183, 16184, 16187, 16198, 16201, 16202, 16218, 16219, 16229, 16230, 16233, 16234, 16239, 16240, 16243, 16247, 16249, 16253, 16261, 16275, 16279, 16281, 16282, 16293, 16296, 16298, 16300, 16302, 16308, 16315, 16325, 16327, 16332, 16336, 16337, 16338, 16351, 16352, 16360, 16364, 16369, 16376, 16378, 16379, 16382, 16384, 16388, 16390, 16393, 16395, 16401, 16403, 16406, 16410, 16414, 16415, 16417, 16421, 16424, 16429, 16431, 16435, 16439, 16451, 16456, 16463, 16465, 16468, 16469, 16473, 16475, 16478, 16481, 16484, 16492, 16494, 16496, 16502, 16505, 16507, 16508, 16509, 16515, 16518, 16520, 16524, 16525, 16526, 16531, 16534, 16545, 16546, 16547, 16549, 16550, 16551, 16561, 16563, 16565, 16568, 16570, 16573, 16574, 16575, 16577, 16586, 16591, 16593, 16594, 16598, 16602, 16606, 16607, 16609, 16610, 16613, 16617, 16626, 16629, 16640, 16643, 16648, 16653, 16654, 16655, 16661, 16666, 16673, 16675, 16676, 16678, 16681, 16682, 16683, 16686, 16687, 16693, 16694, 16695, 16699, 16701, 16710, 16713, 16714, 16720, 16722, 16724, 16739, 16740, 16741, 16742, 16743, 16744, 16747, 16749, 16757, 16758, 16759, 16760, 16761, 16762, 16764, 16767, 16777, 16779, 16781, 16782, 16788, 16794, 16798, 16800, 16807, 16810, 16814, 16815, 16818, 16821, 16827, 16828, 16829, 16836, 16837, 16842, 16853, 16855, 16856, 16860, 16864, 16866, 16871, 16873, 16874, 16876, 16878, 16899, 16900, 16905, 16906, 16908, 16915, 16916, 16917, 16920, 16921, 16923, 16928, 16935, 16936, 16938, 16940, 16943, 16944, 16947, 16954, 16966, 16969, 16973, 16982, 16984, 16986, 16988, 16991, 16995, 16996, 16999, 17003, 17008, 17011, 17012, 17013, 17018, 17024, 17027, 17030, 17038, 17044, 17047, 17049, 17051, 17061, 17068, 17069, 17076, 17077, 17079, 17083, 17085, 17086, 17091, 17101, 17102, 17108, 17114, 17140, 17145, 17149, 17154, 17157, 17158, 17162, 17169, 17171, 17175, 17176, 17180, 17183, 17185, 17190, 17191, 17192, 17198, 17199, 17201, 17207, 17210, 17213, 17217, 17220, 17227, 17233, 17239, 17246, 17248, 17249, 17251, 17265, 17266, 17268, 17270, 17280, 17282, 17283, 17290, 17291, 17294, 17298, 17299, 17310, 17312, 17318, 17325, 17334, 17345, 17367, 17372, 17380, 17382, 17384, 17385, 17393, 17396, 17397, 17399, 17404, 17410, 17413, 17420, 17423, 17425, 17431, 17432, 17435, 17437, 17440, 17448, 17454, 17457, 17458, 17459, 17461, 17465, 17469, 17482, 17486, 17492, 17495, 17498, 17501, 17503, 17509, 17515, 17516, 17526, 17533, 17538, 17543, 17544, 17548, 17554, 17559, 17561, 17564, 17571, 17572, 17573, 17576, 17583, 17587, 17589, 17591, 17592, 17606, 17614, 17615, 17616, 17617, 17620, 17622, 17625, 17628, 17633, 17638, 17640, 17648, 17651, 17655, 17656, 17657, 17661, 17673, 17675, 17684, 17693, 17697, 17699, 17705, 17708, 17710, 17723, 17724, 17726, 17728, 17733, 17736, 17737, 17740, 17743, 17748, 17754, 17758, 17759, 17761, 17763, 17765, 17771, 17773, 17778, 17782, 17789, 17796, 17800, 17801, 17802, 17803, 17812, 17822, 17830, 17831, 17835, 17839, 17848, 17852, 17854, 17856, 17860, 17861, 17865, 17869, 17881, 17887, 17890, 17896, 17898, 17899, 17916, 17917, 17918, 17921, 17928, 17930, 17932, 17934, 17938, 17940, 17954, 17955, 17957, 17961, 17962, 17967, 17971, 17973, 17977, 17978, 17981, 17999, 18000, 18009, 18011, 18016, 18020, 18023, 18025, 18030, 18034, 18036, 18042, 18043, 18047, 18048, 18051, 18053, 18054, 18081, 18082, 18085, 18086, 18089, 18090, 18091, 18092, 18098, 18100, 18101, 18105, 18112, 18114, 18115, 18116, 18117, 18119, 18124, 18125, 18126, 18131, 18136, 18140, 18141, 18146, 18149, 18154, 18155, 18156, 18160, 18165, 18166, 18167, 18169, 18173, 18176, 18182, 18188, 18198, 18200, 18202, 18206, 18207, 18209, 18211, 18212, 18216, 18218, 18223, 18234, 18235, 18236, 18252, 18253, 18255, 18264, 18267, 18268, 18270, 18274, 18280, 18282, 18283, 18284, 18285, 18288, 18290, 18303, 18305, 18306, 18307, 18309, 18313, 18315, 18326, 18327, 18334, 18336, 18340, 18356, 18364, 18369, 18370, 18374, 18375, 18376, 18377, 18382, 18383, 18384, 18385, 18389, 18402, 18404, 18411, 18422, 18425, 18433, 18443, 18445, 18452, 18453, 18456, 18461, 18465, 18468, 18470, 18476, 18479, 18483, 18484, 18493, 18498, 18503, 18509, 18513, 18514, 18516, 18520, 18522, 18526, 18528, 18533, 18539, 18541, 18547, 18550, 18553, 18560, 18564, 18568, 18569, 18574, 18575, 18585, 18586, 18597, 18599, 18605, 18610, 18612, 18618, 18630, 18631, 18632, 18641, 18643, 18648, 18655, 18657, 18658, 18660, 18664, 18675, 18677, 18681, 18682, 18686, 18687, 18697, 18700, 18701, 18702, 18704, 18709, 18713, 18715, 18716, 18721, 18722, 18724, 18725, 18729, 18730, 18738, 18739, 18740, 18741, 18743, 18748, 18751, 18752, 18755, 18761, 18763, 18766, 18769, 18770, 18772, 18773, 18778, 18779, 18781, 18788, 18790, 18794, 18801, 18805, 18810, 18814, 18816, 18817, 18818, 18824, 18830, 18835, 18837, 18839, 18840, 18844, 18857, 18863, 18864, 18865, 18867, 18868, 18869, 18873, 18880, 18890, 18895, 18897, 18900, 18905, 18906, 18907, 18910, 18912, 18913, 18918, 18919, 18922, 18933, 18936, 18937, 18941, 18942, 18948, 18950, 18952, 18955, 18957, 18959, 18961, 18964, 18970, 18971, 18974, 18978, 18983, 18984, 18985, 18987, 18988, 18991, 19000, 19001, 19002, 19011, 19013, 19014, 19019, 19024, 19025, 19027, 19033, 19034, 19039, 19041, 19042, 19044, 19047, 19048, 19055, 19056, 19058, 19060, 19072, 19074, 19083, 19086, 19094, 19100, 19101, 19103, 19107, 19108, 19110, 19111, 19113, 19114, 19119, 19123, 19130, 19133, 19139, 19141, 19143, 19145, 19150, 19153, 19161, 19162, 19163, 19167, 19173, 19174, 19175, 19177, 19178, 19181, 19182, 19186, 19194, 19195, 19201, 19204, 19208, 19213, 19214, 19221, 19222, 19228, 19231, 19233, 19239, 19246, 19248, 19251, 19261, 19264, 19266, 19269, 19272, 19279, 19283, 19287, 19292, 19301, 19304, 19306, 19307, 19312, 19313, 19315, 19317, 19319, 19320, 19321, 19329, 19333, 19334, 19336, 19339, 19342, 19352, 19366, 19368, 19373, 19375, 19382, 19383, 19384, 19387, 19388, 19389, 19390, 19396, 19398, 19403, 19405, 19407, 19417, 19422, 19425, 19427, 19428, 19435, 19441, 19447, 19458, 19461, 19470, 19471, 19481, 19484, 19489, 19492, 19504, 19505, 19509, 19510, 19514, 19524, 19530, 19536, 19537, 19539, 19542, 19545, 19551, 19556, 19561, 19564, 19565, 19567, 19570, 19572, 19576, 19580, 19581, 19588, 19590, 19591, 19592, 19593, 19606, 19613, 19629, 19635, 19639, 19641, 19643, 19644, 19645, 19649, 19654, 19655, 19660, 19661, 19662, 19670, 19671, 19674, 19681, 19684, 19688, 19697, 19698, 19699, 19707, 19708, 19710, 19711, 19715, 19717, 19720, 19723, 19724, 19729, 19730, 19732, 19735, 19739, 19744, 19747, 19753, 19754, 19755, 19760, 19763, 19766, 19772, 19783, 19786, 19787, 19794, 19808, 19809, 19815, 19816, 19820, 19823, 19830, 19831, 19835, 19842, 19845, 19850, 19861, 19865, 19867, 19868, 19873, 19874, 19884, 19888, 19896, 19900, 19902, 19919, 19922, 19927, 19930, 19934, 19936, 19953, 19964, 19965, 19974, 19987, 19988, 20001, 20002, 20003, 20006, 20010, 20017, 20028, 20032, 20034, 20040, 20043, 20046, 20050, 20052, 20058, 20066, 20073, 20078, 20086, 20089, 20094, 20098, 20105, 20113, 20114, 20116, 20125, 20134, 20135, 20139, 20151, 20152, 20153, 20169, 20170, 20171, 20174, 20181, 20182, 20183, 20189, 20194, 20195, 20218, 20220, 20222, 20226, 20230, 20232, 20234, 20235, 20236, 20239, 20254, 20256, 20257, 20260, 20265, 20268, 20270, 20275, 20282, 20285, 20286, 20287, 20288, 20295, 20296, 20306, 20307, 20308, 20321, 20327, 20328, 20330, 20331, 20333, 20335, 20338, 20348, 20352, 20359, 20364, 20366, 20368, 20374, 20377, 20382, 20385, 20391, 20406, 20408, 20409, 20412, 20414, 20417, 20434, 20438, 20439, 20440, 20442, 20444, 20446, 20447, 20448, 20450, 20454, 20456, 20457, 20461, 20464, 20470, 20471, 20480, 20482, 20483, 20485, 20495, 20496, 20497, 20500, 20509, 20510, 20511, 20512, 20523, 20537, 20539, 20543, 20545, 20550, 20555, 20561, 20564, 20566, 20567, 20572, 20577, 20580, 20587, 20590, 20600, 20607, 20611, 20618, 20619, 20621, 20622, 20624, 20627, 20629, 20630, 20631, 20632, 20636, 20648, 20653, 20656, 20658, 20659, 20662, 20663, 20665, 20671, 20674, 20679, 20680, 20682, 20684, 20692, 20695, 20703, 20714, 20717, 20723, 20724, 20728, 20729, 20730, 20734, 20735, 20738, 20740, 20741, 20757, 20762, 20763, 20766, 20770, 20775, 20791, 20796, 20797, 20808, 20810, 20814, 20818, 20819, 20826, 20827, 20834, 20837, 20843, 20852, 20861, 20864, 20865, 20877, 20878, 20894, 20895, 20903, 20908, 20911, 20918, 20937, 20942, 20950, 20952, 20956, 20958, 20960, 20967, 20970, 20972, 20973, 20974, 20978, 20979, 20981, 20984, 20994, 21001, 21004, 21018, 21019, 21024, 21025, 21030, 21033, 21048, 21054, 21055, 21056, 21059, 21068, 21070, 21073, 21076, 21078, 21079, 21080, 21093, 21094, 21100, 21103, 21104, 21106, 21116, 21134, 21135, 21139, 21143, 21156, 21161, 21163, 21164, 21168, 21172, 21173, 21181, 21186, 21192, 21194, 21196, 21200, 21205, 21207, 21211, 21213, 21218, 21219, 21220, 21222, 21226, 21228, 21234, 21241, 21244, 21248, 21252, 21254, 21257, 21261, 21268, 21270, 21274, 21278, 21286, 21294, 21295, 21298, 21299, 21301, 21303, 21307, 21312, 21314, 21316, 21330, 21335, 21342, 21343, 21345, 21346, 21352, 21354, 21356, 21359, 21362, 21365, 21370, 21378, 21394, 21395, 21397, 21402, 21405, 21410, 21412, 21415, 21419, 21422, 21423, 21427, 21433, 21434, 21436, 21451, 21453, 21456, 21461, 21473, 21474, 21475, 21477, 21478, 21479, 21482, 21485, 21487, 21500, 21513, 21516, 21517, 21518, 21531, 21533, 21535, 21539, 21541, 21544, 21549, 21551, 21556, 21572, 21575, 21578, 21579, 21588, 21589, 21592, 21600, 21603, 21605, 21607, 21612, 21616, 21619, 21620, 21621, 21627, 21632, 21637, 21638, 21640, 21650, 21652, 21666, 21681, 21685, 21693, 21700, 21708, 21711, 21716, 21725, 21729, 21730, 21734, 21739, 21747, 21748, 21753, 21755, 21765, 21772, 21773, 21774, 21775, 21778, 21780, 21783, 21784, 21791, 21793, 21798, 21808, 21811, 21814, 21820, 21826, 21835, 21849, 21850, 21851, 21853, 21861, 21863, 21871, 21873, 21874, 21891, 21892, 21893, 21901, 21902, 21905, 21910, 21912, 21914, 21916, 21929, 21947, 21957, 21962, 21965, 21972, 21974, 21976, 21985, 21988, 21989, 21998, 22000, 22002, 22005, 22006, 22014, 22021, 22024, 22027, 22028, 22030, 22031, 22032, 22036, 22038, 22039, 22041, 22044, 22046, 22048, 22052, 22054, 22057, 22060, 22061, 22062, 22081, 22082, 22085, 22090, 22101, 22102, 22103, 22106, 22107, 22108, 22113, 22116, 22120, 22131, 22132, 22134, 22149, 22151, 22152, 22153, 22155, 22163, 22166, 22167, 22177, 22188, 22195, 22217, 22221, 22227, 22230, 22235, 22236, 22238, 22255, 22256, 22258, 22268, 22270, 22273, 22275, 22283, 22284, 22285, 22294, 22302, 22305, 22307, 22315, 22324, 22325, 22327, 22332, 22334, 22339, 22348, 22349, 22360, 22365, 22371, 22372, 22373, 22384, 22386, 22401, 22405, 22411, 22416, 22419, 22421, 22424, 22433, 22437, 22440, 22445, 22446, 22451, 22452, 22460, 22468, 22469, 22471, 22477, 22482, 22484, 22486, 22501, 22510, 22515, 22516, 22517, 22518, 22523, 22524, 22525, 22528, 22532, 22534, 22546, 22565, 22572, 22581, 22584, 22597, 22602, 22610, 22618, 22621, 22626, 22627, 22629, 22631, 22634, 22637, 22638, 22641, 22643, 22645, 22649, 22651, 22666, 22672, 22676, 22678, 22687, 22690, 22702, 22703, 22705, 22707, 22709, 22718, 22728, 22730, 22732, 22733, 22737, 22738, 22739, 22741, 22746, 22756, 22759, 22763, 22767, 22768, 22771, 22773, 22774, 22788, 22791, 22793, 22794, 22800, 22807, 22812, 22817, 22819, 22824, 22825, 22830, 22835, 22839, 22845, 22846, 22864, 22866, 22868, 22869, 22873, 22879, 22882, 22885, 22898, 22902, 22903, 22906, 22910, 22912, 22916, 22921, 22924, 22926, 22927, 22947, 22953, 22957, 22958, 22966, 22967, 22977, 22979, 22980, 22983, 22985, 22997, 23002, 23003, 23007, 23008, 23012, 23014, 23016, 23017, 23018, 23021, 23023, 23024, 23025, 23037, 23040, 23046, 23057, 23060, 23061, 23064, 23065, 23074, 23082, 23085, 23086, 23098, 23104, 23107, 23118, 23122, 23130, 23144, 23145, 23146, 23148, 23150, 23163, 23164, 23165, 23169, 23173, 23177, 23185, 23190, 23191, 23194, 23197, 23214, 23218, 23219, 23225, 23227, 23235, 23238, 23245, 23246, 23251, 23255, 23283, 23285, 23287, 23293, 23294, 23295, 23298, 23299, 23304, 23310, 23311, 23312, 23314, 23326, 23331, 23332, 23342, 23343, 23344, 23346, 23348, 23359, 23377, 23378, 23384, 23389, 23394, 23397, 23398, 23410, 23414, 23417, 23419, 23423, 23424, 23426, 23428, 23429, 23432, 23439, 23443, 23448, 23455, 23459, 23464, 23470, 23471, 23472, 23475, 23478, 23485, 23493, 23495, 23508, 23512, 23515, 23521, 23522, 23531, 23532, 23533, 23535, 23536, 23540, 23544, 23549, 23556, 23557, 23562, 23570, 23571, 23587, 23592, 23595, 23596, 23598, 23600, 23611, 23613, 23616, 23618, 23640, 23641, 23644, 23647, 23665, 23681, 23693, 23697, 23700, 23705, 23709, 23716, 23731, 23734, 23735, 23748, 23756, 23758, 23769, 23776, 23777, 23783, 23792, 23799, 23800, 23810, 23812, 23815, 23816, 23818, 23819, 23823, 23825, 23826, 23833, 23835, 23837, 23838, 23841, 23850, 23851, 23853, 23856, 23857, 23873, 23877, 23879, 23883, 23884, 23893, 23895, 23899, 23914, 23928, 23932, 23934, 23935, 23941, 23944, 23946, 23950, 23951, 23956, 23958, 23961, 23964, 23967, 23982, 23985, 23986, 23989, 23994, 23996, 24005, 24009, 24016, 24024, 24026, 24027, 24028, 24037, 24043, 24049, 24050, 24054, 24080, 24081, 24084, 24095, 24096, 24097, 24101, 24102, 24120, 24123, 24124, 24125, 24142, 24147, 24148, 24151, 24156, 24160, 24163, 24167, 24170, 24173, 24186, 24188, 24192, 24197, 24199, 24200, 24207, 24209, 24217, 24219, 24224, 24225, 24228, 24231, 24233, 24235, 24238, 24242, 24244, 24251, 24253, 24254, 24266, 24268, 24271, 24276, 24277, 24285, 24297, 24298, 24304, 24309, 24315, 24316, 24323, 24332, 24344, 24358, 24366, 24385, 24386, 24390, 24393, 24398, 24401, 24408, 24412, 24419, 24420, 24427, 24443, 24446, 24447, 24458, 24459, 24460, 24463, 24469, 24496, 24501, 24504, 24505, 24507, 24508, 24515, 24529, 24530, 24532, 24542, 24546, 24547, 24553, 24555, 24556, 24557, 24569, 24571, 24572, 24577, 24578, 24579, 24580, 24582, 24588, 24590, 24600, 24606, 24612, 24624, 24626, 24633, 24642, 24655, 24660, 24663, 24667, 24676, 24681, 24695, 24699, 24712, 24716, 24723, 24725, 24728, 24733, 24734, 24738, 24739, 24752, 24755, 24765, 24773, 24776, 24781, 24782, 24785, 24788, 24790, 24792, 24795, 24797, 24804, 24806, 24813, 24814, 24826, 24828, 24829, 24830, 24838, 24850, 24851, 24865, 24868, 24869, 24875, 24876, 24880, 24882, 24890, 24903, 24917, 24918, 24919, 24921, 24927, 24929, 24935, 24936, 24941, 24947, 24956, 24959, 24962, 24963, 24970, 24974, 24979, 24990, 24992, 24993, 24995, 25003, 25012, 25013, 25015, 25020, 25025, 25030, 25033, 25034, 25045, 25050, 25054, 25057, 25064, 25067, 25068, 25072, 25074, 25075, 25077, 25078, 25082, 25083, 25084, 25085, 25091, 25092, 25095, 25098, 25103, 25111, 25113, 25115, 25117, 25118, 25124, 25129, 25133, 25141, 25147, 25148, 25149, 25151, 25152, 25162, 25164, 25170, 25174, 25177, 25181, 25187, 25193, 25201, 25202, 25208, 25212, 25215, 25216, 25220, 25224, 25227, 25244, 25250, 25252, 25256, 25257, 25265, 25270, 25277, 25278, 25284, 25286, 25300, 25304, 25307, 25314, 25317, 25320, 25325, 25328, 25329, 25333, 25334, 25340, 25344, 25347, 25349, 25353, 25364, 25366, 25367, 25368, 25369, 25386, 25393, 25396, 25407, 25408, 25417, 25420, 25423, 25428, 25429, 25432, 25434, 25435, 25442, 25446, 25451, 25456, 25460, 25462, 25464, 25465, 25466, 25467, 25471, 25472, 25490, 25502, 25503, 25504, 25506, 25510, 25511, 25521, 25522, 25524, 25532, 25545, 25549, 25550, 25552, 25558, 25570, 25580, 25586, 25587, 25598, 25602, 25605, 25607, 25611, 25617, 25619, 25626, 25628, 25636, 25643, 25645, 25652, 25654, 25659, 25662, 25663, 25673, 25678, 25681, 25685, 25688, 25689, 25692, 25695, 25704, 25705, 25707, 25709, 25710, 25711, 25712, 25718, 25720, 25721, 25722, 25730, 25731, 25749, 25751, 25758, 25766, 25770, 25773, 25774, 25786, 25790, 25791, 25794, 25798, 25806, 25807, 25812, 25822, 25828, 25830, 25832, 25835, 25849, 25855, 25862, 25875, 25880, 25888, 25891, 25896, 25899, 25900, 25903, 25905, 25910, 25921, 25924, 25930, 25931, 25936, 25941, 25943, 25953, 25959, 25962, 25968, 25969, 25971, 25984, 25989, 26003, 26004, 26007, 26010, 26015, 26022, 26023, 26025, 26026, 26035, 26037, 26038, 26042, 26044, 26048, 26049, 26052, 26053, 26055, 26068, 26072, 26079, 26080, 26093, 26104, 26108, 26115, 26123, 26126, 26137, 26147, 26152, 26159, 26162, 26164, 26167, 26170, 26172, 26184, 26189, 26192, 26193, 26205, 26207, 26209, 26211, 26214, 26222, 26224, 26227, 26241, 26243, 26253, 26254, 26259, 26260, 26264, 26277, 26288, 26289, 26292, 26297, 26307, 26312, 26314, 26317, 26320, 26321, 26334, 26336, 26343, 26344, 26346, 26351, 26353, 26355, 26357, 26358, 26359, 26370, 26373, 26377, 26381, 26383, 26386, 26388, 26401, 26422, 26427, 26429, 26446, 26463, 26472, 26475, 26482, 26486, 26487, 26492, 26493, 26507, 26510, 26517, 26521, 26524, 26542, 26543, 26547, 26548, 26554, 26561, 26564, 26565, 26568, 26569, 26570, 26572, 26579, 26582, 26583, 26586, 26587, 26588, 26597, 26604, 26606, 26609, 26615, 26619, 26634, 26637, 26642, 26644, 26661, 26667, 26670, 26679, 26681, 26684, 26699, 26701, 26704, 26709, 26714, 26715, 26722, 26731, 26746, 26748, 26754, 26756, 26760, 26772, 26782, 26790, 26793, 26802, 26806, 26808, 26812, 26815, 26816, 26817, 26821, 26824, 26832, 26843, 26849, 26857, 26858, 26865, 26876, 26880, 26884, 26885, 26887, 26899, 26901, 26902, 26907, 26930, 26931, 26932, 26933, 26941, 26943, 26946, 26949, 26950, 26952, 26957, 26960, 26963, 26964, 26969, 26982, 26988, 26990, 26998, 27000, 27002, 27006, 27017, 27028, 27032, 27034, 27036, 27041, 27043, 27044, 27050, 27051, 27068, 27073, 27083, 27086, 27092, 27101, 27102, 27107, 27110, 27117, 27120, 27124, 27129, 27130, 27134, 27142, 27153, 27155, 27159, 27167, 27173, 27193, 27204, 27209, 27210, 27212, 27221, 27222, 27224, 27227, 27229, 27230, 27232, 27235, 27242, 27245, 27247, 27248, 27259, 27260, 27271, 27292, 27295, 27315, 27317, 27320, 27324, 27325, 27335, 27336, 27337, 27344, 27345, 27350, 27351, 27355, 27356, 27358, 27365, 27366, 27367, 27372, 27376, 27379, 27382, 27389, 27391, 27392, 27394, 27396, 27400, 27403, 27412, 27419, 27421, 27433, 27437, 27439, 27441, 27443, 27446, 27467, 27469, 27477, 27478, 27482, 27484, 27502, 27510, 27512, 27513, 27517, 27522, 27526, 27531, 27533, 27534, 27538, 27545, 27551, 27555, 27558, 27560, 27562, 27567, 27571, 27572, 27573, 27580, 27583, 27587, 27588, 27593, 27599, 27601, 27609, 27614, 27618, 27632, 27642, 27645, 27649, 27653, 27654, 27655, 27661, 27662, 27665, 27666, 27675, 27676, 27687, 27692, 27700, 27701, 27704, 27709, 27714, 27724, 27735, 27736, 27739, 27743, 27756, 27775, 27776, 27777, 27780, 27781, 27784, 27786, 27794, 27796, 27803, 27811, 27817, 27823, 27837, 27840, 27841, 27846, 27848, 27857, 27864, 27869, 27870, 27879, 27891, 27895, 27900, 27911, 27912, 27916, 27917, 27926, 27929, 27936, 27937, 27948, 27949, 27950, 27958, 27960, 27962, 27965, 27971, 27976, 27980, 27984, 27988, 27991, 27994, 27996, 28000, 28011, 28013, 28014, 28016, 28024, 28025, 28028, 28041, 28042, 28043, 28049, 28053, 28055, 28061, 28068, 28072, 28073, 28075, 28079, 28086, 28090, 28092, 28105, 28106, 28109, 28112, 28114, 28115, 28125, 28133, 28142, 28149, 28151, 28170, 28177, 28183, 28184, 28203, 28211, 28215, 28223, 28224, 28226, 28232, 28236, 28238, 28240, 28241, 28243, 28244, 28253, 28258, 28259, 28263, 28265, 28266, 28279, 28284, 28298, 28299, 28300, 28320, 28323, 28326, 28327, 28328, 28329, 28341, 28343, 28349, 28365, 28377, 28380, 28384, 28390, 28396, 28398, 28404, 28408, 28412, 28425, 28428, 28450, 28457, 28461, 28465, 28473, 28475, 28477, 28482, 28486, 28488, 28498, 28501, 28504, 28505, 28515, 28516, 28517, 28518, 28528, 28530, 28536, 28539, 28546, 28566, 28592, 28594, 28606, 28621, 28622, 28639, 28647, 28648, 28649, 28652, 28654, 28662, 28663, 28671, 28672, 28673, 28674, 28677, 28685, 28689, 28693, 28700, 28701, 28707, 28712, 28713, 28714, 28722, 28731, 28739, 28746, 28758, 28759, 28761, 28763, 28771, 28772, 28778, 28782, 28789, 28794, 28803, 28810, 28811, 28815, 28822, 28825, 28830, 28833, 28837, 28845, 28853, 28867, 28886, 28895, 28896, 28898, 28909, 28913, 28915, 28921, 28924, 28926, 28928, 28945, 28958, 28962, 28964, 28966, 28969, 28970, 28973, 28974, 28983, 28985, 29010, 29012, 29014, 29015, 29018, 29028, 29031, 29035, 29044, 29051, 29052, 29094, 29096, 29101, 29116, 29130, 29133, 29141, 29146, 29159, 29167, 29169, 29174, 29177, 29184, 29193, 29203, 29207, 29208, 29211, 29236, 29238, 29243, 29246, 29249, 29261, 29265, 29269, 29270, 29273, 29274, 29281, 29283, 29285, 29286, 29292, 29293, 29297, 29300, 29303, 29306, 29310, 29313, 29315, 29318, 29327, 29329, 29331, 29346, 29347, 29350, 29365, 29369, 29377, 29379, 29381, 29396, 29398, 29404, 29408, 29415, 29420, 29422, 29430, 29431, 29434, 29435, 29436, 29438, 29440, 29442, 29456, 29457, 29458, 29464, 29465, 29471, 29492, 29493, 29500, 29512, 29527, 29551, 29552, 29560, 29566, 29567, 29569, 29578, 29582, 29588, 29591, 29592, 29593, 29607, 29611, 29612, 29615, 29616, 29617, 29618, 29619, 29621, 29622, 29624, 29625, 29629, 29642, 29647, 29648, 29653, 29663, 29667, 29683, 29686, 29689, 29691, 29693, 29697, 29702, 29703, 29715, 29725, 29730, 29741, 29749, 29752, 29754, 29763, 29768, 29780, 29781, 29783, 29784, 29787, 29796, 29797, 29804, 29806, 29808, 29812, 29815, 29821, 29830, 29833, 29840, 29845, 29846, 29850, 29853, 29859, 29865, 29877, 29880, 29883, 29886, 29890, 29910, 29919, 29921, 29925, 29933, 29935, 29942, 29951, 29954, 29973, 29975, 29976, 29977, 29980, 29985, 29991, 29993, 29998, 30003, 30009, 30021, 30022, 30027, 30028, 30033, 30035, 30039, 30040, 30043, 30056, 30064, 30070, 30075, 30076, 30090, 30092, 30096, 30097, 30098, 30101, 30104, 30108, 30116, 30119, 30120, 30135, 30139, 30140, 30147, 30148, 30150, 30152, 30155, 30157, 30165, 30174, 30182, 30184, 30187, 30192, 30195, 30199, 30200, 30211, 30214, 30216, 30218, 30219, 30235, 30239, 30245, 30246, 30255, 30261, 30262, 30269, 30278, 30282, 30285, 30287, 30296, 30299, 30300, 30303, 30308, 30322, 30323, 30335, 30344, 30358, 30372, 30374, 30394, 30407, 30408, 30411, 30412, 30427, 30437, 30439, 30442, 30451, 30453, 30456, 30474, 30492, 30494, 30506, 30515, 30518, 30520, 30537, 30539, 30558, 30559, 30563, 30571, 30572, 30580, 30591, 30596, 30599, 30600, 30603, 30623, 30637, 30644, 30647, 30656, 30662, 30664, 30672, 30682, 30684, 30686, 30692, 30697, 30712, 30715, 30719, 30738, 30744, 30748, 30753, 30755, 30761, 30764, 30779, 30787, 30790, 30794, 30797, 30798, 30800, 30812, 30825, 30835, 30838, 30851, 30852, 30854, 30863, 30869, 30871, 30875, 30889, 30890, 30895, 30900, 30909, 30914, 30917, 30938, 30941, 30946, 30951, 30957, 30961, 30968, 30982, 30996, 31005, 31009, 31012, 31022, 31032, 31037, 31047, 31049, 31051, 31063, 31075, 31084, 31091, 31092, 31098, 31103, 31109, 31126, 31127, 31128, 31130, 31133, 31135, 31136, 31137, 31138, 31143, 31144, 31150, 31153, 31167, 31169, 31174, 31177, 31179, 31189, 31192, 31201, 31209, 31211, 31215, 31216, 31218, 31220, 31221, 31230, 31233, 31241, 31243, 31248, 31251, 31265, 31267, 31268, 31271, 31276, 31282, 31291, 31296, 31297, 31304, 31312, 31314, 31315, 31324, 31335, 31340, 31341, 31342, 31343, 31350, 31352, 31354, 31355, 31363, 31365, 31379, 31397, 31413, 31433, 31437, 31439, 31446, 31451, 31456, 31462, 31464, 31467, 31474, 31494, 31498, 31502, 31504, 31509, 31510, 31511, 31515, 31519, 31521, 31529, 31540, 31547, 31548, 31559, 31560, 31561, 31564, 31566, 31575, 31580, 31581, 31591, 31595, 31625, 31626, 31629, 31634, 31637, 31639, 31646, 31649, 31650, 31654, 31659, 31660, 31665, 31669, 31670, 31685, 31686, 31693, 31699, 31712, 31719, 31728, 31733, 31750, 31752, 31760, 31762, 31764, 31766, 31768, 31773, 31777, 31788, 31795, 31800, 31802, 31811, 31814, 31821, 31838, 31841, 31850, 31852, 31855, 31859, 31874, 31881, 31882, 31886, 31887, 31888, 31890, 31918, 31921, 31930, 31938, 31943, 31946, 31950, 31954, 31961, 31966, 31969, 31972, 31975, 31979, 31982, 31988, 31995, 31997, 32001, 32002, 32007, 32016, 32023, 32031, 32039, 32040, 32049, 32058, 32073, 32075, 32096, 32107, 32115, 32119, 32122, 32127, 32143, 32144, 32147, 32148, 32153, 32157, 32158, 32160, 32163, 32168, 32170, 32171, 32176, 32184, 32191, 32224, 32225, 32228, 32248, 32249, 32254, 32260, 32283, 32288, 32290, 32303, 32308, 32313, 32314, 32319, 32323, 32327, 32328, 32333, 32335, 32337, 32341, 32343, 32344, 32345, 32346, 32348, 32364, 32365, 32369, 32378, 32384, 32396, 32410, 32412, 32416, 32424, 32428, 32430, 32451, 32453, 32460, 32466, 32469, 32471, 32472, 32480, 32483, 32487, 32493, 32495, 32498, 32503, 32505, 32524, 32530, 32534, 32543, 32555, 32563, 32566, 32588, 32592, 32595, 32603, 32607, 32624, 32625, 32626, 32636, 32641, 32645, 32653, 32655, 32663, 32672, 32680, 32681, 32683, 32689, 32690, 32708, 32709, 32713, 32728, 32735, 32745, 32746, 32749, 32751, 32759, 32768, 32770, 32773, 32785, 32788, 32797, 32817, 32822, 32837, 32848, 32854, 32866, 32878, 32879, 32891, 32895, 32897, 32904, 32917, 32925, 32940, 32942, 32944, 32949, 32956, 32958, 32961, 32969, 32974, 32979, 32988, 33007, 33011, 33017, 33021, 33025, 33026, 33027, 33034, 33040, 33041, 33060, 33061, 33062, 33064, 33073, 33074, 33075, 33088, 33089, 33093, 33094, 33095, 33105, 33107, 33108, 33110, 33115, 33119, 33120, 33124, 33129, 33130, 33149, 33153, 33160, 33164, 33167, 33170, 33188, 33190, 33203, 33204, 33208, 33214, 33218, 33224, 33226, 33229, 33233, 33239, 33241, 33251, 33254, 33256, 33260, 33263, 33265, 33268, 33271, 33273, 33276, 33279, 33283, 33284, 33286, 33295, 33301, 33303, 33318, 33319, 33330, 33331, 33333, 33350, 33358, 33363, 33367, 33382, 33389, 33395, 33401, 33413, 33418, 33419, 33431, 33435, 33436, 33443, 33445, 33448, 33453, 33461, 33464, 33466, 33472, 33488, 33490, 33498, 33503, 33508, 33513, 33528, 33532, 33534, 33539, 33550, 33558, 33565, 33567, 33582, 33586, 33590, 33601, 33606, 33617, 33621, 33625, 33633, 33652, 33654, 33658, 33662, 33667, 33673, 33675, 33685, 33687, 33691, 33692, 33696, 33702, 33712, 33713, 33714, 33720, 33725, 33726, 33730, 33734, 33735, 33742, 33744, 33750, 33756, 33759, 33762, 33764, 33765, 33767, 33773, 33776, 33777, 33779, 33782, 33789, 33791, 33793, 33807, 33810, 33812, 33829, 33850, 33862, 33868, 33872, 33877, 33887, 33892, 33911, 33914, 33917, 33924, 33931, 33932, 33935, 33939, 33950, 33955, 33970, 33971, 33972, 33973, 33992, 33993, 34008, 34016, 34019, 34023, 34030, 34033, 34035, 34045, 34048, 34051, 34082, 34085, 34088, 34094, 34098, 34099, 34105, 34109, 34111, 34112, 34114, 34124, 34126, 34129, 34131, 34136, 34145, 34150, 34154, 34156, 34157, 34163, 34167, 34171, 34192, 34202, 34209, 34211, 34215, 34217, 34219, 34234, 34237, 34250, 34258, 34267, 34276, 34282, 34290, 34294, 34299, 34303, 34310, 34317, 34318, 34322, 34325, 34327, 34337, 34338, 34346, 34352, 34356, 34366, 34369, 34373, 34377, 34378, 34383, 34388, 34410, 34414, 34420, 34429, 34435, 34439, 34445, 34449, 34461, 34468, 34475, 34493, 34500, 34512, 34513, 34523, 34526, 34531, 34532, 34535, 34538, 34559, 34576, 34577, 34578, 34580, 34618, 34624, 34633, 34636, 34639, 34647, 34652, 34653, 34666, 34691, 34707, 34717, 34725, 34736, 34742, 34746, 34749, 34762, 34765, 34773, 34775, 34780, 34795, 34802, 34808, 34811, 34827, 34835, 34841, 34850, 34860, 34864, 34867, 34872, 34880, 34885, 34895, 34901, 34905, 34912, 34919, 34925, 34927, 34932, 34937, 34945, 34973, 34991, 35000, 35003, 35007, 35015, 35016, 35028, 35039, 35044, 35060, 35061, 35068, 35076, 35094, 35095, 35096, 35102, 35103, 35122, 35126, 35133, 35135, 35139, 35140, 35141, 35155, 35166, 35169, 35174, 35175, 35195, 35207, 35209, 35216, 35222, 35226, 35234, 35253, 35257, 35264, 35265, 35282, 35288, 35290, 35296, 35299, 35313, 35331, 35342, 35355, 35358, 35368, 35374, 35393, 35395, 35399, 35406, 35414, 35425, 35426, 35430, 35435, 35436, 35442, 35452, 35456, 35459, 35463, 35466, 35473, 35474, 35475, 35477, 35478, 35479, 35481, 35482, 35497, 35503, 35516, 35525, 35538, 35539, 35542, 35546, 35548, 35559, 35562, 35564, 35571, 35590, 35594, 35600, 35604, 35608, 35617, 35629, 35643, 35649, 35661, 35672, 35679, 35680, 35682, 35684, 35688, 35704, 35705, 35711, 35718, 35721, 35727, 35729, 35732, 35733, 35736, 35742, 35750, 35761, 35778, 35792, 35793, 35808, 35811, 35815, 35821, 35826, 35838, 35843, 35848, 35852, 35859, 35864, 35865, 35870, 35873, 35875, 35880, 35885, 35891, 35893, 35894, 35901, 35910, 35916, 35917, 35923, 35931, 35932, 35934, 35936, 35937, 35943, 35944, 35945, 35960, 35970, 35971, 35978, 35980, 35983, 36002, 36004, 36012, 36034, 36040, 36059, 36071, 36076, 36079, 36110, 36111, 36125, 36139, 36141, 36142, 36146, 36151, 36154, 36158, 36162, 36165, 36176, 36182, 36188, 36193, 36211, 36213, 36215, 36218, 36234, 36238, 36268, 36274, 36275, 36283, 36289, 36295, 36303, 36312, 36319, 36320, 36323, 36334, 36336, 36347, 36352, 36353, 36354, 36358, 36361, 36371, 36375, 36377, 36381, 36397, 36406, 36412, 36414, 36427, 36435, 36465, 36472, 36474, 36479, 36480, 36491, 36497, 36510, 36513, 36514, 36515, 36516, 36519, 36524, 36541, 36546, 36566, 36569, 36571, 36584, 36585, 36586, 36588, 36589, 36590, 36591, 36595, 36597, 36602, 36616, 36622, 36633, 36634, 36641, 36642, 36649, 36655, 36658, 36660, 36667, 36676, 36687, 36700, 36703, 36704, 36714, 36730, 36733, 36744, 36751, 36764, 36766, 36774, 36775, 36783, 36792, 36793, 36797, 36799, 36802, 36817, 36823, 36833, 36851, 36878, 36879, 36883, 36889, 36893, 36900, 36901, 36902, 36933, 36947, 36948, 36953, 36965, 36980, 36983, 36985, 36988, 36990, 36999, 37005, 37019, 37024, 37029, 37043, 37052, 37063, 37066, 37073, 37079, 37088, 37093, 37098, 37118, 37134, 37138, 37140, 37157, 37160, 37169, 37170, 37172, 37174, 37183, 37197, 37198, 37201, 37202, 37208, 37238, 37249, 37274, 37280, 37281, 37291, 37296, 37306, 37310, 37312, 37316, 37339, 37348, 37350, 37361, 37367, 37372, 37393, 37400, 37401, 37403, 37405, 37427, 37431, 37435, 37436, 37437, 37443, 37448, 37453, 37456, 37471, 37481, 37488, 37490, 37493, 37502, 37514, 37531, 37537, 37539, 37556, 37557, 37564, 37568, 37589, 37597, 37611, 37617, 37629, 37630, 37631, 37638, 37640, 37641, 37647, 37652, 37653, 37655, 37657, 37666, 37679, 37680, 37681, 37683, 37685, 37693, 37707, 37715, 37725, 37729, 37730, 37732, 37734, 37740, 37747, 37749, 37753, 37761, 37774, 37781, 37792, 37794, 37796, 37797, 37839, 37844, 37847, 37852, 37860, 37877, 37890, 37894, 37908, 37918, 37922, 37939, 37951, 37955, 37956, 37963, 37990, 37992, 37996, 37997, 38005, 38010, 38011, 38016, 38044, 38056, 38094, 38097, 38098, 38100, 38116, 38125, 38127, 38136, 38137, 38154, 38157, 38175, 38176, 38186, 38188, 38192, 38200, 38205, 38211, 38224, 38225, 38226, 38243, 38247, 38250, 38251, 38252, 38256, 38258, 38269, 38289, 38318, 38321, 38327, 38329, 38333, 38334, 38339, 38344, 38346, 38361, 38362, 38365, 38389, 38394, 38397, 38400, 38402, 38406, 38409, 38416, 38429, 38430, 38437, 38442, 38449, 38453, 38460, 38468, 38473, 38488, 38495, 38508, 38523, 38540, 38553, 38556, 38564, 38568, 38581, 38585, 38587, 38595, 38603, 38608, 38615, 38617, 38618, 38619, 38625, 38627, 38633, 38643, 38645, 38649, 38661, 38666, 38667, 38680, 38686, 38696, 38702, 38704, 38706, 38710, 38715, 38729, 38750, 38754, 38758, 38762, 38764, 38778, 38786, 38787, 38788, 38798, 38806, 38817, 38818, 38821, 38823, 38841, 38845, 38854, 38864, 38866, 38873, 38883, 38891, 38896, 38903, 38920, 38921, 38935, 38951, 38959, 38965, 38968, 38982, 38983, 38985, 38988, 38989, 38999, 39013, 39017, 39019, 39021, 39025, 39031, 39039, 39044, 39045, 39051, 39054, 39056, 39062, 39067, 39072, 39075, 39089, 39094, 39100, 39105, 39108, 39109, 39110, 39115, 39117, 39124, 39129, 39141, 39147, 39149, 39155, 39197, 39204, 39205, 39219, 39222, 39226, 39233, 39238, 39241, 39245, 39248, 39257, 39258, 39262, 39275, 39284, 39286, 39287, 39294, 39299, 39323, 39328, 39350, 39356, 39358, 39368, 39379, 39390, 39407, 39420, 39424, 39426, 39441, 39446, 39456, 39460, 39470, 39481, 39488, 39492, 39498, 39507, 39512, 39527, 39539, 39542, 39552, 39565, 39566, 39577, 39587, 39602, 39603, 39604, 39618, 39640, 39641, 39644, 39647, 39651, 39652, 39655, 39663, 39664, 39672, 39674, 39675, 39676, 39680, 39689, 39691, 39692, 39698, 39708, 39718, 39736, 39740, 39741, 39742, 39745, 39751, 39752, 39761, 39765, 39783, 39792, 39811, 39821, 39826, 39834, 39844, 39846, 39849, 39852, 39859, 39867, 39874, 39875, 39876, 39895, 39923, 39928, 39931, 39943, 39950, 39961, 39964, 39970, 39978, 39981, 39983, 39996, 39997, 39998, 40007, 40018, 40019, 40023, 40028, 40039, 40062, 40064, 40080, 40084, 40087, 40096, 40098, 40103, 40107, 40122, 40129, 40136, 40142, 40143, 40171, 40173, 40181, 40195, 40201, 40212, 40236, 40238, 40246, 40248, 40252, 40253, 40283, 40284, 40289, 40290, 40300, 40302, 40314, 40323, 40335, 40338, 40339, 40350, 40355, 40357, 40362, 40366, 40373, 40387, 40397, 40400, 40415, 40423, 40427, 40441, 40443, 40444, 40452, 40469, 40473, 40476, 40486, 40504, 40512, 40523, 40525, 40541, 40550, 40562, 40568, 40574, 40576, 40577, 40581, 40592, 40617, 40621, 40624, 40628, 40638, 40657, 40659, 40664, 40678, 40683, 40689, 40693, 40718, 40725, 40729, 40731, 40745, 40748, 40751, 40787, 40789, 40790, 40798, 40799, 40802, 40803, 40813, 40821, 40823, 40846, 40857, 40860, 40878, 40889, 40890, 40893, 40904, 40905, 40914, 40920, 40922, 40941, 40948, 40951, 40961, 40973, 40984, 40988, 41009, 41012, 41014, 41019, 41020, 41031, 41036, 41042, 41050, 41054, 41055, 41058, 41076, 41080, 41082, 41095, 41096, 41108, 41125, 41127, 41135, 41139, 41145, 41147, 41150, 41152, 41167, 41177, 41178, 41182, 41185, 41187, 41211, 41212, 41221, 41223, 41224, 41227, 41229, 41244, 41260, 41270, 41275, 41276, 41291, 41302, 41327, 41329, 41330, 41332, 41356, 41359, 41372, 41388, 41396, 41403, 41419, 41439, 41440, 41447, 41453, 41457, 41476, 41484, 41491, 41496, 41499, 41510, 41513, 41514, 41515, 41520, 41523, 41538, 41547, 41553, 41557, 41562, 41567, 41569, 41582, 41587, 41590, 41602, 41611, 41633, 41634, 41641, 41642, 41643, 41647, 41657, 41662, 41664, 41670, 41689, 41713, 41717, 41720, 41738, 41755, 41763, 41772, 41774, 41778, 41784, 41795, 41804, 41813, 41848, 41850, 41863, 41866, 41868, 41882, 41895, 41915, 41930, 41931, 41932, 41933, 41951, 41954, 41955, 41961, 41968, 41979, 41982, 41994, 42000, 42001, 42018, 42029, 42049, 42051, 42054, 42055, 42060, 42065, 42072, 42080, 42084, 42085, 42095, 42099, 42100, 42103, 42108, 42120, 42127, 42138, 42144, 42151, 42155, 42166, 42169, 42170, 42171, 42177, 42182, 42191, 42199, 42208, 42237, 42249, 42258, 42267, 42270, 42281, 42284, 42289, 42303, 42307, 42314, 42318, 42320, 42326, 42328, 42344, 42345, 42349, 42351, 42359, 42360, 42368, 42369, 42370, 42371, 42377, 42382, 42384, 42398, 42412, 42417, 42418, 42423, 42424, 42425, 42436, 42441, 42446, 42450, 42451, 42458, 42465, 42467, 42476, 42480, 42481, 42490, 42491, 42499, 42504, 42507, 42517, 42529, 42535, 42559, 42583, 42584, 42589, 42613, 42629, 42642, 42644, 42655, 42664, 42678, 42685, 42695, 42700, 42703, 42705, 42716, 42726, 42727, 42728, 42741, 42744, 42753, 42768, 42794, 42809, 42810, 42812, 42815, 42820, 42821, 42824, 42826, 42837, 42849, 42850, 42863, 42873, 42889, 42898, 42901, 42903, 42912, 42918, 42936, 42950, 42959, 42962, 42964, 42992, 43000, 43001, 43015, 43022, 43030, 43070, 43072, 43075, 43082, 43083, 43088, 43117, 43120, 43129, 43138, 43143, 43147, 43150, 43181, 43187, 43188, 43195, 43197, 43204, 43217, 43220, 43224, 43230, 43244, 43248, 43254, 43260, 43268, 43274, 43294, 43309, 43311, 43318, 43327, 43329, 43334, 43364, 43365, 43369, 43374, 43375, 43386, 43388, 43400, 43408, 43410, 43439, 43459, 43460, 43469, 43471, 43472, 43478, 43487, 43489, 43510, 43532, 43540, 43546, 43555, 43565, 43568, 43589, 43591, 43614, 43615, 43619, 43625, 43644, 43647, 43650, 43669, 43687, 43697, 43703, 43715, 43716, 43719, 43731, 43736, 43740, 43741, 43765, 43774, 43782, 43792, 43804, 43830, 43831, 43832, 43837, 43844, 43850, 43854, 43867, 43872, 43884, 43896, 43897, 43900, 43913, 43924, 43937, 43938, 43967, 43977, 43982, 43986, 44001, 44007, 44025, 44026, 44030, 44052, 44059, 44094, 44095, 44098, 44102, 44107, 44115, 44133, 44142, 44148, 44149, 44155, 44157, 44171, 44173, 44174, 44182, 44183, 44189, 44195, 44201, 44208, 44212, 44213, 44232, 44237, 44254, 44264, 44265, 44268, 44284, 44312, 44313, 44318, 44331, 44349, 44350, 44372, 44373, 44383, 44388, 44397, 44399, 44401, 44403, 44404, 44408, 44429, 44431, 44439, 44444, 44457, 44487, 44491, 44504, 44543, 44546, 44550, 44556, 44564, 44569, 44572, 44586, 44639, 44646, 44647, 44657, 44670, 44671, 44683, 44706, 44712, 44722, 44732, 44734, 44738, 44739, 44752, 44777, 44778, 44782, 44789, 44798, 44811, 44816, 44817, 44839, 44866, 44868, 44877, 44880, 44883, 44896, 44904, 44917, 44920, 44921, 44925, 44932, 44935, 44937, 44947, 44949, 44962, 44966, 44967, 44968, 44969, 44984, 44986, 44996, 45004, 45035, 45044, 45051, 45052, 45061, 45065, 45070, 45079, 45083, 45106, 45110, 45112, 45115, 45127, 45140, 45148, 45161, 45191, 45193, 45203, 45205, 45217, 45218, 45226, 45227, 45232, 45239, 45243, 45247, 45263, 45271, 45277, 45299, 45305, 45309, 45326, 45328, 45332, 45341, 45345, 45347, 45349, 45351, 45356, 45361, 45369, 45374, 45388, 45392, 45398, 45403, 45417, 45438, 45441, 45447, 45452, 45462, 45465, 45474, 45477, 45487, 45492, 45522, 45525, 45543, 45562, 45563, 45569, 45574, 45583, 45592, 45607, 45620, 45621, 45632, 45636, 45640, 45665, 45668, 45680, 45687, 45690, 45691, 45693, 45696, 45715, 45720, 45724, 45726, 45727, 45735, 45759, 45771, 45776, 45784, 45822, 45847, 45867, 45874, 45881, 45888, 45891, 45899, 45902, 45906, 45918, 45920, 45930, 45941, 45943, 45944, 45967, 45993, 45994, 45999, 46004, 46025, 46026, 46028, 46051, 46055, 46066, 46070, 46073, 46076, 46089, 46091, 46096, 46113, 46118, 46121, 46124, 46127, 46138, 46142, 46148, 46166, 46171, 46184, 46194, 46197, 46206, 46227, 46230, 46261, 46262, 46271, 46272, 46290, 46298, 46306, 46318, 46325, 46333, 46352, 46361, 46365, 46373, 46375, 46379, 46398, 46399, 46420, 46422, 46426, 46429, 46431, 46449, 46451, 46455, 46479, 46480, 46484, 46485, 46491, 46499, 46508, 46522, 46525, 46529, 46531, 46533, 46539, 46549, 46550, 46552, 46560, 46574, 46596, 46598, 46599, 46607, 46608, 46609, 46617, 46625, 46631, 46633, 46636, 46657, 46663, 46668, 46670, 46676, 46678, 46683, 46692, 46705, 46723, 46726, 46727, 46729, 46730, 46734, 46736, 46737, 46754, 46755, 46756, 46767, 46772, 46773, 46780, 46792, 46808, 46814, 46819, 46828, 46830, 46867, 46914, 46923, 46925, 46930, 46931, 46932, 46950, 46966, 46991, 47004, 47013, 47022, 47029, 47034, 47060, 47067, 47069, 47089, 47095, 47096, 47103, 47110, 47132, 47140, 47154, 47160, 47161, 47163, 47164, 47183, 47187, 47193, 47198, 47204, 47219, 47226, 47232, 47240, 47241, 47246, 47248, 47272, 47279, 47283, 47286, 47289, 47319, 47321, 47326, 47338, 47339, 47354, 47364, 47387, 47388, 47389, 47404, 47410, 47433, 47438, 47442, 47446, 47450, 47455, 47461, 47464, 47488, 47491, 47493, 47499, 47500, 47506, 47514, 47517, 47526, 47532, 47535, 47558, 47563, 47567, 47580, 47587, 47589, 47590, 47614, 47622, 47640, 47651, 47656, 47661, 47665, 47675, 47681, 47692, 47696, 47707, 47709, 47720, 47721, 47734, 47737, 47739, 47746, 47753, 47755, 47770, 47771, 47775, 47781, 47793, 47796, 47807, 47808, 47822, 47825, 47843, 47848, 47849, 47851, 47875, 47894, 47897, 47907, 47914, 47929, 47930, 47932, 47939, 47940, 47944, 47946, 47948, 47953, 47954, 47957, 47975, 47980, 47983, 47988, 47989, 47993, 48002, 48005, 48008, 48010, 48025, 48028, 48030, 48039, 48047, 48055, 48056, 48057, 48070, 48085, 48086, 48088, 48093, 48098, 48118, 48129, 48134, 48142, 48143, 48149, 48152, 48162, 48166, 48169, 48170, 48184, 48193, 48200, 48219, 48230, 48233, 48235, 48264, 48271, 48282, 48286, 48296, 48304, 48308, 48326, 48330, 48340, 48350, 48352, 48359, 48366, 48369, 48382, 48384, 48389, 48393, 48396, 48397, 48402, 48403, 48421, 48444, 48455, 48470, 48476, 48482, 48483, 48507, 48518, 48528, 48529, 48575, 48594, 48601, 48603, 48611, 48621, 48624, 48632, 48636, 48651, 48657, 48660, 48662, 48671, 48673, 48683, 48684, 48699, 48700, 48706, 48707, 48716, 48721, 48743, 48755, 48781, 48785, 48791, 48794, 48819, 48821, 48844, 48846, 48857, 48890, 48922, 48926, 48939, 48955, 48994, 49002, 49011, 49017, 49018, 49024, 49034, 49038, 49039, 49040, 49042, 49058, 49070, 49085, 49096, 49105, 49107, 49108, 49112, 49122, 49123, 49135, 49141, 49143, 49146, 49195, 49197, 49198, 49211, 49217, 49219, 49222, 49227, 49231, 49238, 49245, 49259, 49264, 49267, 49282, 49300, 49310, 49314, 49316, 49318, 49320, 49322, 49325, 49329, 49332, 49338, 49340, 49347, 49348, 49364, 49382, 49388, 49401, 49403, 49422, 49458, 49463, 49464, 49465, 49473, 49481, 49485, 49491, 49503, 49512, 49516, 49531, 49558, 49564, 49569, 49573, 49579, 49591, 49596, 49625, 49629, 49657, 49667, 49677, 49682, 49690, 49693, 49695, 49724, 49738, 49760, 49761, 49769, 49779, 49788, 49811, 49820, 49825, 49835, 49839, 49855, 49869, 49876, 49889, 49906, 49916, 49918, 49927, 49930, 49940, 49948, 49971, 49972, 49986, 49988, 49991, 49992, 49995, 50003, 50011, 50029, 50046, 50054, 50067, 50069, 50076, 50079, 50083, 50098, 50099, 50110, 50123, 50128, 50131, 50133, 50151, 50161, 50163, 50181, 50187, 50196, 50215, 50231, 50233, 50240, 50244, 50247, 50252, 50291, 50322, 50333, 50342, 50345, 50346, 50370, 50379, 50394, 50398, 50402, 50407, 50422, 50428, 50438, 50460, 50461, 50465, 50488, 50504, 50506, 50509, 50525, 50539, 50564, 50570, 50572, 50589, 50613, 50636, 50638, 50640, 50656, 50693, 50706, 50713, 50714, 50716, 50717, 50734, 50746, 50757, 50763, 50781, 50799, 50821, 50848, 50850, 50853, 50857, 50863, 50872, 50878, 50882, 50897, 50903, 50906, 50915, 50920, 50921, 50932, 50937, 50940, 50952, 50958, 50959, 50976, 50984, 50992, 51000, 51008, 51019, 51021, 51028, 51030, 51034, 51047, 51070, 51087, 51099, 51102, 51103, 51112, 51119, 51133, 51138, 51139, 51151, 51161, 51184, 51190, 51192, 51201, 51206, 51217, 51241, 51245, 51253, 51273, 51274, 51285, 51287, 51297, 51301, 51308, 51323, 51343, 51344, 51368, 51376, 51381, 51390, 51405, 51407, 51418, 51422, 51442, 51443, 51453, 51473, 51487, 51505, 51520, 51527, 51529, 51534, 51561, 51573, 51586, 51587, 51598, 51623, 51631, 51660, 51666, 51674, 51676, 51690, 51692, 51694, 51730, 51748, 51775, 51792, 51819, 51836, 51842, 51850, 51851, 51852, 51883, 51887, 51898, 51899, 51902, 51924, 51948, 51953, 51960, 51974, 51983, 51985, 52000, 52010, 52014, 52032, 52034, 52041, 52048, 52061, 52062, 52065, 52068, 52091, 52126, 52131, 52139, 52145, 52152, 52167, 52175, 52187, 52191, 52221, 52228, 52230, 52264, 52275, 52289, 52293, 52294, 52298, 52300, 52303, 52316, 52340, 52348, 52349, 52351, 52354, 52369, 52393, 52400, 52405, 52413, 52426, 52444, 52448, 52456, 52457, 52462, 52464, 52465, 52471, 52478, 52483, 52487, 52488, 52492, 52500, 52508, 52513, 52516, 52519, 52520, 52549, 52604, 52609, 52610, 52615, 52634, 52638, 52649, 52652, 52659, 52671, 52680, 52689, 52696, 52701, 52711, 52722, 52735, 52736, 52750, 52761, 52778, 52782, 52783, 52787, 52790, 52798, 52824, 52835, 52837, 52843, 52850, 52851, 52853, 52865, 52870, 52883, 52898, 52900, 52903, 52904, 52910, 52915, 52916, 52920, 52925, 52940, 52943, 52961, 52970, 52971, 52976, 52979, 52993, 52998, 53002, 53007, 53009, 53014, 53017, 53019, 53021, 53030, 53032, 53037, 53041, 53057, 53064, 53070, 53074, 53086, 53125, 53136, 53140, 53150, 53159, 53164, 53187, 53188, 53221, 53230, 53238, 53246, 53253, 53264, 53270, 53277, 53297, 53343, 53367, 53379, 53390, 53396, 53406, 53415, 53424, 53437, 53440, 53442, 53449, 53454, 53457, 53459, 53463, 53470, 53472, 53483, 53499, 53518, 53525, 53530, 53547, 53559, 53566, 53577, 53579, 53580, 53615, 53624, 53630, 53631, 53649, 53651, 53678, 53700, 53703, 53730, 53733, 53739, 53754, 53772, 53779, 53782, 53790, 53797, 53800, 53833, 53843, 53845, 53847, 53863, 53865, 53873, 53878, 53881, 53893, 53897, 53915, 53916, 53924, 53939, 53953, 53961, 53968, 53974, 53977, 53983, 53985, 53991, 53992, 53993, 54004, 54006, 54031, 54032, 54035, 54040, 54042, 54051, 54056, 54061, 54102, 54113, 54117, 54130, 54133, 54150, 54205, 54211, 54220, 54221, 54231, 54237, 54242, 54243, 54249, 54250, 54254, 54264, 54266, 54269, 54315, 54324, 54338, 54352, 54360, 54361, 54369, 54386, 54392, 54397, 54409, 54452, 54453, 54455, 54458, 54488, 54489, 54507, 54509, 54528, 54532, 54543, 54551, 54557, 54580, 54589, 54617, 54622, 54638, 54656, 54668, 54688, 54697, 54700, 54703, 54704, 54714, 54731, 54738, 54757, 54771, 54789, 54791, 54820, 54821, 54830, 54842, 54865, 54874, 54879, 54908, 54917, 54929, 54931, 54935, 54938, 54946, 54964, 54965, 54966, 54987, 54997, 55003, 55008, 55022, 55024, 55045, 55047, 55079, 55089, 55090, 55091, 55116, 55122, 55125, 55135, 55148, 55160, 55179, 55182, 55201, 55204, 55207, 55221, 55224, 55237, 55278, 55288, 55294, 55308, 55314, 55316, 55317, 55333, 55334, 55336, 55346, 55348, 55352, 55399, 55400, 55402, 55404, 55406, 55411, 55438, 55445, 55464, 55465, 55471, 55475, 55492, 55493, 55509, 55512, 55526, 55535, 55593, 55606, 55616, 55637, 55657, 55665, 55670, 55671, 55677, 55678, 55680, 55686, 55687, 55688, 55689, 55692, 55698, 55718, 55722, 55734, 55735, 55761, 55774, 55775, 55787, 55792, 55796, 55803, 55851, 55856, 55872, 55877, 55878, 55882, 55893, 55903, 55906, 55914, 55922, 55945, 55950, 55971, 55979, 56002, 56004, 56009, 56028, 56031, 56040, 56041, 56075, 56091, 56095, 56096, 56098, 56111, 56121, 56124, 56128, 56129, 56132, 56136, 56140, 56147, 56149, 56163, 56166, 56168, 56183, 56209, 56212, 56216, 56240, 56246, 56300, 56320, 56340, 56341, 56346, 56364, 56373, 56374, 56377, 56378, 56380, 56391, 56400, 56415, 56435, 56445, 56462, 56465, 56468, 56470, 56478, 56484, 56489, 56497, 56502, 56507, 56542, 56546, 56548, 56555, 56571, 56578, 56598, 56606, 56617, 56672, 56684, 56690, 56703, 56705, 56727, 56728, 56738, 56739, 56757, 56764, 56780, 56785, 56797, 56801, 56814, 56819, 56826, 56830, 56841, 56845, 56871, 56872, 56894, 56896, 56899, 56911, 56928, 56930, 56931, 56933, 56957, 56992, 56999, 57006, 57013, 57014, 57020, 57055, 57060, 57081, 57104, 57106, 57120, 57121, 57122, 57125, 57131, 57147, 57151, 57153, 57163, 57172, 57181, 57200, 57206, 57208, 57236, 57238, 57248, 57250, 57265, 57267, 57287, 57296, 57297, 57304, 57307, 57316, 57353, 57369, 57371, 57385, 57389, 57391, 57393, 57394, 57398, 57400, 57402, 57413, 57435, 57436, 57441, 57449, 57450, 57467, 57479, 57489, 57491, 57496, 57501, 57513, 57517, 57524, 57538, 57553, 57554, 57555, 57559, 57567, 57573, 57576, 57582, 57597, 57615, 57653, 57658, 57664, 57668, 57675, 57678, 57691, 57710, 57714, 57716, 57728, 57735, 57739, 57756, 57765, 57804, 57805, 57810, 57825, 57835, 57838, 57844, 57871, 57873, 57874, 57876, 57878, 57892, 57893, 57901, 57909, 57913, 57925, 57928, 57946, 57949, 57956, 57972, 57986, 58011, 58064, 58078, 58115, 58118, 58133, 58143, 58146, 58155, 58169, 58174, 58183, 58188, 58196, 58225, 58235, 58236, 58244, 58246, 58251, 58258, 58270, 58289, 58302, 58304, 58317, 58333, 58334, 58351, 58371, 58391, 58405, 58423, 58433, 58435, 58462, 58463, 58469, 58477, 58480, 58487, 58516, 58518, 58533, 58539, 58564, 58571, 58576, 58589, 58603, 58607, 58614, 58621, 58634, 58655, 58688, 58694, 58704, 58706, 58710, 58713, 58721, 58722, 58725, 58728, 58730, 58746, 58768, 58783, 58790, 58793, 58831, 58868, 58877, 58883, 58901, 58917, 58920, 58927, 58941, 59025, 59036, 59040, 59054, 59056, 59064, 59066, 59072, 59076, 59077, 59100, 59111, 59124, 59126, 59138, 59141, 59142, 59143, 59144, 59167, 59176, 59177, 59187, 59190, 59200, 59205, 59207, 59227, 59229, 59234, 59237, 59239, 59243, 59273, 59276, 59291, 59293, 59310, 59315, 59317, 59326, 59347, 59365, 59367, 59373, 59386, 59392, 59400, 59408, 59429, 59431, 59432, 59438, 59440, 59442, 59446, 59454, 59460, 59467, 59476, 59478, 59501, 59513, 59514, 59521, 59543, 59571, 59600, 59609, 59615, 59624, 59626, 59632, 59654, 59661, 59666, 59669, 59678, 59713, 59724, 59728, 59735, 59749, 59770, 59771, 59789, 59794, 59802, 59806, 59810, 59823, 59833, 59869, 59876, 59907, 59920, 59929, 59934, 59936, 59942, 59951, 59970, 59987, 59988, 59993, 59999, 60004, 60018, 60051, 60055, 60059, 60065, 60071, 60099, 60101, 60120, 60162, 60163, 60168, 60173, 60190, 60197, 60200, 60210, 60242, 60246, 60258, 60275, 60286, 60296, 60303, 60307, 60322, 60325, 60337, 60352, 60363, 60372, 60395, 60404, 60406, 60420, 60427, 60436, 60445, 60467, 60470, 60480, 60489, 60512, 60517, 60518, 60523, 60527, 60528, 60585, 60594, 60609, 60610, 60628, 60637, 60638, 60683, 60735, 60756, 60777, 60783, 60786, 60794, 60797, 60804, 60815, 60819, 60826, 60828, 60831, 60836, 60840, 60847, 60851, 60854, 60894, 60915, 60922, 60936, 60943, 60947, 60955, 60967, 60987, 61002, 61005, 61010, 61021, 61036, 61051, 61098, 61103, 61107, 61144, 61148, 61160, 61161, 61170, 61185, 61205, 61209, 61220, 61228, 61235, 61240, 61249, 61265, 61267, 61274, 61282, 61299, 61302, 61310, 61312, 61316, 61338, 61361, 61368, 61369, 61374, 61378, 61383, 61384, 61386, 61387, 61389, 61416, 61429, 61446, 61456, 61478, 61483, 61501, 61507, 61522, 61527, 61536, 61544, 61551, 61553, 61555, 61557, 61568, 61569, 61580, 61584, 61603, 61639, 61663, 61672, 61674, 61688, 61695, 61725, 61732, 61743, 61744, 61774, 61782, 61791, 61794, 61799, 61806, 61809, 61826, 61837, 61844, 61857, 61869, 61880, 61885, 61898, 61899, 61908, 61909, 61921, 61924, 61934, 61937, 61946, 61961, 61971, 61982, 62035, 62076, 62078, 62086, 62088, 62094, 62115, 62116, 62117, 62131, 62140, 62142, 62194, 62236, 62253, 62261, 62262, 62266, 62279, 62292, 62328, 62336, 62355, 62359, 62361, 62366, 62377, 62401, 62404, 62411, 62415, 62420, 62421, 62446, 62453, 62463, 62475, 62532, 62546, 62550, 62570, 62586, 62588, 62611, 62620, 62634, 62643, 62657, 62673, 62680, 62683, 62690, 62700, 62719, 62746, 62774, 62782, 62792, 62796, 62797, 62807, 62810, 62815, 62840, 62861, 62862, 62906, 62910, 62916, 62920, 62943, 62951, 62966, 62976, 62980, 62982, 62984, 62991, 62992, 63001, 63008, 63011, 63020, 63029, 63032, 63040, 63049, 63056, 63062, 63068, 63098, 63106, 63113, 63128, 63135, 63148, 63149, 63151, 63157, 63165, 63175, 63188, 63200, 63205, 63206, 63212, 63219, 63250, 63259, 63262, 63272, 63276, 63285, 63319, 63339, 63352, 63361, 63367, 63368, 63378, 63386, 63398, 63409, 63414, 63421, 63429, 63430, 63452, 63456, 63471, 63512, 63520, 63522, 63523, 63527, 63532, 63557, 63561, 63563, 63575, 63600, 63613, 63628, 63653, 63659, 63682, 63687, 63691, 63704, 63707, 63723, 63725, 63726, 63731, 63745, 63755, 63795, 63799, 63809, 63815, 63816, 63819, 63851, 63858, 63871, 63879, 63885, 63896, 63908, 63927, 63933, 63945, 63946, 63955, 63961, 63982, 63987, 64009, 64016, 64034, 64042, 64045, 64070, 64075, 64151, 64162, 64168, 64173, 64195, 64198, 64206, 64207, 64208, 64214, 64226, 64228, 64234, 64240, 64258, 64263, 64268, 64285, 64288, 64294, 64295, 64298, 64304, 64312, 64316, 64358, 64381, 64406, 64422, 64448, 64452, 64455, 64462, 64467, 64479, 64483, 64494, 64495, 64498, 64503, 64513, 64547, 64548, 64550, 64557, 64561, 64575, 64602, 64616, 64619, 64625, 64635, 64649, 64696, 64704, 64711, 64726, 64731, 64749, 64756, 64763, 64775, 64780, 64784, 64804, 64861, 64874, 64875, 64880, 64913, 64919, 64927, 64941, 64963, 64990, 64998, 65022, 65025, 65048, 65053, 65059, 65060, 65084, 65086, 65088, 65111, 65114, 65134, 65151, 65199, 65219, 65230, 65235, 65244, 65254, 65256, 65261, 65262, 65269, 65274, 65275, 65289, 65297, 65304, 65311, 65313, 65316, 65319, 65332, 65363, 65392, 65393, 65397, 65403, 65409, 65424, 65511, 65529, 65531, 65550, 65553, 65560, 65575, 65581, 65583, 65584, 65591, 65611, 65655, 65662, 65669, 65691, 65696, 65704, 65709, 65728, 65752, 65753, 65759, 65771, 65777, 65787, 65794, 65827, 65837, 65846, 65856, 65861, 65883, 65901, 65903, 65912, 65944, 65964, 65975, 65977, 65982, 65992, 66034, 66035, 66044, 66064, 66073, 66085, 66100, 66120, 66125, 66132, 66135, 66141, 66153, 66169, 66172, 66180, 66186, 66204, 66212, 66219, 66232, 66282, 66299, 66307, 66311, 66319, 66320, 66322, 66328, 66329, 66335, 66337, 66338, 66339, 66362, 66366, 66392, 66420, 66425, 66426, 66438, 66441, 66446, 66477, 66480, 66486, 66487, 66516, 66522, 66526, 66532, 66533, 66538, 66540, 66561, 66581, 66599, 66602, 66612, 66631, 66633, 66662, 66672, 66702, 66713, 66714, 66732, 66740, 66745, 66746, 66749, 66756, 66762, 66768, 66769, 66771, 66775, 66783, 66789, 66793, 66814, 66833, 66835, 66848, 66856, 66858, 66867, 66869, 66870, 66897, 66925, 66926, 66958, 66962, 66967, 66969, 66976, 66988, 67008, 67011, 67016, 67025, 67047, 67051, 67095, 67101, 67115, 67121, 67124, 67127, 67132, 67139, 67140, 67157, 67180, 67190, 67200, 67210, 67211, 67214, 67234, 67263, 67266, 67283, 67291, 67295, 67306, 67313, 67327, 67328, 67330, 67354, 67356, 67360, 67373, 67378, 67387, 67390, 67419, 67433, 67440, 67441, 67447, 67455, 67463, 67479, 67489, 67491, 67499, 67501, 67517, 67519, 67521, 67528, 67534, 67537, 67539, 67542, 67548, 67584, 67605, 67615, 67619, 67623, 67626, 67630, 67632, 67638, 67654, 67657, 67659, 67686, 67710, 67711, 67721, 67729, 67759, 67765, 67797, 67802, 67806, 67819, 67828, 67845, 67848, 67872, 67894, 67895, 67916, 67929, 67947, 67951, 67964, 67976, 67981, 68005, 68024, 68037, 68044, 68065, 68066, 68097, 68111, 68118, 68142, 68159, 68163, 68194, 68197, 68219, 68230, 68275, 68278, 68288, 68290, 68297, 68318, 68336, 68345, 68346, 68360, 68373, 68376, 68392, 68393, 68394, 68397, 68410, 68414, 68417, 68422, 68430, 68434, 68456, 68471, 68473, 68476, 68509, 68526, 68559, 68578, 68600, 68612, 68614, 68647, 68648, 68659, 68672, 68680, 68687, 68692, 68701, 68711, 68733, 68742, 68752, 68754, 68761, 68763, 68766, 68769, 68777, 68792, 68797, 68802, 68816, 68819, 68835, 68841, 68849, 68874, 68890, 68895, 68912, 68923, 68929, 68935, 68936, 68949, 68950, 68951, 68952, 68954, 68957, 68960, 68980, 69041, 69058, 69072, 69089, 69090, 69093, 69095, 69098, 69107, 69111, 69149, 69184, 69207, 69246, 69250, 69268, 69285, 69294, 69305, 69321, 69330, 69343, 69355, 69377, 69402, 69409, 69416, 69446, 69449, 69458, 69464, 69472, 69481, 69497, 69504, 69505, 69512, 69514, 69515, 69540, 69551, 69569, 69628, 69631, 69634, 69635, 69644, 69660, 69684, 69687, 69702, 69711, 69736, 69739, 69743, 69744, 69748, 69761, 69777, 69781, 69785, 69792, 69846, 69851, 69883, 69886, 69892, 69905, 69913, 69921, 69924, 69928, 69931, 69942, 69963, 69970, 69984, 70007, 70013, 70022, 70023, 70042, 70049, 70097, 70112, 70118, 70126, 70143, 70145, 70147, 70157, 70174, 70178, 70206, 70209, 70221, 70294, 70307, 70313, 70316, 70371, 70385, 70392, 70435, 70446, 70448, 70496, 70505, 70508, 70512, 70524, 70527, 70535, 70543, 70546, 70561, 70573, 70575, 70588, 70592, 70605, 70613, 70617, 70664, 70684, 70702, 70703, 70714, 70733, 70735, 70760, 70807, 70818, 70825, 70826, 70827, 70884, 70896, 70915, 70921, 70927, 70942, 70965, 70966, 70980, 70984, 70988, 70997, 71014, 71046, 71065, 71068, 71084, 71093, 71097, 71126, 71128, 71144, 71145, 71152, 71159, 71160, 71183, 71187, 71196, 71222, 71231, 71253, 71257, 71267, 71277, 71295, 71300, 71301, 71323, 71324, 71328, 71336, 71339, 71376, 71404, 71408, 71411, 71428, 71433, 71440, 71453, 71458, 71462, 71465, 71481, 71486, 71507, 71520, 71524, 71534, 71538, 71552, 71565, 71571, 71585, 71588, 71592, 71597, 71603, 71623, 71633, 71642, 71654, 71655, 71659, 71662, 71710, 71743, 71750, 71755, 71759, 71766, 71771, 71779, 71807, 71808, 71813, 71818, 71821, 71831, 71876, 71878, 71881, 71921, 71925, 71946, 71950, 71970, 72010, 72020, 72039, 72073, 72089, 72091, 72093, 72101, 72113, 72119, 72128, 72132, 72149, 72156, 72172, 72179, 72184, 72194, 72218, 72281, 72337, 72345, 72346, 72364, 72374, 72378, 72384, 72388, 72399, 72401, 72405, 72420, 72458, 72474, 72490, 72493, 72550, 72553, 72557, 72560, 72577, 72622, 72632, 72677, 72694, 72710, 72787, 72792, 72795, 72797, 72799, 72801, 72805, 72812, 72814, 72826, 72830, 72853, 72870, 72887, 72888, 72906, 72919, 72957, 72958, 72963, 72975, 73013, 73016, 73017, 73047, 73052, 73053, 73055, 73064, 73071, 73074, 73093, 73096, 73104, 73117, 73118, 73119, 73123, 73129, 73135, 73146, 73165, 73174, 73193, 73220, 73223, 73227, 73260, 73272, 73285, 73313, 73332, 73341, 73360, 73377, 73401, 73406, 73407, 73424, 73432, 73445, 73455, 73486, 73507, 73513, 73520, 73533, 73546, 73549, 73560, 73574, 73585, 73596, 73608, 73619, 73620, 73630, 73635, 73642, 73643, 73666, 73676, 73677, 73681, 73682, 73691, 73695, 73717, 73728, 73732, 73791, 73802, 73815, 73825, 73828, 73829, 73842, 73858, 73875, 73885, 73893, 73910, 73912, 73916, 73922, 73952, 73971, 73982, 74047, 74050, 74060, 74068, 74096, 74113, 74122, 74137, 74153, 74182, 74194, 74206, 74223, 74224, 74270, 74284, 74293, 74321, 74322, 74330, 74336, 74345, 74363, 74366, 74420, 74468, 74470, 74483, 74484, 74492, 74505, 74551, 74554, 74563, 74568, 74574, 74579, 74614, 74641, 74643, 74653, 74661, 74669, 74695, 74700, 74708, 74714, 74730, 74734, 74743, 74757, 74762, 74768, 74799, 74803, 74806, 74828, 74839, 74840, 74854, 74855, 74861, 74864, 74872, 74877, 74880, 74897, 74902, 74913, 74917, 74923, 74945, 74970, 74988, 75027, 75032, 75050, 75067, 75106, 75108, 75115, 75125, 75150, 75166, 75177, 75210, 75230, 75235, 75237, 75275, 75295, 75297, 75305, 75318, 75328, 75329, 75330, 75332, 75342, 75349, 75351, 75352, 75354, 75356, 75389, 75396, 75401, 75412, 75423, 75435, 75477, 75479, 75505, 75507, 75509, 75515, 75516, 75527, 75528, 75530, 75551, 75561, 75563, 75579, 75583, 75599, 75600, 75604, 75605, 75645, 75652, 75657, 75661, 75711, 75724, 75740, 75741, 75751, 75758, 75767, 75800, 75810, 75812, 75823, 75832, 75851, 75853, 75899, 75922, 75932, 75941, 75955, 75959, 75983, 76006, 76031, 76037, 76046, 76049, 76066, 76068, 76100, 76108, 76151, 76168, 76179, 76195, 76225, 76259, 76289, 76308, 76322, 76360, 76373, 76388, 76436, 76475, 76506, 76511, 76516, 76521, 76523, 76524, 76531, 76563, 76588, 76591, 76598, 76599, 76624, 76680, 76713, 76717, 76735, 76751, 76760, 76776, 76787, 76800, 76827, 76828, 76892, 76930, 76932, 76938, 76945, 76952, 76954, 76980, 76990, 76991, 77019, 77028, 77041, 77046, 77047, 77084, 77110, 77154, 77167, 77175, 77217, 77231, 77252, 77256, 77260, 77264, 77267, 77269, 77285, 77291, 77295, 77299, 77302, 77303, 77333, 77340, 77352, 77358, 77370, 77376, 77385, 77397, 77407, 77420, 77439, 77443, 77450, 77459, 77474, 77506, 77516, 77542, 77589, 77609, 77612, 77615, 77625, 77650, 77680, 77684, 77704, 77708, 77716, 77727, 77742, 77750, 77796, 77839, 77875, 77884, 77894, 77898, 77912, 77926, 77929, 77944, 77959, 77976, 77987, 77999, 78020, 78025, 78043, 78054, 78060, 78061, 78062, 78064, 78081, 78096, 78106, 78111, 78112, 78121, 78128, 78157, 78161, 78183, 78194, 78198, 78223, 78224, 78249, 78252, 78253, 78284, 78296, 78297, 78301, 78312, 78315, 78320, 78347, 78349, 78361, 78365, 78416, 78465, 78474, 78479, 78483, 78503, 78535, 78536, 78550, 78571, 78573, 78582, 78592, 78603, 78643, 78654, 78668, 78690, 78724, 78741, 78766, 78773, 78795, 78802, 78823, 78860, 78872, 78885, 78891, 78898, 78899, 78919, 78951, 78971, 78984, 78998, 79017, 79027, 79046, 79062, 79101, 79114, 79133, 79141, 79149, 79174, 79207, 79299, 79304, 79317, 79319, 79336, 79346, 79388, 79432, 79442, 79445, 79470, 79476, 79488, 79503, 79517, 79530, 79531, 79562, 79572, 79592, 79593, 79596, 79602, 79603, 79608, 79609, 79650, 79657, 79665, 79667, 79724, 79726, 79731, 79733, 79744, 79752, 79762, 79774, 79781, 79783, 79801, 79805, 79823, 79845, 79847, 79894, 79909, 79920, 79925, 79968, 79969, 79979, 79987, 80017, 80018, 80019, 80039, 80045, 80077, 80086, 80088, 80089, 80109, 80113, 80126, 80137, 80149, 80172, 80174, 80203, 80205, 80222, 80234, 80246, 80248, 80250, 80255, 80265, 80291, 80306, 80313, 80316, 80321, 80345, 80347, 80351, 80357, 80403, 80419, 80425, 80432, 80435, 80439, 80445, 80456, 80466, 80470, 80480, 80483, 80487, 80516, 80543, 80544, 80555, 80559, 80572, 80573, 80612, 80650, 80657, 80681, 80725, 80747, 80757, 80761, 80789, 80818, 80832, 80842, 80851, 80854, 80872, 80873, 80879, 80897, 80899, 80922, 80931, 80996, 81003, 81029, 81042, 81100, 81106, 81122, 81124, 81125, 81137, 81169, 81179, 81183, 81185, 81194, 81222, 81259, 81288, 81294, 81295, 81308, 81367, 81370, 81402, 81404, 81448, 81451, 81474, 81487, 81499, 81523, 81539, 81544, 81555, 81582, 81588, 81689, 81705, 81715, 81728, 81748, 81756, 81788, 81819, 81844, 81859, 81885, 81897, 81917, 81921, 81925, 81944, 81969, 82000, 82015, 82017, 82024, 82035, 82078, 82089, 82094, 82102, 82115, 82116, 82129, 82139, 82168, 82170, 82178, 82188, 82191, 82206, 82248, 82253, 82256, 82269, 82279, 82295, 82322, 82323, 82353, 82365, 82382, 82391, 82400, 82428, 82431, 82435, 82446, 82490, 82495, 82501, 82508, 82515, 82518, 82520, 82562, 82593, 82634, 82656, 82686, 82698, 82699, 82704, 82729, 82744, 82808, 82816, 82819, 82833, 82841, 82848, 82855, 82860, 82869, 82891, 82892, 82908, 82911, 82923, 82930, 82944, 82951, 82954, 82956, 82980, 82987, 83081, 83082, 83090, 83108, 83115, 83123, 83127, 83146, 83155, 83174, 83185, 83193, 83208, 83212, 83260, 83264, 83273, 83274, 83317, 83331, 83336, 83359, 83374, 83389, 83392, 83445, 83481, 83484, 83485, 83524, 83545, 83548, 83570, 83603, 83608, 83616, 83648, 83652, 83662, 83671, 83681, 83698, 83706, 83719, 83728, 83739, 83749, 83750, 83756, 83822, 83859, 83876, 83879, 83889, 83893, 83924, 83934, 83943, 83947, 83951, 83960, 83965, 83999, 84039, 84055, 84068, 84069, 84086, 84090, 84095, 84109, 84137, 84142, 84171, 84203, 84212, 84228, 84243, 84250, 84270, 84278, 84284, 84290, 84298, 84338, 84398, 84401, 84454, 84457, 84485, 84529, 84540, 84554, 84563, 84567, 84575, 84576, 84591, 84673, 84713, 84716, 84717, 84728, 84770, 84776, 84778, 84791, 84796, 84805, 84818, 84823, 84888, 84890, 84903, 84915, 84921, 84934, 84943, 84969, 84980, 85000, 85027, 85045, 85062, 85074, 85116, 85123, 85130, 85144, 85146, 85152, 85154, 85160, 85175, 85213, 85229, 85232, 85236, 85249, 85286, 85305, 85339, 85343, 85347, 85349, 85357, 85363, 85364, 85378, 85400, 85429, 85439, 85458, 85465, 85491, 85497, 85510, 85514, 85527, 85544, 85549, 85553, 85596, 85625, 85677, 85694, 85733, 85739, 85763, 85765, 85770, 85779, 85789, 85797, 85802, 85811, 85857, 85864, 85881, 85892, 85894, 85912, 85973, 85978, 85985, 85993, 86020, 86037, 86047, 86055, 86097, 86100, 86170, 86171, 86201, 86204, 86236, 86272, 86294, 86297, 86376, 86379, 86403, 86412, 86424, 86443, 86464, 86469, 86480, 86499, 86508, 86521, 86539, 86542, 86543, 86549, 86555, 86582, 86611, 86618, 86630, 86632, 86635, 86659, 86667, 86678, 86703, 86713, 86715, 86725, 86731, 86742, 86771, 86806, 86811, 86824, 86842, 86851, 86886, 86888, 86914, 86918, 86985, 87006, 87012, 87013, 87028, 87030, 87086, 87110, 87138, 87162, 87173, 87197, 87219, 87244, 87245, 87301, 87317, 87344, 87354, 87359, 87392, 87398, 87401, 87406, 87429, 87454, 87465, 87470, 87501, 87502, 87506, 87528, 87562, 87565, 87574, 87579, 87584, 87627, 87634, 87670, 87686, 87695, 87709, 87727, 87804, 87828, 87829, 87834, 87843, 87855, 87863, 87882, 87887, 87923, 87967, 87986, 87987, 87990, 87996, 88007, 88010, 88017, 88050, 88072, 88083, 88086, 88091, 88104, 88128, 88129, 88130, 88144, 88200, 88211, 88283, 88290, 88296, 88300, 88336, 88337, 88341, 88352, 88367, 88378, 88391, 88399, 88422, 88429, 88435, 88463, 88495, 88515, 88517, 88544, 88546, 88558, 88565, 88603, 88635, 88654, 88656, 88673, 88690, 88744, 88762, 88766, 88784, 88803, 88824, 88838, 88842, 88873, 88876, 88884, 88888, 88901, 88986, 88987, 88989, 89012, 89016, 89031, 89050, 89057, 89063, 89123, 89152, 89160, 89179, 89182, 89211, 89213, 89219, 89257, 89274, 89288, 89291, 89303, 89325, 89346, 89356, 89391, 89408, 89413, 89475, 89480, 89482, 89503, 89516, 89538, 89561, 89562, 89574, 89587, 89600, 89617, 89619, 89621, 89622, 89639, 89642, 89665, 89666, 89684, 89691, 89696, 89701, 89704, 89712, 89752, 89759, 89768, 89784, 89848, 89895, 89908, 89931, 89997, 89999, 90004, 90012, 90035, 90069, 90072, 90091, 90096, 90101, 90111, 90146, 90189, 90242, 90247, 90258, 90273, 90277, 90345, 90354, 90387, 90410, 90418, 90423, 90437, 90438, 90463, 90466, 90509, 90532, 90574, 90625, 90629, 90636, 90655, 90670, 90715, 90765, 90766, 90771, 90780, 90788, 90792, 90804, 90809, 90824, 90865, 90871, 90950, 91001, 91015, 91020, 91037, 91046, 91050, 91053, 91068, 91074, 91087, 91104, 91109, 91147, 91195, 91229, 91291, 91310, 91311, 91338, 91345, 91363, 91431, 91466, 91496, 91508, 91530, 91531, 91543, 91546, 91557, 91563, 91600, 91630, 91659, 91676, 91689, 91712, 91783, 91838, 91841, 91853, 91871, 91880, 91892, 91893, 91925, 91935, 91938, 91953, 91965, 91967, 91974, 91993, 92002, 92007, 92011, 92027, 92056, 92061, 92068, 92075, 92121, 92180, 92189, 92217, 92244, 92278, 92285, 92306, 92311, 92335, 92362, 92382, 92383, 92386, 92390, 92423, 92428, 92502, 92517, 92533, 92535, 92564, 92649, 92686, 92700, 92707, 92733, 92734, 92818, 92824, 92840, 92874, 92876, 92892, 92921, 92937, 92954, 92997, 93001, 93007, 93021, 93046, 93048, 93068, 93079, 93082, 93097, 93104, 93133, 93142, 93144, 93172, 93177, 93182, 93207, 93210, 93223, 93225, 93317, 93327, 93351, 93359, 93395, 93418, 93435, 93472, 93485, 93498, 93515, 93528, 93529, 93531, 93534, 93583, 93615, 93633, 93641, 93655, 93742, 93755, 93761, 93766, 93779, 93785, 93812, 93868, 93975, 93987, 94047, 94064, 94075, 94082, 94090, 94102, 94111, 94121, 94123, 94150, 94170, 94172, 94195, 94197, 94205, 94215, 94223, 94247, 94323, 94360, 94361, 94394, 94396, 94411, 94439, 94442, 94453, 94487, 94506, 94516, 94521, 94538, 94542, 94561, 94569, 94589, 94613, 94616, 94639, 94641, 94646, 94687, 94698, 94718, 94751, 94754, 94762, 94766, 94821, 94834, 94851, 94877, 94901, 94964, 94965, 94990, 94995, 95074, 95075, 95085, 95088, 95161, 95225, 95249, 95272, 95288, 95296, 95350, 95361, 95374, 95417, 95451, 95500, 95502, 95516, 95539, 95540, 95559, 95582, 95599, 95610, 95681, 95706, 95707, 95744, 95986, 96053, 96301, 96311, 96702, 96822, 97161, 149837, 150324, 150922, 153934, 154122, 154812, 155029, 157478, 157526, 158740, 159029, 159034, 159282, 159405, 159608, 163036, 163401, 163557, 163719, 164238, 164927, 168889, 168977, 168998, 169042, 169103, 169143, 169553, 169614, 170286, 171973, 172188, 172800, 173001, 173976, 174548, 174759, 174818, 175110, 175267, 176001, 176376, 176991, 177036, 177156, 177251, 177884, 178687, 179463, 179586, 180110, 181422, 181796, 182186, 182254, 183044, 183631, 184290, 184883, 185451, 185672, 187375, 187525, 187767, 188163, 188410, 189200, 189402, 189852, 190632, 191075, 192074, 192398, 192456, 192709, 193138, 193260, 195406, 196457, 197978, 198075, 199662, 200271, 200709, 201143, 201539, 201825, 202082, 202596, 203436, 204205, 204292, 204595, 204890, 205449, 206840, 206991, 207029, 207046, 208207, 208312, 208489, 208632, 208643, 209218, 209370, 209963, 211173, 211178, 211614, 211627, 214679, 215020, 215495, 218048, 218951, 219069, 220390, 221111, 221224, 221535, 222017, 222064, 222214, 222957, 224024, 224535, 224994, 226001, 227512, 228701, 229248, 230279, 230937, 231194, 231442, 231523, 232827, 232888, 233738, 234511, 235673, 236927, 237084, 237114, 237766, 237895, 238341, 238448, 238549, 239890, 240265, 240976, 241326, 242044, 242269, 243004, 243343, 244315, 246280, 246696, 248044, 248046, 248058, 248059, 248068, 248069], "k": 16340, "tokens_counted": 43000495, "distinct": 16336, "coverage": 1.0, "held_out_coverage": 0.0}\', \'src/qwen/vllm_patch/vllm_last_state.py\': \'"""ARC_VLLM_RETAIN_LAST_STATE=1: under halt95\\\'s Merlin engine (h95: block 4096, --prefix-match-unit 64, MTP with its\\nblock drop, VLLM_E2_MAMBA_RETAIN_CHECKPOINT=1), a finished request leaves the newest usable Gated DeltaNet (GDN) state of\\nits decode for the next turn, so a free-agent round no longer re-prefills everything the previous round generated.\\nDesign: docs/design/last_state_retention_design.md (Tier 1).\\n\\nStock h95 keeps GDN states only at prompt boundaries, and --prefix-cache-retention-interval 4096 keeps one every 4096\\ntokens and lets each superseded one age in the LRU, where they evicted whole cached prefixes. Here, engine core only\\n(no worker, kernel or CUDA graph change), per request and GDN group (the PLE state rides in one of them):\\n- the prompt keeps the stock semantic states (retention 0: replay boundaries, the tail, junctions); past the prompt\\n  every block-boundary state (the one RecoverSSM writes at each 4096 crossing) is hashed;\\n- the request\\\'s prompt-tail state stays held and hashed until the request is freed, exactly as in stock, and is freed\\n  last: the decode state is only an additional, newer resume point, so a turn whose decode state or decode-tail block\\n  was evicted still resumes at the prompt tail (with the decode state as the only resume point, 34 of 77 dry-run\\n  turns fully missed under KV pressure on 10-17: the decode-tail block is the first of a request\\\'s blocks the LRU\\n  evicts);\\n- of the decode states the request holds the newest usable one: a state at X serves a hit only once the attention side\\n  holds a key at X + 64 (the MTP block drop), so a new state is held next to the previous one until the committed\\n  tokens pass it by RELEASE_UNITS hash units, then the previous one is released, losing its hash unless another\\n  request holds it too, so it goes back to the head of the free queue as without retention and never pushes cached\\n  prefixes out of the LRU;\\n- at free (finish or preemption), before the grace pin is taken: every full-attention group keys the block holding\\n  the newest usable decode state X (at or below T - 64, T = the committed tokens rounded down to 64) at X + 64 (no stock\\n  path keys a decode position; a key at T would miss whenever the next prompt re-renders the output and diverges from it\\n  below T), the held decode states other\\n  than the newest one at or below T - 64 lose their hash, and so do the table\\\'s decode states above T - 64 (their\\n  stage-0 commit may not have run under pipeline parallelism). States at or below the request\\\'s prefix-cache hit keep\\n  theirs: they were complete before it was admitted (a request preempted again before passing its resume point);\\n- a waiting next turn\\\'s pin also holds, in every attention group\\\'s own pool, the block with the key its hit rests on\\n  (the stock pin looks one hash unit past the hit, where the decode-tail key now lies; the extra pin covers the rest),\\n  and for a block-aligned hit, per GDN group, the next cached state below it (the previous turn\\\'s prompt tail);\\n- a request admitted on a hit past its own prompt (resumed after preemption at a decode state) also holds its\\n  prompt-tail state when cached, the fallback stock would have resumed it at.\\nThe next turn then resumes at the newest 4096 multiple at or below T - 64 (or the prompt tail when higher) instead of\\nthe prompt tail, and at no lower position than without the patch whatever subset of the cached blocks is evicted.\\nKV: a request past its first decode crossing holds one more state than stock (one block per GDN group), two for\\nRELEASE_UNITS x 64 of every 4096 decode tokens.\\n\\nDecode states are hashed when a step\\\'s output arrives (the async scheduler\\\'s cache_blocks, h95\\\'s default), before the\\nnext step retires them; the synchronous scheduler hashes only at allocation, after retiring, so there a state crossed\\nby a multi-token verify step is lost and the next turn falls back to an older one.\\n\\nBetween a request\\\'s free and its next turn\\\'s admission, its chain (the attention blocks up to the retained decode\\nstate, the decode-tail block, the GDN states) is held by stock\\\'s grace pin, but under KV pressure grace pins are\\nyielded first, oldest first, and a yielded chain is evicted tail first, so the decode-only blocks go first and the\\nnext turn falls back to the prompt tail. With ARC_VLLM_RETAIN_LAST_STATE_PROTECT_SECONDS=S (default 0: off), a\\nfinished request that leaves a usable decode state is protected for S seconds or until its next turn is admitted: its\\ngrace pin yields only after every other grace pin and every waiting turn\\\'s pin, just before a preemption would (so it\\nnever causes one, and the head-of-line admission still counts it as yieldable). It holds no block a stock grace does\\nnot, but it changes which pin goes first, so another turn can lose more than without the patch (one block of\\npressure: the protected turn resumes at 12,288 instead of 4,928, while a short turn whose grace now goes first misses\\nfully instead of resuming at 5,888; test/qwen/test_last_state.py), hence off by default.\\n\\nARC_VLLM_RETAIN_LAST_STATE_LOG=1 logs one "ARC_LS <event> key=value ..." line per event to the engine log: publish (at\\na request\\\'s free: the retained decode state, its blocks, the decode-tail blocks, whether it is protected), unhash (a\\ndecode state losing its hash through this patch), evict (a published retained state, prompt-tail state or\\ndecode-tail block evicted by the pool), yield (the grace / waiter pins a yield released, protected ones marked) and\\nlookup (at each admission: the hit, the contiguous attention hit per group, the newest retained decode state\\npublished for that prefix and whether its GDN states are still cached).\\n\\n--prefix-cache-retention-interval is ignored for the GDN states (leave RETENTION unset). Composes with\\nARC_VLLM_QSA_HOST_KV (vllm_qsa_host_kv.py): the decode-tail key goes into each attention group\\\'s own block pool, the host\\nK/V group\\\'s included, and the GDN rules hold per group whatever the grouping. Patched modules: see patches().\\n"""\\nimport logging\\nimport os\\nimport time\\nfrom collections import OrderedDict\\n\\nlogger = logging.getLogger("vllm.arc_last_state")\\n\\nRELEASE_UNITS = 2  # hash units past the newest decode state before the previous hold is released\\nPROTECT_SECONDS = float(os.environ.get("ARC_VLLM_RETAIN_LAST_STATE_PROTECT_SECONDS", "0"))\\nLOG = os.environ.get("ARC_VLLM_RETAIN_LAST_STATE_LOG") == "1"\\n_MARKS: dict = {}  # LOG: (id(pool), block id) -> what a published block holds, for its eviction line\\n_RETAINED: OrderedDict = OrderedDict()  # LOG: prefix hash at a published decode state -> (request id, position)\\n\\n\\ndef log(event: str, **fields) -> None:\\n    logger.info("ARC_LS %s %s", event, " ".join(f"{key}={value}" for key, value in fields.items()))\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Rules (no vLLM needed; test/qwen/test_last_state.py)\\n\\n\\ndef committed_tokens(num_computed: int, num_in_flight: int, num_tokens: int) -> int:\\n    """Tokens whose KV and GDN commits have run on every stage: computed minus in flight, and never the last sampled\\n    token (it is never fed)."""\\n    return max(0, min(num_computed - num_in_flight, num_tokens - 1))\\n\\n\\ndef tail_key(committed: int, num_prompt_tokens: int, unit: int, block: int) -> int:\\n    """Decode-tail attention key a freed request registers (0: none): one unit past the newest decode state a hit can\\n    use (a state at X serves a hit only with a key at X + unit, the MTP block drop), when that state lies past the\\n    prompt. Not at the committed end: the next turn\\\'s prompt re-renders this output (stripped tool-call whitespace,\\n    re-tokenized text) and often matches it only up to a point below the end, where a key at the end cannot match."""\\n    state = (committed // unit * unit - unit) // block * block\\n    return state + unit if state > num_prompt_tokens else 0\\n\\n\\ndef decode_block_ends(num_prompt_tokens: int, num_tokens: int, block: int) -> range:\\n    """Ends of the state blocks that end past the prompt and are full at num_tokens."""\\n    return range((num_prompt_tokens // block + 1) * block, num_tokens + 1, block)\\n\\n\\ndef states_to_unhash(positions: list[int], floor: int, reach: int, unit: int) -> list[int]:\\n    """Held state positions that lose their hash at free: every state above floor (the prompt, or the request\\\'s\\n    prefix-cache hit when higher) but the newest one a hit can use (at or below reach - unit, reach being T, the\\n    committed tokens rounded down to the unit)."""\\n    decode = [p for p in positions if p > floor]\\n    keep = max((p for p in decode if p <= reach - unit), default=None)\\n    return [p for p in decode if p != keep]\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Engine core\\n\\n\\ndef unhash(pool, blocks, floor: int, shared_too: bool = False, request_id=None) -> None:\\n    """Drop the hashes of the states keyed above floor among blocks; shared ones (other refs) only if shared_too."""\\n    for block in blocks:\\n        if (block.block_hash is not None and block.block_hash_num_tokens > floor\\n                and (shared_too or block.ref_cnt == 1)):\\n            if LOG:\\n                mark = _MARKS.pop((id(pool), block.block_id), None)\\n                log("unhash", req=request_id, pos=block.block_hash_num_tokens, block=block.block_id,\\n                    refs=block.ref_cnt, published=bool(mark))\\n            pool._maybe_evict_cached_block(block)\\n\\n\\ndef patch_single_type_manager(module) -> None:\\n    cls = module.MambaManager\\n    init, cache, retire = cls.__init__, cls.cache_blocks, cls._retire_state_blocks\\n    skipped, pop_retained, pop_blocks = cls.remove_skipped_blocks, cls.pop_retained_checkpoints, cls.pop_blocks_for_free\\n    add_hit = cls.add_local_computed_blocks\\n\\n    def __init__(self, *args, **kwargs):\\n        init(self, *args, **kwargs)\\n        if self.mamba_cache_mode == "align" and self.enable_caching and not self._retain_checkpoint:\\n            raise ValueError("ARC_VLLM_RETAIN_LAST_STATE needs VLLM_E2_MAMBA_RETAIN_CHECKPOINT=1")\\n        self._arc_last_state = self._retain_checkpoint  # align mode with prefix caching\\n        self._arc_prompt: dict[str, int] = {}  # request -> num_prompt_tokens\\n        self._arc_previous: dict[str, tuple] = {}  # request -> (position, block) held until the newest is usable\\n        self._arc_reach: dict[str, int] = {}  # request -> T (committed tokens rounded down to 64), set just before its free\\n        self._arc_hit: dict[str, int] = {}  # request -> prefix-cache hit at admission\\n        self._arc_prompt_state: dict[str, tuple] = {}  # request -> (position, block): its prompt state, held until free\\n\\n    def release(self, request_id: str, blocks) -> None:\\n        unhash(self.block_pool, blocks, self._arc_prompt[request_id], request_id=request_id)\\n        self.block_pool.free_blocks(blocks)\\n\\n    def release_previous(self, request_id: str) -> None:\\n        """Release the decode state held until the newest one is usable; it loses its hash unless another request holds\\n        it."""\\n        _, block = self._arc_previous.pop(request_id)\\n        unhash(self.block_pool, [block], 0, request_id=request_id)\\n        self.block_pool.free_blocks([block])\\n\\n    def add_local_computed_blocks(self, request_id, new_computed_blocks, num_local_computed_tokens, *args, **kwargs):\\n        add_hit(self, request_id, new_computed_blocks, num_local_computed_tokens, *args, **kwargs)\\n        if self._arc_last_state:\\n            self._arc_hit[request_id] = num_local_computed_tokens\\n\\n    def cache_blocks(self, request, num_tokens, retention_interval=None, *, replay_boundaries):\\n        if not self._arc_last_state:\\n            return cache(self, request, num_tokens, retention_interval, replay_boundaries=replay_boundaries)\\n        if request.request_id not in self._arc_prompt and self._arc_hit.get(request.request_id, 0) > request.num_prompt_tokens:\\n            hold_prompt_tail(self, request)\\n        prompt = self._arc_prompt[request.request_id] = request.num_prompt_tokens\\n        # The mask keeps every listed boundary\\\'s state: the prompt\\\'s own plus every block end past the prompt\\n        ends = decode_block_ends(prompt, num_tokens, self.block_size)\\n        return cache(self, request, num_tokens, 0, replay_boundaries=(*replay_boundaries, *ends))\\n\\n    def hold_prompt_tail(self, request) -> None:\\n        """A request resumed at a decode state: hold its prompt-tail state (where stock resumes it) as the fallback."""\\n        unit = self.block_pool.hash_block_size\\n        tail = request.num_prompt_tokens // unit * unit - unit\\n        cached = tail > 0 and self.block_pool.get_cached_block(request.block_hashes[tail // unit - 1], [self.kv_cache_group_id])\\n        if cached:\\n            self.block_pool.touch(cached)\\n            self._arc_prompt_state[request.request_id] = (tail, cached[0])\\n\\n    def _retire_state_blocks(self, request_id, retired):\\n        if not self._arc_last_state or request_id not in self._arc_prompt:\\n            return retire(self, request_id, retired)\\n        newest = max(((b.block_hash_num_tokens, b) for _, b in retired if b.block_hash is not None),\\n                     key=lambda entry: entry[0], default=None)\\n        held = self._retained_checkpoints.get(request_id)\\n        if newest is None or (held is not None and newest[0] <= held[0]):\\n            return release(self, request_id, [b for _, b in retired])\\n        freed = [b for _, b in retired if b is not newest[1]]\\n        self._retained_checkpoints[request_id] = newest  # the table\\\'s ref becomes the hold\\\'s\\n        prompt = self._arc_prompt[request_id]\\n        if held is not None:\\n            if newest[0] <= prompt or (held[0] <= prompt and request_id in self._arc_prompt_state):\\n                freed.append(held[1])  # superseded inside the prompt, as in stock\\n            elif held[0] <= prompt:  # the first decode state: the prompt state stays held until free, as in stock\\n                self._arc_prompt_state[request_id] = held\\n            else:  # a decode state: unusable until the attention key passes it\\n                if request_id in self._arc_previous:\\n                    release_previous(self, request_id)\\n                self._arc_previous[request_id] = held\\n        release(self, request_id, freed)\\n\\n    def remove_skipped_blocks(self, request_id, processed_computed_tokens, num_prompt_tokens=None):\\n        skipped(self, request_id, processed_computed_tokens, num_prompt_tokens)\\n        previous = self._arc_previous.get(request_id) if self._arc_last_state else None\\n        held = self._retained_checkpoints.get(request_id)\\n        unit = self.block_pool.hash_block_size\\n        if previous is not None and (held is None or held[0] + RELEASE_UNITS * unit <= processed_computed_tokens):\\n            release_previous(self, request_id)\\n\\n    def pop_retained_checkpoints(self, request_id):\\n        held = self._retained_checkpoints.get(request_id) if self.mamba_cache_mode == "align" else None\\n        blocks = pop_retained(self, request_id)\\n        if not self._arc_last_state:\\n            return blocks\\n        previous = self._arc_previous.pop(request_id, None)\\n        fallback = self._arc_prompt_state.pop(request_id, None)\\n        if previous is None and held is None and fallback is None:\\n            return blocks\\n        holds = [h for h in (previous, held) if h is not None]\\n        floor = max(self._arc_prompt.get(request_id, 0), self._arc_hit.get(request_id, 0))\\n        drop = states_to_unhash([p for p, _ in holds], floor, self._arc_reach.get(request_id, 0),\\n                                self.block_pool.hash_block_size)\\n        unhash(self.block_pool, [b for p, b in holds if p in drop], floor, shared_too=True, request_id=request_id)\\n        # Freed in this order, so evicted in this order: the prompt state last, where stock\\\'s single hold is\\n        return [b for _, b in holds] + ([fallback[1]] if fallback else [])\\n\\n    def pop_blocks_for_free(self, request_id):\\n        blocks = pop_blocks(self, request_id)\\n        prompt, reach = self._arc_prompt.pop(request_id, None), self._arc_reach.pop(request_id, 0)\\n        hit = self._arc_hit.pop(request_id, 0)\\n        if prompt is not None:\\n            unit = self.block_pool.hash_block_size\\n            unhash(self.block_pool, [b for b in blocks if b.block_hash is not None\\n                                     and b.block_hash_num_tokens > reach - unit], max(prompt, hit), shared_too=True,\\n                   request_id=request_id)\\n        return blocks\\n\\n    cls.__init__, cls.cache_blocks, cls._retire_state_blocks = __init__, cache_blocks, _retire_state_blocks\\n    cls.remove_skipped_blocks, cls.pop_retained_checkpoints = remove_skipped_blocks, pop_retained_checkpoints\\n    cls.pop_blocks_for_free, cls.add_local_computed_blocks = pop_blocks_for_free, add_local_computed_blocks\\n\\n\\ndef patch_kv_cache_manager(module) -> None:\\n    from vllm.v1.core.kv_cache_utils import make_block_hash_with_group_id\\n    from vllm.v1.core.single_type_kv_cache_manager import FullAttentionManager, MambaManager\\n\\n    cls = module.KVCacheManager\\n    init, free, pop, pin = cls.__init__, cls.free, cls.pop_blocks_for_free, cls.pin_waiting_hit\\n    allocate, yield_pins = cls.allocate_slots, cls.yield_pinned_hits\\n\\n    def __init__(self, *args, **kwargs):\\n        init(self, *args, **kwargs)\\n        self._arc_last_state = self.enable_caching\\n        self._arc_protected: dict[str, float] = {}  # finished request -> deadline of its grace pin\\\'s protection\\n        if self._arc_last_state:\\n            logger.info("ARC_VLLM_RETAIN_LAST_STATE: each request keeps its newest usable decode GDN state; protection "\\n                        "%.0f s, event log %s", PROTECT_SECONDS, "on" if LOG else "off")\\n\\n    def gdn_managers(self) -> list:\\n        return [m for m in self.coordinator.single_type_managers if isinstance(m, MambaManager) and m._arc_last_state]\\n\\n    def retained(self, request_id: str) -> list:\\n        """Per GDN group, the (position, block) of the decode state the request\\\'s free keeps (None if none): the\\n        newest held one past the prompt (and past its prefix hit) at or below the decode-tail key minus one unit."""\\n        out = []\\n        for manager in gdn_managers(self):\\n            floor = max(manager._arc_prompt.get(request_id, 0), manager._arc_hit.get(request_id, 0))\\n            reach = manager._arc_reach.get(request_id, 0) - manager.block_pool.hash_block_size\\n            holds = [h for h in (manager._retained_checkpoints.get(request_id), manager._arc_previous.get(request_id))\\n                     if h is not None and floor < h[0] <= reach]\\n            out.append(max(holds, key=lambda h: h[0], default=None))\\n        return out\\n\\n    def attention_managers(self) -> list:\\n        return [m for m in self.coordinator.single_type_managers if isinstance(m, FullAttentionManager)\\n                and m.enable_caching and m.kv_cache_spec.prefix_cacheable]\\n\\n    def publish_tail(self, request) -> None:\\n        """Before a request\\\'s blocks are popped: the decode-tail attention keys, and the reach for the GDN managers."""\\n        unit = self.block_pool.hash_block_size\\n        committed = committed_tokens(request.num_computed_tokens, request.num_in_flight_tokens, request.num_tokens)\\n        for manager in self.coordinator.single_type_managers:\\n            if isinstance(manager, MambaManager) and manager._arc_last_state:\\n                manager._arc_reach[request.request_id] = committed // unit * unit\\n        for manager in attention_managers(self):\\n            key = tail_key(committed, request.num_prompt_tokens, unit, manager.block_size)\\n            blocks = manager.req_to_blocks.get(request.request_id, ())\\n            if not key or key // unit > len(request.block_hashes) or key // manager.block_size >= len(blocks):\\n                continue\\n            block = blocks[key // manager.block_size]\\n            if not block.is_null:  # an extra key: the block keeps its prompt-tail key, its K/V is append-only\\n                manager.block_pool._insert_block_hash(\\n                    make_block_hash_with_group_id(request.block_hashes[key // unit - 1], manager.kv_cache_group_id),\\n                    block, num_tokens=key)\\n                if LOG:\\n                    _MARKS[(id(manager.block_pool), block.block_id)] = dict(\\n                        req=request.request_id, kind="tail_key", pos=key, group=manager.kv_cache_group_id)\\n\\n    def after_free(self, request, kept: list) -> None:\\n        """Protect the grace pin of a request that left a usable decode state in every GDN group; log the publish."""\\n        request_id = request.request_id\\n        protect = bool(kept) and all(kept) and len({p for p, _ in kept}) == 1 and request_id in self._grace_pins\\n        if protect and PROTECT_SECONDS > 0:\\n            self._arc_protected[request_id] = time.monotonic() + PROTECT_SECONDS\\n        if not LOG:\\n            return\\n        unit = self.block_pool.hash_block_size\\n        for manager, entry in zip(gdn_managers(self), kept):\\n            if entry is not None:\\n                _MARKS[(id(manager.block_pool), entry[1].block_id)] = dict(\\n                    req=request_id, kind="retained", pos=entry[0], group=manager.kv_cache_group_id)\\n        if kept and all(kept):\\n            _RETAINED[request.block_hashes[kept[0][0] // unit - 1]] = (request_id, kept[0][0])\\n            while len(_RETAINED) > 1024:\\n                _RETAINED.popitem(last=False)\\n        committed = committed_tokens(request.num_computed_tokens, request.num_in_flight_tokens, request.num_tokens)\\n        log("publish", req=request_id, prompt=request.num_prompt_tokens, committed=committed,\\n            retained=kept[0][0] if kept and all(kept) else None,\\n            gdn_blocks=[e[1].block_id if e else None for e in kept],\\n            tail_blocks=sorted(k[1] for k, v in _MARKS.items() if v["req"] == request_id and v["kind"] == "tail_key"),\\n            grace=request_id in self._grace_pins, protected=request_id in self._arc_protected)\\n\\n    def yield_pinned_hits(self, keep=None, limit=None):\\n        """Stock\\\'s order (grace pins oldest first, then waiters\\\' pins newest first) without the protected grace pins,\\n        which go only once nothing else is left (all of them when limit is None)."""\\n        if not self._arc_last_state:\\n            return yield_pins(self, keep, limit)\\n        now = time.monotonic()\\n        for request_id in [r for r, deadline in self._arc_protected.items() if deadline < now or r not in self._grace_pins]:\\n            del self._arc_protected[request_id]\\n        graces, waiters = set(self._grace_pins), len(self._pinned_hits)\\n        hidden = {r: self._grace_pins.pop(r) for r in list(self._grace_pins) if r in self._arc_protected}\\n        try:\\n            released = yield_pins(self, keep, limit)\\n        finally:\\n            self._grace_pins.update(hidden)\\n        protected = []\\n        if not released or limit is None:\\n            for request_id in list(self._arc_protected)[:limit]:  # oldest first\\n                released += self.release_grace_pin(request_id)\\n                del self._arc_protected[request_id]\\n                protected.append(request_id)\\n        if LOG and released:\\n            log("yield", graces=sorted(graces - set(self._grace_pins) - set(protected)), protected=protected,\\n                waiters=waiters - len(self._pinned_hits), blocks=released, limit=limit)\\n        return released\\n\\n    def allocate_slots(self, request, *args, **kwargs):\\n        new = LOG and self._arc_last_state and request.num_computed_tokens == 0 and not any(\\n            request.request_id in m.num_cached_block for m in self.coordinator.single_type_managers)\\n        blocks = allocate(self, request, *args, **kwargs)\\n        if new and blocks is not None:\\n            log_lookup(self, request, args[1] if len(args) > 1 else kwargs.get("num_new_computed_tokens", 0))\\n        return blocks\\n\\n    def log_lookup(self, request, hit: int) -> None:\\n        """An admitted request\\\'s hit, its contiguous attention hit per group (full blocks from the left, then the\\n        highest key inside the next block), and the newest retained decode state published for its prefix."""\\n        unit, hashes = self.block_pool.hash_block_size, request.block_hashes\\n        attention = {}\\n        for manager in attention_managers(self):\\n            cached = lambda k: manager.block_pool.get_cached_block(hashes[k - 1], [manager.kv_cache_group_id])  # noqa: E731\\n            per_block, units = manager.block_size // unit, 0\\n            while units + per_block <= len(hashes) and cached(units + per_block):\\n                units += per_block\\n            units = next((k for k in range(min(units + per_block - 1, len(hashes)), units, -1) if cached(k)), units)\\n            attention[manager.kv_cache_group_id] = units * unit\\n        found = max(((p, _RETAINED[hashes[p // unit - 1]]) for p in range(4096, len(hashes) * unit + 1, 4096)\\n                     if hashes[p // unit - 1] in _RETAINED), key=lambda e: e[0], default=None)\\n        alive = found is not None and all(m.block_pool.get_cached_block(hashes[found[0] // unit - 1], [m.kv_cache_group_id])\\n                                          for m in gdn_managers(self))\\n        log("lookup", req=request.request_id, prompt=request.num_prompt_tokens, hit=hit,\\n            attention_min=min(attention.values(), default=None), attention_max=max(attention.values(), default=None),\\n            short_groups=sorted(g for g, v in attention.items() if v < max(attention.values())),\\n            retained=found[0] if found else None, retained_from=found[1][0] if found else None,\\n            retained_cached=alive)\\n\\n    def pin_waiting_hit(self, request) -> int:\\n        """The stock pin, plus per attention group the block holding the highest key in (hit, hit + block]: a hit at\\n        X needs a key at X + 64 or above (the MTP block drop). The stock pin looks only at X + 64 and only in the GPU\\n        pool; a decode-state hit (X a block end) rests on T, in block X / block outside the hit, which otherwise only\\n        the previous turn\\\'s grace pin holds, and grace pins yield first. For a block-aligned hit also, per GDN group,\\n        the next cached state below it (the previous turn\\\'s prompt tail, the hit without the patch), released last."""\\n        pinned = pin(self, request)\\n        if not (pinned and self._arc_last_state):\\n            return pinned\\n        _, hit, _ = self.get_computed_blocks(request)  # the stock pin\\\'s lookup again, once per waiter\\n        unit, held = self.block_pool.hash_block_size, self._pinned_hits[request.request_id]\\n        for manager in attention_managers(self):\\n            for key in range(min(hit + manager.block_size, len(request.block_hashes) * unit), hit, -unit):\\n                cached = manager.block_pool.get_cached_block(request.block_hashes[key // unit - 1],\\n                                                             [manager.kv_cache_group_id])\\n                if cached:\\n                    if all(block is not cached[0] for block in held):\\n                        manager.block_pool.touch(cached)  # its own pool: the host K/V group\\\'s is private\\n                        held.append(cached[0])\\n                        pinned += 1\\n                    break\\n        for manager in self.coordinator.single_type_managers:\\n            if not (isinstance(manager, MambaManager) and manager._arc_last_state) or hit % manager.block_size:\\n                continue\\n            for key in range(hit - unit, 0, -unit):  # the attention blocks below the hit are pinned already\\n                cached = manager.block_pool.get_cached_block(request.block_hashes[key // unit - 1],\\n                                                             [manager.kv_cache_group_id])\\n                if cached:\\n                    if all(block is not cached[0] for block in held):\\n                        manager.block_pool.touch(cached)\\n                        held.insert(0, cached[0])  # release_pinned_hit frees in reverse: this one last\\n                        pinned += 1\\n                    break\\n        return pinned\\n\\n    def free_with_tail(self, request):\\n        if not self._arc_last_state:\\n            return free(self, request)\\n        publish_tail(self, request)\\n        kept = retained(self, request.request_id)\\n        free(self, request)\\n        after_free(self, request, kept)\\n\\n    def pop_with_tail(self, request):\\n        if not self._arc_last_state:\\n            return pop(self, request)\\n        publish_tail(self, request)\\n        kept = retained(self, request.request_id)\\n        blocks = pop(self, request)\\n        after_free(self, request, kept)\\n        return blocks\\n\\n    cls.__init__, cls.free, cls.pop_blocks_for_free = __init__, free_with_tail, pop_with_tail\\n    cls.pin_waiting_hit, cls.yield_pinned_hits, cls.allocate_slots = pin_waiting_hit, yield_pinned_hits, allocate_slots\\n\\n\\ndef patch_block_pool(module) -> None:\\n    """LOG only: an eviction line when the pool evicts a block this patch published (retained or decode-tail)."""\\n    cls = module.BlockPool\\n    evict = cls._maybe_evict_cached_block\\n\\n    def _maybe_evict_cached_block(self, block):\\n        mark = _MARKS.pop((id(self), block.block_id), None) if _MARKS else None\\n        evicted = evict(self, block)\\n        if mark and evicted:\\n            log("evict", block=block.block_id, **mark)\\n        return evicted\\n\\n    cls._maybe_evict_cached_block = _maybe_evict_cached_block\\n\\n\\nBOUNDARY_ORDER = "for boundary in (False, True) if align else (False,):"\\n\\n\\ndef patch_recoverssm_commit(module) -> None:\\n    """Merlin\\\'s GDNRecoverSSMCommitContext.commit with its two align-mode commit launches in the other order, boundary\\n    states first. Under the V2 runner the verify source is the window\\\'s last block, which is the final state\\\'s block\\n    when a verify step crosses a block end X; launched first, the final fold overwrote it in place and the boundary fold\\n    then re-applied the step\\\'s records for [n_c, X) to that advanced state, so the decode-time state stored at X was\\n    wrong whenever the step accepted tokens past X (most crossings at MTP-3). Boundary first is exact there: its target\\n    (the block ending at X) is never the source, which the final fold then reads unchanged. Only gdn-state-cache reads\\n    those states (a next turn resuming at X); the running request never does. Raises if the method no longer has the\\n    launch loop this rewrites."""\\n    import inspect\\n    import textwrap\\n\\n    cls = module.GDNRecoverSSMCommitContext\\n    source = textwrap.dedent(inspect.getsource(cls.commit))\\n    if source.count(BOUNDARY_ORDER) != 1:\\n        raise RuntimeError("GDNRecoverSSMCommitContext.commit has changed; the boundary-first launch order cannot be applied")\\n    namespace: dict = {}\\n    exec(compile(source.replace(BOUNDARY_ORDER, "for boundary in (True, False) if align else (False,):"),\\n                 inspect.getsourcefile(cls.commit), "exec"), module.__dict__, namespace)\\n    cls.commit = namespace["commit"]\\n    logger.info("ARC_VLLM_RETAIN_LAST_STATE: GDN RecoverSSM commit launches boundary states first")\\n\\n\\ndef patches() -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook (chained after hooks already registered there)."""\\n    return {\\n        "vllm.v1.core.single_type_kv_cache_manager": patch_single_type_manager,\\n        "vllm.v1.core.kv_cache_manager": patch_kv_cache_manager,\\n        "vllm.model_executor.layers.mamba.gdn.recoverssm_gdn": patch_recoverssm_commit,\\n        **({"vllm.v1.core.block_pool": patch_block_pool} if LOG else {}),\\n    }\\n\', \'src/qwen/vllm_patch/vllm_qsa_host_kv.py\': \'"""ARC_VLLM_QSA_HOST_KV=1: Flash-Next\\\'s QSA K/V in host memory under the halt95 Merlin engine (vLLM 0.30.0 + its patch\\nseries, h95), the vLLM counterpart of ARC_SGLANG_QSA_HISPARSE (qsa_hisparse.py). Design and sizing:\\ndocs/design/hisparse_h95_design.md.\\n\\nThe QSA (sparse full-attention) K/V layers of the target and of the MTP drafter leave the GPU pool: they form one extra\\nKV cache group, host_resident with a private block pool (KVCacheConfig.hisparse_host_num_blocks host blocks), whose pages\\nlive in one cudaHostRegister\\\'ed buffer per worker. The QSA compressed index, the key ring, the GDN states and the PLE\\nstate stay in the GPU pool, which is sized after the K/V left it, so the pool stride shrinks to the index page and many\\nmore tokens fit. Per worker, for each QSA layer:\\n- writes (reshape_and_cache_flash) go to host memory through a UVA view, the layer\\\'s bound kv_cache;\\n- decode, verify and draft rows (uniform query length Q) resolve their selected positions against a per-request GPU hot\\n  buffer of H rows with vLLM\\\'s own HiSparse kernels (torch.ops._C_cache_ops.hisparse_resolve_residency: LRU hits and\\n  misses, then hisparse_gather_compact copies the misses from host memory), one call per row step j of every request,\\n  and the unchanged QSA attention kernel reads the hot buffer through an identity block table. H >= Q x the row width\\n  (2,051 entries), so a later row step never evicts what an earlier one of the same forward resolved;\\n- prefill rows stage their requests\\\' context blocks from host memory into a reserved GPU slab (one layer at a time)\\n  and run the same kernel through a remapped block table.\\nAttention gets the same FP8 bytes in the same column order; launches over part of a batch\\\'s rows keep the launch\\nconfiguration of the whole batch, so outputs are bit-identical to the GPU-resident path. Hot entries are keyed by host\\nrow; at the start of every forward, rows this forward writes are dropped from every request\\\'s hot buffer\\n(hisparse_invalidate_written_slots), and a request slot\\\'s hot buffer is reset when a request takes the slot.\\n\\nGPU memory fixed by the patch (hot K/V and maps for --max-num-seqs request slots, row buffers, staging slab), one\\nallocation per worker, is taken from --kv-cache-memory-bytes before the pool is sized, as anything allocated later comes\\nout of the CUDA-graph headroom. Same-size Mamba (GDN) groups whose layers sit on different pipeline stages share one\\ngroup when no worker\\\'s block stride grows (at --block-size 4096 a stage-0 and a stage-1 GDN layer pair up: 19 groups\\ninstead of 36). Engine warmup\\\'s made-up block ids of the host group are folded into the host pool, and those of the\\nslot-packed QSA ring group (128 requests per 2 MiB pool block) are moved to pool blocks no other warmup group holds.\\nA waiting request\\\'s prefix-cache pin finds the host group\\\'s MTP proof block in the host pool.\\n\\nEnvironment:\\n- ARC_VLLM_QSA_HOST_KV_GB (default 10): host K/V budget per GPU in GB (1e9 B); host blocks = the smallest\\n  budget // (QSA layers x page bytes) over the workers.\\n- ARC_VLLM_GDN_PACK=1: Mamba layers whose state fits half a pool block (the bf16 GDN state) take half-block pages, so\\n  the Mamba group merge puts two of one worker\\\'s layers in each block: with --mamba-ssm-cache-dtype bfloat16, 19 GDN\\n  groups become 10 and a running request holds 2 blocks per Mamba group fewer. No effect on fp32 states.\\n  ARC_VLLM_GDN_PACK=tight: the same pairs, with every Mamba page the bf16 GDN state rounded up to 256 B instead of half\\n  the platform\\\'s 2 MiB page, so the pool block shrinks to two such pages (1,893,376 B) and the same KV bytes hold ~10.8%\\n  more blocks (tight_mamba_groups).\\n- ARC_VLLM_QSA_STAGING_BLOCKS (default 0 = two max_model_len contexts): GPU staging slab, in blocks.\\n- ARC_VLLM_QSA_STAGE_STREAM=1: prefill staging overlapped with compute. While one QSA layer attends, a side stream\\n  copies the same blocks of the next QSA layer of the worker into the other half of the staging slab; that layer then\\n  waits for the copy and copies again only the blocks its own forward wrote after the copy read them. A forward whose\\n  prefill blocks need more than half the slab (two long prefills at once) stages in place as without the switch. Same\\n  bytes in the slab, the same kernel, so the attention output is unchanged.\\n- ARC_VLLM_QSA_HOST_KV_FUSE=1: one hisparse_gather_compact per layer after all row steps instead of one per row step\\n  (each row step\\\'s swap plan goes to its own rows of the plan buffers). Same bytes in the same hot rows: a row step\\n  only reads the hot buffer\\\'s maps, never its bytes, and with H >= Q x the row width no two row steps of a forward copy\\n  into the same hot row.\\n- ARC_VLLM_QSA_HOT_STEPS=k (default 0: all Q row steps): H holds k row steps (k x the row width) instead of Q, and a\\n  decode forward resolves and attends k row steps at a time (rows j, j + Q, ... of those steps, through a contiguous copy,\\n  with the whole batch\\\'s launch configuration), so no row step evicts what a not yet attended one resolved. Same bytes,\\n  the same kernel and launch configuration, so the attention output is unchanged; the fixed GPU buffers shrink by about\\n  (Q - k) / Q of the hot K/V, which the pool gets back. More launches per layer, and fewer hot rows kept between forwards.\\nARC_VLLM_QSA_HOT_TOKENS=H (default 0: the smallest, steps x row width rounded up to 16): a larger hot buffer, a multiple\\nof 16 and at least that. At a given H the per-step path touches the same entries in the same order as one launch over\\nall row steps, so its hit rate is the one-launch path\\\'s at that H: with ARC_VLLM_QSA_HOT_STEPS=1, H = 1.5 x row width\\nkeeps more of the previous forwards\\\' rows than one row\\\'s worth while each step still attends before the next resolve\\n(the output is unchanged for any H). ARC_VLLM_QSA_HOT_OVERLAP=1 (with HOT_STEPS, H >= 2 x steps x row width): the next\\ngroup of row steps is resolved and swapped in on a side stream while the current group attends, hiding the host reads\\nbehind the attention. The current group\\\'s entries were touched last, so the least-recently-used evictions of the next\\ngroup\\\'s misses (at most steps x row width per request) never reach them while H holds both groups; each group writes\\nonly its own rows of the hot-index map. Same bytes in the same hot rows, so the output is unchanged. Constant: STATS_EVERY (1024: log the hot-buffer hit rate every this many forwards\\nof the target; 0: off).\\n\\nNeeds qsa_hisparse.py beside it (its register_host_memory pins the host buffer) and VLLM_USE_V2_MODEL_RUNNER=1. Not\\nsupported: KV connectors, native HiSparse (--hisparse-config), DCP, the V1 model runner; DP attention with more than one\\nKV head per GPU is untested. Patched modules: see patches().\\n"""\\nimport logging\\nimport os\\nimport time\\n\\nlogger = logging.getLogger("vllm.arc_qsa_host_kv")\\n\\nROLE = "arc_qsa_host_kv"  # KVCacheGroupSpec.role of the host K/V group, mapped to QSAHostKVManager\\nSMEM_LIMIT = 99 * 1024  # dynamic shared memory per block on sm89; the swap-in kernel\\\'s grows with H\\nALIGN = 256  # byte alignment of every buffer carved from a worker\\\'s one fixed GPU allocation\\nSTATS_EVERY = 1024  # forwards of the target between hot-buffer hit-rate lines (0: none)\\n\\n\\ndef host_budget_bytes() -> int:\\n    return int(float(os.environ.get("ARC_VLLM_QSA_HOST_KV_GB", "10")) * 1e9)\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Sizing and bookkeeping rules (no vLLM or torch needed; test/qwen/test_qsa_host_kv.py)\\n\\n\\ndef resolve_smem_bytes(hot: int, width: int) -> int:\\n    """Dynamic shared memory of hisparse_resolve_residency for H hot rows and `width` selected entries per row."""\\n    chunks = -(-hot // 32)\\n    return 4 * (width + 2 * (chunks + 1) + 2 * width + 3) + 2 * (hot + 2 * width)\\n\\n\\ndef hot_steps(q: int) -> int:\\n    """Row steps resolved before each decode attention launch (ARC_VLLM_QSA_HOT_STEPS; 0 or unset: all q, one launch)."""\\n    steps = int(os.environ.get("ARC_VLLM_QSA_HOT_STEPS", "0"))\\n    if steps < 0:\\n        raise ValueError(f"ARC_VLLM_QSA_HOT_STEPS={steps} is not a row step count")\\n    return min(steps, q) if steps else q\\n\\n\\ndef hot_tokens() -> int:\\n    """ARC_VLLM_QSA_HOT_TOKENS: H, or 0 for the smallest one (hot_rows)."""\\n    value = int(os.environ.get("ARC_VLLM_QSA_HOT_TOKENS", "0") or 0)\\n    if value < 0 or value % 16:\\n        raise ValueError(f"ARC_VLLM_QSA_HOT_TOKENS={value} is not a non-negative multiple of 16")\\n    return value\\n\\n\\ndef hot_rows(rows_per_request: int, width: int, override: int = 0) -> int:\\n    """H: at least one forward\\\'s rows of a request (Q row steps of `width` entries each), a multiple of 16."""\\n    need = rows_per_request * width\\n    hot = override or -(-need // 16) * 16\\n    if hot % 16:\\n        raise ValueError(f"H={hot} is not a multiple of 16")\\n    if hot < need:\\n        raise ValueError(f"H={hot} is below {rows_per_request} rows x {width} entries = {need}")\\n    if hot > 32767:\\n        raise ValueError(f"H={hot} does not fit the swap-in kernel\\\'s int16 LRU slots")\\n    if resolve_smem_bytes(hot, width) > SMEM_LIMIT:\\n        raise ValueError(f"H={hot} needs {resolve_smem_bytes(hot, width)} B of shared memory per block (max {SMEM_LIMIT})")\\n    return hot\\n\\n\\ndef staging_blocks(max_model_len: int, lookahead: int, block_size: int, override: int = 0) -> int:\\n    """Blocks of the prefill staging slab: two longest contexts by default, at least one."""\\n    one = -(-(max_model_len + lookahead) // block_size)\\n    blocks = override or 2 * one\\n    if blocks < one:\\n        raise ValueError(f"ARC_VLLM_QSA_STAGING_BLOCKS={blocks} cannot hold one context of {one} blocks")\\n    return blocks\\n\\n\\ndef buffer_layout(layers: int, geo: dict) -> tuple[dict[str, tuple[int, int]], int]:\\n    """A worker\\\'s fixed GPU buffers for `layers` QSA layers as one allocation: ({name: (byte offset, bytes)}, size).\\n    Hot K/V rows with their host-row keys (int32) and LRU order (int16) per request slot; the staging slab and its block\\n    table; the request-slot maps; the row buffers shared by all layers for the decode rows (attention indices with the\\n    count column, hot indices, the compact swap plan: int32); the hit and miss counters."""\\n    cells, slots, rows, width = layers * geo["slots"] * geo["hot"], geo["slots"], geo["rows"], geo["width"]\\n    sizes = dict(hot_kv=cells * geo["row_bytes"], slab=geo["staging"] * geo["page_bytes"], keys=4 * cells,\\n                 lru=2 * cells, lru_init=2 * geo["hot"], staging_table=4 * slots * geo["staging"], states=4 * slots,\\n                 request_ids=4 * slots, identity=4 * slots, zeros=4 * rows, attention=4 * rows * (width + 1),\\n                 hot_indices=4 * rows * width, swap_host=4 * rows * width, swap_device=4 * rows * width,\\n                 swap_counts=4 * rows, stats=16)\\n    layout, end = {}, 0\\n    for name, size in sizes.items():\\n        layout[name] = (end, size)\\n        end += -(-size // ALIGN) * ALIGN\\n    return layout, end\\n\\n\\ndef allocator_bytes(size: int) -> int:\\n    """GPU memory PyTorch\\\'s caching allocator reserves for one allocation of `size` bytes that no cached segment serves:\\n    a 2 MiB segment up to 1 MiB, 20 MiB below 10 MiB, else `size` rounded up to 2 MiB (expandable segments: at most\\n    this)."""\\n    if size <= 1 << 20:\\n        return 2 << 20\\n    if size < 10 << 20:\\n        return 20 << 20\\n    return -(-size // (2 << 20)) * (2 << 20)\\n\\n\\ndef reserve_bytes(layers: int, geo: dict) -> int:\\n    """GPU memory a worker\\\'s fixed buffers hold (0 without QSA layers)."""\\n    return allocator_bytes(buffer_layout(layers, geo)[1]) if layers else 0\\n\\n\\ndef host_num_blocks(budget_bytes: int, block_bytes_per_worker: list[int]) -> int:\\n    """Host blocks, equal on every worker: the smallest budget // bytes per host block (all of a worker\\\'s layers)."""\\n    return min(budget_bytes // b for b in block_bytes_per_worker if b)\\n\\n\\ndef merge_state_groups(layer_sets: list[list[str]], keys: list, layer_bytes: dict[str, int],\\n                       worker_layers: list[set[str]], strides: list[int]) -> list[list[int]]:\\n    """Merge KV cache groups: group i (key None: never merged) joins the first earlier merged group with its key whose\\n    bytes on every worker stay within that worker\\\'s block stride. Returns the original group indices of each merged\\n    group, in order of first member."""\\n    merged = []  # [key, member indices, bytes per worker]\\n    for i, (names, key) in enumerate(zip(layer_sets, keys)):\\n        own = [sum(layer_bytes[name] for name in names if name in layers) for layers in worker_layers]\\n        for entry in merged:\\n            if key is not None and entry[0] == key and all(a + b <= s for a, b, s in zip(entry[2], own, strides)):\\n                entry[1].append(i)\\n                entry[2] = [a + b for a, b in zip(entry[2], own)]\\n                break\\n        else:\\n            merged.append([key, [i], own])\\n    return [entry[1] for entry in merged]\\n\\n\\ndef packed_page(state_bytes: int, strides: list[int]) -> int | None:\\n    """ARC_VLLM_GDN_PACK: the page a Mamba layer gets so that two of its kind share every worker\\\'s pool block (half the\\n    smallest block stride, ALIGN-aligned), or None when its state does not fit there (the fp32 GDN state)."""\\n    half = min(s for s in strides if s) // 2 // ALIGN * ALIGN\\n    return half if state_bytes <= half else None\\n\\n\\ndef tight_mamba_groups(kvu, groups, worker_specs):\\n    """ARC_VLLM_GDN_PACK=tight: every Mamba layer (the GDN states and the replicated PLE state) gets the largest Mamba\\n    state rounded up to ALIGN as its page, instead of the platform\\\'s padding to the QSA K/V page (2 MiB at block 4096),\\n    which no longer sets the block once that K/V is in host memory. Returns the groups with those pages and the block\\n    caps for the merge, max(stride, two pages) per worker, so two bf16 GDN layers of a worker share a block of about\\n    twice their state (1,893,376 B against 2 MiB: ~10.8% more blocks at the same --kv-cache-memory-bytes)."""\\n    import dataclasses\\n\\n    states = [s.real_page_size_bytes for g in groups for s in kvu._group_layer_specs(g) if isinstance(s, kvu.MambaSpec)]\\n    if not states:\\n        return groups, kvu._worker_strides(groups, worker_specs)\\n    page = -(-max(states) // ALIGN) * ALIGN\\n\\n    def resize(spec):\\n        return dataclasses.replace(spec, page_size_padded=page) if isinstance(spec, kvu.MambaSpec) else spec\\n\\n    out = []\\n    for group in groups:\\n        spec = group.kv_cache_spec\\n        if isinstance(spec, kvu.UniformTypeKVCacheSpecs):\\n            spec = kvu.UniformTypeKVCacheSpecs(block_size=spec.block_size,\\n                                               kv_cache_specs={n: resize(s) for n, s in spec.kv_cache_specs.items()})\\n        out.append(dataclasses.replace(group, kv_cache_spec=resize(spec)))\\n    return out, [max(s, 2 * page) if s else 0 for s in kvu._worker_strides(out, worker_specs)]\\n\\n\\ndef encode_host_copy(src: int, dst: int) -> tuple[int, int]:\\n    """Host block copy-on-write rides in SchedulerOutput.kv_cache_block_copies as negative ids (a separate id space)."""\\n    return -1 - src, -1 - dst\\n\\n\\ndef split_copies(pairs) -> tuple[list, list[tuple[int, int]]]:\\n    """(GPU block copies as given, host copies decoded)."""\\n    gpu = [pair for pair in pairs if pair[0] >= 0]\\n    host = [(-1 - pair[0], -1 - pair[1]) for pair in pairs if pair[0] < 0]\\n    if any(pair[1] < 0 for pair in gpu) or any(dst < 0 for _, dst in host):\\n        raise ValueError(f"QSA host KV: a block copy mixes host and GPU ids: {list(pairs)}")\\n    return gpu, host\\n\\n\\ndef fold_warmup_block_ids(output, host_groups: list[int], num_blocks: int, packed: dict[int, int] | None = None):\\n    """Engine warmup hands out block ids from one counter over every group and request, bounded by the GPU pool only:\\n    fold the host groups\\\' ids into the host pool\\\'s 1 .. num_blocks - 1 (warmup requests may then share host blocks; a\\n    request\\\'s own blocks stay distinct), and turn the id b of a slot-packed ring group ({group: slots per pool block})\\n    into slot 0 of pool block b. A counter id goes to one group of one request, so pool block b holds that ring alone.\\n    Taken as a slot id, b is slot b % slots of pool block b // slots: from b = slots on (128 at the 2 MiB block this\\n    patch leaves; the stock 16.5 MB block gives 1,024, beyond warmup\\\'s ids) a pool block the same warmup hands to another\\n    group, whose writes overwrite the ring rows the prefill step committed. In place on a SchedulerOutput; returns it."""\\n    tables = [request.block_ids for request in output.scheduled_new_reqs]\\n    tables += [ids for ids in output.scheduled_cached_reqs.new_block_ids if ids is not None]\\n    for table in tables:\\n        for group in host_groups:\\n            table[group][:] = [1 + (block - 1) % (num_blocks - 1) for block in table[group]]\\n        for group, slots in (packed or {}).items():\\n            table[group][:] = [block * slots for block in table[group]]\\n    return output\\n\\n\\ndef row_steps(num_rows: int, rows_per_request: int) -> list[tuple[int, int]]:\\n    """(first row, rows) of each row step j of a uniform decode batch: rows j, j + Q, ... (one per request)."""\\n    return [(j, len(range(j, num_rows, rows_per_request))) for j in range(min(rows_per_request, num_rows))]\\n\\n\\ndef layer_number(name: str) -> int:\\n    """The layer index in a module name such as model.layers.11.self_attn.attn (0 when there is none)."""\\n    import re\\n\\n    match = re.search(r"\\\\.(\\\\d+)\\\\.", f".{name}.")\\n    return int(match.group(1)) if match else 0\\n\\n\\ndef staging_plan(seq_lens: list[int], block_size: int, capacity: int) -> list[list[tuple[int, int, int]]]:\\n    """Consecutive requests grouped so each group\\\'s context blocks fit the slab: per group, (position in seq_lens,\\n    first slab page, blocks) of every request."""\\n    chunks, current, used = [], [], 0\\n    for i, seq_len in enumerate(seq_lens):\\n        blocks = -(-seq_len // block_size)\\n        if blocks > capacity:\\n            raise ValueError(f"QSA host KV: a context of {blocks} blocks exceeds the staging slab ({capacity} blocks)")\\n        if used + blocks > capacity:\\n            chunks.append(current)\\n            current, used = [], 0\\n        current.append((i, used, blocks))\\n        used += blocks\\n    if current:\\n        chunks.append(current)\\n    return chunks\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Engine core: grouping, pool sizing, host block manager, copy-on-write\\n\\n\\ndef geometry(vllm_config, spec) -> dict:\\n    """Shared by the engine core (reservation) and the workers (allocation): rows per request in a decode or verify\\n    forward (Q), the selection width, H, staging blocks and the per-token row of the QSA K/V spec."""\\n    text = vllm_config.model_config.hf_text_config\\n    width = int(text.indexer_budget) + int(text.indexer_compress_ratio) - 1\\n    lookahead = int(vllm_config.num_speculative_tokens or 0)\\n    q = 1 + lookahead\\n    slots = int(vllm_config.scheduler_config.max_num_seqs)\\n    page_bytes = int(spec.page_size_bytes)\\n    return dict(\\n        width=width, q=q, slots=slots, rows=slots * q, page_bytes=page_bytes, block_size=int(spec.block_size),\\n        row_bytes=page_bytes // int(spec.block_size),\\n        steps=hot_steps(q), hot=hot_rows(hot_steps(q), width, hot_tokens()),\\n        staging=staging_blocks(vllm_config.model_config.max_model_len, lookahead, int(spec.block_size),\\n                               int(os.environ.get("ARC_VLLM_QSA_STAGING_BLOCKS", "0"))),\\n    )\\n\\n\\ndef worker_budgets(available_memory: list[int], layers_per_worker: list[int], geo: dict) -> tuple[list[int], list[int]]:\\n    """(GPU pool budget, fixed hot-path bytes) per worker: the fixed buffers come out of --kv-cache-memory-bytes before\\n    the pool is sized, so the pool plus the buffers allocated after it stay within the budget."""\\n    reserves = [reserve_bytes(layers, geo) for layers in layers_per_worker]\\n    budgets = [available - reserve for available, reserve in zip(available_memory, reserves)]\\n    if min(budgets) <= 0:\\n        raise ValueError(f"ARC_VLLM_QSA_HOST_KV: the KV budget {available_memory} does not cover the fixed GPU buffers "\\n                         f"{reserves}")\\n    return budgets, reserves\\n\\n\\ndef merge_mamba_groups(kvu, groups, worker_specs):\\n    """Pair same-schedule Mamba groups across workers (merge_state_groups) without widening any worker\\\'s block. With\\n    ARC_VLLM_GDN_PACK=1, a Mamba layer whose state fits half a block (the bf16 GDN state, --mamba-ssm-cache-dtype\\n    bfloat16) gets that half as its page instead of vLLM\\\'s padding to the whole block, so two layers of one worker share\\n    each block of a merged group (the block-outermost layout puts their pages side by side); fp32 states are unchanged."""\\n    import dataclasses\\n\\n    mode = os.environ.get("ARC_VLLM_GDN_PACK", "0")\\n    pack = mode in ("1", "tight")\\n    groups, strides = (tight_mamba_groups(kvu, groups, worker_specs) if mode == "tight"\\n                       else (groups, kvu._worker_strides(groups, worker_specs)))\\n    keys, layer_bytes, layer_specs, halved = [], {}, {}, 0\\n    for group in groups:\\n        specs = kvu._group_layer_specs(group)\\n        mergeable = (specs and group.slots_per_block == 1\\n                     and all(isinstance(s, kvu.MambaSpec) and not s.tp_replicated for s in specs)\\n                     and len({kvu._mamba_schedule_key(s) for s in specs}) == 1)\\n        if pack and mergeable:\\n            pages = [packed_page(s.real_page_size_bytes, strides) for s in specs]\\n            specs = [s if page is None else dataclasses.replace(s, page_size_padded=page) for s, page in zip(specs, pages)]\\n            halved += sum(page is not None for page in pages)\\n        layer_specs.update(zip(group.layer_names, specs))\\n        layer_bytes.update((name, spec.page_size_bytes) for name, spec in zip(group.layer_names, specs))\\n        keys.append((kvu._mamba_schedule_key(specs[0]), group.is_eagle_group) if mergeable else None)\\n    members = merge_state_groups([g.layer_names for g in groups], keys, layer_bytes,\\n                                 [set(spec) for spec in worker_specs], strides)\\n    if len(members) == len(groups):\\n        return groups\\n    merged = []\\n    for indices in members:\\n        if len(indices) == 1:\\n            merged.append(groups[indices[0]])\\n            continue\\n        specs = {name: layer_specs[name] for i in indices for name in groups[i].layer_names}\\n        merged.append(kvu.KVCacheGroupSpec(\\n            list(specs), kvu.UniformTypeKVCacheSpecs(block_size=groups[indices[0]].kv_cache_spec.block_size,\\n                                                     kv_cache_specs=specs),\\n            is_eagle_group=groups[indices[0]].is_eagle_group))\\n    logger.info("QSA host KV: %d KV cache groups -> %d (Mamba groups paired across workers%s, block bytes per worker "\\n                "at most %s)", len(groups), len(merged), f", {halved} Mamba layers on half-block pages" if halved else "", strides)\\n    return merged\\n\\n\\ndef add_host_group(kvu, config, layers: dict, spec, num_blocks: int, eagle: bool) -> None:\\n    """Append the host K/V group (last group, on every worker) and its layer-outermost host tensors to a worker\\\'s\\n    KVCacheConfig."""\\n    from vllm.v1.hisparse.layout import _build_hisparse_kv_cache_tensors\\n    from vllm.v1.kv_cache_layout import KVCacheLayout\\n\\n    group = kvu.KVCacheGroupSpec(list(layers), spec, is_eagle_group=eagle and bool(layers), host_resident=True,\\n                                 role=ROLE)\\n    config.kv_cache_groups.append(group)\\n    config.hisparse_host_num_blocks = num_blocks\\n    if layers:\\n        block_bytes = len(layers) * spec.page_size_bytes\\n        config.kv_cache_tensors.extend(_build_hisparse_kv_cache_tensors(\\n            [group], num_blocks, block_bytes * num_blocks, KVCacheLayout.LBNHC, block_bytes, host_resident=True))\\n\\n\\ndef patch_kv_cache_utils(kvu) -> None:\\n    configs_fn, groups_fn = kvu.get_kv_cache_configs, kvu.get_kv_cache_groups\\n    context = {}\\n\\n    def get_kv_cache_groups(vllm_config, kv_cache_spec):\\n        groups = groups_fn(vllm_config, kv_cache_spec)\\n        worker_specs = context.get("worker_specs")\\n        return groups if worker_specs is None or not groups else merge_mamba_groups(kvu, groups, worker_specs)\\n\\n    def get_kv_cache_configs(vllm_config, kv_cache_specs, available_memory):\\n        host = [{n: s for n, s in specs.items() if type(s) is kvu.FullAttentionSpec} for specs in kv_cache_specs]\\n        if not any(host):\\n            return configs_fn(vllm_config, kv_cache_specs, available_memory)\\n        if vllm_config.attention_config.hisparse_config is not None or vllm_config.kv_transfer_config is not None:\\n            raise NotImplementedError("ARC_VLLM_QSA_HOST_KV: no native HiSparse and no KV connector")\\n        spec = kvu.FullAttentionSpec.merge([s for layers in host for s in layers.values()])\\n        geo = geometry(vllm_config, spec)\\n        budgets, reserves = worker_budgets(list(available_memory), [len(layers) for layers in host], geo)\\n        host_blocks = host_num_blocks(host_budget_bytes(), [len(layers) * geo["page_bytes"] for layers in host])\\n        per_request = -(-(vllm_config.model_config.max_model_len + geo["q"] - 1) // geo["block_size"])\\n        if host_blocks <= per_request:\\n            raise ValueError(f"ARC_VLLM_QSA_HOST_KV_GB holds {host_blocks} host blocks, one request needs {per_request}"\\n                             " + the null block")\\n        context["worker_specs"] = [{n: s for n, s in specs.items() if n not in layers}\\n                                   for specs, layers in zip(kv_cache_specs, host)]\\n        try:\\n            configs = configs_fn(vllm_config, context["worker_specs"], budgets)\\n        finally:\\n            context.clear()\\n        # The drafter\\\'s K/V layer registers last, as its index cache does on the index group, which vLLM flags as the\\n        # EAGLE group: the host group shares that flag (the two shared one packed group before).\\n        eagle = any(g.is_eagle_group and all(isinstance(s, kvu.FullAttentionSpec) for s in kvu._group_layer_specs(g))\\n                    for config in configs for g in config.kv_cache_groups if g.layer_names)\\n        for config, layers in zip(configs, host):\\n            add_host_group(kvu, config, layers, spec, host_blocks, eagle)\\n        logger.info(\\n            "QSA host KV: %s QSA K/V layers per worker in host memory, %d host blocks (%.1f GB pinned in all, %d tokens "\\n            "per worker); GPU KV budget %s B - fixed %s B (H=%d rows x %d request slots, staging %d blocks) -> %d GPU "\\n            "blocks of %s B", [len(layers) for layers in host], host_blocks,\\n            sum(len(layers) for layers in host) * geo["page_bytes"] * host_blocks / 1e9, host_blocks * geo["block_size"],\\n            list(available_memory), reserves, geo["hot"], geo["slots"], geo["staging"], configs[0].num_blocks,\\n            [kvu._get_kv_cache_bytes_per_block([g for g in c.kv_cache_groups if not g.host_resident]) for c in configs])\\n        return configs\\n\\n    kvu.get_kv_cache_groups, kvu.get_kv_cache_configs = get_kv_cache_groups, get_kv_cache_configs\\n\\n\\ndef patch_single_type_manager(module) -> None:\\n    class QSAHostKVManager(module.FullAttentionManager):\\n        """Full-attention manager over a private host block pool. Host capacity gates admission: no GPU copy exists."""\\n\\n        def __init__(self, *args, **kwargs):\\n            super().__init__(*args, **kwargs)\\n            self._record_new_block_ids = False  # host ids never reach the GPU zeroer\\n            self.device_pool = self.block_pool\\n            self.host_shortfall = 0  # of the last get_num_blocks_to_allocate, read by the coordinator\\n\\n        def take_new_block_ids(self) -> list[int]:\\n            self.new_block_ids = []  # _apply_cow records copy-on-write targets here unconditionally\\n            return []\\n\\n        def bind_host_pool(self, num_blocks: int) -> None:\\n            device_pool = self.device_pool\\n            self.block_pool = module.SharedEventQueueBlockPool(\\n                num_gpu_blocks=num_blocks, enable_caching=self.enable_caching and self.kv_cache_spec.prefix_cacheable,\\n                hash_block_size=device_pool.hash_block_size, enable_kv_cache_events=device_pool.enable_kv_cache_events,\\n                metrics_collector=device_pool.metrics_collector, medium=module.MEDIUM_CPU, event_owner=device_pool)\\n            self._null_block = self.block_pool.null_block\\n\\n        def take_pending_cow_copies(self):\\n            return []  # host ids: take_host_cow_copies hands them over separately\\n\\n        def take_host_cow_copies(self):\\n            copies, self._pending_cow_copies = self._pending_cow_copies, []\\n            return copies\\n\\n        def get_num_blocks_to_allocate(self, request_id, num_tokens, new_computed_blocks, total_computed_tokens,\\n                                       num_local_computed_tokens, num_tokens_main_model, apply_admission_cap=False):\\n            required = super().get_num_blocks_to_allocate(\\n                request_id, num_tokens, new_computed_blocks, total_computed_tokens, num_local_computed_tokens,\\n                num_tokens_main_model, apply_admission_cap=apply_admission_cap)\\n            # Host blocks are not GPU blocks: the coordinator turns a host shortfall into a GPU-pool need that fails\\n            self.host_shortfall = max(required - self.block_pool.get_num_free_blocks(), 0)\\n            return 0\\n\\n    QSAHostKVManager.__qualname__ = QSAHostKVManager.__name__\\n    module.QSAHostKVManager = QSAHostKVManager\\n    module.KVCacheSpecRegistry.register_role_manager(ROLE, QSAHostKVManager)\\n\\n\\ndef patch_coordinator(module) -> None:\\n    init, count = module.KVCacheCoordinator.__init__, module.KVCacheCoordinator.get_num_blocks_to_allocate\\n\\n    def __init__(self, kv_cache_config, *args, **kwargs):\\n        init(self, kv_cache_config, *args, **kwargs)  # subclasses\\\' hit-lookup setup reads the pools after this\\n        for group, manager in zip(kv_cache_config.kv_cache_groups, self.single_type_managers):\\n            if group.role == ROLE:\\n                manager.bind_host_pool(kv_cache_config.hisparse_host_num_blocks)\\n\\n    def get_num_blocks_to_allocate(self, *args, **kwargs):\\n        need = count(self, *args, **kwargs)\\n        short = sum(getattr(manager, "host_shortfall", 0) for manager in self.single_type_managers)\\n        # The scheduler measures the need against the GPU pool\\\'s free blocks. A host shortfall puts it past them by that\\n        # many blocks, so the allocation fails and last_alloc_shortfall is the GPU deficit plus the host deficit: what\\n        # released pins must cover (scheduler.py\\\'s yield test compares it with num_yieldable_blocks over both pools).\\n        return max(need, self.block_pool.get_num_free_blocks()) + short if short else need\\n\\n    module.KVCacheCoordinator.__init__ = __init__\\n    module.KVCacheCoordinator.get_num_blocks_to_allocate = get_num_blocks_to_allocate\\n\\n\\ndef patch_kv_cache_manager(module) -> None:\\n    take = module.KVCacheManager.take_kv_cache_block_copies\\n\\n    def pin_waiting_hit(self, request) -> int:\\n        """Merlin\\\'s pin_waiting_hit (kv_cache_manager.py) with its MTP proof block (the block holding the hash unit past\\n        the hit, a block of its own when the hit ends a block) looked up in each group\\\'s own pool and told apart by\\n        object: Merlin looks in the GPU pool for every group, whose map never holds the host group\\\'s blocks, and skips\\n        a proof block that shares its id with a hit block, although host and GPU block ids overlap."""\\n        request_id = request.request_id\\n        if (request_id in self._pinned_hits or not self.enable_caching or not self.prefix_cache_lookup_enabled(request)\\n                or len(self._pinned_hits) >= self.MAX_PINNED_WAITERS):\\n            return 0\\n        computed, num_hit, _ = self.get_computed_blocks(request)\\n        blocks = [block for group in computed.blocks for block in group if not block.is_null]\\n        if num_hit == 0 or not blocks:\\n            return 0\\n        proof = num_hit // self.block_pool.hash_block_size\\n        if proof < len(request.block_hashes):\\n            seen = {id(block) for block in blocks}\\n            for group_id, manager in enumerate(self.coordinator.single_type_managers):\\n                cached = manager.block_pool.get_cached_block(request.block_hashes[proof], [group_id])\\n                if cached and id(cached[0]) not in seen:\\n                    seen.add(id(cached[0]))\\n                    blocks.append(cached[0])\\n        for block in blocks:\\n            (block.pool or self.block_pool).touch([block])\\n        self._pinned_hits[request_id] = blocks\\n        return len(blocks)\\n\\n    def take_kv_cache_block_copies(self):\\n        copies, retained = take(self)\\n        pairs = [pair for manager in self.coordinator.single_type_managers\\n                 if hasattr(manager, "take_host_cow_copies") for pair in manager.take_host_cow_copies()]\\n        if not pairs:\\n            return copies, retained\\n        # Retained endpoints are freed through the GPU pool, which routes each block to its own pool\\n        return ([*copies, *(module.KVCacheBlockCopy(*encode_host_copy(s.block_id, d.block_id)) for s, d in pairs)],\\n                [*retained, *(block for pair in pairs for block in pair)])\\n\\n    module.KVCacheManager.take_kv_cache_block_copies = take_kv_cache_block_copies\\n    module.KVCacheManager.pin_waiting_hit = pin_waiting_hit\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Worker: allocation, request slots, copy-on-write, the QSA forward\\n\\n_FORCED_ROWS: list[int] = []  # _select_config\\\'s row count while a launch covers part of a batch\\n_WORKER = {}  # "vllm_config", "host_group_ids" (set by initialize_kv_cache), "state" (HostKV)\\n\\n\\nclass HostKV:\\n    """A worker\\\'s host K/V: per QSA layer its host pages and GPU hot buffer, plus buffers shared by all layers."""\\n\\n    def __init__(self, kv_cache_config, host_tensors, device, vllm_config):\\n        import torch\\n        from qsa_hisparse import register_host_memory\\n        from vllm.utils.torch_utils import get_accelerator_view_from_cpu_tensor\\n        from vllm.v1.kv_cache_interface import create_kv_cache_views\\n        from vllm.v1.kv_cache_layout import KVCacheLayout\\n\\n        import vllm._C_stable_libtorch  # noqa: F401 (registers torch.ops._C_cache_ops.hisparse_*)\\n\\n        self.ops = torch.ops._C_cache_ops\\n        group = next(g for g in kv_cache_config.kv_cache_groups if g.role == ROLE)\\n        spec = group.kv_cache_spec\\n        geo = geometry(vllm_config, spec)\\n        self.geo, self.device, self.dtype = geo, device, spec.dtype\\n        self.kv_heads, self.head_size = spec.num_kv_heads, spec.head_size\\n        n, hot, slots = len(group.layer_names), geo["hot"], geo["slots"]\\n        num_blocks = kv_cache_config.hisparse_host_num_blocks\\n        cols = geo["row_bytes"] // spec.dtype.itemsize\\n\\n        started = time.perf_counter()\\n        self.host = torch.empty(host_tensors[0].size, dtype=torch.int8)\\n        register_host_memory(self.host)\\n        registered = time.perf_counter() - started\\n        layout, size = buffer_layout(n, geo)\\n        reserved = torch.cuda.memory_reserved(device)\\n        self.buffer = torch.empty(size, dtype=torch.uint8, device=device)  # one allocation: one rounding, as planned\\n        reserved = torch.cuda.memory_reserved(device) - reserved\\n\\n        def carve(name, dtype, *shape):\\n            start, nbytes = layout[name]\\n            return self.buffer[start:start + nbytes].view(dtype).view(shape)\\n\\n        hot_kv = carve("hot_kv", spec.dtype, n, slots, hot, cols)\\n        keys = carve("keys", torch.int32, n, slots, hot).fill_(-1)\\n        self.lru_init = carve("lru_init", torch.int16, hot).copy_(torch.arange(hot, dtype=torch.int16))\\n        lru = carve("lru", torch.int16, n, slots, hot).copy_(self.lru_init.expand(n, slots, hot))\\n        self.keys, self.lru = keys, lru\\n        self.layers, self.views = {}, {}\\n        index = 0\\n        for tensor in host_tensors:\\n            for name, view in zip(tensor.layers, create_kv_cache_views(self.host, spec, num_blocks, KVCacheLayout.LBNHC,\\n                                                                       tensor)):\\n                uva = get_accelerator_view_from_cpu_tensor(view)\\n                self.views[name] = uva\\n                rows = view.transpose(1, 2)  # [blocks, tokens, heads, width]: one contiguous row per token\\n                self.layers[name] = dict(\\n                    host=rows.reshape(-1, cols), hot=hot_kv[index], keys=keys[index], lru=lru[index],\\n                    hot_kv=hot_kv[index].view(slots, hot, self.kv_heads, -1).split(self.head_size, dim=-1),\\n                    pages=uva.transpose(1, 2).view(torch.uint8).reshape(num_blocks, -1).view(torch.int64))\\n                index += 1\\n        rows, width = geo["rows"], geo["width"]\\n        self.states = carve("states", torch.int32, slots).fill_(-1)  # batch request -> request slot\\n        self.hot_table = self.states.view(slots, 1)  # one hot block of H rows per request slot\\n        self.request_ids = carve("request_ids", torch.int32, slots).copy_(torch.arange(slots, dtype=torch.int32))\\n        self.identity = carve("identity", torch.int32, 1, slots).copy_(self.request_ids.view(1, slots))\\n        self.zeros = carve("zeros", torch.int32, rows).zero_()\\n        self.attention = carve("attention", torch.int32, rows, width + 1).fill_(-1)\\n        self.hot_indices = carve("hot_indices", torch.int32, rows, width)\\n        self.swap_host = carve("swap_host", torch.int32, rows, width)\\n        self.swap_device = carve("swap_device", torch.int32, rows, width)\\n        self.swap_counts = carve("swap_counts", torch.int32, rows).zero_()\\n        self.stats_int = carve("stats", torch.int64, 2).zero_()  # uint64 has few torch ops\\n        self.stats = self.stats_int.view(torch.uint64)  # hits, misses (the swap-in kernel\\\'s atomics)\\n        self.slab = carve("slab", spec.dtype, geo["staging"], geo["block_size"], self.kv_heads, cols // self.kv_heads)\\n        self.slab_kv = self.slab.split(self.head_size, dim=-1)\\n        self.slab_pages = self.slab.view(torch.uint8).view(geo["staging"], -1).view(torch.int64)\\n        self.staging_entries = carve("staging_table", torch.int32, slots * geo["staging"])\\n        self.fuse = os.environ.get("ARC_VLLM_QSA_HOST_KV_FUSE") == "1"\\n        # stage stream: while one QSA layer attends, the next one\\\'s prefill blocks are copied into the other half of the\\n        # slab (target layers in model order; the drafter\\\'s layer is staged in place)\\n        self.stage_stream = torch.cuda.Stream(device) if os.environ.get("ARC_VLLM_QSA_STAGE_STREAM") == "1" else None\\n        # ARC_VLLM_QSA_HOT_OVERLAP: the next row-step group\\\'s swap-in on a side stream while the current group attends\\n        self.overlap_stream = None\\n        if os.environ.get("ARC_VLLM_QSA_HOT_OVERLAP") == "1":\\n            if geo["hot"] < 2 * geo["steps"] * geo["width"]:\\n                raise ValueError(f"ARC_VLLM_QSA_HOT_OVERLAP needs H >= 2 x {geo[\\\'steps\\\']} row steps x {geo[\\\'width\\\']} "\\n                                 f"entries, H is {geo[\\\'hot\\\']}")\\n            self.overlap_stream = torch.cuda.Stream(device)\\n        order = sorted((name for name in self.layers if "mtp" not in name), key=layer_number)\\n        self.next_layer = dict(zip(order, order[1:]))\\n        self.pending, self.prefetch_hits = None, 0\\n        self.forwards = 0\\n        planned = reserve_bytes(n, geo)\\n        (logger.warning if reserved > planned else logger.info)(\\n            "QSA host KV: %d layers x %d host blocks = %.2f GB registered in %.1f s; GPU hot buffers (H=%d x %d slots), "\\n            "maps, row buffers and staging slab (%d blocks): %.3f GB in one allocation, %.3f GB newly reserved by the "\\n            "allocator, %.3f GB reserved in pool sizing", n, num_blocks, self.host.numel() / 1e9, registered, hot, slots,\\n            geo["staging"], size / 1e9, reserved / 1e9, planned / 1e9)\\n\\n    # -- per step (outside CUDA graphs) -------------------------------------------------------------------------------\\n\\n    def set_requests(self, idx_mapping, num_reqs: int) -> None:\\n        """Request slot of every batch request (-1 for padding requests): the hot path\\\'s request_state_indices."""\\n        if num_reqs:\\n            self.states[:num_reqs].copy_(idx_mapping[:num_reqs])\\n        self.states[num_reqs:].fill_(-1)\\n        every = STATS_EVERY\\n        if not num_reqs or every <= 0:\\n            return\\n        self.forwards += 1\\n        if self.forwards % every == 0:\\n            hits, misses = self.stats_int.tolist()\\n            self.stats_int.zero_()\\n            if hits + misses:\\n                logger.info("QSA host KV: last %d forwards: %.1f%% of %d selected entries served from the GPU hot "\\n                            "buffer, %.1f MB read from host per forward", every, 100.0 * hits / (hits + misses),\\n                            hits + misses, misses * self.geo["row_bytes"] / every / 1e6)\\n\\n    def reset_slot(self, req_index: int) -> None:\\n        """A request takes slot req_index: its hot buffers hold nothing (the previous owner\\\'s host blocks are gone)."""\\n        self.keys[:, req_index].fill_(-1)\\n        self.lru[:, req_index].copy_(self.lru_init)\\n\\n    def copy_host_blocks(self, pairs: list[tuple[int, int]]) -> None:\\n        """Copy-on-write of host blocks for every QSA layer of this worker, in stream order."""\\n        # ponytail: host copies (here and in prefill staging) are GPU kernels reading host memory through UVA views;\\n        # move to cuMemcpyBatchAsync (vllm.v1.simple_kv_offload.cuda_mem_ops) if they show up in step times\\n        import torch\\n\\n        src = torch.tensor([s for s, _ in pairs], dtype=torch.int64).to(self.device, non_blocking=True)\\n        dst = torch.tensor([d for _, d in pairs], dtype=torch.int64).to(self.device, non_blocking=True)\\n        for layer in self.layers.values():\\n            layer["pages"].index_copy_(0, dst, layer["pages"].index_select(0, src))\\n\\n    # -- QSA forward --------------------------------------------------------------------------------------------------\\n\\n    def attend(self, layer, host: dict, query, attn_metadata, side, output, token_to_req, use_prefill_config,\\n               output_gate) -> None:\\n        num_tokens = attn_metadata.num_actual_tokens\\n        selected = layer.topk_indices_buffer[:num_tokens]\\n        written = attn_metadata.slot_mapping[:num_tokens]\\n        self.ops.hisparse_invalidate_written_slots(host["keys"], self.states, token_to_req[:written.shape[0]], written)\\n        decode_rows = side.num_decode_tokens if side.decode_query_len else 0\\n        partial = num_tokens if decode_rows < num_tokens else None\\n        q = side.decode_query_len\\n        if decode_rows and self.geo["steps"] >= q:\\n            self.resolve(host, selected, decode_rows, q, attn_metadata.block_table)\\n            self.launch(layer, query[:decode_rows], host["hot_kv"], self.attention[:decode_rows], self.identity,\\n                        self.zeros[:decode_rows], use_prefill_config, output[:decode_rows],\\n                        output_gate[:decode_rows], partial)\\n        elif decode_rows:  # ARC_VLLM_QSA_HOT_STEPS: H holds `steps` row steps, so each group attends before the next\\n            import torch\\n\\n            steps, config_rows = self.geo["steps"], partial or decode_rows\\n            requests = torch.arange(decode_rows // q, device=query.device)[:, None] * q\\n            groups = [range(first, min(first + steps, q)) for first in range(0, q, steps)]\\n            stream = self.overlap_stream if len(groups) > 1 else None\\n            if stream is not None:\\n                self.resolve(host, selected, decode_rows, q, attn_metadata.block_table, groups[0])\\n            for i, group in enumerate(groups):\\n                ready = None\\n                if stream is None:\\n                    self.resolve(host, selected, decode_rows, q, attn_metadata.block_table, group)\\n                elif i + 1 < len(groups):  # the next group\\\'s swap-in, after this group\\\'s, beside this group\\\'s attention\\n                    ready = torch.cuda.Event()\\n                    stream.wait_stream(torch.cuda.current_stream())\\n                    with torch.cuda.stream(stream):\\n                        self.resolve(host, selected, decode_rows, q, attn_metadata.block_table, groups[i + 1])\\n                        ready.record(stream)\\n                rows = (requests + torch.arange(group.start, group.stop, device=query.device)[None, :]).reshape(-1)\\n                out = output.index_select(0, rows)\\n                self.launch(layer, query.index_select(0, rows), host["hot_kv"], self.attention.index_select(0, rows),\\n                            self.identity, self.zeros[:rows.shape[0]], use_prefill_config, out,\\n                            output_gate.index_select(0, rows), config_rows)\\n                output.index_copy_(0, rows, out)\\n                if ready is not None:\\n                    torch.cuda.current_stream().wait_event(ready)\\n        if decode_rows < num_tokens:\\n            self.stage_prefills(layer, host, query, selected, attn_metadata, side, output, token_to_req,\\n                                use_prefill_config, output_gate, num_tokens)\\n\\n    def resolve(self, host: dict, selected, num_rows: int, rows_per_request: int, block_table, group=None) -> None:\\n        """Hot rows for the decode rows [0, num_rows) into self.attention: one swap-in call per row step (of `group`, a\\n        range of row steps; default all), and one gather per row step or (fuse) one after all of them."""\\n        width, hot = selected.shape[1] - 1, self.geo["hot"]\\n        if width != self.geo["width"] or num_rows > self.geo["rows"]:\\n            raise ValueError(f"QSA host KV: {num_rows} rows of width {width}, buffers hold {self.geo[\\\'rows\\\']} rows of "\\n                             f"width {self.geo[\\\'width\\\']}")\\n        plan = 0  # first row of this row step\\\'s swap plan\\n        for first, rows in row_steps(num_rows, rows_per_request):\\n            if group is not None and first not in group:\\n                continue\\n            swap = slice(plan, plan + rows)\\n            self.ops.hisparse_resolve_residency(\\n                host["host"], host["hot"], self.hot_table, selected[first:num_rows:rows_per_request, :width],\\n                self.hot_indices[:rows], host["keys"], host["lru"], self.states, hot, None, self.stats,\\n                self.attention[first:num_rows:rows_per_request, :width], hot, self.request_ids[:rows], block_table,\\n                self.geo["block_size"], None, None, self.swap_host[swap], self.swap_device[swap],\\n                self.swap_counts[swap], None, 0, 0)\\n            if self.fuse:\\n                plan += rows\\n            else:\\n                self.ops.hisparse_gather_compact(host["host"], host["hot"], self.swap_host[swap],\\n                                                 self.swap_device[swap], self.swap_counts[swap])\\n        if self.fuse:\\n            self.ops.hisparse_gather_compact(host["host"], host["hot"], self.swap_host[:plan], self.swap_device[:plan],\\n                                             self.swap_counts[:plan])\\n        if group is None:\\n            self.attention[:num_rows, width].copy_(selected[:num_rows, width])\\n        else:  # only this group\\\'s rows: another group may be attending beside it (ARC_VLLM_QSA_HOT_OVERLAP)\\n            for first in group:\\n                self.attention[first:num_rows:rows_per_request, width].copy_(selected[first:num_rows:rows_per_request, width])\\n\\n    def stage_prefills(self, layer, host: dict, query, selected, attn_metadata, side, output, token_to_req,\\n                       use_prefill_config, output_gate, num_tokens: int) -> None:\\n        """Prefill rows: copy their requests\\\' context blocks to the slab, attend through a remapped block table."""\\n        import numpy as np\\n        import torch\\n        from vllm.utils.torch_utils import async_tensor_h2d\\n\\n        starts = side.arc_query_start_loc_cpu.tolist()\\n        seq_lens = side.arc_seq_lens_cpu\\n        seq_lens = [attn_metadata.max_seq_len] * (len(starts) - 1) if seq_lens is None else seq_lens.tolist()\\n        requests = [r for r in range(side.num_decodes, len(starts) - 1) if starts[r + 1] > starts[r]]\\n        block_table = attn_metadata.block_table\\n        table = self.staging_table(block_table.shape[1])[:block_table.shape[0]]\\n        limit = block_table.shape[1] * self.geo["block_size"]  # a block table row covers at most this many tokens\\n        plan = staging_plan([min(seq_lens[r], limit) for r in requests], self.geo["block_size"], self.geo["staging"])\\n        prefetched, name = None, getattr(layer, "layer_name", None)\\n        # what makes two QSA layers\\\' staging the same: the batch\\\'s block table and the plan (the CPU lengths)\\n        key = (id(block_table), tuple(starts), tuple(seq_lens[r] for r in requests), tuple(map(tuple, plan[0])))\\n        if self.stage_stream is not None:\\n            prefetched = self.take_prefetch(name, key)\\n            if not (len(plan) == 1 and sum(blocks for _, _, blocks in plan[0]) <= self.geo["staging"] // 2):\\n                prefetched = None  # several chunks or one over half the slab: the whole slab, staged here\\n        for chunk in plan:\\n            members = [requests[i] for i, _, _ in chunk]\\n            pages = sum(blocks for _, _, blocks in chunk)\\n            ids = torch.cat([block_table[r, :blocks] for r, (_, _, blocks) in zip(members, chunk)]).long()\\n            base = 0\\n            if prefetched is None:\\n                torch.index_select(host["pages"], 0, ids, out=self.slab_pages[:pages])\\n            else:\\n                base = prefetched * (self.geo["staging"] // 2)\\n                # this forward wrote its own tokens\\\' K/V after the prefetch read them: those blocks again (from one\\n                # hash unit below the first new token, as seq_lens is an upper bound)\\n                fresh = [first + b for r, (_, first, blocks) in zip(members, chunk)\\n                         for b in range(max(0, seq_lens[r] - (starts[r + 1] - starts[r]) - 64) // self.geo["block_size"],\\n                                        blocks)]\\n                self.prefetch_hits += 1\\n                if fresh:\\n                    slots = torch.tensor(fresh, dtype=torch.int64).to(self.device, non_blocking=True)\\n                    self.slab_pages[base:base + pages].index_copy_(0, slots, host["pages"].index_select(0, ids[slots]))\\n            if self.stage_stream is not None and len(plan) == 1 and pages <= self.geo["staging"] // 2:\\n                self.prefetch(name, key, ids, pages, 0 if prefetched == 1 else 1)\\n            remap = np.full((len(members), block_table.shape[1]), -1, dtype=np.int32)\\n            for k, (_, first, blocks) in enumerate(chunk):\\n                remap[k, :blocks] = np.arange(base + first, base + first + blocks)\\n            table.fill_(-1)\\n            table[async_tensor_h2d(np.array(members), device=self.device, dtype=torch.int64)] = async_tensor_h2d(\\n                remap, device=self.device)\\n            rows = slice(starts[members[0]], starts[members[-1] + 1])\\n            self.launch(layer, query[rows], self.slab_kv, selected[rows], table, token_to_req[rows],\\n                        use_prefill_config, output[rows], output_gate[rows], num_tokens)\\n\\n    def take_prefetch(self, name, key):\\n        """The slab half holding this layer\\\'s prefetched blocks (0 or 1), or None; any other pending prefetch is waited\\n        for first, so the slab is never written by the copy stream while this layer reads it."""\\n        import torch\\n\\n        pending, self.pending = self.pending, None\\n        if pending is None:\\n            return None\\n        torch.cuda.current_stream(self.device).wait_event(pending["done"])\\n        return pending["half"] if pending["layer"] == name and pending["key"] == key else None\\n\\n    def prefetch(self, name, key, ids, pages: int, half: int) -> None:\\n        """On the copy stream, the next QSA layer\\\'s copy of the same blocks into slab half `half`, after everything the\\n        compute stream has queued so far (the previous reader of that half, the ids)."""\\n        import torch\\n\\n        following = self.next_layer.get(name)\\n        if following is None:\\n            return\\n        compute = torch.cuda.current_stream(self.device)\\n        self.stage_stream.wait_stream(compute)\\n        start = half * (self.geo["staging"] // 2)\\n        with torch.cuda.stream(self.stage_stream):\\n            torch.index_select(self.layers[following]["pages"], 0, ids, out=self.slab_pages[start:start + pages])\\n            done = torch.cuda.Event()\\n            done.record(self.stage_stream)\\n        ids.record_stream(self.stage_stream)\\n        self.pending = {"layer": following, "key": key, "half": half, "done": done}\\n\\n    def staging_table(self, width: int):\\n        """A [slots, width] block table for the slab (width: the batch block table\\\'s, at most the slab\\\'s blocks)."""\\n        if width > self.geo["staging"]:\\n            raise ValueError(f"QSA host KV: a block table of width {width} exceeds the staging slab "\\n                             f"({self.geo[\\\'staging\\\']} blocks)")\\n        return self.staging_entries[:self.geo["slots"] * width].view(self.geo["slots"], width)\\n\\n    @staticmethod\\n    def launch(layer, query, kv, indices, block_table, token_to_req, use_prefill_config, out, output_gate,\\n               config_rows) -> None:\\n        """The unchanged QSA kernel over some rows; config_rows: the batch\\\'s row count for its launch configuration."""\\n        from vllm.models.qwen4_exp.nvidia.ops.qsa import qsa_sparse_paged_attention\\n\\n        if config_rows is not None:\\n            _FORCED_ROWS.append(config_rows)\\n        try:\\n            qsa_sparse_paged_attention(query, kv[0], kv[1], indices, block_table, token_to_req, use_prefill_config,\\n                                       out=out, k_scale=layer._k_scale, v_scale=layer._v_scale,\\n                                       output_gate=output_gate)\\n        finally:\\n            if config_rows is not None:\\n                _FORCED_ROWS.pop()\\n\\n\\ndef state() -> HostKV | None:\\n    return _WORKER.get("state")\\n\\n\\ndef patch_attn_utils(module) -> None:\\n    allocate = module.allocate_kv_cache\\n\\n    def allocate_kv_cache(kv_cache_config, device, layout, kernel_block_sizes=None):\\n        import copy\\n\\n        host = [t for t in kv_cache_config.kv_cache_tensors if t.host_resident]\\n        if not host:\\n            return allocate(kv_cache_config, device, layout, kernel_block_sizes)\\n        device_config = copy.copy(kv_cache_config)\\n        device_config.kv_cache_tensors = [t for t in kv_cache_config.kv_cache_tensors if not t.host_resident]\\n        if state() is not None:  # a second 10 GB registration would never be freed\\n            raise RuntimeError("QSA host KV: KV cache initialized twice in one worker (needs "\\n                               "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0)")\\n        caches = allocate(device_config, device, layout, kernel_block_sizes)\\n        host_kv = _WORKER["state"] = HostKV(kv_cache_config, host, device, _WORKER["vllm_config"])\\n        caches.update(host_kv.views)  # layers bind their UVA views: reshape_and_cache_flash writes to host memory\\n        return caches\\n\\n    module.allocate_kv_cache = allocate_kv_cache\\n\\n\\ndef patch_model_runner(module) -> None:\\n    cls = module.GPUModelRunner\\n    initialize = cls.initialize_kv_cache\\n\\n    def initialize_kv_cache(self, kv_cache_config, *args, **kwargs):\\n        _WORKER.update(vllm_config=self.vllm_config, host_group_ids=kv_cache_config.host_group_ids)\\n        initialize(self, kv_cache_config, *args, **kwargs)\\n        host_kv = state()\\n        if host_kv is not None:  # generic copy-on-write must not see host views (separate block ids)\\n            host = {view.data_ptr() for view in host_kv.views.values()}\\n            self.kv_caches = [cache for cache in self.kv_caches if cache.data_ptr() not in host]\\n\\n    class KVBlockZeroer(module.KVBlockZeroer):\\n        """The zeroer skips the host group: its block ids are not GPU pool ids (and attention reads written rows only)."""\\n\\n        def __init__(self, *args, skip_kv_cache_group_ids=(), **kwargs):\\n            skip = (*skip_kv_cache_group_ids, *_WORKER.get("host_group_ids", ()))\\n            super().__init__(*args, skip_kv_cache_group_ids=skip, **kwargs)\\n\\n    copy_blocks = module.copy_kv_cache_blocks_inplace\\n\\n    def copy_kv_cache_blocks_inplace(kv_caches, num_blocks, kv_cache_block_copies):\\n        gpu, host = split_copies(kv_cache_block_copies or [])\\n        copy_blocks(kv_caches, num_blocks, gpu)\\n        if host:\\n            state().copy_host_blocks(host)\\n\\n    cls.initialize_kv_cache = initialize_kv_cache\\n    module.KVBlockZeroer = KVBlockZeroer\\n    module.copy_kv_cache_blocks_inplace = copy_kv_cache_blocks_inplace\\n\\n\\ndef patch_warmup(module) -> None:\\n    """warmup_kernels\\\' scheduler outputs (warmup.py:_warmup_kernels): host-group block ids folded into the host pool,\\n    which may be far smaller than the GPU pool that bounds them, and QSA ring ids moved to pool blocks of their own\\n    (fold_warmup_block_ids). run_mixed_prefill_decode_warmup only runs for DeepSeek V4\\\'s sparse MLA."""\\n    warmup = module._warmup_kernels\\n\\n    def _warmup_kernels(model_runner, worker_execute_model, worker_sample_tokens):\\n        config = model_runner.kv_cache_config\\n        host = [i for i, group in enumerate(config.kv_cache_groups) if group.role == ROLE]\\n        packed = {i: group.slots_per_block for i, group in enumerate(config.kv_cache_groups)\\n                  if group.slots_per_block > 1}\\n        execute = worker_execute_model\\n        if host or packed:\\n            def worker_execute_model(output):\\n                return execute(fold_warmup_block_ids(output, host, config.hisparse_host_num_blocks, packed))\\n        return warmup(model_runner, worker_execute_model, worker_sample_tokens)\\n\\n    module._warmup_kernels = _warmup_kernels\\n\\n\\ndef patch_model_state(module) -> None:\\n    cls = module.Qwen4ExpModelState\\n    prepare, prepare_dummy, add = cls.prepare_inputs, cls.prepare_dummy_inputs, cls.add_request\\n\\n    def prepare_inputs(self, input_batch, req_states):\\n        if state() is not None:  # the drafter\\\'s forwards of this step use the same batch order\\n            state().set_requests(input_batch.idx_mapping, input_batch.num_reqs)\\n        return prepare(self, input_batch, req_states)\\n\\n    def prepare_dummy_inputs(self, num_reqs, num_tokens):\\n        if state() is not None:  # capture and dummy runs leave every hot buffer alone\\n            state().set_requests(None, 0)\\n        return prepare_dummy(self, num_reqs, num_tokens)\\n\\n    def add_request(self, req_index, new_req_data):\\n        add(self, req_index, new_req_data)\\n        if state() is not None:\\n            state().reset_slot(req_index)\\n\\n    cls.prepare_inputs, cls.prepare_dummy_inputs, cls.add_request = prepare_inputs, prepare_dummy_inputs, add_request\\n\\n\\ndef patch_qsa_cache(module) -> None:\\n    """The prefill staging plan reads the batch\\\'s request boundaries and sequence lengths on the CPU."""\\n    cls = module.QSAMetadataBuilder\\n    build = cls.build\\n\\n    def build_with_cpu_lengths(self, common_prefix_len, common_attn_metadata, fast_build=False):\\n        metadata = build(self, common_prefix_len, common_attn_metadata, fast_build)\\n        metadata.arc_query_start_loc_cpu = common_attn_metadata.query_start_loc_cpu\\n        metadata.arc_seq_lens_cpu = common_attn_metadata.seq_lens_cpu_upper_bound\\n        return metadata\\n\\n    cls.build = build_with_cpu_lengths\\n\\n\\ndef patch_qsa(module) -> None:\\n    impl = module.Qwen4ExpQSAFlashAttentionImpl\\n    forward = impl.forward_qsa\\n\\n    def forward_qsa(self, layer, query, key, value, kv_cache, attn_metadata, output, token_to_req, use_prefill_config,\\n                    output_gate, output_scale=None, output_block_scale=None):\\n        host_kv = state()\\n        host = None if host_kv is None else host_kv.layers.get(getattr(layer, "layer_name", None))\\n        if host is None or output_scale is not None or output_block_scale is not None:\\n            return forward(self, layer, query, key, value, kv_cache, attn_metadata, output, token_to_req,\\n                           use_prefill_config, output_gate, output_scale=output_scale,\\n                           output_block_scale=output_block_scale)\\n        from vllm.forward_context import get_forward_context\\n\\n        output.zero_()\\n        if attn_metadata.num_actual_tokens == 0:\\n            return output\\n        metadata = get_forward_context().attn_metadata\\n        side = (metadata[0] if isinstance(metadata, list) else metadata)[layer.indexer.raw_key_cache.prefix]\\n        host_kv.attend(layer, host, query, attn_metadata, side, output, token_to_req, use_prefill_config, output_gate)\\n        return output\\n\\n    impl.forward_qsa = forward_qsa\\n\\n\\ndef patch_qsa_ops(module) -> None:\\n    select = module._select_config\\n\\n    def _select_config(num_rows, num_kv_heads, use_prefill_config, num_columns):\\n        return select(_FORCED_ROWS[-1] if _FORCED_ROWS else num_rows, num_kv_heads, use_prefill_config, num_columns)\\n\\n    module._select_config = _select_config\\n\\n\\ndef patches() -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook (chained after hooks already registered there)."""\\n    return {\\n        "vllm.v1.core.kv_cache_utils": patch_kv_cache_utils,\\n        "vllm.v1.core.single_type_kv_cache_manager": patch_single_type_manager,\\n        "vllm.v1.core.kv_cache_coordinator": patch_coordinator,\\n        "vllm.v1.core.kv_cache_manager": patch_kv_cache_manager,\\n        "vllm.v1.worker.gpu.attn_utils": patch_attn_utils,\\n        "vllm.v1.worker.gpu.model_runner": patch_model_runner,\\n        "vllm.v1.worker.gpu.warmup": patch_warmup,\\n        "vllm.models.qwen4_exp.nvidia.model_state": patch_model_state,\\n        "vllm.models.qwen4_exp.common.qsa_cache": patch_qsa_cache,\\n        "vllm.models.qwen4_exp.nvidia.qsa": patch_qsa,\\n        "vllm.models.qwen4_exp.nvidia.ops.qsa": patch_qsa_ops,\\n    }\\n\', \'src/qwen/vllm_patch/qsa_hisparse.py\': \'"""Host-resident K/V for Qwen3.8-Flash-Next\\\'s QSA layers under SGLang 0.5.21 (ARC_SGLANG_QSA_HISPARSE=1).\\n\\nThe QSA (sparse full-attention) layers keep their K/V cache in registered host memory instead of on the GPU: the\\ntarget\\\'s 12 layers and, unless ARC_SGLANG_QSA_HOST_DRAFT=0, the MTP drafter\\\'s layer. Per token the GPU then holds only\\nthe QSA compressed indexes (832 B at TP4; 1,344 B with the drafter\\\'s K/V on the GPU), so the KV pool grows several\\nfold on a 4x L4 server. Every decode, verify or draft step reads only the entries QSA selected for each query row:\\n\\n- ARC_SGLANG_QSA_HOT_TOKENS=H (default 4096): each request keeps a GPU hot buffer of H entries per QSA layer, managed by\\n  HiSparse\\\'s LRU swap-in kernel (sglang/kernels/jit/csrc/kvcacheio/hisparse.cuh, token mode with separate K and V,\\n  JIT-compiled from a copy that keeps -1 padding out of its hash, see _TOKEN_MODE_GUARD, written to\\n  $TMPDIR/arc_qsa_hisparse); only entries that are not in the buffer cross PCIe. The kernel runs once per query row of\\n  every request (4 rows in MTP verify and draft extend), and each row is packed into the attention scratch right after\\n  its call, so the rows of a request load a shared entry once. H must be at\\n  least the row width plus a margin (2,051 + 16; the drafter\\\'s decode lookup adds speculative_num_steps + 1 columns).\\n- ARC_SGLANG_QSA_HOT_TOKENS=0: no hot buffer, every selected entry is read from host memory (a correctness reference;\\n  about 1 GB of PCIe reads per MTP verify step at 20 requests).\\n\\nOutputs are unchanged: attention receives the same FP8 bytes, converted to bf16 the same way, as from the GPU pool.\\nHot entries are keyed by position; at the start of every forward, entries at positions the forward writes again are\\ndropped (verify: >= seq_lens; draft extend: >= its window; decode and draft steps: >= seq_lens - 1; extend: >= the\\nprefix, which also resets a new or retracted request), and an int64 position->slot copy of req_to_token is kept for\\nthe kernel\\\'s host addresses.\\n\\nGPU memory: the hot buffers (target and drafter), their maps and one bf16 pack scratch shared by every QSA backend\\n(target, draft extend, each draft decode step; SGLang otherwise allocates one per backend during CUDA graph capture,\\n~0.5 GiB at 24 requests, out of the --mem-fraction-static headroom) are taken from the KV budget before the pool is\\nsized (log: "KV budget ... - fixed ..."). After CUDA graph capture the server logs the memory left for activations\\nand warns below ARC_SGLANG_MIN_FREE_GIB (default 0.5).\\n\\nPatched modules (import hooks, see register):\\n- sglang.srt.model_executor.pool_configurator: GPU bytes per token and the fixed reservation.\\n- sglang.srt.mem_cache.kv_cache_configurator: host MHA pools (one cudaHostRegister\\\'ed buffer, K and V views per\\n  layer) with their hot buffers for the target\\\'s QSA pool and the drafter\\\'s.\\n- sglang.srt.layers.attention.qwen_sparse_attn_backend: the K/V pack before sparse attention reads host / hot memory,\\n  the shared scratch, chunked prefill with a prefix gathers its context from host memory, and the metadata hooks run\\n  the per-forward bookkeeping (eager, CUDA-graph capture and replay, the drafter\\\'s multi-step decode backend).\\n- sglang.srt.managers.scheduler: the free-memory check after CUDA graph capture.\\nNeeds --max-running-requests (it sizes the hot buffers) and FP8 or bf16 KV with equal K/V head sizes; the swap-in\\nkernel is JIT-compiled by SGLang (nvcc and ninja, as for the Marlin repack). Every STATS_EVERY (256) forwards of each\\nmodel the server logs the share of selected entries served from the GPU and the bytes read from host. The plan stream\\n(SGLANG_ENABLE_OVERLAP_PLAN_STREAM, off by default) is not supported.\\n"""\\nimport functools\\nimport logging\\nimport os\\nimport time\\n\\nlogger = logging.getLogger(__name__)\\n\\nSWAP_IN_BLOCK = 960  # threads per request block of HiSparse\\\'s swap-in kernel (its default swap_in_block_size)\\nPACK_BLOCK_N = 16  # selected entries per program of the pack kernel (as SGLang\\\'s _compact_kv)\\n_FAKE_SEQ_LEN = 1 << 30  # > any position: turns off the kernel\\\'s short-sequence fast path and newest-slot binding\\nWIDTH_MARGIN = 16  # extra row width over QSA\\\'s 2,051: the MTP decode lookup appends speculative_num_steps + 1 tail columns\\nSTATS_EVERY = 256  # forwards of a model between its hot-buffer statistics lines (0: none)\\n_HOST_TENSORS = {}  # data_ptr of a host K or V layer buffer -> (QSAHostKV, layer index)\\n# The bf16 pack scratch of every QSA backend in the process: (kv heads, head dim, dtype, device index) -> (k, v).\\n# SGLang gives each backend (target, draft extend, each draft decode step) its own scratch, allocated during CUDA graph\\n# capture from the --mem-fraction-static headroom; they run one after another on the forward stream, so one buffer,\\n# allocated with the hot buffer and reserved in pool sizing, serves them all.\\n_SHARED_SCRATCH = {}\\n# Registered host buffers are never freed: freeing one without cudaHostUnregister leaves a stale registration that\\n# breaks later copies from memory reusing those addresses ("invalid argument"). Pools live as long as the server.\\n_REGISTERED = []\\n\\n\\ndef hot_tokens() -> int:\\n    return int(os.environ.get("ARC_SGLANG_QSA_HOT_TOKENS", "4096"))\\n\\n\\ndef host_draft() -> bool:\\n    return os.environ.get("ARC_SGLANG_QSA_HOST_DRAFT", "1") != "0"\\n\\n\\ndef min_free_gib() -> float:\\n    return float(os.environ.get("ARC_SGLANG_MIN_FREE_GIB", "0.5"))\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Kernels (Triton is imported only when a patched SGLang module loads)\\n\\n\\n@functools.cache\\ndef kernels():\\n    import triton\\n    import triton.language as tl\\n\\n    @triton.jit\\n    def store_rows(src, src_stride, dst, loc, ROW: tl.constexpr, BLOCK: tl.constexpr):\\n        """dst[loc[i]] = src[i] for rows of ROW elements; dst may be host memory."""\\n        i = tl.program_id(0)\\n        cols = tl.arange(0, BLOCK)\\n        mask = cols < ROW\\n        slot = tl.load(loc + i).to(tl.int64)\\n        tl.store(dst + slot * ROW + cols, tl.load(src + i.to(tl.int64) * src_stride + cols, mask=mask), mask=mask)\\n\\n    @triton.jit\\n    def gather_rows(src, slots, out, ROW: tl.constexpr, BLOCK: tl.constexpr):\\n        """out[i] = src[slots[i]] for rows of ROW elements; src may be host memory."""\\n        i = tl.program_id(0)\\n        cols = tl.arange(0, BLOCK)\\n        mask = cols < ROW\\n        slot = tl.load(slots + i).to(tl.int64)\\n        tl.store(out + i.to(tl.int64) * ROW + cols, tl.load(src + slot * ROW + cols, mask=mask), mask=mask)\\n\\n    @triton.jit\\n    def pack_selected(k, v, table, table_rows, table_stride, positions, positions_stride, lengths, cu_k, out_k, out_v,\\n                      row0, row_step, HEADS: tl.constexpr, DIM: tl.constexpr, BY_POSITION: tl.constexpr,\\n                      BLOCK_N: tl.constexpr, BLOCK_D: tl.constexpr):\\n        """SGLang\\\'s _compact_kv (qsa/sparse_attn.py) for query rows row0 + i * row_step: entry c of row r is valid if\\n        c < cu_k[r+1] - cu_k[r] and 0 <= positions[r, c] < lengths[r], and then converted to out\\\'s dtype and stored at\\n        cu_k[r] + c. BY_POSITION: its slot is table[table_rows[r], positions[r, c]] (req_to_token, host memory as k/v).\\n        Otherwise it is table[r, c] (the swap-in kernel\\\'s hot-buffer slots, -1 = not loaded)."""\\n        i = tl.program_id(0)\\n        head = tl.program_id(1)\\n        block = tl.program_id(2)\\n        row = row0 + i * row_step\\n        start = tl.load(cu_k + row)\\n        count = tl.load(cu_k + row + 1) - start\\n        cols = block * BLOCK_N + tl.arange(0, BLOCK_N)\\n        valid = cols < count\\n        pos = tl.load(positions + row.to(tl.int64) * positions_stride + cols, mask=valid, other=-1)\\n        length = tl.load(lengths + row)\\n        valid = valid & (pos >= 0) & (pos < length)\\n        if BY_POSITION:\\n            table_row = tl.load(table_rows + row).to(tl.int64)\\n            slot = tl.load(table + table_row * table_stride + tl.where(valid, pos, 0), mask=valid, other=0)\\n        else:\\n            slot = tl.load(table + row.to(tl.int64) * table_stride + cols, mask=valid, other=-1)\\n            valid = valid & (slot >= 0)\\n        dims = tl.arange(0, BLOCK_D)\\n        src = slot.to(tl.int64)[:, None] * (HEADS * DIM) + head * DIM + dims[None, :]\\n        dst = (start + cols).to(tl.int64)[:, None] * (HEADS * DIM) + head * DIM + dims[None, :]\\n        mask = valid[:, None] & (dims[None, :] < DIM)\\n        out_dtype = out_k.dtype.element_ty\\n        tl.store(out_k + dst, tl.load(k + src, mask=mask, other=0.0).to(out_dtype), mask=mask)\\n        tl.store(out_v + dst, tl.load(v + src, mask=mask, other=0.0).to(out_dtype), mask=mask)\\n\\n    return store_rows, gather_rows, pack_selected\\n\\n\\ndef _rows(t):\\n    """2-D [rows, elements] view of a [rows, heads, dim] tensor with contiguous rows."""\\n    t = t.reshape(t.shape[0], -1)\\n    return t if t.stride(1) == 1 else t.contiguous()\\n\\n\\ndef store_rows(src, dst, loc) -> None:\\n    import triton\\n\\n    src = _rows(src)\\n    if src.shape[0]:\\n        row = src.shape[1]\\n        kernels()[0][(src.shape[0],)](src, src.stride(0), dst, loc, ROW=row, BLOCK=triton.next_power_of_2(row))\\n\\n\\ndef gather_rows(src, slots, out) -> None:\\n    import triton\\n\\n    row = out[0].numel() if out.shape[0] else 0\\n    if out.shape[0]:\\n        kernels()[1][(out.shape[0],)](src, slots, out, ROW=row, BLOCK=triton.next_power_of_2(row))\\n\\n\\ndef pack_selected(k, v, table, table_rows, positions, lengths, cu_k, out_k, out_v, *, rows, row0=0, row_step=1,\\n                  by_position=True, width) -> None:\\n    import triton\\n\\n    if rows == 0:\\n        return\\n    heads, dim = k.shape[1], k.shape[2]\\n    kernels()[2][(rows, heads, triton.cdiv(width, PACK_BLOCK_N))](\\n        k, v, table, table_rows, table.stride(0), positions, positions.stride(0), lengths, cu_k, out_k, out_v,\\n        row0, row_step, HEADS=heads, DIM=dim, BY_POSITION=by_position, BLOCK_N=PACK_BLOCK_N,\\n        BLOCK_D=triton.next_power_of_2(dim), num_warps=8)\\n\\n\\n# SGLang 0.5.21\\\'s swap-in kernel skips invalid selections before its shared-memory hash only in block mode. In token\\n# mode a -1 column enters the hash: its atomicCAS(EMPTY, -1) "succeeds" on an empty slot and its index store can land\\n# after a real token claimed that slot, so the real token is not seen as a hit, is loaded again, and its copies pile up\\n# in the hot buffer until misses outnumber the evictable slots and two misses share one (wrong K/V). QSA rows end in -1\\n# padding, so the kernel is compiled from a copy with the block-mode guard applied to token mode as well.\\n_TOKEN_MODE_GUARD = (\\n    "    if constexpr (TopKIsBlocks) {\\\\n      if (token_idx < 0 || token_idx >= seq_len) {",\\n    "    {  // qsa_hisparse.py: token mode skips invalid (-1) selections too\\\\n"\\n    "      if (token_idx < 0 || token_idx >= seq_len) {",\\n)\\n\\n\\ndef guarded_kernel_source() -> str:\\n    """Path of hisparse.cuh with the token-mode guard, written once per content hash (atomically: TP ranks race)."""\\n    import hashlib\\n    import tempfile\\n    from pathlib import Path\\n\\n    import sglang.kernels.ops.kvcache.hisparse as ops\\n\\n    source = (Path(ops.__file__).resolve().parents[2] / "jit/csrc/kvcacheio/hisparse.cuh").read_text()\\n    old, new = _TOKEN_MODE_GUARD\\n    if source.count(old) != 1:\\n        raise RuntimeError("QSA host KV: hisparse.cuh no longer has the expected token insert loop (SGLang != 0.5.21?)")\\n    patched = source.replace(old, new)\\n    directory = Path(tempfile.gettempdir()) / "arc_qsa_hisparse"\\n    path = directory / f"hisparse_token_guard_{hashlib.sha256(patched.encode()).hexdigest()[:16]}.cuh"\\n    if not path.exists():\\n        directory.mkdir(parents=True, exist_ok=True)\\n        partial = path.with_suffix(f".{os.getpid()}.part")\\n        partial.write_text(patched)\\n        os.replace(partial, path)\\n    return str(path)\\n\\n\\n@functools.cache\\ndef swap_in_module(item_bytes: int, width: int, hot: int):\\n    """HiSparse\\\'s swap-in kernel (guarded copy): token mode, separate K and V (is_mla=False), miss plan recorded for\\n    the statistics. Same template arguments as sglang.kernels.ops.kvcache.hisparse._jit_sparse_module."""\\n    from sglang.kernels.jit.utils import load_jit, make_cpp_args\\n\\n    flags = (SWAP_IN_BLOCK, width, hot, False, False, 1, False, True, False)  # is_mla, dsv4, block 1, blocks, plan, skip\\n    template_args = make_cpp_args(*flags)\\n    return load_jit("qsa_sparse_cache", *make_cpp_args(item_bytes, *flags), cuda_files=[guarded_kernel_source()],\\n                    cuda_wrappers=[("load_cache_to_device_buffer", f"load_cache_to_device_buffer<{template_args}>")])\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Host memory\\n\\n\\ndef register_host_memory(tensor) -> None:\\n    """Pin an existing CPU tensor (exact size, unlike pin_memory=True\\\'s power-of-two blocks) and map it for the GPU."""\\n    import torch\\n\\n    flags = 1 | 2  # cudaHostRegisterPortable | cudaHostRegisterMapped\\n    error = torch.cuda.cudart().cudaHostRegister(tensor.data_ptr(), tensor.numel() * tensor.element_size(), flags)\\n    if error != torch.cuda.cudart().cudaError.success:\\n        raise RuntimeError(f"cudaHostRegister of {tensor.numel() * tensor.element_size() / 2**30:.2f} GiB failed: {error}")\\n    _REGISTERED.append(tensor)\\n\\n\\n@functools.cache\\ndef host_pool_class(base):\\n    """base (SGLang\\\'s MHATokenToKVPool) with its K/V buffers in registered host memory."""\\n\\n    class QSAHostMHATokenToKVPool(base):\\n        def _create_buffers_normal(self):\\n            import torch\\n\\n            if self.use_hnd or self.kv_cache_layout != "nhd":\\n                raise NotImplementedError(f"QSA host KV supports the NHD layout only, got {self.kv_cache_layout}")\\n            k_shape, v_shape = self._kv_buffer_shapes()\\n            itemsize = self.store_dtype.itemsize\\n            k_bytes, v_bytes = (int(torch.Size(s).numel()) * itemsize for s in (k_shape, v_shape))\\n            started = time.perf_counter()\\n            self._host_storage = torch.empty(self.layer_num * (k_bytes + v_bytes), dtype=torch.uint8)\\n            register_host_memory(self._host_storage)\\n            views, offset = [], 0\\n            for shape, nbytes in [(k_shape, k_bytes)] * self.layer_num + [(v_shape, v_bytes)] * self.layer_num:\\n                views.append(self._host_storage[offset:offset + nbytes].view(self.store_dtype).view(shape))\\n                offset += nbytes\\n            self.k_buffer, self.v_buffer = views[:self.layer_num], views[self.layer_num:]\\n            logger.info("QSA host KV: %d layers x %d slots in host memory, %.2f GiB registered in %.1f s",\\n                        self.layer_num, k_shape[0], self._host_storage.numel() / 2**30, time.perf_counter() - started)\\n\\n        def _init_kv_copy_and_warmup(self):\\n            self._kv_copy_config = None  # its warm-up copy treats buffer addresses as device pointers\\n\\n        def _store_kv_layer(self, layer_idx, loc, cache_k, cache_v):\\n            store_rows(cache_k, self.k_buffer[layer_idx], loc)\\n            store_rows(cache_v, self.v_buffer[layer_idx], loc)\\n\\n        def get_kv_size_bytes(self):\\n            return 0, 0  # nothing on the GPU; the hot buffer is reported by QSAHostKV\\n\\n        def move_kv_cache(self, tgt_loc, src_loc):\\n            raise NotImplementedError("QSA host KV does not move KV slots (tree drafting is not supported)")\\n\\n        def get_cpu_copy(self, *args, **kwargs):\\n            raise NotImplementedError("QSA host KV has no CPU copy path")\\n\\n        def load_cpu_copy(self, *args, **kwargs):\\n            raise NotImplementedError("QSA host KV has no CPU copy path")\\n\\n    QSAHostMHATokenToKVPool.__qualname__ = QSAHostMHATokenToKVPool.__name__\\n    return QSAHostMHATokenToKVPool\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# Per-pool state\\n\\n\\nclass QSAHostKV:\\n    """The host K/V of one QSA pool (a QSATokenToKVPool whose full_kv_pool is a host pool) and its GPU hot buffer.\\n    One per model: the target\\\'s 12 QSA layers and, with ARC_SGLANG_QSA_HOST_DRAFT, the MTP drafter\\\'s layer."""\\n\\n    def __init__(self, pool, req_to_token, hot: int, device, name: str = "target", scratch_dtype=None):\\n        import torch\\n\\n        full = pool.full_kv_pool\\n        if full.head_dim != full.v_head_dim:\\n            raise NotImplementedError("QSA host KV needs equal K and V head sizes")\\n        self.name = name\\n        self.layers, self.heads, self.head_dim = full.layer_num, full.head_num, full.head_dim\\n        self.dtype, self.store_dtype = full.dtype, full.store_dtype\\n        self.row_bytes = self.heads * self.head_dim * self.store_dtype.itemsize\\n        self.host_k, self.host_v = full.k_buffer, full.v_buffer\\n        self.req_to_token = req_to_token\\n        self.width = pool.qsa_token_topk + pool.qsa_compress_ratio - 1  # QSA\\\'s row width: blocks x ratio + tail\\n        self.width_cap = self.width + WIDTH_MARGIN\\n        self.max_rows_per_req = pool.qsa_compress_ratio  # QSA allows at most `ratio` MTP rows per request\\n        self.hot = hot\\n        self.device = device\\n        self.rows_per_req = 1\\n        self.forwards = 0\\n        self.num_real = torch.zeros(1, dtype=torch.int32, device=device)\\n        self.stats = torch.zeros(2, dtype=torch.int64, device=device)  # host loads, selected (row, entry) reads\\n        slots, context = req_to_token.shape\\n        self.slots, self.context = slots, context\\n        self._row_buffers = []  # every size ever allocated stays referenced: captured CUDA graphs point into them\\n        if hot:\\n            if hot < self.width_cap:\\n                raise ValueError(f"ARC_SGLANG_QSA_HOT_TOKENS={hot} is below the row width {self.width_cap}")\\n            if hot > 32767:\\n                raise ValueError("ARC_SGLANG_QSA_HOT_TOKENS must fit the swap-in kernel\\\'s int16 LRU slots")\\n            # One allocation for every layer\\\'s hot K and V: no per-tensor rounding of the caching allocator outside\\n            # what pool sizing reserved (fixed_gpu_bytes).\\n            hot_kv = torch.empty((2, self.layers, slots * hot, self.heads, self.head_dim), dtype=self.store_dtype,\\n                                 device=device)\\n            self.hot_k, self.hot_v = list(hot_kv[0].unbind(0)), list(hot_kv[1].unbind(0))\\n            self.tokens = torch.full((self.layers, slots, hot), -1, dtype=torch.int32, device=device)\\n            self._lru_init = torch.arange(hot, dtype=torch.int16, device=device)\\n            self.lru = self._lru_init.repeat(self.layers, slots, 1).contiguous()\\n            self.locs = (torch.arange(slots, dtype=torch.int32, device=device)[:, None] * hot\\n                         + torch.arange(hot, dtype=torch.int32, device=device)[None, :]).contiguous()\\n            self.mirror = torch.zeros((slots, context), dtype=torch.int64, device=device)\\n            self.mirror.copy_(req_to_token)\\n            self._grow(slots * self.max_rows_per_req)\\n        if scratch_dtype is not None:\\n            self.scratch_rows = slots * self.max_rows_per_req * self.width_cap\\n            register_shared_scratch(self.scratch_rows, self.heads, self.head_dim, scratch_dtype, device)\\n        for layer in range(self.layers):\\n            _HOST_TENSORS[self.host_k[layer].data_ptr()] = (self, layer)\\n            _HOST_TENSORS[self.host_v[layer].data_ptr()] = (self, layer)\\n        logger.info("QSA host KV (%s): hot buffer %d entries x %d request slots x %d layers = %.3f GiB GPU (%s)%s",\\n                    name, hot, slots, self.layers, self.gpu_bytes() / 2**30,\\n                    "LRU swap-in" if hot else "every selected entry read from host",\\n                    "" if scratch_dtype is None else\\n                    f"; shared pack scratch {scratch_bytes(self.scratch_rows, self.heads, self.head_dim) / 2**30:.3f} GiB")\\n\\n    @staticmethod\\n    def fixed_gpu_bytes(*, layers, slots, context, hot, row_bytes, width_cap, max_rows_per_req) -> int:\\n        """GPU bytes allocated by __init__ apart from the shared scratch (pool sizing reserves them before dividing the\\n        KV budget): hot K and V, entry keys (int32) and LRU order (int16) per layer, slot ids, the int64 position map,\\n        the per-row swap-in buffers (out int32, miss plan int64 + int32 per entry; 5 small per-row vectors), plus 4 MiB\\n        for the caching allocator\\\'s rounding of the small tensors."""\\n        if not hot:\\n            return 0\\n        per_layer = slots * hot * (2 * row_bytes + 4 + 2)\\n        rows = slots * max_rows_per_req\\n        return layers * per_layer + slots * hot * 4 + slots * context * 8 + rows * (width_cap * 16 + 24) + (4 << 20)\\n\\n    def gpu_bytes(self) -> int:\\n        return self.fixed_gpu_bytes(layers=self.layers, slots=self.slots, context=self.context, hot=self.hot,\\n                                    row_bytes=self.row_bytes, width_cap=self.width_cap,\\n                                    max_rows_per_req=self.max_rows_per_req)\\n\\n    def _grow(self, rows: int) -> None:\\n        import torch\\n\\n        if self._row_buffers and self._row_buffers[-1]["rows"] >= rows:\\n            return\\n        device = self.device\\n        buffers = dict(\\n            rows=rows,\\n            out=torch.full((rows, self.width_cap), -1, dtype=torch.int32, device=device),\\n            miss_src=torch.zeros((rows, self.width_cap), dtype=torch.int64, device=device),\\n            miss_dst=torch.zeros((rows, self.width_cap), dtype=torch.int32, device=device),\\n            miss_count=torch.zeros(rows, dtype=torch.int32, device=device),\\n            fake_seq=torch.full((rows,), _FAKE_SEQ_LEN, dtype=torch.int32, device=device),\\n            real=torch.zeros(rows, dtype=torch.int64, device=device),\\n            arange=torch.arange(rows, dtype=torch.int32, device=device),\\n        )\\n        self._row_buffers.append(buffers)\\n\\n    @property\\n    def buffers(self):\\n        return self._row_buffers[-1]\\n\\n    # -- per forward --------------------------------------------------------------------------------------------------\\n\\n    def begin_forward(self, forward_batch, *, real: int, rows_per_req: int, in_capture: bool) -> None:\\n        """Bookkeeping before a forward of this pool\\\'s model: real request count, query rows per request, the\\n        position->slot map of the positions this forward writes, and dropping hot entries at positions it rewrites\\n        (>= the first position it writes for that request)."""\\n        import torch\\n\\n        mode = forward_batch.forward_mode\\n        self.rows_per_req = rows_per_req\\n        self.num_real.fill_(real)\\n        if not self.hot:\\n            return\\n        self._grow(max(int(forward_batch.batch_size), real) * max(rows_per_req, 1))\\n        buffers = self.buffers\\n        buffers["real"].copy_(buffers["arange"] < self.num_real)\\n        if in_capture or real == 0 or mode.is_idle():\\n            return\\n        self.forwards += 1\\n        rids = forward_batch.req_pool_indices[:real].long()\\n        seq = forward_batch.seq_lens[:real].long()\\n        lru_reset = False\\n        if mode.is_target_verify():  # rows at seq .. seq + k - 1\\n            first = seq\\n            rows, cols = rids[:, None], seq[:, None] + torch.arange(rows_per_req, device=seq.device)[None, :]\\n        elif mode.is_draft_extend_v2():  # the drafter re-reads the verified window: k rows per request, any positions\\n            positions = getattr(forward_batch, "positions", None)\\n            if positions is None:  # the draft-extend graph runner\\\'s metadata view carries them in spec_info\\n                positions = forward_batch.spec_info.positions\\n            positions = positions[0] if positions.ndim == 2 else positions\\n            cols = positions[:real * rows_per_req].long().view(real, rows_per_req)\\n            rows, first = rids[:, None], cols.amin(1)\\n        elif mode.is_decode():  # one new token at seq - 1 (draft decode steps included)\\n            first = seq - 1\\n            rows, cols = rids[:, None], first[:, None]\\n        elif mode.is_extend():  # [prefix, seq) of every request, lengths known on the host\\n            lengths = forward_batch.extend_seq_lens[:real].long()\\n            first = seq - lengths\\n            total = int(sum(forward_batch.extend_seq_lens_cpu[:real]))\\n            owner = torch.repeat_interleave(torch.arange(real, device=seq.device), lengths, output_size=total)\\n            starts = torch.cumsum(lengths, 0) - lengths\\n            rows = rids[owner]\\n            cols = first[owner] + torch.arange(total, device=seq.device) - starts[owner]\\n            lru_reset = True\\n        else:\\n            raise NotImplementedError(f"QSA host KV ({self.name}): unexpected forward mode {mode}")\\n        self.mirror[rows, cols] = self.req_to_token[rows, cols].long()\\n        tokens = self.tokens[:, rids]\\n        self.tokens[:, rids] = tokens.masked_fill(tokens >= first.to(torch.int32).view(1, -1, 1), -1)\\n        if lru_reset:\\n            self.lru[:, rids] = self._lru_init  # any order is valid; this also repairs warm-up races on dummy slots\\n        self._log_stats()\\n\\n    def _log_stats(self) -> None:\\n        every = STATS_EVERY\\n        if every <= 0 or self.forwards % every:\\n            return\\n        loads, selected = (int(x) for x in self.stats.tolist())\\n        self.stats.zero_()\\n        if selected:\\n            logger.info("QSA host KV (%s): last %d forwards: %.1f%% of %d selected entries served from the GPU hot "\\n                        "buffer, %.1f MB read from host per forward", self.name, every, 100.0 * (1 - loads / selected),\\n                        selected, loads * 2 * self.row_bytes / every / 1e6)\\n\\n    # -- pack ---------------------------------------------------------------------------------------------------------\\n\\n    def pack(self, layer, k, v, req_to_token, req_indices, indices, seq_lens, cu_k, out_k, out_v, batch, topk) -> None:\\n        """Replacement of SGLang\\\'s qwen_sparse_kv_extraction_compact_triton for this pool\\\'s layer."""\\n        if not self.hot:\\n            pack_selected(k, v, req_to_token, req_indices, indices, seq_lens, cu_k, out_k, out_v, rows=batch,\\n                          width=topk)\\n            self.stats[1] += cu_k[batch] - cu_k[0]\\n            self.stats[0] += cu_k[batch] - cu_k[0]\\n            return\\n        per_req = self.rows_per_req\\n        if batch % per_req or topk > self.width_cap or indices.dtype.itemsize != 4:\\n            raise ValueError(f"QSA host KV ({self.name}): {batch} rows of width {topk} for {per_req} rows per request "\\n                             f"(at most width {self.width_cap}, int32 positions)")\\n        requests = batch // per_req\\n        self._grow(batch)\\n        buffers = self.buffers\\n        out = buffers["out"][:batch]\\n        miss_count = buffers["miss_count"][:batch].view(per_req, requests)\\n        module = swap_in_module(self.row_bytes, topk, self.hot)\\n        hot_k, hot_v = self.hot_k[layer].view(self.dtype), self.hot_v[layer].view(self.dtype)\\n        for row in range(per_req):  # row j of every request: one LRU state and one kernel block per request\\n            module.load_cache_to_device_buffer(\\n                indices[row::per_req], self.tokens[layer], self.mirror, self.locs, self.host_k[layer],\\n                self.host_v[layer], self.hot_k[layer], self.hot_v[layer], out[row::per_req],\\n                req_indices[row::per_req].contiguous(), buffers["fake_seq"][:requests], self.lru[layer], self.num_real,\\n                1, self.row_bytes, buffers["miss_src"][:requests], buffers["miss_dst"][:requests], miss_count[row])\\n            # Pack this row now: the next row\\\'s call may evict entries this row selected.\\n            pack_selected(hot_k, hot_v, out, out, indices, seq_lens, cu_k, out_k, out_v, rows=requests, row0=row,\\n                          row_step=per_req, by_position=False, width=topk)\\n        real = buffers["real"][:requests]\\n        self.stats[0] += (miss_count.long() * real).sum()\\n        self.stats[1] += ((cu_k[1:batch + 1] - cu_k[:batch]).view(requests, per_req).long() * real[:, None]).sum()\\n\\n    # -- chunked prefill with a prefix --------------------------------------------------------------------------------\\n\\n    def gather_context(self, layer, slots):\\n        """K and V rows of the given slots, copied from host memory into new GPU tensors of the pool dtype."""\\n        import torch\\n\\n        out_k = torch.empty((slots.numel(), self.heads, self.head_dim), dtype=self.store_dtype, device=self.device)\\n        out_v = torch.empty_like(out_k)\\n        gather_rows(self.host_k[layer], slots, out_k)\\n        gather_rows(self.host_v[layer], slots, out_v)\\n        return out_k.view(self.dtype), out_v.view(self.dtype)\\n\\n\\ndef scratch_bytes(rows: int, heads: int, head_dim: int, itemsize: int = 2) -> int:\\n    """Bytes of the shared bf16 pack scratch: K and V for `rows` selected entries."""\\n    return 2 * rows * heads * head_dim * itemsize\\n\\n\\ndef _device_index(device) -> int:\\n    import torch\\n\\n    device = torch.device(device)\\n    if device.type != "cuda" or device.index is None:\\n        return torch.cuda.current_device()\\n    return device.index\\n\\n\\ndef register_shared_scratch(rows: int, heads: int, head_dim: int, dtype, device) -> None:\\n    import torch\\n\\n    key = (heads, head_dim, dtype, _device_index(device))\\n    current = _SHARED_SCRATCH.get(key)\\n    if current is not None and current[0].shape[0] >= rows:\\n        return\\n    shape = (rows, heads, head_dim)\\n    _SHARED_SCRATCH[key] = (torch.empty(shape, dtype=dtype, device=device),\\n                            torch.empty(shape, dtype=dtype, device=device))\\n\\n\\ndef host_kv_of(pool):\\n    return getattr(pool, "qsa_host_kv", None)\\n\\n\\n# --------------------------------------------------------------------------------------------------------------------\\n# SGLang patches\\n\\n\\ndef _geometry(kvc):\\n    """Per-GPU QSA geometry of a KVCacheConfigurator\\\'s model, or None if it has no QSA."""\\n    import torch\\n    from sglang.srt.layers.attention.qsa.config import parse_qsa_profile\\n    from sglang.srt.mem_cache.qsa_kv_pool import QSATokenToKVPool\\n    from sglang.srt.runtime_context import get_parallel, get_schedule\\n\\n    profile = parse_qsa_profile(kvc.model_config.hf_config)\\n    if profile is None:\\n        return None\\n    max_running = get_schedule().max_running_requests\\n    if max_running is None:\\n        raise ValueError("ARC_SGLANG_QSA_HISPARSE=1 needs --max-running-requests (it sizes the GPU hot buffer)")\\n    heads = kvc.model_config.get_num_kv_heads(get_parallel().attn_tp_size, get_parallel().attn_dcp_size)\\n    item = torch._utils._element_size(kvc.kv_cache_dtype)\\n    width_cap = profile.budget + profile.compress_ratio - 1 + WIDTH_MARGIN\\n    hot = hot_tokens()\\n    if hot and hot < width_cap:\\n        raise ValueError(f"ARC_SGLANG_QSA_HOT_TOKENS={hot} is below the row width {width_cap}")\\n    slots = max_running + 1  # request slot 0 is the padding slot\\n    return dict(\\n        kv_layer=heads * (kvc.model_config.head_dim + kvc.model_config.v_head_dim) * item,\\n        index_layer=QSATokenToKVPool.qsa_bytes_per_token(kv_heads=profile.kv_heads, head_dim=profile.head_dim,\\n                                                         compress_ratio=profile.compress_ratio, num_layers=1),\\n        heads=heads, head_dim=kvc.model_config.head_dim,\\n        hot_kwargs=dict(slots=slots, context=kvc.model_config.context_len + 64, hot=hot,\\n                        row_bytes=heads * kvc.model_config.head_dim * item, width_cap=width_cap,\\n                        max_rows_per_req=profile.compress_ratio),\\n        scratch=scratch_bytes(slots * profile.compress_ratio * width_cap, heads, kvc.model_config.head_dim),\\n    )\\n\\n\\ndef target_gpu_plan(kvc, log: bool = True):\\n    """(GPU bytes per token, fixed GPU bytes) of the target\\\'s KV pool with host QSA K/V, or None if not applicable.\\n    Per token the GPU keeps the target\\\'s QSA index, and the drafter\\\'s layer: K/V and index, or only its index with\\n    ARC_SGLANG_QSA_HOST_DRAFT (default). The fixed bytes are the hot buffers QSAHostKV allocates for\\n    --max-running-requests (+1 padding slot) for the target and the host drafter, and the shared pack scratch; pool\\n    sizing takes them from the KV budget before dividing it by the per-token bytes."""\\n    from sglang.srt.configs.hybrid_arch import mambaish_config\\n\\n    if kvc.is_draft_worker:\\n        return None\\n    geometry, mambaish = _geometry(kvc), mambaish_config(kvc.model_config)\\n    if geometry is None or mambaish is None:\\n        return None\\n    layers = sum(1 for i in mambaish.full_attention_layer_ids\\n                 if kvc.layer_info.start_layer <= i < kvc.layer_info.end_layer)\\n    draft_layers = 0\\n    if kvc.spec_algorithm.is_eagle() or kvc.spec_algorithm.is_standalone():\\n        draft_layers = int(kvc.spec_aux_config.eagle_draft_num_layers or 0)\\n    index, kv = geometry["index_layer"], geometry["kv_layer"]\\n    on_host = draft_layers and host_draft()\\n    cell = layers * index + draft_layers * (index if on_host else kv + index)\\n    target_fixed = QSAHostKV.fixed_gpu_bytes(layers=layers, **geometry["hot_kwargs"])\\n    draft_fixed = QSAHostKV.fixed_gpu_bytes(layers=draft_layers, **geometry["hot_kwargs"]) if on_host else 0\\n    fixed = target_fixed + draft_fixed + geometry["scratch"]\\n    if log:\\n        logger.info("QSA host KV: GPU bytes per token %d (QSA index of %d layers + %d drafter layers%s), host bytes per "\\n                    "token %d; fixed GPU %.3f GiB = hot buffer %.3f (target) + %.3f (drafter) + shared pack scratch "\\n                    "%.3f", cell, layers, draft_layers, " on host" if on_host else "",\\n                    layers * kv + (draft_layers * kv if on_host else 0), fixed / 2**30, target_fixed / 2**30,\\n                    draft_fixed / 2**30, geometry["scratch"] / 2**30)\\n    return cell, fixed\\n\\n\\ndef patch_pool_configurator(module) -> None:\\n    cls = module.DefaultPoolConfigurator\\n    init, calculate = cls.__init__, cls.calculate_pool_sizes\\n\\n    def __init__(self, kvc):\\n        init(self, kvc)\\n        plan = target_gpu_plan(kvc)\\n        if plan is not None:\\n            self._cell_size, self._qsa_host_fixed_bytes = plan\\n\\n    def calculate_pool_sizes(self, available_bytes, page_size):\\n        fixed = getattr(self, "_qsa_host_fixed_bytes", None)\\n        if fixed is None:\\n            return calculate(self, available_bytes, page_size)\\n        config = calculate(self, available_bytes - fixed, page_size)\\n        logger.info("QSA host KV: KV budget %.3f GiB - fixed %.3f GiB = %.3f GiB at %d B/token -> %d tokens",\\n                    available_bytes / 2**30, fixed / 2**30, (available_bytes - fixed) / 2**30, self._cell_size,\\n                    config.max_total_num_tokens)\\n        return config\\n\\n    cls.__init__, cls.calculate_pool_sizes = __init__, calculate_pool_sizes\\n\\n\\ndef patch_kv_cache_configurator(module) -> None:\\n    cls = module.KVCacheConfigurator\\n    build = cls._build_hybrid_linear_kv_pool\\n\\n    def _build_hybrid_linear_kv_pool(self, *, max_total_num_tokens, req_to_token_pool, mha_pool_class):\\n        import torch\\n\\n        geometry = _geometry(self)\\n        if geometry is None or (self.is_draft_worker and not host_draft()):\\n            return build(self, max_total_num_tokens=max_total_num_tokens, req_to_token_pool=req_to_token_pool,\\n                         mha_pool_class=mha_pool_class)\\n        host_class = host_pool_class(mha_pool_class)\\n        pool = build(self, max_total_num_tokens=max_total_num_tokens, req_to_token_pool=req_to_token_pool,\\n                     mha_pool_class=host_class)\\n        if not isinstance(pool.full_kv_pool, host_class):\\n            raise RuntimeError(f"QSA host KV: SGLang built {type(pool.full_kv_pool).__name__} instead of the host pool")\\n        name = "drafter" if self.is_draft_worker else "target"\\n        scratch_dtype = None if self.is_draft_worker else getattr(self, "model_dtype", None) or torch.bfloat16\\n        allocated, reserved = torch.cuda.memory_allocated(), torch.cuda.memory_reserved()\\n        pool.qsa_host_kv = QSAHostKV(pool, req_to_token_pool.req_to_token, hot_tokens(), self.device, name=name,\\n                                     scratch_dtype=scratch_dtype)\\n        allocated, reserved = torch.cuda.memory_allocated() - allocated, torch.cuda.memory_reserved() - reserved\\n        planned = QSAHostKV.fixed_gpu_bytes(layers=pool.full_kv_pool.layer_num, **geometry["hot_kwargs"])\\n        planned += 0 if self.is_draft_worker else geometry["scratch"]\\n        (logger.warning if allocated > planned else logger.info)(\\n            "QSA host KV (%s): hot buffer, maps%s allocated %.3f GiB (caching allocator +%.3f GiB), reserved in pool "\\n            "sizing %.3f GiB", name, "" if self.is_draft_worker else " and pack scratch", allocated / 2**30,\\n            reserved / 2**30, planned / 2**30)\\n        return pool\\n\\n    cls._build_hybrid_linear_kv_pool = _build_hybrid_linear_kv_pool\\n\\n\\ndef _rows_per_request(forward_batch) -> int:\\n    mode = forward_batch.forward_mode\\n    if mode.is_target_verify():\\n        return int(forward_batch.spec_info.draft_token_num)\\n    if mode.is_draft_extend_v2():  # EAGLE v2 draft extend: a constant num_tokens_per_req rows per request\\n        return int(forward_batch.spec_info.num_tokens_per_req)\\n    return 1\\n\\n\\ndef patch_qsa_backend(module) -> None:\\n    compact = module.qwen_sparse_kv_extraction_compact_triton\\n\\n    def qwen_sparse_kv_extraction_compact_triton(k, v, req_to_token, req_indices, indices, seq_lens, cu_k, out_k,\\n                                                 out_v, batch, topk, zero_fill_cols: int = 0):\\n        owner = _HOST_TENSORS.get(k.data_ptr())\\n        if owner is None:\\n            return compact(k, v, req_to_token, req_indices, indices, seq_lens, cu_k, out_k, out_v, batch, topk,\\n                           zero_fill_cols=zero_fill_cols)\\n        if zero_fill_cols:\\n            raise NotImplementedError("QSA host KV does not support the trtllm (page-strided) sparse decode path")\\n        state, layer = owner\\n        state.pack(layer, k, v, req_to_token, req_indices, indices, seq_lens, cu_k, out_k, out_v, batch, topk)\\n\\n    module.qwen_sparse_kv_extraction_compact_triton = qwen_sparse_kv_extraction_compact_triton\\n    cls = module.QwenSparseAttnBackend\\n    get_scratch, forward_extend = cls._get_fa2_scratch, cls.forward_extend\\n    init_metadata, init_metadata_out_graph = cls.init_forward_metadata, cls.init_forward_metadata_out_graph\\n\\n    def _get_fa2_scratch(self, capacity, num_kv_heads, head_dim, dtype, device):\\n        import torch\\n\\n        if device.type == "cpu":  # k_buffer.device of a host pool: the scratch belongs on the GPU\\n            device = torch.device("cuda", torch.cuda.current_device())\\n        shared = _SHARED_SCRATCH.get((num_kv_heads, head_dim, dtype, _device_index(device)))\\n        if shared is not None and shared[0].shape[0] >= capacity:\\n            return shared[0][:capacity], shared[1][:capacity]\\n        if shared is not None:\\n            logger.warning("QSA host KV: a pack scratch of %d rows exceeds the shared one (%d rows); allocating it "\\n                           "outside the pool-sizing reservation", capacity, shared[0].shape[0])\\n        return get_scratch(self, capacity, num_kv_heads, head_dim, dtype, device)\\n\\n    def init_forward_metadata(self, forward_batch):\\n        init_metadata(self, forward_batch)\\n        state = host_kv_of(self.token_to_kv_pool)\\n        if state is not None:\\n            state.begin_forward(forward_batch, real=int(forward_batch.batch_size),\\n                                rows_per_req=_rows_per_request(forward_batch), in_capture=False)\\n\\n    def init_forward_metadata_out_graph(self, forward_batch, in_capture: bool = False):\\n        init_metadata_out_graph(self, forward_batch, in_capture=in_capture)\\n        state = host_kv_of(self.token_to_kv_pool)\\n        if state is not None:\\n            padding = 0 if in_capture else int(getattr(forward_batch, "num_padding", None) or 0)\\n            real = int(forward_batch.batch_size) - padding\\n            raw = None if in_capture else getattr(getattr(forward_batch, "spec_info", None), "hisparse_raw_bs", None)\\n            state.begin_forward(forward_batch, real=real if raw is None else min(real, int(raw)),\\n                                rows_per_req=_rows_per_request(forward_batch), in_capture=in_capture)\\n\\n    def forward_extend_host(self, q, k, v, layer, forward_batch, save_kv_cache=True, topk_indices=None, **kwargs):\\n        """forward_extend\\\'s chunked-prefill branch (context with a prefix) with the context read from host memory."""\\n        import torch\\n        import torch.nn.functional as F\\n\\n        state = host_kv_of(self.token_to_kv_pool)\\n        if (state is None or topk_indices is None or not q.is_cuda\\n                or self._is_speculative_paged_mode(forward_batch.forward_mode)):\\n            return forward_extend(self, q, k, v, layer, forward_batch, save_kv_cache=save_kv_cache,\\n                                  topk_indices=topk_indices, **kwargs)\\n        extend_lens = [int(x) for x in forward_batch.extend_seq_lens_cpu]\\n        sequence_lens = [int(x) for x in forward_batch.seq_lens_cpu]\\n        if not any(s - e for s, e in zip(sequence_lens, extend_lens)):  # no prefix: attention reads only this chunk\\n            return forward_extend(self, q, k, v, layer, forward_batch, save_kv_cache=save_kv_cache,\\n                                  topk_indices=topk_indices, **kwargs)\\n        if save_kv_cache:\\n            self.token_to_kv_pool.set_kv_buffer(layer, forward_batch.out_cache_loc, k, v)\\n        q = q.reshape(-1, layer.tp_q_head_num, layer.head_dim)\\n        num_output_rows, num_valid_rows = q.shape[0], topk_indices.shape[0]\\n        if num_valid_rows > num_output_rows:\\n            raise ValueError(f"QSA top-k rows exceed query rows: topk={num_valid_rows}, query={num_output_rows}")\\n        q = q[:num_valid_rows]\\n        topk_indices = topk_indices.to(torch.int32).contiguous()\\n        cu_seqlens_q = F.pad(forward_batch.extend_seq_lens.to(q.device, dtype=torch.int32).cumsum(0), (1, 0)).contiguous()\\n        req_indices = forward_batch.req_pool_indices.tolist()\\n        req_to_token = self.req_to_token_pool.req_to_token\\n        slots = torch.cat([req_to_token[req_indices[i], :sequence_lens[i]] for i in range(len(sequence_lens))])\\n        layer_index = self.token_to_kv_pool._transfer_full_attention_id(layer.layer_id)\\n        k_full, v_full = state.gather_context(layer_index, slots.long())\\n        sequence_lens_tensor = torch.tensor(sequence_lens, dtype=torch.int32, device=q.device)\\n        cu_seqlens_k = F.pad(sequence_lens_tensor.cumsum(0), (1, 0)).contiguous()\\n        output = module.sparse_gqa_fwd_interface_triton_ck(q.contiguous(), k_full, v_full, topk_indices, cu_seqlens_q,\\n                                                           cu_seqlens_k, sequence_lens_tensor, layer.scaling)\\n        return self._pad_extend_output(output, num_output_rows)\\n\\n    cls._get_fa2_scratch = _get_fa2_scratch\\n    cls.init_forward_metadata = init_forward_metadata\\n    cls.init_forward_metadata_out_graph = init_forward_metadata_out_graph\\n    cls.forward_extend = forward_extend_host\\n\\n    # The MTP draft decode steps: in CUDA-graph replay the multi-step backend calls its step backends\\\'\\n    # _capture/_replay_cuda_graph_metadata directly, past init_forward_metadata_out_graph, so its own entry point\\n    # runs the bookkeeping of every step (eager steps go through init_forward_metadata above).\\n    steps = module.QwenSparseMultiStepDraftBackend\\n    steps_out_graph = steps.init_forward_metadata_out_graph\\n\\n    def steps_init_forward_metadata_out_graph(self, forward_batch, in_capture: bool = False):\\n        steps_out_graph(self, forward_batch, in_capture=in_capture)\\n        state = host_kv_of(self.attn_backends[0].token_to_kv_pool) if self.attn_backends else None\\n        if state is None:\\n            return\\n        padding = 0 if in_capture else int(getattr(forward_batch, "num_padding", None) or 0)\\n        for step in range(len(self.attn_backends)):\\n            step_batch = self._make_step_forward_batch(forward_batch, step, num_padding=padding)\\n            state.begin_forward(step_batch, real=int(forward_batch.batch_size) - padding, rows_per_req=1,\\n                                in_capture=in_capture)\\n\\n    steps.init_forward_metadata_out_graph = steps_init_forward_metadata_out_graph\\n\\n\\ndef free_memory_after_capture() -> None:\\n    """Log (and warn below ARC_SGLANG_MIN_FREE_GIB, default 0.5) the GPU memory left for activations after every\\n    CUDA graph is captured: what CUDA reports free plus what PyTorch holds cached but unallocated."""\\n    import torch\\n\\n    free, _ = torch.cuda.mem_get_info()\\n    cached = torch.cuda.memory_reserved() - torch.cuda.memory_allocated()\\n    usable = (free + cached) / 2**30\\n    threshold = min_free_gib()\\n    (logger.warning if usable < threshold else logger.info)(\\n        "QSA host KV: after CUDA graph capture %.2f GiB free + %.2f GiB cached by PyTorch = %.2f GiB for activations "\\n        "(ARC_SGLANG_MIN_FREE_GIB=%.2f)%s", free / 2**30, cached / 2**30, usable, threshold,\\n        "; prefill or decode may run out of memory: lower --mem-fraction-static (0.01 frees ~1% of the GPU) or "\\n        "--chunked-prefill-size" if usable < threshold else "")\\n\\n\\ndef patch_scheduler(module) -> None:\\n    cls = module.Scheduler\\n    init_graphs = cls.init_all_cuda_graphs\\n\\n    def init_all_cuda_graphs(self):\\n        init_graphs(self)\\n        free_memory_after_capture()\\n\\n    cls.init_all_cuda_graphs = init_all_cuda_graphs\\n\\n\\ndef patch_draft_extend_runner(module) -> None:\\n    """The draft-extend CUDA graph runner hands the attention backend a metadata view without positions or a padding\\n    count, and fills spec_info.positions only for padded batches; the host-KV bookkeeping needs the window and the real\\n    request count either way, so for the duration of execute an unpadded batch lends its own positions to spec_info\\n    and spec_info.hisparse_raw_bs carries the batch size before padding to a captured size."""\\n    runner = module.EAGLEDraftExtendCudaGraphRunner\\n    execute = runner.execute\\n\\n    def execute_with_positions(self, forward_batch, *args, **kwargs):\\n        spec = getattr(forward_batch, "spec_info", None)\\n        if spec is None:\\n            return execute(self, forward_batch, *args, **kwargs)\\n        lent = getattr(spec, "positions", None) is None and getattr(forward_batch, "positions", None) is not None\\n        if lent:\\n            spec.positions = forward_batch.positions\\n        spec.hisparse_raw_bs = forward_batch.batch_size\\n        try:\\n            return execute(self, forward_batch, *args, **kwargs)\\n        finally:\\n            if lent:\\n                spec.positions = None\\n            spec.hisparse_raw_bs = None\\n\\n    runner.execute = execute_with_positions\\n\\n\\nPATCHES = {\\n    "sglang.srt.speculative.eagle_draft_extend_cuda_graph_runner": patch_draft_extend_runner,\\n    "sglang.srt.model_executor.pool_configurator": patch_pool_configurator,\\n    "sglang.srt.mem_cache.kv_cache_configurator": patch_kv_cache_configurator,\\n    "sglang.srt.layers.attention.qwen_sparse_attn_backend": patch_qsa_backend,\\n    "sglang.srt.managers.scheduler": patch_scheduler,\\n}\\n\\n\\ndef register(patches: dict) -> None:\\n    """Add this module\\\'s import hooks to sitecustomize\\\'s table, after any patch already registered for a module."""\\n    for name, apply in PATCHES.items():\\n        previous = patches.get(name)\\n        patches[name] = apply if previous is None else (lambda module, a=previous, b=apply: (a(module), b(module)))\\n\', \'src/qwen/fnweights.py\': \'"""Weights for training Qwen3.8-Flash-Next against the deployed model: the bf16 training directory whose routed experts are\\nthe deployed int4 values, the override file trained tensors go back in, and its loader.\\n\\nThe deployed routed experts are Intel\\\'s int4 GPTQ tensors (AutoRound, symmetric, zero point 8). Training keeps them\\nfrozen at the values SGLang\\\'s Marlin MoE multiplies with, bf16(q - 8) * bf16(scale), so build writes the kept experts\\ndequantized as per-expert bf16 .weight tensors renumbered from 0 (src.qwen.reap\\\'s order), slices the routers to the kept\\nrows, drops mtp.*, and declares the kept expert count without a quantization_config; transformers\\\' qwen4_exp loader\\nfuses the per-expert tensors. transformers reads every tensor of a shard, not only the index\\\'s, so a shard is linked\\nonly when all its tensors are kept as they are, and the kept tensors of the other shards are copied.\\n\\nThe trained tensors go back as one override.safetensors under their checkpoint names and source dtypes (routers with\\ntheir kept rows) with the sha256 of the keep list they were trained under. patch_weight_override substitutes them while\\na vLLM (h95) server loads (ARC_VLLM_WEIGHT_OVERRIDE); patched before src/qwen/vllm_patch\\\'s REAP keep patch, it sees the pruned, renumbered stream, so\\nthe int4 files, the Marlin kernels and the keep list stay as deployed.\\n    python -m src.qwen.fnweights keep --stats STATS.rank0 ... [--mtp-stats OLD.rank0 ...] --keep 256 --output KEEP.json\\n    python -m src.qwen.fnweights build --source INTEL_DIR --keep KEEP.json --output TRAIN_DIR [--ngram-from DIR]\\n    python -m src.qwen.fnweights export --checkpoint CHECKPOINT_DIR --source INTEL_DIR --keep KEEP.json --output OVERRIDE\\n    python -m src.qwen.fnweights export --checkpoint RUN/hf/step_N --base TRAIN_DIR --source INTEL_DIR --keep KEEP.json --output OVERRIDE\\n"""\\nimport argparse\\nimport json\\nimport os\\nimport re\\nimport sys\\nfrom contextlib import ExitStack\\nfrom pathlib import Path\\n\\nimport torch\\n\\nfrom src.qwen.flashnext import link_others\\nfrom src.qwen.reap import DRAFT, EXPERT, ROUTER, TARGET, keep_sets, keep_sha256, saliency\\nfrom src.utils.tensors import safetensors_header\\n\\nTABLE = re.compile(r"\\\\.ngram_embedding\\\\.shard_\\\\d+\\\\.weight$")\\nFLOAT_DTYPES = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32}\\nSYMMETRIC_ZEROS = 0x77777777  # eight 4-bit zero points of 7: GPTQ v1 stores zero point 8 as 8 - 1\\nPROJECTIONS = ("gate_proj", "up_proj", "down_proj")\\n\\n\\ndef read_keep(path: Path) -> dict[tuple[str, int], list[int]]:\\n    """A keep list {"model.language_model.layers.<i>" | "mtp.layers.0": [expert ids]} keyed (prefix, layer)."""\\n    return {(key.rsplit(".", 1)[0], int(key.rsplit(".", 1)[1])): experts for key, experts in json.loads(Path(path).read_text()).items()}\\n\\n\\ndef keep_list(stats: list[Path], keep: int, mtp_stats: list[Path] | None = None, target_layers: int = 48) -> dict[str, list[int]]:\\n    """REAP keep list from router statistics; the MTP layer\\\'s entry comes from mtp_stats when given (a scoring server\\n    without the drafter records no MTP statistics)."""\\n    scores = saliency(stats, target_layers)\\n    if mtp_stats:\\n        scores = {key: value for key, value in scores.items() if key[0] != DRAFT}\\n        scores.update({key: value for key, value in saliency(mtp_stats, target_layers).items() if key[0] == DRAFT})\\n    if not any(prefix == DRAFT for prefix, _ in scores):\\n        raise ValueError("no statistics for the MTP layer: pass --mtp-stats")\\n    return {f"{prefix}.{layer}": experts for (prefix, layer), experts in sorted(keep_sets(scores, keep).items())}\\n\\n\\ndef unpack_int4(qweight: torch.Tensor) -> torch.Tensor:\\n    """GPTQ int32 [in/8, out] -> uint8 codes [in, out]; row 8j+i is bits 4i..4i+3 of packed row j."""\\n    shifts = torch.arange(0, 32, 4, dtype=torch.int32, device=qweight.device)\\n    return ((qweight.unsqueeze(1) >> shifts[None, :, None]) & 0xF).reshape(-1, qweight.shape[1]).to(torch.uint8)\\n\\n\\ndef dequantize(qweight: torch.Tensor, scales: torch.Tensor, qzeros: torch.Tensor, group_size: int = 128) -> torch.Tensor:\\n    """[out, in] bf16 = bf16(q - 8) * bf16(scale), the product SGLang\\\'s Marlin MoE computes with our bf16 scales;\\n    raises unless every qzeros nibble is 7 (symmetric)."""\\n    if not torch.equal(qzeros, torch.full_like(qzeros, SYMMETRIC_ZEROS)):\\n        raise ValueError("asymmetric qzeros: only symmetric int4 (zero point 8) is supported")\\n    codes = unpack_int4(qweight).to(torch.int16) - 8\\n    return (codes.to(torch.bfloat16) * scales.to(torch.bfloat16).repeat_interleave(group_size, 0)).t().contiguous()\\n\\n\\ndef build_training_dir(source: Path, keep: Path, output: Path, ngram_from: Path | None = None) -> dict:\\n    """Kept experts as per-expert bf16 .weight tensors (renumbered from 0), routers sliced, other tensors linked or\\n    copied, mtp.* dropped, config num_experts = K without quantization_config, the keep list copied to reap_keep.json (the\\n    trainer refuses to export under another list). ngram_from replaces the n-gram table shards with the ones in that\\n    directory. Returns counts and bytes written."""\\n    from safetensors import safe_open\\n    from safetensors.torch import save_file\\n\\n    kept = read_keep(keep)\\n    config = json.loads((source / "config.json").read_text())\\n    quantization = config.pop("quantization_config")\\n    if quantization.get("bits") != 4 or not quantization.get("sym"):\\n        raise ValueError(f"expected symmetric int4 experts, got {quantization}")\\n    group = quantization["group_size"]\\n    weight_map = json.loads((source / "model.safetensors.index.json").read_text())["weight_map"]\\n    if any(name.endswith(".g_idx") for name in weight_map):\\n        raise ValueError("act-order g_idx tensors are not supported")\\n    moe_layers = {int(match.group(2)) for match in map(ROUTER.match, weight_map) if match and match.group(1) == TARGET}\\n    if moe_layers != {layer for prefix, layer in kept if prefix == TARGET}:\\n        raise ValueError("the keep list does not cover exactly the checkpoint\\\'s MoE layers")\\n    tables = {}\\n    if ngram_from is not None:\\n        for path in sorted(ngram_from.glob("*.safetensors")):\\n            tables.update({name: path for name in safetensors_header(path)[0] if TABLE.search(name)})\\n\\n    def unchanged(name: str) -> bool:\\n        return not (EXPERT.match(name) or ROUTER.match(name) or name.startswith("mtp.") or (tables and TABLE.search(name)))\\n\\n    output.mkdir(parents=True, exist_ok=False)\\n    (output / "reap_keep.json").write_text(Path(keep).read_text())\\n    new_map, report = {}, {"linked": [], "copied": [], "bytes_written": 0}\\n    with ExitStack() as stack:\\n        handles = {}\\n\\n        def tensor(file: str, name: str) -> torch.Tensor:\\n            if file not in handles:\\n                handles[file] = stack.enter_context(safe_open(str(source / file), framework="pt"))\\n            return handles[file].get_tensor(name)\\n\\n        def write(tensors: dict, file: str) -> None:\\n            save_file(tensors, str(output / file), metadata={"format": "pt"})\\n            new_map.update(dict.fromkeys(tensors, file))\\n            report["bytes_written"] += (output / file).stat().st_size\\n\\n        for file in sorted(set(weight_map.values())):\\n            names = list(safetensors_header(source / file)[0])\\n            wanted = [name for name in names if unchanged(name)]\\n            if wanted and len(wanted) == len(names):\\n                os.symlink(os.path.realpath(source / file), output / file)\\n                new_map.update(dict.fromkeys(wanted, file))\\n                report["linked"].append(file)\\n            elif wanted:\\n                write({name: tensor(file, name) for name in wanted}, f"kept-{file}")\\n                report["copied"].append(file)\\n        routers = {}\\n        for name, file in weight_map.items():\\n            match = ROUTER.match(name)\\n            if match and match.group(1) == TARGET:\\n                routers[name] = tensor(file, name)[torch.tensor(kept[(TARGET, int(match.group(2)))])].contiguous()\\n        write(routers, "routers.safetensors")\\n        for (prefix, layer), experts in sorted(kept.items()):\\n            if prefix != TARGET:\\n                continue\\n            tensors = {}\\n            for new, old in enumerate(experts):\\n                for projection in PROJECTIONS:\\n                    base = f"{prefix}.{layer}.mlp.experts.{old}.{projection}"\\n                    parts = [tensor(weight_map[f"{base}.{part}"], f"{base}.{part}") for part in ("qweight", "scales", "qzeros")]\\n                    tensors[f"{prefix}.{layer}.mlp.experts.{new}.{projection}.weight"] = dequantize(*parts, group_size=group)\\n            write(tensors, f"experts-{layer:03d}.safetensors")\\n    for name, path in tables.items():\\n        if not (output / path.name).exists():\\n            os.symlink(os.path.realpath(path), output / path.name)\\n        new_map[name] = path.name\\n    counts = {len(experts) for experts in kept.values()}\\n    if len(counts) != 1:\\n        raise ValueError(f"layers keep different expert counts: {counts}")\\n    config.get("text_config", config)["num_experts"] = report["num_experts"] = counts.pop()\\n    (output / "config.json").write_text(json.dumps(config, indent=2))\\n    (output / "model.safetensors.index.json").write_text(json.dumps({"metadata": {}, "weight_map": new_map}))\\n    link_others(source, output, {"config.json", "model.safetensors.index.json", "quantization_config.json"})\\n    return report\\n\\n\\ndef trained_tensors(checkpoint: Path, base: Path | None = None) -> tuple[dict, dict]:\\n    """(name -> trained tensor, {"step", "tokens"}) from checkpoint/masters.safetensors + state.json, or from a Miles\\n    --save-hf directory (.../step_<n>), where the trained tensors are those that differ from base, the training directory\\n    the run started from (routed experts and the n-gram table are frozen and skipped)."""\\n    from safetensors import safe_open\\n\\n    if (checkpoint / "masters.safetensors").exists():\\n        state = json.loads((checkpoint / "state.json").read_text())\\n        with safe_open(str(checkpoint / "masters.safetensors"), framework="pt") as masters:\\n            return {name: masters.get_tensor(name) for name in masters.keys()}, {"step": state["step"], "tokens": state["tokens"]}\\n    if base is None:\\n        raise ValueError(f"{checkpoint} has no masters.safetensors; a Miles --save-hf directory needs base")\\n    base_map = json.loads((base / "model.safetensors.index.json").read_text())["weight_map"]\\n    changed = {}\\n    with ExitStack() as stack:\\n        references = {}\\n        for path in sorted(checkpoint.glob("*.safetensors")):\\n            stream = stack.enter_context(safe_open(str(path), framework="pt"))\\n            for name in stream.keys():\\n                if ".mlp.experts." in name or TABLE.search(name):\\n                    continue\\n                file = base_map[name]\\n                if file not in references:\\n                    references[file] = stack.enter_context(safe_open(str(base / file), framework="pt"))\\n                old, new = references[file].get_tensor(name), stream.get_tensor(name)\\n                if not torch.equal(new.to(old.dtype), old):\\n                    changed[name] = new\\n    step = re.search(r"step_(\\\\d+)$", checkpoint.name)\\n    return changed, {"step": int(step.group(1)) if step else 0, "tokens": 0}\\n\\n\\ndef export_override(checkpoint: Path, source: Path, keep: Path, output: Path, base: Path | None = None) -> dict:\\n    """override.safetensors of the trained tensors (trained_tensors) under checkpoint names and source dtypes; shapes\\n    checked against the source headers (router rows = kept count); metadata reap_keep_sha256, num_experts, step, tokens."""\\n    trained, state = trained_tensors(checkpoint, base)\\n    return write_override(trained, source, keep, output, state["step"], state["tokens"])\\n\\n\\ndef write_override(trained: dict, source: Path, keep: Path, output: Path, step: int = 0, tokens: int = 0) -> dict:\\n    """export_override\\\'s file from {checkpoint name: tensor} (archive.rl.hfrl.train\\\'s parameters as well)."""\\n    from safetensors.torch import save_file\\n\\n    kept = read_keep(keep)\\n    weight_map = json.loads((source / "model.safetensors.index.json").read_text())["weight_map"]\\n    headers, tensors = {}, {}\\n    for name, value in trained.items():\\n        if name not in weight_map:\\n            raise KeyError(f"{name} is not a tensor of {source}")\\n        file = weight_map[name]\\n        entry = (headers[file] if file in headers else headers.setdefault(file, safetensors_header(source / file)[0]))[name]\\n        if entry["dtype"] not in FLOAT_DTYPES:\\n            raise TypeError(f"{name} is {entry[\\\'dtype\\\']} in {source}, not a trainable float tensor")\\n        shape, router = list(entry["shape"]), ROUTER.match(name)\\n        if router:\\n            shape[0] = len(kept[(router.group(1), int(router.group(2)))])\\n        if list(value.shape) != shape:\\n            raise ValueError(f"{name}: trained shape {list(value.shape)}, source shape {shape}")\\n        tensors[name] = value.to(FLOAT_DTYPES[entry["dtype"]]).contiguous()\\n    counts = {len(experts) for experts in kept.values()}\\n    metadata = {"reap_keep_sha256": keep_sha256(keep), "num_experts": str(max(counts)), "step": str(step), "tokens": str(tokens)}\\n    save_file(tensors, str(output), metadata=metadata)\\n    return {"tensors": len(tensors), "bytes": output.stat().st_size, **metadata}\\n\\n\\ndef patch_weight_override(module, path: str, class_name: str, keep: str | None = None) -> None:\\n    """class_name\\\'s load_weights takes every tensor of the override file in place of the checkpoint\\\'s tensor of the same\\n    name, same shape and dtype; raises when a load stream holding a tensor of a module the override touches misses one\\n    of its tensors (a stream without any, such as Merlin\\\'s PLE offload worker\\\'s, which loads only PLE weights through\\n    load_weights, is another loader\\\'s and passes untouched), or when the file was trained under another REAP keep list\\n    than keep (a keep list of the same size but other experts would load silently into a wrong model). Patch before src/qwen/vllm_patch\\\'s patch_reap_keep, which then wraps this loader and hands it\\n    the pruned stream (renumbered experts, routers already sliced)."""\\n    from safetensors import safe_open\\n\\n    with safe_open(path, framework="pt") as stream:\\n        names, metadata = set(stream.keys()), stream.metadata() or {}\\n    expected, actual = metadata.get("reap_keep_sha256"), keep_sha256(Path(keep)) if keep else None\\n    if expected != actual:\\n        raise ValueError(f"{path} was trained under REAP keep list sha256 {expected}, the server prunes with {actual}")\\n    cls = getattr(module, class_name)\\n    original = cls.load_weights\\n    modules = {name.rsplit(".", 2)[0] for name in names} | {name.rsplit(".", 1)[0] for name in names}\\n\\n    def load_weights(self, weights):\\n        seen, touched = set(), []\\n        with safe_open(path, framework="pt") as stream:\\n            def substituted():\\n                for name, tensor in weights:\\n                    if not touched and (name.rsplit(".", 1)[0] in modules or name.rsplit(".", 2)[0] in modules):\\n                        touched.append(name)\\n                    if name in names:\\n                        new = stream.get_tensor(name)\\n                        if new.shape != tensor.shape or new.dtype != tensor.dtype:\\n                            raise ValueError(f"override {name} is {new.dtype} {tuple(new.shape)}, the checkpoint loads "\\n                                             f"{tensor.dtype} {tuple(tensor.shape)}")\\n                        seen.add(name)\\n                        tensor = new\\n                    yield name, tensor\\n\\n            result = original(self, substituted())\\n        if names - seen and (seen or touched):\\n            raise ValueError(f"{len(names - seen)} override tensors never reached the loader, e.g. {sorted(names - seen)[:3]}")\\n        # one line per load_weights call (every TP / PP worker; the PLE offload worker\\\'s call shows 0), so a server log\\n        # proves the substitution reached each worker\\n        print(f"[weight_override] pid {os.getpid()}: substituted {len(seen)} of {len(names)} tensors from {path}",\\n              file=sys.stderr, flush=True)\\n        return result\\n\\n    cls.load_weights = load_weights\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    commands = parser.add_subparsers(dest="command", required=True)\\n    keep = commands.add_parser("keep", help="write a REAP keep list from router statistics (no pruned files)")\\n    keep.add_argument("--stats", type=Path, nargs="+", required=True, help="ARC_SGLANG_ROUTER_STATS files, one per TP rank")\\n    keep.add_argument("--mtp-stats", type=Path, nargs="+", help="statistics files that hold the MTP layer\\\'s saliency")\\n    keep.add_argument("--keep", type=int, required=True, help="experts kept per MoE layer")\\n    keep.add_argument("--target-layers", type=int, default=48, help="TopK modules of the target before the MTP layer")\\n    keep.add_argument("--output", type=Path, required=True)\\n    build = commands.add_parser("build", help="bf16 training directory with the kept experts dequantized")\\n    build.add_argument("--source", type=Path, required=True, help="Intel\\\'s Qwen3.8-Flash-Next-W4A16-AutoRound files")\\n    build.add_argument("--keep", type=Path, required=True, help="REAP keep list")\\n    build.add_argument("--output", type=Path, required=True)\\n    build.add_argument("--ngram-from", type=Path, help="directory whose n-gram table shards replace the source\\\'s")\\n    export = commands.add_parser("export", help="override.safetensors from a training checkpoint")\\n    export.add_argument("--checkpoint", type=Path, required=True,\\n                        help="checkpoint directory (masters.safetensors, state.json), or a Miles --save-hf step_<n> directory")\\n    export.add_argument("--base", type=Path, help="with a Miles directory: the training directory the run started from")\\n    export.add_argument("--source", type=Path, required=True, help="the full checkpoint the server loads (names, dtypes, shapes)")\\n    export.add_argument("--keep", type=Path, required=True, help="REAP keep list the run trained under")\\n    export.add_argument("--output", type=Path, required=True)\\n    args = parser.parse_args()\\n    if args.command == "keep":\\n        args.output.write_text(json.dumps(keep_list(args.stats, args.keep, args.mtp_stats, args.target_layers)))\\n        print(json.dumps({"output": str(args.output)}))\\n    elif args.command == "build":\\n        print(json.dumps(build_training_dir(args.source, args.keep, args.output, args.ngram_from)))\\n    else:\\n        print(json.dumps(export_override(args.checkpoint, args.source, args.keep, args.output, args.base)))\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\', \'src/qwen/vllm_patch/vllm_read_filter.py\': \'"""Checkpoint reads of Flash-Next GPU workers on the halt95 Merlin engine (vLLM 0.30.0 + its patch series).\\n\\nARC_VLLM_READ_FILTER=1: a GPU worker skips, before reading them, the target\\\'s tensors it would read and then drop: the\\nlayers of the other pipeline stage, the PLE table\\\'s shards (serve_l4.sh\\\'s pull mode: the PLE offload worker loads the\\ntable, a GPU worker keeps only its FP8 scale), the experts REAP pruned (ARC_VLLM_REAP_KEEP) and, under expert\\nparallelism, the other EP rank\\\'s experts, counted after REAP\\\'s renumbering; Merlin\\\'s PLE offload worker reads only the\\ntensors named like a PLE module\\\'s (".ple.", a superset of its own prefix filter, which still refuses a load that misses\\na PLE parameter). vLLM\\\'s safetensors iterator calls should_skip_weight before get_tensor but skips only through\\nlocal_expert_ids (enable_ep_weight_filter, off by default), which counts the checkpoint\\\'s own expert ids, wrong once\\nREAP renumbers them. Without the filter every process reads the whole checkpoint (116 GiB for Flash-Next REAP448) in the\\nsame file order and the page cache serves the ones that read a file after the first; with it the four GPU workers\\n(~18 GiB each) and the PLE worker (48 GiB) read disjoint parts at the same time. The drafter\\\'s tensors (named mtp.*) are\\nnever filtered. With ARC_VLLM_WEIGHT_OVERRIDE a GPU worker still reads every checkpoint tensor the override file\\nreplaces (src.qwen.fnweights requires a stream that touches an overridden module to carry all of them; ~0.9 GB for\\nAZ-Focus\\\'s 50).\\n\\nARC_VLLM_WEIGHT_DIGEST=1: every GPU worker logs, after each model\\\'s load_weights, a digest of its parameters and\\nbuffers, how many tensors the filter skipped, and its /proc/self/io read_bytes, so a start with the filter can be\\ncompared with one without (equal digests: the same weights were loaded).\\n"""\\nimport json\\nimport re\\nimport sys\\n\\nLAYER = re.compile(r"model\\\\.language_model\\\\.layers\\\\.(\\\\d+)\\\\.")\\nEXPERT = re.compile(r"(model\\\\.language_model\\\\.layers)\\\\.(\\\\d+)\\\\.mlp\\\\.experts\\\\.(\\\\d+)\\\\.")\\nPLE_TABLE = re.compile(r"\\\\.ple\\\\.ple_embedding\\\\.ngram_embedding\\\\.shard_\\\\d+\\\\.weight$")\\n# as vLLM\\\'s own EP filter: only the heavy tensors of the other EP rank\\\'s experts, never their scales\\nHEAVY = (".qweight", ".weight", ".weight_packed")\\nTARGET = "Qwen4ExpForConditionalGeneration"\\nSKIPPED = [0]\\n\\n\\ndef rule(start: int, end: int, local: set[int] | None, positions: dict | None, exempt: frozenset = frozenset()):\\n    """name -> True to skip it: [start, end) the stage\\\'s layers; local the EP rank\\\'s experts (kept indices) or None for\\n    all; positions {(prefix, layer): {checkpoint expert: kept index}} from REAP\\\'s keep list, or None without REAP;\\n    exempt names are never skipped."""\\n    def skip(name: str) -> bool:\\n        if name in exempt:\\n            return False\\n        layer = LAYER.match(name)\\n        if layer is None:\\n            return False\\n        if not start <= int(layer.group(1)) < end or PLE_TABLE.search(name):\\n            return True\\n        expert = EXPERT.match(name)\\n        if expert is None:\\n            return False\\n        number = int(expert.group(3))\\n        if positions is not None:\\n            number = positions.get((expert.group(1), int(expert.group(2))), {}).get(number)\\n            if number is None:  # pruned\\n                return True\\n        return local is not None and number not in local and name.endswith(HEAVY)\\n\\n    return skip\\n\\n\\ndef ple_only(name: str) -> bool:\\n    """The PLE offload worker\\\'s rule: skip every tensor not named like a PLE module\\\'s."""\\n    return ".ple." not in name\\n\\n\\ndef keep_positions(path: str) -> dict:\\n    with open(path) as stream:\\n        kept = json.load(stream)\\n    return {(key.rsplit(".", 1)[0], int(key.rsplit(".", 1)[1])): {expert: new for new, expert in enumerate(experts)}\\n            for key, experts in kept.items()}\\n\\n\\ndef current_rule(keep: str | None):\\n    """The rule of this process for the model vLLM is loading: ple_only in the PLE offload worker, rule(...) in a GPU\\n    worker loading the target, None (nothing skipped) otherwise."""\\n    from vllm.config import get_current_vllm_config\\n    from vllm.model_executor.layers.ple_offload_layer import is_offload_process\\n\\n    if is_offload_process():\\n        print("read filter: PLE offload worker, PLE tensors only", file=sys.stderr, flush=True)\\n        return ple_only\\n    config = get_current_vllm_config()\\n    if TARGET not in (config.model_config.hf_config.architectures or []):\\n        return None\\n    from vllm.distributed import get_pp_group, get_tensor_model_parallel_rank\\n    from vllm.distributed.utils import get_pp_indices\\n    from vllm.model_executor.model_loader.ep_weight_filter import compute_local_expert_ids\\n\\n    parallel = config.parallel_config\\n    pp = get_pp_group()\\n    start, end = get_pp_indices(config.model_config.hf_text_config.num_hidden_layers, pp.rank_in_group, pp.world_size)\\n    local = None\\n    if parallel.enable_expert_parallel and not parallel.enable_eplb and parallel.data_parallel_size == 1 \\\\\\n            and parallel.prefill_context_parallel_size == 1:  # ep_rank = tp_rank, as DefaultModelLoader._init_ep_weight_filter\\n        local = compute_local_expert_ids(config.model_config.get_num_experts(), parallel.tensor_parallel_size,\\n                                         get_tensor_model_parallel_rank(), placement=parallel.expert_placement_strategy)\\n    print(f"read filter: layers [{start}, {end}), {len(local) if local is not None else \\\'all\\\'} local experts, "\\n          f"REAP keep {\\\'on\\\' if keep else \\\'off\\\'}", file=sys.stderr, flush=True)\\n    return rule(start, end, local, keep_positions(keep) if keep else None, override_names())\\n\\n\\ndef override_names() -> frozenset:\\n    """The tensor names of ARC_VLLM_WEIGHT_OVERRIDE\\\'s file (empty without one)."""\\n    import os\\n\\n    path = os.environ.get("ARC_VLLM_WEIGHT_OVERRIDE")\\n    if not path:\\n        return frozenset()\\n    from safetensors import safe_open\\n\\n    with safe_open(path, framework="pt") as stream:\\n        return frozenset(stream.keys())\\n\\n\\ndef patch_skip(module, keep: str | None) -> None:\\n    """Wraps vllm.model_executor.model_loader.weight_utils.should_skip_weight (looked up there at each call)."""\\n    original, rules = module.should_skip_weight, {}\\n\\n    def should_skip_weight(weight_name, local_expert_ids):\\n        if original(weight_name, local_expert_ids):\\n            return True\\n        from vllm.config import get_current_vllm_config\\n\\n        key = id(get_current_vllm_config().model_config)\\n        if key not in rules:\\n            rules[key] = current_rule(keep)\\n        if rules[key] is not None and rules[key](weight_name):\\n            SKIPPED[0] += 1\\n            return True\\n        return False\\n\\n    module.should_skip_weight = should_skip_weight\\n\\n\\ndef digest(model) -> tuple[str, int]:\\n    """sha256 over every parameter\\\'s and buffer\\\'s name, shape and int32-word sum (tail bytes summed apart)."""\\n    import hashlib\\n\\n    import torch\\n\\n    hasher, count = hashlib.sha256(), 0\\n    for name, tensor in [*model.named_parameters(), *model.named_buffers()]:\\n        if tensor.is_meta:\\n            continue\\n        data = tensor.detach().reshape(-1).view(torch.uint8)\\n        body = data.numel() // 4 * 4\\n        try:\\n            words = data[:body].view(torch.int32)\\n        except RuntimeError:  # a view at a storage offset not divisible by 4\\n            words = data[:body].clone().view(torch.int32)\\n        hasher.update(f"{name}:{tuple(tensor.shape)}:{tensor.dtype}:{int(words.sum(dtype=torch.int64))}:"\\n                      f"{int(data[body:].sum(dtype=torch.int64))};".encode())\\n        count += 1\\n    return hasher.hexdigest(), count\\n\\n\\ndef patch_digest(module) -> None:\\n    loader = module.DefaultModelLoader\\n    original = loader.load_weights\\n\\n    def load_weights(self, model, model_config):\\n        result = original(self, model, model_config)\\n        try:  # a diagnostic: it never stops a start\\n            from vllm.distributed import get_pp_group, get_tensor_model_parallel_rank\\n\\n            value, count = digest(model)\\n            with open("/proc/self/io") as stream:\\n                io = dict(line.split(": ") for line in stream.read().splitlines())\\n            print(f"weight digest {type(model).__name__} pp{get_pp_group().rank_in_group} tp{get_tensor_model_parallel_rank()}: "\\n                  f"{value} over {count} tensors; filter skipped {SKIPPED[0]}; read_bytes {int(io[\\\'read_bytes\\\']) / 2**30:.2f} GiB, "\\n                  f"rchar {int(io[\\\'rchar\\\']) / 2**30:.2f} GiB", file=sys.stderr, flush=True)\\n        except Exception as error:  # noqa: BLE001\\n            print(f"weight digest failed: {type(error).__name__}: {error}", file=sys.stderr, flush=True)\\n        return result\\n\\n    loader.load_weights = load_weights\\n\\n\\ndef patches(read_filter: bool, keep: str | None, weight_digest: bool) -> dict:\\n    """{module name: hook} for sitecustomize\\\'s import hook."""\\n    hooks = {}\\n    if read_filter:\\n        hooks["vllm.model_executor.model_loader.weight_utils"] = lambda module: patch_skip(module, keep)\\n    if weight_digest:\\n        hooks["vllm.model_executor.model_loader.default_loader"] = patch_digest\\n    return hooks\\n\', \'src/agent/__init__.py\': \'"""Code-generating ARC solver, verifier, and training utilities."""\\n\', \'src/agent/loop.py\': \'"""The agent loop: one trajectory is render -> generate -> parse -> tool reply or final answer -> next round.\\n\\ntrajectory() runs it against a server and writes every request, response and tool execution of a round under\\nROOT/<key>/; replay_trajectory() runs the same steps again from those files without a server and must reproduce each one\\nbefore the trajectory\\\'s prediction counts.\\n\\nagent_mode "code" (default): the model must return a transform(grid) program; the first program that passes the\\ntraining pairs and returns a grid for every test input is the prediction. agent_mode "free": the direct-answer prompt\\nwith an optional tool; the model may run programs on the training pairs (and see their test outputs), and its final\\nmessage\\\'s grids (src.utils.prompt.answer_grids) are the prediction, never a program\\\'s output.\\n\\nTool "f3" (apply_f3_turn) adds arc_submit_answer: the answer named by the session function or variable that computed it\\n(a plain-text answer still counts); the result also keeps the last passing program\\\'s test outputs when they differ\\nfrom the prediction (program_predictions), and the context can end a trajectory early (final_round). Its result marks\\na trajectory that ended in a final round (forced) and one whose final round the context brake alone caused (brake).\\n"""\\n\\nimport contextlib\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport re\\nfrom threading import Lock\\nimport time\\n\\nfrom src.utils.prompt import answer_grids\\nfrom src.utils.grid import View, apply_task, invert_grid\\nfrom .executor import grid_problem\\nfrom .model_inventory import digest\\nfrom .prompt import (ARC_EXECUTOR_TOOL, ARC_SUBMIT_TOOL, ARC_SUBMIT_TOOL_PERMISSIVE, F3_PYTHON_TOOL, F3_PYTHON_TOOL_BARE, FREE_PYTHON_TOOL, FREE_TOOL,\\n                     SECOND_HYPOTHESIS, SECOND_HYPOTHESIS_EDGES, free_tool, initial_messages)\\nfrom .protocol import (collapse_think_end, decode_native, parse_assistant, reasoning_tokens, render_messages, tool_arguments,\\n                       validate_response)\\nfrom .tools import evaluate_message, grid_difference, run_f3_tool, run_free_tool, session_entry, submit_answer\\n\\n\\nRESULT_LOG_LOCK = Lock()\\n\\n\\ndef write_json(path: Path, value) -> None:\\n    with path.open("x") as stream:\\n        json.dump(value, stream, indent=2)\\n\\n\\nclass TokenBudgetSpent(Exception):\\n    """The trajectory\\\'s total_tokens allowance leaves nothing for another round."""\\n\\n\\ndef request_payload(tokenizer, messages: list[dict], model: str, settings: dict, seed: int,\\n                    first_prompt_tokens: int | None = None, thinking_used: int = 0, prefill: str | None = None,\\n                    continuation: tuple[str, list[int]] | None = None, allowance: tuple[int, int] | None = None,\\n                    cap: int | None = None) -> dict:\\n    """With settings total_tokens (the RL agent loop\\\'s response_length), every token after the first round\\\'s prompt\\n    (generated and tool feedback) counts against one allowance; a round may generate at most the remainder. With\\n    agent_mode "free", thinking_token_budget and max_tokens belong to the whole trajectory: earlier rounds\\\' reasoning\\n    tokens (thinking_used) come off both, tool feedback does not, and a round asks for at most what the context has left.\\n    With thinking_scope "round" they are each round\\\'s caps instead, and the context bounds the trajectory: a round thinks\\n    at most what the context leaves after the answer room (max_tokens - thinking_token_budget).\\n    prefill (final_prefill): text appended after the generation prompt, which closes the reasoning, so the round carries\\n    no thinking budget. continuation (settings token_continuity): the previous round\\\'s (text, ids), its prompt and the tokens\\n    it generated; when this round\\\'s rendering starts with that text, the prompt is those ids plus the rest encoded, so the\\n    server sees the tokens it sampled (an encoding of the same text can split them otherwise and fork the cached prefix);\\n    payload["token_continuity"] says whether it held (the request leaves it out). allowance (start, cap), a second\\n    hypothesis\\\'s rounds: every token after a prompt of start tokens (None: this round\\\'s prompt) counts against cap, and a\\n    round thinks at most what leaves FINAL_ANSWER_RESERVE of it."""\\n    free = settings.get("agent_mode") == "free"\\n    per_round = free and thinking_scope(settings) == "round"\\n    text, ids = render_messages(tokenizer, messages, settings.get("earlier_reasoning", "keep"), tool_spec(settings),\\n                                settings.get("reasoning_effort"))\\n    kept = None\\n    if continuation is not None:\\n        previous_text, previous_ids = continuation\\n        kept = False\\n        if text.startswith(previous_text):\\n            joined = previous_ids + tokenizer.encode(text[len(previous_text):], add_special_tokens=False)\\n            if decode_native(tokenizer, joined) == text:\\n                ids, kept = joined, True\\n    if prefill:\\n        text, ids = text + prefill, ids + tokenizer.encode(prefill, add_special_tokens=False)\\n    max_tokens, total = settings["max_tokens"], settings.get("total_tokens")\\n    budget = None if prefill else settings.get("thinking_token_budget")\\n    if free and budget is not None:\\n        budget = (max(0, min(budget, settings["max_model_len"] - len(ids) - answer_room(settings) - capped_margin(settings)))\\n                  if per_round else max(0, budget - thinking_used))\\n    if free:\\n        max_tokens = min(max_tokens - (0 if per_round else thinking_used), settings["max_model_len"] - len(ids))\\n        if max_tokens <= 0:\\n            raise TokenBudgetSpent\\n    if cap is not None:  # round_clamp: what one request can still generate before the deadline, never below the answer room\\n        max_tokens = min(max_tokens, max(cap, answer_room(settings)))\\n        if budget is not None:\\n            budget = max(0, min(budget, max_tokens - answer_room(settings)))\\n    if total is not None:\\n        max_tokens = min(max_tokens, total - (len(ids) - (len(ids) if first_prompt_tokens is None else first_prompt_tokens)))\\n        if max_tokens <= 0:\\n            raise TokenBudgetSpent\\n    if allowance is not None:  # start None: this prompt is the start\\n        max_tokens = min(max_tokens, allowance[1] - (len(ids) - (len(ids) if allowance[0] is None else allowance[0])))\\n        if max_tokens <= 0:\\n            raise TokenBudgetSpent\\n        if budget is not None:\\n            budget = max(0, min(budget, max_tokens - FINAL_ANSWER_RESERVE))\\n    if len(ids) + max_tokens > settings["max_model_len"]:\\n        raise ValueError("Full conversation and generation allowance exceed the context")\\n    if budget is not None and (type(budget) is not int or budget < 0 or (total is None and not free and budget >= max_tokens)):\\n        raise ValueError("Thinking token budget must leave room for the program")\\n    return {\\n        "model": model, "prompt": ids, "n": 1, "temperature": settings["temperature"],\\n        "top_p": settings["top_p"], "max_tokens": max_tokens, "seed": seed,\\n        "extra_body": {\\n            "top_k": settings["top_k"], "return_token_ids": True, "add_special_tokens": False,\\n            "skip_special_tokens": False, "include_stop_str_in_output": True,\\n            "spaces_between_special_tokens": False, "stop_token_ids": tokenizer.screen_terminal_ids,\\n            **({"thinking_token_budget": budget} if budget is not None else {}),\\n            **({"min_p": settings["min_p"]} if "min_p" in settings else {}),\\n        },\\n        "prompt_sha256": hashlib.sha256(text.encode()).hexdigest(),\\n        **({"token_continuity": kept} if kept is not None else {}),\\n    }\\n\\n\\ndef apply_turn(body, task, index, timeout):\\n    if body is None:\\n        return {"status": "TOKEN_LIMIT", "assistant": None, "evaluations": [], "replies": [], "prediction": None}\\n    try:\\n        message = parse_assistant(body)\\n    except ValueError as error:\\n        return {"status": "INVALID_FORMAT", "assistant": None, "evaluations": [], "replies": [],\\n                "prediction": None, "error": str(error)}\\n    evaluations, replies, prediction = evaluate_message(message, task, index, timeout)\\n    return {"status": "VERIFIED" if prediction is not None else "REVISE", "assistant": message,\\n            "evaluations": evaluations, "replies": replies, "prediction": prediction}\\n\\n\\nFINAL_ANSWER = {"role": "user", "content": "The tool budget is used up: do not call the tool again. Give your final "\\n                                            "answer now, the output grid for each test input in the required format."}\\n# tool "f3" once a program has passed every training pair: the final round may name it in arc_submit_answer, so its\\n# prefill only closes the reasoning (THINK_END)\\nF3_FINAL_ANSWER = {"role": "user", "content": "The tool budget is used up: do not call arc_python_executor again. Give "\\n                                               "your final answer now: call arc_submit_answer with the function or "\\n                                               "variable that computed it, or write the output grid for each test input "\\n                                               "in the required format."}\\nFINAL_ANSWER_RESERVE = 2048\\nMODEL_CONTEXT = 131072  # the deployed server\\\'s max_model_len: a lower settings max_model_len is a cap (capped_margin)\\nF3_CONTEXT_BRAKE = 98304  # tool "f3": context_brake\\\'s default\\nNOTICE_AFTER = 65536  # tool "f3" with budget_notice: tool replies report the context beyond this many tokens\\nTHINK_CLOSED = "<think>\\\\n</think>\\\\n"\\nTHINK_END = "\\\\n</think>\\\\n\\\\n"\\n\\n\\ndef final_prefill(task: dict) -> str:\\n    """What the final free round starts with after the generation prompt: the reasoning closed and the first grid opened\\n    (after the first output\\\'s label when the task has several test inputs), so the round answers instead of calling the\\n    tool again (most trajectories without a grid had ignored FINAL_ANSWER and called it)."""\\n    return "\\\\n</think>\\\\n\\\\n" + ("--Test Output 1--\\\\n" if len(task["test"]) > 1 else "") + "[["\\n\\n\\ndef absorb_prefill(body, prefill: str):\\n    """The final round\\\'s assistant body with the prefilled answer start put back in front of the generated answer."""\\n    if body is None or not body.startswith(THINK_CLOSED):\\n        return body\\n    return THINK_CLOSED + prefill.split("</think>\\\\n\\\\n", 1)[1] + body[len(THINK_CLOSED):]\\n\\n\\ndef tool_spec(settings: dict) -> dict | list[dict]:\\n    """The tool schema the model sees: agent_mode free\\\'s FREE_TOOL, FREE_PYTHON_TOOL with tool "free", or with tool "f3"\\n    F3_PYTHON_TOOL (F3_PYTHON_TOOL_BARE with helpers False) and ARC_SUBMIT_TOOL (ARC_SUBMIT_TOOL_PERMISSIVE with\\n    permissive). context_brake (a positive int or None), budget_notice, helpers (default True: objects / diff\\n    predefined) and permissive (default False: F3\\\' wording) apply to tool "f3" only."""\\n    if settings.get("tool") != "f3" and {"context_brake", "budget_notice", "helpers", "permissive"} & set(settings):\\n        raise ValueError("context_brake, budget_notice, helpers and permissive apply to tool f3")\\n    brake = settings.get("context_brake")\\n    if brake is not None and (type(brake) is not int or brake <= 0):\\n        raise ValueError(f"context_brake must be a positive int or None, not {brake!r}")\\n    if settings.get("agent_mode") != "free":\\n        return ARC_EXECUTOR_TOOL\\n    tool = free_tool(settings.get("tool", "transform"))\\n    return [F3_PYTHON_TOOL if settings.get("helpers", True) else F3_PYTHON_TOOL_BARE,\\n            ARC_SUBMIT_TOOL_PERMISSIVE if settings.get("permissive") else ARC_SUBMIT_TOOL] if tool == "f3" else FREE_PYTHON_TOOL if tool == "free" else FREE_TOOL\\n\\n\\ndef thinking_scope(settings: dict) -> str:\\n    """agent_mode free: "trajectory" (default; one thinking_token_budget and max_tokens shared by every round) or "round"\\n    (each round\\\'s caps, the context bounding the whole trajectory)."""\\n    scope = settings.get("thinking_scope", "trajectory")\\n    if scope not in ("trajectory", "round"):\\n        raise ValueError(f"Unknown thinking_scope {scope!r}")\\n    return scope\\n\\n\\ndef answer_room(settings: dict) -> int:\\n    return settings["max_tokens"] - settings["thinking_token_budget"]\\n\\n\\ndef capped_margin(settings: dict) -> int:\\n    """A context cap below the model\\\'s (MODEL_CONTEXT; a capped later attempt, src.qwen.runtime verified_later_attempts):\\n    the final round comes this much earlier and a round before it thinks this much less, so the round\\\'s tool turn cannot\\n    use up the room the final round answers in (at the full context the margin is 0 and nothing changes)."""\\n    return answer_room(settings) + FINAL_ANSWER_RESERVE if settings["max_model_len"] < MODEL_CONTEXT else 0\\n\\n\\ndef context_tokens(tokenizer, messages: list[dict], settings: dict) -> int | None:\\n    """The rendered conversation\\\'s length, which decides the final round under thinking_scope "round" and with tool "f3"\\n    (else None)."""\\n    if settings.get("agent_mode") != "free" or (thinking_scope(settings) != "round" and settings.get("tool") != "f3"):\\n        return None\\n    return len(render_messages(tokenizer, messages, settings.get("earlier_reasoning", "keep"), tool_spec(settings),\\n                               settings.get("reasoning_effort"))[1])\\n\\n\\ndef final_round(settings: dict, turn_index: int, thinking_used: int, context: int | None = None, passed: bool = False) -> bool:\\n    """agent_mode free: the round that must answer (start_round appends the final-answer message): the last allowed\\n    round, or the first one with less than FINAL_ANSWER_RESERVE thinking tokens left (thinking_scope "round": the first\\n    one whose context (context_tokens) leaves less than the answer room plus FINAL_ANSWER_RESERVE). Tool "f3" (context\\n    counted in either scope) also ends at that context in thinking_scope "trajectory", and while no program has passed\\n    every training pair (passed) at the first round whose context reaches context_brake (default F3_CONTEXT_BRAKE; None\\n    turns it off)."""\\n    if settings.get("agent_mode") != "free":\\n        return False\\n    if turn_index == settings["max_rounds"] - 1:\\n        return True\\n    short = context is not None and (settings["max_model_len"] - context\\n                                     < answer_room(settings) + FINAL_ANSWER_RESERVE + capped_margin(settings))\\n    if settings.get("tool") == "f3" and (short or braking(settings, context, passed)):\\n        return True\\n    if thinking_scope(settings) == "round":\\n        return short\\n    return settings["thinking_token_budget"] - thinking_used < FINAL_ANSWER_RESERVE\\n\\n\\ndef braking(settings: dict, context: int | None, passed: bool) -> bool:\\n    """Tool "f3"\\\'s context brake: no program has passed every training pair (passed) and the context has reached\\n    context_brake (default F3_CONTEXT_BRAKE; None turns it off)."""\\n    brake = settings.get("context_brake", F3_CONTEXT_BRAKE)\\n    return settings.get("tool") == "f3" and brake is not None and not passed and context is not None and context >= brake\\n\\n\\ndef round_clamp(settings: dict, now: float) -> int | None:\\n    """settings round_clamp_at and round_clamp_rate (src.qwen.runtime: the generation deadline less round_clamp seconds,\\n    and one request\\\'s generated tokens per second when the job started): the tokens a round started at `now` can still\\n    generate (None without them). A round whose clamp leaves no more than the answer room is the final round."""\\n    at, rate = settings.get("round_clamp_at"), settings.get("round_clamp_rate")\\n    return None if at is None or not rate else max(0, int((at - now) * rate))\\n\\n\\ndef start_round(settings: dict, task: dict, tokenizer, messages: list[dict], turn_index: int, thinking_used: int,\\n                turns: list[dict], force: bool = False) -> tuple[bool, str | None, dict | None]:\\n    """(final, prefill, brake) of the next round (final_round, or force: the deadline brake); a final round\\\'s message is\\n    appended to messages: FINAL_ANSWER with final_prefill, or with tool "f3" once a program has passed every training pair\\n    F3_FINAL_ANSWER with THINK_END. brake: {"round" (the round\\\'s index, as its files are numbered), "context" (tokens)} when\\n    the context brake made the round final and it would not have been otherwise, else None."""\\n    passed = last_passing(turns, None) is not None\\n    context = context_tokens(tokenizer, messages, settings)\\n    if not (force or final_round(settings, turn_index, thinking_used, context, passed)):\\n        return False, None, None\\n    brake = None\\n    if braking(settings, context, passed) and not final_round({**settings, "context_brake": None}, turn_index, thinking_used,\\n                                                               context, passed):\\n        brake = {"round": turn_index, "context": context}\\n    submit = settings.get("tool") == "f3" and passed\\n    messages.append(F3_FINAL_ANSWER if submit else FINAL_ANSWER)\\n    return True, THINK_END if submit else final_prefill(task), brake\\n\\n\\ndef passing_outputs(calls: list[dict], view) -> list | None:\\n    """The test outputs of the last of these free-mode tool calls whose program passed every training pair and returned\\n    a grid for every test input, mapped back from the job\\\'s view."""\\n    for call in reversed(calls):\\n        outputs = call.get("test_outputs")\\n        if call.get("train_passed") == call.get("train_total") and outputs and all(grid is not None for grid in outputs):\\n            return [invert_grid(grid, view) if view else grid for grid in outputs]\\n    return None\\n\\n\\ndef last_passing(turns: list[dict], view) -> list | None:\\n    """passing_outputs over every tool call of the trajectory\\\'s turns."""\\n    return passing_outputs([call for turn in turns for call in turn.get("calls") or []], view)\\n\\n\\ndef program_fallback(turns: list[dict], view) -> list | None:\\n    """The test outputs of the last free-mode tool call whose program passed every training pair, mapped back from the\\n    job\\\'s view, or the readable grids of a later answer that was returned for correction: the prediction of a trajectory\\n    that ends without grids (the model has seen these outputs). After a second_hypothesis turn: that turn\\\'s answer, the\\n    first, whatever its continuation recorded."""\\n    marked = next((turn for turn in turns if turn.get("second_hypothesis")), None)\\n    if marked is not None:\\n        return marked["prediction"]\\n    for turn in reversed(turns):\\n        if any(grid is not None for grid in turn.get("prediction") or []):  # an answer returned for correction (tool free)\\n            return turn["prediction"]\\n        outputs = passing_outputs(turn.get("calls") or [], view)\\n        if outputs is not None:\\n            return outputs\\n    return None\\n\\n\\ndef program_predictions(settings: dict, turns: list[dict], view, prediction) -> list | None:\\n    """tool "f3", whatever the trajectory\\\'s status: the last passing program\\\'s test outputs (last_passing) when they\\n    differ from its prediction, kept with it so the submission can hold both (src.qwen.runtime.finalize,\\n    src.agent.evaluate.aggregate); else None."""\\n    if settings.get("tool") != "f3":\\n        return None\\n    outputs = last_passing(turns, view)\\n    return outputs if outputs is not None and outputs != prediction else None\\n\\n\\ndef verified(turns: list[dict], view, prediction) -> bool:\\n    """Whether every grid of the prediction is a test output of a program that passed every training pair (a tool call or\\n    an f3 function of the trajectory, mapped back from the job\\\'s view): the answer is a verified program\\\'s\\n    (src.qwen.runtime\\\'s JobQueue runs the later attempts of these tasks last)."""\\n    if not prediction or any(grid is None for grid in prediction):\\n        return False\\n    runs = [run for turn in turns for call in turn.get("calls") or [] for run in [call, *(call.get("functions") or {}).values()]]\\n    passing = [[invert_grid(grid, view) if view and grid is not None else grid for grid in run["test_outputs"]] for run in runs\\n               if run.get("train_total") and run.get("train_passed") == run["train_total"] and run.get("test_outputs")]\\n    return all(any(i < len(outputs) and outputs[i] == grid for outputs in passing) for i, grid in enumerate(prediction))\\n\\n\\nSESSION_NAMES = {"train", "test", "checks", "objects", "diff"}  # what the f3 session predefines (executor._f3_session)\\n\\n\\ndef free_names(tree) -> set[str]:\\n    """Names a module loads and does not bind itself (builtins and the session\\\'s predefined names left out)."""\\n    import ast\\n    import builtins\\n\\n    bound, loaded = set(), set()\\n    for node in ast.walk(tree):\\n        if isinstance(node, ast.Name):\\n            (loaded if isinstance(node.ctx, ast.Load) else bound).add(node.id)\\n        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\\n            bound.add(node.name)\\n        elif isinstance(node, ast.arg):\\n            bound.add(node.arg)\\n        elif isinstance(node, (ast.Import, ast.ImportFrom)):\\n            bound.update((alias.asname or alias.name).split(".")[0] for alias in node.names)\\n    return loaded - bound - set(dir(builtins)) - SESSION_NAMES\\n\\n\\ndef defining_statements(code: str, earlier: list[str]) -> str:\\n    """The latest top-level definitions (def, class, import, assignment) in earlier calls\\\' code of the names code needs\\n    and does not define, taken transitively, in the order they ran; "" when nothing is missing or found."""\\n    import ast\\n\\n    try:\\n        needed = free_names(ast.parse(code))\\n    except SyntaxError:\\n        return ""\\n    found = {}  # name -> (call index, line, source)\\n    while needed:\\n        name = needed.pop()\\n        for index in range(len(earlier) - 1, -1, -1):\\n            try:\\n                tree = ast.parse(earlier[index])\\n            except SyntaxError:\\n                continue\\n            hit = None\\n            for node in tree.body:\\n                names = ({node.name} if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) else\\n                         {(a.asname or a.name).split(".")[0] for a in node.names} if isinstance(node, (ast.Import, ast.ImportFrom)) else\\n                         {t.id for t in getattr(node, "targets", [getattr(node, "target", None)]) if isinstance(t, ast.Name)}\\n                         if isinstance(node, (ast.Assign, ast.AnnAssign)) else set())\\n                if name in names:\\n                    hit = node  # the last one in this call wins\\n            if hit is not None:\\n                key = (index, hit.lineno)\\n                if key not in {v[:2] for v in found.values()}:\\n                    found[name] = (index, hit.lineno, ast.get_source_segment(earlier[index], hit))\\n                    needed |= free_names(hit) - set(found)\\n                break\\n    parts = sorted({v for v in found.values() if v[2]})\\n    return "".join(f"{source}\\\\n" for _, _, source in parts)\\n\\n\\ndef previous_program(directory: Path, task: dict, slice_definitions: bool = False) -> dict | None:\\n    """F4 (settings "previous_program"): what a later attempt is shown of an earlier free-mode trajectory (its\\n    NN_execution.json records, in the task\\\'s own orientation) beside its answer: the code of the last executor call\\n    that defines a function the answer came from, else of the last call that ran to the end, else of the last call,\\n    with that call\\\'s train_passed and its check on each training pair (grid_difference). None without an executor call.\\n    slice_definitions (settings previous_program_slice): that code is preceded by the earlier calls\\\' definitions it\\n    uses (defining_statements, from calls that ran to the end), so a call that calls helpers defined turns before is\\n    shown whole."""\\n    turns = [json.loads(path.read_text()) for path in sorted(directory.glob("[0-9][0-9]_execution.json"))]\\n    calls = [call for turn in turns for call in turn.get("calls") or [] if call.get("code")]\\n    if not calls:\\n        return None\\n    sources = next((turn["sources"] for turn in reversed(turns) if turn.get("sources")), None) or []\\n    names = {source.split(":", 1)[1] for source in sources if source.startswith("function:")}\\n    call = (next((c for c in reversed(calls) if names & set(c.get("functions") or {})), None)\\n            or next((c for c in reversed(calls) if c.get("ok")), calls[-1]))\\n    outputs = call.get("train_outputs") or [None] * len(task["train"])\\n    error = call.get("error") or call.get("script_error")\\n    report = [f"Training pair {number}: {grid_difference(pair[\\\'output\\\'], actual, error)}"\\n              for number, (pair, actual) in enumerate(zip(task["train"], outputs), 1)]\\n    code = call["code"]\\n    if slice_definitions:\\n        earlier = [c["code"] for c in calls[:calls.index(call)] if c.get("ok")]\\n        prefix = defining_statements(code, earlier)\\n        code = f"# defined in earlier calls\\\\n{prefix}\\\\n{code}" if prefix else code\\n    return {"code": code, "train_passed": call.get("train_passed") or 0, "train_total": len(task["train"]),\\n            "report": report}\\n\\n\\nSECOND_HYPOTHESIS_TOKENS = 12288  # settings second_hypothesis_tokens\\\' default: the continuation\\\'s allowance\\n\\n\\ndef second_hypothesis_turn(settings: dict, turns: list[dict], turn: dict, view, final: bool) -> dict:\\n    """settings "second_hypothesis" (tool "f3"): a first answer every grid of which came from a program passing every\\n    training pair (verified) does not end the trajectory: the turn goes on as REVISE, marked second_hypothesis, with\\n    SECOND_HYPOTHESIS as the reply (the submit call\\\'s tool reply, or a user message after a plain-text answer), and the\\n    same conversation looks for a second such program (second_result); settings second_hypothesis_prompt "edges" sends\\n    SECOND_HYPOTHESIS_EDGES instead (the test input\\\'s cases no training input decides). Not in a final round, once per\\n    trajectory, and\\n    not after a message whose submit call came with other tool calls (their replies are not kept)."""\\n    calls = (turn.get("assistant") or {}).get("tool_calls") or []\\n    if not (settings.get("second_hypothesis") and settings.get("tool") == "f3" and turn["status"] == "ANSWERED" and not final\\n            and len(calls) <= 1 and not any(earlier.get("second_hypothesis") for earlier in turns)\\n            and verified([*turns, turn], view, turn["prediction"])):\\n        return turn\\n    return {**turn, "status": "REVISE", "second_hypothesis": True,\\n            "replies": [{"role": "tool" if calls else "user",\\n                         "content": SECOND_HYPOTHESIS_EDGES if settings.get("second_hypothesis_prompt") == "edges" else SECOND_HYPOTHESIS}]}\\n\\n\\ndef second_result(turns: list[dict], view) -> dict | None:\\n    """A trajectory with a second_hypothesis turn (None without one): {"mark": its index, "prediction": its answer (the\\n    first, kept whatever the continuation did), "second": the continuation\\\'s answer when every grid of it came from a\\n    program passing every training pair and it differs from the first, else the test outputs of the continuation\\\'s last\\n    program passing every training pair that differ from the first, else None}."""\\n    mark = next((index for index, turn in enumerate(turns) if turn.get("second_hypothesis")), None)\\n    if mark is None:\\n        return None\\n    first, last = turns[mark]["prediction"], turns[-1]\\n    if mark < len(turns) - 1 and last["status"] == "ANSWERED" and last["prediction"] != first and verified(turns, view, last["prediction"]):\\n        return {"mark": mark, "prediction": first, "second": last["prediction"]}\\n    runs = [run for turn in turns[mark + 1:] for call in turn.get("calls") or [] for run in [call, *(call.get("functions") or {}).values()]]\\n    for run in reversed(runs):\\n        outputs = run.get("test_outputs")\\n        if run.get("train_total") and run.get("train_passed") == run["train_total"] and outputs and all(grid is not None for grid in outputs):\\n            mapped = [invert_grid(grid, view) if view else grid for grid in outputs]\\n            if mapped != first:\\n                return {"mark": mark, "prediction": first, "second": mapped}\\n    return {"mark": mark, "prediction": first, "second": None}\\n\\n\\ndef answer_retry(answer: list, problems: list[str] | None = None) -> dict:\\n    """The user message that returns a final answer whose grids cannot all be read (tool "free"; tool "f3" passes the\\n    problems (answer_problems) and also offers arc_submit_answer)."""\\n    count, read = len(answer), sum(grid is not None for grid in answer)\\n    labels = " Put \\\\"--Test Output k--\\\\" on the line before the k-th grid." if count > 1 else ""\\n    head = (f"Your final answer must give {count} output grid{\\\'s\\\' if count > 1 else \\\'\\\'}, each as a JSON list of rows of "\\n            f"integers 0-9 such as [[0, 1], [2, 3]]; {read} of {count} could be read")\\n    if problems is None:\\n        return {"role": "user", "content": f"{head}.{labels} Give the complete final answer again."}\\n    return {"role": "user", "content": f"{head} ({\\\'; \\\'.join(problems)}).{labels} If a function or variable computed the "\\n                                       "answer, call arc_submit_answer with it instead of writing the grids; otherwise "\\n                                       "give the complete final answer again."}\\n\\n\\ndef json_lists(text: str) -> list:\\n    """Every top-level bracketed span of text that parses as a JSON list, in order."""\\n    found, depth, start = [], 0, 0\\n    for position, char in enumerate(text):\\n        if char == "[":\\n            start = position if depth == 0 else start\\n            depth += 1\\n        elif char == "]" and depth:\\n            depth -= 1\\n            if depth == 0:\\n                with contextlib.suppress(ValueError, RecursionError):\\n                    found.append(json.loads(text[start:position + 1]))\\n    return found\\n\\n\\ndef answer_problems(text: str, answer: list) -> list[str]:\\n    """Why each unreadable grid of a final answer could not be read: the problem (grid_problem) of the last JSON list in\\n    its "--Test Output k--" part (in the whole text without labels) that is not a valid grid."""\\n    parts = re.split(r"--\\\\s*Test Output\\\\s*(\\\\d+)\\\\s*--", text)\\n    labelled = {int(label): body for label, body in zip(parts[1::2], parts[2::2])}\\n    problems = []\\n    for number, grid in enumerate(answer, 1):\\n        if grid is None:\\n            found = [problem for problem in map(grid_problem, json_lists(labelled.get(number, "") if labelled else text)) if problem]\\n            problems.append(f"output {number}: {found[-1] if found else \\\'no grid in JSON form, such as [[0, 1], [2, 3]], was found\\\'}")\\n    return problems\\n\\n\\ndef tool_retry(error: str, tool: str = "transform") -> dict:\\n    """The user message that returns a tool call the parser cannot run, e.g. an invented tool name."""\\n    if tool == "f3":\\n        return {"role": "user", "content": f"Your tool call was not run ({error}). The tools are arc_python_executor, with the "\\n                                           "parameter code and, optionally, target, and arc_submit_answer, with the parameter "\\n                                           "outputs. Call one of them that way, or give the final answer."}\\n    return {"role": "user", "content": f"Your tool call was not run ({error}). The only tool is arc_python_executor, with the "\\n                                       "parameter code and, optionally, target. Call it that way, or give the final answer."}\\n\\n\\ndef apply_free_turn(body, task: dict, view, timeout: float, final: bool = False, tool: str = "transform",\\n                    history: list[str] | None = None, exact: bool = False) -> dict:\\n    """agent_mode free: tool calls run and the conversation goes on; a message without one is the final answer, whose\\n    grids (answer_grids, ```python blocks removed) are the prediction, mapped back from the job\\\'s view. In the final\\n    round a tool call is not run and ends the trajectory without grids. Before the final round a tool call the parser\\n    cannot run is returned with tool_retry (status REVISE) instead of ending the trajectory. tool "free": calls run after\\n    the trajectory\\\'s earlier successful programs (history), and before the final round a final answer whose grids cannot\\n    all be read is returned with answer_retry."""\\n    empty = {"assistant": None, "calls": [], "replies": [], "answer": None, "prediction": None}\\n    if body is None:\\n        return {"status": "TOKEN_LIMIT", **empty}\\n    body = collapse_think_end(body)\\n    try:\\n        message = parse_assistant(body, optional=("target",), exact=exact)\\n    except ValueError as error:\\n        if not final and body.startswith("<think>") and body.count("</think>") == 1 and "<tool_call>" in body:\\n            reasoning, _, content = body[len("<think>"):].partition("</think>")\\n            return {"status": "REVISE", **empty, "assistant": {"role": "assistant", "reasoning_content": reasoning.strip(),\\n                                                                "content": content.strip()}, "replies": [tool_retry(str(error))]}\\n        return {"status": "INVALID_FORMAT", **empty, "error": str(error)}\\n    if message.get("tool_calls") and final:\\n        return {"status": "NO_GRID", **empty, "assistant": message}\\n    if message.get("tool_calls"):\\n        history, runs = list(history or []), []\\n        for call in message["tool_calls"]:\\n            runs.append(run_free_tool(tool_arguments(call), task, timeout, tool, history))\\n            if runs[-1][0].get("ok"):\\n                history.append(runs[-1][0]["code"])\\n        return {"status": "REVISE", **empty, "assistant": message, "calls": [record for record, _ in runs],\\n                "replies": [{"role": "tool", "content": reply} for _, reply in runs]}\\n    text = re.sub(r"```python.*?```", "", message["content"], flags=re.DOTALL | re.IGNORECASE)\\n    answer = answer_grids(text, len(task["test"]))\\n    if tool == "free" and not final and not all(grid is not None for grid in answer):\\n        return {"status": "REVISE", **empty, "assistant": message, "replies": [answer_retry(answer)], "answer": answer,\\n                "prediction": [invert_grid(grid, view) if view and grid is not None else grid for grid in answer]}\\n    return {"status": "ANSWERED" if any(grid is not None for grid in answer) else "NO_GRID", **empty, "assistant": message,\\n            "answer": answer, "prediction": [invert_grid(grid, view) if view and grid is not None else grid for grid in answer]}\\n\\n\\ndef pending_confirmation(turns: list[dict]):\\n    """The outputs (parsed) of the previous round\\\'s last tool call when it was an arc_submit_answer that asked for\\n    confirmation (a function failing a training pair), else None."""\\n    previous = turns[-1] if turns else {}\\n    calls = (previous.get("assistant") or {}).get("tool_calls") or []\\n    if calls and calls[-1]["function"]["name"] == "arc_submit_answer" and previous["submissions"][-1]["confirm"]:\\n        return json.loads(previous["submissions"][-1]["outputs"])\\n    return None\\n\\n\\ndef apply_f3_turn(body, task: dict, view, timeout: float, final: bool = False, turns: list[dict] = (),\\n                  notice: str | None = None, helpers: bool = True, exact: bool = False, previous: list | None = None,\\n                  doubt_hint: bool = False, ndimage: bool = False) -> dict:\\n    """tool "f3": as apply_free_turn with the free tool, plus arc_submit_answer. Executor calls run in the session\\n    (run_f3_tool); an accepted submission (submit_answer) ends the trajectory as ANSWERED with its grids, mapped back\\n    from the job\\\'s view, and their sources, and the calls after it do not run; a refused one goes back as its tool\\n    reply. In the final round programs do not run but a submission still counts (a failing function without the\\n    confirmation round); a final round without an accepted one ends without grids. A plain-text answer counts as with\\n    the free tool; one whose grids cannot all be read goes back with answer_retry naming the problems, and the same\\n    unreadable answer twice in a row ends the trajectory with the last passing program\\\'s outputs (fallback "program")\\n    when there is one. notice: a line added to the round\\\'s last tool reply (budget_notice); helpers: objects / diff\\n    predefined in the session; previous: an earlier attempt\\\'s answer that a submission must not repeat unconfirmed."""\\n    empty = {"assistant": None, "calls": [], "submissions": [], "replies": [], "answer": None, "prediction": None}\\n    if body is None:\\n        return {"status": "TOKEN_LIMIT", **empty}\\n    body = collapse_think_end(body)\\n    try:\\n        message = parse_assistant(body, optional=("target",), submit=True, exact=exact)\\n    except ValueError as error:\\n        if not final and body.startswith("<think>") and body.count("</think>") == 1 and "<tool_call>" in body:\\n            reasoning, _, content = body[len("<think>"):].partition("</think>")\\n            return {"status": "REVISE", **empty, "assistant": {"role": "assistant", "reasoning_content": reasoning.strip(),\\n                                                                "content": content.strip()}, "replies": [tool_retry(str(error), "f3")]}\\n        return {"status": "INVALID_FORMAT", **empty, "error": str(error)}\\n    mapped = lambda grids: [invert_grid(grid, view) if view and grid is not None else grid for grid in grids]  # noqa: E731\\n    if message.get("tool_calls"):\\n        session = [session_entry(call) for turn in turns for call in turn.get("calls", [])]\\n        pending, calls, submissions, replies = pending_confirmation(turns), [], [], []\\n        for call in message["tool_calls"]:\\n            arguments = tool_arguments(call)\\n            if call["function"]["name"] == "arc_submit_answer":\\n                record, reply = submit_answer(arguments["outputs"], task, timeout, session, pending, final, helpers, previous,\\n                                              ndimage)\\n                submissions.append(record)\\n                if reply is None:\\n                    return {"status": "ANSWERED", **empty, "assistant": message, "calls": calls, "submissions": submissions,\\n                            "answer": record["grids"], "prediction": mapped(record["grids"]), "sources": record["sources"]}\\n            elif final:\\n                continue\\n            else:\\n                record, reply = run_f3_tool(arguments, task, timeout, session, helpers, doubt_hint, ndimage)\\n                calls.append(record)\\n                session.append(session_entry(record))\\n            replies.append({"role": "tool", "content": reply})\\n        if final:\\n            return {"status": "NO_GRID", **empty, "assistant": message, "submissions": submissions}\\n        if notice:\\n            replies[-1] = {**replies[-1], "content": f"{replies[-1][\\\'content\\\']}\\\\n{notice}"}\\n        return {"status": "REVISE", **empty, "assistant": message, "calls": calls, "submissions": submissions, "replies": replies}\\n    text = re.sub(r"```python.*?```", "", message["content"], flags=re.DOTALL | re.IGNORECASE)\\n    answer = answer_grids(text, len(task["test"]))\\n    if not final and not all(grid is not None for grid in answer):\\n        previous = turns[-1] if turns else None\\n        if (previous and previous["status"] == "REVISE" and previous["answer"] is not None\\n                and previous["assistant"]["content"] == message["content"]):\\n            outputs = last_passing(turns, view)\\n            return {"status": "NO_GRID", **empty, "assistant": message, "answer": answer,\\n                    "prediction": mapped(answer) if outputs is None else outputs, **({} if outputs is None else {"fallback": "program"})}\\n        return {"status": "REVISE", **empty, "assistant": message, "replies": [answer_retry(answer, answer_problems(text, answer))],\\n                "answer": answer, "prediction": mapped(answer)}\\n    return {"status": "ANSWERED" if any(grid is not None for grid in answer) else "NO_GRID", **empty, "assistant": message,\\n            "answer": answer, "prediction": mapped(answer)}\\n\\n\\ndef job_view(job: dict, task: dict) -> tuple[dict, View | None]:\\n    """The task as the job\\\'s trajectory sees it: under job["view"] ([transpose, k], see jobs_for) when it has one."""\\n    if not job.get("view"):\\n        return task, None\\n    view = View(job["view"][0], job["view"][1], tuple(range(10)), tuple(range(len(task["train"]))))\\n    return apply_task(task, view), view\\n\\n\\ndef run_turn(body, task: dict, job: dict, view, settings: dict, final: bool = False, turns: list[dict] = (),\\n             context: int | None = None) -> dict:\\n    """turns: the trajectory\\\'s earlier turns (the free tool reruns their successful programs); context: the\\n    conversation\\\'s tokens after this round\\\'s generation, which tool "f3" with budget_notice reports in its tool replies\\n    beyond NOTICE_AFTER."""\\n    if settings.get("agent_mode") == "free" and settings.get("tool") == "f3":\\n        notice = (f"context used: {context} of {settings[\\\'max_model_len\\\']} tokens"\\n                  if settings.get("budget_notice") and context is not None and context > NOTICE_AFTER else None)\\n        return apply_f3_turn(body, task, view, settings["execution_timeout"], final, turns, notice,\\n                             settings.get("helpers", True), bool(settings.get("exact_tool_text")),\\n                             settings.get("previous_answer") if settings.get("repeat_refuse") else None,\\n                             bool(settings.get("doubt_hint")), bool(settings.get("scipy_ndimage")))\\n    if settings.get("agent_mode") == "free":\\n        history = [record["code"] for turn in turns for record in turn.get("calls", []) if record.get("ok")]\\n        return apply_free_turn(body, task, view, settings["execution_timeout"], final, settings.get("tool", "transform"), history,\\n                               bool(settings.get("exact_tool_text")))\\n    return apply_turn(body, task, job["test_index"], settings["execution_timeout"])\\n\\n\\ndef resume_rounds(settings: dict, task: dict, view, record: dict, messages: list[dict], directory: Path) -> tuple[list, list]:\\n    """settings "resume" ({"turns": an earlier first attempt\\\'s NN_execution.json records, "thinking_tokens", "forced"}, from\\n    src.qwen.runtime\\\'s resume_turns): the turns and messages that attempt ended with, its last turn marked by\\n    second_hypothesis_turn, each written again as NN_execution.json (no request or response files: the trajectory is not\\n    replayable). Raises when that turn does not continue (not a verified answer, a final round, or the setting off)."""\\n    turns = []\\n    for index, turn in enumerate(record["turns"]):\\n        last = index == len(record["turns"]) - 1\\n        if last:\\n            turn = second_hypothesis_turn(settings, turns, turn, view, bool(record.get("forced")))\\n        if (turn["status"] != "REVISE") or (last and not turn.get("second_hypothesis")):\\n            raise ValueError(f"resume: round {index} does not continue ({turn[\\\'status\\\']})")\\n        write_json(directory / f"{index:02d}_execution.json", turn)\\n        turns.append(turn)\\n        messages = [*messages, turn["assistant"], *turn["replies"]]\\n    return turns, messages\\n\\n\\ndef trajectory(job: dict, task: dict, root: Path, model: str, settings: dict, tokenizer, client) -> dict:\\n    directory = root / job["key"]\\n    directory.mkdir()\\n    task, view = job_view(job, task)\\n    if view and settings.get("previous_answer") is not None:\\n        raise ValueError("previous_answer is given in the task\\\'s own orientation; a view would need it mapped")\\n    messages = initial_messages(task, job["test_index"], settings.get("agent_mode", "code"), settings.get("tool", "transform"),\\n                                settings.get("helpers", True), settings.get("permissive", False), settings.get("previous_answer"),\\n                                settings.get("previous_program"), settings.get("previous_program_prompt"),\\n                                ndimage=bool(settings.get("scipy_ndimage")))\\n    result = {**job, "status": "RUNNING", "prediction": None, "turns": 0, "thinking_tokens": 0}\\n    started = time.monotonic()\\n    first, turns, continuation, allowance, marked_thinking, error = None, [], None, None, None, None\\n    free = settings.get("agent_mode") == "free"\\n    if settings.get("resume") is not None:  # an earlier first attempt\\\'s recorded rounds, continued by the second hypothesis\\n        turns, messages = resume_rounds(settings, task, view, settings["resume"], messages, directory)\\n        result.update(turns=len(turns), prediction=turns[-1]["prediction"], status="REVISE",\\n                      thinking_tokens=settings["resume"]["thinking_tokens"])\\n        marked_thinking = result["thinking_tokens"]\\n        allowance = (None, settings.get("second_hypothesis_tokens", SECOND_HYPOTHESIS_TOKENS))\\n    try:\\n        for turn_index in range(len(turns), settings["max_rounds"]):\\n            # settings deadline_brake_at (src.qwen.runtime: the generation deadline less deadline_brake): a round that\\n            # starts after it is the final round, recorded as deadline_brake for the replay\\n            # round_clamp (the same deadline in tokens of one request) caps the round, recorded for the replay, and a\\n            # clamp that leaves only the answer room forces the final round as the brake does\\n            now = time.time()\\n            cap = round_clamp(settings, now)\\n            if cap is not None:\\n                result.setdefault("round_clamp", {})[str(turn_index)] = cap\\n            force = now >= settings.get("deadline_brake_at", float("inf")) or (cap is not None and cap <= answer_room(settings))\\n            final, prefill, brake = start_round(settings, task, tokenizer, messages, turn_index, result["thinking_tokens"],\\n                                                turns, force)\\n            if force and "deadline_brake" not in result:\\n                result["deadline_brake"] = {"round": turn_index}\\n            try:\\n                payload = request_payload(tokenizer, messages, model, settings, job["seed"] + turn_index, first,\\n                                          result["thinking_tokens"], prefill, continuation, allowance, cap)\\n            except TokenBudgetSpent:\\n                result["status"] = "TOKEN_LIMIT"\\n                break\\n            first = len(payload["prompt"]) if first is None else first\\n            if allowance is not None and allowance[0] is None:\\n                allowance = (len(payload["prompt"]), allowance[1])\\n            write_json(directory / f"{turn_index:02d}_request.json", payload)\\n            kwargs = {key: value for key, value in payload.items() if key not in ("prompt_sha256", "token_continuity")}\\n            response = client.completions.create(**kwargs, timeout=settings["request_timeout"]).model_dump()\\n            write_json(directory / f"{turn_index:02d}_response.json", response)\\n            if settings.get("token_continuity"):  # the next round\\\'s prompt continues these ids when its text does\\n                sampled = payload["prompt"] + response["choices"][0]["token_ids"]\\n                continuation = (decode_native(tokenizer, sampled), sampled)\\n                result["continuity_fallbacks"] = result.get("continuity_fallbacks", 0) + (payload.get("token_continuity") is False)\\n            if response["model"] != model:\\n                raise ValueError("Server returned the wrong model identity")\\n            body = validate_response(\\n                tokenizer, payload["prompt"], response, payload["max_tokens"],\\n                payload["extra_body"].get("thinking_token_budget"),\\n                allow_implicit_thinking=any(message["role"] == "tool" for message in messages), free=free,\\n            )\\n            if prefill:\\n                body = absorb_prefill(body, prefill)\\n            turn = run_turn(body, task, job, view, settings, final, turns,\\n                            len(payload["prompt"]) + len(response["choices"][0]["token_ids"]))\\n            # settings second_hypothesis_until (src.qwen.runtime: the generation deadline less its guard): no\\n            # continuation starts after it\\n            if time.time() < settings.get("second_hypothesis_until", float("inf")):\\n                turn = second_hypothesis_turn(settings, turns, turn, view, final)\\n            write_json(directory / f"{turn_index:02d}_execution.json", turn)\\n            turns.append(turn)\\n            result.update(turns=turn_index + 1, prediction=turn["prediction"], status=turn["status"],\\n                          thinking_tokens=result["thinking_tokens"] + (0 if prefill else reasoning_tokens(tokenizer, response)))\\n            if marked_thinking is None:  # a continuation\\\'s final round never marks the first answer forced\\n                if final and settings.get("tool") == "f3":  # the answer was asked for: src.qwen.runtime\\\'s natural is False\\n                    result["forced"] = True\\n                if brake:\\n                    result["brake"] = brake\\n            if turn.get("second_hypothesis"):\\n                marked_thinking = result["thinking_tokens"]\\n                allowance = (None, settings.get("second_hypothesis_tokens", SECOND_HYPOTHESIS_TOKENS))\\n            if turn["status"] != "REVISE" or final:\\n                break\\n            messages.extend([turn["assistant"], *turn["replies"]])\\n    except Exception as failure:  # after a second_hypothesis turn the first answer stands (second_result)\\n        error = f"{type(failure).__name__}: {failure}"\\n    try:\\n        second = second_result(turns, view)\\n        if error is not None and second is None:\\n            raise RuntimeError(error)\\n        if result["status"] == "REVISE":\\n            result["status"] = "ROUND_LIMIT"\\n        kept = turns if second is None else turns[:second["mark"] + 1]\\n        result.update({key: kept[-1][key] for key in ("sources", "fallback") if kept and key in kept[-1]})\\n        if second is not None:\\n            result.update(status="ANSWERED", prediction=second["prediction"], second_hypothesis={\\n                "rounds": len(turns) - len(kept), "found": second["second"] is not None,\\n                "thinking_tokens": result["thinking_tokens"] - marked_thinking, **({"error": error} if error else {})})\\n        if free and not any(grid is not None for grid in result["prediction"] or []):\\n            fallback = program_fallback(turns, view)\\n            if fallback is not None:\\n                result.update(prediction=fallback, fallback="program")\\n        programs = (second["second"] if second is not None and second["second"] is not None\\n                    else program_predictions(settings, kept, view, result["prediction"]))\\n        if programs is not None:\\n            result["program_predictions"] = programs\\n        result["verified"] = verified(kept, view, result["prediction"])\\n    except Exception as failure:\\n        result.update(status="ERROR", error=error or f"{type(failure).__name__}: {failure}")\\n    result["elapsed_seconds"] = time.monotonic() - started\\n    write_json(directory / "result.json", result)\\n    with RESULT_LOG_LOCK:\\n        print(json.dumps({"model": model, **result}), flush=True)\\n    return result\\n\\n\\ndef replay_trajectory(job: dict, receipt: dict, task: dict, settings: dict, model: str, directory: Path, tokenizer) -> dict:\\n    """Validate one completed trajectory, including every real execution and request."""\\n    before = {p.name: digest(p) for p in directory.glob("*.json")}\\n    if any(receipt[key] != value for key, value in job.items()) or json.loads((directory / "result.json").read_text()) != receipt:\\n        raise ValueError("Trajectory receipt identity differs")\\n    if not 1 <= receipt["turns"] <= settings["max_rounds"]:\\n        raise ValueError("Invalid trajectory length")\\n    if len(list(directory.glob("*_request.json"))) != receipt["turns"]:\\n        raise ValueError("Unaccounted request in trajectory")\\n    task, view = job_view(job, task)\\n    if view and settings.get("previous_answer") is not None:\\n        raise ValueError("previous_answer is given in the task\\\'s own orientation; a view would need it mapped")\\n    messages = initial_messages(task, job["test_index"], settings.get("agent_mode", "code"), settings.get("tool", "transform"),\\n                                settings.get("helpers", True), settings.get("permissive", False), settings.get("previous_answer"),\\n                                settings.get("previous_program"), settings.get("previous_program_prompt"),\\n                                ndimage=bool(settings.get("scipy_ndimage")))\\n    turn, generated_tokens, first, thinking, turns, continuation, allowance = None, 0, None, 0, [], None, None\\n    free = settings.get("agent_mode") == "free"\\n    for index in range(receipt["turns"]):\\n        final, prefill, brake = start_round(settings, task, tokenizer, messages, index, thinking, turns,\\n                                            (receipt.get("deadline_brake") or {}).get("round") == index)\\n        payload = request_payload(tokenizer, messages, model, settings, job["seed"] + index, first, thinking, prefill,\\n                                  continuation, allowance, (receipt.get("round_clamp") or {}).get(str(index)))\\n        first = len(payload["prompt"]) if first is None else first\\n        if allowance is not None and allowance[0] is None:\\n            allowance = (len(payload["prompt"]), allowance[1])\\n        if json.loads((directory / f"{index:02d}_request.json").read_text()) != payload:\\n            raise ValueError("Native conversation, sampling or prompt differs")\\n        response = json.loads((directory / f"{index:02d}_response.json").read_text())\\n        if settings.get("token_continuity"):\\n            sampled = payload["prompt"] + response["choices"][0]["token_ids"]\\n            continuation = (decode_native(tokenizer, sampled), sampled)\\n        if response["model"] != model:\\n            raise ValueError("Response model differs")\\n        body = validate_response(\\n            tokenizer, payload["prompt"], response, payload["max_tokens"],\\n            payload["extra_body"].get("thinking_token_budget"),\\n            allow_implicit_thinking=any(message["role"] == "tool" for message in messages), free=free,\\n        )\\n        if prefill:\\n            body = absorb_prefill(body, prefill)\\n        thinking += 0 if prefill else reasoning_tokens(tokenizer, response)\\n        turn = run_turn(body, task, job, view, settings, final, turns,\\n                        len(payload["prompt"]) + len(response["choices"][0]["token_ids"]))\\n        recorded = json.loads((directory / f"{index:02d}_execution.json").read_text())\\n        if recorded.get("second_hypothesis"):  # the continuation\\\'s start was the deadline\\\'s call; its rules are checked\\n            turn = second_hypothesis_turn(settings, turns, turn, view, final)\\n            allowance = (None, settings.get("second_hypothesis_tokens", SECOND_HYPOTHESIS_TOKENS))\\n        if recorded != turn:\\n            raise ValueError("Independent execution or feedback differs")\\n        turns.append(turn)\\n        generated_tokens += response["usage"]["completion_tokens"]\\n        if index < receipt["turns"] - 1 and (turn["status"] != "REVISE" or final):\\n            raise ValueError("Trajectory continued after termination")\\n        if turn["assistant"] is not None:\\n            messages.extend([turn["assistant"], *turn["replies"]])\\n    if turn is None:\\n        raise ValueError("Missing trajectory")\\n    status = "ROUND_LIMIT" if turn["status"] == "REVISE" else turn["status"]\\n    second = second_result(turns, view)\\n    if status == "ROUND_LIMIT" and receipt["turns"] != settings["max_rounds"] and "error" not in (receipt.get("second_hypothesis") or {}):\\n        try:  # only a spent total_tokens (or second hypothesis) allowance ends a revising trajectory early\\n            request_payload(tokenizer, messages, model, settings, job["seed"] + receipt["turns"], first, thinking,\\n                            continuation=continuation, allowance=allowance)\\n        except TokenBudgetSpent:\\n            status = "TOKEN_LIMIT"\\n        else:\\n            raise ValueError("Trajectory ended before its round limit")\\n    kept, prediction, forced = turns, turn["prediction"], True if final and settings.get("tool") == "f3" else None\\n    if second is not None:\\n        kept, prediction, status, forced, brake = turns[:second["mark"] + 1], second["prediction"], "ANSWERED", None, None\\n        info = receipt.get("second_hypothesis") or {}\\n        if (info.get("rounds"), info.get("found")) != (len(turns) - len(kept), second["second"] is not None):\\n            raise ValueError("Second hypothesis result differs")\\n    if free and not any(grid is not None for grid in prediction or []):\\n        prediction = program_fallback(turns, view) or prediction\\n    programs = (second["second"] if second is not None and second["second"] is not None\\n                else program_predictions(settings, kept, view, prediction))\\n    if (status != receipt["status"] or prediction != receipt["prediction"]\\n            or receipt.get("thinking_tokens", thinking) != thinking\\n            or receipt.get("program_predictions") != programs\\n            or receipt.get("forced") != forced or receipt.get("brake") != brake):\\n        raise ValueError("Trajectory terminal result differs")\\n    if before != {p.name: digest(p) for p in directory.glob("*.json")}:\\n        raise ValueError("Trajectory artifacts changed during replay")\\n    return {"requests": receipt["turns"], "generated_tokens": generated_tokens, "thinking_tokens": thinking,\\n            "artifact_hashes": before}\\n\', \'src/agent/prompt.py\': \'"""What the model sees first: the system text, the tool schema and the task prompt of each agent mode.\\n\\nagent_mode "code": SOAR_SYSTEM and build_user_prompt (the program must be returned); agent_mode "free": FREE_SYSTEM and\\nthe direct runs\\\' task prompt (src.utils.prompt.task_prompt), with FREE_TOOL optional; with tool "free", FREE_PYTHON_SYSTEM\\nand FREE_PYTHON_TOOL (any program, the task\\\'s grids predefined, state kept across calls, printed output returned); with\\ntool "f3", F3_SYSTEM, F3_PYTHON_TOOL (the same session, every transform / solve variant verified) and ARC_SUBMIT_TOOL\\n(the answer named by the function or variable that computed it).\\n"""\\nfrom __future__ import annotations\\n\\nimport re\\nfrom typing import Any\\n\\nfrom src.utils.prompt import format_grid, task_prompt\\nfrom src.utils.grid import serialize_spaced\\n\\n\\nARC_EXECUTOR_TOOL = {\\n    "type": "function",\\n    "function": {\\n        "name": "arc_python_executor",\\n        "description": "Execute transform(grid) on every training pair and report exact verification results.",\\n        "parameters": {\\n            "type": "object",\\n            "properties": {"code": {\\n                "type": "string",\\n                "description": "Python source defining transform(grid); both the input and return value are one rectangular list of lists of integers from 0 through 9.",\\n            }},\\n            "required": ["code"],\\n        },\\n    },\\n}\\n\\nFREE_TOOL = {\\n    "type": "function",\\n    "function": {\\n        "name": "arc_python_executor",\\n        "description": "Run transform(grid) on every training pair and report, per pair, whether its output matches "\\n                       "exactly and how it differs; with target \\\\"test\\\\" also show its output for each test input.",\\n        "parameters": {\\n            "type": "object",\\n            "properties": {\\n                "code": ARC_EXECUTOR_TOOL["function"]["parameters"]["properties"]["code"],\\n                "target": {"type": "string", "enum": ["train", "test"],\\n                           "description": "train (default): verify on the training pairs; test: also return the output grid for each test input."},\\n            },\\n            "required": ["code"],\\n        },\\n    },\\n}\\n\\nFREE_PYTHON_TOOL = {\\n    "type": "function",\\n    "function": {\\n        "name": "arc_python_executor",\\n        "description": "Run Python code and return what it prints (up to 4,000 characters) and any error. The task\\\'s grids "\\n                       "are predefined: train is a list of {\\\\"input\\\\": grid, \\\\"output\\\\": grid} and test a list of "\\n                       "{\\\\"input\\\\": grid}, each grid a list of rows of integers 0-9. Variables, functions and imports "\\n                       "from your earlier calls that ran without error are kept. If the code defines transform(grid), it is "\\n                       "also run on every training input and checked against the expected output, and with target \\\\"test\\\\" "\\n                       "its output for each test input is shown.",\\n        "parameters": {\\n            "type": "object",\\n            "properties": {\\n                "code": {"type": "string", "description": "Python source: any statements; print what you want to see."},\\n                "target": FREE_TOOL["function"]["parameters"]["properties"]["target"],\\n            },\\n            "required": ["code"],\\n        },\\n    },\\n}\\n\\nF3_PYTHON_TOOL = {\\n    "type": "function",\\n    "function": {\\n        "name": "arc_python_executor",\\n        "description": "Run Python code in this task\\\'s session and return what it prints (up to 4,000 characters) and any "\\n                       "error. train (a list of {\\\\"input\\\\": grid, \\\\"output\\\\": grid}) and test (a list of {\\\\"input\\\\": "\\n                       "grid}) are predefined afresh for every call, each grid a list of rows of integers 0-9. Variables, "\\n                       "functions and imports from earlier calls are kept (from a call that raised, those defined before "\\n                       "the failing statement). Every function the code defines named transform, solve, transform<N> or "\\n                       "solve<N> is run on every training input and checked against the expected output; when it passes "\\n                       "them all, its output for each test input is shown too (with target \\\\"test\\\\", in any case). checks "\\n                       "maps each function checked in an earlier call to {\\\\"train_passed\\\\", \\\\"train_total\\\\", "\\n                       "\\\\"test_outputs\\\\"}. objects(grid, diagonal=False, background=None) is predefined and returns the "\\n                       "connected components of one color in a grid (a list of rows or a numpy array; 4-connected, or "\\n                       "8-connected with diagonal=True; background=c skips color c) as a list of {\\\\"color\\\\", \\\\"cells\\\\" (a "\\n                       "list of (row, col)), \\\\"bbox\\\\" (top, left, bottom, right; inclusive), \\\\"size\\\\"} in row-major order. "\\n                       "diff(a, b) "\\n                       "is predefined and returns \\\\"same\\\\", \\\\"shape differs: HxW vs HxW\\\\", or how many cells of the two "\\n                       "grids differ and the first 50 of them as (row, col, a, b). Both are given afresh to every call, like "\\n                       "train and test.",\\n        "parameters": {\\n            "type": "object",\\n            "properties": {\\n                "code": FREE_PYTHON_TOOL["function"]["parameters"]["properties"]["code"],\\n                "target": FREE_TOOL["function"]["parameters"]["properties"]["target"],\\n            },\\n            "required": ["code"],\\n        },\\n    },\\n}\\n\\n# settings "helpers" False (the objects / diff ablation): the same tool without the two predefined helpers\\nF3_PYTHON_TOOL_BARE = {**F3_PYTHON_TOOL, "function": {\\n    **F3_PYTHON_TOOL["function"], "description": F3_PYTHON_TOOL["function"]["description"].split(" objects(grid")[0]}}\\n\\nARC_SUBMIT_TOOL = {\\n    "type": "function",\\n    "function": {\\n        "name": "arc_submit_answer",\\n        "description": "Submit the final answer, one output grid per test input, and end the task. Name what computed "\\n                       "each grid instead of writing it: {\\\\"function\\\\": name} runs that session function on the test "\\n                       "input, {\\\\"variable\\\\": name} takes the grid that session variable holds (a list of rows or a numpy "\\n                       "array), and {\\\\"grid\\\\": [[...]]} is a grid written by hand, only for an output nothing computed. A "\\n                       "function that fails a training pair is submitted only when the identical call is repeated.",\\n        "parameters": {\\n            "type": "object",\\n            "properties": {"outputs": {\\n                "type": "array",\\n                "description": "One entry per test input, in order, e.g. [{\\\\"function\\\\": \\\\"transform\\\\"}] or "\\n                               "[{\\\\"variable\\\\": \\\\"out1\\\\"}, {\\\\"grid\\\\": [[0, 1], [1, 0]]}]; a single {\\\\"function\\\\": name} "\\n                               "object instead of the list applies that function to every test input.",\\n                "items": {"type": "object", "properties": {\\n                    "function": {"type": "string"}, "variable": {"type": "string"},\\n                    "grid": {"type": "array", "items": {"type": "array", "items": {"type": "integer"}}}}},\\n            }},\\n            "required": ["outputs"],\\n        },\\n    },\\n}\\n\\nF3_SYSTEM = (\\n    "You may solve the puzzle directly or with the arc_python_executor tool, a Python session on this task: train and "\\n    "test are predefined and hold its grids, anything you print comes back, and your variables and functions stay "\\n    "defined from call to call. Never retype a grid: read inputs from train and test, and compare grids in code (==), "\\n    "not by eye. The predefined objects(grid) returns the connected components of one color in a grid, each with its "\\n    "color, cells, inclusive bbox and size (diagonal=True for 8-connectivity, background=c to skip color c). The "\\n    "predefined "\\n    "diff(a, b) returns \\\\"same\\\\", the two shapes when they differ, or the cells where two grids differ; both are given "\\n    "afresh to every call, so do not reuse their names. Read and count "\\n    "grid cells with Python (print, objects, diff) instead of in your reasoning. A function named transform or solve, or "\\n    "either followed by a number (transform2, solve3), is checked on "\\n    "every training pair whenever you define it; once it passes them all, its outputs for the test inputs are shown too, "\\n    "and the variable checks holds the result of every check from your earlier calls. Available libraries: numpy, "\\n    "collections, itertools, functools, math, copy, re, heapq, bisect, json, random, statistics, fractions; no "\\n    "filesystem, network or subprocess access. Give your final answer with the arc_submit_answer tool: whenever a function or a variable computed it, name "\\n    "that function or variable instead of writing the grids; write a grid by hand only when nothing computed it. Only "\\n    "the grids of your final answer are graded."\\n)\\n\\n# settings "permissive" (F3\\\'): russcore\\\'s framing of the task (RULE_FRAMING), then FREE_PYTHON_SYSTEM\\\'s framing (F2) with\\n# F3\\\'s tools, so the one addition over F2 is the path that spares retyping a computed grid (arc_submit_answer naming a\\n# session function or variable); none of F3_SYSTEM\\\'s rules on retyping, comparing or reading grids, and the grid entry of\\n# arc_submit_answer is neutral. The framing came in after the 37-task prompt ablation (A3 24.5 against 22.0 without it)\\nRULE_FRAMING = ("Each training pair maps an input grid to an output grid (digits 0-9 are colors; 0 is usually background). "\\n                "Infer the ONE rule that maps every input to its output, then apply it to the test inputs. ")\\nF3P_SYSTEM = RULE_FRAMING + (\\n    "You may solve the puzzle directly or with the arc_python_executor tool, a Python session on this task: train and "\\n    "test hold its grids, anything you print comes back, and your variables and functions stay defined from call to "\\n    "call. Use it as much as helps: inspect the grids, measure objects and counts (objects(grid) and diff(a, b) are "\\n    "predefined for this), test hypotheses, or define transform(grid) or solve(grid), or either followed by a number, to "\\n    "check it on every training pair; once it passes them all, its outputs for the test inputs are shown too. Available "\\n    "libraries: numpy, collections, itertools, functools, math, copy, re, heapq, bisect, json, random, statistics, "\\n    "fractions; no filesystem, network or subprocess access. Whichever way you work, your final answer must give the "\\n    "output grid for each test input: write the grids in your final message in the requested format, or, when a "\\n    "function or variable of your session computed them, submit them with arc_submit_answer by naming it, so they need "\\n    "not be retyped. Only those grids are graded, and a final message whose grids cannot all be read is returned to you "\\n    "to correct."\\n)\\nF3P_SYSTEM_BARE = F3P_SYSTEM.replace(" (objects(grid) and diff(a, b) are predefined for this)", "")\\n\\n\\nF3_SYSTEM_BARE = re.sub(r" The predefined objects\\\\(grid\\\\).*?so do not reuse their names\\\\.", "", F3_SYSTEM).replace(\\n    "(print, objects, diff)", "(print)")\\n\\nFREE_PYTHON_SYSTEM = (\\n    "You may solve the puzzle directly or with the arc_python_executor tool, a Python session on this task: train and "\\n    "test hold its grids, anything you print comes back, and your variables and functions stay defined from call to "\\n    "call. Use it as much as helps: inspect the grids, measure objects and counts, test hypotheses, or define "\\n    "transform(grid) to check it on every training pair (and, with target \\\\"test\\\\", see its test outputs). Available "\\n    "libraries: numpy, collections, itertools, functools, math, copy, re, heapq, bisect, json, random, statistics, "\\n    "fractions; no filesystem, network or subprocess access. Whichever way you work, your final message must give the "\\n    "output grid for each test input in the requested format; only those grids are graded, and a final message whose "\\n    "grids cannot all be read is returned to you to correct."\\n)\\n\\nFREE_SYSTEM = (\\n    "You may solve the puzzle directly or with the arc_python_executor tool, which runs a Python transform(grid) on the "\\n    "training pairs and, with target \\\\"test\\\\", on the test inputs. The tool is optional: use it whenever testing a "\\n    "hypothesis or a program helps. Available libraries: numpy, collections, itertools, functools, math; no filesystem, "\\n    "network or subprocess access. Whichever way you work, your final message must give the output grid for each test "\\n    "input in the requested format; only those grids are graded."\\n)\\n\\nSOAR_SYSTEM = (\\n    "You are an ARC-AGI coding agent. Infer one general transformation from the training "\\n    "examples, write transform(grid), execute it on every training pair, and return the "\\n    "verified Python program. Do not hardcode individual examples."\\n)\\n\\n\\nARC_SUBMIT_TOOL_PERMISSIVE = {**ARC_SUBMIT_TOOL, "function": {**ARC_SUBMIT_TOOL["function"], "description": (\\n    "Submit the final answer, one output grid per test input, and end the task. Each output can be named instead of "\\n    "retyped: {\\\\"function\\\\": name} runs that session function on the test input and {\\\\"variable\\\\": name} takes the grid "\\n    "that session variable holds (a list of rows or a numpy array); {\\\\"grid\\\\": [[...]]} gives the grid itself. A function "\\n    "that fails a training pair is submitted only when the identical call is repeated.")}}\\n\\n\\ndef free_tool(tool: str) -> str:\\n    """agent_mode free\\\'s tool: "transform" (verifies transform(grid); the default), "free" (FREE_PYTHON_TOOL) or "f3"\\n    (F3_PYTHON_TOOL and ARC_SUBMIT_TOOL)."""\\n    if tool not in ("transform", "free", "f3"):\\n        raise ValueError(f"Unknown tool {tool!r}")\\n    return tool\\n\\n\\ndef build_user_prompt(task: dict[str, Any], test_index: int | None = 0) -> str:\\n    """test_index None shows every test input of the task (numbered when there are several)."""\\n    tests = task.get("test", [])\\n    if test_index is not None and not 0 <= test_index < len(tests):\\n        raise IndexError("test_index is outside the task\\\'s test list")\\n    sections: list[str] = []\\n    for pair_index, pair in enumerate(task.get("train", []), 1):\\n        sections.extend([\\n            f"Training pair {pair_index} input:",\\n            serialize_spaced(pair["input"]),\\n            f"Training pair {pair_index} output:",\\n            serialize_spaced(pair["output"]),\\n            "",\\n        ])\\n    shown = range(len(tests)) if test_index is None else [test_index]\\n    for number, index in enumerate(shown, 1):\\n        label = f"Test input {number}:" if len(shown) > 1 else "Test input:"\\n        sections.extend([label, serialize_spaced(tests[index]["input"]), ""])\\n    sections.append("Return the Python transform(grid) function.")\\n    return "\\\\n".join(sections)\\n\\n\\n# settings "previous_answer" (a later attempt with avoid_previous, see src.qwen.runtime): the task prompt followed by an\\n# earlier attempt\\\'s answer, which this attempt must not repeat\\nAVOID_PREVIOUS = ("\\\\n\\\\nA previous attempt at this task gave the answer below. It may be wrong. Your final answer must differ "\\n                  "from it; if after checking you still believe it is right, give your second most likely output instead."\\n                  "\\\\n\\\\n{answer}")\\n\\n\\n# settings "previous_program" (F4, with avoid_previous): the earlier attempt\\\'s answer with its last program and that\\n# program\\\'s check on the training pairs (src.agent.loop.previous_program); the attempt starts from a new hypothesis\\nPREVIOUS_PROGRAM = ("\\\\n\\\\nA previous attempt at this task ended with the program and the answer below. {verdict} The answer may "\\n                    "be wrong. Work out the rule again from the training pairs, starting from a different hypothesis rather "\\n                    "than patching this program. Your final answer must differ from the previous answer; if after checking "\\n                    "you still believe it is right, give your second most likely output instead.\\\\n\\\\nPrevious program:\\\\n"\\n                    "```python\\\\n{code}\\\\n```\\\\n\\\\nIts check on the training pairs:\\\\n{report}\\\\n\\\\nPrevious answer:\\\\n{answer}")\\nPREVIOUS_CODE_LIMIT = 12000  # characters of the earlier program shown; a longer one is cut at its end\\n# settings previous_program_prompt "edges": for a previous program that passed every training pair, a wrong answer is\\n# usually a few cells off where the test input has a case no training input shows, so look there first\\nPREVIOUS_PROGRAM_EDGES = ("\\\\n\\\\nA previous attempt at this task ended with the program and the answer below. Its program passed "\\n                          "every training pair, which does not prove it right on the test input. The answer may be wrong: such "\\n                          "an answer is usually off in a few cells, where the test input has a case no training input shows. "\\n                          "First list what the test input contains that no training input covers, for example ties, "\\n                          "overlapping or touching objects, objects at the border, another count of objects or colors, another "\\n                          "orientation, or another size, and check in code what the program does on each. Then either change "\\n                          "how the program treats those cases or change the hypothesis. Your final answer must differ from the "\\n                          "previous answer; if after checking you still believe it is right, give your second most likely "\\n                          "output instead.\\\\n\\\\nPrevious program:\\\\n```python\\\\n{code}\\\\n```\\\\n\\\\nIts check on the training "\\n                          "pairs:\\\\n{report}\\\\n\\\\nPrevious answer:\\\\n{answer}")\\n\\n\\n# settings "second_hypothesis" (tool f3, first attempts): the reply to an answer every grid of which a program passing\\n# every training pair computed (src.agent.loop.second_hypothesis_turn); the same conversation then looks for a second one\\nSECOND_HYPOTHESIS = ("Your answer is recorded as the first answer; it stays as it is. Passing every training pair does not "\\n                     "prove a program right on the test input, and a second answer is allowed. Look for a structurally "\\n                     "different program, built on a different hypothesis about the rule, that also passes every training "\\n                     "pair. If its test output differs from your first answer, submit it with arc_submit_answer: it becomes "\\n                     "the second answer. If you find no such program, submit your first answer again.")\\n# settings second_hypothesis_prompt "edges": the continuation looks where a verified wrong answer usually is, a few cells\\n# the training pairs do not decide\\nSECOND_HYPOTHESIS_EDGES = ("Your answer is recorded as the first answer; it stays as it is. Passing every training pair does "\\n                           "not prove a program right on the test input: a wrong answer usually differs from the right one in "\\n                           "a few cells, where the test input has a case no training input shows. List what the test input "\\n                           "contains that no training input exercises, for example ties, overlapping or touching objects, "\\n                           "objects at the border, another count of objects or colors, another orientation, or another size. "\\n                           "Check in code what your program does on each of those parts of the test input, and decide how the "\\n                           "rule most likely treats them where the training pairs leave it open. If another treatment is "\\n                           "plausible and changes the test output, write a program that passes every training pair with it and "\\n                           "submit its output with arc_submit_answer: it becomes the second answer. If the training pairs "\\n                           "decide every part of the test input, submit your first answer again.")\\n\\n\\ndef previous_program_text(program: dict, grids: list, prompt: str | None = None) -> str:\\n    """PREVIOUS_PROGRAM filled from previous_program\\\'s record and the earlier answer\\\'s grids; with prompt "edges",\\n    PREVIOUS_PROGRAM_EDGES when that program passed every training pair."""\\n    passed, total = program["train_passed"], program["train_total"]\\n    verdict = ("Its program passed every training pair, which does not prove it right on the test input." if passed == total\\n               else f"Its program passed {passed} of {total} training pairs.")\\n    code = program["code"] if len(program["code"]) <= PREVIOUS_CODE_LIMIT else program["code"][:PREVIOUS_CODE_LIMIT] + "\\\\n# ... (cut)"\\n    if prompt == "edges" and passed == total:\\n        return PREVIOUS_PROGRAM_EDGES.format(code=code, report="\\\\n".join(program["report"]), answer=previous_answer_text(grids))\\n    return PREVIOUS_PROGRAM.format(verdict=verdict, code=code, report="\\\\n".join(program["report"]),\\n                                   answer=previous_answer_text(grids))\\n\\n\\ndef previous_answer_text(grids: list) -> str:\\n    """An earlier answer as the task prompt shows grids: one grid alone, several under "--Test Output k--" labels."""\\n    shown = [format_grid(grid) if grid is not None else "(no grid)" for grid in grids]\\n    return shown[0] if len(shown) == 1 else "\\\\n\\\\n".join(f"--Test Output {k}--\\\\n{grid}" for k, grid in enumerate(shown, 1))\\n\\n\\ndef initial_messages(task: dict, test_index: int | None, mode: str = "code", tool: str = "transform",\\n                     helpers: bool = True, permissive: bool = False, previous: list | None = None,\\n                     program: dict | None = None, program_prompt: str | None = None, ndimage: bool = False) -> list[dict]:\\n    """helpers: tool "f3" tells of the predefined objects / diff (F3_SYSTEM) or not (F3_SYSTEM_BARE); permissive: tool\\n    "f3" with F3\\\' framing (F3P_SYSTEM, F3P_SYSTEM_BARE); previous: agent_mode free, an earlier attempt\\\'s grids, one per\\n    test input, shown after the task with AVOID_PREVIOUS, or with program (that attempt\\\'s previous_program record) with\\n    PREVIOUS_PROGRAM (program_prompt: previous_program_text\\\'s prompt)."""\\n    if any("output" in test for test in task["test"]):\\n        raise ValueError("Test labels must not enter generation or replay")\\n    if mode == "free":\\n        if test_index is not None:\\n            raise ValueError("agent_mode free runs one trajectory per task (job_unit task)")\\n        system = {"transform": FREE_SYSTEM, "free": FREE_PYTHON_SYSTEM, "f3": F3_SYSTEM if helpers else F3_SYSTEM_BARE}[free_tool(tool)]\\n        if permissive:\\n            system = (F3P_SYSTEM if helpers else F3P_SYSTEM_BARE) if tool == "f3" else None\\n            if system is None:\\n                raise ValueError("permissive applies to tool f3")\\n        if ndimage:  # tool f3 with settings scipy_ndimage: its library list also names scipy.ndimage\\n            if tool != "f3" or "libraries: numpy, " not in system:\\n                raise ValueError("scipy_ndimage applies to tool f3")\\n            system = system.replace("libraries: numpy, ", "libraries: numpy, scipy.ndimage (predefined as ndimage, or from "\\n                                    "scipy import ndimage), ", 1)\\n        if previous is not None and len(previous) != len(task["test"]):\\n            raise ValueError(f"previous answer has {len(previous)} grids for {len(task[\\\'test\\\'])} test inputs")\\n        if program is not None and previous is None:\\n            raise ValueError("a previous program is shown with the previous answer")\\n        content = task_prompt(task) + ("" if previous is None else AVOID_PREVIOUS.format(answer=previous_answer_text(previous))\\n                                       if program is None else previous_program_text(program, previous, program_prompt))\\n        return [{"role": "system", "content": system}, {"role": "user", "content": content}]\\n    return [\\n        {"role": "system", "content": SOAR_SYSTEM + " Available libraries: numpy, collections, itertools, functools, math. Use only pure transformations; no filesystem, network or subprocess access."},\\n        {"role": "user", "content": build_user_prompt(task, test_index)},\\n    ]\\n\', \'src/agent/protocol.py\': \'"""The wire format of one round: lossless tokenizer helpers, prompt rendering with the tool schema, and the checks and\\nparsing of the generated tokens (reasoning boundary, XML tool calls, terminator, thinking budget)."""\\nimport json\\nfrom pathlib import Path\\nimport re\\nfrom typing import Any\\n\\nfrom .prompt import ARC_EXECUTOR_TOOL\\n\\n\\nclass NativeProtocolError(ValueError):\\n    """Native bytes or token boundaries disagree with the request."""\\n\\n\\ndef token_ids(value):\\n    if not isinstance(value, list) or not value or any(type(t) is not int or t < 0 for t in value):\\n        raise NativeProtocolError("missing or invalid flat token IDs")\\n    return value\\n\\n\\ndef decode_native(tokenizer, ids):\\n    return tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)\\n\\n\\ndef load_native_tokenizer(model, format_name="qwen"):\\n    """format_name "qwen": Qwen3.5 / Qwen3.8 turns open with <think>; "qwen3_coder": Qwen3-Coder-Next, no reasoning block.\\n    Both call tools in the same XML (<tool_call><function=...><parameter=...>)."""\\n    if format_name not in ("qwen", "qwen3_coder"):\\n        raise NativeProtocolError("unsupported native format")\\n    from transformers import AutoTokenizer\\n\\n    tokenizer = AutoTokenizer.from_pretrained(str(model), local_files_only=True, trust_remote_code=False)\\n    terminals = {tokenizer.eos_token_id}\\n    for name in ("config.json", "generation_config.json"):\\n        path = Path(model) / name\\n        if path.exists():\\n            config = json.loads(path.read_text())\\n            value = config.get("eos_token_id", config.get("text_config", {}).get("eos_token_id"))\\n            terminals.update(value if isinstance(value, list) else [value])\\n    terminals.discard(None)\\n    tokenizer.screen_terminal_ids = token_ids(sorted(terminals))\\n    tokenizer.native_format = format_name\\n    return tokenizer\\n\\n\\ndef render_messages(tokenizer, messages: list[dict], earlier_reasoning: str = "keep",\\n                    tool: dict | list[dict] = ARC_EXECUTOR_TOOL, reasoning_effort: str | None = None) -> tuple[str, list[int]]:\\n    # "keep": every turn keeps its reasoning in the history. Qwen3.5-style templates drop earlier reasoning unless\\n    # patched; newer ones gate it on preserve_thinking (templates without that variable ignore the argument, so their\\n    # prompts are unchanged) or always keep it. "drop": every earlier assistant turn is rendered without its reasoning,\\n    # the way the template renders turns before the last user query (within one tool-calling query it would keep\\n    # them). A probe with two user turns checks the property itself. A qwen3_coder tokenizer has no reasoning to keep.\\n    # tool: one tool schema, or a list of them. reasoning_effort: the template\\\'s variable (Qwen3.8-Flash-Next: xhigh, its\\n    # default, medium or low), passed only when set, since the template\\\'s default() applies to an undefined one only.\\n    tools = tool if isinstance(tool, list) else [tool]\\n    if getattr(tokenizer, "native_format", "qwen") == "qwen3_coder":\\n        text = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True)\\n        return encode_prompt(tokenizer, text, "<|im_start|>assistant\\\\n")\\n    template = tokenizer.get_chat_template()\\n    marker = "{%- if loop.index0 > ns.last_query_index %}"\\n    if earlier_reasoning == "keep":\\n        if template.count(marker) == 1:\\n            template = template.replace(marker, "{%- if true %}")\\n    elif earlier_reasoning == "drop":\\n        if template.count(marker) == 1:\\n            template = template.replace(marker, "{%- if false %}")\\n        else:  # "preserve_thinking ... or loop.index0 > ns.last_query_index", with or without parentheses\\n            template, count = re.subn(r"\\\\s+or\\\\s+\\\\(?loop\\\\.index0 > ns\\\\.last_query_index\\\\)?(?=\\\\s*%\\\\})", "", template)\\n            if count != 1:\\n                raise ValueError("Unsupported native reasoning template")\\n    else:\\n        raise ValueError(f"Unknown earlier_reasoning {earlier_reasoning!r}")\\n    render = lambda conversation: tokenizer.apply_chat_template(\\n        conversation, tools=tools, chat_template=template, tokenize=False, add_generation_prompt=True,\\n        enable_thinking=True, preserve_thinking=earlier_reasoning == "keep",\\n        truncate_history_thinking=earlier_reasoning != "keep",  # Nemotron 3.5\\\'s name for the same switch\\n        **({"reasoning_effort": reasoning_effort} if reasoning_effort is not None else {}),\\n    )\\n    probe = [{"role": "user", "content": "a"},\\n             {"role": "assistant", "content": "b", "reasoning_content": "EARLIER-REASONING-PROBE"},\\n             {"role": "user", "content": "c"}]\\n    if ("EARLIER-REASONING-PROBE" in render(probe)) != (earlier_reasoning == "keep"):\\n        raise ValueError("Unsupported native reasoning template")\\n    return encode_prompt(tokenizer, render(messages), "<|im_start|>assistant\\\\n<think>\\\\n")\\n\\n\\ndef encode_prompt(tokenizer, text: str, generation_prompt: str) -> tuple[str, list[int]]:\\n    ids = tokenizer.encode(text, add_special_tokens=False)\\n    if decode_native(tokenizer, ids) != text or not text.endswith(generation_prompt):\\n        raise ValueError("Native tool prompt does not round-trip")\\n    return text, ids\\n\\n\\nREPEATED_THINK_END = re.compile(r"</think>(?:\\\\s*</think>)+")\\n\\n\\ndef collapse_think_end(body: str) -> str:\\n    """One </think> where the model repeated it right after the first (with only whitespace between), as it does after\\n    a budget-forced end (P 0.98 once the forced </think> is followed by a newline). A body with a single </think> is\\n    unchanged, so only rounds that would fail the reasoning boundary are affected."""\\n    return REPEATED_THINK_END.sub("</think>", body, count=1)\\n\\n\\ndef parse_assistant(body: str, optional: tuple[str, ...] = (), submit: bool = False, exact: bool = False) -> dict:\\n    """Parse Qwen XML calls without interpreting Python comparison operators as XML; a call takes code and the\\n    optional parameters named, or with submit (tool "f3") may be arc_submit_answer with outputs only. exact (settings\\n    exact_tool_text): the message keeps each parameter\\\'s text exactly as generated (tool_arguments strips it for the\\n    tools), so the next round\\\'s rendering repeats the generated tokens and its prompt extends the cached prefix."""\\n    body = collapse_think_end(body)\\n    if not body.startswith("<think>") or body.count("</think>") != 1:\\n        raise ValueError("Missing complete reasoning boundary")\\n    reasoning, _, content = body[len("<think>"):].partition("</think>")\\n    content = content.strip()\\n    message: dict[str, Any] = {"role": "assistant", "reasoning_content": reasoning.strip(), "content": content}\\n    if "<tool_call>" not in content:\\n        return message\\n    prefix, _, _ = content.partition("<tool_call>")\\n    blocks = re.findall(r"<tool_call>\\\\s*<function=([^>]+)>\\\\s*(.*?)\\\\s*</function>\\\\s*</tool_call>", content, re.DOTALL)\\n    remainder = re.sub(r"<tool_call>\\\\s*<function=[^>]+>\\\\s*.*?\\\\s*</function>\\\\s*</tool_call>", "", content[len(prefix):], flags=re.DOTALL)\\n    if not blocks or remainder.strip() or len(blocks) > 4:\\n        raise ValueError("Malformed or excessive tool calls")\\n    calls = []\\n    for name, arguments in blocks:\\n        pairs = re.findall(r"<parameter=(\\\\w+)>\\\\s*\\\\n?(.*?)\\\\n?\\\\s*</parameter>", arguments, re.DOTALL)\\n        values = {key: value.strip() for key, value in pairs}\\n        # the exact text between the template\\\'s own "<parameter=x>\\\\n" and "\\\\n</parameter>", so that the next round\\\'s\\n        # rendering of this message repeats the generated tokens (a stripped blank line forked the prompt from the cached\\n        # prefix); the tools take the stripped values (tool_arguments)\\n        exact_pairs = re.findall(r"<parameter=(\\\\w+)>\\\\n(.*?)\\\\n</parameter>", arguments, re.DOTALL)\\n        kept = dict(exact_pairs) if exact and [key for key, _ in exact_pairs] == [key for key, _ in pairs] else values\\n        required, allowed = ("outputs", ()) if submit and name == "arc_submit_answer" else ("code", optional)\\n        if ((required == "code" and name != "arc_python_executor") or required not in values or len(values) != len(pairs)\\n                or set(values) - {required, *allowed}\\n                or re.sub(r"<parameter=\\\\w+>.*?</parameter>", "", arguments, flags=re.DOTALL).strip()):\\n            raise ValueError("Unsupported tool name or parameters")\\n        calls.append({"type": "function", "function": {"name": name, "arguments": kept}})\\n    message.update(content=prefix.strip(), tool_calls=calls)\\n    return message\\n\\n\\ndef tool_arguments(call: dict) -> dict:\\n    """A parsed call\\\'s parameters as the tools take them, each value stripped (the message keeps the generated text)."""\\n    return {key: value.strip() for key, value in call["function"]["arguments"].items()}\\n\\n\\ndef validate_response(tokenizer, ids: list[int], response: dict, max_tokens: int,\\n                     thinking_token_budget: int | None = None,\\n                     allow_implicit_thinking: bool = False, free: bool = False) -> str | None:\\n    """The round\\\'s assistant body for parse_assistant (None when the round\\\'s allowance ended inside the reasoning).\\n    free (agent_mode "free"): a message that ends without </think> before the budget is a final answer written inside\\n    the reasoning, as direct answers are parsed (src.utils.prompt.parse_answer reads the whole text then); its whole\\n    text is the content."""\\n    if len(response["choices"]) != 1:\\n        raise ValueError("Expected one native completion")\\n    choice = response["choices"][0]\\n    if choice.get("finish_reason") == "abort":  # the server ended the request, e.g. Miles\\\' abort_request after an RL step\\n        raise ValueError("Native request aborted by the server")\\n    generated = choice.get("token_ids")\\n    if (choice.get("prompt_token_ids") != ids or not isinstance(generated, list)\\n            or len(generated) != response["usage"]["completion_tokens"]\\n            or response["usage"]["prompt_tokens"] != len(ids)\\n            or not 0 < len(generated) <= max_tokens\\n            or decode_native(tokenizer, generated) != choice["text"]):\\n        raise ValueError("Native response token accounting differs")\\n    coder = getattr(tokenizer, "native_format", "qwen") == "qwen3_coder"\\n    if coder and thinking_token_budget is not None:\\n        raise ValueError("qwen3_coder has no reasoning to budget")\\n    end_ids = tokenizer.encode("</think>", add_special_tokens=False)\\n    body = decode_native(tokenizer, generated[:-1])\\n    has_thinking_end = len(end_ids) == 1 and end_ids[0] in generated\\n    if (choice["finish_reason"] == "length" and thinking_token_budget is not None and not has_thinking_end\\n            and len(generated) == max_tokens <= thinking_token_budget):\\n        return None  # the round\\\'s allowance (total_tokens) ended inside the reasoning, before the budget could\\n    if thinking_token_budget is not None:\\n        if len(end_ids) != 1 or (not has_thinking_end and not allow_implicit_thinking and not free):\\n            raise ValueError("Native thinking token budget was not enforced")\\n        if has_thinking_end and generated.index(end_ids[0]) >= max(1, thinking_token_budget):\\n            raise ValueError("Native thinking token budget was not enforced")\\n    if choice["finish_reason"] == "length":\\n        return None\\n    terminal_ids = tokenizer.screen_terminal_ids\\n    if (choice["finish_reason"] != "stop" or generated[-1] not in terminal_ids\\n            or any(t in terminal_ids for t in generated[:-1])):\\n        raise ValueError("Native message terminator differs")\\n    if coder:\\n        return "<think>\\\\n</think>\\\\n" + body  # an empty reasoning block: the form parse_assistant reads\\n    if has_thinking_end:\\n        return "<think>\\\\n" + body\\n    if (allow_implicit_thinking or free) and "<tool_call>" in body:\\n        reasoning, marker, content = body.partition("<tool_call>")\\n        return f"<think>\\\\n{reasoning.rstrip()}\\\\n</think>\\\\n{marker}{content}"\\n    if free:\\n        return "<think>\\\\n</think>\\\\n" + body\\n    raise ValueError("Missing complete reasoning boundary")\\n\\n\\ndef reasoning_tokens(tokenizer, response: dict) -> int:\\n    """A round\\\'s generated tokens before </think> (all of them when it has none)."""\\n    generated = response["choices"][0]["token_ids"]\\n    end = tokenizer.encode("</think>", add_special_tokens=False)\\n    return generated.index(end[0]) if len(end) == 1 and end[0] in generated else len(generated)\\n\', \'src/agent/tools.py\': \'"""What the arc_python_executor tool does with a call, and the reply the model reads.\\n\\nagent_mode "code": evaluate_message verifies each program (candidates.evaluate_program) and replies with\\nformat_verifier_feedback; agent_mode "free": run_free_tool reports every training pair and, for target "test", the test\\noutputs; with the free tool (settings tool "free") it runs any program with the task\\\'s grids predefined and the\\ntrajectory\\\'s earlier programs\\\' state, and returns what it printed as well. Tool "f3": run_f3_tool runs the session\\n(session_entry) and verifies every transform / solve variant, submit_answer evaluates arc_submit_answer.\\n"""\\nfrom __future__ import annotations\\n\\nimport ast\\nfrom collections import Counter\\nfrom dataclasses import asdict\\nimport json\\nfrom typing import Any\\n\\nfrom src.utils.prompt import format_grid\\nfrom src.utils.grid import normalize_grid\\nfrom .candidates import ProgramEvaluation, evaluate_program\\nfrom .protocol import tool_arguments\\nfrom .executor import (STDOUT_LIMIT, grid_problem, parse_python_code, run_free_program, run_session, run_transforms,\\n                       safety_error)\\n\\n\\ndef format_verifier_feedback(evaluation: ProgramEvaluation | None,\\n                             task: dict[str, Any]) -> str:\\n    """Create train-only feedback suitable for a revision prompt."""\\n    if evaluation is None:\\n        return "No transform-like Python function was found. Return one complete function."\\n    if evaluation.safety_error:\\n        return f"The program was rejected by the static gate: {evaluation.safety_error}."\\n    if evaluation.execution_failed:\\n        return "The program timed out or the worker process failed. Replace the algorithm."\\n    if evaluation.train_exact and evaluation.test_prediction is None:\\n        return "All training pairs passed, but the test call did not return a valid ARC grid."\\n    if evaluation.accepted:\\n        return "All training pairs passed and the test prediction is a valid ARC grid."\\n\\n    for pair_index, (pair, actual) in enumerate(zip(task.get("train", []),\\n                                                    evaluation.train_predictions)):\\n        if actual != pair["output"]:\\n            error = evaluation.train_errors[pair_index] if pair_index < len(evaluation.train_errors) else None\\n            error_line = f"Execution error: {error}\\\\n" if error else ""\\n            return (\\n                f"Training pair {pair_index} failed.\\\\n"\\n                f"Input: {json.dumps(pair[\\\'input\\\'], separators=(\\\',\\\', \\\':\\\'))}\\\\n"\\n                f"Expected: {json.dumps(pair[\\\'output\\\'], separators=(\\\',\\\', \\\':\\\'))}\\\\n"\\n                f"Actual: {json.dumps(actual, separators=(\\\',\\\', \\\':\\\'))}\\\\n"\\n                f"{error_line}"\\n                "Revise the general rule; do not hardcode this pair."\\n            )\\n    return (\\n        f"Only {evaluation.train_passed}/{evaluation.train_total} training pairs passed. "\\n        "Revise the general rule."\\n    )\\n\\n\\ndef evaluate_message(message: dict, task: dict, test_index: int | None, timeout: float) -> tuple[list[dict], list[dict], object]:\\n    """test_index None (job_unit "task"): a program is accepted only when it passes the training pairs and returns a\\n    valid grid for every test input; its prediction is then the list of those grids. The recorded evaluation is the\\n    first test input\\\'s that failed, else the last one\\\'s."""\\n    calls = message.get("tool_calls", [])\\n    candidates = [tool_arguments(call)["code"] for call in calls] if calls else [message["content"]]\\n    indices = range(len(task["test"])) if test_index is None else [test_index]\\n    evaluations, replies = [], []\\n    prediction = None\\n    for candidate in candidates:\\n        code = (candidate.strip() if candidate.strip().startswith(("import ", "from ", "def "))\\n                else parse_python_code(candidate))\\n        # ponytail: the training pairs rerun once per test input; fold into one run_transforms call if execution time matters\\n        checked = []\\n        for index in indices if code else []:\\n            checked.append(evaluate_program(code, task, index, timeout=timeout))\\n            if not checked[-1].accepted:\\n                break\\n        evaluation = checked[-1] if checked else None\\n        feedback = format_verifier_feedback(evaluation, task)\\n        evaluations.append({"code": code, "evaluation": asdict(evaluation) if evaluation else None, "feedback": feedback})\\n        if len(checked) == len(indices) and evaluation.accepted and prediction is None:\\n            prediction = evaluation.test_prediction if test_index is not None else [e.test_prediction for e in checked]\\n        replies.append({"role": "tool" if calls else "user", "content": feedback})\\n    return evaluations, replies, prediction\\n\\n\\ndef grid_difference(expected: list, actual, error: str | None) -> str:\\n    if actual == expected:\\n        return "passed."\\n    if actual is None:\\n        return f"failed: no valid grid ({error or \\\'invalid output\\\'})."\\n    shape = lambda grid: f"{len(grid)}x{len(grid[0])}"\\n    if shape(actual) != shape(expected):\\n        return f"failed: expected a {shape(expected)} grid, got {shape(actual)}:\\\\n{format_grid(actual)}"\\n    cells = [(r, c, e, a) for r, (want, got) in enumerate(zip(expected, actual))\\n             for c, (e, a) in enumerate(zip(want, got)) if e != a]\\n    shown = ", ".join(f"({r},{c}) expected {e} got {a}" for r, c, e, a in cells[:20])\\n    return f"failed: {len(cells)} of {sum(map(len, expected))} cells differ: {shown}{\\\', ...\\\' if len(cells) > 20 else \\\'\\\'}."\\n\\n\\ndef verification_lines(task: dict, record: dict, outputs: list, errors: list, target: str) -> list[str]:\\n    """Per training pair verification and, for target "test", the test outputs; fills the record\\\'s output fields."""\\n    count = len(task["train"])\\n    record.update(train_outputs=outputs[:count], test_outputs=outputs[count:],\\n                  train_passed=sum(actual == pair["output"] for actual, pair in zip(outputs, task["train"])))\\n    lines = [f"Training pair {number}: {grid_difference(pair[\\\'output\\\'], actual, error)}"\\n             for number, (pair, actual, error) in enumerate(zip(task["train"], outputs, errors), 1)]\\n    lines.append(f"{record[\\\'train_passed\\\']}/{count} training pairs passed.")\\n    for number, (grid, error) in enumerate(zip(outputs[count:], errors[count:]) if target == "test" else [], 1):\\n        lines += [f"Test input {number} output:", format_grid(grid) if grid is not None else f"no valid grid ({error or \\\'invalid output\\\'})"]\\n    return lines\\n\\n\\ndef run_free_tool(arguments: dict, task: dict, timeout: float, tool: str = "transform",\\n                  history: list[str] | None = None) -> tuple[dict, str]:\\n    """(record, reply) of one free-mode call: the program runs on every training pair and test input; the reply shows\\n    each training pair\\\'s verification and, for target "test", the test outputs (the record keeps them either way).\\n    tool "free": any program runs once with train / test predefined after the earlier successful programs (history);\\n    the reply is what it printed (bounded), its error if it raised, and the verification when it defines transform.\\n    The record\\\'s "ok" marks a program that ran to the end, so later calls rerun it."""\\n    code, target = arguments["code"], arguments.get("target", "train")\\n    count = len(task["train"])\\n    record = {"code": code, "target": target, "error": None, "train_passed": 0, "train_total": count,\\n              "train_outputs": None, "test_outputs": None}\\n    if target not in ("train", "test"):\\n        return {**record, "error": "unknown target"}, \\\'The target must be "train" or "test".\\\'\\n    free = tool == "free"\\n    if safety_error(code, free):\\n        return {**record, "error": safety_error(code, free)}, f"The program was rejected by the static gate: {safety_error(code, free)}."\\n    if free:\\n        result = run_free_program(code, task["train"], [{"input": test["input"]} for test in task["test"]], timeout,\\n                                  history or [])\\n        if result is None:\\n            return {**record, "error": "timeout or worker failure", "ok": False}, "The program timed out or the worker process failed."\\n        record.update(stdout=result["stdout"], script_error=result["error"], ok=result["error"] is None)\\n        lines = printed_lines(result)\\n        if result["outputs"] is not None:\\n            lines += verification_lines(task, record, result["outputs"], result["errors"], target)\\n        return record, "\\\\n".join(lines) if lines else "The program ran and printed nothing."\\n    errors: list[str | None] = []\\n    outputs = run_transforms(code, [pair["input"] for pair in task["train"]] + [test["input"] for test in task["test"]],\\n                             timeout=timeout, errors=errors)\\n    if outputs is None:\\n        return {**record, "error": "timeout or worker failure"}, "The program timed out or the worker process failed."\\n    return record, "\\\\n".join(verification_lines(task, record, outputs, errors, target))\\n\\n\\ndef printed_lines(result: dict) -> list[str]:\\n    """A free-tool run\\\'s printed output (bounded) and its error."""\\n    return (["Output:", result["stdout"].rstrip("\\\\n")] if result["stdout"] else []) + (\\n        [f"... (output truncated at {STDOUT_LIMIT} characters)"] if result["truncated"] else []) + (\\n        [f"Error: {result[\\\'error\\\']}"] if result["error"] else [])\\n\\n\\ndef session_entry(record: dict) -> tuple[str | None, dict]:\\n    """One f3 executor record as run_session\\\'s session entry: the program later calls rerun (all of it when it ran to\\n    the end, the statements before the failing one when it raised, else None) and its functions\\\' results."""\\n    return record["code"] if record.get("ok") else record.get("kept"), record.get("functions") or {}\\n\\n\\ndef kept_statements(code: str, completed: int) -> str:\\n    """The source of code\\\'s first `completed` top-level statements (ast end positions count UTF-8 bytes)."""\\n    end = ast.parse(code).body[completed - 1]\\n    data = code.encode()\\n    lines = data.splitlines(keepends=True)\\n    return data[:sum(map(len, lines[:end.end_lineno - 1])) + end.end_col_offset].decode()\\n\\n\\n# The three invariants of settings doubt_rules / doubt_hint (src.qwen.runtime.rule_doubt; this tool\\\'s reply): what every\\n# training output of a task keeps of its input, which a test output breaking it is checked for (a grid\\\'s background is\\n# its most common color). On the ARC-AGI-2 training set\\\'s true test outputs they break 0.4% (colors), 4.9% (background)\\n# and 2.8% (count) of the rules their training pairs keep, so a broken one is a reason to check, not proof of an error.\\nINVARIANTS = {"colors": "uses exactly its input\\\'s colors",\\n              "background": "keeps every background cell of its input at the input\\\'s shape",\\n              "count": "has as many non-background cells as its input"}\\n\\n\\ndef broken_invariants(task: dict, predictions) -> list[tuple[int, str, str]]:\\n    """(test output number from 1, invariant, how this output breaks it) for every invariant that every training pair of\\n    the task keeps and a valid test grid of predictions breaks; [] for a task or grid of another shape than its rows\\n    claim."""\\n    def colors(grid):\\n        return {cell for row in grid for cell in row}\\n\\n    def background(grid):\\n        return Counter(cell for row in grid for cell in row).most_common(1)[0][0]\\n\\n    def non_background(grid):\\n        own = background(grid)\\n        return sum(cell != own for row in grid for cell in row)\\n\\n    def colors_rule(source, output):\\n        added, dropped = sorted(colors(output) - colors(source)), sorted(colors(source) - colors(output))\\n        parts = [f"adds color{\\\'s\\\' * (len(added) > 1)} {\\\', \\\'.join(map(str, added))}"] * bool(added)\\n        parts += [f"drops color{\\\'s\\\' * (len(dropped) > 1)} {\\\', \\\'.join(map(str, dropped))}"] * bool(dropped)\\n        return " and ".join(parts) or None\\n\\n    def background_rule(source, output):\\n        if len(source) != len(output) or any(len(a) != len(b) for a, b in zip(source, output)):\\n            return "has another shape than its input"\\n        own = background(source)\\n        changed = sum(o != s for a, b in zip(source, output) for s, o in zip(a, b) if s == own)\\n        return f"changes {changed} of its input\\\'s background cells" if changed else None\\n\\n    def count_rule(source, output):\\n        have, want = non_background(output), non_background(source)\\n        return f"has {have} non-background cells where its input has {want}" if have != want else None\\n\\n    rules = {"colors": colors_rule, "background": background_rule, "count": count_rule}\\n    try:\\n        kept = [key for key, rule in rules.items() if all(rule(pair["input"], pair["output"]) is None for pair in task["train"])]\\n        broken = []\\n        for number, (grid, test) in enumerate(zip(predictions or [], task["test"]), 1):\\n            if isinstance(grid, list) and grid and all(isinstance(row, list) and row for row in grid):\\n                broken += [(number, key, detail) for key in kept if (detail := rules[key](test["input"], grid)) is not None]\\n        return broken\\n    except (IndexError, TypeError):\\n        return []\\n\\n\\ndef invariant_hints(task: dict, outputs: list) -> list[str]:\\n    """settings doubt_hint: one sentence per invariant broken by these test outputs (broken_invariants), asking for a\\n    check rather than a change."""\\n    return [f"Note: every training output {INVARIANTS[key]}, but this output for test input {number} {detail}. "\\n            "Check whether that is intended before you submit." for number, key, detail in broken_invariants(task, outputs)]\\n\\n\\ndef run_f3_tool(arguments: dict, task: dict, timeout: float, session: list, helpers: bool = True,\\n                doubt_hint: bool = False, ndimage: bool = False) -> tuple[dict, str]:\\n    """(record, reply) of one f3 executor call: the program runs in the session (session_entry, run_session) and\\n    the reply is what it printed, its error, and the verification of every transform / solve variant it defines, with\\n    the test outputs of each one that passes every training pair (of every one with target "test"). The record keeps\\n    each variant\\\'s result under functions (checks\\\' entries), one variant\\\'s in the call\\\'s train / test fields as the free\\n    tool does (the last that passes every training pair with a grid for every test input, else the last that passes\\n    them, else the last one), and for a call that raised, the statements before the failing one (kept)."""\\n    code, target = arguments["code"], arguments.get("target", "train")\\n    count = len(task["train"])\\n    record = {"code": code, "target": target, "error": None, "train_passed": 0, "train_total": count,\\n              "train_outputs": None, "test_outputs": None}\\n    if target not in ("train", "test"):\\n        return {**record, "error": "unknown target"}, \\\'The target must be "train" or "test".\\\'\\n    error = safety_error(code, True, f3=True, ndimage=ndimage)\\n    if error:\\n        return {**record, "error": error}, f"The program was rejected by the static gate: {error}."\\n    result = run_session(code, task["train"], [{"input": test["input"]} for test in task["test"]], timeout, session,\\n                         helpers=helpers, ndimage=ndimage)\\n    if result is None:\\n        return {**record, "error": "timeout or worker failure", "ok": False}, "The program timed out or the worker process failed."\\n    record.update(stdout=result["stdout"], script_error=result["error"], ok=result["error"] is None, functions={})\\n    if result["error"] and result["completed"]:\\n        record["kept"] = kept_statements(code, result["completed"])\\n    lines, best = printed_lines(result), -1\\n    for name, run in result["functions"].items():\\n        checked: dict = {}\\n        passed = all(actual == pair["output"] for actual, pair in zip(run["outputs"], task["train"]))\\n        lines += [f"{name}:", *verification_lines(task, checked, run["outputs"], run["errors"], "test" if passed else target)]\\n        if doubt_hint and passed and all(grid is not None for grid in checked["test_outputs"]):\\n            lines += invariant_hints(task, checked["test_outputs"])\\n        record["functions"][name] = {"train_passed": checked["train_passed"], "train_total": count,\\n                                     "test_outputs": checked["test_outputs"]}\\n        rank = 2 if passed and all(grid is not None for grid in checked["test_outputs"]) else int(passed)\\n        if rank >= best:\\n            record.update(checked)\\n            best = rank\\n    return record, "\\\\n".join(lines) if lines else "The program ran and printed nothing."\\n\\n\\nSUBMIT_KINDS = ("function", "variable", "grid")\\n\\n\\ndef submit_answer(outputs: str, task: dict, timeout: float, session: list, pending=None,\\n                  final: bool = False, helpers: bool = True, previous: list | None = None,\\n                  ndimage: bool = False) -> tuple[dict, str | None]:\\n    """(record, reply) of one arc_submit_answer call; reply None means submitted. outputs (JSON): one entry per test\\n    input, each {"function": name} (that session function on the test input), {"variable": name} (a grid that session\\n    variable holds) or {"grid": grid}; a single {"function": name} applies to every test input (a single entry of either\\n    kind, to a task\\\'s one test input). Only each grid\\\'s format is checked; any problem goes back as the reply. A\\n    function that fails a training pair is submitted only when the call repeats pending (the outputs of the previous\\n    round\\\'s last call, which asked for that confirmation) or in the final round; so is an answer whose every grid equals\\n    previous (a later attempt\\\'s earlier answer, settings repeat_refuse), whose prompt asked for a different one. The\\n    record keeps the grids (the job\\\'s view) and the per-output sources ("function:name", "variable:name" or "grid"), or\\n    the error, or confirm, and failing (each submitted function\\\'s failing training pairs, empty when all passed)."""\\n    record = {"outputs": outputs, "error": None, "grids": None, "sources": None, "confirm": False}\\n\\n    def refuse(problem: str) -> tuple[dict, str]:\\n        return {**record, "error": problem}, f"The answer was not submitted: {problem}. Fix it and call arc_submit_answer again."\\n\\n    count = len(task["test"])\\n    try:\\n        value = json.loads(outputs)\\n    except (ValueError, RecursionError) as error:  # deep nesting exceeds the decoder\\\'s recursion limit\\n        return refuse(f\\\'outputs is not valid JSON ({error}); give e.g. [{{"function": "transform"}}]\\\')\\n    entries = [value] * count if isinstance(value, dict) and ("function" in value or count == 1) else value\\n    if not isinstance(entries, list) or len(entries) != count:\\n        return refuse(f"outputs needs one entry per test input, in order ({count})")\\n    problems, grids, named = [], [None] * count, []\\n    for index, entry in enumerate(entries):\\n        kind = next(iter(entry)) if isinstance(entry, dict) and len(entry) == 1 else None\\n        if kind not in SUBMIT_KINDS:\\n            problems.append(f"output {index + 1}: give exactly one of {\\\', \\\'.join(SUBMIT_KINDS)}")\\n        elif kind == "grid":\\n            grids[index] = normalize_grid(entry["grid"])\\n            if grids[index] is None:\\n                problems.append(f"output {index + 1}: the grid is not valid: {grid_problem(entry[\\\'grid\\\']) or \\\'not a list of rows of integers 0-9\\\'}")\\n        elif not (isinstance(entry[kind], str) and entry[kind].isidentifier() and not entry[kind].startswith("_")):\\n            problems.append(f"output {index + 1}: {kind} must be the name of a session {kind}")\\n        else:\\n            named.append([index, kind, entry[kind]])\\n    if problems:\\n        return refuse("; ".join(problems))\\n    failing = {}\\n    if named:\\n        result = run_session("", task["train"], [{"input": test["input"]} for test in task["test"]], timeout, session,\\n                             submit=named, helpers=helpers, ndimage=ndimage)\\n        if result is None:\\n            return refuse("evaluating it timed out or the worker process failed")\\n        problems = [f"output {index + 1}: {entry[\\\'problem\\\']}" for (index, _, _), entry in zip(named, result["entries"])\\n                    if entry["problem"]]\\n        if problems:\\n            return refuse("; ".join(problems))\\n        for (index, _, _), entry in zip(named, result["entries"]):\\n            grids[index] = entry["grid"]\\n        failing = {name: pairs for name, pairs in result["failing"].items() if pairs}\\n    if failing and not final and value != pending:\\n        lines = [f"{name} fails training pair{\\\'s\\\' if len(pairs) > 1 else \\\'\\\'} {\\\', \\\'.join(map(str, pairs))} of "\\n                 f"{len(task[\\\'train\\\'])}." for name, pairs in failing.items()]\\n        return {**record, "confirm": True}, "\\\\n".join(lines + [\\n            "To submit it anyway, call arc_submit_answer again with identical arguments; otherwise fix it first."])\\n    if previous is not None and grids == [normalize_grid(grid) for grid in previous] and not final and value != pending:\\n        return {**record, "confirm": True}, ("These outputs are the previous attempt\\\'s answer, and your final answer must differ "\\n                                             "from it. To submit it anyway, call arc_submit_answer again with identical "\\n                                             "arguments; otherwise change it first.")\\n    sources = ["grid" if kind == "grid" else f"{kind}:{name}" for kind, name in (next(iter(e.items())) for e in entries)]\\n    return {**record, "grids": grids, "sources": sources, "failing": failing}, None\\n\', \'src/agent/candidates.py\': \'"""Execution verification of one ARC program against every training pair and one test input.\\n\\nOptional D8 equivariance checks are available as a diagnostic, but are disabled as a hard gate by default because\\nsome ARC rules encode an absolute direction and are not D8-equivariant.\\n"""\\nfrom __future__ import annotations\\n\\nimport hashlib\\nfrom dataclasses import dataclass, field\\nfrom typing import Any\\n\\nfrom .executor import run_transforms, safety_error\\nfrom src.utils.grid import d8_transform\\n\\n\\nGrid = list[list[int]]\\n\\n\\n@dataclass\\nclass ProgramEvaluation:\\n    candidate_index: int\\n    code_sha256: str\\n    safety_error: str | None\\n    execution_failed: bool\\n    train_passed: int\\n    train_total: int\\n    train_exact: bool\\n    train_predictions: list[Grid | None]\\n    test_prediction: Grid | None\\n    d8_passed: int\\n    d8_total: int\\n    accepted: bool\\n    train_errors: list[str | None] = field(default_factory=list)\\n\\n    @property\\n    def d8_fraction(self) -> float:\\n        return self.d8_passed / self.d8_total if self.d8_total else 0.0\\n\\n\\ndef evaluate_program(code: str, task: dict[str, Any], test_index: int,\\n                     timeout: float = 5.0, d8_diagnostic: bool = False,\\n                     candidate_index: int = 0, min_d8_fraction: float = 0.0) -> ProgramEvaluation:\\n    """Evaluate one program on all training pairs and one test input."""\\n    if not 0 <= test_index < len(task.get("test", [])):\\n        raise IndexError("test_index is outside the task\\\'s test list")\\n    train_pairs = task.get("train", [])\\n    static_error = safety_error(code)\\n    digest = hashlib.sha256(code.encode("utf-8")).hexdigest()\\n    if static_error:\\n        return ProgramEvaluation(\\n            candidate_index=candidate_index,\\n            code_sha256=digest,\\n            safety_error=static_error,\\n            execution_failed=False,\\n            train_passed=0,\\n            train_total=len(train_pairs),\\n            train_exact=False,\\n            train_predictions=[],\\n            test_prediction=None,\\n            d8_passed=0,\\n            d8_total=0,\\n            accepted=False,\\n        )\\n\\n    inputs = [pair["input"] for pair in train_pairs] + [task["test"][test_index]["input"]]\\n    errors: list[str | None] = []\\n    outputs = run_transforms(code, inputs, timeout=timeout, errors=errors)\\n    if outputs is None:\\n        return ProgramEvaluation(\\n            candidate_index=candidate_index,\\n            code_sha256=digest,\\n            safety_error=None,\\n            execution_failed=True,\\n            train_passed=0,\\n            train_total=len(train_pairs),\\n            train_exact=False,\\n            train_predictions=[],\\n            test_prediction=None,\\n            d8_passed=0,\\n            d8_total=0,\\n            accepted=False,\\n        )\\n\\n    train_outputs = outputs[:-1]\\n    prediction = outputs[-1]\\n    train_passed = sum(output == pair["output"]\\n                       for output, pair in zip(train_outputs, train_pairs))\\n    train_total = len(train_pairs)\\n    train_exact = train_total > 0 and train_passed == train_total\\n\\n    d8_passed = 0\\n    d8_total = 0\\n    if d8_diagnostic and train_pairs:\\n        augmented_inputs = []\\n        augmented_expected = []\\n        for pair in train_pairs:\\n            for transform_index in range(1, 8):\\n                augmented_inputs.append(d8_transform(pair["input"], transform_index))\\n                augmented_expected.append(d8_transform(pair["output"], transform_index))\\n        augmented_outputs = run_transforms(code, augmented_inputs, timeout=timeout)\\n        d8_total = len(augmented_expected)\\n        if augmented_outputs is not None:\\n            d8_passed = sum(output == expected\\n                            for output, expected in zip(augmented_outputs, augmented_expected))\\n\\n    d8_fraction = d8_passed / d8_total if d8_total else 0.0\\n    d8_accepted = not d8_diagnostic or min_d8_fraction <= 0.0 or d8_fraction >= min_d8_fraction\\n    accepted = train_exact and prediction is not None and d8_accepted\\n    return ProgramEvaluation(\\n        candidate_index=candidate_index,\\n        code_sha256=digest,\\n        safety_error=None,\\n        execution_failed=False,\\n        train_passed=train_passed,\\n        train_total=train_total,\\n        train_exact=train_exact,\\n        train_predictions=train_outputs,\\n        test_prediction=prediction,\\n        d8_passed=d8_passed,\\n        d8_total=d8_total,\\n        accepted=accepted,\\n        train_errors=errors[:-1],\\n    )\\n\', \'src/agent/executor.py\': \'"""Execute model-written ARC transforms in a terminable worker process.\\n\\nThe verifier accepts a Python function such as ``transform(grid)`` and evaluates it on one\\nor more ARC grids. A strict AST gate rejects common filesystem, network, process, reflection,\\nand dynamic-code constructs. Each candidate runs in a separate Python process so a timeout\\ncan be terminated instead of leaving a live daemon thread behind.\\n\\nThe free tool (run_free_program) runs any program that passes the same gate, without requiring a function: the task\\\'s\\ngrids are predefined as train and test (inputs only), the trajectory\\\'s earlier successful programs run first (silently)\\nso their variables, functions and imports carry over, print output comes back (at most STDOUT_LIMIT characters), and a\\ntransform-like function, when the program defines one, is called on every training and test input. It also allows a few more pure\\ncomputation modules (FREE_IMPORTS) and json\\\'s string forms (dumps, loads).\\n\\nThe f3 tool (run_session) keeps the same session with four differences: train, test, checks (each verified function\\\'s\\nresult, kept by the caller) and the helpers objects and diff are given afresh to every program, so rebinding them lasts\\none call; the program runs one top-level statement at a time, so the statements before a failing one are known (the caller reruns them as history);\\nevery function it defines named transform, solve, transform<N> or solve<N> is run on every training and test input; and\\nthe gate allows a few more builtins and attributes and blocks a few more attributes (_F3_BLOCKED). With submit it runs no\\nprogram and evaluates arc_submit_answer\\\'s function and variable entries in the session instead. Its worker writes the\\nresult to a copy of fd 1 (the programs\\\' fd 1 is /dev/null), and its result\\\'s shape is checked (_session_shape).\\n\\nThe AST gate is advisory: it turns away the obvious routes to files, processes and interpreter internals, not every\\none Python has. The boundary for untrusted code is an OS-level sandbox or disposable container: configure(sandbox="bwrap")\\nstarts every worker in bubblewrap (no network, its own process tree, a private /tmp, and read-only access to the system\\nlibraries, the Python installation and this repository\\\'s src/ only, so data/ and the rest of the filesystem are out of\\nreach; optional). configure(memory_gb=...) caps each worker\\\'s address space (set on shared machines).\\nThe entry points call configure: src.agent.run (--executor-sandbox, --executor-memory-gb), src.qwen.runtime and\\nsrc.rl.miles.fnrl (their settings\\\' "executor": {"sandbox": ..., "memory_gb": ...}).\\n"""\\nfrom __future__ import annotations\\n\\nimport ast\\nimport builtins\\nimport collections\\nimport contextlib\\nimport functools\\nimport importlib\\nimport io\\nimport itertools\\nimport json\\nimport math\\nimport os\\nimport random\\nimport re\\nimport subprocess\\nimport sys\\nimport tempfile\\nfrom typing import Any, Sequence\\n\\nimport numpy as np\\n\\nif not __package__:  # the --worker subprocess runs this file as a script\\n    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))\\n\\nfrom src.utils.grid import normalize_grid\\n\\n\\n_FN_NAMES = ("transform", "solve", "p", "solve_task", "apply_rule")\\n_ALLOWED_IMPORTS = {\\n    "collections",\\n    "functools",\\n    "itertools",\\n    "math",\\n    "numpy",\\n}\\n# copy, re, heapq, bisect, statistics, fractions and json only compute; random is seeded per run. Not operator (attrgetter\\n# reaches underscore attributes), string (Formatter.get_field does too), typing (get_type_hints evaluates strings), sys.\\nFREE_IMPORTS = _ALLOWED_IMPORTS | {"bisect", "copy", "fractions", "heapq", "json", "random", "re", "statistics"}\\n# f3 with settings scipy_ndimage: scipy.ndimage (image morphology, labelling, filters; no file access) is also predefined\\n# as ndimage and importable, never scipy itself\\nNDIMAGE = "scipy.ndimage"\\n_FREE_ATTRIBUTES = {"dumps", "loads"}  # json\\\'s string forms; the file forms (dump, load) stay blocked\\n# f3 also allows type(x) (the one-argument form only, see visit_Call) and .remove / .replace (list and str methods), and\\n# blocks _F3_BLOCKED as attributes and imported names (not as plain names: none of these modules can be imported, so a\\n# variable called os or types is harmless): modules and attributes that lead to the OS, the interpreter, C pointers,\\n# frames and code objects. re.compile stays allowed (the builtin compile is not in the f3 builtins). This narrows the\\n# gate; it does not make it a boundary (see the module docstring).\\n_F3_ATTRIBUTES = {"remove", "replace"}\\n_F3_BLOCKED = {"os", "sys", "subprocess", "builtins", "codecs", "types", "f2py", "ctypes", "ctypeslib", "operator",\\n               "attrgetter", "methodcaller", "exec", "eval", "gi_frame", "gi_code", "cr_frame", "cr_code", "ag_frame",\\n               "ag_code", "f_back", "f_globals", "f_locals", "f_builtins", "f_code", "tb_frame"}\\n_F3_BUILTIN_NAMES = {"chr", "frozenset", "repr", "type"}  # and id, numbered in call order (_stable_id)\\nF3_FUNCTION = re.compile(r"(?:transform|solve)\\\\d*")  # the f3 functions verified when a program defines them\\n# generators seeded from the OS or the clock: a replay must print the same, so the free tool keeps the seeded module-level\\n# generators (and seeds a missing seed with 0, see _run_free)\\n_FREE_BLOCKED = {"BitGenerator", "Generator", "MT19937", "PCG64", "PCG64DXSM", "Philox", "Random", "RandomState",\\n                 "SFC64", "SeedSequence", "SystemRandom", "datetime64"}\\nSTDOUT_LIMIT = 4000\\n_ADDRESS = re.compile(r\\\' at 0x[0-9a-fA-F]+(?=>)\\\')  # object addresses differ per process; replays must match byte for byte\\n_BLOCKED_NAMES = {\\n    "breakpoint",\\n    "compile",\\n    "delattr",\\n    "dir",\\n    "eval",\\n    "exec",\\n    "exit",\\n    "getattr",\\n    "globals",\\n    "help",\\n    "input",\\n    "locals",\\n    "memoryview",\\n    "open",\\n    "quit",\\n    "setattr",\\n    "type",\\n    "vars",\\n}\\n_BLOCKED_ATTRIBUTES = {\\n    "chdir",\\n    "chmod",\\n    "connect",\\n    "ctypeslib",\\n    "DataSource",\\n    "dump",\\n    "dumps",\\n    "fromfile",\\n    "fromregex",\\n    "fromtextfile",\\n    "genfromtxt",\\n    "load",\\n    "loads",\\n    "loadtxt",\\n    "memmap",\\n    "mkdir",\\n    "open",\\n    "open_memmap",\\n    "popen",\\n    "remove",\\n    "rename",\\n    "replace",\\n    "rmdir",\\n    "save",\\n    "savetxt",\\n    "savez",\\n    "savez_compressed",\\n    "socket",\\n    "spawn",\\n    "system",\\n    "tofile",\\n    "unlink",\\n    "write",\\n}\\n_SAFE_BUILTIN_NAMES = {\\n    "Exception",\\n    "IndexError",\\n    "KeyError",\\n    "RuntimeError",\\n    "TypeError",\\n    "ValueError",\\n    "ZeroDivisionError",\\n    "abs",\\n    "all",\\n    "any",\\n    "bool",\\n    "dict",\\n    "divmod",\\n    "enumerate",\\n    "filter",\\n    "float",\\n    "int",\\n    "isinstance",\\n    "iter",\\n    "len",\\n    "list",\\n    "map",\\n    "max",\\n    "min",\\n    "next",\\n    "pow",\\n    "range",\\n    "reversed",\\n    "round",\\n    "set",\\n    "slice",\\n    "sorted",\\n    "str",\\n    "sum",\\n    "tuple",\\n    "zip",\\n}\\n\\n\\ndef parse_python_code(text: str) -> str | None:\\n    """Extract the most likely Python program from a model completion."""\\n    if not text:\\n        return None\\n    for tag in ("analysis", "think"):\\n        if text.lstrip().startswith(f"<{tag}>"):\\n            _, closing, text = text.partition(f"</{tag}>")\\n            if not closing:\\n                return None\\n    blocks = re.findall(r"```python\\\\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)\\n    if not blocks:\\n        blocks = [block for block in re.findall(r"```\\\\s*(.*?)```", text, re.DOTALL)\\n                  if "def " in block]\\n    if blocks:\\n        return max(blocks, key=len).strip()\\n    match = re.search(r"(def\\\\s+\\\\w+\\\\s*\\\\(.*)", text, re.DOTALL)\\n    return match.group(1).strip() if match else None\\n\\n\\nclass _SafetyVisitor(ast.NodeVisitor):\\n    def __init__(self, free: bool = False, f3: bool = False, ndimage: bool = False) -> None:\\n        self.error: str | None = None\\n        self.f3 = f3\\n        self.ndimage = ndimage\\n        self.imports = FREE_IMPORTS if free else _ALLOWED_IMPORTS\\n        self.attributes = (_BLOCKED_ATTRIBUTES - _FREE_ATTRIBUTES) | _FREE_BLOCKED if free else _BLOCKED_ATTRIBUTES\\n        self.names = _BLOCKED_NAMES\\n        if f3:\\n            self.attributes = (self.attributes - _F3_ATTRIBUTES - {"compile"}) | _F3_BLOCKED\\n\\n    def reject(self, message: str) -> None:\\n        if self.error is None:\\n            self.error = message\\n\\n    def visit_Import(self, node: ast.Import) -> None:\\n        for alias in node.names:\\n            if self.ndimage and alias.name == NDIMAGE:\\n                # bare "import scipy.ndimage" binds the whole scipy package (scipy.io reads and writes files)\\n                if alias.asname is None:\\n                    self.reject("import is not allowed: scipy.ndimage without a name; use from scipy import ndimage")\\n                continue\\n            root, *parts = alias.name.split(".")\\n            # f3: import numpy.f2py as x would bypass the attribute check\\n            if root not in self.imports or (self.f3 and any(part in self.attributes or part.startswith("_") for part in parts)):\\n                self.reject(f"import is not allowed: {alias.name}")\\n        self.generic_visit(node)\\n\\n    def visit_Call(self, node: ast.Call) -> None:\\n        # f3: type(x) with one positional argument; the name type anywhere else (type(name, bases, namespace), t = type)\\n        # stays blocked by visit_Name\\n        if (self.f3 and isinstance(node.func, ast.Name) and node.func.id == "type" and len(node.args) == 1\\n                and not node.keywords and not isinstance(node.args[0], ast.Starred)):\\n            self.visit(node.args[0])\\n            return\\n        self.generic_visit(node)\\n\\n    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:\\n        if self.ndimage and not node.level and (node.module == NDIMAGE or (node.module == "scipy" and node.names\\n                                                                         and all(a.name == "ndimage" for a in node.names))):\\n            for alias in node.names:\\n                if alias.name == "*" or alias.name.startswith("_") or alias.name in self.attributes:\\n                    self.reject(f"import is not allowed: {alias.name}")\\n            self.generic_visit(node)\\n            return\\n        root, *parts = (node.module or "").split(".")\\n        # f3: from numpy.ctypeslib import x would bypass the attribute check as import numpy.f2py would\\n        if node.level or root not in self.imports or (self.f3 and any(part in self.attributes or part.startswith("_") for part in parts)):\\n            self.reject(f"import is not allowed: {node.module}")\\n        for alias in node.names:  # from numpy import load would bypass the attribute check\\n            if alias.name == "*" or alias.name.startswith("_") or alias.name in self.attributes:\\n                self.reject(f"import is not allowed: {alias.name}")\\n        self.generic_visit(node)\\n\\n    def visit_Name(self, node: ast.Name) -> None:\\n        if node.id in self.names or node.id.startswith("__"):\\n            self.reject(f"name is not allowed: {node.id}")\\n\\n    def visit_Attribute(self, node: ast.Attribute) -> None:\\n        if node.attr.startswith("_") or node.attr in self.attributes:\\n            self.reject(f"attribute is not allowed: {node.attr}")\\n        self.generic_visit(node)\\n\\n    def visit_Global(self, node: ast.Global) -> None:\\n        self.reject("global declarations are not allowed")\\n\\n    def visit_Nonlocal(self, node: ast.Nonlocal) -> None:\\n        self.reject("nonlocal declarations are not allowed")\\n\\n\\ndef safety_error(code: str, free: bool = False, f3: bool = False, ndimage: bool = False) -> str | None:\\n    """Return a rejection reason, or ``None`` when the program passes the AST gate (free: the free tool\\\'s imports and\\n    attributes, and no function required; f3 with free: also type(x) and .remove / .replace, and _F3_BLOCKED blocked;\\n    ndimage: also scipy.ndimage, as from scipy import ndimage, from scipy.ndimage import ... or import scipy.ndimage as x)."""\\n    if not isinstance(code, str) or not code.strip():\\n        return "program is empty"\\n    try:\\n        tree = ast.parse(code)\\n        visitor = _SafetyVisitor(free, f3, ndimage)\\n        visitor.visit(tree)\\n    except SyntaxError as exc:\\n        return f"syntax error: {exc.msg}"\\n    except RecursionError:\\n        return "syntax tree exceeds the verifier recursion limit"\\n    if visitor.error:\\n        return visitor.error\\n    if free:\\n        return None\\n    functions = {node.name for node in tree.body\\n                 if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))}\\n    if not functions.intersection(_FN_NAMES):\\n        return f"program must define one of: {\\\', \\\'.join(_FN_NAMES)}"\\n    return None\\n\\n\\ndef _normalize(output: Any) -> list[list[int]] | None:\\n    return normalize_grid(output)\\n\\n\\ndef grid_problem(value: Any) -> str | None:\\n    """Why value is not an ARC grid (1-30 rows of the same 1-30 integers 0-9), e.g. "row 3 has 22 values, the others\\n    23"; None when no problem is found."""\\n    if hasattr(value, "tolist"):\\n        value = value.tolist()\\n    if isinstance(value, (list, tuple)):\\n        value = [row.tolist() if hasattr(row, "tolist") else row for row in value]\\n    if not isinstance(value, list) or not all(isinstance(row, (list, tuple)) for row in value):\\n        return "it is not a list of rows"\\n    if not value:\\n        return "it has no rows"\\n    if len(value) > 30:\\n        return f"it has {len(value)} rows (at most 30)"\\n    lengths = [len(row) for row in value]\\n    common = collections.Counter(lengths).most_common(1)[0][0]\\n    odd = [f"row {number} has {length} value{\\\'s\\\' if length != 1 else \\\'\\\'}" for number, length in enumerate(lengths, 1)\\n           if length != common]\\n    if odd:\\n        return ", ".join(odd[:5]) + (", ..." if len(odd) > 5 else "") + f", the others {common}"\\n    if not 0 < common <= 30:\\n        return f"its rows have {common} values (1 to 30)"\\n    for number, row in enumerate(value, 1):\\n        for cell in row:\\n            if isinstance(cell, np.generic):  # numpy scalars count as their Python value\\n                cell = cell.item()\\n            integral = (isinstance(cell, int) and not isinstance(cell, bool)) or (isinstance(cell, float) and cell.is_integer())\\n            if not integral or not 0 <= cell <= 9:\\n                # only plain values are shown as written: any other object\\\'s repr is its own code\\n                shown = repr(cell)[:40] if type(cell) in (int, float, bool, str, type(None)) else type(cell).__name__\\n                return f"row {number} holds {shown} (cells are integers 0-9)"\\n    return None\\n\\n\\ndef _restricted_import(name: str, globals_: Any = None, locals_: Any = None,\\n                       fromlist: Sequence[str] = (), level: int = 0, allowed: frozenset | set = _ALLOWED_IMPORTS,\\n                       ndimage: bool = False) -> Any:\\n    root = name.split(".", 1)[0]\\n    if ndimage and not level and (name == NDIMAGE or (name == "scipy" and fromlist and set(fromlist) == {"ndimage"})):\\n        return builtins.__import__(name, globals_, locals_, fromlist, level)\\n    if level or root not in allowed:\\n        raise ImportError(f"import is not allowed: {name}")\\n    return builtins.__import__(name, globals_, locals_, fromlist, level)\\n\\n\\ndef _stable_id():\\n    """id() numbered in call order: real addresses differ per process, and replays must print the same."""\\n    seen: dict[int, tuple[int, Any]] = {}\\n\\n    def stable(value: Any) -> int:\\n        # ponytail: keeps every object passed to id() alive so a number is never reused; fine for one call\\\'s worker\\n        return seen.setdefault(builtins.id(value), (len(seen), value))[0]\\n    return stable\\n\\n\\ndef _safe_builtins(free: bool = False, f3: bool = False, ndimage: bool = False) -> dict[str, Any]:\\n    """free: print writes to sys.stdout, which run_free_program captures; f3: also chr, frozenset, repr, type and id;\\n    ndimage: scipy.ndimage importable (see _restricted_import)."""\\n    namespace = {name: getattr(builtins, name) for name in (_SAFE_BUILTIN_NAMES | _F3_BUILTIN_NAMES if f3 else _SAFE_BUILTIN_NAMES)}\\n    namespace["__import__"] = (functools.partial(_restricted_import, allowed=FREE_IMPORTS, ndimage=ndimage) if free\\n                               else _restricted_import)\\n    namespace["print"] = builtins.print if free else lambda *args, **kwargs: None\\n    if f3:\\n        namespace["id"] = _stable_id()\\n    return namespace\\n\\n\\nclass _BoundedOutput(io.StringIO):\\n    """Keeps the first `limit` characters written and notes whether more came."""\\n\\n    def __init__(self, limit: int) -> None:\\n        super().__init__()\\n        self.limit, self.truncated = limit, False\\n\\n    def write(self, text: str) -> int:\\n        text = _ADDRESS.sub(" at <address>", text)\\n        room = self.limit - self.tell()\\n        if len(text) > room:\\n            self.truncated = True\\n        return super().write(text[:max(0, room)]) if room > 0 else 0\\n\\n\\ndef _error_line(exc: Exception) -> str:\\n    """Exception type, the model program\\\'s line number when the traceback reaches it, and a bounded message."""\\n    import traceback\\n\\n    lines = [frame.lineno for frame in traceback.extract_tb(exc.__traceback__) if frame.filename == "<model-program>"]\\n    message = _ADDRESS.sub(" at <address>", str(exc))\\n    where = f" (line {lines[-1]})" if lines else ""\\n    return f"{type(exc).__name__}{where}: {\\\' \\\'.join(message.split())[:240]}"\\n\\n\\ndef _execute_one(code: str, grid: list[list[int]]) -> list[list[int]] | None:\\n    namespace = {\\n        "__builtins__": _safe_builtins(),\\n        "collections": __import__("collections"),\\n        "functools": functools,\\n        "itertools": itertools,\\n        "math": math,\\n        "np": np,\\n        "numpy": np,\\n    }\\n    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n        exec(compile(code, "<model-transform>", "exec"), namespace)\\n        function = next((namespace[name] for name in _FN_NAMES\\n                         if callable(namespace.get(name))), None)\\n        if function is None:\\n            return None\\n        output = function([list(row) for row in grid])\\n    return _normalize(output)\\n\\n\\ndef _defines_function(code: str) -> bool:\\n    return any(isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name in _FN_NAMES\\n               for node in ast.parse(code).body)\\n\\n\\ndef _fresh(value: Any) -> Any:\\n    return json.loads(json.dumps(value))\\n\\n\\ndef _free_namespace(train: list[dict], test: list[dict], f3: bool = False, ndimage: bool = False) -> dict:\\n    """A free-tool session: the generators seeded (a missing seed becomes 0), the safe builtins and modules, train / test."""\\n    seed_random, seed_numpy, numpy_rng = random.seed, np.random.seed, np.random.default_rng\\n    random.seed = lambda a=None, version=2: seed_random(0 if a is None else a, version)\\n    np.random.seed = lambda seed=None: seed_numpy(0 if seed is None else seed)\\n    np.random.default_rng = lambda seed=None: numpy_rng(0 if seed is None else seed)\\n    random.seed(0)\\n    np.random.seed(0)\\n    return {"__builtins__": _safe_builtins(free=True, f3=f3, ndimage=ndimage), "collections": __import__("collections"),\\n            "functools": functools, "itertools": itertools, "math": math, "np": np, "numpy": np,\\n            "train": _fresh(train), "test": _fresh(test)}\\n\\n\\ndef _apply_function(function, grids: list) -> tuple[list, list]:\\n    """(outputs, errors) of function on each grid (a copy); prints are discarded."""\\n    outputs, errors = [], []\\n    for grid in grids:\\n        try:\\n            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n                result = _normalize(function([list(row) for row in grid]))\\n            outputs.append(result)\\n            errors.append(None if result is not None else "InvalidGrid: function did not return a valid ARC grid")\\n        except Exception as exc:\\n            outputs.append(None)\\n            errors.append(_error_line(exc))\\n    return outputs, errors\\n\\n\\ndef _run_free(code: str, train: list[dict], test: list[dict], history: Sequence[str] = ()) -> dict:\\n    """One free-tool run: the earlier programs (history) silently, then the program with train / test predefined and print\\n    captured, then, if the program defines a transform-like function, that function on every training and test input\\n    (prints there are discarded)."""\\n    namespace = _free_namespace(train, test)\\n    for earlier in history:\\n        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n            exec(compile(earlier, "<earlier-program>", "exec"), namespace)\\n    output, error = _BoundedOutput(STDOUT_LIMIT), None\\n    with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):\\n        try:\\n            exec(compile(code, "<model-program>", "exec"), namespace)\\n        except Exception as exc:\\n            error = _error_line(exc)\\n    function = None if error or not _defines_function(code) else next(\\n        (namespace[name] for name in _FN_NAMES if callable(namespace.get(name))), None)\\n    outputs = errors = None\\n    if function is not None:\\n        outputs, errors = _apply_function(function, [pair["input"] for pair in train] + [pair["input"] for pair in test])\\n    return {"ok": True, "stdout": output.getvalue(), "truncated": output.truncated, "error": error,\\n            "outputs": outputs, "errors": errors}\\n\\n\\ndef _plain(grid: Any) -> list[list]:\\n    """A grid (a list of rows or a numpy array) as lists of rows of Python values."""\\n    rows = grid.tolist() if isinstance(grid, np.ndarray) else grid\\n    return [[cell.item() if isinstance(cell, np.generic) else cell for cell in row] for row in rows]\\n\\n\\ndef objects(grid: Any, diagonal: bool = False, background: int | None = None) -> list[dict]:\\n    """f3 helper: the connected components of equal color (4-connected, or 8-connected with diagonal; background None\\n    keeps every color, an int skips that color) as {"color", "cells" ((row, col) in row-major order), "bbox" (top, left,\\n    bottom, right, inclusive), "size"}, in row-major order of their first cell."""\\n    grid = _plain(grid)\\n    steps = [(-1, 0), (1, 0), (0, -1), (0, 1)] + ([(-1, -1), (-1, 1), (1, -1), (1, 1)] if diagonal else [])\\n    seen, found = set(), []\\n    for r, row in enumerate(grid):\\n        for c, color in enumerate(row):\\n            if (r, c) in seen or color == background:\\n                continue\\n            seen.add((r, c))\\n            cells, stack = [], [(r, c)]\\n            while stack:\\n                y, x = stack.pop()\\n                cells.append((y, x))\\n                for near in ((y + dy, x + dx) for dy, dx in steps):\\n                    if (near not in seen and 0 <= near[0] < len(grid) and 0 <= near[1] < len(grid[near[0]])\\n                            and grid[near[0]][near[1]] == color):\\n                        seen.add(near)\\n                        stack.append(near)\\n            cells.sort()\\n            columns = [x for _, x in cells]\\n            found.append({"color": color, "cells": cells, "bbox": (cells[0][0], min(columns), cells[-1][0], max(columns)),\\n                          "size": len(cells)})\\n    return found\\n\\n\\ndef diff(a: Any, b: Any) -> str:\\n    """f3 helper: "same", "shape differs: HxW vs HxW", or how many cells differ and the first 50 as (row, col, a\\\'s value,\\n    b\\\'s value)."""\\n    a, b = _plain(a), _plain(b)\\n    if a == b:\\n        return "same"\\n    if [len(row) for row in a] != [len(row) for row in b]:\\n        shapes = [f"{len(grid)}x{len(grid[0]) if grid else 0}" + (" (rows of unequal length)" if len({*map(len, grid)}) > 1 else "")\\n                  for grid in (a, b)]\\n        return "shape differs: {} vs {}".format(*shapes)\\n    cells = [(r, c, x, y) for r, (row_a, row_b) in enumerate(zip(a, b)) for c, (x, y) in enumerate(zip(row_a, row_b)) if x != y]\\n    return (f"{len(cells)} of {sum(map(len, a))} cells differ, {\\\'the first 50 \\\' if len(cells) > 50 else \\\'\\\'}as "\\n            f"(row, col, a, b): {cells[:50]}")\\n\\n\\ndef _f3_session(train: list[dict], test: list[dict], session: Sequence, helpers: bool = True, ndimage: bool = False) -> dict:\\n    """The f3 session after the earlier calls (session: [program or None, its functions\\\' results] each): each program\\n    silently with fresh train / test, the checks so far and the helpers (objects, diff), the results then added to checks;\\n    then fresh train / test, the current checks and the helpers (none with helpers False)."""\\n    namespace, checks = _free_namespace(train, test, f3=True, ndimage=ndimage), {}\\n    given = {"objects": objects, "diff": diff} if helpers else {}\\n    if ndimage:\\n        given["ndimage"] = sys.modules[NDIMAGE]  # imported by the worker before its limits\\n    for earlier, functions in session:\\n        if earlier:\\n            namespace.update(train=_fresh(train), test=_fresh(test), checks=_fresh(checks), **given)\\n            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n                exec(compile(earlier, "<earlier-program>", "exec"), namespace)\\n        checks.update(functions)\\n    namespace.update(train=_fresh(train), test=_fresh(test), checks=_fresh(checks), **given)\\n    return namespace\\n\\n\\ndef _run_f3(code: str, train: list[dict], test: list[dict], session: Sequence, submit: list | None = None,\\n            call_seconds: int | None = None, helpers: bool = True, ndimage: bool = False) -> dict:\\n    """One f3 run (see run_session): the program one top-level statement at a time, then each transform / solve variant\\n    it defined before any failing statement on every training and test input; with submit, the entries instead. After\\n    the session\\\'s replay the CPU limit drops to call_seconds more, so the new program (or the submission) and its checks\\n    get one call\\\'s time whatever the replay left over."""\\n    namespace = _f3_session(train, test, session, helpers, ndimage)\\n    if call_seconds:\\n        _set_worker_limits(call_seconds, narrow=True)\\n    if submit is not None:\\n        return _run_submission(namespace, train, test, submit)\\n    tree = ast.parse(code)\\n    output, error, completed = _BoundedOutput(STDOUT_LIMIT), None, 0\\n    with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):\\n        try:\\n            for statement in tree.body:\\n                exec(compile(ast.Module([statement], []), "<model-program>", "exec"), namespace)\\n                completed += 1\\n        except Exception as exc:\\n            error = _error_line(exc)\\n    names = dict.fromkeys(node.name for node in tree.body[:completed]\\n                          if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and F3_FUNCTION.fullmatch(node.name))\\n    inputs = [pair["input"] for pair in train] + [pair["input"] for pair in test]\\n    functions = {name: dict(zip(("outputs", "errors"), _apply_function(namespace[name], inputs)))\\n                 for name in names if callable(namespace.get(name))}\\n    return {"ok": True, "stdout": output.getvalue(), "truncated": output.truncated, "error": error, "completed": completed,\\n            "functions": functions}\\n\\n\\ndef _run_submission(namespace: dict, train: list[dict], test: list[dict], entries: list) -> dict:\\n    """arc_submit_answer\\\'s entries ([test index, "function" | "variable", name]) in the session: per entry {"grid",\\n    "problem"}, and per function the training pairs (numbered from 1) it fails."""\\n    results, failing = [], {}\\n    for index, kind, name in entries:\\n        value, problem = namespace.get(name), None\\n        what = f"variable {name}" if kind == "variable" else f"the output of {name} for test input {index + 1}"\\n        if kind == "variable" and name not in namespace:\\n            problem = f"no variable named {name} is defined"\\n        elif kind == "function" and not callable(value):\\n            problem = f"no function named {name} is defined" if value is None else f"{name} is not a function"\\n        elif kind == "function":\\n            if name not in failing:\\n                outputs, _ = _apply_function(value, [pair["input"] for pair in train])\\n                failing[name] = [number for number, (actual, pair) in enumerate(zip(outputs, train), 1) if actual != pair["output"]]\\n            try:\\n                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n                    value = value([list(row) for row in test[index]["input"]])\\n            except Exception as exc:\\n                problem = f"{name} raised on test input {index + 1}: {_error_line(exc)}"\\n        grid = None\\n        try:  # the value can be any session object: reading it as a grid can run its code\\n            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):\\n                grid = None if problem else _normalize(value)\\n                if problem is None and grid is None:\\n                    problem = f"{what} is not a valid grid: {grid_problem(value) or \\\'not a list of rows of integers 0-9\\\'}"\\n        except Exception as exc:\\n            grid, problem = None, f"{what} could not be read as a grid: {_error_line(exc)}"\\n        results.append({"grid": grid, "problem": problem})\\n    return {"ok": True, "entries": results, "failing": failing}\\n\\n\\ndef _set_worker_limits(cpu_seconds: int, memory_bytes: int | None = None, narrow: bool = False) -> None:\\n    """narrow (f3, after the session\\\'s replay): the CPU limit never rises above the one already set."""\\n    with contextlib.suppress(ImportError, OSError, ValueError):\\n        import resource\\n\\n        if memory_bytes:\\n            resource.setrlimit(resource.RLIMIT_AS, (memory_bytes, memory_bytes))\\n        usage = resource.getrusage(resource.RUSAGE_SELF)\\n        consumed = usage.ru_utime + usage.ru_stime\\n        soft = _absolute_cpu_limit(cpu_seconds, consumed)\\n        current = resource.getrlimit(resource.RLIMIT_CPU)[0]\\n        if narrow and current != resource.RLIM_INFINITY:\\n            soft = min(soft, current)\\n        resource.setrlimit(resource.RLIMIT_CPU, (soft, soft + 1))\\n        resource.setrlimit(resource.RLIMIT_FSIZE, (0, 0))\\n        resource.setrlimit(resource.RLIMIT_NOFILE, (32, 32))\\n\\n\\ndef _absolute_cpu_limit(cpu_seconds: int, consumed_seconds: float) -> int:\\n    """Convert a per-candidate CPU allowance to RLIMIT_CPU\\\'s absolute limit."""\\n    return max(1, math.ceil(consumed_seconds + max(1, cpu_seconds)))\\n\\n\\ndef _worker_main() -> int:\\n    try:\\n        payload = json.load(sys.stdin)\\n        code = payload["code"]\\n        mode = payload.get("mode")\\n        free, f3 = mode in ("free", "f3"), mode == "f3"\\n        earlier = [program for program, _ in payload["session"] if program] if f3 else payload.get("history", [])\\n        programs = [*earlier, *([] if "submit" in payload else [code])]\\n        ndimage = bool(f3 and payload.get("ndimage"))\\n        error = next(filter(None, (safety_error(program, free, f3, ndimage) for program in programs)), None)\\n        if error:\\n            json.dump({"ok": False, "error": error}, sys.stdout)\\n            return 2\\n        if ndimage:  # before the limits, so no program pays scipy\\\'s import\\n            try:\\n                importlib.import_module(NDIMAGE)\\n            except Exception:  # a scipy that does not import leaves the programs without ndimage, never fails the worker\\n                ndimage = False\\n        _set_worker_limits(int(payload.get("cpu_seconds", 5)), payload.get("memory_bytes"))\\n        if f3:\\n            # nothing the programs write (fd 1 included) can reach the result: it goes to a copy of fd 1, and fd 1 and\\n            # sys.stdout point at /dev/null before any program runs; os._exit skips the objects\\\' finalizers\\n            result = os.dup(1)\\n            os.dup2(os.open(os.devnull, os.O_WRONLY), 1)\\n            sys.stdout = open(os.devnull, "w")\\n            output = _run_f3(code, payload["train"], payload["test"], payload["session"], payload.get("submit"),\\n                             payload.get("call_seconds"), payload.get("helpers", True), ndimage)\\n            with os.fdopen(result, "w") as stream:\\n                json.dump(output, stream)\\n                stream.flush()\\n            os._exit(0)\\n        if free:\\n            json.dump(_run_free(code, payload["train"], payload["test"], payload.get("history", [])), sys.stdout)\\n            return 0\\n        grids = payload["grids"]\\n        outputs = []\\n        errors = []\\n        for grid in grids:\\n            try:\\n                output = _execute_one(code, grid)\\n                outputs.append(output)\\n                errors.append(None if output is not None else "InvalidGrid: function did not return a valid ARC grid")\\n            except Exception as exc:\\n                outputs.append(None)\\n                message = _ADDRESS.sub(" at <address>", str(exc))\\n                errors.append(f"{type(exc).__name__}: {\\\' \\\'.join(message.split())[:240]}")\\n        json.dump({"ok": True, "outputs": outputs, "errors": errors}, sys.stdout)\\n        return 0\\n    except Exception as exc:\\n        json.dump({"ok": False, "error": f"{type(exc).__name__}: {exc}"}, sys.stdout)\\n        return 3\\n\\n\\ndef run_transforms(code: str, grids: Sequence[list[list[int]]],\\n                   timeout: float = 5.0, *, errors: list[str | None] | None = None) -> list[list[list[int]] | None] | None:\\n    """Execute a program on several grids in one terminable worker.\\n\\n    The program namespace is recreated for every grid, preventing a candidate from depending on\\n    train-pair execution order. ``None`` means the worker failed or timed out; individual entries\\n    can also be ``None`` when a call raised or returned an invalid ARC grid. If supplied,\\n    ``errors`` receives one bounded diagnostic per call, without changing predictions.\\n    """\\n    if errors is not None:\\n        errors.clear()\\n    if safety_error(code) is not None or timeout <= 0:\\n        return None\\n    clean_grids = []\\n    for grid in grids:\\n        normalized = _normalize(grid)\\n        if normalized is None:\\n            return None\\n        clean_grids.append(normalized)\\n    result = _run_worker({"code": code, "grids": clean_grids, "cpu_seconds": max(1, math.ceil(timeout))}, timeout)\\n    if (result is None or not result.get("ok") or not isinstance(result.get("outputs"), list)\\n            or len(result["outputs"]) != len(clean_grids)):\\n        return None\\n    if errors is not None:\\n        call_errors = result.get("errors")\\n        if (not isinstance(call_errors, list) or len(call_errors) != len(clean_grids)\\n                or any(error is not None and not isinstance(error, str) for error in call_errors)):\\n            return None\\n        errors.extend(call_errors)\\n    return result["outputs"]\\n\\n\\ndef run_free_program(code: str, train: list[dict], test: list[dict], timeout: float = 5.0,\\n                     history: Sequence[str] = ()) -> dict | None:\\n    """The free tool\\\'s run in a terminable worker: {"stdout", "truncated", "error" (the program raised; a line number\\n    when the traceback reaches it), "outputs" / "errors" (per training then test input, None unless the program defines a\\n    transform-like function)}. history: the trajectory\\\'s earlier programs that ran without error, rerun first so their\\n    state carries over; each one adds a timeout to the allowance, so a program that fit its own call still fits when\\n    rerun and the new program always keeps a full timeout. test must hold inputs only. None means the gate rejected it,\\n    or the worker failed or timed out."""\\n    if (safety_error(code, free=True) is not None or timeout <= 0 or any(set(pair) != {"input"} for pair in test)\\n            or any(safety_error(program, free=True) for program in history)):\\n        return None\\n    allowance = timeout * (1 + len(history))\\n    result = _run_worker({"code": code, "mode": "free", "train": train, "test": test, "history": list(history),\\n                          "cpu_seconds": max(1, math.ceil(allowance))}, allowance)\\n    count = len(train) + len(test)\\n    if (result is None or not result.get("ok") or not isinstance(result.get("stdout"), str)\\n            or (result.get("outputs") is not None and len(result["outputs"]) != count)):\\n        return None\\n    return result\\n\\n\\ndef run_session(code: str, train: list[dict], test: list[dict], timeout: float = 5.0, session: Sequence = (),\\n                submit: list | None = None, helpers: bool = True, ndimage: bool = False) -> dict | None:\\n    """The f3 tool\\\'s run in a terminable worker. session: the trajectory\\\'s earlier calls as [program or None, its\\n    functions\\\' results ({name: {"train_passed", "train_total", "test_outputs"}})]; each program reruns first with fresh\\n    train / test and checks (the results so far). Returns {"stdout", "truncated", "error", "completed" (top-level\\n    statements that ran), "functions" ({name: {"outputs", "errors"}} per training then test input, for every transform /\\n    solve variant defined before any failing statement)}. With submit ([[test index, "function" | "variable", name],\\n    ...]) code does not run: {"entries": [{"grid", "problem"}], "failing": {function: [training pairs it fails]}}. The\\n    allowance and the None cases are run_free_program\\\'s, except that the new program (or the submission) and its checks\\n    get one timeout of CPU time whatever the replay left over; a result of any other shape is None as well. helpers:\\n    objects / diff predefined (_f3_session)."""\\n    earlier = [program for program, _ in session if program]\\n    if (timeout <= 0 or any(set(pair) != {"input"} for pair in test)\\n            or any(safety_error(program, True, True, ndimage) for program in earlier + ([] if submit is not None else [code]))):\\n        return None\\n    allowance = timeout * (1 + len(earlier))\\n    result = _run_worker({"code": code, "mode": "f3", "train": train, "test": test, "session": [list(entry) for entry in session],\\n                          **({"submit": submit} if submit is not None else {}), "cpu_seconds": max(1, math.ceil(allowance)),\\n                          "call_seconds": max(1, math.ceil(timeout)), "helpers": helpers, "ndimage": ndimage}, allowance)\\n    return result if _session_shape(result, code, len(train) + len(test), submit) else None\\n\\n\\ndef _session_shape(result: Any, code: str, count: int, submit: list | None) -> bool:\\n    """Whether a worker\\\'s f3 result has the shape run_session returns (the worker runs model code, so it is checked)."""\\n    grid = lambda value: value is None or normalize_grid(value) is not None  # noqa: E731\\n    text = lambda value: value is None or isinstance(value, str)  # noqa: E731\\n    if not isinstance(result, dict) or result.get("ok") is not True:\\n        return False\\n    if submit is not None:\\n        entries, failing = result.get("entries"), result.get("failing")\\n        return (isinstance(entries, list) and len(entries) == len(submit)\\n                and all(isinstance(entry, dict) and set(entry) == {"grid", "problem"} and grid(entry["grid"])\\n                        and text(entry["problem"]) and (entry["grid"] is None) != (entry["problem"] is None) for entry in entries)\\n                and isinstance(failing, dict) and all(isinstance(pairs, list) for pairs in failing.values()))\\n    completed, functions = result.get("completed"), result.get("functions")\\n    return (isinstance(result.get("stdout"), str) and isinstance(result.get("truncated"), bool) and text(result.get("error"))\\n            and type(completed) is int and 0 <= completed <= len(ast.parse(code).body)\\n            and isinstance(functions, dict)\\n            and all(isinstance(run, dict) and set(run) == {"outputs", "errors"}\\n                    and all(isinstance(run[key], list) and len(run[key]) == count for key in run)\\n                    and all(map(grid, run["outputs"])) and all(map(text, run["errors"])) for run in functions.values()))\\n\\n\\n_CONFIG = {"sandbox": None, "memory_gb": None}\\n\\n\\ndef configure(sandbox: str | None = None, memory_gb: float | None = None) -> None:\\n    """The process\\\'s worker settings (see the module docstring): sandbox None or "bwrap", memory_gb None or a cap."""\\n    if sandbox not in (None, "bwrap"):\\n        raise ValueError(f"Unknown executor sandbox {sandbox!r}")\\n    if memory_gb is not None and not memory_gb > 0:\\n        raise ValueError(f"executor memory_gb must be positive, got {memory_gb!r}")\\n    _CONFIG.update(sandbox=sandbox, memory_gb=memory_gb)\\n\\n\\ndef _sandbox(command: list[str], worker_path: str) -> list[str]:\\n    """command inside bubblewrap when configured with sandbox "bwrap" (see the module docstring), else unchanged."""\\n    if not _CONFIG["sandbox"]:\\n        return command\\n    arguments = ["bwrap", "--unshare-all", "--die-with-parent", "--new-session", "--proc", "/proc", "--dev", "/dev",\\n                 "--tmpfs", "/tmp", "--ro-bind", "/usr", "/usr", "--ro-bind-try", "/etc/ld.so.cache", "/etc/ld.so.cache"]\\n    for directory in ("/bin", "/lib", "/lib32", "/lib64", "/sbin"):\\n        if os.path.islink(directory):\\n            arguments += ["--symlink", os.readlink(directory), directory]\\n        elif os.path.isdir(directory):\\n            arguments += ["--ro-bind", directory, directory]\\n    source = os.path.dirname(os.path.dirname(worker_path))  # .../src: the worker imports src.utils.grid only\\n    # the interpreter\\\'s own tree (a uv or conda base and the venv, each under its own path and its resolved one) and src/;\\n    # not the base\\\'s parent directory, which can be the home directory\\n    bound = sorted({os.path.realpath(sys.base_prefix), sys.base_prefix, os.path.realpath(sys.prefix), sys.prefix, source} - {"/"})\\n    for path in bound:\\n        arguments += ["--ro-bind", path, path]\\n    for target, link in _outside_links(command[0], bound + ["/usr", "/bin", "/lib", "/lib32", "/lib64", "/sbin"]):\\n        arguments += ["--symlink", target, link]\\n    return arguments + ["--chdir", "/tmp", "--"] + command\\n\\n\\ndef _outside_links(path: str, bound: list[str]) -> list[tuple[str, str]]:\\n    """(target, link) for every symlink on the way from path to the file it names that lies outside the bound\\n    directories, so the sandbox can recreate it: a uv venv\\\'s python names its base through a minor-version link\\n    (cpython-3.12-linux-x86_64-gnu -> cpython-3.12.15-linux-x86_64-gnu) that the bound real base does not contain."""\\n    links, seen = [], set()\\n    while path not in seen:\\n        seen.add(path)\\n        prefix, parts = "/", path.strip("/").split("/")\\n        for index, part in enumerate(parts):\\n            prefix = os.path.join(prefix, part)\\n            if os.path.islink(prefix):\\n                target = os.readlink(prefix)\\n                if not any(prefix == base or prefix.startswith(base + os.sep) for base in bound):\\n                    links.append((target, prefix))\\n                path = os.path.normpath(os.path.join(os.path.dirname(prefix), target, *parts[index + 1:]))\\n                break\\n        else:\\n            break\\n    return links\\n\\n\\n_WORKER_ENVIRONMENT = {\\n    "PYTHONHASHSEED": "0",\\n    "PYTHONIOENCODING": "utf-8",\\n    "BLIS_NUM_THREADS": "1",\\n    "GOTO_NUM_THREADS": "1",\\n    "MKL_NUM_THREADS": "1",\\n    "NUMEXPR_NUM_THREADS": "1",\\n    "OMP_NUM_THREADS": "1",\\n    "OPENBLAS_NUM_THREADS": "1",\\n    "VECLIB_MAXIMUM_THREADS": "1",\\n}\\n\\n\\ndef ndimage_error(timeout: float = 120.0) -> str | None:\\n    """settings scipy_ndimage\\\'s check before a run: None when this interpreter imports scipy.ndimage in a worker\\\'s\\n    environment and sandbox, else the last line of why it does not."""\\n    try:\\n        done = subprocess.run(_sandbox([sys.executable, "-c", f"import {NDIMAGE}"], os.path.abspath(__file__)),\\n                              capture_output=True, text=True, env=_WORKER_ENVIRONMENT, timeout=timeout, check=False)\\n    except (OSError, subprocess.SubprocessError) as error:\\n        return f"{type(error).__name__}: {error}"[:300]\\n    if done.returncode == 0:\\n        return None\\n    return (done.stderr.strip().splitlines() or [f"exit code {done.returncode}"])[-1][:300]\\n\\n\\ndef _run_worker(request: dict, timeout: float) -> dict | None:\\n    memory = _CONFIG["memory_gb"]\\n    payload = json.dumps({**request, **({"memory_bytes": int(memory * 2**30)} if memory else {})})\\n    worker_path = os.path.abspath(__file__)\\n    try:\\n        with tempfile.TemporaryDirectory(prefix="arc-codeexec-") as workdir:\\n            completed = subprocess.run(\\n                _sandbox([sys.executable, worker_path, "--worker"], worker_path),\\n                input=payload,\\n                text=True,\\n                capture_output=True,\\n                cwd=workdir,\\n                env=_WORKER_ENVIRONMENT,\\n                timeout=timeout + 1.0,\\n                check=False,\\n            )\\n    except (OSError, subprocess.TimeoutExpired):\\n        return None\\n    if completed.returncode != 0 or not completed.stdout:\\n        return None\\n    try:\\n        return json.loads(completed.stdout)\\n    except json.JSONDecodeError:\\n        return None\\n\\n\\nif __name__ == "__main__" and "--worker" in sys.argv:\\n    raise SystemExit(_worker_main())\\n\', \'src/agent/model_inventory.py\': \'"""Hash local model files and validate safetensors index and payload coverage."""\\n\\nimport argparse\\nfrom datetime import datetime, timezone\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\n\\nfrom src.utils.tensors import safetensors_header\\n\\ndef digest(path: Path) -> str:\\n    with path.open("rb") as handle:\\n        return hashlib.file_digest(handle, "sha256").hexdigest()\\n\\n\\ndef inspect_model(model: Path) -> dict:\\n    model = model.resolve()\\n    mapping = json.loads((model / "model.safetensors.index.json").read_text())["weight_map"]\\n    shards = sorted(set(mapping.values()))\\n    seen = {}\\n    files, stats = {}, {}\\n    for name in shards:\\n        if Path(name).name != name:\\n            raise ValueError("model index contains a nonlocal shard path")\\n        path = model / name\\n        before = path.stat()\\n        header, data_start = safetensors_header(path)\\n        spans = []\\n        for tensor, value in header.items():\\n            if tensor == "__metadata__":\\n                continue\\n            if tensor in seen or mapping.get(tensor) != name:\\n                raise ValueError("duplicate or incorrectly indexed tensor")\\n            start, end = value["data_offsets"]\\n            if not 0 <= start < end <= before.st_size - data_start:\\n                raise ValueError("tensor payload is empty or truncated")\\n            spans.append((start, end))\\n            seen[tensor] = name\\n        spans.sort()\\n        if (not spans or spans[0][0] != 0 or spans[-1][1] != before.st_size - data_start\\n                or any(a[1] != b[0] for a, b in zip(spans, spans[1:]))):\\n            raise ValueError("tensor payload coverage is not contiguous")\\n        checksum = digest(path)\\n        after = path.stat()\\n        if (before.st_size, before.st_mtime_ns, before.st_ctime_ns) != (after.st_size, after.st_mtime_ns, after.st_ctime_ns):\\n            raise ValueError("model shard changed during inspection")\\n        files[name] = {"size_bytes": before.st_size, "sha256": checksum}\\n        stats[name] = {"size_bytes": before.st_size, "mtime_ns": before.st_mtime_ns, "ctime_ns": before.st_ctime_ns}\\n    if seen != mapping:\\n        raise ValueError("model index coverage differs from tensors")\\n    for path in model.iterdir():\\n        if path.is_file() and (path.suffix in (".json", ".jinja") or path.name in ("merges.txt", "LICENSE", "USAGE_POLICY", "README.md")):\\n            files[path.name] = {"size_bytes": path.stat().st_size, "sha256": digest(path)}\\n    return {"status": "PASS", "model": str(model), "created_utc": datetime.now(timezone.utc).isoformat(),\\n            "shards": len(shards), "tensors": len(seen), "weight_bytes": sum(files[name]["size_bytes"] for name in shards),\\n            "files": files, "shard_stats": stats, "source_sha256": digest(Path(__file__)),\\n            "scope": "Local file integrity and complete index/payload bounds; not vendor authentication, GPU execution, model accuracy or L4 deployment evidence."}\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--model", type=Path, required=True)\\n    parser.add_argument("--output", type=Path, required=True)\\n    args = parser.parse_args()\\n    if args.output.exists():\\n        parser.error("inventory output exists")\\n    result = inspect_model(args.model)\\n    with args.output.open("x") as handle:\\n        handle.write(json.dumps(result, indent=2) + "\\\\n")\\n    print(json.dumps({key: result[key] for key in ("status", "model", "shards", "tensors", "weight_bytes")}))\\n\', \'src/agent/evaluate.py\': \'"""Run every trajectory of a frozen plan\\\'s evaluation (run_arm), replay it without labels (replay_arm), and score."""\\n\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom datetime import datetime, timezone\\nimport json\\nfrom pathlib import Path\\n\\nfrom src.utils.prompt import second_view\\nfrom .loop import replay_trajectory, trajectory, write_json\\nfrom .model_inventory import digest\\nfrom .prompt import initial_messages\\nfrom .protocol import load_native_tokenizer\\n\\n\\ndef jobs_for(tasks: dict, settings: dict) -> list[dict]:\\n    """Jobs in task order; job_order "attempt_major" runs every output\\\'s first attempt before any second attempt\\n    with the same keys and seeds. job_unit "output" (default) runs each test output\\\'s attempts on its own; "task" runs\\n    each task\\\'s attempts once over all its test inputs (test_index None). With second_view (agent_mode free), attempts\\n    after the first see the task under the fixed D8 symmetry of src.utils.prompt.second_view, as the direct runs do."""\\n    order = settings.get("job_order", "task_major")\\n    if order not in ("task_major", "attempt_major"):\\n        raise ValueError(f"Unknown job order {order!r}")\\n    unit = settings.get("job_unit", "output")\\n    if unit not in ("output", "task"):\\n        raise ValueError(f"Unknown job unit {unit!r}")\\n    if settings.get("second_view") and settings.get("agent_mode") != "free":\\n        raise ValueError("second_view needs agent_mode free")\\n    jobs = []\\n    for task_id, task in tasks.items():\\n        for index in range(len(task["test"])) if unit == "output" else [None]:\\n            initial_messages(task, index, settings.get("agent_mode", "code"), settings.get("tool", "transform"),\\n                             settings.get("helpers", True), settings.get("permissive", False))\\n            for attempt in range(settings.get("attempts", 2)):\\n                jobs.append({"key": f"{len(jobs):04d}", "task_id": task_id, "test_index": index,\\n                             "attempt": attempt, "seed": settings["seed"] + len(jobs) * settings["max_rounds"]})\\n                if attempt and settings.get("second_view"):\\n                    jobs[-1]["view"] = second_view(task_id, 0 if index is None else index)\\n    if order == "attempt_major":\\n        jobs.sort(key=lambda job: job["attempt"])  # stable: task order within each attempt\\n    return jobs\\n\\n\\ndef shard_jobs(jobs: list[dict], output: Path, index: int, count: int) -> list[dict]:\\n    """Jobs with key % count == index and no finished result.json; an interrupted or ERROR attempt directory is\\n    moved aside so the attempt reruns with its original key and seed."""\\n    pending = []\\n    for job in jobs:\\n        directory = output / job["key"]\\n        if int(job["key"]) % count != index:\\n            continue\\n        if (directory / "result.json").exists() and json.loads((directory / "result.json").read_text())["status"] != "ERROR":\\n            continue\\n        if directory.exists():\\n            directory.rename(output / f"{job[\\\'key\\\']}.interrupted-{datetime.now(timezone.utc):%Y%m%dT%H%M%S}")\\n        pending.append(job)\\n    return pending\\n\\n\\ndef run_arm(plan: dict, arm: str, output: Path, client, shard: tuple[int, int] | None = None) -> None:\\n    """Without shard, run every job into a new directory. With shard (index, count), run only that shard\\\'s unfinished\\n    jobs inside an existing directory and leave aggregation and replay to the merge step."""\\n    settings = plan["evaluation"]\\n    tasks = json.loads(Path(plan["challenges"]).read_text())\\n    tokenizer = load_native_tokenizer(plan["model"], settings.get("native_format", "qwen"))\\n    jobs = jobs_for(tasks, settings)\\n    output.mkdir(exist_ok=shard is not None)\\n    model = plan["served_models"][arm]\\n    if shard is None:\\n        write_json(output / "inputs.json", {"tasks": tasks, "settings": settings, "model": model, "jobs": jobs})\\n    else:\\n        jobs = shard_jobs(jobs, output, *shard)\\n    with ThreadPoolExecutor(max_workers=settings["concurrency"]) as pool:\\n        futures = [pool.submit(trajectory, job, tasks[job["task_id"]], output, model, settings, tokenizer, client) for job in jobs]\\n        results = [future.result() for future in futures]\\n    if shard is not None:\\n        if any(row["status"] == "ERROR" for row in results):\\n            raise RuntimeError("Native tool shard has infrastructure failures; rerun the shard to retry them")\\n        return\\n    write_json(output / "schedule.json", {"status": "ERROR" if any(r["status"] == "ERROR" for r in results) else "COMPLETE", "results": results})\\n    if any(row["status"] == "ERROR" for row in results):\\n        raise RuntimeError("Native tool evaluation has infrastructure failures; do not score")\\n    write_json(output / "predictions.json", aggregate(tasks, results))\\n\\n\\ndef replay_arm(plan: dict, arm: str, output: Path) -> dict:\\n    """Rebuild every conversation and rerun code without reading solution labels."""\\n    before = {str(p.relative_to(output)): digest(p) for p in output.rglob("*.json")}\\n    tasks = json.loads(Path(plan["challenges"]).read_text())\\n    settings = plan["evaluation"]\\n    tokenizer = load_native_tokenizer(plan["model"], settings.get("native_format", "qwen"))\\n    jobs = jobs_for(tasks, settings)\\n    model = plan["served_models"][arm]\\n    expected_inputs = {"tasks": tasks, "settings": settings, "model": model, "jobs": jobs}\\n    if json.loads((output / "inputs.json").read_text()) != expected_inputs:\\n        raise ValueError("Frozen evaluation input differs")\\n    schedule = json.loads((output / "schedule.json").read_text())\\n    if schedule["status"] != "COMPLETE" or len(schedule["results"]) != len(jobs):\\n        raise ValueError("Evaluation is incomplete")\\n    results, generated_tokens, total_requests = [], 0, 0\\n    for job, receipt in zip(jobs, schedule["results"]):\\n        directory = output / job["key"]\\n        checked = replay_trajectory(job, receipt, tasks[job["task_id"]], settings, model, directory, tokenizer)\\n        total_requests += checked["requests"]\\n        generated_tokens += checked["generated_tokens"]\\n        results.append(receipt)\\n    predictions = aggregate(tasks, results)\\n    if json.loads((output / "predictions.json").read_text()) != predictions:\\n        raise ValueError("Aggregate predictions differ")\\n    if before != {str(p.relative_to(output)): digest(p) for p in output.rglob("*.json")}:\\n        raise ValueError("Evaluation artifacts changed during replay")\\n    return {"status": "ALL_NATIVE_TOOL_TRAJECTORIES_REPLAYED", "labels_loaded": False,\\n            "finished_utc": datetime.now(timezone.utc).isoformat(), "artifact_hashes": before,\\n            "requests": total_requests, "generated_tokens": generated_tokens, "scoring_submission": predictions}\\n\\n\\ndef aggregate(tasks: dict, results: list[dict]) -> dict:\\n    """attempt_k of each output: attempt k - 1\\\'s grid. A task with a row holding program_predictions (tool "f3",\\n    src.agent.loop.program_predictions) fills its outputs as src.qwen.runtime.assemble does: each attempt\\\'s grid then\\n    its program output, in attempt order, repeats removed, the first two (None where fewer remain)."""\\n    scoring = {key: [{"attempt_1": None, "attempt_2": None} for _ in task["test"]] for key, task in tasks.items()}\\n    for row in results:\\n        rows = scoring[row["task_id"]]\\n        if row["test_index"] is not None:\\n            rows[row["test_index"]][f"attempt_{row[\\\'attempt\\\'] + 1}"] = row["prediction"]\\n        elif row["prediction"] is not None:\\n            for entry, grid in zip(rows, row["prediction"], strict=True):\\n                entry[f"attempt_{row[\\\'attempt\\\'] + 1}"] = grid\\n    for task_id in {row["task_id"] for row in results if row.get("program_predictions") and row["test_index"] is None}:\\n        ordered = sorted((row for row in results if row["task_id"] == task_id and row["test_index"] is None),\\n                         key=lambda row: row["attempt"])\\n        for index, entry in enumerate(scoring[task_id]):\\n            found = []\\n            for row in ordered:\\n                for grids in (row["prediction"], row.get("program_predictions")):\\n                    grid = grids[index] if grids and index < len(grids) else None\\n                    if grid is not None and grid not in found:\\n                        found.append(grid)\\n            found += [None, None]\\n            entry.update(attempt_1=found[0], attempt_2=found[1])\\n    return scoring\\n\\n\\ndef train_cell_share(task: dict, outputs: list) -> float:\\n    """Mean share of matching cells over the training pairs (0 for a pair whose output shape differs)."""\\n    def share(actual, expected):\\n        if not isinstance(actual, list) or [len(r) for r in actual] != [len(r) for r in expected]:\\n            return 0.0\\n        return sum(a == e for got, want in zip(actual, expected) for a, e in zip(got, want)) / sum(map(len, expected))\\n    return sum(share(actual, pair["output"]) for actual, pair in zip(outputs, task["train"])) / len(task["train"])\\n\\n\\ndef fallback_grids(task: dict, directory: Path, test_index: int | None) -> list[tuple]:\\n    """(rank, test input, grid) for every test grid a trajectory\\\'s recorded rounds produced, unfinished trajectories\\n    included. A verified turn ranks first; then more training pairs passed, then a higher training cell share. With\\n    test_index None (job_unit "task") an unverified grid belongs to the first test input: evaluate_message stops\\n    there, since a program with a test grid that is not accepted failed the training pairs."""\\n    found = []\\n    for path in sorted(directory.glob("[0-9][0-9]_execution.json")):\\n        turn = json.loads(path.read_text())\\n        if turn["prediction"] is not None:\\n            indices = range(len(task["test"])) if test_index is None else [test_index]\\n            grids = turn["prediction"] if test_index is None else [turn["prediction"]]\\n            found += [((2, 0, 0.0), index, grid) for index, grid in zip(indices, grids)]\\n        for entry in turn["evaluations"]:\\n            evaluation = entry["evaluation"]\\n            if evaluation and not evaluation["accepted"] and evaluation["test_prediction"] is not None:\\n                rank = (1, evaluation["train_passed"], train_cell_share(task, evaluation["train_predictions"]))\\n                found.append((rank, test_index or 0, evaluation["test_prediction"]))\\n    return found\\n\\n\\ndef fill_empty(predictions: dict, fallback: dict) -> int:\\n    """Fill attempts without a verified grid from fallback[task_id] (fallback_grids rows), best first, never\\n    repeating a grid already in that output\\\'s slots. Returns the number of filled slots."""\\n    filled = 0\\n    for task_id, rows in predictions.items():\\n        ranked = sorted(fallback.get(task_id, []), key=lambda row: row[0], reverse=True)\\n        for index, slots in enumerate(rows):\\n            for attempt in slots:\\n                if slots[attempt] is None:\\n                    taken = [grid for grid in slots.values() if grid is not None]\\n                    grid = next((grid for _, i, grid in ranked if i == index and grid not in taken), None)\\n                    filled += grid is not None\\n                    slots[attempt] = grid\\n    return filled\\n\\n\\ndef verified_outcomes(tasks: dict, answers: dict, predictions: dict) -> dict:\\n    """Score verified predictions; None represents an output without a verified candidate."""\\n    if set(tasks) != set(answers) or set(tasks) != set(predictions):\\n        raise ValueError("scoring task coverage differs")\\n    hits, verified = {}, 0\\n    for task_id, task in tasks.items():\\n        if len(answers[task_id]) != len(task["test"]) or len(predictions[task_id]) != len(task["test"]):\\n            raise ValueError("scoring output coverage differs")\\n        hits[task_id] = []\\n        for expected, entry in zip(answers[task_id], predictions[task_id]):\\n            if set(entry) != {"attempt_1", "attempt_2"}:\\n                raise ValueError("scoring attempts differ")\\n            actual = [grid for grid in entry.values() if grid is not None]\\n            verified += bool(actual)\\n            hits[task_id].append(any(grid == expected for grid in actual))\\n    correct = sum(sum(rows) for rows in hits.values())\\n    count = sum(len(rows) for rows in hits.values())\\n    return {"correct_outputs": correct, "total_outputs": count, "verified_outputs": verified,\\n            "accuracy_percent": 100 * correct / count if count else 0,\\n            "fully_correct_tasks": sum(all(rows) for rows in hits.values()), "total_tasks": len(tasks),\\n            "hits_by_task": hits}\\n\', \'src/agent/run.py\': \'"""The entry point: serve a frozen plan (freeze.py) and run its evaluation.\\n\\nvLLM serves a base/adapter pair or a base model alone; with engine "sglang" in the plan, SGLang serves the base model\\n(src.qwen.sglang_eval.serve; the job\\\'s environment variables, e.g. PYTHONPATH with src/qwen/vllm_patch and ARC_SGLANG_*,\\npass through) and is queried on /generate with token ids. Without --shard every arm is generated, replayed and scored\\nunder ROOT/runtime; with --shard I/N only that shard\\\'s unfinished trajectories are generated, and merge.py replays and\\nscores the shard roots.\\n    python -m src.agent.run --root ROOT --plan-sha256 SHA [--shard I/N] [--weights DIR]\\n"""\\n\\nimport argparse\\nfrom contextlib import contextmanager\\nfrom datetime import datetime, timezone\\nimport importlib.metadata\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport signal\\nimport socket\\nimport subprocess\\nimport sys\\nimport threading\\nimport time\\nfrom types import SimpleNamespace\\n\\nfrom src.utils.completion import budgeted\\nfrom src.utils.kaggle_runtime import atomic_json, capture_server_log, stop_group\\nfrom .evaluate import replay_arm, run_arm, verified_outcomes\\nfrom .executor import configure, run_session\\nfrom .loop import request_payload\\nfrom .model_inventory import digest\\nfrom .prompt import initial_messages\\nfrom .protocol import decode_native, load_native_tokenizer, validate_response\\n\\n\\ndef _assert_isolated(environment: Path, source: Path) -> tuple[Path, Path]:\\n    environment = environment.resolve()\\n    source = source.resolve()\\n    executable = Path(sys.executable).resolve()\\n    prefix = Path(sys.prefix).resolve()\\n    if prefix != environment or environment not in executable.parents:\\n        raise ValueError("launcher must run with the selected isolated environment")\\n    return environment, source\\n\\n\\ndef _server_environment(environment: Path, source: Path, cuda_compat: Path | None) -> dict[str, str]:\\n    """Build the server environment; ``cuda_compat`` is only needed on hosts whose driver is too old."""\\n    environment, source = _assert_isolated(environment, source)\\n    cuda_libraries = environment / "targets/x86_64-linux/lib"\\n    required = [environment / "bin/vllm"]\\n    if cuda_compat is not None:\\n        cuda_compat = cuda_compat.resolve()\\n        required += [environment / "bin/nvcc", environment / "lib/libstdc++.so.6", cuda_libraries / "libcudart.so",\\n                     cuda_compat / "libcuda.so", cuda_compat / "libcuda.so.1"]\\n    missing = [str(path) for path in required if not path.is_file()]\\n    if missing:\\n        raise FileNotFoundError(f"isolated serving environment is incomplete: {missing}")\\n\\n    result = os.environ.copy()\\n    for key in tuple(result):\\n        if key.lower() in {"http_proxy", "https_proxy", "all_proxy"}:\\n            result.pop(key)\\n    result.update({\\n        "HF_HUB_OFFLINE": "1",\\n        "PYTHONNOUSERSITE": "1",\\n        "PYTHONDONTWRITEBYTECODE": "1",\\n        "PYTHONPATH": str(source),\\n        "TOKENIZERS_PARALLELISM": "false",\\n        "PATH": f"{environment / \\\'bin\\\'}:{os.environ.get(\\\'PATH\\\', \\\'\\\')}",\\n    })\\n    if cuda_compat is not None:\\n        result.update({\\n            "CUDA_HOME": str(environment),\\n            "CUDA_PATH": str(environment),\\n            "LIBRARY_PATH": f"{cuda_libraries}:{cuda_compat}",\\n            "LD_PRELOAD": str(environment / "lib/libstdc++.so.6"),\\n            "LD_LIBRARY_PATH": f"{cuda_compat}:{environment / \\\'lib\\\'}:{cuda_libraries}",\\n        })\\n    return result\\n\\n\\ndef verify_plan(path: Path, expected: str) -> dict:\\n    if digest(path) != expected:\\n        raise ValueError("Evaluation plan changed")\\n    plan = json.loads(path.read_text())\\n    if plan.get("mode") == "training_collection":\\n        raise ValueError("Training collection is no longer supported; scoring would read labels")\\n    for name, sha in plan["artifact_hashes"].items():\\n        if digest(Path(name)) != sha:\\n            raise ValueError("Bound evaluation artifact changed: " + name)\\n    if plan.get("adapter") is not None:\\n        reload = json.loads(Path(plan["adapter_reload"]).read_text())\\n        if reload["status"] != "ADAPTER_RELOAD_VERIFIED" or reload["adapter_sha256"] != digest(Path(plan["adapter"]) / "adapter_model.safetensors"):\\n            raise ValueError("Adapter has not passed exact tensor reload verification")\\n    engine = plan.get("engine", "vllm")\\n    if importlib.metadata.version(engine) != plan[engine + "_version"]:\\n        raise ValueError("Serving environment changed")\\n    return plan\\n\\n\\ndef score_pair(plan: dict, output: Path) -> dict:\\n    reports = {arm: json.loads((output / f"{arm}_replay.json").read_text()) for arm in plan["served_models"]}\\n    for arm, report in reports.items():\\n        if report["status"] != "ALL_NATIVE_TOOL_TRAJECTORIES_REPLAYED" or report["labels_loaded"]:\\n            raise ValueError("Both arms must independently replay before scoring")\\n        for name, sha in report["artifact_hashes"].items():\\n            if digest(output / arm / name) != sha:\\n                raise ValueError("Replayed evidence changed")\\n    tasks = json.loads(Path(plan["challenges"]).read_text())\\n    if digest(Path(plan["solutions"])) != plan["solutions_sha256"]:\\n        raise ValueError("Evaluation solutions differ")\\n    solutions = json.loads(Path(plan["solutions"]).read_text())\\n    scores = {arm: verified_outcomes(tasks, solutions, report["scoring_submission"]) for arm, report in reports.items()}\\n    for score in scores.values():\\n        score["kaggle_style_score"] = 100 * sum(sum(rows) / len(rows) for rows in score["hits_by_task"].values()) / len(tasks)\\n    pairs = None\\n    if "adapter" in scores:\\n        pairs = {"both_correct": 0, "base_only": 0, "adapter_only": 0, "both_wrong": 0}\\n        for task_id in tasks:\\n            for base, adapter in zip(scores["base"]["hits_by_task"][task_id], scores["adapter"]["hits_by_task"][task_id]):\\n                key = "both_correct" if base and adapter else "base_only" if base else "adapter_only" if adapter else "both_wrong"\\n                pairs[key] += 1\\n    return {"status": "COMPLETE_REPLAYED_SCORED", "scores": scores, "paired_outputs": pairs,\\n            "labels_loaded_after_both_replays": True, "official_kaggle_score": False,\\n            "scope": "ARC evaluation tasks excluded from this training cache; previously used development evaluation, not an untouched test set."}\\n\\n\\ndef verify_adapter_route(plan: dict, output: Path, client, *, deadline: float | None = None) -> None:\\n    reference = json.loads(Path(plan["adapter_reload"]).read_text())\\n    witness = {}\\n    for arm in ["base", "adapter"]:\\n        timeout = 180 if deadline is None else min(180, deadline - time.time())\\n        if timeout <= 0:\\n            raise TimeoutError("Adapter route verification exhausted the startup budget")\\n        response = client.completions.create(\\n            model=plan["served_models"][arm], prompt=reference["prompt_token_ids"],\\n            temperature=0, max_tokens=1, timeout=timeout,\\n            extra_body={"return_token_ids": True, "add_special_tokens": False},\\n        ).model_dump()\\n        witness[arm] = response\\n        atomic_json(output / "adapter_route.json", witness)\\n        if (response["model"] != plan["served_models"][arm]\\n                or response["choices"][0].get("token_ids") != [reference[f"{arm}_next_token"]]):\\n            raise ValueError("Served adapter/base next token differs from independent Transformers reload")\\n\\n\\ndef verify_thinking_budget(plan: dict, output: Path, client) -> None:\\n    """Witness the native budget in served tokens without reading evaluation tasks."""\\n    tokenizer = load_native_tokenizer(plan["model"])\\n    task = {"train": [{"input": [[1]], "output": [[1]]}], "test": [{"input": [[2]]}]}\\n    messages = initial_messages(task, 0)\\n    witness = []\\n    for arm in plan["served_models"]:\\n        for budget in (0, 4):\\n            settings = {**plan["evaluation"], "thinking_token_budget": budget, "max_tokens": 512}\\n            payload = request_payload(tokenizer, messages, plan["served_models"][arm], settings, settings["seed"])\\n            response = client.completions.create(\\n                **{key: value for key, value in payload.items() if key != "prompt_sha256"}, timeout=180,\\n            ).model_dump()\\n            witness.append({"arm": arm, "budget": budget, "request": payload, "response": response})\\n            atomic_json(output / "thinking_budget_witness.json", witness)\\n            if response["model"] != plan["served_models"][arm]:\\n                raise ValueError("Thinking witness model identity differs")\\n            validate_response(tokenizer, payload["prompt"], response, 512, budget)\\n            tokens = response["choices"][0]["token_ids"]\\n            end = tokens.index(tokenizer.encode("</think>", add_special_tokens=False)[0])\\n            if len(tokens) <= end + 1:\\n                raise ValueError("Thinking witness did not continue after the reasoning boundary")\\n    atomic_json(output / "thinking_budget_check.json", {\\n        "status": "NATIVE_BUDGET_WITNESSED", "arms": ["base", "adapter"], "budgets": [0, 4],\\n        "scope": "Served-token control only; no ARC quality claim. Every formal response is checked again.",\\n    })\\n\\n\\ndef budgeted_response(tokenizer, generate, model: str, prompt: list[int], max_tokens: int, budget: int | None) -> dict:\\n    """vLLM\\\'s /v1/completions response (token ids in and out) for generate(ids, limit) -> (ids, finish), the thinking\\n    budget applied as src.utils.completion.budgeted does; a spent budget (limit <= 0) sends no request."""\\n    step = lambda ids, limit: ([], "length") if limit <= 0 else generate(ids, limit)\\n    think = tokenizer.convert_tokens_to_ids\\n    tokens, finish = budgeted(step, prompt, {"thinking_token_budget": budget}, max_tokens, think("<think>"), think("</think>"))\\n    return {"model": model, "choices": [{"text": decode_native(tokenizer, tokens), "token_ids": tokens,\\n                                         "prompt_token_ids": prompt, "finish_reason": finish}],\\n            "usage": {"prompt_tokens": len(prompt), "completion_tokens": len(tokens)}}\\n\\n\\nclass SGLangCompletions:\\n    """client.completions.create for the agent loop on SGLang\\\'s /generate, answered in the shape of vLLM\\\'s\\n    /v1/completions with token ids; the request\\\'s thinking_token_budget is applied as src.utils.completion.budgeted does."""\\n\\n    def __init__(self, http, url: str, tokenizer):\\n        self.http, self.url, self.tokenizer, self.completions = http, url, tokenizer, self\\n\\n    def create(self, *, model, prompt, n, temperature, top_p, max_tokens, seed, extra_body, timeout):\\n        if n != 1:\\n            raise ValueError("Expected one native completion")\\n\\n        def generate(ids, limit):\\n            params = {"temperature": temperature, "top_p": top_p, "top_k": extra_body["top_k"], "max_new_tokens": limit,\\n                      "stop_token_ids": extra_body["stop_token_ids"], "skip_special_tokens": False, "sampling_seed": seed,\\n                      **({"min_p": extra_body["min_p"]} if "min_p" in extra_body else {})}\\n            response = self.http.post(self.url, json={"input_ids": ids, "sampling_params": params}, timeout=timeout)\\n            response.raise_for_status()\\n            out = response.json()\\n            if out["meta_info"]["prompt_tokens"] != len(ids):\\n                raise ValueError("SGLang prompt token count differs")\\n            return list(out["output_ids"]), out["meta_info"]["finish_reason"]["type"]\\n\\n        response = budgeted_response(self.tokenizer, generate, model, prompt, max_tokens, extra_body.get("thinking_token_budget"))\\n        return SimpleNamespace(model_dump=lambda: response)\\n\\n\\nclass VLLMCompletions:\\n    """client.completions.create for the agent loop on a vLLM /v1/completions that budgets nothing itself (h95 as the\\n    notebook serves it); each request\\\'s prefix-cache hit is kept in usage.cached_tokens. With logprobs, each sampled\\n    token\\\'s log-prob as the server reports it (logprobs 0, on the scale of its --logprobs-mode) is kept in\\n    choices[0].token_logprobs, aligned with token_ids; the </think> the budget inserted, which nothing sampled, gets None\\n    (the RL trainer\\\'s rollout log-probs)."""\\n\\n    def __init__(self, http, url: str, tokenizer, logprobs: bool = False):\\n        self.http, self.url, self.tokenizer, self.completions, self.logprobs = http, url, tokenizer, self, logprobs\\n\\n    def create(self, *, model, prompt, n, temperature, top_p, max_tokens, seed, extra_body, timeout):\\n        if n != 1:\\n            raise ValueError("Expected one native completion")\\n        cached, sampled = [], []  # sampled: (offset in the completion, log-probs) per request\\n\\n        def generate(ids, limit):\\n            body = {"model": model, "prompt": ids, "max_tokens": limit, "temperature": temperature, "top_p": top_p,\\n                    "seed": seed, "top_k": extra_body["top_k"], "stop_token_ids": extra_body["stop_token_ids"],\\n                    "return_token_ids": True, "add_special_tokens": False, "skip_special_tokens": False,\\n                    "include_stop_str_in_output": True, "spaces_between_special_tokens": False,\\n                    **({"min_p": extra_body["min_p"]} if "min_p" in extra_body else {}),\\n                    **({"logprobs": 0} if self.logprobs else {})}\\n            response = self.http.post(self.url, json=body, timeout=timeout)\\n            response.raise_for_status()\\n            out = response.json()\\n            choice = out["choices"][0]\\n            if out["model"] != model or out["usage"]["prompt_tokens"] != len(ids) or choice.get("prompt_token_ids") != ids:\\n                raise ValueError("vLLM model or prompt tokens differ")\\n            cached.append((out["usage"].get("prompt_tokens_details") or {}).get("cached_tokens"))\\n            if self.logprobs:\\n                values = (choice.get("logprobs") or {}).get("token_logprobs")\\n                if values is None or len(values) != len(choice["token_ids"]):\\n                    raise ValueError("vLLM log-probs missing or not one per generated token")\\n                sampled.append((len(ids) - len(prompt), values))\\n            return list(choice["token_ids"]), choice["finish_reason"]\\n\\n        response = budgeted_response(self.tokenizer, generate, model, prompt, max_tokens, extra_body.get("thinking_token_budget"))\\n        response["usage"]["cached_tokens"] = cached\\n        if self.logprobs:\\n            aligned = [None] * len(response["choices"][0]["token_ids"])\\n            for offset, values in sampled:\\n                aligned[offset: offset + len(values)] = values\\n            response["choices"][0]["token_logprobs"] = aligned\\n        return SimpleNamespace(model_dump=lambda: response)\\n\\n\\n@contextmanager\\ndef vllm_server(plan: dict, output: Path, state: dict, model: str):\\n    """Yield (OpenAI client, alive) once vLLM serves every arm of the plan; the server group is stopped on exit."""\\n    import httpx\\n    from openai import OpenAI\\n\\n    server = None\\n    try:\\n        gpus = plan.get("gpu_visibility")\\n        environment = Path(plan["environment"])\\n        env = _server_environment(environment, Path(plan["source"]),\\n                                  Path(plan["cuda_compat"]) if plan.get("cuda_compat") else None)\\n        env.update(OMP_NUM_THREADS="4", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")\\n        if gpus:\\n            env["CUDA_VISIBLE_DEVICES"] = gpus\\n        command = [str(environment / "bin/vllm"), "serve", model, "--host", "127.0.0.1", "--port", str(plan["port"]),\\n                   "--served-model-name", plan["served_models"]["base"], "--tensor-parallel-size", str(plan.get("tensor_parallel_size", 2)), "--dtype", "bfloat16",\\n                   "--max-model-len", str(plan["evaluation"]["max_model_len"]), "--max-num-seqs", str(plan["evaluation"]["concurrency"]),\\n                   "--max-num-batched-tokens", "4096", "--gpu-memory-utilization", "0.85", "--language-model-only",\\n                   "--generation-config", "vllm", "--compilation-config", json.dumps(plan["compilation_config"])]\\n        if plan.get("adapter") is not None:\\n            command.extend(["--enable-lora", "--max-lora-rank", "16", "--lora-modules", json.dumps({\\n                "name": plan["served_models"]["adapter"], "path": plan["adapter"], "base_model_name": plan["served_models"]["base"]})])\\n        if plan["evaluation"].get("thinking_token_budget") is not None:\\n            command.extend(["--reasoning-parser", "qwen3"])\\n        if plan.get("kv_cache_dtype"):\\n            command.extend(["--kv-cache-dtype", plan["kv_cache_dtype"]])\\n        if plan.get("quantization"):\\n            command.extend(["--quantization", plan["quantization"]])\\n        command.extend(plan.get("server_args", []))\\n        server = subprocess.Popen(command, cwd=plan["source"], env=env, stdout=subprocess.PIPE,\\n                                  stderr=subprocess.STDOUT, text=True, start_new_session=True)\\n        state.update(server_pid=server.pid, server_command=command)\\n        atomic_json(output / "lifecycle.json", state)\\n        ready, complete = threading.Event(), threading.Event()\\n        threading.Thread(target=capture_server_log, args=(server.stdout, output / "server.log", ready, complete), daemon=True).start()\\n        if not ready.wait(1800) or not complete.is_set() or server.poll() is not None:\\n            raise RuntimeError("Native tool server failed during startup")\\n        with httpx.Client(trust_env=False, follow_redirects=False) as transport, OpenAI(\\n                base_url=f"http://127.0.0.1:{plan[\\\'port\\\']}/v1", api_key="EMPTY", max_retries=0, http_client=transport) as client:\\n            if sorted(row.id for row in client.models.list().data) != sorted(plan["served_models"].values()):\\n                raise ValueError("Served base and adapter identities differ")\\n            yield client, lambda: server.poll() is None\\n    finally:\\n        stop_group(server)\\n\\n\\n@contextmanager\\ndef sglang_server(plan: dict, output: Path, model: str):\\n    """Yield (SGLangCompletions, alive) once SGLang serves the plan\\\'s base model; stopped on exit."""\\n    from src.qwen.sglang_eval import serve\\n\\n    if plan.get("adapter") is not None:\\n        raise ValueError("An SGLang plan serves the base model alone")\\n    settings = plan["evaluation"]\\n    config = {"python": str(Path(plan["environment"]) / "bin/python"), "model": model,\\n              "tp": plan["tensor_parallel_size"], "max_model_len": settings["max_model_len"],\\n              "concurrency": settings["concurrency"], "port": plan["port"], "server_args": plan.get("server_args", [])}\\n    tokenizer = load_native_tokenizer(plan["model"], settings.get("native_format", "qwen"))\\n    with serve(config, output / "server.log") as (http, base):\\n        yield SGLangCompletions(http, base + "/generate", tokenizer), lambda: True\\n\\n\\ndef require_executor(settings: dict) -> None:\\n    """Tool "f3" runs only if its executor worker starts (with any configured sandbox and memory cap), checked with one\\n    trivial program before the server starts."""\\n    if settings.get("tool") == "f3" and run_session("x = 1", [], [], 5.0) is None:\\n        raise RuntimeError("the f3 executor worker did not run a trivial program (memory cap or sandbox setup)")\\n\\n\\ndef run(root: Path, expected: str, shard: tuple[int, int] | None = None, weights: Path | None = None) -> None:\\n    """weights: serve this copy of the plan\\\'s model (e.g. on node-local NVMe); the tokenizer is still the plan\\\'s."""\\n    plan_path = root / "plan.json"\\n    plan = verify_plan(plan_path, expected)\\n    sglang = plan.get("engine") == "sglang"\\n    output = root / "runtime"\\n    output.mkdir(exist_ok=shard is not None)\\n    state = {"status": "STARTING", "started_utc": datetime.now(timezone.utc).isoformat(),\\n             "plan_sha256": expected, "labels_loaded": False, **({"shard": "%d/%d" % shard} if shard else {})}\\n    atomic_json(output / "lifecycle.json", state)\\n    try:\\n        require_executor(plan["evaluation"])\\n        # Without gpu_visibility (scheduler-assigned GPUs, e.g. a batch job) every visible GPU must be idle.\\n        gpus = plan.get("gpu_visibility")\\n        used = subprocess.check_output(["nvidia-smi", *(["--id=" + gpus] if gpus else []), "--query-gpu=memory.used",\\n                                        "--format=csv,noheader,nounits"], text=True)\\n        if any(int(value) >= 1024 for value in used.splitlines()):\\n            raise RuntimeError("Selected inference GPUs are occupied")\\n        with socket.socket() as probe:\\n            probe.bind(("127.0.0.1", plan["port"]))\\n        model = str(weights or plan["model"])\\n        with (sglang_server(plan, output, model) if sglang else vllm_server(plan, output, state, model)) as (client, alive):\\n            if plan.get("adapter") is not None:\\n                verify_adapter_route(plan, output, client)\\n            if not sglang and plan["evaluation"].get("thinking_token_budget") is not None:\\n                verify_thinking_budget(plan, output, client)\\n            for arm in plan["served_models"]:\\n                verify_plan(plan_path, expected)\\n                state.update(status="GENERATING", active_arm=arm)\\n                atomic_json(output / "lifecycle.json", state)\\n                run_arm(plan, arm, output / arm, client, shard)\\n                if not alive():\\n                    raise RuntimeError("Server exited during evaluation")\\n                if shard is not None:\\n                    continue\\n                state.update(status="REPLAYING", active_arm=arm)\\n                atomic_json(output / "lifecycle.json", state)\\n                atomic_json(output / f"{arm}_replay.json", replay_arm(plan, arm, output / arm))\\n        verify_plan(plan_path, expected)\\n        if shard is not None:\\n            state.update(status="SHARD_COMPLETE_UNREPLAYED")\\n        else:\\n            atomic_json(output / "paired_scored.json", score_pair(plan, output))\\n            state.update(status="COMPLETE_REPLAYED_SCORED", labels_loaded=True)\\n    except BaseException as error:\\n        state.update(status="ERROR", error=f"{type(error).__name__}: {error}")\\n        raise\\n    finally:\\n        state["released_utc"] = datetime.now(timezone.utc).isoformat()\\n        atomic_json(output / "lifecycle.json", state)\\n\\n\\nif __name__ == "__main__":\\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\\n    parser.add_argument("--root", type=Path, required=True)\\n    parser.add_argument("--plan-sha256", required=True)\\n    parser.add_argument("--shard", type=lambda text: tuple(map(int, text.split("/"))), metavar="I/N",\\n                        help="run only keys with key %% N == I, skipping finished keys already in the root; replay and "\\n                             "scoring are left to merge.py")\\n    parser.add_argument("--weights", type=Path, help="serve this copy of the plan\\\'s model (e.g. on node-local NVMe)")\\n    parser.add_argument("--executor-sandbox", choices=["bwrap"], help="run every program worker in bubblewrap (src.agent.executor)")\\n    parser.add_argument("--executor-memory-gb", type=float, help="address-space cap of each program worker (shared machines)")\\n    args = parser.parse_args()\\n    configure(args.executor_sandbox, args.executor_memory_gb)\\n    if args.shard is not None and (len(args.shard) != 2 or not 0 <= args.shard[0] < args.shard[1]):\\n        parser.error("--shard must be I/N with 0 <= I < N")\\n\\n    def terminate(signum, _frame):\\n        raise SystemExit(128 + signum)\\n\\n    signal.signal(signal.SIGTERM, terminate)\\n    signal.signal(signal.SIGINT, terminate)\\n    exit_code = 1\\n    try:\\n        run(args.root, args.plan_sha256, args.shard, args.weights)\\n        exit_code = 0\\n    finally:\\n        (args.root / "runtime.exit").write_text(str(exit_code) + "\\\\n")\\n\'}\nfor name, source in sources.items():\n    path = Path(\'/kaggle/working\') / name\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(source)\n', 'notebook_cell_4', 'exec'))


In [6]:
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    exec(compile("from src.utils.kaggle_runtime import validate_model_files\nconfigs = list(Path('/kaggle/input').rglob('config.json'))\nmodels = [p.parent for p in configs if json.loads(p.read_text()).get('architectures') == [deployment['model_architecture']]\n          and deployment['model_dir_contains'] in str(p)]\nassert len(models) == 1, models\nif deployment['model_files']:\n    validate_model_files(models[0], deployment['model_files'])\nsubprocess.run([python, '-c', 'import torch, transformers, vllm; print(torch.__version__, transformers.__version__, vllm.__version__)'], check=True)\nversions = 'import json, sys; from importlib.metadata import version; print(json.dumps({n: version(n) for n in sys.argv[1:]}))'\nreport = dict(status='PASS', python=sys.version, libc=platform.libc_ver(), machine=platform.machine(),\n              model=str(models[0]), model_files=len(deployment['model_files']), wheel_files=len(deployment['wheel_files']),\n              packages=json.loads(subprocess.check_output([python, '-c', versions, deployment['package'], 'torch', 'transformers',\n                                                           'flashinfer-python'], text=True)),\n              isolated_wheel_install=deployment['isolated_wheel_install'],\n              elapsed_seconds=time.time()-started, scope='CPU install, import and model-byte verification only; no generation or GPU validation')\nPath('deployment_preflight.json').write_text(json.dumps(report, indent=2))\nprint(json.dumps(report))\n", 'notebook_cell_5', 'exec'))


In [7]:
if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    exec(compile('os.environ[\'PYTHONPATH\'] = \'/kaggle/working/src/qwen/vllm_patch:/kaggle/working:\' + os.environ.get(\'PYTHONPATH\', \'\')\nos.environ[\'ARC_VLLM_DRAFT_VOCAB\'] = \'/kaggle/working/kaggle/qwen-transductive/draft_vocab/qwen38_arc_16384.json\'\nfrom src.qwen.flashnext import assemble_drafter, assemble_model, kept_experts\nfp8_tables = [p.parent for p in Path(\'/kaggle/input\').rglob(\'model-plefp8-00000.safetensors\')]\nflashnext_drafters = [p.parent for p in Path(\'/kaggle/input\').rglob(\'mtp-routed-experts-int4.safetensors\')]\nassert len(fp8_tables) == 1 and len(flashnext_drafters) == 1, (fp8_tables, flashnext_drafters)\nreap_keep = Path(\'/kaggle/working/kaggle/qwen-transductive/flashnext/reap448_keep.json\')\nassembled = Path(\'/kaggle/tmp/flashnext\')\nshutil.rmtree(assembled, ignore_errors=True)\nserved_model = assemble_model(models[0], fp8_tables[0], assembled / \'model\', kept_experts(reap_keep))\nserved_drafter = assemble_drafter(flashnext_drafters[0], assembled / \'drafter\', kept_experts(reap_keep))\npurelib = Path(subprocess.check_output([python, \'-c\', \'import sysconfig; print(sysconfig.get_paths()["purelib"])\'], text=True).strip())\nfor aot in sorted(wheel_directories[deployment[\'wheel_sources\'][0]].glob(\'flashinfer-aot-sm89-*.so\')):\n    name = aot.stem.removeprefix(\'flashinfer-aot-sm89-\')\n    (purelib / \'flashinfer/data/aot\' / name).mkdir(parents=True, exist_ok=True)\n    shutil.copy(aot, purelib / \'flashinfer/data/aot\' / name / f\'{name}.so\')\n    print(\'FlashInfer AOT module\', name, flush=True)\nprefetch = subprocess.Popen([sys.executable, \'-c\', \'\\nimport json, os, sys, time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nmodel, readers, log = Path(sys.argv[1]), int(sys.argv[2]), Path(sys.argv[3])\\ncopy = Path(sys.argv[4]) if len(sys.argv) > 4 else None\\ncache = len(sys.argv) > 5 and sys.argv[5] == "keep"  # copy-first: nothing else reads meanwhile, so the copy stays cached\\nCHUNK = (64 if copy else 16) << 20\\nfiles = sorted(path for path in model.glob("*.safetensors") if "plefp8" not in path.name)\\nif copy:\\n    copy.mkdir(parents=True, exist_ok=True)\\n    files += sorted(path for path in model.glob("*.safetensors") if "plefp8" in path.name)\\nFLUSH = 256 << 20  # the copy is synced and dropped from the page cache every FLUSH bytes, so it cannot evict the source\\ndef meminfo():\\n    with open("/proc/meminfo") as lines:\\n        return {line.split(":")[0]: int(line.split()[1]) for line in lines}\\ndef dirty_mib():\\n    return meminfo().get("Dirty", 0) // 1024\\ndef read(path):\\n    began, size, writing, dropped, peak = time.time(), 0, 0.0, 0, 0\\n    part = copy / (path.name + ".part") if copy else None\\n    try:\\n        with open(path, "rb", buffering=0) as handle, (part.open("wb") if part else open(os.devnull, "wb")) as out:\\n            while chunk := handle.read(CHUNK):\\n                size += len(chunk)\\n                if part:\\n                    mark = time.time()\\n                    out.write(chunk)\\n                    if size - dropped >= FLUSH:\\n                        peak = max(peak, dirty_mib())\\n                        out.flush()\\n                        os.fdatasync(out.fileno())\\n                        if not cache:\\n                            os.posix_fadvise(out.fileno(), dropped, size - dropped, os.POSIX_FADV_DONTNEED)\\n                        dropped = size\\n                    writing += time.time() - mark\\n            if part:\\n                mark = time.time()\\n                peak = max(peak, dirty_mib())\\n                out.flush()\\n                os.fsync(out.fileno())\\n                if not cache:\\n                    os.posix_fadvise(out.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)\\n                writing += time.time() - mark\\n        if part:\\n            part.replace(copy / path.name)\\n    except OSError as error:\\n        if part:\\n            part.unlink(missing_ok=True)\\n        return {"file": path.name, "error": f"{type(error).__name__}: {error}"}\\n    row = {"file": path.name, "gib": round(size / 2**30, 3), "seconds": round(time.time() - began, 1)}\\n    return {**row, "write_seconds": round(writing, 1), "dirty_mib_peak": peak} if part else row\\nbegan = time.time()\\nwith ThreadPoolExecutor(readers) as pool, log.open("w") as out:\\n    total, peak = 0.0, 0\\n    for row in pool.map(read, files):\\n        total += row.get("gib", 0.0)\\n        peak = max(peak, row.get("dirty_mib_peak", 0))\\n        out.write(json.dumps(row) + "\\\\n")\\n        out.flush()\\n    seconds = time.time() - began\\n    memory = meminfo()\\n    out.write(json.dumps({"files": len(files), "readers": readers, "gib": round(total, 2), "seconds": round(seconds, 1),\\n                          "gib_per_s": round(total / seconds, 3) if seconds else None,\\n                          "mb_per_s": round(total * 1024 / seconds, 1) if seconds else None,\\n                          **({"dirty_mib_peak": peak, "copy_cached": cache} if copy else {}),\\n                          "mem_available_gib": round(memory.get("MemAvailable", 0) / 2**20, 1),\\n                          "cached_gib": round(memory.get("Cached", 0) / 2**20, 1)}) + "\\\\n")\\n\', str(served_model), \'16\', \'/kaggle/working/prefetch.log\'])\nchallenges = list(Path(\'/kaggle/input\').rglob(\'arc-agi_test_challenges.json\'))\nassert len(challenges) == 1, challenges\ntasks = json.loads(challenges[0].read_text())\nbudget = 42300\nif not os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\'):  # commit run: smoke test only\n    tasks = dict(list(tasks.items())[:4])\n    challenges = [Path(\'/kaggle/working/commit_challenges.json\')]\n    challenges[0].write_text(json.dumps(tasks))\n    budget = min(budget, time.time() - started + 2700)\nconfig = {**{\'served_model\': \'qwen\', \'port\': 8580, \'gpus\': [0, 1, 2, 3], \'evaluation\': {\'seed\': 20260926, \'attempts\': 2, \'concurrency\': 64, \'temperature\': 0.7, \'top_p\': 0.95, \'top_k\': 20, \'thinking_token_budget\': 131072, \'max_tokens\': 139264, \'max_model_len\': 131072, \'second_priority\': True, \'deadline_budget\': 16384, \'job_unit\': \'task\', \'agent_mode\': \'free\', \'max_rounds\': 1000, \'execution_timeout\': 5.0, \'tool\': \'f3\', \'context_brake\': 131072, \'thinking_scope\': \'round\', \'exact_tool_text\': True, \'token_continuity\': True, \'later_attempts\': {\'avoid_previous\': True, \'previous_program\': True, \'repeat_refuse\': True, \'previous_program_slice\': True}, \'later_order\': [\'capped\', \'unverified\', \'running\', \'verified\'], \'doubt_rules\': True, \'deadline_brake\': 900, \'round_clamp\': 180, \'doubt_hint\': True, \'scipy_ndimage\': True}, \'engine\': \'openai\', \'command\': [\'bash\', \'/kaggle/working/kaggle/qwen-transductive/h95/serve_l4.sh\'], \'env\': {\'VENV\': \'/kaggle/tmp/h95-venv\', \'REPO\': \'/kaggle/working\', \'KEEP\': \'/kaggle/working/kaggle/qwen-transductive/flashnext/reap448_keep.json\', \'ARC_VLLM_GDN_REUSE\': \'1\', \'KV_BYTES\': \'3900000000\', \'UTIL\': \'0.95\', \'MAX_SEQS\': \'64\', \'PORT\': \'8580\', \'CACHE_ROOT\': \'/kaggle/tmp/vllm-cache-merlin\', \'SIZES\': \'1,2,3,4,6,8,12,16,24,32,48,64,96,128,144,160,176,192,208,224,240,256\', \'ARC_VLLM_TOPK_LOGITS\': \'20\', \'ARC_VLLM_RETAIN_LAST_STATE\': \'1\', \'ARC_VLLM_RETAIN_LAST_STATE_PROTECT_SECONDS\': \'300\', \'ARC_VLLM_QSA_HOST_KV\': \'1\', \'ARC_VLLM_QSA_HOST_KV_GB\': \'15\', \'ARC_VLLM_READ_FILTER\': \'1\', \'ARC_VLLM_LATE_COMMIT\': \'1\', \'SSM_DTYPE\': \'bfloat16\', \'ARC_VLLM_GDN_PACK\': \'tight\', \'ARC_VLLM_QSA_HOT_STEPS\': \'1\', \'ARC_VLLM_QSA_HOT_TOKENS\': \'4112\', \'BLOCK_SIZE\': \'8192\', \'INDEXER_KV_DTYPE\': \'fp8\', \'ARC_VLLM_GDN_PARK\': \'2048\', \'ARC_VLLM_QSA_HOT_OVERLAP\': \'1\', \'CUDA_DEVICE_MAX_CONNECTIONS\': \'32\', \'ARC_VLLM_MEM_REPORT\': \'500\'}, \'startup_timeout\': 10800, \'cleanup_reserve\': 300, \'trajectories\': \'/kaggle/tmp/qwen_trajectories\', \'stall_restart\': 600, \'fallback_server_args\': [], \'fallback_command\': [\'bash\', \'/kaggle/working/kaggle/qwen-transductive/h95/serve_l4.sh\'], \'fallback_env\': [{\'MAX_SEQS\': \'48\', \'KV_BYTES\': \'4080000000\', \'SIZES\': \'1,2,3,4,6,8,12,16,24,32,48,64,96,128,144,160,176,192\', \'ARC_VLLM_QSA_HOT_STEPS\': \'1\', \'ARC_VLLM_QSA_HOT_TOKENS\': \'3088\', \'ARC_VLLM_GDN_PACK\': \'tight\', \'ARC_VLLM_QSA_HOST_KV_GB\': \'12\', \'SSM_DTYPE\': \'bfloat16\', \'ARC_VLLM_MEM_REPORT\': \'500\', \'BLOCK_SIZE\': \'4096\', \'INDEXER_KV_DTYPE\': \'\', \'ARC_VLLM_GDN_PARK\': \'0\'}, {\'MAX_SEQS\': \'36\', \'KV_BYTES\': \'4100000000\', \'SIZES\': \'1,2,3,4,6,8,12,16,24,32,48,64,96,128\', \'ARC_VLLM_QSA_HOT_STEPS\': \'0\', \'ARC_VLLM_QSA_HOT_TOKENS\': \'0\', \'ARC_VLLM_GDN_PACK\': \'1\', \'ARC_VLLM_QSA_HOST_KV_GB\': \'10\', \'SSM_DTYPE\': \'bfloat16\', \'ARC_VLLM_MEM_REPORT\': \'\', \'BLOCK_SIZE\': \'4096\', \'INDEXER_KV_DTYPE\': \'\', \'ARC_VLLM_GDN_PARK\': \'0\'}, {\'MAX_SEQS\': \'20\', \'KV_BYTES\': \'4170000000\', \'SIZES\': \'1,2,3,4,6,8,12,16,24,32,48,64,80\', \'ARC_VLLM_QSA_HOST_KV_GB\': \'10\', \'ARC_VLLM_QSA_HOT_STEPS\': \'0\', \'ARC_VLLM_QSA_HOT_TOKENS\': \'0\', \'ARC_VLLM_GDN_PACK\': \'1\', \'SSM_DTYPE\': \'bfloat16\', \'ARC_VLLM_MEM_REPORT\': \'\', \'BLOCK_SIZE\': \'4096\', \'INDEXER_KV_DTYPE\': \'\', \'ARC_VLLM_GDN_PARK\': \'0\'}]}, \'model\': str(served_model), \'challenges\': str(challenges[0]), \'deadline\': started + budget,\n          \'output\': \'/kaggle/working/qwen_run\', \'final_submission\': \'/kaggle/working/submission.json\'}\nconfig[\'env\'].update(MODEL=str(served_model), DRAFTER=str(served_drafter))\nconfig[\'restart_prefetch\'] = [sys.executable, \'-c\', \'\\nimport json, os, sys, time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nmodel, readers, log = Path(sys.argv[1]), int(sys.argv[2]), Path(sys.argv[3])\\ncopy = Path(sys.argv[4]) if len(sys.argv) > 4 else None\\ncache = len(sys.argv) > 5 and sys.argv[5] == "keep"  # copy-first: nothing else reads meanwhile, so the copy stays cached\\nCHUNK = (64 if copy else 16) << 20\\nfiles = sorted(path for path in model.glob("*.safetensors") if "plefp8" not in path.name)\\nif copy:\\n    copy.mkdir(parents=True, exist_ok=True)\\n    files += sorted(path for path in model.glob("*.safetensors") if "plefp8" in path.name)\\nFLUSH = 256 << 20  # the copy is synced and dropped from the page cache every FLUSH bytes, so it cannot evict the source\\ndef meminfo():\\n    with open("/proc/meminfo") as lines:\\n        return {line.split(":")[0]: int(line.split()[1]) for line in lines}\\ndef dirty_mib():\\n    return meminfo().get("Dirty", 0) // 1024\\ndef read(path):\\n    began, size, writing, dropped, peak = time.time(), 0, 0.0, 0, 0\\n    part = copy / (path.name + ".part") if copy else None\\n    try:\\n        with open(path, "rb", buffering=0) as handle, (part.open("wb") if part else open(os.devnull, "wb")) as out:\\n            while chunk := handle.read(CHUNK):\\n                size += len(chunk)\\n                if part:\\n                    mark = time.time()\\n                    out.write(chunk)\\n                    if size - dropped >= FLUSH:\\n                        peak = max(peak, dirty_mib())\\n                        out.flush()\\n                        os.fdatasync(out.fileno())\\n                        if not cache:\\n                            os.posix_fadvise(out.fileno(), dropped, size - dropped, os.POSIX_FADV_DONTNEED)\\n                        dropped = size\\n                    writing += time.time() - mark\\n            if part:\\n                mark = time.time()\\n                peak = max(peak, dirty_mib())\\n                out.flush()\\n                os.fsync(out.fileno())\\n                if not cache:\\n                    os.posix_fadvise(out.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)\\n                writing += time.time() - mark\\n        if part:\\n            part.replace(copy / path.name)\\n    except OSError as error:\\n        if part:\\n            part.unlink(missing_ok=True)\\n        return {"file": path.name, "error": f"{type(error).__name__}: {error}"}\\n    row = {"file": path.name, "gib": round(size / 2**30, 3), "seconds": round(time.time() - began, 1)}\\n    return {**row, "write_seconds": round(writing, 1), "dirty_mib_peak": peak} if part else row\\nbegan = time.time()\\nwith ThreadPoolExecutor(readers) as pool, log.open("w") as out:\\n    total, peak = 0.0, 0\\n    for row in pool.map(read, files):\\n        total += row.get("gib", 0.0)\\n        peak = max(peak, row.get("dirty_mib_peak", 0))\\n        out.write(json.dumps(row) + "\\\\n")\\n        out.flush()\\n    seconds = time.time() - began\\n    memory = meminfo()\\n    out.write(json.dumps({"files": len(files), "readers": readers, "gib": round(total, 2), "seconds": round(seconds, 1),\\n                          "gib_per_s": round(total / seconds, 3) if seconds else None,\\n                          "mb_per_s": round(total * 1024 / seconds, 1) if seconds else None,\\n                          **({"dirty_mib_peak": peak, "copy_cached": cache} if copy else {}),\\n                          "mem_available_gib": round(memory.get("MemAvailable", 0) / 2**20, 1),\\n                          "cached_gib": round(memory.get("Cached", 0) / 2**20, 1)}) + "\\\\n")\\n\', str(served_model), \'16\']\noverrides = list(Path(\'/kaggle/input\').rglob(\'override.safetensors\'))\nassert len(overrides) == 1, overrides\nlocal_override = Path(\'/kaggle/tmp/override.safetensors\')\nlocal_override.parent.mkdir(parents=True, exist_ok=True)\nshutil.copyfile(overrides[0], local_override)\nconfig[\'env\'][\'ARC_VLLM_WEIGHT_OVERRIDE\'] = str(local_override)\nPath(\'qwen_config.json\').write_text(json.dumps(config))\n# The runtime writes a valid submission before serving and rewrites it at the end, so its exit code must not fail\n# the notebook; only an invalid file may. A competition rerun writes none until it ends, and none at all when the\n# server was never ready or no output was answered: the run then raises (a failed submission, not a 0.00).\nif not os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\'):  # commit run: per-second GPU use, the server\'s counters every 30 s\n    import threading, urllib.request\n    monitor = Path(\'/kaggle/working/commit_monitor\')\n    monitor.mkdir(exist_ok=True)\n    dmon = subprocess.Popen([\'nvidia-smi\', \'dmon\', \'-s\', \'u\', \'-d\', \'1\'], stdout=open(monitor / \'dmon.log\', \'w\'), stderr=subprocess.STDOUT)\n    stop = threading.Event()\n    counters = (\'vllm:num_preemptions_total\', \'vllm:prefix_cache_hits_total\', \'vllm:prefix_cache_queries_total\',\n                \'vllm:num_requests_running\', \'vllm:num_requests_waiting\', \'vllm:kv_cache_usage_perc\',\n                \'vllm:prompt_tokens_total\', \'vllm:generation_tokens_total\')\n    def scrape():\n        while not stop.is_set():\n            try:\n                url = \'http://127.0.0.1:\' + str(config.get(\'port\', 8580)) + \'/metrics\'\n                text = urllib.request.urlopen(url, timeout=5).read().decode()\n                with open(monitor / \'metrics.jsonl\', \'a\') as log:\n                    log.write(json.dumps({\'seconds\': round(time.time() - started, 1),\n                                          \'lines\': [line for line in text.splitlines() if line.startswith(counters)]}) + \'\\n\')\n            except Exception:\n                pass\n            stop.wait(30)\n    threading.Thread(target=scrape, daemon=True).start()\nruntime = subprocess.run([python, \'-B\', \'-m\', \'src.qwen.runtime\', \'--config\', \'qwen_config.json\'])\nprint(\'runtime exit code\', runtime.returncode)\nif not os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\'):\n    stop.set()\n    dmon.terminate()\nstatus = json.loads(Path(\'qwen_run/status.json\').read_text())\nprint(json.dumps(status, indent=2)[:20000])\nif os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\') and not Path(\'/kaggle/working/submission.json\').exists():\n    message = (\'competition rerun failed: \' + str(status.get(\'failed_loudly\')) + \'; error \' + str(status.get(\'error\'))\n               + \'; timing \' + json.dumps(status.get(\'timing\')) + \'; copy \' + json.dumps(status.get(\'copy\')))\n    print(message, file=sys.stderr, flush=True)\n    raise RuntimeError(message)\nif not os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\'):  # the server\'s 10 s windows: L4 speed on the deployment\'s workload\n    import re\n    fields = {\'prompt\': r\'Avg prompt throughput: ([\\d.]+)\', \'generation\': r\'Avg generation throughput: ([\\d.]+)\',\n              \'running\': r\'Running: (\\d+) reqs\', \'waiting\': r\'Waiting: (\\d+) reqs\', \'kv_percent\': r\'KV cache usage: ([\\d.]+)%\',\n              \'prefix_hit_percent\': r\'Prefix cache hit rate: ([\\d.]+)%\',\n              \'time\': r\'(\\d\\d-\\d\\d \\d\\d:\\d\\d:\\d\\d)\'}\n    log_path = Path(\'qwen_run/server.log\')\n    windows = [{name: (m.group(1) if name == \'time\' else float(m.group(1))) for name, m in\n                ((name, re.search(pattern, line)) for name, pattern in fields.items()) if m}\n               for line in (log_path.read_text(errors=\'replace\').splitlines() if log_path.exists() else [])\n               if \'Avg generation throughput\' in line]\n    full = [w[\'generation\'] for w in windows if w.get(\'running\', 0) >= config[\'evaluation\'][\'concurrency\'] - 1]\n    speed = {\'windows\': windows, \'full_windows\': len(full), \'full_mean_generation\': round(sum(full) / len(full), 1) if full else None}\n    Path(\'qwen_run/speed.json\').write_text(json.dumps(speed))\n    print(\'speed\', {key: value for key, value in speed.items() if key != \'windows\'}, flush=True)\n# A commit run fails loudly when the runtime failed (e.g. the server never started); a competition rerun keeps the\n# valid placeholder file. Too few tokens for a grid in the short commit budget is not a failure.\nif not os.environ.get(\'KAGGLE_IS_COMPETITION_RERUN\'):\n    assert status[\'status\'] == \'COMPLETE\', status\nfrom src.utils.kaggle_runtime import validate_submission\nvalidate_submission(tasks, json.loads(Path(\'/kaggle/working/submission.json\').read_text()))\n', 'notebook_cell_6', 'exec'))


In [8]:
if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    try:
        probe, lines = '/kaggle/tmp/scipy-probe', []
        subprocess.check_call([sys.executable, '-m', 'venv', '--without-pip', '--clear', probe])
        for package in ('numpy', 'scipy'):
            name, record = next((n, r) for n, r in deployment['wheel_files'].items() if n.startswith(package + '-'))
            directory = next(p for p in Path('/kaggle/input').rglob(record['source'].split('/')[1]) if p.is_dir())
            lines.append(next(directory.rglob(name)).as_uri() + ' --hash=sha256:' + record['sha256'])
        Path(probe + '.txt').write_text('\n'.join(lines) + '\n')
        subprocess.check_call([sys.executable, '-m', 'pip', '--python', probe + '/bin/python', 'install', '--no-index', '--no-deps',
                               '--require-hashes', '--only-binary=:all:', '-q', '-r', probe + '.txt'])
        done = subprocess.run([probe + '/bin/python', '-c', 'import numpy, scipy, scipy.ndimage; '
                               'print(numpy.__version__, scipy.__version__, int(scipy.ndimage.label([[1, 0, 1]])[1]))'],
                              capture_output=True, text=True, timeout=300)
        print('SCIPY_PROBE', 'ok' if done.returncode == 0 else 'FAILED', done.stdout.strip(), done.stderr.strip()[-300:], flush=True)
    except Exception as error:
        print('SCIPY_PROBE FAILED', repr(error)[:300], flush=True)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


SCIPY_PROBE ok 2.4.6 1.18.1 2 Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


In [9]:
if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    tasks = json.loads(next(Path('/kaggle/input').rglob('arc-agi_test_challenges.json')).read_text())
    Path('/kaggle/working/submission.json').write_text(json.dumps(
        {task_id: [{'attempt_1': [[0]], 'attempt_2': [[0]]} for _ in task['test']] for task_id, task in tasks.items()}))
